In [1]:
# Cell 2A — Build all-tenor Corsi call candidate panel
# Purpose:
#   Build the raw all-tenor candidate panel for the Corsi-based short-call sleeve.
#
# Scope:
#   - Research only
#   - All existing tenors: 9, 12, 15, 18, 21, 24, 27, 30, 33
#   - Uses existing SPX-derived VIX-style vol and Corsi VRP inputs
#   - Builds raw theoretical 1SD / 3SD call strikes using SPY close
#   - No ThetaData option pricing yet
#   - No parameter sweep yet
#   - No selected trade logic yet
#
# Trade structure:
#   - Sell 1SD call
#   - Buy 3SD call
#   - SD = vix_style_vol_pct / 100 * sqrt(tenor / 365)
#
# Signal model to be researched later:
#   - Corsi VRP = ln(implied_variance / forecast_variance)
#   - Use model_vrp_log_final, z_3m_final, z_1y_final
#   - During initial sweeps, 3m z threshold = 1y z threshold

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

OUTPUT_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

print("=" * 100)
print("Cell 2A — Build all-tenor Corsi call candidate panel")
print("=" * 100)
print(f"Input:   {FINAL_SIGNAL_PATH}")
print(f"Output:  {OUTPUT_PANEL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------

EXPECTED_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

# --------------------------------------------------------------------------------------
# Load source
# --------------------------------------------------------------------------------------

df = pd.read_parquet(FINAL_SIGNAL_PATH)

required_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "implied_variance_final",
    "vix_style_vol_final",
    "forecast_variance_final",
    "forecast_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns from final signal panel: {missing_cols}")

# --------------------------------------------------------------------------------------
# Build candidate panel
# --------------------------------------------------------------------------------------

panel = df.loc[df["tenor"].isin(EXPECTED_TENORS), required_cols].copy()

panel = panel.rename(
    columns={
        "trade_date": "trade_date",
        "tenor": "tenor",
        "tenor_bucket": "tenor_bucket",
        "spy_close": "spy_close",
        "implied_variance_final": "implied_variance",
        "vix_style_vol_final": "vix_style_vol_pct",
        "forecast_variance_final": "forecast_variance",
        "forecast_vol_final": "forecast_vol_pct",
        "model_vrp_log_final": "corsi_vrp_log",
        "z_3m_final": "corsi_vrp_z_3m",
        "z_1y_final": "corsi_vrp_z_1y",
        "rsi14_final": "rsi14",
    }
)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel = panel.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Guardrails
# --------------------------------------------------------------------------------------

if panel.empty:
    raise ValueError("All-tenor Corsi call candidate panel is empty.")

duplicate_date_tenor = panel.duplicated(["trade_date", "tenor"]).sum()
if duplicate_date_tenor > 0:
    raise ValueError(f"Duplicate trade_date × tenor rows found: {duplicate_date_tenor:,}")

actual_tenors = sorted(panel["tenor"].dropna().unique().tolist())
missing_tenors = sorted(set(EXPECTED_TENORS) - set(actual_tenors))
unexpected_tenors = sorted(set(actual_tenors) - set(EXPECTED_TENORS))

if missing_tenors:
    raise ValueError(f"Missing expected tenors: {missing_tenors}")

if unexpected_tenors:
    raise ValueError(f"Unexpected tenors found: {unexpected_tenors}")

vix_median = panel["vix_style_vol_pct"].dropna().median()
forecast_vol_median = panel["forecast_vol_pct"].dropna().median()

if pd.notna(vix_median) and vix_median <= 1.0:
    raise ValueError(
        "vix_style_vol_pct appears to be decimal format, not percent format. "
        f"Median value is {vix_median:.6f}. Expected values like 12, 18, 25."
    )

if pd.notna(forecast_vol_median) and forecast_vol_median <= 1.0:
    raise ValueError(
        "forecast_vol_pct appears to be decimal format, not percent format. "
        f"Median value is {forecast_vol_median:.6f}. Expected values like 8, 14, 25."
    )

# --------------------------------------------------------------------------------------
# Raw theoretical strike construction
# --------------------------------------------------------------------------------------

panel["sqrt_t"] = np.sqrt(panel["tenor"] / 365.0)
panel["sd_move_decimal"] = (panel["vix_style_vol_pct"] / 100.0) * panel["sqrt_t"]

panel["short_call_1sd_raw"] = panel["spy_close"] * (1.0 + panel["sd_move_decimal"])
panel["long_call_3sd_raw"] = panel["spy_close"] * (1.0 + 3.0 * panel["sd_move_decimal"])

panel["call_spread_width_raw"] = (
    panel["long_call_3sd_raw"] - panel["short_call_1sd_raw"]
)

panel["short_call_otm_pct_raw"] = (
    panel["short_call_1sd_raw"] / panel["spy_close"] - 1.0
)

panel["long_call_otm_pct_raw"] = (
    panel["long_call_3sd_raw"] / panel["spy_close"] - 1.0
)

# Useful selection/sweep helper fields, but no selection decision yet.
panel["avg_corsi_z"] = (
    panel["corsi_vrp_z_3m"] + panel["corsi_vrp_z_1y"]
) / 2.0

panel["min_corsi_z"] = panel[["corsi_vrp_z_3m", "corsi_vrp_z_1y"]].min(axis=1)

# --------------------------------------------------------------------------------------
# Missingness and coverage audits
# --------------------------------------------------------------------------------------

core_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "implied_variance",
    "vix_style_vol_pct",
    "forecast_variance",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "call_spread_width_raw",
]

missing_summary = (
    panel[core_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(panel)

panel["candidate_inputs_complete"] = panel[core_cols].notna().all(axis=1)

tenor_summary = (
    panel
    .groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "count"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        complete_rows=("candidate_inputs_complete", "sum"),
        median_vix_style_vol_pct=("vix_style_vol_pct", "median"),
        median_forecast_vol_pct=("forecast_vol_pct", "median"),
        median_corsi_vrp_log=("corsi_vrp_log", "median"),
        median_z_3m=("corsi_vrp_z_3m", "median"),
        median_z_1y=("corsi_vrp_z_1y", "median"),
        median_rsi14=("rsi14", "median"),
        median_short_call_otm_pct_raw=("short_call_otm_pct_raw", "median"),
        median_long_call_otm_pct_raw=("long_call_otm_pct_raw", "median"),
        median_call_spread_width_raw=("call_spread_width_raw", "median"),
    )
)

tenor_summary["complete_row_pct"] = tenor_summary["complete_rows"] / tenor_summary["rows"]

date_coverage = (
    panel
    .groupby("trade_date", as_index=False)
    .agg(
        tenor_count=("tenor", "nunique"),
        complete_candidate_count=("candidate_inputs_complete", "sum"),
    )
)

date_coverage["has_all_expected_tenors"] = date_coverage["tenor_count"] == len(EXPECTED_TENORS)
date_coverage["has_all_complete_candidates"] = (
    date_coverage["complete_candidate_count"] == len(EXPECTED_TENORS)
)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_panel_path = AUDIT_DIR / f"02A_call_sleeve_corsi_all_tenor_raw_candidate_panel_{timestamp}.csv"
missing_summary_path = AUDIT_DIR / f"02A_call_sleeve_corsi_missing_summary_{timestamp}.csv"
tenor_summary_path = AUDIT_DIR / f"02A_call_sleeve_corsi_tenor_summary_{timestamp}.csv"
date_coverage_path = AUDIT_DIR / f"02A_call_sleeve_corsi_date_coverage_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02A_call_sleeve_corsi_raw_candidate_manifest_{timestamp}.json"

panel.to_parquet(OUTPUT_PANEL_PATH, index=False)
panel.to_csv(audit_panel_path, index=False)
missing_summary.to_csv(missing_summary_path, index=False)
tenor_summary.to_csv(tenor_summary_path, index=False)
date_coverage.to_csv(date_coverage_path, index=False)

manifest = {
    "cell": "2A",
    "description": "All-tenor Corsi short-call raw candidate panel",
    "created_at": timestamp,
    "input_path": str(FINAL_SIGNAL_PATH),
    "output_panel_path": str(OUTPUT_PANEL_PATH),
    "audit_panel_path": str(audit_panel_path),
    "missing_summary_path": str(missing_summary_path),
    "tenor_summary_path": str(tenor_summary_path),
    "date_coverage_path": str(date_coverage_path),
    "expected_tenors": EXPECTED_TENORS,
    "actual_tenors": actual_tenors,
    "date_min": str(panel["trade_date"].min().date()),
    "date_max": str(panel["trade_date"].max().date()),
    "rows": int(len(panel)),
    "unique_dates": int(panel["trade_date"].nunique()),
    "duplicate_date_tenor_rows": int(duplicate_date_tenor),
    "complete_candidate_rows": int(panel["candidate_inputs_complete"].sum()),
    "complete_candidate_pct": float(panel["candidate_inputs_complete"].mean()),
    "date_rows_with_all_expected_tenors": int(date_coverage["has_all_expected_tenors"].sum()),
    "date_rows_with_all_complete_candidates": int(date_coverage["has_all_complete_candidates"].sum()),
    "sqrt_t_convention": "sqrt(tenor / 365)",
    "short_call_raw_formula": "spy_close * (1 + vix_style_vol_pct / 100 * sqrt(tenor / 365))",
    "long_call_raw_formula": "spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt(tenor / 365))",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("Built all-tenor Corsi call candidate panel")
print(f"Rows:                              {len(panel):,}")
print(f"Unique dates:                      {panel['trade_date'].nunique():,}")
print(f"Date range:                        {panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
print(f"Tenors:                            {actual_tenors}")
print(f"Duplicate date × tenor rows:       {duplicate_date_tenor:,}")
print(f"Complete candidate rows:           {int(panel['candidate_inputs_complete'].sum()):,}")
print(f"Complete candidate pct:            {panel['candidate_inputs_complete'].mean():.2%}")
print()
print("Strike convention:")
print("  sqrt_t = sqrt(tenor / 365)")
print("  short_call_1sd_raw = spy_close * (1 + 1 * vix_style_vol_pct / 100 * sqrt_t)")
print("  long_call_3sd_raw  = spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt_t)")
print()
print("Sanity checks:")
print(f"  Median vix_style_vol_pct:         {vix_median:.4f}")
print(f"  Median forecast_vol_pct:          {forecast_vol_median:.4f}")
print(f"  Dates with all expected tenors:   {int(date_coverage['has_all_expected_tenors'].sum()):,} / {len(date_coverage):,}")
print(f"  Dates with all complete rows:     {int(date_coverage['has_all_complete_candidates'].sum()):,} / {len(date_coverage):,}")
print()
print("Saved:")
print(f"  Output parquet:                   {OUTPUT_PANEL_PATH}")
print(f"  Audit panel CSV:                  {audit_panel_path}")
print(f"  Missing summary CSV:              {missing_summary_path}")
print(f"  Tenor summary CSV:                {tenor_summary_path}")
print(f"  Date coverage CSV:                {date_coverage_path}")
print(f"  Manifest JSON:                    {manifest_path}")

print()
print("=" * 100)
print("Missing summary")
print("=" * 100)
display(missing_summary)

print()
print("=" * 100)
print("Tenor summary")
print("=" * 100)
display(tenor_summary)

print()
print("=" * 100)
print("Latest date candidates")
print("=" * 100)
latest_date = panel["trade_date"].max()
display(
    panel.loc[panel["trade_date"] == latest_date]
    .sort_values("tenor")
    [
        [
            "trade_date",
            "tenor",
            "tenor_bucket",
            "spy_close",
            "vix_style_vol_pct",
            "forecast_vol_pct",
            "corsi_vrp_log",
            "corsi_vrp_z_3m",
            "corsi_vrp_z_1y",
            "rsi14",
            "short_call_1sd_raw",
            "long_call_3sd_raw",
            "call_spread_width_raw",
            "candidate_inputs_complete",
        ]
    ]
)

Cell 2A — Build all-tenor Corsi call candidate panel
Input:   C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet
Output:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet

Built all-tenor Corsi call candidate panel
Rows:                              14,715
Unique dates:                      1,635
Date range:                        2020-01-02 to 2026-07-08
Tenors:                            [9, 12, 15, 18, 21, 24, 27, 30, 33]
Duplicate date × tenor rows:       0
Complete candidate rows:           12,420
Complete candidate pct:            84.40%

Strike convention:
  sqrt_t = sqrt(tenor / 365)
  short_call_1sd_raw = spy_close * (1 + 1 * vix_style_vol_pct / 100 * sqrt_t)
  long_call_3sd_raw  = spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt_t)

Sanity checks:
  Median vix_style_vol_pct:         18.4128
  Median forecast_vol_pct:          13.9888
  Dates with al

,column,missing_count,missing_pct
0,trade_date,0,0.000000
1,tenor,0,0.000000
2,tenor_bucket,27,0.001835
3,spy_close,27,0.001835
4,implied_variance,0,0.000000
5,vix_style_vol_pct,0,0.000000
6,forecast_variance,0,0.000000
7,forecast_vol_pct,0,0.000000
8,corsi_vrp_log,0,0.000000
9,corsi_vrp_z_3m,567,0.038532



Tenor summary


,tenor,rows,date_min,date_max,complete_rows,median_vix_style_vol_pct,median_forecast_vol_pct,median_corsi_vrp_log,median_z_3m,median_z_1y,median_rsi14,median_short_call_otm_pct_raw,median_long_call_otm_pct_raw,median_call_spread_width_raw,complete_row_pct
0,9,1635,2020-01-02,2026-07-08,1380,17.515786,13.224036,0.540406,-0.042942,-0.153739,57.195255,0.027534,0.082602,26.165654,0.844037
1,12,1635,2020-01-02,2026-07-08,1380,17.809555,13.192627,0.564562,-0.041767,-0.145995,57.195255,0.032343,0.097029,30.728249,0.844037
2,15,1635,2020-01-02,2026-07-08,1380,17.917531,13.778479,0.499975,-0.067391,-0.177360,57.195255,0.036426,0.109279,34.794209,0.844037
3,18,1635,2020-01-02,2026-07-08,1380,18.212769,13.715855,0.522889,-0.060154,-0.210078,57.195255,0.040473,0.121419,38.699576,0.844037
4,21,1635,2020-01-02,2026-07-08,1380,18.384014,14.054020,0.498948,-0.068424,-0.242724,57.195255,0.044123,0.132368,42.294111,0.844037
5,24,1635,2020-01-02,2026-07-08,1380,18.580106,14.087548,0.512516,-0.093706,-0.253646,57.195255,0.047659,0.142976,45.520278,0.844037
6,27,1635,2020-01-02,2026-07-08,1380,18.750958,14.236235,0.518564,-0.117991,-0.234771,57.195255,0.051085,0.153255,48.83928,0.844037
7,30,1635,2020-01-02,2026-07-08,1380,18.950384,14.441723,0.501646,-0.140280,-0.229082,57.195255,0.054371,0.163112,51.952965,0.844037
8,33,1635,2020-01-02,2026-07-08,1380,19.104130,14.449255,0.514078,-0.148171,-0.238308,57.195255,0.057456,0.172367,54.956408,0.844037



Latest date candidates


,trade_date,tenor,tenor_bucket,spy_close,vix_style_vol_pct,forecast_vol_pct,corsi_vrp_log,corsi_vrp_z_3m,corsi_vrp_z_1y,rsi14,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw,candidate_inputs_complete
14706,2026-07-08,9,None,NaN,14.142112,11.315075,0.446042,0.294892,-0.387636,47.0441,<NA>,<NA>,<NA>,False
14707,2026-07-08,12,None,NaN,14.623586,11.416124,0.495218,0.322649,-0.383688,47.0441,<NA>,<NA>,<NA>,False
14708,2026-07-08,15,None,NaN,14.905007,12.018682,0.430470,0.292371,-0.407554,47.0441,<NA>,<NA>,<NA>,False
14709,2026-07-08,18,None,NaN,15.478564,11.963703,0.515158,0.457871,-0.270658,47.0441,<NA>,<NA>,<NA>,False
14710,2026-07-08,21,None,NaN,16.035208,12.209146,0.545203,0.608556,-0.130088,47.0441,<NA>,<NA>,<NA>,False
14711,2026-07-08,24,None,NaN,16.397687,12.266581,0.580523,0.691769,-0.065768,47.0441,<NA>,<NA>,<NA>,False
14712,2026-07-08,27,None,NaN,16.599352,12.330703,0.594543,0.711117,-0.058179,47.0441,<NA>,<NA>,<NA>,False
14713,2026-07-08,30,None,NaN,16.758938,12.532342,0.581238,0.723058,-0.056647,47.0441,<NA>,<NA>,<NA>,False
14714,2026-07-08,33,None,NaN,16.945993,12.561058,0.598860,0.712617,-0.058700,47.0441,<NA>,<NA>,<NA>,False


In [2]:
# Cell 2B — ThetaData option price source inventory for SPY call spreads
# Purpose:
#   Find existing local ThetaData / option-chain files and scripts that can be used
#   to attach actual SPY option bid/ask entry prices to the Corsi call candidate panel.
#
# Scope:
#   - Inventory only
#   - No ThetaData API pulls yet
#   - No option pricing yet
#   - No parameter sweep yet
#
# Looking for:
#   - SPY option chain / quote files
#   - ThetaData pull scripts
#   - Existing option pricing / backtest artifacts from prior put work
#   - Columns for date, expiration, strike, right/call-put, bid, ask, mid, quote timestamp

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime
import os
import re

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B — ThetaData option price source inventory")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Audit dir:    {AUDIT_DIR}")
print()

# --------------------------------------------------------------------------------------
# Search configuration
# --------------------------------------------------------------------------------------

SEARCH_ROOTS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "scripts",
]

SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]

DATA_EXTENSIONS = {".parquet", ".csv", ".feather", ".pkl", ".pickle"}
TEXT_EXTENSIONS = {".py", ".ipynb", ".txt", ".md", ".json"}

# Filename/path terms that suggest options / ThetaData relevance
PATH_KEYWORDS = [
    "theta",
    "thetadata",
    "option",
    "options",
    "chain",
    "quote",
    "quotes",
    "contract",
    "expiry",
    "expiration",
    "strike",
    "bid",
    "ask",
    "mid",
    "spy",
    "spx",
    "put",
    "call",
]

# Column terms required for actual option pricing
COLUMN_GROUPS = {
    "date": ["date", "trade_date", "quote_date", "asof_date", "timestamp", "datetime", "ms_of_day"],
    "expiration": ["expiration", "expiry", "exp", "expiration_date"],
    "strike": ["strike"],
    "right": ["right", "option_type", "cp", "call_put", "put_call", "root"],
    "bid": ["bid", "bid_price"],
    "ask": ["ask", "ask_price"],
    "mid": ["mid", "mark", "price", "close"],
    "underlying": ["underlying", "root", "symbol", "ticker"],
}

TEXT_SEARCH_TERMS = [
    "thetadata",
    "ThetaData",
    "option chain",
    "bulk_hist",
    "hist/option",
    "bid",
    "ask",
    "strike",
    "expiration",
    "SPY",
    "SPX",
]

MAX_FILE_SIZE_TO_TEXT_SCAN_MB = 10
MAX_CSV_ROWS_TO_SAMPLE = 5
MAX_DISPLAY_ROWS = 200

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def path_score(path: Path) -> int:
    s = str(path).lower()
    return sum(1 for k in PATH_KEYWORDS if k in s)

def safe_file_size_mb(path: Path) -> float:
    try:
        return path.stat().st_size / (1024 * 1024)
    except Exception:
        return np.nan

def detect_column_groups(columns):
    cols = [str(c) for c in columns]
    lower_map = {c: c.lower() for c in cols}
    
    detected = {}
    for group, terms in COLUMN_GROUPS.items():
        matches = []
        for col, low in lower_map.items():
            if any(term in low for term in terms):
                matches.append(col)
        detected[group] = matches
    
    return detected

def option_pricing_column_score(detected_groups) -> int:
    required_groups = ["date", "expiration", "strike", "right", "bid", "ask"]
    return sum(1 for g in required_groups if len(detected_groups.get(g, [])) > 0)

def read_parquet_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        out["rows"] = pf.metadata.num_rows
        out["cols"] = len(pf.schema.names)
        out["columns"] = list(pf.schema.names)
    except Exception as e1:
        try:
            sample = pd.read_parquet(path)
            out["rows"] = len(sample)
            out["cols"] = len(sample.columns)
            out["columns"] = list(sample.columns)
        except Exception as e2:
            out["error"] = f"pyarrow_error={repr(e1)} | pandas_error={repr(e2)}"
    
    return out

def read_csv_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        sample = pd.read_csv(path, nrows=MAX_CSV_ROWS_TO_SAMPLE)
        out["cols"] = len(sample.columns)
        out["columns"] = list(sample.columns)
        # Row count for huge CSVs can be expensive; skip exact rows.
        out["rows"] = None
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def read_feather_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        sample = pd.read_feather(path)
        out["rows"] = len(sample)
        out["cols"] = len(sample.columns)
        out["columns"] = list(sample.columns)
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def read_pickle_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        obj = pd.read_pickle(path)
        if isinstance(obj, pd.DataFrame):
            out["rows"] = len(obj)
            out["cols"] = len(obj.columns)
            out["columns"] = list(obj.columns)
        else:
            out["error"] = f"pickle object is {type(obj)} not DataFrame"
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def scan_text_file(path: Path):
    result = {
        "text_hits": [],
        "text_hit_count": 0,
        "error": None,
    }
    
    size_mb = safe_file_size_mb(path)
    if pd.notna(size_mb) and size_mb > MAX_FILE_SIZE_TO_TEXT_SCAN_MB:
        result["error"] = f"skipped_text_scan_file_too_large_{size_mb:.2f}mb"
        return result
    
    try:
        text = path.read_text(errors="ignore")
        hits = []
        for term in TEXT_SEARCH_TERMS:
            if term.lower() in text.lower():
                hits.append(term)
        result["text_hits"] = hits
        result["text_hit_count"] = len(hits)
    except Exception as e:
        result["error"] = repr(e)
    
    return result

# --------------------------------------------------------------------------------------
# Inventory files
# --------------------------------------------------------------------------------------

all_paths = []

for root in SEARCH_ROOTS:
    for path in root.rglob("*"):
        if path.is_file():
            ext = path.suffix.lower()
            if ext in DATA_EXTENSIONS or ext in TEXT_EXTENSIONS:
                all_paths.append(path)

print(f"Scanned roots: {[str(p) for p in SEARCH_ROOTS]}")
print(f"Files considered: {len(all_paths):,}")
print()

data_rows = []
text_rows = []

for path in all_paths:
    ext = path.suffix.lower()
    rel_path = str(path.relative_to(PROJECT_ROOT))
    pscore = path_score(path)
    size_mb = safe_file_size_mb(path)
    
    if ext in DATA_EXTENSIONS:
        metadata = {
            "rows": None,
            "cols": None,
            "columns": [],
            "error": None,
        }
        
        if ext == ".parquet":
            metadata = read_parquet_metadata(path)
        elif ext == ".csv":
            metadata = read_csv_metadata(path)
        elif ext == ".feather":
            metadata = read_feather_metadata(path)
        elif ext in {".pkl", ".pickle"}:
            metadata = read_pickle_metadata(path)
        
        columns = metadata.get("columns", []) or []
        detected_groups = detect_column_groups(columns)
        col_score = option_pricing_column_score(detected_groups)
        
        # Candidate if path suggests relevance OR columns suggest option pricing.
        is_candidate = (pscore >= 2) or (col_score >= 3)
        
        data_rows.append({
            "relative_path": rel_path,
            "full_path": str(path),
            "extension": ext,
            "size_mb": size_mb,
            "path_keyword_score": pscore,
            "option_pricing_column_score": col_score,
            "is_candidate": is_candidate,
            "rows": metadata.get("rows"),
            "cols": metadata.get("cols"),
            "date_cols": ", ".join(detected_groups.get("date", [])),
            "expiration_cols": ", ".join(detected_groups.get("expiration", [])),
            "strike_cols": ", ".join(detected_groups.get("strike", [])),
            "right_cols": ", ".join(detected_groups.get("right", [])),
            "bid_cols": ", ".join(detected_groups.get("bid", [])),
            "ask_cols": ", ".join(detected_groups.get("ask", [])),
            "mid_price_cols": ", ".join(detected_groups.get("mid", [])),
            "underlying_cols": ", ".join(detected_groups.get("underlying", [])),
            "columns": ", ".join([str(c) for c in columns]),
            "error": metadata.get("error"),
        })
    
    elif ext in TEXT_EXTENSIONS:
        # Only scan text files that have at least a weak path keyword score,
        # plus notebooks/scripts generally worth scanning.
        should_scan = (pscore >= 1) or (ext in {".py", ".ipynb"})
        
        if should_scan:
            text_scan = scan_text_file(path)
            is_candidate = pscore >= 1 or text_scan["text_hit_count"] >= 2
            
            text_rows.append({
                "relative_path": rel_path,
                "full_path": str(path),
                "extension": ext,
                "size_mb": size_mb,
                "path_keyword_score": pscore,
                "text_hit_count": text_scan["text_hit_count"],
                "text_hits": ", ".join(text_scan["text_hits"]),
                "is_candidate": is_candidate,
                "error": text_scan["error"],
            })

data_inventory = pd.DataFrame(data_rows)
text_inventory = pd.DataFrame(text_rows)

if data_inventory.empty:
    data_candidates = pd.DataFrame()
else:
    data_candidates = (
        data_inventory.loc[data_inventory["is_candidate"]]
        .sort_values(
            ["option_pricing_column_score", "path_keyword_score", "size_mb"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

if text_inventory.empty:
    text_candidates = pd.DataFrame()
else:
    text_candidates = (
        text_inventory.loc[text_inventory["is_candidate"]]
        .sort_values(
            ["text_hit_count", "path_keyword_score", "size_mb"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

# --------------------------------------------------------------------------------------
# Try to identify strongest pricing-source candidates
# --------------------------------------------------------------------------------------

pricing_ready_mask = pd.Series(False, index=data_candidates.index)

if not data_candidates.empty:
    pricing_ready_mask = (
        data_candidates["date_cols"].astype(str).str.len().gt(0)
        & data_candidates["expiration_cols"].astype(str).str.len().gt(0)
        & data_candidates["strike_cols"].astype(str).str.len().gt(0)
        & data_candidates["bid_cols"].astype(str).str.len().gt(0)
        & data_candidates["ask_cols"].astype(str).str.len().gt(0)
    )

pricing_ready_candidates = data_candidates.loc[pricing_ready_mask].copy()

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data_inventory_path = AUDIT_DIR / f"02B_option_price_source_data_inventory_all_{timestamp}.csv"
data_candidates_path = AUDIT_DIR / f"02B_option_price_source_data_candidates_{timestamp}.csv"
pricing_ready_path = AUDIT_DIR / f"02B_option_price_source_pricing_ready_candidates_{timestamp}.csv"
text_inventory_path = AUDIT_DIR / f"02B_option_price_source_text_inventory_all_{timestamp}.csv"
text_candidates_path = AUDIT_DIR / f"02B_option_price_source_text_candidates_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02B_option_price_source_inventory_manifest_{timestamp}.json"

data_inventory.to_csv(data_inventory_path, index=False)
data_candidates.to_csv(data_candidates_path, index=False)
pricing_ready_candidates.to_csv(pricing_ready_path, index=False)
text_inventory.to_csv(text_inventory_path, index=False)
text_candidates.to_csv(text_candidates_path, index=False)

manifest = {
    "cell": "2B",
    "description": "ThetaData / option price source inventory for SPY call spreads",
    "created_at": timestamp,
    "project_root": str(PROJECT_ROOT),
    "search_roots": [str(p) for p in SEARCH_ROOTS],
    "files_considered": len(all_paths),
    "data_files_inventory_count": int(len(data_inventory)),
    "data_candidate_count": int(len(data_candidates)),
    "pricing_ready_candidate_count": int(len(pricing_ready_candidates)),
    "text_files_inventory_count": int(len(text_inventory)),
    "text_candidate_count": int(len(text_candidates)),
    "data_inventory_path": str(data_inventory_path),
    "data_candidates_path": str(data_candidates_path),
    "pricing_ready_path": str(pricing_ready_path),
    "text_inventory_path": str(text_inventory_path),
    "text_candidates_path": str(text_candidates_path),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Inventory summary")
print("=" * 100)
print(f"Data files inventoried:              {len(data_inventory):,}")
print(f"Data candidates:                     {len(data_candidates):,}")
print(f"Pricing-ready candidates:            {len(pricing_ready_candidates):,}")
print(f"Text/script files inventoried:        {len(text_inventory):,}")
print(f"Text/script candidates:               {len(text_candidates):,}")
print()

print("Saved:")
print(f"  Data inventory all:                 {data_inventory_path}")
print(f"  Data candidates:                    {data_candidates_path}")
print(f"  Pricing-ready candidates:           {pricing_ready_path}")
print(f"  Text inventory all:                 {text_inventory_path}")
print(f"  Text/script candidates:             {text_candidates_path}")
print(f"  Manifest:                           {manifest_path}")

print()
print("=" * 100)
print("Pricing-ready data candidates")
print("=" * 100)

if pricing_ready_candidates.empty:
    print("No pricing-ready data files found with date + expiration + strike + bid + ask columns.")
else:
    display(
        pricing_ready_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "option_pricing_column_score",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
            ]
        ]
    )

print()
print("=" * 100)
print("Top data candidates")
print("=" * 100)

if data_candidates.empty:
    print("No data candidates found.")
else:
    display(
        data_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "path_keyword_score",
                "option_pricing_column_score",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
                "error",
            ]
        ]
    )

print()
print("=" * 100)
print("Top text/script candidates")
print("=" * 100)

if text_candidates.empty:
    print("No text/script candidates found.")
else:
    display(
        text_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "size_mb",
                "path_keyword_score",
                "text_hit_count",
                "text_hits",
                "error",
            ]
        ]
    )

Cell 2B — ThetaData option price source inventory
Project root: C:\Users\patri\vrp_project
Audit dir:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research

Scanned roots: ['C:\\Users\\patri\\vrp_project\\data', 'C:\\Users\\patri\\vrp_project\\notebooks']
Files considered: 14,515

Inventory summary
Data files inventoried:              14,084
Data candidates:                     11,496
Pricing-ready candidates:            11,058
Text/script files inventoried:        137
Text/script candidates:               119

Saved:
  Data inventory all:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B_option_price_source_data_inventory_all_20260710_094512.csv
  Data candidates:                    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B_option_price_source_data_candidates_20260710_094512.csv
  Pricing-ready candidates:           C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B_option_price_source_pricing_ready_candidates_20260

,relative_path,extension,rows,cols,size_mb,option_pricing_column_score,date_cols,expiration_cols,strike_cols,right_cols,bid_cols,ask_cols,mid_price_cols,underlying_cols
0,data\raw\thetadata_chains\SPX_20260529_2026061...,.pkl,1278.0,14.0,0.151597,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
1,data\raw\thetadata_chains\SPX_20260601_2026061...,.pkl,1278.0,14.0,0.151597,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
2,data\raw\thetadata_chains\SPX_20260602_2026061...,.pkl,1278.0,14.0,0.151597,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
3,data\raw\thetadata_chains\SPX_20260603_2026061...,.pkl,1278.0,14.0,0.151597,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
4,data\raw\thetadata_chains\SPX_20260604_2026061...,.pkl,1278.0,14.0,0.151597,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,data\raw\thetadata_chains\SPXW_20260512_202605...,.pkl,984.0,14.0,0.117907,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
196,data\raw\thetadata_chains\SPXW_20260513_202605...,.pkl,984.0,14.0,0.117907,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
197,data\raw\thetadata_chains\SPX_20250627_2025071...,.pkl,988.0,14.0,0.117439,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root
198,data\raw\thetadata_chains\SPX_20250630_2025071...,.pkl,988.0,14.0,0.117439,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root



Top data candidates


,relative_path,extension,rows,cols,size_mb,path_keyword_score,option_pricing_column_score,date_cols,expiration_cols,strike_cols,right_cols,bid_cols,ask_cols,mid_price_cols,underlying_cols,error
0,data\raw\thetadata_chains\SPX_20260529_2026061...,.pkl,1278.0,14.0,0.151597,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
1,data\raw\thetadata_chains\SPX_20260601_2026061...,.pkl,1278.0,14.0,0.151597,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
2,data\raw\thetadata_chains\SPX_20260602_2026061...,.pkl,1278.0,14.0,0.151597,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
3,data\raw\thetadata_chains\SPX_20260603_2026061...,.pkl,1278.0,14.0,0.151597,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
4,data\raw\thetadata_chains\SPX_20260604_2026061...,.pkl,1278.0,14.0,0.151597,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,data\raw\thetadata_chains\SPXW_20260512_202605...,.pkl,984.0,14.0,0.117907,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
196,data\raw\thetadata_chains\SPXW_20260513_202605...,.pkl,984.0,14.0,0.117907,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
197,data\raw\thetadata_chains\SPX_20250627_2025071...,.pkl,988.0,14.0,0.117439,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None
198,data\raw\thetadata_chains\SPX_20250630_2025071...,.pkl,988.0,14.0,0.117439,4,6,timestamp,expiration,strike,"root, right","bid, bid_size, bid_exchange, bid_condition","ask, ask_size, ask_exchange, ask_condition",mid,root,None



Top text/script candidates


,relative_path,extension,size_mb,path_keyword_score,text_hit_count,text_hits,error
0,notebooks\vrp_intraday_source_inventory_v1.py,.py,0.020200,0,10,"thetadata, ThetaData, option chain, bulk_hist,...",None
1,notebooks\vrp_upstream_source_deep_dive_v1.py,.py,0.031092,0,9,"thetadata, ThetaData, option chain, bid, ask, ...",None
2,notebooks\vrp_intraday_thetadata_live_probe_v1.py,.py,0.023219,2,8,"thetadata, ThetaData, bid, ask, strike, expira...",None
3,notebooks\vrp_intraday_thetadata_working_endpo...,.py,0.015119,2,8,"thetadata, ThetaData, bid, ask, strike, expira...",None
4,data\audit\production_inventory\vrp_vix_functi...,.py,0.132316,0,8,"thetadata, ThetaData, option chain, bid, ask, ...",None
...,...,...,...,...,...,...,...
114,data\audit\call_sleeve_research\01D_call_sleev...,.json,0.001266,1,0,,None
115,data\audit\put_phase3_portfolio_config_v0_1.json,.json,0.000952,1,0,,None
116,data\audit\put_phase4_vol_of_vol_diagnostic_co...,.json,0.000910,1,0,,None
117,notebooks\call_sleeve_research\02_call_sleeve_...,.ipynb,NaN,1,0,,"FileNotFoundError(2, 'No such file or directory')"


In [3]:
# Cell 2B.1 — Identify SPY option-chain availability
# Purpose:
#   Narrow the broad Cell 2B inventory to determine whether local SPY option-chain
#   files already exist and are usable for actual SPY call-spread entry pricing.
#
# Scope:
#   - Read-only inspection
#   - No ThetaData API pulls
#   - No option pricing yet
#   - No parameter sweep yet
#
# Goal:
#   Decide whether Cell 2C can price from existing local SPY chains,
#   or whether we first need a ThetaData SPY pull/cache step.

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B.1 — Identify SPY option-chain availability")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Audit dir:    {AUDIT_DIR}")
print()

# --------------------------------------------------------------------------------------
# Locate latest Cell 2B pricing-ready inventory
# --------------------------------------------------------------------------------------

pricing_ready_files = sorted(
    AUDIT_DIR.glob("02B_option_price_source_pricing_ready_candidates_*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not pricing_ready_files:
    raise FileNotFoundError(
        "No Cell 2B pricing-ready inventory found. Run Cell 2B first."
    )

PRICING_READY_PATH = pricing_ready_files[0]

print(f"Using pricing-ready inventory: {PRICING_READY_PATH}")
print()

inv = pd.read_csv(PRICING_READY_PATH)

if inv.empty:
    raise ValueError("Pricing-ready inventory is empty.")

required_inv_cols = ["relative_path", "full_path", "extension", "rows", "cols"]
missing_inv_cols = [c for c in required_inv_cols if c not in inv.columns]
if missing_inv_cols:
    raise ValueError(f"Pricing-ready inventory missing required columns: {missing_inv_cols}")

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def normalize_path_string(x):
    return str(x).replace("/", "\\").lower()

def infer_root_from_filename(path_str: str):
    """
    Infer root from filename patterns like:
      SPY_20260529_20260619...
      SPX_20260529_20260619...
      SPXW_20260529_20260619...
    """
    name = Path(str(path_str)).name.upper()
    
    if re.match(r"^SPY[_\-]", name):
        return "SPY"
    if re.match(r"^SPXW[_\-]", name):
        return "SPXW"
    if re.match(r"^SPX[_\-]", name):
        return "SPX"
    
    # Slightly looser fallback, but avoid classifying SPX/SPXW as SPY.
    if "SPY" in name and "SPX" not in name:
        return "SPY"
    if "SPXW" in name:
        return "SPXW"
    if "SPX" in name:
        return "SPX"
    
    return "UNKNOWN"

def parse_dates_from_filename(path_str: str):
    """
    Try to pull all YYYYMMDD tokens from filename.
    Usually first token is quote/trade date and second token may be expiration.
    """
    name = Path(str(path_str)).name
    tokens = re.findall(r"(20\d{6})", name)
    dates = []
    for t in tokens:
        try:
            dates.append(pd.to_datetime(t, format="%Y%m%d"))
        except Exception:
            pass
    
    return dates

def load_option_file(path: Path):
    ext = path.suffix.lower()
    if ext == ".pkl" or ext == ".pickle":
        return pd.read_pickle(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".feather":
        return pd.read_feather(path)
    raise ValueError(f"Unsupported extension for sample load: {ext}")

def safe_unique_values(df, col, max_vals=20):
    if col not in df.columns:
        return []
    vals = pd.Series(df[col]).dropna().unique().tolist()
    vals = sorted(vals, key=lambda x: str(x))
    return vals[:max_vals]

def safe_min_max_date(df, col):
    if col not in df.columns:
        return (None, None)
    s = pd.to_datetime(df[col], errors="coerce")
    if s.notna().sum() == 0:
        return (None, None)
    return (s.min(), s.max())

def safe_min_max_numeric(df, col):
    if col not in df.columns:
        return (None, None)
    s = pd.to_numeric(df[col], errors="coerce")
    if s.notna().sum() == 0:
        return (None, None)
    return (s.min(), s.max())

def find_first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# --------------------------------------------------------------------------------------
# Classify inventory by root
# --------------------------------------------------------------------------------------

inv["root_from_filename"] = inv["relative_path"].apply(infer_root_from_filename)

inv["filename_dates"] = inv["relative_path"].apply(parse_dates_from_filename)
inv["filename_first_date"] = inv["filename_dates"].apply(lambda xs: xs[0] if len(xs) >= 1 else pd.NaT)
inv["filename_second_date"] = inv["filename_dates"].apply(lambda xs: xs[1] if len(xs) >= 2 else pd.NaT)

inv["file_exists"] = inv["full_path"].apply(lambda x: Path(str(x)).exists())

root_summary = (
    inv
    .groupby(["root_from_filename", "extension"], as_index=False)
    .agg(
        file_count=("relative_path", "count"),
        existing_file_count=("file_exists", "sum"),
        total_rows=("rows", "sum"),
        min_filename_first_date=("filename_first_date", "min"),
        max_filename_first_date=("filename_first_date", "max"),
        min_filename_second_date=("filename_second_date", "min"),
        max_filename_second_date=("filename_second_date", "max"),
    )
    .sort_values(["root_from_filename", "extension"])
)

spy_candidates = (
    inv.loc[inv["root_from_filename"] == "SPY"]
    .sort_values(["filename_first_date", "filename_second_date", "relative_path"])
    .reset_index(drop=True)
)

spx_candidates = inv.loc[inv["root_from_filename"].isin(["SPX", "SPXW"])].copy()
unknown_candidates = inv.loc[inv["root_from_filename"] == "UNKNOWN"].copy()

# --------------------------------------------------------------------------------------
# SPY date coverage from filenames
# --------------------------------------------------------------------------------------

if not spy_candidates.empty:
    spy_date_coverage = (
        spy_candidates
        .groupby("filename_first_date", as_index=False)
        .agg(
            file_count=("relative_path", "count"),
            min_expiry_hint=("filename_second_date", "min"),
            max_expiry_hint=("filename_second_date", "max"),
            total_rows=("rows", "sum"),
        )
        .sort_values("filename_first_date")
    )
else:
    spy_date_coverage = pd.DataFrame(
        columns=[
            "filename_first_date",
            "file_count",
            "min_expiry_hint",
            "max_expiry_hint",
            "total_rows",
        ]
    )

# --------------------------------------------------------------------------------------
# Sample-load SPY files if found
# --------------------------------------------------------------------------------------

sample_rows = []
sampled_dfs_preview = []

if not spy_candidates.empty:
    # Sample first 3, middle 3, last 3 unique files to check actual root/right/bid/ask structure.
    n = len(spy_candidates)
    sample_indices = sorted(set(
        list(range(min(3, n)))
        + [max(0, n // 2 - 1), n // 2, min(n - 1, n // 2 + 1)]
        + list(range(max(0, n - 3), n))
    ))
    
    sample_candidates = spy_candidates.iloc[sample_indices].copy()
    
    for _, row in sample_candidates.iterrows():
        path = Path(row["full_path"])
        result = {
            "relative_path": row["relative_path"],
            "full_path": str(path),
            "extension": path.suffix.lower(),
            "file_exists": path.exists(),
            "load_ok": False,
            "rows": None,
            "cols": None,
            "columns": None,
            "root_values": None,
            "right_values": None,
            "expiration_min": None,
            "expiration_max": None,
            "timestamp_min": None,
            "timestamp_max": None,
            "strike_min": None,
            "strike_max": None,
            "bid_min": None,
            "bid_max": None,
            "ask_min": None,
            "ask_max": None,
            "mid_min": None,
            "mid_max": None,
            "bid_missing": None,
            "ask_missing": None,
            "error": None,
        }
        
        try:
            df_sample = load_option_file(path)
            
            result["load_ok"] = True
            result["rows"] = len(df_sample)
            result["cols"] = len(df_sample.columns)
            result["columns"] = ", ".join(map(str, df_sample.columns))
            
            root_col = find_first_existing_col(df_sample, ["root", "underlying", "symbol", "ticker"])
            right_col = find_first_existing_col(df_sample, ["right", "option_type", "cp", "call_put", "put_call"])
            exp_col = find_first_existing_col(df_sample, ["expiration", "expiry", "expiration_date", "exp"])
            ts_col = find_first_existing_col(df_sample, ["timestamp", "quote_date", "trade_date", "date", "datetime"])
            strike_col = find_first_existing_col(df_sample, ["strike"])
            bid_col = find_first_existing_col(df_sample, ["bid", "bid_price"])
            ask_col = find_first_existing_col(df_sample, ["ask", "ask_price"])
            mid_col = find_first_existing_col(df_sample, ["mid", "mark"])
            
            result["root_values"] = ", ".join(map(str, safe_unique_values(df_sample, root_col))) if root_col else None
            result["right_values"] = ", ".join(map(str, safe_unique_values(df_sample, right_col))) if right_col else None
            
            exp_min, exp_max = safe_min_max_date(df_sample, exp_col) if exp_col else (None, None)
            ts_min, ts_max = safe_min_max_date(df_sample, ts_col) if ts_col else (None, None)
            strike_min, strike_max = safe_min_max_numeric(df_sample, strike_col) if strike_col else (None, None)
            bid_min, bid_max = safe_min_max_numeric(df_sample, bid_col) if bid_col else (None, None)
            ask_min, ask_max = safe_min_max_numeric(df_sample, ask_col) if ask_col else (None, None)
            mid_min, mid_max = safe_min_max_numeric(df_sample, mid_col) if mid_col else (None, None)
            
            result["expiration_min"] = exp_min
            result["expiration_max"] = exp_max
            result["timestamp_min"] = ts_min
            result["timestamp_max"] = ts_max
            result["strike_min"] = strike_min
            result["strike_max"] = strike_max
            result["bid_min"] = bid_min
            result["bid_max"] = bid_max
            result["ask_min"] = ask_min
            result["ask_max"] = ask_max
            result["mid_min"] = mid_min
            result["mid_max"] = mid_max
            
            result["bid_missing"] = int(df_sample[bid_col].isna().sum()) if bid_col else None
            result["ask_missing"] = int(df_sample[ask_col].isna().sum()) if ask_col else None
            
            # Save a tiny preview with source path attached.
            preview = df_sample.head(5).copy()
            preview.insert(0, "source_relative_path", row["relative_path"])
            sampled_dfs_preview.append(preview)
            
        except Exception as e:
            result["error"] = repr(e)
        
        sample_rows.append(result)

spy_sample_summary = pd.DataFrame(sample_rows)

if sampled_dfs_preview:
    spy_sample_preview = pd.concat(sampled_dfs_preview, ignore_index=True)
else:
    spy_sample_preview = pd.DataFrame()

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

classified_inventory_path = AUDIT_DIR / f"02B1_option_chain_inventory_classified_{timestamp}.csv"
root_summary_path = AUDIT_DIR / f"02B1_option_chain_root_summary_{timestamp}.csv"
spy_candidates_path = AUDIT_DIR / f"02B1_spy_option_chain_candidates_{timestamp}.csv"
spy_date_coverage_path = AUDIT_DIR / f"02B1_spy_option_chain_date_coverage_{timestamp}.csv"
spy_sample_summary_path = AUDIT_DIR / f"02B1_spy_option_chain_sample_summary_{timestamp}.csv"
spy_sample_preview_path = AUDIT_DIR / f"02B1_spy_option_chain_sample_preview_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02B1_spy_option_chain_availability_manifest_{timestamp}.json"

inv.to_csv(classified_inventory_path, index=False)
root_summary.to_csv(root_summary_path, index=False)
spy_candidates.to_csv(spy_candidates_path, index=False)
spy_date_coverage.to_csv(spy_date_coverage_path, index=False)
spy_sample_summary.to_csv(spy_sample_summary_path, index=False)
spy_sample_preview.to_csv(spy_sample_preview_path, index=False)

manifest = {
    "cell": "2B.1",
    "description": "Identify local SPY option-chain availability",
    "created_at": timestamp,
    "pricing_ready_inventory_path": str(PRICING_READY_PATH),
    "classified_inventory_path": str(classified_inventory_path),
    "root_summary_path": str(root_summary_path),
    "spy_candidates_path": str(spy_candidates_path),
    "spy_date_coverage_path": str(spy_date_coverage_path),
    "spy_sample_summary_path": str(spy_sample_summary_path),
    "spy_sample_preview_path": str(spy_sample_preview_path),
    "pricing_ready_rows": int(len(inv)),
    "spy_candidate_files": int(len(spy_candidates)),
    "spx_spxw_candidate_files": int(len(spx_candidates)),
    "unknown_candidate_files": int(len(unknown_candidates)),
    "spy_unique_quote_dates_from_filename": int(spy_candidates["filename_first_date"].nunique()) if not spy_candidates.empty else 0,
    "spy_filename_date_min": str(spy_candidates["filename_first_date"].min().date()) if not spy_candidates.empty and pd.notna(spy_candidates["filename_first_date"].min()) else None,
    "spy_filename_date_max": str(spy_candidates["filename_first_date"].max().date()) if not spy_candidates.empty and pd.notna(spy_candidates["filename_first_date"].max()) else None,
    "recommendation": (
        "SPY option chains found locally; inspect sample summary and proceed to Cell 2C pricing cache."
        if not spy_candidates.empty
        else "No SPY option chains found locally; build a ThetaData SPY pull/cache step before Cell 2C."
    ),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Root summary")
print("=" * 100)
display(root_summary)

print()
print("=" * 100)
print("SPY availability summary")
print("=" * 100)

print(f"SPY candidate files:                 {len(spy_candidates):,}")
print(f"SPX/SPXW candidate files:            {len(spx_candidates):,}")
print(f"Unknown candidate files:             {len(unknown_candidates):,}")

if spy_candidates.empty:
    print()
    print("Result: No local SPY option-chain files found in the pricing-ready inventory.")
    print("Next step: build a ThetaData SPY option-chain pull/cache cell before pricing.")
else:
    print(f"SPY filename quote-date min:          {spy_candidates['filename_first_date'].min().date()}")
    print(f"SPY filename quote-date max:          {spy_candidates['filename_first_date'].max().date()}")
    print(f"SPY unique quote dates from filename: {spy_candidates['filename_first_date'].nunique():,}")
    print()
    print("Result: Local SPY option-chain candidates exist.")
    print("Next step: inspect sample summary, then build Cell 2C priced candidate cache.")

print()
print("Saved:")
print(f"  Classified inventory:              {classified_inventory_path}")
print(f"  Root summary:                      {root_summary_path}")
print(f"  SPY candidates:                    {spy_candidates_path}")
print(f"  SPY date coverage:                 {spy_date_coverage_path}")
print(f"  SPY sample summary:                {spy_sample_summary_path}")
print(f"  SPY sample preview:                {spy_sample_preview_path}")
print(f"  Manifest:                          {manifest_path}")

print()
print("=" * 100)
print("SPY date coverage preview")
print("=" * 100)

if spy_date_coverage.empty:
    print("No SPY date coverage to display.")
else:
    display(spy_date_coverage.head(20))
    print("...")
    display(spy_date_coverage.tail(20))

print()
print("=" * 100)
print("SPY candidate files preview")
print("=" * 100)

if spy_candidates.empty:
    print("No SPY candidates.")
else:
    display(
        pd.concat(
            [
                spy_candidates.head(20),
                spy_candidates.tail(20),
            ],
            ignore_index=True,
        )
        .drop_duplicates(subset=["relative_path"])
        [
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "filename_first_date",
                "filename_second_date",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
            ]
        ]
    )

print()
print("=" * 100)
print("SPY sample file summary")
print("=" * 100)

if spy_sample_summary.empty:
    print("No SPY sample files loaded.")
else:
    display(spy_sample_summary)

print()
print("=" * 100)
print("SPY sample preview rows")
print("=" * 100)

if spy_sample_preview.empty:
    print("No SPY sample preview rows.")
else:
    display(spy_sample_preview.head(30))

Cell 2B.1 — Identify SPY option-chain availability
Project root: C:\Users\patri\vrp_project
Audit dir:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research

Using pricing-ready inventory: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B_option_price_source_pricing_ready_candidates_20260710_094512.csv

Root summary


,root_from_filename,extension,file_count,existing_file_count,total_rows,min_filename_first_date,max_filename_first_date,min_filename_second_date,max_filename_second_date
0,SPX,.csv,2,2,0.0,2026-07-17,2026-07-17,2026-07-09,2026-07-09
1,SPX,.pkl,2530,2530,1976595.0,2018-06-25,2026-07-08,2018-07-20,2026-07-17
2,SPXW,.csv,16,16,0.0,2026-07-14,2026-07-23,2026-07-09,2026-07-09
3,SPXW,.pkl,8439,8439,3967310.0,2018-06-25,2026-07-08,2018-06-29,2026-08-14
4,SPY,.csv,1,1,0.0,2026-07-09,2026-07-09,NaT,NaT
5,UNKNOWN,.csv,24,24,0.0,2018-09-24,2026-07-09,2026-06-03,2026-07-01
6,UNKNOWN,.parquet,46,46,14761822.0,2018-09-24,2018-09-24,2026-07-01,2026-07-08



SPY availability summary
SPY candidate files:                 1
SPX/SPXW candidate files:            10,987
Unknown candidate files:             70
SPY filename quote-date min:          2026-07-09
SPY filename quote-date max:          2026-07-09
SPY unique quote dates from filename: 1

Result: Local SPY option-chain candidates exist.
Next step: inspect sample summary, then build Cell 2C priced candidate cache.

Saved:
  Classified inventory:              C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B1_option_chain_inventory_classified_20260710_094516.csv
  Root summary:                      C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B1_option_chain_root_summary_20260710_094516.csv
  SPY candidates:                    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B1_spy_option_chain_candidates_20260710_094516.csv
  SPY date coverage:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B1_spy_option_chain_date_cov

,filename_first_date,file_count,min_expiry_hint,max_expiry_hint,total_rows
0,2026-07-09,1,NaT,NaT,0.0


...


...


SPY candidate files preview


,relative_path,extension,rows,cols,size_mb,filename_first_date,filename_second_date,date_cols,expiration_cols,strike_cols,right_cols,bid_cols,ask_cols,mid_price_cols,underlying_cols
0,data\audit\call_sleeve_research\02B1_spy_optio...,.csv,NaN,24.0,0.000298,2026-07-09,NaT,"is_candidate, date_cols, filename_dates, filen...",expiration_cols,strike_cols,"right_cols, root_from_filename",bid_cols,ask_cols,mid_price_cols,"underlying_cols, root_from_filename"



SPY sample file summary


,relative_path,full_path,extension,file_exists,load_ok,rows,cols,columns,root_values,right_values,...,strike_max,bid_min,bid_max,ask_min,ask_max,mid_min,mid_max,bid_missing,ask_missing,error
0,data\audit\call_sleeve_research\02B1_spy_optio...,C:\Users\patri\vrp_project\data\audit\call_sle...,.csv,True,True,0,24,"relative_path, full_path, extension, size_mb, ...",None,None,...,None,None,None,None,None,None,None,None,None,None



SPY sample preview rows
No SPY sample preview rows.


In [4]:
# Cell 2B.2 — Inspect ThetaData pull code and test one SPY chain pull
# Purpose:
#   We confirmed no local SPY option chains exist. This cell:
#   1. Inspects existing ThetaData-related scripts/notebooks for endpoint patterns.
#   2. Inspects an existing SPX/SPXW chain file to confirm local saved schema.
#   3. Attempts one small SPY chain pull from local ThetaData.
#   4. Saves the test SPY chain only if the result has usable quote columns.
#
# Scope:
#   - Tiny probe only
#   - No historical bulk SPY download
#   - No parameter sweep
#   - No priced candidate cache yet
#
# Output folder:
#   data\raw\thetadata_chains_spy
#
# Expected saved naming:
#   SPY_YYYYMMDD_YYYYMMDD.pkl
#
# Notes:
#   - This is intentionally defensive because we need to infer the exact local ThetaData endpoint format.
#   - If the pull fails, paste the console output and we will adapt the endpoint.

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import requests
from io import StringIO
from datetime import datetime, timedelta

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
SPY_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy"
EXISTING_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPY_CHAIN_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B.2 — Inspect ThetaData pull code and test one SPY chain pull")
print("=" * 100)
print(f"Project root:            {PROJECT_ROOT}")
print(f"SPY chain output dir:    {SPY_CHAIN_DIR}")
print()

# --------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------

THETADATA_BASE_URLS = [
    "http://127.0.0.1:25510",
    "http://localhost:25510",
]

# Pick a recent date that should exist in your local final panel.
# This is only a one-chain probe. It does not need to be a selected trade.
RAW_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

TEST_ROOT = "SPY"
TEST_TENOR = 30

# If None, auto-pick latest complete candidate date from Cell 2A.
OVERRIDE_TEST_TRADE_DATE = None  # Example: "2026-07-01"

# Endpoint candidates. We try these in order.
# Some ThetaData installs support /v2/list/expirations and /v2/bulk_hist/option/quote.
EXPIRATION_ENDPOINT_CANDIDATES = [
    "/v2/list/expirations",
    "/v2/list/expirations/option",
]

CHAIN_ENDPOINT_CANDIDATES = [
    "/v2/bulk_hist/option/quote",
    "/v2/bulk_hist/option/eod",
]

# Quote interval guesses. 900000 ms = 15 minutes. Existing files may have one snapshot.
# If an endpoint ignores this, harmless.
IVL_CANDIDATES = [900000, 0]

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def yyyymmdd(dt) -> str:
    return pd.Timestamp(dt).strftime("%Y%m%d")

def normalize_expiration_value(x):
    """
    Convert likely expiration representations into YYYYMMDD string if possible.
    """
    if pd.isna(x):
        return None
    
    s = str(x).strip()
    
    # Already YYYYMMDD
    if re.fullmatch(r"20\d{6}", s):
        return s
    
    # Numeric like 20260731.0
    if re.fullmatch(r"20\d{6}\.0", s):
        return s.split(".")[0]
    
    # ISO date
    try:
        return pd.to_datetime(s).strftime("%Y%m%d")
    except Exception:
        return None

def response_to_dataframe(resp: requests.Response) -> pd.DataFrame:
    """
    Parse common ThetaData response formats:
    - JSON with header/response
    - JSON list/dict
    - CSV text
    """
    text = resp.text.strip()
    
    if not text:
        return pd.DataFrame()
    
    # Try JSON first
    try:
        obj = resp.json()
        
        if isinstance(obj, dict):
            # Common pattern: {"header": {"format": [...]}, "response": [[...], ...]}
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)
            
            # Otherwise flatten dict.
            return pd.json_normalize(obj)
        
        if isinstance(obj, list):
            return pd.DataFrame(obj)
    
    except Exception:
        pass
    
    # Try CSV
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass
    
    # Last resort: line split
    return pd.DataFrame({"raw_response": text.splitlines()})

def request_df(base_url, endpoint, params, timeout=30):
    url = base_url.rstrip("/") + endpoint
    resp = requests.get(url, params=params, timeout=timeout)
    status = resp.status_code
    df = pd.DataFrame()
    error = None
    
    try:
        resp.raise_for_status()
        df = response_to_dataframe(resp)
    except Exception as e:
        error = repr(e)
    
    return {
        "url": url,
        "params": params,
        "status_code": status,
        "ok": error is None,
        "rows": len(df) if isinstance(df, pd.DataFrame) else 0,
        "cols": len(df.columns) if isinstance(df, pd.DataFrame) else 0,
        "df": df,
        "error": error,
        "text_preview": resp.text[:500] if hasattr(resp, "text") else "",
    }

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    
    # Case-insensitive fallback
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    
    return None

def has_usable_chain_schema(df):
    if df is None or df.empty:
        return False
    
    strike_col = first_col(df, ["strike"])
    right_col = first_col(df, ["right", "option_type", "cp", "call_put", "put_call"])
    bid_col = first_col(df, ["bid", "bid_price"])
    ask_col = first_col(df, ["ask", "ask_price"])
    
    return all(c is not None for c in [strike_col, right_col, bid_col, ask_col])

def standardize_chain_schema(df, root, quote_date, expiration):
    out = df.copy()
    
    col_map = {}
    for canonical, candidates in {
        "timestamp": ["timestamp", "datetime", "date", "trade_date", "quote_date"],
        "root": ["root", "underlying", "symbol", "ticker"],
        "expiration": ["expiration", "expiry", "expiration_date", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type", "cp", "call_put", "put_call"],
        "bid": ["bid", "bid_price"],
        "ask": ["ask", "ask_price"],
        "mid": ["mid", "mark"],
    }.items():
        c = first_col(out, candidates)
        if c is not None:
            col_map[c] = canonical
    
    out = out.rename(columns=col_map)
    
    if "root" not in out.columns:
        out["root"] = root
    
    if "expiration" not in out.columns:
        out["expiration"] = expiration
    
    if "timestamp" not in out.columns:
        out["timestamp"] = quote_date
    
    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (pd.to_numeric(out["bid"], errors="coerce") + pd.to_numeric(out["ask"], errors="coerce")) / 2.0
    
    # Normalize key fields
    out["root"] = out["root"].astype(str).str.upper()
    out["expiration"] = out["expiration"].apply(normalize_expiration_value)
    out["strike"] = pd.to_numeric(out["strike"], errors="coerce")
    out["bid"] = pd.to_numeric(out["bid"], errors="coerce")
    out["ask"] = pd.to_numeric(out["ask"], errors="coerce")
    out["mid"] = pd.to_numeric(out["mid"], errors="coerce")
    
    if "right" in out.columns:
        out["right"] = out["right"].astype(str).str.upper().str[0]
    
    return out

def find_candidate_expirations_from_response(df, target_date):
    """
    Extract expiration candidates from an expiration-list response.
    """
    if df is None or df.empty:
        return []
    
    candidates = []
    
    for col in df.columns:
        vals = df[col].dropna().tolist()
        for v in vals:
            exp = normalize_expiration_value(v)
            if exp is not None:
                candidates.append(exp)
    
    candidates = sorted(set(candidates))
    
    # Keep expirations near the target date.
    target = pd.Timestamp(target_date)
    near = []
    for exp in candidates:
        dt = pd.to_datetime(exp, format="%Y%m%d", errors="coerce")
        if pd.notna(dt) and abs((dt - target).days) <= 14:
            near.append(exp)
    
    return near

def fallback_expirations_around_target(target_date, radius_days=10):
    """
    Generate likely SPY expirations around target date.
    SPY has frequent expiries; this fallback tries every business day around target.
    """
    target = pd.Timestamp(target_date)
    dates = pd.date_range(target - pd.Timedelta(days=radius_days), target + pd.Timedelta(days=radius_days), freq="B")
    return [yyyymmdd(d) for d in dates]

# --------------------------------------------------------------------------------------
# 1) Inspect existing ThetaData-related scripts/notebooks
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 1 — Existing ThetaData code snippets")
print("=" * 100)

text_paths = []
for root in [PROJECT_ROOT / "notebooks", PROJECT_ROOT / "scripts"]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".py", ".ipynb", ".txt", ".md"}:
                text_paths.append(p)

terms = [
    "bulk_hist",
    "hist/option",
    "list/expirations",
    "thetadata",
    "127.0.0.1",
    "25510",
    "bid",
    "ask",
]

snippet_rows = []

for path in text_paths:
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue
    
    low = text.lower()
    if not any(t.lower() in low for t in terms):
        continue
    
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if any(t.lower() in line.lower() for t in terms):
            start = max(0, i - 2)
            end = min(len(lines), i + 3)
            snippet = "\n".join(lines[start:end])
            snippet_rows.append({
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "line_number": i + 1,
                "matched_line": line.strip()[:300],
                "snippet": snippet[:1500],
            })

snippet_df = pd.DataFrame(snippet_rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
snippet_path = AUDIT_DIR / f"02B2_thetadata_code_snippets_{timestamp}.csv"
snippet_df.to_csv(snippet_path, index=False)

print(f"Snippet rows found: {len(snippet_df):,}")
print(f"Saved snippets:     {snippet_path}")

if not snippet_df.empty:
    display(snippet_df.head(30)[["relative_path", "line_number", "matched_line"]])
else:
    print("No ThetaData snippets found.")

print()

# --------------------------------------------------------------------------------------
# 2) Inspect existing local SPX/SPXW chain schema
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 2 — Existing SPX/SPXW chain schema sample")
print("=" * 100)

existing_chain_files = []
if EXISTING_CHAIN_DIR.exists():
    existing_chain_files = sorted(
        list(EXISTING_CHAIN_DIR.glob("SPX_*.pkl")) + list(EXISTING_CHAIN_DIR.glob("SPXW_*.pkl")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

existing_schema_summary = pd.DataFrame()
existing_preview = pd.DataFrame()

if existing_chain_files:
    sample_file = existing_chain_files[0]
    try:
        sample_df = pd.read_pickle(sample_file)
        
        existing_schema_summary = pd.DataFrame([{
            "sample_file": str(sample_file.relative_to(PROJECT_ROOT)),
            "rows": len(sample_df),
            "cols": len(sample_df.columns),
            "columns": ", ".join(map(str, sample_df.columns)),
            "root_values": ", ".join(map(str, sorted(sample_df["root"].dropna().unique().tolist()))) if "root" in sample_df.columns else None,
            "right_values": ", ".join(map(str, sorted(sample_df["right"].dropna().unique().tolist()))) if "right" in sample_df.columns else None,
            "expiration_values": ", ".join(map(str, sorted(sample_df["expiration"].dropna().astype(str).unique().tolist())[:10])) if "expiration" in sample_df.columns else None,
            "timestamp_values": ", ".join(map(str, sorted(sample_df["timestamp"].dropna().astype(str).unique().tolist())[:10])) if "timestamp" in sample_df.columns else None,
            "strike_min": pd.to_numeric(sample_df["strike"], errors="coerce").min() if "strike" in sample_df.columns else np.nan,
            "strike_max": pd.to_numeric(sample_df["strike"], errors="coerce").max() if "strike" in sample_df.columns else np.nan,
            "bid_missing": int(sample_df["bid"].isna().sum()) if "bid" in sample_df.columns else None,
            "ask_missing": int(sample_df["ask"].isna().sum()) if "ask" in sample_df.columns else None,
        }])
        
        existing_preview = sample_df.head(20).copy()
        
        display(existing_schema_summary)
        display(existing_preview)
    
    except Exception as e:
        print(f"Failed to inspect sample existing chain file {sample_file}: {repr(e)}")
else:
    print(f"No existing SPX/SPXW chain files found under {EXISTING_CHAIN_DIR}")

existing_schema_path = AUDIT_DIR / f"02B2_existing_chain_schema_summary_{timestamp}.csv"
existing_preview_path = AUDIT_DIR / f"02B2_existing_chain_preview_{timestamp}.csv"
existing_schema_summary.to_csv(existing_schema_path, index=False)
existing_preview.to_csv(existing_preview_path, index=False)

print(f"Saved existing schema summary: {existing_schema_path}")
print(f"Saved existing preview:        {existing_preview_path}")
print()

# --------------------------------------------------------------------------------------
# 3) Choose one test trade date / target expiration
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 3 — Choose one SPY chain test date")
print("=" * 100)

if OVERRIDE_TEST_TRADE_DATE is not None:
    test_trade_date = pd.Timestamp(OVERRIDE_TEST_TRADE_DATE)
else:
    raw_panel = pd.read_parquet(RAW_CANDIDATE_PANEL_PATH)
    raw_panel["trade_date"] = pd.to_datetime(raw_panel["trade_date"], errors="coerce")
    
    complete = raw_panel.loc[
        (raw_panel["tenor"] == TEST_TENOR)
        & (raw_panel["candidate_inputs_complete"])
    ].copy()
    
    if complete.empty:
        raise ValueError("No complete 30D candidates found to choose a test trade date.")
    
    test_trade_date = complete["trade_date"].max()

target_expiration_date = test_trade_date + pd.Timedelta(days=TEST_TENOR)

print(f"Test root:              {TEST_ROOT}")
print(f"Test trade date:        {test_trade_date.date()}")
print(f"Target expiration date: {target_expiration_date.date()}")
print()

# --------------------------------------------------------------------------------------
# 4) Check ThetaData local service availability
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 4 — Probe local ThetaData service")
print("=" * 100)

service_probe_rows = []
working_base_url = None

for base in THETADATA_BASE_URLS:
    for endpoint in ["/v2/system/status", "/v2/list/roots/option", "/"]:
        try:
            url = base.rstrip("/") + endpoint
            resp = requests.get(url, timeout=5)
            service_probe_rows.append({
                "base_url": base,
                "endpoint": endpoint,
                "url": url,
                "status_code": resp.status_code,
                "ok": resp.ok,
                "text_preview": resp.text[:300],
                "error": None,
            })
            
            if resp.status_code < 500 and working_base_url is None:
                working_base_url = base
        
        except Exception as e:
            service_probe_rows.append({
                "base_url": base,
                "endpoint": endpoint,
                "url": base.rstrip("/") + endpoint,
                "status_code": None,
                "ok": False,
                "text_preview": "",
                "error": repr(e),
            })

service_probe_df = pd.DataFrame(service_probe_rows)
service_probe_path = AUDIT_DIR / f"02B2_thetadata_service_probe_{timestamp}.csv"
service_probe_df.to_csv(service_probe_path, index=False)

display(service_probe_df)

if working_base_url is None:
    print()
    print("No local ThetaData service endpoint responded.")
    print("Start ThetaData Terminal / local API, then rerun this cell.")
else:
    print(f"Working base URL candidate: {working_base_url}")

print()

# --------------------------------------------------------------------------------------
# 5) Pull expiration list if possible
# --------------------------------------------------------------------------------------

expiration_probe_rows = []
expiration_candidates = []

if working_base_url is not None:
    print("=" * 100)
    print("Step 5 — Try expiration-list endpoints")
    print("=" * 100)
    
    for endpoint in EXPIRATION_ENDPOINT_CANDIDATES:
        params = {"root": TEST_ROOT}
        result = request_df(working_base_url, endpoint, params, timeout=20)
        df_exp = result["df"]
        
        near_exp = find_candidate_expirations_from_response(df_exp, target_expiration_date)
        
        expiration_probe_rows.append({
            "endpoint": endpoint,
            "params": json.dumps(params),
            "status_code": result["status_code"],
            "ok": result["ok"],
            "rows": result["rows"],
            "cols": result["cols"],
            "near_expiration_count": len(near_exp),
            "near_expirations": ", ".join(near_exp[:30]),
            "columns": ", ".join(map(str, df_exp.columns)) if isinstance(df_exp, pd.DataFrame) else "",
            "error": result["error"],
            "text_preview": result["text_preview"],
        })
        
        expiration_candidates.extend(near_exp)
    
    expiration_candidates = sorted(set(expiration_candidates))
    
    if not expiration_candidates:
        expiration_candidates = fallback_expirations_around_target(target_expiration_date, radius_days=10)
        expiration_source = "fallback_business_days_around_target"
    else:
        expiration_source = "thetadata_expiration_list"
    
    expiration_probe_df = pd.DataFrame(expiration_probe_rows)
    display(expiration_probe_df)
    
    print()
    print(f"Expiration source:       {expiration_source}")
    print(f"Expiration candidates:   {expiration_candidates[:30]}")
else:
    expiration_probe_df = pd.DataFrame()
    expiration_source = "no_service"
    expiration_candidates = []

expiration_probe_path = AUDIT_DIR / f"02B2_spy_expiration_probe_{timestamp}.csv"
expiration_probe_df.to_csv(expiration_probe_path, index=False)
print()

# --------------------------------------------------------------------------------------
# 6) Try one SPY chain pull
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 6 — Try one SPY chain pull")
print("=" * 100)

chain_attempt_rows = []
saved_chain_path = None
saved_chain_df = pd.DataFrame()

if working_base_url is not None and expiration_candidates:
    test_trade_yyyymmdd = yyyymmdd(test_trade_date)
    
    # Closest expirations first
    expiration_candidates_sorted = sorted(
        expiration_candidates,
        key=lambda exp: abs((pd.to_datetime(exp, format="%Y%m%d") - target_expiration_date).days)
    )
    
    for exp in expiration_candidates_sorted[:20]:
        if saved_chain_path is not None:
            break
        
        for endpoint in CHAIN_ENDPOINT_CANDIDATES:
            if saved_chain_path is not None:
                break
            
            for ivl in IVL_CANDIDATES:
                params = {
                    "root": TEST_ROOT,
                    "exp": exp,
                    "start_date": test_trade_yyyymmdd,
                    "end_date": test_trade_yyyymmdd,
                }
                
                # Try with ivl; if endpoint ignores it, fine.
                if ivl is not None:
                    params["ivl"] = ivl
                
                result = request_df(working_base_url, endpoint, params, timeout=60)
                df_chain = result["df"]
                
                usable = has_usable_chain_schema(df_chain)
                
                chain_attempt_rows.append({
                    "endpoint": endpoint,
                    "exp": exp,
                    "ivl": ivl,
                    "params": json.dumps(params),
                    "status_code": result["status_code"],
                    "ok": result["ok"],
                    "rows": result["rows"],
                    "cols": result["cols"],
                    "usable_schema": usable,
                    "columns": ", ".join(map(str, df_chain.columns)) if isinstance(df_chain, pd.DataFrame) else "",
                    "error": result["error"],
                    "text_preview": result["text_preview"],
                })
                
                if usable and not df_chain.empty:
                    standardized = standardize_chain_schema(
                        df_chain,
                        root=TEST_ROOT,
                        quote_date=test_trade_yyyymmdd,
                        expiration=exp,
                    )
                    
                    # Keep rows with valid strike/right/bid/ask.
                    valid = standardized[
                        standardized["strike"].notna()
                        & standardized["right"].notna()
                        & standardized["bid"].notna()
                        & standardized["ask"].notna()
                    ].copy()
                    
                    if not valid.empty:
                        # Save exactly what we got, standardized.
                        out_name = f"{TEST_ROOT}_{test_trade_yyyymmdd}_{exp}.pkl"
                        saved_chain_path = SPY_CHAIN_DIR / out_name
                        valid.to_pickle(saved_chain_path)
                        saved_chain_df = valid
                        break

chain_attempt_df = pd.DataFrame(chain_attempt_rows)
chain_attempt_path = AUDIT_DIR / f"02B2_spy_chain_pull_attempts_{timestamp}.csv"
chain_attempt_df.to_csv(chain_attempt_path, index=False)

if chain_attempt_df.empty:
    print("No chain pull attempts were made.")
else:
    display(chain_attempt_df[[
        "endpoint",
        "exp",
        "ivl",
        "status_code",
        "ok",
        "rows",
        "cols",
        "usable_schema",
        "columns",
        "error",
    ]].head(100))

print()

if saved_chain_path is None:
    print("No usable SPY chain was saved.")
    print("Paste the chain attempt table and text previews if needed; we will adapt the endpoint.")
else:
    print(f"Saved usable SPY test chain: {saved_chain_path}")
    print(f"Saved rows:                  {len(saved_chain_df):,}")
    print(f"Columns:                     {list(saved_chain_df.columns)}")
    print()
    print("Saved chain summary:")
    
    saved_summary = pd.DataFrame([{
        "path": str(saved_chain_path),
        "rows": len(saved_chain_df),
        "root_values": ", ".join(map(str, sorted(saved_chain_df["root"].dropna().unique().tolist()))),
        "right_values": ", ".join(map(str, sorted(saved_chain_df["right"].dropna().unique().tolist()))),
        "expiration_values": ", ".join(map(str, sorted(saved_chain_df["expiration"].dropna().unique().tolist()))),
        "timestamp_values_preview": ", ".join(map(str, sorted(saved_chain_df["timestamp"].dropna().astype(str).unique().tolist())[:10])),
        "strike_min": saved_chain_df["strike"].min(),
        "strike_max": saved_chain_df["strike"].max(),
        "bid_min": saved_chain_df["bid"].min(),
        "bid_max": saved_chain_df["bid"].max(),
        "ask_min": saved_chain_df["ask"].min(),
        "ask_max": saved_chain_df["ask"].max(),
        "mid_min": saved_chain_df["mid"].min(),
        "mid_max": saved_chain_df["mid"].max(),
    }])
    
    display(saved_summary)
    print()
    print("Saved chain preview:")
    display(saved_chain_df.head(30))

# --------------------------------------------------------------------------------------
# 7) Save manifest
# --------------------------------------------------------------------------------------

manifest_path = AUDIT_DIR / f"02B2_spy_chain_probe_manifest_{timestamp}.json"

manifest = {
    "cell": "2B.2",
    "description": "Inspect existing ThetaData code/schema and test one SPY option-chain pull",
    "created_at": timestamp,
    "snippet_path": str(snippet_path),
    "existing_schema_path": str(existing_schema_path),
    "existing_preview_path": str(existing_preview_path),
    "service_probe_path": str(service_probe_path),
    "expiration_probe_path": str(expiration_probe_path),
    "chain_attempt_path": str(chain_attempt_path),
    "spy_chain_dir": str(SPY_CHAIN_DIR),
    "test_root": TEST_ROOT,
    "test_tenor": TEST_TENOR,
    "test_trade_date": str(test_trade_date.date()),
    "target_expiration_date": str(target_expiration_date.date()),
    "working_base_url": working_base_url,
    "expiration_source": expiration_source,
    "expiration_candidates": expiration_candidates[:50],
    "saved_chain_path": str(saved_chain_path) if saved_chain_path is not None else None,
    "saved_chain_rows": int(len(saved_chain_df)) if saved_chain_path is not None else 0,
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print()
print("=" * 100)
print("Cell 2B.2 complete")
print("=" * 100)
print(f"Manifest: {manifest_path}")

if saved_chain_path is not None:
    print("Result: SPY test chain pull worked. Next cell can build the SPY pull/cache plan.")
else:
    print("Result: SPY test chain pull did not produce a usable saved chain. Review endpoint attempts.")

Cell 2B.2 — Inspect ThetaData pull code and test one SPY chain pull
Project root:            C:\Users\patri\vrp_project
SPY chain output dir:    C:\Users\patri\vrp_project\data\raw\thetadata_chains_spy

Step 1 — Existing ThetaData code snippets
Snippet rows found: 1,669
Saved snippets:     C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2_thetadata_code_snippets_20260710_094521.csv


,relative_path,line_number,matched_line
0,notebooks\02_call_sleeve_corsi_parameter_sweep...,724,"""# - No ThetaData option pricing yet\n"","
1,notebooks\02_call_sleeve_corsi_parameter_sweep...,1095,"""Cell 2B â€” ThetaData option price source inv..."
2,notebooks\02_call_sleeve_corsi_parameter_sweep...,1107,"""# Cell 2B â€” ThetaData option price source i..."
3,notebooks\02_call_sleeve_corsi_parameter_sweep...,1109,"""# Find existing local ThetaData / option-ch..."
4,notebooks\02_call_sleeve_corsi_parameter_sweep...,1110,"""# to attach actual SPY option bid/ask entry..."
5,notebooks\02_call_sleeve_corsi_parameter_sweep...,1114,"""# - No ThetaData API pulls yet\n"","
6,notebooks\02_call_sleeve_corsi_parameter_sweep...,1120,"""# - ThetaData pull scripts\n"","
7,notebooks\02_call_sleeve_corsi_parameter_sweep...,1122,"""# - Columns for date, expiration, strike, r..."
8,notebooks\02_call_sleeve_corsi_parameter_sweep...,1141,"""print(\""Cell 2B â€” ThetaData option price so..."
9,notebooks\02_call_sleeve_corsi_parameter_sweep...,1162,"""# Filename/path terms that suggest options / ..."



Step 2 — Existing SPX/SPXW chain schema sample


,sample_file,rows,cols,columns,root_values,right_values,expiration_values,timestamp_values,strike_min,strike_max,bid_missing,ask_missing
0,data\raw\thetadata_chains\SPX_20260708_2026071...,1186,14,"root, expiration, strike, right, bid, ask, mid...",SPX,"C, P",20260717,2026-07-08T16:00:00.000,200.0,12400.0,0,0


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
0,SPX,20260717,5925.0,P,0.25,0.45,0.350,54,153,5,5,50,50,2026-07-08T16:00:00.000
1,SPX,20260717,5925.0,C,1549.10,1569.60,1559.350,1,1,5,5,50,50,2026-07-08T16:00:00.000
2,SPX,20260717,6650.0,C,826.50,847.00,836.750,1,1,5,5,50,50,2026-07-08T16:00:00.000
3,SPX,20260717,6650.0,P,0.85,1.00,0.925,35,103,5,5,50,50,2026-07-08T16:00:00.000
4,SPX,20260717,7375.0,P,25.20,25.80,25.500,45,30,5,5,50,50,2026-07-08T16:00:00.000
5,SPX,20260717,7375.0,C,133.50,136.90,135.200,2,2,5,5,50,50,2026-07-08T16:00:00.000
6,SPX,20260717,5200.0,C,2273.90,2294.40,2284.150,1,1,5,5,50,50,2026-07-08T16:00:00.000
7,SPX,20260717,5200.0,P,0.05,0.20,0.125,180,168,5,5,50,50,2026-07-08T16:00:00.000
8,SPX,20260717,6480.0,C,998.10,1016.60,1007.350,2,1,5,5,50,50,2026-07-08T16:00:00.000
9,SPX,20260717,6480.0,P,0.65,0.80,0.725,34,117,5,5,50,50,2026-07-08T16:00:00.000


Saved existing schema summary: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2_existing_chain_schema_summary_20260710_094521.csv
Saved existing preview:        C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2_existing_chain_preview_20260710_094521.csv

Step 3 — Choose one SPY chain test date
Test root:              SPY
Test trade date:        2026-07-01
Target expiration date: 2026-07-31

Step 4 — Probe local ThetaData service


,base_url,endpoint,url,status_code,ok,text_preview,error
0,http://127.0.0.1:25510,/v2/system/status,http://127.0.0.1:25510/v2/system/status,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."
1,http://127.0.0.1:25510,/v2/list/roots/option,http://127.0.0.1:25510/v2/list/roots/option,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."
2,http://127.0.0.1:25510,/,http://127.0.0.1:25510/,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."
3,http://localhost:25510,/v2/system/status,http://localhost:25510/v2/system/status,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."
4,http://localhost:25510,/v2/list/roots/option,http://localhost:25510/v2/list/roots/option,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."
5,http://localhost:25510,/,http://localhost:25510/,None,False,,"ConnectionError(MaxRetryError(""HTTPConnectionP..."



No local ThetaData service endpoint responded.
Start ThetaData Terminal / local API, then rerun this cell.


Step 6 — Try one SPY chain pull
No chain pull attempts were made.

No usable SPY chain was saved.
Paste the chain attempt table and text previews if needed; we will adapt the endpoint.

Cell 2B.2 complete
Manifest: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2_spy_chain_probe_manifest_20260710_094521.json
Result: SPY test chain pull did not produce a usable saved chain. Review endpoint attempts.


In [8]:
# Cell 2B.2c — ThetaData v3 option quote endpoint probe
# Purpose:
#   Theta Terminal v3 is reachable on port 25503, but v2 endpoints return 410.
#   This cell probes likely v3 option quote/history endpoint shapes.
#
# Scope:
#   - Probe only
#   - No bulk download
#   - No parameter sweep
#   - No production changes

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
SPY_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SPY_CHAIN_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "http://127.0.0.1:25503"

# Test contract near the 2026-07-01 30D call-spread candidate.
# If 780 is not listed or has no response, the endpoint can still reveal the correct route/parameter error.
TEST_SYMBOL = "SPY"
TEST_RIGHT = "C"
TEST_STRIKE = 780.0
TEST_EXPIRATION_ISO = "2026-07-31"
TEST_EXPIRATION_YYYYMMDD = "20260731"
TEST_DATE_ISO = "2026-07-01"
TEST_DATE_YYYYMMDD = "20260701"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2B.2c — ThetaData v3 option quote endpoint probe")
print("=" * 100)
print(f"Base URL:     {BASE_URL}")
print(f"Symbol:       {TEST_SYMBOL}")
print(f"Right:        {TEST_RIGHT}")
print(f"Strike:       {TEST_STRIKE}")
print(f"Expiration:   {TEST_EXPIRATION_ISO}")
print(f"Date:         {TEST_DATE_ISO}")
print()

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    # JSON
    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)

            # Some v3 endpoints may return {"data": ...}
            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    # CSV / NDJSON-ish fallback
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def get_df(endpoint, params, timeout=(2, 20)):
    url = BASE_URL.rstrip("/") + endpoint

    try:
        r = session.get(url, params=params, timeout=timeout)
        df = response_to_dataframe(r)

        return {
            "endpoint": endpoint,
            "url": r.url,
            "params": json.dumps(params),
            "status_code": r.status_code,
            "ok": r.ok,
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(map(str, df.columns)),
            "text_preview": r.text[:1000],
            "error": None,
            "df": df,
        }

    except Exception as e:
        return {
            "endpoint": endpoint,
            "url": url,
            "params": json.dumps(params),
            "status_code": None,
            "ok": False,
            "rows": 0,
            "cols": 0,
            "columns": "",
            "text_preview": "",
            "error": repr(e),
            "df": pd.DataFrame(),
        }

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c

    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]

    return None

def has_quote_schema(df):
    if df is None or df.empty:
        return False

    bid_col = first_col(df, ["bid", "bid_price"])
    ask_col = first_col(df, ["ask", "ask_price"])
    mid_col = first_col(df, ["mid", "mark"])
    close_col = first_col(df, ["close", "price"])

    return (bid_col is not None and ask_col is not None) or mid_col is not None or close_col is not None

# Endpoint/parameter candidates based on v3 route style.
# We test both ISO and YYYYMMDD dates because v3 examples use ISO, while our old cache used YYYYMMDD.
attempts = []

endpoint_param_sets = []

single_contract_endpoints = [
    "/v3/option/history/quote/eod",
    "/v3/option/history/quote",
    "/v3/option/history/trade_quote",
    "/v3/option/history/greeks/eod",   # control endpoint from Theta's example shape
    "/v3/option/quote/history/eod",
    "/v3/option/quote/eod",
]

for endpoint in single_contract_endpoints:
    # ISO-style params
    endpoint_param_sets.append((
        endpoint,
        {
            "symbol": TEST_SYMBOL,
            "right": TEST_RIGHT,
            "strike": TEST_STRIKE,
            "expiration": TEST_EXPIRATION_ISO,
            "start_date": TEST_DATE_ISO,
            "end_date": TEST_DATE_ISO,
            "format": "csv",
        }
    ))

    # YYYYMMDD-style params
    endpoint_param_sets.append((
        endpoint,
        {
            "symbol": TEST_SYMBOL,
            "right": TEST_RIGHT,
            "strike": TEST_STRIKE,
            "expiration": TEST_EXPIRATION_YYYYMMDD,
            "start_date": TEST_DATE_YYYYMMDD,
            "end_date": TEST_DATE_YYYYMMDD,
            "format": "csv",
        }
    ))

    # Old-style exp key, just in case
    endpoint_param_sets.append((
        endpoint,
        {
            "root": TEST_SYMBOL,
            "right": TEST_RIGHT,
            "strike": TEST_STRIKE,
            "exp": TEST_EXPIRATION_YYYYMMDD,
            "start_date": TEST_DATE_YYYYMMDD,
            "end_date": TEST_DATE_YYYYMMDD,
            "format": "csv",
        }
    ))

chain_or_snapshot_endpoints = [
    "/v3/option/snapshot/quote",
    "/v3/option/snapshot/chain",
    "/v3/option/history/quote/snapshot",
    "/v3/option/history/eod",
    "/v3/option/history/quote/eod/chain",
]

for endpoint in chain_or_snapshot_endpoints:
    endpoint_param_sets.append((
        endpoint,
        {
            "symbol": TEST_SYMBOL,
            "expiration": TEST_EXPIRATION_ISO,
            "date": TEST_DATE_ISO,
            "format": "csv",
        }
    ))

    endpoint_param_sets.append((
        endpoint,
        {
            "root": TEST_SYMBOL,
            "exp": TEST_EXPIRATION_YYYYMMDD,
            "date": TEST_DATE_YYYYMMDD,
            "format": "csv",
        }
    ))

# Run attempts
success_dfs = []

for endpoint, params in endpoint_param_sets:
    r = get_df(endpoint, params)
    df = r["df"]

    schema_ok = has_quote_schema(df)

    row = {k: v for k, v in r.items() if k != "df"}
    row["has_quote_schema"] = schema_ok
    attempts.append(row)

    if r["ok"] and r["rows"] > 0:
        preview = df.head(10).copy()
        preview.insert(0, "source_endpoint", endpoint)
        preview.insert(1, "source_params", json.dumps(params))
        success_dfs.append(preview)

attempts_df = pd.DataFrame(attempts)

attempts_path = AUDIT_DIR / f"02B2c_thetadata_v3_option_quote_endpoint_attempts_{timestamp}.csv"
attempts_df.to_csv(attempts_path, index=False)

if success_dfs:
    success_preview = pd.concat(success_dfs, ignore_index=True)
else:
    success_preview = pd.DataFrame()

success_preview_path = AUDIT_DIR / f"02B2c_thetadata_v3_option_quote_success_preview_{timestamp}.csv"
success_preview.to_csv(success_preview_path, index=False)

print("=" * 100)
print("Endpoint attempts")
print("=" * 100)
display(
    attempts_df[
        [
            "endpoint",
            "status_code",
            "ok",
            "rows",
            "cols",
            "columns",
            "has_quote_schema",
            "error",
            "text_preview",
        ]
    ]
)

print()
print("Saved:")
print(f"  Attempts:        {attempts_path}")
print(f"  Success preview: {success_preview_path}")

print()
print("=" * 100)
print("Successful / non-empty previews")
print("=" * 100)

if success_preview.empty:
    print("No non-empty endpoint response found.")
else:
    display(success_preview.head(50))

Cell 2B.2c — ThetaData v3 option quote endpoint probe
Base URL:     http://127.0.0.1:25503
Symbol:       SPY
Right:        C
Strike:       780.0
Expiration:   2026-07-31
Date:         2026-07-01

Endpoint attempts


,endpoint,status_code,ok,rows,cols,columns,has_quote_schema,error,text_preview
0,/v3/option/history/quote/eod,404,False,14,1,<html>,False,None,"<html>\n<head>\n<meta http-equiv=""Content-Type..."
1,/v3/option/history/quote/eod,404,False,14,1,<html>,False,None,"<html>\n<head>\n<meta http-equiv=""Content-Type..."
2,/v3/option/history/quote/eod,410,False,4,1,We have upgraded to API v3. Please use API v3 ...,False,None,We have upgraded to API v3. Please use API v3 ...
3,/v3/option/history/quote,200,True,23401,13,"symbol, expiration, strike, right, timestamp, ...",True,None,"symbol,expiration,strike,right,timestamp,bid_s..."
4,/v3/option/history/quote,200,True,23401,13,"symbol, expiration, strike, right, timestamp, ...",True,None,"symbol,expiration,strike,right,timestamp,bid_s..."
5,/v3/option/history/quote,410,False,4,1,We have upgraded to API v3. Please use API v3 ...,False,None,We have upgraded to API v3. Please use API v3 ...
6,/v3/option/history/trade_quote,200,True,127,23,"symbol, expiration, strike, right, trade_times...",True,None,"symbol,expiration,strike,right,trade_timestamp..."
7,/v3/option/history/trade_quote,200,True,127,23,"symbol, expiration, strike, right, trade_times...",True,None,"symbol,expiration,strike,right,trade_timestamp..."
8,/v3/option/history/trade_quote,410,False,4,1,We have upgraded to API v3. Please use API v3 ...,False,None,We have upgraded to API v3. Please use API v3 ...
9,/v3/option/history/greeks/eod,200,True,1,43,"symbol, expiration, strike, right, timestamp, ...",True,None,"symbol,expiration,strike,right,timestamp,open,..."



Saved:
  Attempts:        C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2c_thetadata_v3_option_quote_endpoint_attempts_20260710_095422.csv
  Success preview: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B2c_thetadata_v3_option_quote_success_preview_20260710_095422.csv

Successful / non-empty previews


,source_endpoint,source_params,symbol,expiration,strike,right,timestamp,bid_size,bid_exchange,bid,...,color,ultima,d1,d2,dual_delta,dual_gamma,implied_vol,iv_error,underlying_timestamp,underlying_price
0,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:00.000,0,0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:01.000,108,73,1.23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:02.000,108,73,1.24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:03.000,28,42,1.24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:04.000,216,6,1.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:05.000,216,6,1.23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:06.000,226,6,1.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:07.000,339,6,1.21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:08.000,108,73,1.22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,/v3/option/history/quote,"{""symbol"": ""SPY"", ""right"": ""C"", ""strike"": 780....",SPY,2026-07-31,780.0,CALL,2026-07-01T09:30:09.000,364,42,1.19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Cell 2B.3 — Test SPY EOD greeks pricing for raw call-spread legs
# Purpose:
#   Test whether we can price actual SPY call-spread entry legs from ThetaData v3
#   using /v3/option/history/greeks/eod.
#
# Scope:
#   - Tiny sample only
#   - No historical bulk cache yet
#   - No parameter sweep yet
#   - No selected-trade logic yet
#
# Pricing assumption:
#   - Short call sold at bid
#   - Long call bought at ask
#   - Conservative entry credit = short_bid - long_ask
#
# Endpoint:
#   /v3/option/history/greeks/eod
#
# Contract selection:
#   - expiration target = trade_date + tenor calendar days
#   - test nearby expirations on/after target date
#   - short strike = ceil(raw 1SD strike to nearest $1, then try nearby higher strikes
#   - long strike  = ceil(raw 3SD strike to nearest $1, then try nearby higher strikes

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json
import math

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
SPY_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPY_CHAIN_DIR.mkdir(parents=True, exist_ok=True)

RAW_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

BASE_URL = "http://127.0.0.1:25503"
ENDPOINT = "/v3/option/history/greeks/eod"

SYMBOL = "SPY"
RIGHT = "C"

# Keep this small. We only need to prove the leg-pricing workflow works.
TEST_TENORS = [9, 30, 33]
MAX_EXPIRATION_DAYS_FORWARD = 10
MAX_STRIKE_INCREMENT = 10

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2B.3 — Test SPY EOD greeks pricing for raw call-spread legs")
print("=" * 100)
print(f"Raw candidate panel: {RAW_CANDIDATE_PANEL_PATH}")
print(f"ThetaData base URL:  {BASE_URL}")
print(f"Endpoint:            {ENDPOINT}")
print()

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def get_df(endpoint, params, timeout=(2, 30)):
    url = BASE_URL.rstrip("/") + endpoint

    try:
        r = session.get(url, params=params, timeout=timeout)
        df = response_to_dataframe(r)

        return {
            "endpoint": endpoint,
            "url": r.url,
            "params": params,
            "status_code": r.status_code,
            "ok": r.ok,
            "rows": len(df),
            "cols": len(df.columns),
            "columns": list(df.columns),
            "text_preview": r.text[:500],
            "error": None,
            "df": df,
        }

    except Exception as e:
        return {
            "endpoint": endpoint,
            "url": url,
            "params": params,
            "status_code": None,
            "ok": False,
            "rows": 0,
            "cols": 0,
            "columns": [],
            "text_preview": "",
            "error": repr(e),
            "df": pd.DataFrame(),
        }

def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None

def standardize_price_row(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    # Normalize expected columns from v3.
    rename_map = {}
    lower_to_col = {str(c).lower(): c for c in out.columns}

    for canonical, candidates in {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "underlying_price": ["underlying_price"],
        "implied_vol": ["implied_vol", "iv"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
    }.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    for col in ["strike", "bid", "ask", "mid", "underlying_price", "implied_vol", "delta", "theta", "vega", "gamma"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    return out

def expiration_candidates(trade_date, tenor):
    target = pd.Timestamp(trade_date) + pd.Timedelta(days=int(tenor))

    # Prefer expirations on/after target, then allow small fallback before target.
    forward = pd.date_range(
        target,
        target + pd.Timedelta(days=MAX_EXPIRATION_DAYS_FORWARD),
        freq="D",
    )
    fallback_before = pd.date_range(
        target - pd.Timedelta(days=3),
        target - pd.Timedelta(days=1),
        freq="D",
    )

    candidates = list(forward) + list(fallback_before)
    candidates = sorted(set(candidates), key=lambda d: (d < target, abs((d - target).days), d))

    return [d.strftime("%Y-%m-%d") for d in candidates]

def strike_candidates(raw_strike, minimum_strike=None):
    base = int(math.ceil(float(raw_strike)))

    candidates = [base + i for i in range(0, MAX_STRIKE_INCREMENT + 1)]

    # Add nearest $5 strike at/above raw as fallback.
    base5 = int(math.ceil(float(raw_strike) / 5.0) * 5)
    candidates += [base5 + 5 * i for i in range(0, 4)]

    # Keep unique sorted, enforce minimum if supplied.
    candidates = sorted(set(candidates))

    if minimum_strike is not None:
        candidates = [x for x in candidates if x > minimum_strike]

    return candidates

def fetch_contract_eod(trade_date, expiration_iso, strike, right="C"):
    params = {
        "symbol": SYMBOL,
        "right": right,
        "strike": float(strike),
        "expiration": expiration_iso,
        "start_date": pd.Timestamp(trade_date).strftime("%Y-%m-%d"),
        "end_date": pd.Timestamp(trade_date).strftime("%Y-%m-%d"),
        "format": "csv",
    }

    result = get_df(ENDPOINT, params)
    raw = result["df"]
    std = standardize_price_row(raw)

    attempt = {
        "trade_date": pd.Timestamp(trade_date).date().isoformat(),
        "expiration": expiration_iso,
        "strike": float(strike),
        "right": right,
        "status_code": result["status_code"],
        "ok": result["ok"],
        "rows": result["rows"],
        "cols": result["cols"],
        "columns": ", ".join(map(str, result["columns"])),
        "error": result["error"],
        "text_preview": result["text_preview"],
        "has_bid_ask": False,
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "timestamp": None,
        "underlying_price": np.nan,
        "implied_vol": np.nan,
        "delta": np.nan,
    }

    if not std.empty and {"bid", "ask"}.issubset(std.columns):
        # Pick first row with valid bid/ask.
        valid = std.loc[
            std["bid"].notna()
            & std["ask"].notna()
        ].copy()

        if not valid.empty:
            row = valid.iloc[-1]
            attempt["has_bid_ask"] = True
            attempt["bid"] = row.get("bid", np.nan)
            attempt["ask"] = row.get("ask", np.nan)
            attempt["mid"] = row.get("mid", np.nan)
            attempt["timestamp"] = row.get("timestamp", None)
            attempt["underlying_price"] = row.get("underlying_price", np.nan)
            attempt["implied_vol"] = row.get("implied_vol", np.nan)
            attempt["delta"] = row.get("delta", np.nan)

    return attempt

def price_leg_with_fallback(trade_date, tenor, raw_strike, leg_label, minimum_strike=None):
    exp_list = expiration_candidates(trade_date, tenor)
    attempt_rows = []

    for exp_iso in exp_list:
        for strike in strike_candidates(raw_strike, minimum_strike=minimum_strike):
            attempt = fetch_contract_eod(
                trade_date=trade_date,
                expiration_iso=exp_iso,
                strike=strike,
                right=RIGHT,
            )
            attempt["leg_label"] = leg_label
            attempt["raw_strike"] = raw_strike
            attempt["target_tenor"] = tenor
            attempt_rows.append(attempt)

            if attempt["has_bid_ask"]:
                return attempt, attempt_rows

    return None, attempt_rows

# --------------------------------------------------------------------------------------
# Load sample candidates
# --------------------------------------------------------------------------------------

raw_panel = pd.read_parquet(RAW_CANDIDATE_PANEL_PATH)
raw_panel["trade_date"] = pd.to_datetime(raw_panel["trade_date"], errors="coerce")

complete = raw_panel.loc[
    raw_panel["candidate_inputs_complete"]
    & raw_panel["tenor"].isin(TEST_TENORS)
].copy()

if complete.empty:
    raise ValueError("No complete test candidates found.")

# Use latest complete trade date available across requested test tenors.
latest_complete_date = complete["trade_date"].max()

sample = (
    complete.loc[complete["trade_date"] == latest_complete_date]
    .sort_values("tenor")
    .copy()
)

# If all requested tenors are not present on latest date, take latest per tenor.
if sample["tenor"].nunique() < len(TEST_TENORS):
    sample = (
        complete
        .sort_values(["tenor", "trade_date"])
        .groupby("tenor", as_index=False)
        .tail(1)
        .sort_values(["trade_date", "tenor"])
        .copy()
    )

sample = sample.loc[sample["tenor"].isin(TEST_TENORS)].copy()

print("Sample candidates:")
display(
    sample[
        [
            "trade_date",
            "tenor",
            "spy_close",
            "vix_style_vol_pct",
            "short_call_1sd_raw",
            "long_call_3sd_raw",
            "call_spread_width_raw",
            "corsi_vrp_log",
            "corsi_vrp_z_3m",
            "corsi_vrp_z_1y",
            "rsi14",
        ]
    ]
)

# --------------------------------------------------------------------------------------
# Price sample spreads
# --------------------------------------------------------------------------------------

priced_rows = []
all_attempts = []

for _, row in sample.iterrows():
    trade_date = row["trade_date"]
    tenor = int(row["tenor"])

    short_raw = float(row["short_call_1sd_raw"])
    long_raw = float(row["long_call_3sd_raw"])

    print()
    print("-" * 100)
    print(f"Pricing sample row: trade_date={trade_date.date()}, tenor={tenor}")
    print(f"  short raw strike: {short_raw:.4f}")
    print(f"  long raw strike:  {long_raw:.4f}")

    short_leg, short_attempts = price_leg_with_fallback(
        trade_date=trade_date,
        tenor=tenor,
        raw_strike=short_raw,
        leg_label="short_call_1sd",
        minimum_strike=None,
    )
    all_attempts.extend(short_attempts)

    if short_leg is None:
        print("  Short leg: no usable bid/ask found")
        continue

    long_leg, long_attempts = price_leg_with_fallback(
        trade_date=trade_date,
        tenor=tenor,
        raw_strike=long_raw,
        leg_label="long_call_3sd",
        minimum_strike=short_leg["strike"],
    )
    all_attempts.extend(long_attempts)

    if long_leg is None:
        print("  Long leg: no usable bid/ask found")
        continue

    # Prefer same expiration for both legs. If fallback picked different expirations, flag it.
    same_expiration = short_leg["expiration"] == long_leg["expiration"]

    entry_credit_bid_ask = short_leg["bid"] - long_leg["ask"]
    entry_credit_mid = short_leg["mid"] - long_leg["mid"]
    listed_width = long_leg["strike"] - short_leg["strike"]

    priced = {
        "trade_date": trade_date,
        "tenor": tenor,
        "spy_close": row["spy_close"],
        "vix_style_vol_pct": row["vix_style_vol_pct"],
        "corsi_vrp_log": row["corsi_vrp_log"],
        "corsi_vrp_z_3m": row["corsi_vrp_z_3m"],
        "corsi_vrp_z_1y": row["corsi_vrp_z_1y"],
        "rsi14": row["rsi14"],
        "short_call_1sd_raw": short_raw,
        "long_call_3sd_raw": long_raw,
        "short_expiration": short_leg["expiration"],
        "long_expiration": long_leg["expiration"],
        "same_expiration": same_expiration,
        "short_strike_listed": short_leg["strike"],
        "long_strike_listed": long_leg["strike"],
        "listed_width": listed_width,
        "short_bid": short_leg["bid"],
        "short_ask": short_leg["ask"],
        "short_mid": short_leg["mid"],
        "long_bid": long_leg["bid"],
        "long_ask": long_leg["ask"],
        "long_mid": long_leg["mid"],
        "entry_credit_bid_ask": entry_credit_bid_ask,
        "entry_credit_mid": entry_credit_mid,
        "short_timestamp": short_leg["timestamp"],
        "long_timestamp": long_leg["timestamp"],
        "short_underlying_price": short_leg["underlying_price"],
        "long_underlying_price": long_leg["underlying_price"],
        "short_implied_vol": short_leg["implied_vol"],
        "long_implied_vol": long_leg["implied_vol"],
        "short_delta": short_leg["delta"],
        "long_delta": long_leg["delta"],
    }

    priced_rows.append(priced)

    print(f"  Short leg: exp={short_leg['expiration']}, strike={short_leg['strike']}, bid={short_leg['bid']}, ask={short_leg['ask']}")
    print(f"  Long leg:  exp={long_leg['expiration']}, strike={long_leg['strike']}, bid={long_leg['bid']}, ask={long_leg['ask']}")
    print(f"  Same expiration: {same_expiration}")
    print(f"  Listed width:    {listed_width:.2f}")
    print(f"  Credit bid/ask:  {entry_credit_bid_ask:.4f}")
    print(f"  Credit mid:      {entry_credit_mid:.4f}")

priced_sample = pd.DataFrame(priced_rows)
attempts_df = pd.DataFrame(all_attempts)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

priced_sample_path = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_priced_sample_v1.parquet"
)

priced_sample_csv_path = (
    AUDIT_DIR / f"02B3_call_sleeve_corsi_spy_priced_sample_{timestamp}.csv"
)

attempts_path = (
    AUDIT_DIR / f"02B3_call_sleeve_corsi_spy_pricing_attempts_{timestamp}.csv"
)

manifest_path = (
    AUDIT_DIR / f"02B3_call_sleeve_corsi_spy_pricing_sample_manifest_{timestamp}.json"
)

if not priced_sample.empty:
    priced_sample.to_parquet(priced_sample_path, index=False)
else:
    # Save an empty placeholder with CSV only if no rows priced.
    priced_sample_path = None

priced_sample.to_csv(priced_sample_csv_path, index=False)
attempts_df.to_csv(attempts_path, index=False)

manifest = {
    "cell": "2B.3",
    "description": "Tiny SPY EOD greeks pricing sample for Corsi call-spread raw candidates",
    "created_at": timestamp,
    "raw_candidate_panel_path": str(RAW_CANDIDATE_PANEL_PATH),
    "base_url": BASE_URL,
    "endpoint": ENDPOINT,
    "symbol": SYMBOL,
    "right": RIGHT,
    "test_tenors": TEST_TENORS,
    "latest_complete_date": str(latest_complete_date.date()),
    "sample_rows": int(len(sample)),
    "priced_rows": int(len(priced_sample)),
    "attempt_rows": int(len(attempts_df)),
    "priced_sample_path": str(priced_sample_path) if priced_sample_path is not None else None,
    "priced_sample_csv_path": str(priced_sample_csv_path),
    "attempts_path": str(attempts_path),
    "contract_selection": {
        "expiration": "near trade_date + tenor, prefer on/after target date",
        "short_strike": "ceil raw 1SD strike to nearest $1, fallback to higher listed strikes",
        "long_strike": "ceil raw 3SD strike to nearest $1, fallback to higher listed strikes",
        "pricing": "short bid minus long ask",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Pricing sample summary")
print("=" * 100)
print(f"Sample rows attempted:   {len(sample):,}")
print(f"Priced rows:             {len(priced_sample):,}")
print(f"Total endpoint attempts: {len(attempts_df):,}")
print()

if priced_sample.empty:
    print("No complete call spreads priced.")
else:
    display(priced_sample)

print()
print("=" * 100)
print("Attempt summary")
print("=" * 100)

if attempts_df.empty:
    print("No attempts recorded.")
else:
    summary = (
        attempts_df
        .groupby(["leg_label", "status_code", "has_bid_ask"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["leg_label", "status_code", "has_bid_ask"])
    )
    display(summary)

print()
print("Saved:")
print(f"  Priced sample parquet: {priced_sample_path}")
print(f"  Priced sample CSV:     {priced_sample_csv_path}")
print(f"  Attempts CSV:          {attempts_path}")
print(f"  Manifest:              {manifest_path}")

Cell 2B.3 — Test SPY EOD greeks pricing for raw call-spread legs
Raw candidate panel: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet
ThetaData base URL:  http://127.0.0.1:25503
Endpoint:            /v3/option/history/greeks/eod

Sample candidates:


,trade_date,tenor,spy_close,vix_style_vol_pct,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw,corsi_vrp_log,corsi_vrp_z_3m,corsi_vrp_z_1y,rsi14
14679,2026-07-01,9,745.76,12.841456,760.797934,790.873803,30.075869,0.026507,-1.727898,-1.529773,54.736742
14686,2026-07-01,30,745.76,16.396160,780.815441,850.926322,70.110881,0.297561,-1.029999,-1.140252,54.736742
14687,2026-07-01,33,745.76,16.567266,782.910141,857.210422,74.300282,0.315352,-1.049221,-1.181808,54.736742



----------------------------------------------------------------------------------------------------
Pricing sample row: trade_date=2026-07-01, tenor=9
  short raw strike: 760.7979
  long raw strike:  790.8738
  Short leg: exp=2026-07-10, strike=761.0, bid=0.44, ask=0.45
  Long leg:  exp=2026-07-10, strike=791.0, bid=0.01, ask=0.02
  Same expiration: True
  Listed width:    30.00
  Credit bid/ask:  0.4200
  Credit mid:      0.4300

----------------------------------------------------------------------------------------------------
Pricing sample row: trade_date=2026-07-01, tenor=30
  short raw strike: 780.8154
  long raw strike:  850.9263
  Short leg: exp=2026-07-31, strike=781.0, bid=0.95, ask=0.97
  Long leg:  exp=2026-07-31, strike=855.0, bid=0.01, ask=0.02
  Same expiration: True
  Listed width:    74.00
  Credit bid/ask:  0.9300
  Credit mid:      0.9450

----------------------------------------------------------------------------------------------------
Pricing sample row: trade

,trade_date,tenor,spy_close,vix_style_vol_pct,corsi_vrp_log,corsi_vrp_z_3m,corsi_vrp_z_1y,rsi14,short_call_1sd_raw,long_call_3sd_raw,...,entry_credit_bid_ask,entry_credit_mid,short_timestamp,long_timestamp,short_underlying_price,long_underlying_price,short_implied_vol,long_implied_vol,short_delta,long_delta
0,2026-07-01,9,745.76,12.841456,0.026507,-1.727898,-1.529773,54.736742,760.797934,790.873803,...,0.42,0.430,2026-07-01T16:07:37.143,2026-07-01T00:00:00.000,745.76,745.76,0.0911,0.1289,0.0895,0.0021
1,2026-07-01,30,745.76,16.396160,0.297561,-1.029999,-1.140252,54.736742,780.815441,850.926322,...,0.93,0.945,2026-07-01T14:22:31.742,2026-07-01T15:35:22.314,745.76,745.76,0.1104,0.1523,0.0889,0.0011
2,2026-07-01,33,745.76,16.567266,0.315352,-1.049221,-1.181808,54.736742,782.910141,857.210422,...,1.37,1.395,2026-07-01T15:31:42.156,2026-07-01T00:00:00.000,745.76,745.76,0.1138,0.1503,0.1104,0.0020



Attempt summary


,leg_label,status_code,has_bid_ask,count
0,long_call_3sd,200,True,3
1,long_call_3sd,472,False,58
2,short_call_1sd,200,True,3
3,short_call_1sd,472,False,52



Saved:
  Priced sample parquet: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_priced_sample_v1.parquet
  Priced sample CSV:     C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B3_call_sleeve_corsi_spy_priced_sample_20260710_095921.csv
  Attempts CSV:          C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B3_call_sleeve_corsi_spy_pricing_attempts_20260710_095921.csv
  Manifest:              C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02B3_call_sleeve_corsi_spy_pricing_sample_manifest_20260710_095921.json


In [10]:
# Cell 2C — Build efficient SPY pricing request plan
# Purpose:
#   Build a deterministic, deduplicated SPY contract pricing request plan for the Corsi
#   all-tenor short-call candidate panel.
#
# Why this exists:
#   Cell 2B.3 proved pricing works, but the fallback search made 116 endpoint attempts
#   for only 3 spreads. This cell creates a clean primary request plan first.
#
# Scope:
#   - No ThetaData calls
#   - No pricing pull
#   - No parameter sweep
#   - No final trade selection
#
# Primary contract convention:
#   - expiration = first Friday on or after trade_date + tenor
#   - short call strike = ceil raw 1SD strike to nearest $1
#   - long call strike  = ceil raw 3SD strike to nearest $1
#
# Fallback contract convention, saved but not pulled yet:
#   - same expiration
#   - nearest $5 strike at/above raw strike when different from $1 rounded strike
#
# Pricing endpoint intended for next cell:
#   /v3/option/history/greeks/eod
#
# Output:
#   Processed:
#     call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet
#     call_sleeve_corsi_spy_unique_contract_request_plan_primary_v1.parquet
#     call_sleeve_corsi_spy_candidate_leg_plan_fallback_v1.parquet
#     call_sleeve_corsi_spy_unique_contract_request_plan_fallback_v1.parquet
#
#   Audit:
#     02C_* timestamped CSVs + manifest

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import json
import math

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

SYMBOL = "SPY"
RIGHT = "C"
ENDPOINT = "/v3/option/history/greeks/eod"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2C — Build efficient SPY pricing request plan")
print("=" * 100)
print(f"Raw candidate panel: {RAW_CANDIDATE_PANEL_PATH}")
print(f"Symbol:              {SYMBOL}")
print(f"Right:               {RIGHT}")
print(f"Intended endpoint:   {ENDPOINT}")
print()

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def ceil_to_increment(x, increment):
    if pd.isna(x):
        return np.nan
    return float(math.ceil(float(x) / float(increment)) * float(increment))

def next_friday_on_or_after(dt):
    d = pd.Timestamp(dt).normalize()
    # Monday=0, Friday=4
    days_forward = (4 - d.weekday()) % 7
    return d + pd.Timedelta(days=days_forward)

def make_candidate_id(trade_date, tenor):
    return f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_T{int(tenor):02d}"

def make_contract_request_id(symbol, trade_date, expiration, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def add_contract_fields(df):
    out = df.copy()
    out["contract_request_id"] = out.apply(
        lambda r: make_contract_request_id(
            r["symbol"],
            r["trade_date"],
            r["expiration_date"],
            r["strike_listed"],
            r["right"],
        ),
        axis=1,
    )
    out["trade_date_str"] = out["trade_date"].dt.strftime("%Y-%m-%d")
    out["expiration_str"] = out["expiration_date"].dt.strftime("%Y-%m-%d")
    out["expiration_yyyymmdd"] = out["expiration_date"].dt.strftime("%Y%m%d")
    return out

def build_unique_contract_plan(leg_plan):
    cols = [
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "endpoint",
    ]

    unique = (
        leg_plan[cols]
        .drop_duplicates()
        .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
        .reset_index(drop=True)
    )

    # Useful counts for joining back later.
    usage = (
        leg_plan
        .groupby("contract_request_id")
        .agg(
            mapped_candidate_legs=("candidate_id", "size"),
            mapped_unique_candidates=("candidate_id", "nunique"),
            mapped_tenors=("tenor", lambda x: ",".join(map(str, sorted(set(x.astype(int)))))),
            mapped_leg_labels=("leg_label", lambda x: ",".join(sorted(set(map(str, x))))),
        )
        .reset_index()
    )

    unique = unique.merge(usage, on="contract_request_id", how="left")

    return unique

# --------------------------------------------------------------------------------------
# Load candidate panel
# --------------------------------------------------------------------------------------

raw = pd.read_parquet(RAW_CANDIDATE_PANEL_PATH)
raw["trade_date"] = pd.to_datetime(raw["trade_date"], errors="coerce")

required_cols = [
    "trade_date",
    "tenor",
    "candidate_inputs_complete",
    "spy_close",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
]

missing_required = [c for c in required_cols if c not in raw.columns]
if missing_required:
    raise ValueError(f"Missing required columns in raw candidate panel: {missing_required}")

complete = raw.loc[raw["candidate_inputs_complete"]].copy()

if complete.empty:
    raise ValueError("No complete candidate rows found.")

complete["tenor"] = complete["tenor"].astype(int)
complete["candidate_id"] = complete.apply(
    lambda r: make_candidate_id(r["trade_date"], r["tenor"]),
    axis=1,
)

# Expiration convention
complete["target_expiration_date"] = complete["trade_date"] + pd.to_timedelta(complete["tenor"], unit="D")
complete["primary_expiration_date"] = complete["target_expiration_date"].apply(next_friday_on_or_after)
complete["primary_effective_dte"] = (
    complete["primary_expiration_date"] - complete["trade_date"]
).dt.days
complete["expiration_slippage_days"] = (
    complete["primary_expiration_date"] - complete["target_expiration_date"]
).dt.days

# Strike convention
complete["short_strike_ceil_1"] = complete["short_call_1sd_raw"].apply(lambda x: ceil_to_increment(x, 1.0))
complete["long_strike_ceil_1"] = complete["long_call_3sd_raw"].apply(lambda x: ceil_to_increment(x, 1.0))

complete["short_strike_ceil_5"] = complete["short_call_1sd_raw"].apply(lambda x: ceil_to_increment(x, 5.0))
complete["long_strike_ceil_5"] = complete["long_call_3sd_raw"].apply(lambda x: ceil_to_increment(x, 5.0))

complete["short_rounding_distance_1"] = complete["short_strike_ceil_1"] - complete["short_call_1sd_raw"]
complete["long_rounding_distance_1"] = complete["long_strike_ceil_1"] - complete["long_call_3sd_raw"]
complete["short_rounding_distance_5"] = complete["short_strike_ceil_5"] - complete["short_call_1sd_raw"]
complete["long_rounding_distance_5"] = complete["long_strike_ceil_5"] - complete["long_call_3sd_raw"]

# --------------------------------------------------------------------------------------
# Build primary candidate-leg plan
# --------------------------------------------------------------------------------------

base_cols = [
    "candidate_id",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "vix_style_vol_pct",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "target_expiration_date",
    "primary_expiration_date",
    "primary_effective_dte",
    "expiration_slippage_days",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
]

available_base_cols = [c for c in base_cols if c in complete.columns]

short_primary = complete[available_base_cols].copy()
short_primary["leg_label"] = "short_call_1sd"
short_primary["symbol"] = SYMBOL
short_primary["right"] = RIGHT
short_primary["endpoint"] = ENDPOINT
short_primary["expiration_date"] = short_primary["primary_expiration_date"]
short_primary["raw_strike"] = complete["short_call_1sd_raw"].values
short_primary["strike_listed"] = complete["short_strike_ceil_1"].values
short_primary["strike_rounding_increment"] = 1.0
short_primary["strike_rounding_distance"] = complete["short_rounding_distance_1"].values
short_primary["leg_side"] = "sell"
short_primary["price_field_for_entry"] = "bid"
short_primary["contract_priority"] = 1
short_primary["plan_type"] = "primary"

long_primary = complete[available_base_cols].copy()
long_primary["leg_label"] = "long_call_3sd"
long_primary["symbol"] = SYMBOL
long_primary["right"] = RIGHT
long_primary["endpoint"] = ENDPOINT
long_primary["expiration_date"] = long_primary["primary_expiration_date"]
long_primary["raw_strike"] = complete["long_call_3sd_raw"].values
long_primary["strike_listed"] = complete["long_strike_ceil_1"].values
long_primary["strike_rounding_increment"] = 1.0
long_primary["strike_rounding_distance"] = complete["long_rounding_distance_1"].values
long_primary["leg_side"] = "buy"
long_primary["price_field_for_entry"] = "ask"
long_primary["contract_priority"] = 1
long_primary["plan_type"] = "primary"

primary_leg_plan = pd.concat([short_primary, long_primary], ignore_index=True)

primary_leg_plan = primary_leg_plan.loc[
    primary_leg_plan["trade_date"].notna()
    & primary_leg_plan["expiration_date"].notna()
    & primary_leg_plan["strike_listed"].notna()
].copy()

primary_leg_plan = add_contract_fields(primary_leg_plan)

# Ensure long strike is above short strike inside each candidate for primary plan.
primary_width_check = (
    primary_leg_plan
    .pivot_table(
        index="candidate_id",
        columns="leg_label",
        values="strike_listed",
        aggfunc="first",
    )
    .reset_index()
)

primary_width_check["primary_width_listed"] = (
    primary_width_check["long_call_3sd"] - primary_width_check["short_call_1sd"]
)

bad_width_ids = primary_width_check.loc[
    primary_width_check["primary_width_listed"] <= 0,
    "candidate_id"
].tolist()

if bad_width_ids:
    raise ValueError(f"Primary plan produced non-positive listed width for {len(bad_width_ids)} candidates.")

# Add listed width back to leg plan.
primary_leg_plan = primary_leg_plan.merge(
    primary_width_check[["candidate_id", "primary_width_listed"]],
    on="candidate_id",
    how="left",
)

primary_unique_contract_plan = build_unique_contract_plan(primary_leg_plan)

# --------------------------------------------------------------------------------------
# Build fallback candidate-leg plan
# --------------------------------------------------------------------------------------

fallback_rows = []

# Only create fallback rows where $5 rounded strike differs from the primary $1 strike.
for leg_label, raw_col, strike_1_col, strike_5_col, side, price_field in [
    ("short_call_1sd", "short_call_1sd_raw", "short_strike_ceil_1", "short_strike_ceil_5", "sell", "bid"),
    ("long_call_3sd", "long_call_3sd_raw", "long_strike_ceil_1", "long_strike_ceil_5", "buy", "ask"),
]:
    fb = complete.loc[
        complete[strike_5_col].notna()
        & complete[strike_1_col].notna()
        & (complete[strike_5_col] != complete[strike_1_col])
    ][available_base_cols].copy()

    if fb.empty:
        continue

    fb["leg_label"] = leg_label
    fb["symbol"] = SYMBOL
    fb["right"] = RIGHT
    fb["endpoint"] = ENDPOINT
    fb["expiration_date"] = fb["primary_expiration_date"]
    fb["raw_strike"] = complete.loc[fb.index, raw_col].values
    fb["strike_listed"] = complete.loc[fb.index, strike_5_col].values
    fb["strike_rounding_increment"] = 5.0
    fb["strike_rounding_distance"] = fb["strike_listed"] - fb["raw_strike"]
    fb["leg_side"] = side
    fb["price_field_for_entry"] = price_field
    fb["contract_priority"] = 2
    fb["plan_type"] = "fallback_strike_5"

    fallback_rows.append(fb)

if fallback_rows:
    fallback_leg_plan = pd.concat(fallback_rows, ignore_index=True)
    fallback_leg_plan = add_contract_fields(fallback_leg_plan)
else:
    fallback_leg_plan = pd.DataFrame(columns=primary_leg_plan.columns)

if not fallback_leg_plan.empty:
    fallback_unique_contract_plan = build_unique_contract_plan(fallback_leg_plan)
else:
    fallback_unique_contract_plan = pd.DataFrame(columns=primary_unique_contract_plan.columns)

# --------------------------------------------------------------------------------------
# Save processed outputs
# --------------------------------------------------------------------------------------

primary_leg_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet"
)
primary_unique_contract_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_unique_contract_request_plan_primary_v1.parquet"
)
fallback_leg_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_candidate_leg_plan_fallback_v1.parquet"
)
fallback_unique_contract_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_unique_contract_request_plan_fallback_v1.parquet"
)

primary_leg_plan.to_parquet(primary_leg_plan_path, index=False)
primary_unique_contract_plan.to_parquet(primary_unique_contract_plan_path, index=False)
fallback_leg_plan.to_parquet(fallback_leg_plan_path, index=False)
fallback_unique_contract_plan.to_parquet(fallback_unique_contract_plan_path, index=False)

# Audit CSVs
primary_leg_plan_csv = AUDIT_DIR / f"02C_candidate_leg_plan_primary_{timestamp}.csv"
primary_unique_contract_plan_csv = AUDIT_DIR / f"02C_unique_contract_request_plan_primary_{timestamp}.csv"
fallback_leg_plan_csv = AUDIT_DIR / f"02C_candidate_leg_plan_fallback_{timestamp}.csv"
fallback_unique_contract_plan_csv = AUDIT_DIR / f"02C_unique_contract_request_plan_fallback_{timestamp}.csv"

primary_leg_plan.to_csv(primary_leg_plan_csv, index=False)
primary_unique_contract_plan.to_csv(primary_unique_contract_plan_csv, index=False)
fallback_leg_plan.to_csv(fallback_leg_plan_csv, index=False)
fallback_unique_contract_plan.to_csv(fallback_unique_contract_plan_csv, index=False)

# --------------------------------------------------------------------------------------
# Diagnostics
# --------------------------------------------------------------------------------------

candidate_summary = pd.DataFrame([{
    "raw_rows": len(raw),
    "complete_candidate_rows": len(complete),
    "unique_candidate_ids": complete["candidate_id"].nunique(),
    "date_min": complete["trade_date"].min().date(),
    "date_max": complete["trade_date"].max().date(),
    "tenors": ", ".join(map(str, sorted(complete["tenor"].unique()))),
    "primary_leg_rows": len(primary_leg_plan),
    "primary_unique_contract_requests": len(primary_unique_contract_plan),
    "fallback_leg_rows": len(fallback_leg_plan),
    "fallback_unique_contract_requests": len(fallback_unique_contract_plan),
}])

tenor_summary = (
    complete
    .groupby("tenor")
    .agg(
        candidate_rows=("candidate_id", "size"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        median_spy_close=("spy_close", "median"),
        median_vix_style_vol_pct=("vix_style_vol_pct", "median"),
        median_primary_effective_dte=("primary_effective_dte", "median"),
        min_primary_effective_dte=("primary_effective_dte", "min"),
        max_primary_effective_dte=("primary_effective_dte", "max"),
        median_expiration_slippage_days=("expiration_slippage_days", "median"),
        max_expiration_slippage_days=("expiration_slippage_days", "max"),
        median_short_rounding_distance_1=("short_rounding_distance_1", "median"),
        median_long_rounding_distance_1=("long_rounding_distance_1", "median"),
        median_short_rounding_distance_5=("short_rounding_distance_5", "median"),
        median_long_rounding_distance_5=("long_rounding_distance_5", "median"),
    )
    .reset_index()
)

leg_summary = (
    primary_leg_plan
    .groupby("leg_label")
    .agg(
        leg_rows=("candidate_id", "size"),
        unique_contract_requests=("contract_request_id", "nunique"),
        min_strike=("strike_listed", "min"),
        median_strike=("strike_listed", "median"),
        max_strike=("strike_listed", "max"),
        median_rounding_distance=("strike_rounding_distance", "median"),
        max_rounding_distance=("strike_rounding_distance", "max"),
    )
    .reset_index()
)

slippage_summary = (
    complete
    .groupby(["tenor", "expiration_slippage_days"])
    .size()
    .reset_index(name="candidate_rows")
    .sort_values(["tenor", "expiration_slippage_days"])
)

width_summary = (
    primary_width_check
    .agg(
        candidate_rows=("candidate_id", "size"),
        min_primary_width_listed=("primary_width_listed", "min"),
        median_primary_width_listed=("primary_width_listed", "median"),
        max_primary_width_listed=("primary_width_listed", "max"),
    )
    .reset_index(drop=True)
)

candidate_summary_csv = AUDIT_DIR / f"02C_candidate_summary_{timestamp}.csv"
tenor_summary_csv = AUDIT_DIR / f"02C_tenor_summary_{timestamp}.csv"
leg_summary_csv = AUDIT_DIR / f"02C_leg_summary_{timestamp}.csv"
slippage_summary_csv = AUDIT_DIR / f"02C_expiration_slippage_summary_{timestamp}.csv"
width_summary_csv = AUDIT_DIR / f"02C_width_summary_{timestamp}.csv"

candidate_summary.to_csv(candidate_summary_csv, index=False)
tenor_summary.to_csv(tenor_summary_csv, index=False)
leg_summary.to_csv(leg_summary_csv, index=False)
slippage_summary.to_csv(slippage_summary_csv, index=False)
width_summary.to_csv(width_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02C_spy_pricing_request_plan_manifest_{timestamp}.json"

manifest = {
    "cell": "2C",
    "description": "Efficient deterministic SPY contract request plan for Corsi call-sleeve candidates",
    "created_at": timestamp,
    "raw_candidate_panel_path": str(RAW_CANDIDATE_PANEL_PATH),
    "endpoint": ENDPOINT,
    "symbol": SYMBOL,
    "right": RIGHT,
    "primary_contract_convention": {
        "expiration": "first Friday on or after trade_date + tenor",
        "short_call_strike": "ceil raw 1SD strike to nearest $1",
        "long_call_strike": "ceil raw 3SD strike to nearest $1",
    },
    "fallback_contract_convention": {
        "expiration": "same as primary",
        "strike": "ceil raw strike to nearest $5 when different from primary $1 strike",
        "note": "fallback plan is saved but should only be used for misses after primary pricing coverage is checked",
    },
    "counts": candidate_summary.iloc[0].to_dict(),
    "processed_outputs": {
        "primary_leg_plan_path": str(primary_leg_plan_path),
        "primary_unique_contract_plan_path": str(primary_unique_contract_plan_path),
        "fallback_leg_plan_path": str(fallback_leg_plan_path),
        "fallback_unique_contract_plan_path": str(fallback_unique_contract_plan_path),
    },
    "audit_outputs": {
        "primary_leg_plan_csv": str(primary_leg_plan_csv),
        "primary_unique_contract_plan_csv": str(primary_unique_contract_plan_csv),
        "fallback_leg_plan_csv": str(fallback_leg_plan_csv),
        "fallback_unique_contract_plan_csv": str(fallback_unique_contract_plan_csv),
        "candidate_summary_csv": str(candidate_summary_csv),
        "tenor_summary_csv": str(tenor_summary_csv),
        "leg_summary_csv": str(leg_summary_csv),
        "slippage_summary_csv": str(slippage_summary_csv),
        "width_summary_csv": str(width_summary_csv),
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Candidate / request summary")
print("=" * 100)
display(candidate_summary)

print()
print("=" * 100)
print("Tenor summary")
print("=" * 100)
display(tenor_summary)

print()
print("=" * 100)
print("Primary leg summary")
print("=" * 100)
display(leg_summary)

print()
print("=" * 100)
print("Primary listed-width summary")
print("=" * 100)
display(width_summary)

print()
print("=" * 100)
print("Expiration slippage preview")
print("=" * 100)
display(slippage_summary.head(50))

print()
print("=" * 100)
print("Primary request plan preview")
print("=" * 100)
display(
    primary_unique_contract_plan[
        [
            "contract_request_id",
            "symbol",
            "trade_date_str",
            "expiration_str",
            "strike_listed",
            "right",
            "mapped_candidate_legs",
            "mapped_tenors",
            "mapped_leg_labels",
        ]
    ].head(30)
)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Primary leg plan:              {primary_leg_plan_path}")
print(f"Primary unique contract plan:  {primary_unique_contract_plan_path}")
print(f"Fallback leg plan:             {fallback_leg_plan_path}")
print(f"Fallback unique contract plan: {fallback_unique_contract_plan_path}")
print(f"Manifest:                      {manifest_path}")

print()
print("Result: Request plan built. Next cell should price PRIMARY unique contracts only, then audit coverage before using fallback.")

Cell 2C — Build efficient SPY pricing request plan
Raw candidate panel: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet
Symbol:              SPY
Right:               C
Intended endpoint:   /v3/option/history/greeks/eod

Candidate / request summary


,raw_rows,complete_candidate_rows,unique_candidate_ids,date_min,date_max,tenors,primary_leg_rows,primary_unique_contract_requests,fallback_leg_rows,fallback_unique_contract_requests
0,14715,12420,12420,2020-12-31,2026-07-01,"9, 12, 15, 18, 21, 24, 27, 30, 33",24840,24825,19855,17101



Tenor summary


,tenor,candidate_rows,date_min,date_max,median_spy_close,median_vix_style_vol_pct,median_primary_effective_dte,min_primary_effective_dte,max_primary_effective_dte,median_expiration_slippage_days,max_expiration_slippage_days,median_short_rounding_distance_1,median_long_rounding_distance_1,median_short_rounding_distance_5,median_long_rounding_distance_5
0,9,1380,2020-12-31,2026-07-01,459.035,16.565065,11.0,9,15,2.0,6,0.503259,0.522067,2.518796,2.442357
1,12,1380,2020-12-31,2026-07-01,459.035,16.754888,16.0,14,18,4.0,6,0.497643,0.511353,2.401681,2.525145
2,15,1380,2020-12-31,2026-07-01,459.035,17.065347,17.0,15,21,2.0,6,0.506683,0.503743,2.625151,2.525474
3,18,1380,2020-12-31,2026-07-01,459.035,17.342120,22.0,18,24,4.0,6,0.487758,0.4998,2.432439,2.550556
4,21,1380,2020-12-31,2026-07-01,459.035,17.594531,23.0,21,25,2.0,4,0.510872,0.492001,2.404912,2.604865
5,24,1380,2020-12-31,2026-07-01,459.035,17.766361,28.0,24,30,4.0,6,0.501171,0.522823,2.488841,2.590599
6,27,1380,2020-12-31,2026-07-01,459.035,17.942570,30.0,28,32,3.0,5,0.510794,0.499919,2.549604,2.440878
7,30,1380,2020-12-31,2026-07-01,459.035,18.118708,32.0,30,36,2.0,6,0.500069,0.49628,2.535878,2.420196
8,33,1380,2020-12-31,2026-07-01,459.035,18.277626,37.0,35,39,4.0,6,0.483177,0.498522,2.40769,2.519922



Primary leg summary


...


Expiration slippage preview


,tenor,expiration_slippage_days,candidate_rows
0,9,0,284
1,9,1,286
2,9,2,256
3,9,5,277
4,9,6,277
5,12,2,277
6,12,3,277
7,12,4,284
8,12,5,286
9,12,6,256



Primary request plan preview


,contract_request_id,symbol,trade_date_str,expiration_str,strike_listed,right,mapped_candidate_legs,mapped_tenors,mapped_leg_labels
0,SPY_20201231_20210115_386.0_C,SPY,2020-12-31,2021-01-15,386.0,C,2,"9,12",short_call_1sd
1,SPY_20201231_20210115_387.0_C,SPY,2020-12-31,2021-01-15,387.0,C,1,15,short_call_1sd
2,SPY_20201231_20210115_409.0_C,SPY,2020-12-31,2021-01-15,409.0,C,1,9,long_call_3sd
3,SPY_20201231_20210115_410.0_C,SPY,2020-12-31,2021-01-15,410.0,C,1,12,long_call_3sd
4,SPY_20201231_20210115_413.0_C,SPY,2020-12-31,2021-01-15,413.0,C,1,15,long_call_3sd
5,SPY_20201231_20210122_391.0_C,SPY,2020-12-31,2021-01-22,391.0,C,1,18,short_call_1sd
6,SPY_20201231_20210122_393.0_C,SPY,2020-12-31,2021-01-22,393.0,C,1,21,short_call_1sd
7,SPY_20201231_20210122_423.0_C,SPY,2020-12-31,2021-01-22,423.0,C,1,18,long_call_3sd
8,SPY_20201231_20210122_432.0_C,SPY,2020-12-31,2021-01-22,432.0,C,1,21,long_call_3sd
9,SPY_20201231_20210129_395.0_C,SPY,2020-12-31,2021-01-29,395.0,C,1,24,short_call_1sd



Saved
Primary leg plan:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet
Primary unique contract plan:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_unique_contract_request_plan_primary_v1.parquet
Fallback leg plan:             C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_candidate_leg_plan_fallback_v1.parquet
Fallback unique contract plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_unique_contract_request_plan_fallback_v1.parquet
Manifest:                      C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02C_spy_pricing_request_plan_manifest_20260710_100939.json

Result: Request plan built. Next cell should price PRIMARY unique contracts only, then audit coverage before using fallback.


In [11]:
# Cell 2C.1 — Test SPY historical EOD chain endpoint
# Purpose:
#   Test whether ThetaData v3 /v3/option/history/eod can return a full historical
#   EOD option chain for one SPY trade_date + expiration.
#
# Why:
#   Cell 2C built a contract-level pricing plan with ~24,825 unique contract requests.
#   If /v3/option/history/eod can return the whole chain for one date/expiration,
#   we can instead cache at the chain level and select both call-spread legs locally.
#
# Scope:
#   - Tiny endpoint probe only
#   - No bulk cache
#   - No parameter sweep
#
# Test:
#   1. Load primary leg plan from Cell 2C.
#   2. Pick one real candidate spread.
#   3. Probe /v3/option/history/eod with start_date/end_date.
#   4. Check if returned data looks like a chain.
#   5. Select planned short/long call legs locally from returned chain.
#   6. Compare with contract-level /v3/option/history/greeks/eod for same legs.

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
CHAIN_TEST_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy_eod_test"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHAIN_TEST_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_LEG_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet"
)

BASE_URL = "http://127.0.0.1:25503"

CHAIN_ENDPOINT = "/v3/option/history/eod"
CONTRACT_ENDPOINT = "/v3/option/history/greeks/eod"

SYMBOL = "SPY"
RIGHT = "C"

# Prefer the known recent sample date from prior test if available.
PREFERRED_TRADE_DATE = pd.Timestamp("2026-07-01")
PREFERRED_TENOR = 30

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2C.1 — Test SPY historical EOD chain endpoint")
print("=" * 100)
print(f"Primary leg plan:       {PRIMARY_LEG_PLAN_PATH}")
print(f"Base URL:               {BASE_URL}")
print(f"Chain endpoint:          {CHAIN_ENDPOINT}")
print(f"Contract endpoint ctrl:  {CONTRACT_ENDPOINT}")
print()

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    # JSON
    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    # CSV
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    # NDJSON
    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def get_df(endpoint, params, timeout=(2, 60)):
    url = BASE_URL.rstrip("/") + endpoint

    try:
        r = session.get(url, params=params, timeout=timeout)
        df = response_to_dataframe(r)

        return {
            "endpoint": endpoint,
            "url": r.url,
            "params": params,
            "status_code": r.status_code,
            "ok": r.ok,
            "rows": len(df),
            "cols": len(df.columns),
            "columns": list(df.columns),
            "text_preview": r.text[:1000],
            "error": None,
            "df": df,
        }

    except Exception as e:
        return {
            "endpoint": endpoint,
            "url": url,
            "params": params,
            "status_code": None,
            "ok": False,
            "rows": 0,
            "cols": 0,
            "columns": [],
            "text_preview": "",
            "error": repr(e),
            "df": pd.DataFrame(),
        }

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c

    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]

    return None

def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None

def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    rename_map = {}
    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "underlying_price": ["underlying_price"],
        "implied_vol": ["implied_vol", "iv"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
    }

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    for col in [
        "strike",
        "bid",
        "ask",
        "mid",
        "underlying_price",
        "implied_vol",
        "delta",
        "theta",
        "vega",
        "gamma",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out

def has_chain_schema(df):
    if df is None or df.empty:
        return False

    required = [
        first_col(df, ["strike"]),
        first_col(df, ["right", "option_type"]),
        first_col(df, ["bid"]),
        first_col(df, ["ask"]),
    ]

    return all(c is not None for c in required)

def pick_leg_from_chain(chain_df, expiration, strike, right="C"):
    if chain_df is None or chain_df.empty:
        return pd.Series(dtype="object")

    expiration_iso = pd.Timestamp(expiration).strftime("%Y-%m-%d")

    mask = (
        (chain_df["strike"].round(4) == round(float(strike), 4))
        & (chain_df["right"] == right)
    )

    if "expiration" in chain_df.columns:
        mask = mask & (chain_df["expiration"] == expiration_iso)

    leg = chain_df.loc[mask].copy()

    if leg.empty:
        return pd.Series(dtype="object")

    # If multiple rows, take last valid bid/ask row.
    leg = leg.loc[leg["bid"].notna() & leg["ask"].notna()].copy()

    if leg.empty:
        return pd.Series(dtype="object")

    return leg.iloc[-1]

def contract_level_price(trade_date, expiration, strike, right="C"):
    params = {
        "symbol": SYMBOL,
        "right": right,
        "strike": float(strike),
        "expiration": pd.Timestamp(expiration).strftime("%Y-%m-%d"),
        "start_date": pd.Timestamp(trade_date).strftime("%Y-%m-%d"),
        "end_date": pd.Timestamp(trade_date).strftime("%Y-%m-%d"),
        "format": "csv",
    }

    result = get_df(CONTRACT_ENDPOINT, params=params, timeout=(2, 45))
    std = standardize_option_rows(result["df"])

    row = {
        "status_code": result["status_code"],
        "ok": result["ok"],
        "rows": result["rows"],
        "cols": result["cols"],
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "timestamp": None,
        "underlying_price": np.nan,
        "implied_vol": np.nan,
        "delta": np.nan,
        "url": result["url"],
        "error": result["error"],
        "text_preview": result["text_preview"],
    }

    if not std.empty and {"bid", "ask"}.issubset(std.columns):
        valid = std.loc[std["bid"].notna() & std["ask"].notna()].copy()
        if not valid.empty:
            x = valid.iloc[-1]
            for c in ["bid", "ask", "mid", "timestamp", "underlying_price", "implied_vol", "delta"]:
                if c in valid.columns:
                    row[c] = x.get(c, row[c])

    return row

# --------------------------------------------------------------------------------------
# Load a real candidate spread from primary plan
# --------------------------------------------------------------------------------------

leg_plan = pd.read_parquet(PRIMARY_LEG_PLAN_PATH)
leg_plan["trade_date"] = pd.to_datetime(leg_plan["trade_date"], errors="coerce")
leg_plan["expiration_date"] = pd.to_datetime(leg_plan["expiration_date"], errors="coerce")

candidate_ids_on_preferred = leg_plan.loc[
    (leg_plan["trade_date"] == PREFERRED_TRADE_DATE)
    & (leg_plan["tenor"] == PREFERRED_TENOR),
    "candidate_id"
].drop_duplicates().tolist()

if candidate_ids_on_preferred:
    test_candidate_id = candidate_ids_on_preferred[0]
else:
    # Fallback to most recent available 30D, then most recent overall.
    candidate_ids_30d = leg_plan.loc[
        leg_plan["tenor"] == PREFERRED_TENOR
    ].sort_values("trade_date")["candidate_id"].drop_duplicates().tolist()

    if candidate_ids_30d:
        test_candidate_id = candidate_ids_30d[-1]
    else:
        test_candidate_id = leg_plan.sort_values("trade_date")["candidate_id"].drop_duplicates().iloc[-1]

test_legs = (
    leg_plan.loc[leg_plan["candidate_id"] == test_candidate_id]
    .sort_values("leg_label")
    .copy()
)

if test_legs["leg_label"].nunique() != 2:
    raise ValueError(f"Expected exactly two legs for candidate_id={test_candidate_id}")

short_plan = test_legs.loc[test_legs["leg_label"] == "short_call_1sd"].iloc[0]
long_plan = test_legs.loc[test_legs["leg_label"] == "long_call_3sd"].iloc[0]

test_trade_date = pd.Timestamp(short_plan["trade_date"])
test_expiration = pd.Timestamp(short_plan["expiration_date"])

if pd.Timestamp(long_plan["expiration_date"]) != test_expiration:
    raise ValueError("Short and long legs have different planned expirations.")

print("=" * 100)
print("Test candidate")
print("=" * 100)

test_candidate_display_cols = [
    "candidate_id",
    "trade_date",
    "tenor",
    "leg_label",
    "expiration_date",
    "raw_strike",
    "strike_listed",
    "strike_rounding_increment",
    "strike_rounding_distance",
    "primary_width_listed",
]

display(test_legs[test_candidate_display_cols])

# --------------------------------------------------------------------------------------
# Probe chain endpoint with likely v3 parameter variants
# --------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Probe /v3/option/history/eod chain endpoint")
print("=" * 100)

trade_iso = test_trade_date.strftime("%Y-%m-%d")
trade_yyyymmdd = test_trade_date.strftime("%Y%m%d")
exp_iso = test_expiration.strftime("%Y-%m-%d")
exp_yyyymmdd = test_expiration.strftime("%Y%m%d")

chain_attempt_param_sets = [
    {
        "symbol": SYMBOL,
        "expiration": exp_iso,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "expiration": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "exp": exp_iso,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "exp": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "root": SYMBOL,
        "exp": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
]

chain_attempt_rows = []
selected_chain_raw = pd.DataFrame()
selected_chain_std = pd.DataFrame()
selected_attempt = None

for params in chain_attempt_param_sets:
    result = get_df(CHAIN_ENDPOINT, params=params, timeout=(2, 90))
    raw_df = result["df"]
    std_df = standardize_option_rows(raw_df)
    schema_ok = has_chain_schema(raw_df)
    standardized_schema_ok = has_chain_schema(std_df)

    unique_strikes = (
        int(std_df["strike"].nunique())
        if "strike" in std_df.columns and not std_df.empty
        else 0
    )
    unique_rights = (
        ",".join(sorted(std_df["right"].dropna().astype(str).unique()))
        if "right" in std_df.columns and not std_df.empty
        else ""
    )
    unique_expirations = (
        ",".join(sorted(std_df["expiration"].dropna().astype(str).unique())[:10])
        if "expiration" in std_df.columns and not std_df.empty
        else ""
    )

    short_leg_from_chain = pick_leg_from_chain(
        std_df,
        expiration=test_expiration,
        strike=short_plan["strike_listed"],
        right=RIGHT,
    )

    long_leg_from_chain = pick_leg_from_chain(
        std_df,
        expiration=test_expiration,
        strike=long_plan["strike_listed"],
        right=RIGHT,
    )

    found_short = not short_leg_from_chain.empty
    found_long = not long_leg_from_chain.empty

    chain_attempt_rows.append({
        "endpoint": CHAIN_ENDPOINT,
        "params": json.dumps(params),
        "status_code": result["status_code"],
        "ok": result["ok"],
        "rows": result["rows"],
        "cols": result["cols"],
        "columns": ", ".join(map(str, result["columns"])),
        "raw_schema_ok": schema_ok,
        "standardized_schema_ok": standardized_schema_ok,
        "unique_strikes": unique_strikes,
        "unique_rights": unique_rights,
        "unique_expirations": unique_expirations,
        "found_short_leg": found_short,
        "found_long_leg": found_long,
        "error": result["error"],
        "text_preview": result["text_preview"],
        "url": result["url"],
    })

    if (
        result["ok"]
        and result["rows"] > 0
        and standardized_schema_ok
        and found_short
        and found_long
        and selected_attempt is None
    ):
        selected_attempt = chain_attempt_rows[-1]
        selected_chain_raw = raw_df.copy()
        selected_chain_std = std_df.copy()

chain_attempts_df = pd.DataFrame(chain_attempt_rows)

display(
    chain_attempts_df[
        [
            "status_code",
            "ok",
            "rows",
            "cols",
            "standardized_schema_ok",
            "unique_strikes",
            "unique_rights",
            "unique_expirations",
            "found_short_leg",
            "found_long_leg",
            "error",
            "text_preview",
        ]
    ]
)

# --------------------------------------------------------------------------------------
# If chain works, select planned legs locally
# --------------------------------------------------------------------------------------

chain_leg_comparison = pd.DataFrame()
chain_spread_summary = pd.DataFrame()

if selected_attempt is not None:
    print()
    print("Selected working chain attempt:")
    print(selected_attempt["url"])

    short_chain = pick_leg_from_chain(
        selected_chain_std,
        expiration=test_expiration,
        strike=short_plan["strike_listed"],
        right=RIGHT,
    )

    long_chain = pick_leg_from_chain(
        selected_chain_std,
        expiration=test_expiration,
        strike=long_plan["strike_listed"],
        right=RIGHT,
    )

    # Contract-level control prices from known working endpoint.
    short_contract = contract_level_price(
        trade_date=test_trade_date,
        expiration=test_expiration,
        strike=short_plan["strike_listed"],
        right=RIGHT,
    )

    long_contract = contract_level_price(
        trade_date=test_trade_date,
        expiration=test_expiration,
        strike=long_plan["strike_listed"],
        right=RIGHT,
    )

    comparison_rows = []

    for leg_label, plan, chain_row, contract_row in [
        ("short_call_1sd", short_plan, short_chain, short_contract),
        ("long_call_3sd", long_plan, long_chain, long_contract),
    ]:
        comparison_rows.append({
            "candidate_id": test_candidate_id,
            "trade_date": test_trade_date,
            "tenor": int(plan["tenor"]),
            "leg_label": leg_label,
            "expiration": test_expiration,
            "strike": float(plan["strike_listed"]),
            "raw_strike": float(plan["raw_strike"]),
            "chain_bid": chain_row.get("bid", np.nan),
            "chain_ask": chain_row.get("ask", np.nan),
            "chain_mid": chain_row.get("mid", np.nan),
            "chain_timestamp": chain_row.get("timestamp", None),
            "chain_underlying_price": chain_row.get("underlying_price", np.nan),
            "chain_implied_vol": chain_row.get("implied_vol", np.nan),
            "chain_delta": chain_row.get("delta", np.nan),
            "contract_status_code": contract_row["status_code"],
            "contract_rows": contract_row["rows"],
            "contract_bid": contract_row["bid"],
            "contract_ask": contract_row["ask"],
            "contract_mid": contract_row["mid"],
            "contract_timestamp": contract_row["timestamp"],
            "contract_underlying_price": contract_row["underlying_price"],
            "contract_implied_vol": contract_row["implied_vol"],
            "contract_delta": contract_row["delta"],
            "bid_diff_chain_minus_contract": chain_row.get("bid", np.nan) - contract_row["bid"],
            "ask_diff_chain_minus_contract": chain_row.get("ask", np.nan) - contract_row["ask"],
            "mid_diff_chain_minus_contract": chain_row.get("mid", np.nan) - contract_row["mid"],
        })

    chain_leg_comparison = pd.DataFrame(comparison_rows)

    short_cmp = chain_leg_comparison.loc[chain_leg_comparison["leg_label"] == "short_call_1sd"].iloc[0]
    long_cmp = chain_leg_comparison.loc[chain_leg_comparison["leg_label"] == "long_call_3sd"].iloc[0]

    chain_credit_bid_ask = short_cmp["chain_bid"] - long_cmp["chain_ask"]
    chain_credit_mid = short_cmp["chain_mid"] - long_cmp["chain_mid"]
    contract_credit_bid_ask = short_cmp["contract_bid"] - long_cmp["contract_ask"]
    contract_credit_mid = short_cmp["contract_mid"] - long_cmp["contract_mid"]

    chain_spread_summary = pd.DataFrame([{
        "candidate_id": test_candidate_id,
        "trade_date": test_trade_date,
        "tenor": int(short_plan["tenor"]),
        "expiration": test_expiration,
        "short_strike": float(short_plan["strike_listed"]),
        "long_strike": float(long_plan["strike_listed"]),
        "listed_width": float(long_plan["strike_listed"] - short_plan["strike_listed"]),
        "chain_credit_bid_ask": chain_credit_bid_ask,
        "chain_credit_mid": chain_credit_mid,
        "contract_credit_bid_ask": contract_credit_bid_ask,
        "contract_credit_mid": contract_credit_mid,
        "credit_bid_ask_diff_chain_minus_contract": chain_credit_bid_ask - contract_credit_bid_ask,
        "credit_mid_diff_chain_minus_contract": chain_credit_mid - contract_credit_mid,
        "chain_rows": len(selected_chain_std),
        "chain_unique_strikes": selected_chain_std["strike"].nunique(),
        "chain_unique_rights": ",".join(sorted(selected_chain_std["right"].dropna().astype(str).unique())),
    }])

    print()
    print("=" * 100)
    print("Chain-selected leg comparison vs contract-level control")
    print("=" * 100)
    display(chain_leg_comparison)

    print()
    print("=" * 100)
    print("Spread comparison")
    print("=" * 100)
    display(chain_spread_summary)

    print()
    print("=" * 100)
    print("Working chain preview")
    print("=" * 100)
    display(selected_chain_std.head(30))

else:
    print()
    print("No chain endpoint attempt returned both planned legs.")
    print("If all attempts fail, next step is contract-level cache with checkpointing.")

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

attempts_path = AUDIT_DIR / f"02C1_spy_eod_chain_endpoint_attempts_{timestamp}.csv"
comparison_path = AUDIT_DIR / f"02C1_spy_eod_chain_vs_contract_comparison_{timestamp}.csv"
spread_summary_path = AUDIT_DIR / f"02C1_spy_eod_chain_spread_summary_{timestamp}.csv"
chain_preview_path = AUDIT_DIR / f"02C1_spy_eod_chain_preview_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02C1_spy_eod_chain_endpoint_test_manifest_{timestamp}.json"

chain_attempts_df.to_csv(attempts_path, index=False)
chain_leg_comparison.to_csv(comparison_path, index=False)
chain_spread_summary.to_csv(spread_summary_path, index=False)

if selected_attempt is not None:
    selected_chain_std.head(500).to_csv(chain_preview_path, index=False)

    test_chain_cache_path = (
        CHAIN_TEST_DIR /
        f"SPY_EOD_CHAIN_TEST_{trade_yyyymmdd}_{exp_yyyymmdd}.parquet"
    )
    selected_chain_std.to_parquet(test_chain_cache_path, index=False)
else:
    test_chain_cache_path = None
    pd.DataFrame().to_csv(chain_preview_path, index=False)

manifest = {
    "cell": "2C.1",
    "description": "Test ThetaData v3 historical EOD chain endpoint for SPY chain-level pricing",
    "created_at": timestamp,
    "base_url": BASE_URL,
    "chain_endpoint": CHAIN_ENDPOINT,
    "contract_endpoint_control": CONTRACT_ENDPOINT,
    "primary_leg_plan_path": str(PRIMARY_LEG_PLAN_PATH),
    "test_candidate_id": test_candidate_id,
    "test_trade_date": trade_iso,
    "test_expiration": exp_iso,
    "test_short_strike": float(short_plan["strike_listed"]),
    "test_long_strike": float(long_plan["strike_listed"]),
    "chain_endpoint_worked": selected_attempt is not None,
    "selected_attempt_url": selected_attempt["url"] if selected_attempt is not None else None,
    "attempts_path": str(attempts_path),
    "comparison_path": str(comparison_path),
    "spread_summary_path": str(spread_summary_path),
    "chain_preview_path": str(chain_preview_path),
    "test_chain_cache_path": str(test_chain_cache_path) if test_chain_cache_path is not None else None,
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Cell 2C.1 complete")
print("=" * 100)
print(f"Attempts:        {attempts_path}")
print(f"Comparison:      {comparison_path}")
print(f"Spread summary:  {spread_summary_path}")
print(f"Chain preview:   {chain_preview_path}")
print(f"Manifest:        {manifest_path}")

if selected_attempt is not None:
    print()
    print("Result: Chain-level EOD endpoint worked. Next cell should build a chain-level cache plan.")
else:
    print()
    print("Result: Chain-level EOD endpoint did not work for this test. Use contract-level cache with checkpointing.")

Cell 2C.1 — Test SPY historical EOD chain endpoint
Primary leg plan:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet
Base URL:               http://127.0.0.1:25503
Chain endpoint:          /v3/option/history/eod
Contract endpoint ctrl:  /v3/option/history/greeks/eod

Test candidate


,candidate_id,trade_date,tenor,leg_label,expiration_date,raw_strike,strike_listed,strike_rounding_increment,strike_rounding_distance,primary_width_listed
24838,20260701_T30,2026-07-01,30,long_call_3sd,2026-07-31,850.926322,851.0,1.0,0.073678,70.0
12418,20260701_T30,2026-07-01,30,short_call_1sd,2026-07-31,780.815441,781.0,1.0,0.184559,70.0



Probe /v3/option/history/eod chain endpoint


KeyError: 'strike'

In [12]:
# Cell 2C.1 repair — Robust SPY historical EOD chain endpoint probe
# Purpose:
#   Repair the prior 2C.1 crash. The crash happened because one endpoint response
#   did not contain a 'strike' column, but the code tried to select legs anyway.
#
# Scope:
#   - Tiny endpoint probe only
#   - No bulk cache
#   - No parameter sweep

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_LEG_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet"
)

BASE_URL = "http://127.0.0.1:25503"
CHAIN_ENDPOINT = "/v3/option/history/eod"
CONTRACT_ENDPOINT = "/v3/option/history/greeks/eod"

SYMBOL = "SPY"
RIGHT = "C"

PREFERRED_TRADE_DATE = pd.Timestamp("2026-07-01")
PREFERRED_TENOR = 30

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2C.1 repair — Robust SPY historical EOD chain endpoint probe")
print("=" * 100)
print(f"Primary leg plan: {PRIMARY_LEG_PLAN_PATH}")
print(f"Chain endpoint:   {CHAIN_ENDPOINT}")
print()

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        obj = resp.json()
        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)
            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)
            return pd.json_normalize(obj)
        if isinstance(obj, list):
            return pd.DataFrame(obj)
    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def get_df(endpoint, params, timeout=(2, 90)):
    url = BASE_URL.rstrip("/") + endpoint

    try:
        r = session.get(url, params=params, timeout=timeout)
        df = response_to_dataframe(r)
        return {
            "endpoint": endpoint,
            "url": r.url,
            "params": params,
            "status_code": r.status_code,
            "ok": r.ok,
            "rows": len(df),
            "cols": len(df.columns),
            "columns": list(df.columns),
            "text_preview": r.text[:1200],
            "error": None,
            "df": df,
        }
    except Exception as e:
        return {
            "endpoint": endpoint,
            "url": url,
            "params": params,
            "status_code": None,
            "ok": False,
            "rows": 0,
            "cols": 0,
            "columns": [],
            "text_preview": "",
            "error": repr(e),
            "df": pd.DataFrame(),
        }

def first_col(df, candidates):
    if df is None or df.empty:
        return None
    for c in candidates:
        if c in df.columns:
            return c
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None

def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    rename_map = {}
    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "underlying_price": ["underlying_price"],
        "implied_vol": ["implied_vol", "iv"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
    }

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    for col in ["strike", "bid", "ask", "mid", "underlying_price", "implied_vol", "delta", "theta", "vega", "gamma"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out

def has_required_chain_cols(df):
    if df is None or df.empty:
        return False
    needed = ["strike", "right", "bid", "ask"]
    return all(c in df.columns for c in needed)

def pick_leg_from_chain(chain_df, expiration, strike, right="C"):
    if not has_required_chain_cols(chain_df):
        return pd.Series(dtype="object")

    expiration_iso = pd.Timestamp(expiration).strftime("%Y-%m-%d")

    mask = (
        (chain_df["strike"].round(4) == round(float(strike), 4))
        & (chain_df["right"] == right)
    )

    if "expiration" in chain_df.columns:
        mask = mask & (chain_df["expiration"] == expiration_iso)

    leg = chain_df.loc[mask].copy()

    if leg.empty:
        return pd.Series(dtype="object")

    leg = leg.loc[leg["bid"].notna() & leg["ask"].notna()].copy()

    if leg.empty:
        return pd.Series(dtype="object")

    return leg.iloc[-1]

# --------------------------------------------------------------------------------------
# Load one real test candidate
# --------------------------------------------------------------------------------------

leg_plan = pd.read_parquet(PRIMARY_LEG_PLAN_PATH)
leg_plan["trade_date"] = pd.to_datetime(leg_plan["trade_date"], errors="coerce")
leg_plan["expiration_date"] = pd.to_datetime(leg_plan["expiration_date"], errors="coerce")

candidate_ids = leg_plan.loc[
    (leg_plan["trade_date"] == PREFERRED_TRADE_DATE)
    & (leg_plan["tenor"] == PREFERRED_TENOR),
    "candidate_id"
].drop_duplicates().tolist()

if candidate_ids:
    test_candidate_id = candidate_ids[0]
else:
    test_candidate_id = (
        leg_plan.loc[leg_plan["tenor"] == PREFERRED_TENOR]
        .sort_values("trade_date")["candidate_id"]
        .drop_duplicates()
        .iloc[-1]
    )

test_legs = leg_plan.loc[leg_plan["candidate_id"] == test_candidate_id].copy()

short_plan = test_legs.loc[test_legs["leg_label"] == "short_call_1sd"].iloc[0]
long_plan = test_legs.loc[test_legs["leg_label"] == "long_call_3sd"].iloc[0]

trade_date = pd.Timestamp(short_plan["trade_date"])
expiration = pd.Timestamp(short_plan["expiration_date"])

trade_iso = trade_date.strftime("%Y-%m-%d")
trade_yyyymmdd = trade_date.strftime("%Y%m%d")
exp_iso = expiration.strftime("%Y-%m-%d")
exp_yyyymmdd = expiration.strftime("%Y%m%d")

print("=" * 100)
print("Test candidate")
print("=" * 100)
display(
    test_legs[
        [
            "candidate_id",
            "trade_date",
            "tenor",
            "leg_label",
            "expiration_date",
            "raw_strike",
            "strike_listed",
            "primary_width_listed",
        ]
    ]
)

# --------------------------------------------------------------------------------------
# Probe chain endpoint variants
# --------------------------------------------------------------------------------------

param_sets = [
    {
        "symbol": SYMBOL,
        "expiration": exp_iso,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "expiration": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "exp": exp_iso,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "exp": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "root": SYMBOL,
        "exp": exp_yyyymmdd,
        "start_date": trade_yyyymmdd,
        "end_date": trade_yyyymmdd,
        "format": "csv",
    },
    {
        "symbol": SYMBOL,
        "start_date": trade_iso,
        "end_date": trade_iso,
        "format": "csv",
    },
]

attempt_rows = []
successful_chain = pd.DataFrame()
selected_url = None

for params in param_sets:
    result = get_df(CHAIN_ENDPOINT, params)
    raw_df = result["df"]
    std_df = standardize_option_rows(raw_df)

    required_cols = has_required_chain_cols(std_df)

    unique_strikes = int(std_df["strike"].nunique()) if "strike" in std_df.columns and not std_df.empty else 0
    unique_rights = ",".join(sorted(std_df["right"].dropna().astype(str).unique())) if "right" in std_df.columns and not std_df.empty else ""
    unique_expirations = ",".join(sorted(std_df["expiration"].dropna().astype(str).unique())[:20]) if "expiration" in std_df.columns and not std_df.empty else ""

    short_row = pick_leg_from_chain(std_df, expiration, short_plan["strike_listed"], RIGHT)
    long_row = pick_leg_from_chain(std_df, expiration, long_plan["strike_listed"], RIGHT)

    found_short = not short_row.empty
    found_long = not long_row.empty

    attempt_rows.append({
        "params": json.dumps(params),
        "url": result["url"],
        "status_code": result["status_code"],
        "ok": result["ok"],
        "rows": result["rows"],
        "cols": result["cols"],
        "raw_columns": ", ".join(map(str, result["columns"])),
        "std_columns": ", ".join(map(str, std_df.columns)),
        "required_chain_cols": required_cols,
        "unique_strikes": unique_strikes,
        "unique_rights": unique_rights,
        "unique_expirations": unique_expirations,
        "found_short_leg": found_short,
        "found_long_leg": found_long,
        "error": result["error"],
        "text_preview": result["text_preview"],
    })

    if result["ok"] and required_cols and found_short and found_long and successful_chain.empty:
        successful_chain = std_df.copy()
        selected_url = result["url"]

attempts_df = pd.DataFrame(attempt_rows)

print()
print("=" * 100)
print("Endpoint attempts")
print("=" * 100)
display(
    attempts_df[
        [
            "status_code",
            "ok",
            "rows",
            "cols",
            "required_chain_cols",
            "unique_strikes",
            "unique_rights",
            "unique_expirations",
            "found_short_leg",
            "found_long_leg",
            "raw_columns",
            "std_columns",
            "text_preview",
        ]
    ]
)

# --------------------------------------------------------------------------------------
# Result if chain endpoint worked
# --------------------------------------------------------------------------------------

spread_summary = pd.DataFrame()

if not successful_chain.empty:
    short_chain = pick_leg_from_chain(successful_chain, expiration, short_plan["strike_listed"], RIGHT)
    long_chain = pick_leg_from_chain(successful_chain, expiration, long_plan["strike_listed"], RIGHT)

    spread_summary = pd.DataFrame([{
        "candidate_id": test_candidate_id,
        "trade_date": trade_date,
        "tenor": int(short_plan["tenor"]),
        "expiration": expiration,
        "short_strike": float(short_plan["strike_listed"]),
        "long_strike": float(long_plan["strike_listed"]),
        "listed_width": float(long_plan["strike_listed"] - short_plan["strike_listed"]),
        "short_bid": short_chain["bid"],
        "short_ask": short_chain["ask"],
        "short_mid": short_chain.get("mid", np.nan),
        "long_bid": long_chain["bid"],
        "long_ask": long_chain["ask"],
        "long_mid": long_chain.get("mid", np.nan),
        "entry_credit_bid_ask": short_chain["bid"] - long_chain["ask"],
        "entry_credit_mid": short_chain.get("mid", np.nan) - long_chain.get("mid", np.nan),
        "chain_rows": len(successful_chain),
        "chain_unique_strikes": successful_chain["strike"].nunique(),
        "chain_unique_rights": ",".join(sorted(successful_chain["right"].dropna().astype(str).unique())),
        "selected_url": selected_url,
    }])

    print()
    print("=" * 100)
    print("Chain endpoint worked — selected spread")
    print("=" * 100)
    display(spread_summary)

    print()
    print("=" * 100)
    print("Working chain preview")
    print("=" * 100)
    display(successful_chain.head(30))

else:
    print()
    print("=" * 100)
    print("Chain endpoint did NOT return a usable full chain for this test")
    print("=" * 100)
    print("Next step should be contract-level cache with checkpointing using /v3/option/history/greeks/eod.")

# --------------------------------------------------------------------------------------
# Save audit outputs
# --------------------------------------------------------------------------------------

attempts_path = AUDIT_DIR / f"02C1_repair_spy_eod_chain_endpoint_attempts_{timestamp}.csv"
spread_summary_path = AUDIT_DIR / f"02C1_repair_spy_eod_chain_spread_summary_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02C1_repair_spy_eod_chain_endpoint_manifest_{timestamp}.json"

attempts_df.to_csv(attempts_path, index=False)
spread_summary.to_csv(spread_summary_path, index=False)

manifest = {
    "cell": "2C.1 repair",
    "description": "Robust test of ThetaData v3 /v3/option/history/eod as possible chain endpoint",
    "created_at": timestamp,
    "chain_endpoint": CHAIN_ENDPOINT,
    "contract_endpoint_fallback": CONTRACT_ENDPOINT,
    "test_candidate_id": test_candidate_id,
    "trade_date": trade_iso,
    "expiration": exp_iso,
    "short_strike": float(short_plan["strike_listed"]),
    "long_strike": float(long_plan["strike_listed"]),
    "chain_endpoint_worked": not successful_chain.empty,
    "selected_url": selected_url,
    "attempts_path": str(attempts_path),
    "spread_summary_path": str(spread_summary_path),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Attempts:       {attempts_path}")
print(f"Spread summary: {spread_summary_path}")
print(f"Manifest:       {manifest_path}")

if successful_chain.empty:
    print()
    print("Result: Use contract-level cache with checkpointing.")
else:
    print()
    print("Result: Chain-level endpoint works. Build chain-level cache next.")

Cell 2C.1 repair — Robust SPY historical EOD chain endpoint probe
Primary leg plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_spy_candidate_leg_plan_primary_v1.parquet
Chain endpoint:   /v3/option/history/eod

Test candidate


,candidate_id,trade_date,tenor,leg_label,expiration_date,raw_strike,strike_listed,primary_width_listed
12418,20260701_T30,2026-07-01,30,short_call_1sd,2026-07-31,780.815441,781.0,70.0
24838,20260701_T30,2026-07-01,30,long_call_3sd,2026-07-31,850.926322,851.0,70.0



Endpoint attempts


,status_code,ok,rows,cols,required_chain_cols,unique_strikes,unique_rights,unique_expirations,found_short_leg,found_long_leg,raw_columns,std_columns,text_preview
0,200,True,500,20,True,250,"C,P",2026-07-31,True,False,"symbol, expiration, strike, right, created, la...","symbol, expiration, strike, right, created, la...","symbol,expiration,strike,right,created,last_tr..."
1,200,True,500,20,True,250,"C,P",2026-07-31,True,False,"symbol, expiration, strike, right, created, la...","symbol, expiration, strike, right, created, la...","symbol,expiration,strike,right,created,last_tr..."
2,410,False,3,1,False,0,,,False,False,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...
3,410,False,3,1,False,0,,,False,False,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...
4,410,False,4,1,False,0,,,False,False,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...,We have upgraded to API v3. Please use API v3 ...
5,400,False,0,1,False,0,,,False,False,An expiration must be specified,,An expiration must be specified



Chain endpoint did NOT return a usable full chain for this test
Next step should be contract-level cache with checkpointing using /v3/option/history/greeks/eod.

Saved
Attempts:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02C1_repair_spy_eod_chain_endpoint_attempts_20260710_101535.csv
Spread summary: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02C1_repair_spy_eod_chain_spread_summary_20260710_101535.csv
Manifest:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02C1_repair_spy_eod_chain_endpoint_manifest_20260710_101535.json

Result: Use contract-level cache with checkpointing.


In [14]:
# Cell 2D — Build selected-trade pricing universe from Corsi parameter grid
# Purpose:
#   Build the actual selected-trade universe from Corsi call-sleeve parameter rules,
#   then deduplicate the SPY option legs that require ThetaData prices.
#
# Key point:
#   This cell DOES NOT call ThetaData.
#
# Workflow:
#   1. Load all-tenor Corsi raw candidate panel.
#   2. Apply Corsi call parameter grid.
#   3. Apply selection rules: one selected call spread per date per rule.
#   4. Compute unpriced expiry diagnostics using SPY close.
#   5. Deduplicate selected trades.
#   6. Deduplicate option contracts needed for pricing.
#
# Output:
#   - selected trades by rule
#   - unique selected trade universe
#   - selected option-leg request plan
#   - unique selected contract request plan
#   - pre-price rule summary
#
# No pricing / no P&L ranking yet.

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import json
import math

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

SPY_EOD_PATH = PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_eod_prices_v1.parquet"

SYMBOL = "SPY"
RIGHT = "C"
PRICING_ENDPOINT = "/v3/option/history/greeks/eod"

# --------------------------------------------------------------------------------------------------
# Parameter grid
# --------------------------------------------------------------------------------------------------
# Keep 3m and 1y z thresholds equal, per your instruction.
# These are screening parameters; we will narrow after seeing selected-trade counts.
Z_THRESHOLDS = [-0.25, 0.00, 0.10, 0.25, 0.50, 0.75, 1.00]
RSI_FLOORS = [55, 58, 60, 62, 65, 68]
VRP_LOG_FLOORS = [None, 0.00, 0.20, 0.40]

SELECTION_RULES = [
    "highest_z_avg",
    "highest_z_min",
    "highest_vrp_log",
    "shortest_tenor",
    "longest_tenor",
]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2D — Build selected-trade pricing universe from Corsi parameter grid")
print("=" * 100)
print(f"Raw candidate panel: {RAW_CANDIDATE_PANEL_PATH}")
print(f"SPY EOD path:        {SPY_EOD_PATH}")
print(f"Pricing endpoint:    {PRICING_ENDPOINT}")
print()
print(f"Z thresholds:        {Z_THRESHOLDS}")
print(f"RSI floors:          {RSI_FLOORS}")
print(f"VRP log floors:      {VRP_LOG_FLOORS}")
print(f"Selection rules:     {SELECTION_RULES}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def ceil_to_increment(x, increment):
    if pd.isna(x):
        return np.nan
    return float(math.ceil(float(x) / float(increment)) * float(increment))

def next_friday_on_or_after(dt):
    d = pd.Timestamp(dt).normalize()
    days_forward = (4 - d.weekday()) % 7
    return d + pd.Timedelta(days=days_forward)

def clean_float_label(x):
    if x is None or pd.isna(x):
        return "NONE"
    s = f"{float(x):.2f}"
    s = s.replace("-", "m").replace(".", "p")
    return s

def make_rule_id(z_threshold, rsi_floor, vrp_log_floor, selection_rule):
    return (
        f"Z{clean_float_label(z_threshold)}_"
        f"RSI{int(rsi_floor)}_"
        f"VRP{clean_float_label(vrp_log_floor)}_"
        f"SEL_{selection_rule}"
    )

def make_trade_id(trade_date, tenor, expiration_date, short_strike, long_strike):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"T{int(tenor):02d}_"
        f"EXP{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"S{float(short_strike):.1f}_"
        f"L{float(long_strike):.1f}"
    )

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def infer_date_col(df):
    candidates = ["date", "trade_date", "timestamp", "datetime"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not infer date column from columns: {list(df.columns)}")

def infer_close_col(df):
    candidates = ["spy_close", "close", "Close", "adj_close", "Adj Close", "last"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not infer close column from columns: {list(df.columns)}")

def select_one_per_date(pass_df, selection_rule):
    if pass_df.empty:
        return pass_df.copy()

    work = pass_df.copy()

    if selection_rule == "highest_z_avg":
        sort_cols = ["trade_date", "z_avg", "z_min", "corsi_vrp_log", "tenor"]
        ascending = [True, False, False, False, True]

    elif selection_rule == "highest_z_min":
        sort_cols = ["trade_date", "z_min", "z_avg", "corsi_vrp_log", "tenor"]
        ascending = [True, False, False, False, True]

    elif selection_rule == "highest_vrp_log":
        sort_cols = ["trade_date", "corsi_vrp_log", "z_avg", "z_min", "tenor"]
        ascending = [True, False, False, False, True]

    elif selection_rule == "shortest_tenor":
        sort_cols = ["trade_date", "tenor", "z_avg", "corsi_vrp_log"]
        ascending = [True, True, False, False]

    elif selection_rule == "longest_tenor":
        sort_cols = ["trade_date", "tenor", "z_avg", "corsi_vrp_log"]
        ascending = [True, False, False, False]

    else:
        raise ValueError(f"Unknown selection rule: {selection_rule}")

    selected = (
        work
        .sort_values(sort_cols, ascending=ascending)
        .groupby("trade_date", as_index=False)
        .head(1)
        .copy()
    )

    return selected

# --------------------------------------------------------------------------------------------------
# Load Corsi candidate panel
# --------------------------------------------------------------------------------------------------

raw = pd.read_parquet(RAW_CANDIDATE_PANEL_PATH).copy()
raw["trade_date"] = pd.to_datetime(raw["trade_date"], errors="coerce")

required_cols = [
    "trade_date",
    "tenor",
    "candidate_inputs_complete",
    "spy_close",
    "vix_style_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
]

missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required raw candidate columns: {missing}")

base = raw.loc[raw["candidate_inputs_complete"]].copy()
base["tenor"] = base["tenor"].astype(int)

base["z_avg"] = (base["corsi_vrp_z_3m"] + base["corsi_vrp_z_1y"]) / 2.0
base["z_min"] = base[["corsi_vrp_z_3m", "corsi_vrp_z_1y"]].min(axis=1)

# Deterministic expiry / strike convention
base["target_expiration_date"] = base["trade_date"] + pd.to_timedelta(base["tenor"], unit="D")
base["expiration_date"] = base["target_expiration_date"].apply(next_friday_on_or_after)
base["effective_dte"] = (base["expiration_date"] - base["trade_date"]).dt.days
base["expiration_slippage_days"] = (base["expiration_date"] - base["target_expiration_date"]).dt.days

base["short_strike"] = base["short_call_1sd_raw"].apply(lambda x: ceil_to_increment(x, 1.0))
base["long_strike"] = base["long_call_3sd_raw"].apply(lambda x: ceil_to_increment(x, 1.0))
base["listed_width"] = base["long_strike"] - base["short_strike"]

base = base.loc[
    base["trade_date"].notna()
    & base["expiration_date"].notna()
    & base["short_strike"].notna()
    & base["long_strike"].notna()
    & (base["listed_width"] > 0)
].copy()

base["candidate_trade_id"] = base.apply(
    lambda r: make_trade_id(
        r["trade_date"],
        r["tenor"],
        r["expiration_date"],
        r["short_strike"],
        r["long_strike"],
    ),
    axis=1,
)

print("=" * 100)
print("Loaded candidate base")
print("=" * 100)
print(f"Raw rows:              {len(raw):,}")
print(f"Complete base rows:    {len(base):,}")
print(f"Date range:            {base['trade_date'].min().date()} to {base['trade_date'].max().date()}")
print(f"Tenors:                {sorted(base['tenor'].unique())}")
print()

# --------------------------------------------------------------------------------------------------
# Build selected trades across parameter grid
# --------------------------------------------------------------------------------------------------

selected_chunks = []
param_combo_count = 0

for z_threshold in Z_THRESHOLDS:
    for rsi_floor in RSI_FLOORS:
        for vrp_log_floor in VRP_LOG_FLOORS:
            param_combo_count += 1

            mask = (
                (base["corsi_vrp_z_3m"] > z_threshold)
                & (base["corsi_vrp_z_1y"] > z_threshold)
                & (base["rsi14"] > rsi_floor)
            )

            if vrp_log_floor is not None:
                mask = mask & (base["corsi_vrp_log"] > vrp_log_floor)

            pass_df = base.loc[mask].copy()

            if pass_df.empty:
                continue

            for selection_rule in SELECTION_RULES:
                selected = select_one_per_date(pass_df, selection_rule)

                if selected.empty:
                    continue

                selected["z_threshold_shared"] = z_threshold
                selected["rsi_floor"] = rsi_floor
                selected["vrp_log_floor"] = np.nan if vrp_log_floor is None else vrp_log_floor
                selected["vrp_log_floor_label"] = "NONE" if vrp_log_floor is None else f"{vrp_log_floor:.2f}"
                selected["selection_rule"] = selection_rule
                selected["rule_id"] = make_rule_id(
                    z_threshold=z_threshold,
                    rsi_floor=rsi_floor,
                    vrp_log_floor=vrp_log_floor,
                    selection_rule=selection_rule,
                )

                selected_chunks.append(selected)

if selected_chunks:
    selected_trades = pd.concat(selected_chunks, ignore_index=True)
else:
    selected_trades = pd.DataFrame()

print("=" * 100)
print("Selected-trade grid built")
print("=" * 100)
print(f"Parameter combos tested:       {param_combo_count:,}")
print(f"Selection rules tested:        {len(SELECTION_RULES):,}")
print(f"Selected rows by rule:         {len(selected_trades):,}")
print()

if selected_trades.empty:
    raise ValueError("No selected trades produced. Grid is too strict or input panel has no qualifying rows.")

# --------------------------------------------------------------------------------------------------
# Attach expiry SPY close for unpriced diagnostics
# --------------------------------------------------------------------------------------------------

spy = pd.read_parquet(SPY_EOD_PATH).copy()
spy_date_col = infer_date_col(spy)
spy_close_col = infer_close_col(spy)

spy_prices = spy[[spy_date_col, spy_close_col]].copy()
spy_prices.columns = ["spy_price_date", "expiry_spy_close"]
spy_prices["spy_price_date"] = pd.to_datetime(spy_prices["spy_price_date"], errors="coerce")
spy_prices["expiry_spy_close"] = pd.to_numeric(spy_prices["expiry_spy_close"], errors="coerce")

spy_prices = (
    spy_prices
    .dropna(subset=["spy_price_date", "expiry_spy_close"])
    .sort_values("spy_price_date")
    .drop_duplicates("spy_price_date", keep="last")
    .reset_index(drop=True)
)

unique_expirations = (
    selected_trades[["expiration_date"]]
    .drop_duplicates()
    .sort_values("expiration_date")
    .reset_index(drop=True)
)

expiry_map = pd.merge_asof(
    unique_expirations,
    spy_prices,
    left_on="expiration_date",
    right_on="spy_price_date",
    direction="backward",
)

expiry_map["expiry_price_date_exact"] = (
    expiry_map["expiration_date"] == expiry_map["spy_price_date"]
)

selected_trades = selected_trades.merge(
    expiry_map,
    on="expiration_date",
    how="left",
)

selected_trades["short_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["short_strike"],
    0.0,
)

selected_trades["long_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["long_strike"],
    0.0,
)

selected_trades["terminal_spread_intrinsic_before_premium"] = (
    selected_trades["short_call_terminal_intrinsic"]
    - selected_trades["long_call_terminal_intrinsic"]
)

selected_trades["terminal_intrinsic_pct_width"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"]
    / selected_trades["listed_width"]
)

selected_trades["expired_short_call_otm"] = (
    selected_trades["expiry_spy_close"] <= selected_trades["short_strike"]
)

selected_trades["expired_spread_zero_intrinsic"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"] <= 0
)

selected_trades["trade_year"] = selected_trades["trade_date"].dt.year

# --------------------------------------------------------------------------------------------------
# Build unique selected trade universe
# --------------------------------------------------------------------------------------------------

selected_trades["selected_trade_id"] = selected_trades["candidate_trade_id"]

rule_usage = (
    selected_trades
    .groupby("selected_trade_id")
    .agg(
        selected_by_rule_count=("rule_id", "nunique"),
        selection_rules=("selection_rule", lambda x: ",".join(sorted(set(map(str, x))))),
        min_z_threshold=("z_threshold_shared", "min"),
        max_z_threshold=("z_threshold_shared", "max"),
        min_rsi_floor=("rsi_floor", "min"),
        max_rsi_floor=("rsi_floor", "max"),
        vrp_log_floor_labels=("vrp_log_floor_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

trade_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "vix_style_vol_pct",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "z_avg",
    "z_min",
    "rsi14",
    "target_expiration_date",
    "expiration_date",
    "effective_dte",
    "expiration_slippage_days",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "short_strike",
    "long_strike",
    "listed_width",
    "spy_price_date",
    "expiry_spy_close",
    "expiry_price_date_exact",
    "short_call_terminal_intrinsic",
    "long_call_terminal_intrinsic",
    "terminal_spread_intrinsic_before_premium",
    "terminal_intrinsic_pct_width",
    "expired_short_call_otm",
    "expired_spread_zero_intrinsic",
]

trade_cols = [c for c in trade_cols if c in selected_trades.columns]

unique_selected_trades = (
    selected_trades[trade_cols]
    .drop_duplicates("selected_trade_id")
    .merge(rule_usage, on="selected_trade_id", how="left")
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Build selected option leg request plan
# --------------------------------------------------------------------------------------------------

short_legs = unique_selected_trades.copy()
short_legs["leg_label"] = "short_call_1sd"
short_legs["leg_side"] = "sell"
short_legs["price_field_for_entry"] = "bid"
short_legs["symbol"] = SYMBOL
short_legs["right"] = RIGHT
short_legs["strike_listed"] = short_legs["short_strike"]

long_legs = unique_selected_trades.copy()
long_legs["leg_label"] = "long_call_3sd"
long_legs["leg_side"] = "buy"
long_legs["price_field_for_entry"] = "ask"
long_legs["symbol"] = SYMBOL
long_legs["right"] = RIGHT
long_legs["strike_listed"] = long_legs["long_strike"]

selected_leg_plan = pd.concat([short_legs, long_legs], ignore_index=True)

selected_leg_plan["trade_date_str"] = selected_leg_plan["trade_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_str"] = selected_leg_plan["expiration_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_yyyymmdd"] = selected_leg_plan["expiration_date"].dt.strftime("%Y%m%d")
selected_leg_plan["endpoint"] = PRICING_ENDPOINT

selected_leg_plan["contract_request_id"] = selected_leg_plan.apply(
    lambda r: make_contract_request_id(
        r["symbol"],
        r["trade_date"],
        r["expiration_date"],
        r["strike_listed"],
        r["right"],
    ),
    axis=1,
)

contract_usage = (
    selected_leg_plan
    .groupby("contract_request_id")
    .agg(
        mapped_trade_legs=("selected_trade_id", "size"),
        mapped_unique_trades=("selected_trade_id", "nunique"),
        mapped_tenors=("tenor", lambda x: ",".join(map(str, sorted(set(x.astype(int)))))),
        mapped_leg_labels=("leg_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

unique_contract_request_plan = (
    selected_leg_plan[
        [
            "contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "strike_listed",
            "endpoint",
        ]
    ]
    .drop_duplicates()
    .merge(contract_usage, on="contract_request_id", how="left")
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Pre-price rule summaries
# --------------------------------------------------------------------------------------------------

def summarize_rule(g):
    g = g.sort_values("trade_date").copy()

    intrinsic = g["terminal_spread_intrinsic_before_premium"].fillna(0.0)
    intrinsic_pct = g["terminal_intrinsic_pct_width"].fillna(0.0)

    rolling_20_intrinsic = intrinsic.rolling(20, min_periods=1).sum()
    rolling_20_intrinsic_pct = intrinsic_pct.rolling(20, min_periods=1).sum()

    year_counts = g.groupby("trade_year").size()

    return pd.Series({
        "selected_trades": len(g),
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "years_traded": ",".join(map(str, sorted(g["trade_year"].dropna().astype(int).unique()))),
        "n_years_traded": g["trade_year"].nunique(),
        "median_trades_per_year": year_counts.median() if not year_counts.empty else np.nan,
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "avg_effective_dte": g["effective_dte"].mean(),
        "avg_listed_width": g["listed_width"].mean(),
        "median_listed_width": g["listed_width"].median(),
        "short_call_otm_rate": g["expired_short_call_otm"].mean(),
        "zero_intrinsic_rate": g["expired_spread_zero_intrinsic"].mean(),
        "avg_terminal_intrinsic": intrinsic.mean(),
        "median_terminal_intrinsic": intrinsic.median(),
        "p95_terminal_intrinsic": intrinsic.quantile(0.95),
        "worst_terminal_intrinsic": intrinsic.max(),
        "avg_terminal_intrinsic_pct_width": intrinsic_pct.mean(),
        "p95_terminal_intrinsic_pct_width": intrinsic_pct.quantile(0.95),
        "worst_terminal_intrinsic_pct_width": intrinsic_pct.max(),
        "worst_20_trade_intrinsic_sum": rolling_20_intrinsic.max(),
        "worst_20_trade_intrinsic_pct_width_sum": rolling_20_intrinsic_pct.max(),
        "expiry_price_exact_rate": g["expiry_price_date_exact"].mean(),
    })

rule_summary = (
    selected_trades
    .groupby(
        [
            "rule_id",
            "z_threshold_shared",
            "rsi_floor",
            "vrp_log_floor_label",
            "selection_rule",
        ],
        dropna=False,
    )
    .apply(summarize_rule, include_groups=False)
    .reset_index()
)

rule_year_summary = (
    selected_trades
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_trades=("selected_trade_id", "size"),
        short_call_otm_rate=("expired_short_call_otm", "mean"),
        zero_intrinsic_rate=("expired_spread_zero_intrinsic", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
    .sort_values(["rule_id", "trade_year"])
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet"
)

unique_selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_v1.parquet"
)

selected_leg_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_request_plan_v1.parquet"
)

unique_contract_request_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_v1.parquet"
)

rule_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_preprice_summary_v1.parquet"
)

rule_year_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_preprice_summary_v1.parquet"
)

selected_trades.to_parquet(selected_trades_path, index=False)
unique_selected_trades.to_parquet(unique_selected_trades_path, index=False)
selected_leg_plan.to_parquet(selected_leg_plan_path, index=False)
unique_contract_request_plan.to_parquet(unique_contract_request_plan_path, index=False)
rule_summary.to_parquet(rule_summary_path, index=False)
rule_year_summary.to_parquet(rule_year_summary_path, index=False)

# Audit CSVs
selected_trades_csv = AUDIT_DIR / f"02D_rule_selected_trades_unpriced_{timestamp}.csv"
unique_selected_trades_csv = AUDIT_DIR / f"02D_unique_selected_trade_universe_{timestamp}.csv"
selected_leg_plan_csv = AUDIT_DIR / f"02D_selected_leg_request_plan_{timestamp}.csv"
unique_contract_request_plan_csv = AUDIT_DIR / f"02D_selected_unique_contract_request_plan_{timestamp}.csv"
rule_summary_csv = AUDIT_DIR / f"02D_rule_preprice_summary_{timestamp}.csv"
rule_year_summary_csv = AUDIT_DIR / f"02D_rule_year_preprice_summary_{timestamp}.csv"

selected_trades.to_csv(selected_trades_csv, index=False)
unique_selected_trades.to_csv(unique_selected_trades_csv, index=False)
selected_leg_plan.to_csv(selected_leg_plan_csv, index=False)
unique_contract_request_plan.to_csv(unique_contract_request_plan_csv, index=False)
rule_summary.to_csv(rule_summary_csv, index=False)
rule_year_summary.to_csv(rule_year_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02D_selected_trade_pricing_universe_manifest_{timestamp}.json"

manifest = {
    "cell": "2D",
    "description": "Build selected-trade pricing universe from Corsi call parameter grid; no ThetaData calls",
    "created_at": timestamp,
    "raw_candidate_panel_path": str(RAW_CANDIDATE_PANEL_PATH),
    "spy_eod_path": str(SPY_EOD_PATH),
    "pricing_endpoint_for_next_cell": PRICING_ENDPOINT,
    "z_thresholds": Z_THRESHOLDS,
    "rsi_floors": RSI_FLOORS,
    "vrp_log_floors": ["NONE" if x is None else x for x in VRP_LOG_FLOORS],
    "selection_rules": SELECTION_RULES,
    "counts": {
        "base_candidate_rows": int(len(base)),
        "rule_selected_rows": int(len(selected_trades)),
        "unique_selected_trades": int(len(unique_selected_trades)),
        "selected_leg_rows": int(len(selected_leg_plan)),
        "unique_contract_requests": int(len(unique_contract_request_plan)),
        "rule_count": int(rule_summary["rule_id"].nunique()),
    },
    "outputs": {
        "selected_trades_path": str(selected_trades_path),
        "unique_selected_trades_path": str(unique_selected_trades_path),
        "selected_leg_plan_path": str(selected_leg_plan_path),
        "unique_contract_request_plan_path": str(unique_contract_request_plan_path),
        "rule_summary_path": str(rule_summary_path),
        "rule_year_summary_path": str(rule_year_summary_path),
    },
    "audit_outputs": {
        "selected_trades_csv": str(selected_trades_csv),
        "unique_selected_trades_csv": str(unique_selected_trades_csv),
        "selected_leg_plan_csv": str(selected_leg_plan_csv),
        "unique_contract_request_plan_csv": str(unique_contract_request_plan_csv),
        "rule_summary_csv": str(rule_summary_csv),
        "rule_year_summary_csv": str(rule_year_summary_csv),
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------------------

summary_counts = pd.DataFrame([{
    "base_candidate_rows": len(base),
    "rule_selected_rows": len(selected_trades),
    "unique_selected_trades": len(unique_selected_trades),
    "selected_leg_rows": len(selected_leg_plan),
    "unique_contract_requests_to_price": len(unique_contract_request_plan),
    "rule_count": rule_summary["rule_id"].nunique(),
    "date_min": selected_trades["trade_date"].min().date(),
    "date_max": selected_trades["trade_date"].max().date(),
}])

print()
print("=" * 100)
print("Selected pricing universe summary")
print("=" * 100)
display(summary_counts)

print()
print("=" * 100)
print("Rule summary preview")
print("=" * 100)

preview = (
    rule_summary
    .sort_values(
        [
            "short_call_otm_rate",
            "worst_terminal_intrinsic_pct_width",
            "selected_trades",
        ],
        ascending=[False, True, False],
    )
    .head(30)
)

display(
    preview[
        [
            "rule_id",
            "selected_trades",
            "selection_rule",
            "z_threshold_shared",
            "rsi_floor",
            "vrp_log_floor_label",
            "tenors_selected",
            "short_call_otm_rate",
            "zero_intrinsic_rate",
            "avg_terminal_intrinsic",
            "worst_terminal_intrinsic",
            "worst_terminal_intrinsic_pct_width",
            "worst_20_trade_intrinsic_pct_width_sum",
        ]
    ]
)

print()
print("=" * 100)
print("Unique selected trade universe preview")
print("=" * 100)
display(
    unique_selected_trades[
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "long_strike",
            "listed_width",
            "expiry_spy_close",
            "expired_short_call_otm",
            "terminal_spread_intrinsic_before_premium",
            "selected_by_rule_count",
            "selection_rules",
        ]
    ].head(30)
)

print()
print("=" * 100)
print("Unique contract request plan preview")
print("=" * 100)
display(
    unique_contract_request_plan[
        [
            "contract_request_id",
            "symbol",
            "trade_date_str",
            "expiration_str",
            "strike_listed",
            "right",
            "mapped_unique_trades",
            "mapped_tenors",
            "mapped_leg_labels",
        ]
    ].head(30)
)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Selected trades by rule:          {selected_trades_path}")
print(f"Unique selected trade universe:   {unique_selected_trades_path}")
print(f"Selected leg request plan:        {selected_leg_plan_path}")
print(f"Unique contract request plan:     {unique_contract_request_plan_path}")
print(f"Rule pre-price summary:           {rule_summary_path}")
print(f"Rule-year pre-price summary:      {rule_year_summary_path}")
print(f"Manifest:                         {manifest_path}")

print()
print("Result: selected-trade pricing universe built. Next step is to price only this selected unique contract plan.")

Cell 2D — Build selected-trade pricing universe from Corsi parameter grid
Raw candidate panel: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet
SPY EOD path:        C:\Users\patri\vrp_project\data\processed\market_data\spy_eod_prices_v1.parquet
Pricing endpoint:    /v3/option/history/greeks/eod

Z thresholds:        [-0.25, 0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
RSI floors:          [55, 58, 60, 62, 65, 68]
VRP log floors:      [None, 0.0, 0.2, 0.4]
Selection rules:     ['highest_z_avg', 'highest_z_min', 'highest_vrp_log', 'shortest_tenor', 'longest_tenor']

Loaded candidate base
Raw rows:              14,715
Complete base rows:    12,420
Date range:            2020-12-31 to 2026-07-01
Tenors:                [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Selected-trade grid built
Parameter combos tested:       168
Selection rules tested:     

,base_candidate_rows,rule_selected_rows,unique_selected_trades,selected_leg_rows,unique_contract_requests_to_price,rule_count,date_min,date_max
0,12420,155550,1741,3482,3481,840,2020-12-31,2026-06-02



Rule summary preview


,rule_id,selected_trades,selection_rule,z_threshold_shared,rsi_floor,vrp_log_floor_label,tenors_selected,short_call_otm_rate,zero_intrinsic_rate,avg_terminal_intrinsic,worst_terminal_intrinsic,worst_terminal_intrinsic_pct_width,worst_20_trade_intrinsic_pct_width_sum
601,Z1p00_RSI55_VRP0p00_SEL_highest_z_avg,107,highest_z_avg,1.0,55,0.00,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.136667,0.413519
606,Z1p00_RSI55_VRP0p20_SEL_highest_z_avg,107,highest_z_avg,1.0,55,0.20,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.136667,0.413519
616,Z1p00_RSI55_VRPNONE_SEL_highest_z_avg,107,highest_z_avg,1.0,55,NONE,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.136667,0.413519
611,Z1p00_RSI55_VRP0p40_SEL_highest_z_avg,104,highest_z_avg,1.0,55,0.40,"9,12,15,18,21,24,27,30,33",0.951923,0.951923,0.129519,4.10,0.136667,0.413519
661,Z1p00_RSI62_VRP0p00_SEL_highest_z_avg,76,highest_z_avg,1.0,62,0.00,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852
662,Z1p00_RSI62_VRP0p00_SEL_highest_z_min,76,highest_z_min,1.0,62,0.00,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852
666,Z1p00_RSI62_VRP0p20_SEL_highest_z_avg,76,highest_z_avg,1.0,62,0.20,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852
667,Z1p00_RSI62_VRP0p20_SEL_highest_z_min,76,highest_z_min,1.0,62,0.20,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852
676,Z1p00_RSI62_VRPNONE_SEL_highest_z_avg,76,highest_z_avg,1.0,62,NONE,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852
677,Z1p00_RSI62_VRPNONE_SEL_highest_z_min,76,highest_z_min,1.0,62,NONE,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.095652,0.276852



Unique selected trade universe preview


,selected_trade_id,trade_date,tenor,expiration_date,short_strike,long_strike,listed_width,expiry_spy_close,expired_short_call_otm,terminal_spread_intrinsic_before_premium,selected_by_rule_count,selection_rules
0,SPY_CALL_20201231_T09_EXP20210115_S386.0_L409.0,2020-12-31,9,2021-01-15,386.0,409.0,23.0,375.70,True,0.0,600,"highest_vrp_log,highest_z_avg,highest_z_min,lo..."
1,SPY_CALL_20201231_T27_EXP20210129_S397.0_L442.0,2020-12-31,27,2021-01-29,397.0,442.0,45.0,370.07,True,0.0,20,longest_tenor
2,SPY_CALL_20201231_T33_EXP20210205_S400.0_L451.0,2020-12-31,33,2021-02-05,400.0,451.0,51.0,387.71,True,0.0,80,longest_tenor
3,SPY_CALL_20210106_T09_EXP20210115_S389.0_L419.0,2021-01-06,9,2021-01-15,389.0,419.0,30.0,375.70,True,0.0,60,"highest_vrp_log,highest_z_avg,highest_z_min,lo..."
4,SPY_CALL_20210111_T09_EXP20210122_S392.0_L418.0,2021-01-11,9,2021-01-22,392.0,418.0,26.0,382.88,True,0.0,80,"highest_vrp_log,highest_z_avg,highest_z_min,lo..."
5,SPY_CALL_20210119_T09_EXP20210129_S392.0_L417.0,2021-01-19,9,2021-01-29,392.0,417.0,25.0,370.07,True,0.0,156,"highest_vrp_log,highest_z_avg,highest_z_min,lo..."
6,SPY_CALL_20210119_T12_EXP20210205_S394.0_L424.0,2021-01-19,12,2021-02-05,394.0,424.0,30.0,387.71,True,0.0,24,"highest_vrp_log,longest_tenor"
7,SPY_CALL_20210125_T09_EXP20210205_S397.0_L423.0,2021-01-25,9,2021-02-05,397.0,423.0,26.0,387.71,True,0.0,240,"highest_z_avg,highest_z_min,shortest_tenor"
8,SPY_CALL_20210125_T12_EXP20210212_S400.0_L429.0,2021-01-25,12,2021-02-12,400.0,429.0,29.0,392.64,True,0.0,100,"highest_vrp_log,longest_tenor"
9,SPY_CALL_20210125_T15_EXP20210212_S402.0_L436.0,2021-01-25,15,2021-02-12,402.0,436.0,34.0,392.64,True,0.0,40,longest_tenor



Unique contract request plan preview


,contract_request_id,symbol,trade_date_str,expiration_str,strike_listed,right,mapped_unique_trades,mapped_tenors,mapped_leg_labels
0,SPY_20201231_20210115_386.0_C,SPY,2020-12-31,2021-01-15,386.0,C,1,9,short_call_1sd
1,SPY_20201231_20210115_409.0_C,SPY,2020-12-31,2021-01-15,409.0,C,1,9,long_call_3sd
2,SPY_20201231_20210129_397.0_C,SPY,2020-12-31,2021-01-29,397.0,C,1,27,short_call_1sd
3,SPY_20201231_20210129_442.0_C,SPY,2020-12-31,2021-01-29,442.0,C,1,27,long_call_3sd
4,SPY_20201231_20210205_400.0_C,SPY,2020-12-31,2021-02-05,400.0,C,1,33,short_call_1sd
5,SPY_20201231_20210205_451.0_C,SPY,2020-12-31,2021-02-05,451.0,C,1,33,long_call_3sd
6,SPY_20210106_20210115_389.0_C,SPY,2021-01-06,2021-01-15,389.0,C,1,9,short_call_1sd
7,SPY_20210106_20210115_419.0_C,SPY,2021-01-06,2021-01-15,419.0,C,1,9,long_call_3sd
8,SPY_20210111_20210122_392.0_C,SPY,2021-01-11,2021-01-22,392.0,C,1,9,short_call_1sd
9,SPY_20210111_20210122_418.0_C,SPY,2021-01-11,2021-01-22,418.0,C,1,9,long_call_3sd



Saved
Selected trades by rule:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet
Unique selected trade universe:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_v1.parquet
Selected leg request plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_v1.parquet
Unique contract request plan:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_v1.parquet
Rule pre-price summary:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_preprice_summary_v1.parquet
Rule-year pre-price summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_preprice_summary_v1.parquet
Manifest:                        

In [15]:
# Cell 2E — Price selected unique contract request plan with checkpointing
# Purpose:
#   Price only the selected-trade contract universe built in Cell 2D.
#
# Input:
#   call_sleeve_corsi_call_selected_unique_contract_request_plan_v1.parquet
#
# Endpoint:
#   /v3/option/history/greeks/eod
#
# Output:
#   call_sleeve_corsi_call_selected_contract_price_cache_v1.parquet
#
# Scope:
#   - Selected contracts only
#   - No fallback strikes yet
#   - No spread P&L join yet
#   - No parameter ranking yet
#
# First run:
#   MAX_REQUESTS_THIS_RUN = 250
#
# After the first batch looks clean:
#   set MAX_REQUESTS_THIS_RUN = None and rerun to finish the cache.

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json
import time
import traceback

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_UNIQUE_CONTRACT_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_v1.parquet"
)

PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_v1.parquet"
)

PRICE_CACHE_CSV_SNAPSHOT_PATH = (
    AUDIT_DIR / "02E_call_sleeve_corsi_call_selected_contract_price_cache_latest.csv"
)

BASE_URL = "http://127.0.0.1:25503"
ENDPOINT = "/v3/option/history/greeks/eod"

# Runtime controls
CHECKPOINT_EVERY = 100
PROGRESS_EVERY = 25
REQUEST_SLEEP_SECONDS = 0.02

# First validation run. Set to None after the first batch looks clean.
MAX_REQUESTS_THIS_RUN = 250

# Usually False. If True, cached rows with price_found=False are retried.
RETRY_FAILED_OR_MISSING = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_AUDIT_PATH = AUDIT_DIR / f"02E_selected_contract_price_cache_run_audit_{timestamp}.csv"
RUN_MANIFEST_PATH = AUDIT_DIR / f"02E_selected_contract_price_cache_manifest_{timestamp}.json"

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2E — Price selected unique contract request plan")
print("=" * 100)
print(f"Selected unique contract plan: {SELECTED_UNIQUE_CONTRACT_PLAN_PATH}")
print(f"Price cache path:              {PRICE_CACHE_PATH}")
print(f"ThetaData base URL:            {BASE_URL}")
print(f"Endpoint:                      {ENDPOINT}")
print(f"Checkpoint every:              {CHECKPOINT_EVERY:,} attempts")
print(f"Max requests this run:         {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed/missing:          {RETRY_FAILED_OR_MISSING}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None

                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break

                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)

                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None

def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "created": ["created"],
        "last_trade": ["last_trade"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "bid_size": ["bid_size"],
        "ask_size": ["ask_size"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close"],
        "volume": ["volume"],
        "count": ["count"],
        "underlying_price": ["underlying_price"],
        "underlying_timestamp": ["underlying_timestamp"],
        "implied_vol": ["implied_vol", "iv"],
        "iv_error": ["iv_error"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
        "rho": ["rho"],
        "epsilon": ["epsilon"],
        "lambda": ["lambda"],
        "omega": ["omega"],
        "vanna": ["vanna"],
        "charm": ["charm"],
        "vomma": ["vomma"],
        "veta": ["veta"],
        "vera": ["vera"],
        "speed": ["speed"],
        "zomma": ["zomma"],
        "color": ["color"],
        "ultima": ["ultima"],
        "d1": ["d1"],
        "d2": ["d2"],
        "dual_delta": ["dual_delta"],
        "dual_gamma": ["dual_gamma"],
    }

    rename_map = {}

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    numeric_cols = [
        "strike", "bid", "ask", "mid", "bid_size", "ask_size",
        "open", "high", "low", "close", "volume", "count",
        "underlying_price", "implied_vol", "iv_error",
        "delta", "theta", "vega", "gamma", "rho",
        "epsilon", "lambda", "omega", "vanna", "charm",
        "vomma", "veta", "vera", "speed", "zomma",
        "color", "ultima", "d1", "d2", "dual_delta", "dual_gamma",
    ]

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out

def atomic_write_parquet(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def build_empty_cache():
    return pd.DataFrame(columns=[
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "endpoint",
        "status_code",
        "http_ok",
        "response_rows",
        "response_cols",
        "response_columns",
        "price_found",
        "price_status",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "timestamp",
        "created",
        "last_trade",
        "underlying_price",
        "underlying_timestamp",
        "implied_vol",
        "iv_error",
        "delta",
        "theta",
        "vega",
        "gamma",
        "rho",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "count",
        "raw_text_preview",
        "error",
        "request_url",
        "pulled_at",
    ])

def get_completed_ids(cache_df):
    if cache_df.empty:
        return set()

    if RETRY_FAILED_OR_MISSING:
        if "price_found" in cache_df.columns:
            return set(cache_df.loc[cache_df["price_found"] == True, "contract_request_id"].astype(str))
        return set()

    return set(cache_df["contract_request_id"].dropna().astype(str))

def price_contract(plan_row):
    symbol = str(plan_row["symbol"]).upper()
    right = normalize_right(plan_row["right"])
    strike = float(plan_row["strike_listed"])
    trade_date_str = str(plan_row["trade_date_str"])
    expiration_str = str(plan_row["expiration_str"])

    params = {
        "symbol": symbol,
        "right": right,
        "strike": strike,
        "expiration": expiration_str,
        "start_date": trade_date_str,
        "end_date": trade_date_str,
        "format": "csv",
    }

    url = BASE_URL.rstrip("/") + ENDPOINT
    pulled_at = datetime.now().isoformat(timespec="seconds")

    base_result = {
        "contract_request_id": str(plan_row["contract_request_id"]),
        "symbol": symbol,
        "right": right,
        "trade_date": pd.Timestamp(plan_row["trade_date"]),
        "trade_date_str": trade_date_str,
        "expiration_date": pd.Timestamp(plan_row["expiration_date"]),
        "expiration_str": expiration_str,
        "expiration_yyyymmdd": str(plan_row["expiration_yyyymmdd"]),
        "strike_listed": strike,
        "endpoint": ENDPOINT,
        "status_code": np.nan,
        "http_ok": False,
        "response_rows": 0,
        "response_cols": 0,
        "response_columns": "",
        "price_found": False,
        "price_status": "not_attempted",
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_size": np.nan,
        "ask_size": np.nan,
        "timestamp": None,
        "created": None,
        "last_trade": None,
        "underlying_price": np.nan,
        "underlying_timestamp": None,
        "implied_vol": np.nan,
        "iv_error": np.nan,
        "delta": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "gamma": np.nan,
        "rho": np.nan,
        "open": np.nan,
        "high": np.nan,
        "low": np.nan,
        "close": np.nan,
        "volume": np.nan,
        "count": np.nan,
        "raw_text_preview": "",
        "error": None,
        "request_url": "",
        "pulled_at": pulled_at,
    }

    try:
        resp = session.get(url, params=params, timeout=(2, 45))

        base_result["status_code"] = resp.status_code
        base_result["http_ok"] = bool(resp.ok)
        base_result["raw_text_preview"] = resp.text[:500] if hasattr(resp, "text") else ""
        base_result["request_url"] = resp.url

        raw_df = response_to_dataframe(resp)
        std_df = standardize_option_rows(raw_df)

        base_result["response_rows"] = int(len(raw_df))
        base_result["response_cols"] = int(len(raw_df.columns))
        base_result["response_columns"] = ", ".join(map(str, raw_df.columns))

        if not resp.ok:
            base_result["price_status"] = f"http_{resp.status_code}"
            return base_result

        if std_df.empty:
            base_result["price_status"] = "empty_response"
            return base_result

        if not {"bid", "ask"}.issubset(std_df.columns):
            base_result["price_status"] = "missing_bid_ask_columns"
            return base_result

        valid = std_df.loc[
            std_df["bid"].notna()
            & std_df["ask"].notna()
        ].copy()

        if valid.empty:
            base_result["price_status"] = "no_valid_bid_ask"
            return base_result

        # Prefer exact requested contract when identifiers exist.
        if {"strike", "right"}.issubset(valid.columns):
            exact_mask = (
                valid["strike"].round(4).eq(round(strike, 4))
                & valid["right"].eq(right)
            )

            if "expiration" in valid.columns:
                exact_mask = exact_mask & valid["expiration"].eq(expiration_str)

            exact = valid.loc[exact_mask].copy()

            if not exact.empty:
                valid = exact

        px = valid.iloc[-1]

        base_result["price_found"] = True
        base_result["price_status"] = "priced"

        for col in [
            "bid", "ask", "mid", "bid_size", "ask_size",
            "timestamp", "created", "last_trade",
            "underlying_price", "underlying_timestamp",
            "implied_vol", "iv_error",
            "delta", "theta", "vega", "gamma", "rho",
            "open", "high", "low", "close", "volume", "count",
        ]:
            if col in valid.columns:
                base_result[col] = px.get(col, base_result[col])

        return base_result

    except Exception as e:
        base_result["price_status"] = "exception"
        base_result["error"] = repr(e)
        base_result["raw_text_preview"] = traceback.format_exc()[:500]
        return base_result

def save_checkpoint(cache_df, run_audit_df=None):
    cache_df = cache_df.copy()

    if not cache_df.empty:
        cache_df["contract_request_id"] = cache_df["contract_request_id"].astype(str)
        cache_df["pulled_at"] = cache_df["pulled_at"].astype(str)

        cache_df = (
            cache_df
            .sort_values(["contract_request_id", "pulled_at"])
            .drop_duplicates("contract_request_id", keep="last")
            .sort_values(["trade_date_str", "expiration_str", "strike_listed", "right"])
            .reset_index(drop=True)
        )

    atomic_write_parquet(cache_df, PRICE_CACHE_PATH)
    atomic_write_csv(cache_df, PRICE_CACHE_CSV_SNAPSHOT_PATH)

    if run_audit_df is not None:
        atomic_write_csv(run_audit_df, RUN_AUDIT_PATH)

    return cache_df

# --------------------------------------------------------------------------------------------------
# Load selected contract plan and existing cache
# --------------------------------------------------------------------------------------------------

plan = pd.read_parquet(SELECTED_UNIQUE_CONTRACT_PLAN_PATH).copy()

required_plan_cols = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "trade_date_str",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "strike_listed",
]

missing_plan_cols = [c for c in required_plan_cols if c not in plan.columns]
if missing_plan_cols:
    raise ValueError(f"Missing required columns from selected unique contract plan: {missing_plan_cols}")

plan["contract_request_id"] = plan["contract_request_id"].astype(str)
plan["trade_date"] = pd.to_datetime(plan["trade_date"], errors="coerce")
plan["expiration_date"] = pd.to_datetime(plan["expiration_date"], errors="coerce")
plan["strike_listed"] = pd.to_numeric(plan["strike_listed"], errors="coerce")

plan = (
    plan
    .dropna(subset=["trade_date", "expiration_date", "strike_listed"])
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

if PRICE_CACHE_PATH.exists():
    cache = pd.read_parquet(PRICE_CACHE_PATH).copy()

    if "contract_request_id" in cache.columns:
        cache["contract_request_id"] = cache["contract_request_id"].astype(str)
    else:
        cache = build_empty_cache()
else:
    cache = build_empty_cache()

completed_ids = get_completed_ids(cache)

remaining_full = plan.loc[
    ~plan["contract_request_id"].astype(str).isin(completed_ids)
].copy()

remaining = remaining_full.copy()

if MAX_REQUESTS_THIS_RUN is not None:
    remaining = remaining.head(int(MAX_REQUESTS_THIS_RUN)).copy()

print("=" * 100)
print("Loaded selected contract plan/cache")
print("=" * 100)
print(f"Plan rows:                 {len(plan):,}")
print(f"Existing cache rows:       {len(cache):,}")
print(f"Completed/skipped IDs:     {len(completed_ids):,}")
print(f"Remaining full:            {len(remaining_full):,}")
print(f"Remaining this run:        {len(remaining):,}")
print()

if not cache.empty:
    existing_summary = (
        cache
        .groupby(["price_found", "price_status"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["price_found", "price_status"])
    )

    print("Existing cache status summary:")
    display(existing_summary)
    print()

# --------------------------------------------------------------------------------------------------
# Main pricing loop
# --------------------------------------------------------------------------------------------------

run_rows = []
new_rows = []
start_time = time.time()

try:
    if remaining.empty:
        print("No remaining selected contract requests to price.")
    else:
        for i, (_, row) in enumerate(remaining.iterrows(), start=1):
            result = price_contract(row)
            new_rows.append(result)
            run_rows.append(result)

            if i % PROGRESS_EVERY == 0 or i == 1:
                elapsed = time.time() - start_time
                priced_count = sum(1 for r in run_rows if r.get("price_found") is True)
                rate = i / elapsed if elapsed > 0 else np.nan
                remaining_count = len(remaining) - i
                eta_minutes = remaining_count / rate / 60 if rate and rate > 0 else np.nan

                print(
                    f"[{i:>5,}/{len(remaining):,}] "
                    f"priced={priced_count:,} "
                    f"last_status={result['price_status']} "
                    f"last_http={result['status_code']} "
                    f"rate={rate:.2f}/sec "
                    f"ETA={eta_minutes:.1f} min"
                )

            if i % CHECKPOINT_EVERY == 0:
                new_df = pd.DataFrame(new_rows)
                cache = pd.concat([cache, new_df], ignore_index=True)
                run_audit_df = pd.DataFrame(run_rows)
                cache = save_checkpoint(cache, run_audit_df=run_audit_df)
                new_rows = []

                checkpoint_summary = (
                    cache
                    .groupby(["price_found", "price_status"], dropna=False)
                    .size()
                    .reset_index(name="count")
                    .sort_values(["price_found", "price_status"])
                )

                print()
                print(f"Checkpoint saved after {i:,} attempts this run.")
                display(checkpoint_summary)
                print()

            if REQUEST_SLEEP_SECONDS and REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

except KeyboardInterrupt:
    print()
    print("KeyboardInterrupt received. Saving checkpoint before stopping...")

except Exception as e:
    print()
    print("Unexpected exception. Saving checkpoint before raising...")
    print(repr(e))
    print(traceback.format_exc())
    raise

finally:
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        cache = pd.concat([cache, new_df], ignore_index=True)

    run_audit_df = pd.DataFrame(run_rows)
    cache = save_checkpoint(cache, run_audit_df=run_audit_df)

# --------------------------------------------------------------------------------------------------
# Final diagnostics
# --------------------------------------------------------------------------------------------------

elapsed = time.time() - start_time
final_cache = pd.read_parquet(PRICE_CACHE_PATH)

status_summary = (
    final_cache
    .groupby(["price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["price_found", "price_status"])
)

status_code_summary = (
    final_cache
    .groupby(["status_code", "price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["status_code", "price_found", "price_status"])
)

date_coverage = (
    final_cache
    .groupby("trade_date_str")
    .agg(
        cached_contracts=("contract_request_id", "size"),
        priced_contracts=("price_found", "sum"),
    )
    .reset_index()
)

date_coverage["priced_rate"] = (
    date_coverage["priced_contracts"] / date_coverage["cached_contracts"]
)

missing_or_failed = final_cache.loc[
    final_cache["price_found"] != True
].copy()

missing_or_failed_preview = (
    missing_or_failed
    .sort_values(["trade_date_str", "expiration_str", "strike_listed"])
    .head(50)
)

# Save audit snapshots
status_summary_path = AUDIT_DIR / f"02E_selected_contract_price_cache_status_summary_{timestamp}.csv"
status_code_summary_path = AUDIT_DIR / f"02E_selected_contract_price_cache_status_code_summary_{timestamp}.csv"
date_coverage_path = AUDIT_DIR / f"02E_selected_contract_price_cache_date_coverage_{timestamp}.csv"
missing_preview_path = AUDIT_DIR / f"02E_selected_contract_price_cache_missing_preview_{timestamp}.csv"

status_summary.to_csv(status_summary_path, index=False)
status_code_summary.to_csv(status_code_summary_path, index=False)
date_coverage.to_csv(date_coverage_path, index=False)
missing_or_failed_preview.to_csv(missing_preview_path, index=False)

manifest = {
    "cell": "2E",
    "description": "Selected SPY contract price cache using ThetaData v3 /v3/option/history/greeks/eod",
    "created_at": timestamp,
    "base_url": BASE_URL,
    "endpoint": ENDPOINT,
    "selected_unique_contract_plan_path": str(SELECTED_UNIQUE_CONTRACT_PLAN_PATH),
    "price_cache_path": str(PRICE_CACHE_PATH),
    "price_cache_csv_snapshot_path": str(PRICE_CACHE_CSV_SNAPSHOT_PATH),
    "run_audit_path": str(RUN_AUDIT_PATH),
    "status_summary_path": str(status_summary_path),
    "status_code_summary_path": str(status_code_summary_path),
    "date_coverage_path": str(date_coverage_path),
    "missing_preview_path": str(missing_preview_path),
    "runtime_seconds": elapsed,
    "plan_rows": int(len(plan)),
    "cache_rows_final": int(len(final_cache)),
    "run_attempts": int(len(run_audit_df)),
    "run_priced": int(run_audit_df["price_found"].sum()) if not run_audit_df.empty and "price_found" in run_audit_df.columns else 0,
    "final_priced": int(final_cache["price_found"].sum()) if "price_found" in final_cache.columns else 0,
    "final_missing_or_failed": int((final_cache["price_found"] != True).sum()) if "price_found" in final_cache.columns else int(len(final_cache)),
    "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
    "retry_failed_or_missing": RETRY_FAILED_OR_MISSING,
}

with open(RUN_MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Cell 2E complete")
print("=" * 100)
print(f"Runtime minutes:       {elapsed / 60:.2f}")
print(f"Run attempts:          {len(run_audit_df):,}")
print(f"Final cache rows:      {len(final_cache):,}")
print(f"Final priced rows:     {int(final_cache['price_found'].sum()):,}")
print(f"Final missing/failed:  {int((final_cache['price_found'] != True).sum()):,}")
print()

print("Status summary:")
display(status_summary)

print()
print("Status-code summary:")
display(status_code_summary)

print()
print("Date coverage preview:")
display(date_coverage.head(30))

print()
print("Missing/failed preview:")
if missing_or_failed_preview.empty:
    print("No missing/failed rows in cache so far.")
else:
    display(
        missing_or_failed_preview[
            [
                "contract_request_id",
                "trade_date_str",
                "expiration_str",
                "strike_listed",
                "right",
                "status_code",
                "price_status",
                "raw_text_preview",
            ]
        ].head(50)
    )

print()
print("Saved:")
print(f"  Price cache parquet:       {PRICE_CACHE_PATH}")
print(f"  Price cache CSV snapshot:  {PRICE_CACHE_CSV_SNAPSHOT_PATH}")
print(f"  Run audit:                 {RUN_AUDIT_PATH}")
print(f"  Status summary:            {status_summary_path}")
print(f"  Status-code summary:       {status_code_summary_path}")
print(f"  Date coverage:             {date_coverage_path}")
print(f"  Missing preview:           {missing_preview_path}")
print(f"  Manifest:                  {RUN_MANIFEST_PATH}")

if len(final_cache) < len(plan):
    print()
    print("Result: Cache is partially complete. Review first-batch coverage; then rerun with MAX_REQUESTS_THIS_RUN = None to finish.")
else:
    print()
    print("Result: Selected contract cache completed. Next cell should join prices back to selected trades and compute real spread P&L.")

Cell 2E — Price selected unique contract request plan
Selected unique contract plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_v1.parquet
Price cache path:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_v1.parquet
ThetaData base URL:            http://127.0.0.1:25503
Endpoint:                      /v3/option/history/greeks/eod
Checkpoint every:              100 attempts
Max requests this run:         250
Retry failed/missing:          False

Loaded selected contract plan/cache
Plan rows:                 3,481
Existing cache rows:       0
Completed/skipped IDs:     0
Remaining full:            3,481
Remaining this run:        250

[    1/250] priced=1 last_status=priced last_http=200 rate=7.08/sec ETA=0.6 min
[   25/250] priced=15 last_status=priced last_http=200 rate=9.87/sec ETA=0.4 min
[   50/250] priced=33 last_status=pri

C:\Users\patri\AppData\Local\Temp\ipykernel_6252\574349498.py:592: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat([cache, new_df], ignore_index=True)


,price_found,price_status,count
0,False,http_472,26
1,True,priced,74



[  125/250] priced=95 last_status=priced last_http=200 rate=8.61/sec ETA=0.2 min
[  150/250] priced=113 last_status=http_472 last_http=472 rate=8.68/sec ETA=0.2 min
[  175/250] priced=134 last_status=priced last_http=200 rate=8.57/sec ETA=0.1 min
[  200/250] priced=155 last_status=http_472 last_http=472 rate=8.84/sec ETA=0.1 min

Checkpoint saved after 200 attempts this run.


,price_found,price_status,count
0,False,http_472,45
1,True,priced,155


...


Date coverage preview:


,trade_date_str,cached_contracts,priced_contracts,priced_rate
0,2020-12-31,6,4,0.666667
1,2021-01-06,2,1,0.500000
2,2021-01-11,2,1,0.500000
3,2021-01-19,4,3,0.750000
4,2021-01-25,8,4,0.500000
5,2021-01-26,10,6,0.600000
6,2021-02-16,10,8,0.800000
7,2021-02-17,10,8,0.800000
8,2021-02-18,6,3,0.500000
9,2021-02-19,4,2,0.500000



Missing/failed preview:


,contract_request_id,trade_date_str,expiration_str,strike_listed,right,status_code,price_status,raw_text_preview
3,SPY_20201231_20210129_442.0_C,2020-12-31,2021-01-29,442.0,C,472,http_472,No data found for your request
5,SPY_20201231_20210205_451.0_C,2020-12-31,2021-02-05,451.0,C,472,http_472,No data found for your request
7,SPY_20210106_20210115_419.0_C,2021-01-06,2021-01-15,419.0,C,472,http_472,No data found for your request
9,SPY_20210111_20210122_418.0_C,2021-01-11,2021-01-22,418.0,C,472,http_472,No data found for your request
13,SPY_20210119_20210205_424.0_C,2021-01-19,2021-02-05,424.0,C,472,http_472,No data found for your request
15,SPY_20210125_20210205_423.0_C,2021-01-25,2021-02-05,423.0,C,472,http_472,No data found for your request
19,SPY_20210125_20210212_429.0_C,2021-01-25,2021-02-12,429.0,C,472,http_472,No data found for your request
20,SPY_20210125_20210212_436.0_C,2021-01-25,2021-02-12,436.0,C,472,http_472,No data found for your request
21,SPY_20210125_20210212_442.0_C,2021-01-25,2021-02-12,442.0,C,472,http_472,No data found for your request
23,SPY_20210126_20210205_423.0_C,2021-01-26,2021-02-05,423.0,C,472,http_472,No data found for your request



Saved:
  Price cache parquet:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_v1.parquet
  Price cache CSV snapshot:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_call_sleeve_corsi_call_selected_contract_price_cache_latest.csv
  Run audit:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_selected_contract_price_cache_run_audit_20260710_104429.csv
  Status summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_selected_contract_price_cache_status_summary_20260710_104429.csv
  Status-code summary:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_selected_contract_price_cache_status_code_summary_20260710_104429.csv
  Date coverage:             C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_selected_contract_price_cache_date_coverage_20260710_104429.csv
  Missing preview:           C:\Users\patri\vrp_proje

In [16]:
# Cell 2D.1 — Rebuild selected-trade pricing universe with 3SD long strike rounded to nearest $5
# Purpose:
#   Keep the selected-trade universe from Cell 2D, but change the long 3SD call strike convention:
#
#       short 1SD call: ceil raw strike to nearest $1
#       long 3SD call:  round raw strike to nearest $5
#
# Key point:
#   This cell DOES NOT call ThetaData.
#
# It creates new *_long5_v1 files so we do not mix the old $1-long-strike cache with the new convention.

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import json
import math

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_TRADES_OLD_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet"
)

SYMBOL = "SPY"
RIGHT = "C"
PRICING_ENDPOINT = "/v3/option/history/greeks/eod"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2D.1 — Rebuild selected-trade pricing universe with 3SD long strike rounded to nearest $5")
print("=" * 100)
print(f"Input selected trades: {SELECTED_TRADES_OLD_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def ceil_to_increment(x, increment):
    if pd.isna(x):
        return np.nan
    return float(math.ceil(float(x) / float(increment)) * float(increment))

def round_to_nearest_increment(x, increment):
    if pd.isna(x):
        return np.nan
    return float(round(float(x) / float(increment)) * float(increment))

def make_trade_id(trade_date, tenor, expiration_date, short_strike, long_strike):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"T{int(tenor):02d}_"
        f"EXP{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"S{float(short_strike):.1f}_"
        f"L{float(long_strike):.1f}"
    )

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def summarize_rule(g):
    g = g.sort_values("trade_date").copy()

    intrinsic = g["terminal_spread_intrinsic_before_premium"].fillna(0.0)
    intrinsic_pct = g["terminal_intrinsic_pct_width"].fillna(0.0)

    rolling_20_intrinsic = intrinsic.rolling(20, min_periods=1).sum()
    rolling_20_intrinsic_pct = intrinsic_pct.rolling(20, min_periods=1).sum()

    year_counts = g.groupby("trade_year").size()

    return pd.Series({
        "selected_trades": len(g),
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "years_traded": ",".join(map(str, sorted(g["trade_year"].dropna().astype(int).unique()))),
        "n_years_traded": g["trade_year"].nunique(),
        "median_trades_per_year": year_counts.median() if not year_counts.empty else np.nan,
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "avg_effective_dte": g["effective_dte"].mean(),
        "avg_listed_width": g["listed_width"].mean(),
        "median_listed_width": g["listed_width"].median(),
        "short_call_otm_rate": g["expired_short_call_otm"].mean(),
        "zero_intrinsic_rate": g["expired_spread_zero_intrinsic"].mean(),
        "avg_terminal_intrinsic": intrinsic.mean(),
        "median_terminal_intrinsic": intrinsic.median(),
        "p95_terminal_intrinsic": intrinsic.quantile(0.95),
        "worst_terminal_intrinsic": intrinsic.max(),
        "avg_terminal_intrinsic_pct_width": intrinsic_pct.mean(),
        "p95_terminal_intrinsic_pct_width": intrinsic_pct.quantile(0.95),
        "worst_terminal_intrinsic_pct_width": intrinsic_pct.max(),
        "worst_20_trade_intrinsic_sum": rolling_20_intrinsic.max(),
        "worst_20_trade_intrinsic_pct_width_sum": rolling_20_intrinsic_pct.max(),
        "expiry_price_exact_rate": g["expiry_price_date_exact"].mean(),
    })

# --------------------------------------------------------------------------------------------------
# Load old selected-trade-by-rule table
# --------------------------------------------------------------------------------------------------

selected_trades = pd.read_parquet(SELECTED_TRADES_OLD_PATH).copy()

selected_trades["trade_date"] = pd.to_datetime(selected_trades["trade_date"], errors="coerce")
selected_trades["expiration_date"] = pd.to_datetime(selected_trades["expiration_date"], errors="coerce")
selected_trades["target_expiration_date"] = pd.to_datetime(selected_trades["target_expiration_date"], errors="coerce")
selected_trades["spy_price_date"] = pd.to_datetime(selected_trades["spy_price_date"], errors="coerce")
selected_trades["tenor"] = selected_trades["tenor"].astype(int)

required_cols = [
    "trade_date",
    "tenor",
    "expiration_date",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "expiry_spy_close",
    "rule_id",
    "selection_rule",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
]

missing = [c for c in required_cols if c not in selected_trades.columns]
if missing:
    raise ValueError(f"Missing required columns from selected trades: {missing}")

# --------------------------------------------------------------------------------------------------
# Apply revised strike convention
# --------------------------------------------------------------------------------------------------

selected_trades["short_strike"] = selected_trades["short_call_1sd_raw"].apply(
    lambda x: ceil_to_increment(x, 1.0)
)

selected_trades["long_strike"] = selected_trades["long_call_3sd_raw"].apply(
    lambda x: round_to_nearest_increment(x, 5.0)
)

# Safety: if nearest $5 somehow lands at/below short strike, push long to next $5 above short.
bad_long = selected_trades["long_strike"] <= selected_trades["short_strike"]
selected_trades.loc[bad_long, "long_strike"] = selected_trades.loc[bad_long, "short_strike"].apply(
    lambda x: ceil_to_increment(float(x) + 0.01, 5.0)
)

selected_trades["listed_width"] = selected_trades["long_strike"] - selected_trades["short_strike"]

selected_trades = selected_trades.loc[selected_trades["listed_width"] > 0].copy()

selected_trades["short_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["short_strike"],
    0.0,
)

selected_trades["long_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["long_strike"],
    0.0,
)

selected_trades["terminal_spread_intrinsic_before_premium"] = (
    selected_trades["short_call_terminal_intrinsic"]
    - selected_trades["long_call_terminal_intrinsic"]
)

selected_trades["terminal_intrinsic_pct_width"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"]
    / selected_trades["listed_width"]
)

selected_trades["expired_short_call_otm"] = (
    selected_trades["expiry_spy_close"] <= selected_trades["short_strike"]
)

selected_trades["expired_spread_zero_intrinsic"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"] <= 0
)

selected_trades["trade_year"] = selected_trades["trade_date"].dt.year

selected_trades["selected_trade_id"] = selected_trades.apply(
    lambda r: make_trade_id(
        r["trade_date"],
        r["tenor"],
        r["expiration_date"],
        r["short_strike"],
        r["long_strike"],
    ),
    axis=1,
)

# --------------------------------------------------------------------------------------------------
# Rebuild unique selected trade universe
# --------------------------------------------------------------------------------------------------

rule_usage = (
    selected_trades
    .groupby("selected_trade_id")
    .agg(
        selected_by_rule_count=("rule_id", "nunique"),
        selection_rules=("selection_rule", lambda x: ",".join(sorted(set(map(str, x))))),
        min_z_threshold=("z_threshold_shared", "min"),
        max_z_threshold=("z_threshold_shared", "max"),
        min_rsi_floor=("rsi_floor", "min"),
        max_rsi_floor=("rsi_floor", "max"),
        vrp_log_floor_labels=("vrp_log_floor_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

trade_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "vix_style_vol_pct",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "z_avg",
    "z_min",
    "rsi14",
    "target_expiration_date",
    "expiration_date",
    "effective_dte",
    "expiration_slippage_days",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "short_strike",
    "long_strike",
    "listed_width",
    "spy_price_date",
    "expiry_spy_close",
    "expiry_price_date_exact",
    "short_call_terminal_intrinsic",
    "long_call_terminal_intrinsic",
    "terminal_spread_intrinsic_before_premium",
    "terminal_intrinsic_pct_width",
    "expired_short_call_otm",
    "expired_spread_zero_intrinsic",
]

trade_cols = [c for c in trade_cols if c in selected_trades.columns]

unique_selected_trades = (
    selected_trades[trade_cols]
    .drop_duplicates("selected_trade_id")
    .merge(rule_usage, on="selected_trade_id", how="left")
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Rebuild selected option leg request plan
# --------------------------------------------------------------------------------------------------

short_legs = unique_selected_trades.copy()
short_legs["leg_label"] = "short_call_1sd"
short_legs["leg_side"] = "sell"
short_legs["price_field_for_entry"] = "bid"
short_legs["symbol"] = SYMBOL
short_legs["right"] = RIGHT
short_legs["strike_listed"] = short_legs["short_strike"]

long_legs = unique_selected_trades.copy()
long_legs["leg_label"] = "long_call_3sd"
long_legs["leg_side"] = "buy"
long_legs["price_field_for_entry"] = "ask"
long_legs["symbol"] = SYMBOL
long_legs["right"] = RIGHT
long_legs["strike_listed"] = long_legs["long_strike"]

selected_leg_plan = pd.concat([short_legs, long_legs], ignore_index=True)

selected_leg_plan["trade_date_str"] = selected_leg_plan["trade_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_str"] = selected_leg_plan["expiration_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_yyyymmdd"] = selected_leg_plan["expiration_date"].dt.strftime("%Y%m%d")
selected_leg_plan["endpoint"] = PRICING_ENDPOINT

selected_leg_plan["contract_request_id"] = selected_leg_plan.apply(
    lambda r: make_contract_request_id(
        r["symbol"],
        r["trade_date"],
        r["expiration_date"],
        r["strike_listed"],
        r["right"],
    ),
    axis=1,
)

contract_usage = (
    selected_leg_plan
    .groupby("contract_request_id")
    .agg(
        mapped_trade_legs=("selected_trade_id", "size"),
        mapped_unique_trades=("selected_trade_id", "nunique"),
        mapped_tenors=("tenor", lambda x: ",".join(map(str, sorted(set(x.astype(int)))))),
        mapped_leg_labels=("leg_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

unique_contract_request_plan = (
    selected_leg_plan[
        [
            "contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "strike_listed",
            "endpoint",
        ]
    ]
    .drop_duplicates()
    .merge(contract_usage, on="contract_request_id", how="left")
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Rebuild pre-price rule summaries
# --------------------------------------------------------------------------------------------------

rule_summary = (
    selected_trades
    .groupby(
        [
            "rule_id",
            "z_threshold_shared",
            "rsi_floor",
            "vrp_log_floor_label",
            "selection_rule",
        ],
        dropna=False,
    )
    .apply(summarize_rule, include_groups=False)
    .reset_index()
)

rule_year_summary = (
    selected_trades
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_trades=("selected_trade_id", "size"),
        short_call_otm_rate=("expired_short_call_otm", "mean"),
        zero_intrinsic_rate=("expired_spread_zero_intrinsic", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
    .sort_values(["rule_id", "trade_year"])
)

# --------------------------------------------------------------------------------------------------
# Save long5 outputs
# --------------------------------------------------------------------------------------------------

selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_v1.parquet"
)

unique_selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_long5_v1.parquet"
)

selected_leg_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_request_plan_long5_v1.parquet"
)

unique_contract_request_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet"
)

rule_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_preprice_summary_long5_v1.parquet"
)

rule_year_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_preprice_summary_long5_v1.parquet"
)

selected_trades.to_parquet(selected_trades_path, index=False)
unique_selected_trades.to_parquet(unique_selected_trades_path, index=False)
selected_leg_plan.to_parquet(selected_leg_plan_path, index=False)
unique_contract_request_plan.to_parquet(unique_contract_request_plan_path, index=False)
rule_summary.to_parquet(rule_summary_path, index=False)
rule_year_summary.to_parquet(rule_year_summary_path, index=False)

selected_trades_csv = AUDIT_DIR / f"02D1_rule_selected_trades_unpriced_long5_{timestamp}.csv"
unique_selected_trades_csv = AUDIT_DIR / f"02D1_unique_selected_trade_universe_long5_{timestamp}.csv"
selected_leg_plan_csv = AUDIT_DIR / f"02D1_selected_leg_request_plan_long5_{timestamp}.csv"
unique_contract_request_plan_csv = AUDIT_DIR / f"02D1_selected_unique_contract_request_plan_long5_{timestamp}.csv"
rule_summary_csv = AUDIT_DIR / f"02D1_rule_preprice_summary_long5_{timestamp}.csv"
rule_year_summary_csv = AUDIT_DIR / f"02D1_rule_year_preprice_summary_long5_{timestamp}.csv"

selected_trades.to_csv(selected_trades_csv, index=False)
unique_selected_trades.to_csv(unique_selected_trades_csv, index=False)
selected_leg_plan.to_csv(selected_leg_plan_csv, index=False)
unique_contract_request_plan.to_csv(unique_contract_request_plan_csv, index=False)
rule_summary.to_csv(rule_summary_csv, index=False)
rule_year_summary.to_csv(rule_year_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02D1_selected_trade_pricing_universe_long5_manifest_{timestamp}.json"

manifest = {
    "cell": "2D.1",
    "description": "Rebuild selected-trade pricing universe with long 3SD call rounded to nearest $5",
    "created_at": timestamp,
    "input_selected_trades_old_path": str(SELECTED_TRADES_OLD_PATH),
    "pricing_endpoint_for_next_cell": PRICING_ENDPOINT,
    "strike_convention": {
        "short_call_1sd": "ceil raw strike to nearest $1",
        "long_call_3sd": "round raw strike to nearest $5",
    },
    "counts": {
        "rule_selected_rows": int(len(selected_trades)),
        "unique_selected_trades": int(len(unique_selected_trades)),
        "selected_leg_rows": int(len(selected_leg_plan)),
        "unique_contract_requests": int(len(unique_contract_request_plan)),
        "rule_count": int(rule_summary["rule_id"].nunique()),
    },
    "outputs": {
        "selected_trades_path": str(selected_trades_path),
        "unique_selected_trades_path": str(unique_selected_trades_path),
        "selected_leg_plan_path": str(selected_leg_plan_path),
        "unique_contract_request_plan_path": str(unique_contract_request_plan_path),
        "rule_summary_path": str(rule_summary_path),
        "rule_year_summary_path": str(rule_year_summary_path),
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------------------

summary_counts = pd.DataFrame([{
    "rule_selected_rows": len(selected_trades),
    "unique_selected_trades": len(unique_selected_trades),
    "selected_leg_rows": len(selected_leg_plan),
    "unique_contract_requests_to_price": len(unique_contract_request_plan),
    "rule_count": rule_summary["rule_id"].nunique(),
    "date_min": selected_trades["trade_date"].min().date(),
    "date_max": selected_trades["trade_date"].max().date(),
}])

print()
print("=" * 100)
print("Long5 selected pricing universe summary")
print("=" * 100)
display(summary_counts)

print()
print("=" * 100)
print("Strike convention check")
print("=" * 100)

strike_check = pd.DataFrame([{
    "short_non_integer_count": int((selected_trades["short_strike"] % 1 != 0).sum()),
    "long_not_multiple_of_5_count": int((selected_trades["long_strike"] % 5 != 0).sum()),
    "min_width": selected_trades["listed_width"].min(),
    "median_width": selected_trades["listed_width"].median(),
    "max_width": selected_trades["listed_width"].max(),
}])

display(strike_check)

print()
print("=" * 100)
print("Rule summary preview")
print("=" * 100)

preview = (
    rule_summary
    .sort_values(
        [
            "short_call_otm_rate",
            "worst_terminal_intrinsic_pct_width",
            "selected_trades",
        ],
        ascending=[False, True, False],
    )
    .head(30)
)

display(
    preview[
        [
            "rule_id",
            "selected_trades",
            "selection_rule",
            "z_threshold_shared",
            "rsi_floor",
            "vrp_log_floor_label",
            "tenors_selected",
            "short_call_otm_rate",
            "zero_intrinsic_rate",
            "avg_terminal_intrinsic",
            "worst_terminal_intrinsic",
            "worst_terminal_intrinsic_pct_width",
            "worst_20_trade_intrinsic_pct_width_sum",
        ]
    ]
)

print()
print("=" * 100)
print("Unique contract request plan preview")
print("=" * 100)
display(
    unique_contract_request_plan[
        [
            "contract_request_id",
            "symbol",
            "trade_date_str",
            "expiration_str",
            "strike_listed",
            "right",
            "mapped_unique_trades",
            "mapped_tenors",
            "mapped_leg_labels",
        ]
    ].head(40)
)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Selected trades by rule:          {selected_trades_path}")
print(f"Unique selected trade universe:   {unique_selected_trades_path}")
print(f"Selected leg request plan:        {selected_leg_plan_path}")
print(f"Unique contract request plan:     {unique_contract_request_plan_path}")
print(f"Rule pre-price summary:           {rule_summary_path}")
print(f"Rule-year pre-price summary:      {rule_year_summary_path}")
print(f"Manifest:                         {manifest_path}")

print()
print("Result: long5 selected-trade pricing universe rebuilt. Next price the long5 unique contract plan with a fresh cache.")

Cell 2D.1 — Rebuild selected-trade pricing universe with 3SD long strike rounded to nearest $5
Input selected trades: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet


Long5 selected pricing universe summary


,rule_selected_rows,unique_selected_trades,selected_leg_rows,unique_contract_requests_to_price,rule_count,date_min,date_max
0,155550,1741,3482,3437,840,2020-12-31,2026-06-02



Strike convention check


,short_non_integer_count,long_not_multiple_of_5_count,min_width,median_width,max_width
0,0,0,14.0,31.0,87.0



Rule summary preview


,rule_id,selected_trades,selection_rule,z_threshold_shared,rsi_floor,vrp_log_floor_label,tenors_selected,short_call_otm_rate,zero_intrinsic_rate,avg_terminal_intrinsic,worst_terminal_intrinsic,worst_terminal_intrinsic_pct_width,worst_20_trade_intrinsic_pct_width_sum
601,Z1p00_RSI55_VRP0p00_SEL_highest_z_avg,107,highest_z_avg,1.0,55,0.00,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.151852,0.442533
606,Z1p00_RSI55_VRP0p20_SEL_highest_z_avg,107,highest_z_avg,1.0,55,0.20,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.151852,0.442533
616,Z1p00_RSI55_VRPNONE_SEL_highest_z_avg,107,highest_z_avg,1.0,55,NONE,"9,12,15,18,21,24,27,30,33",0.953271,0.953271,0.125888,4.10,0.151852,0.442533
611,Z1p00_RSI55_VRP0p40_SEL_highest_z_avg,104,highest_z_avg,1.0,55,0.40,"9,12,15,18,21,24,27,30,33",0.951923,0.951923,0.129519,4.10,0.151852,0.442533
661,Z1p00_RSI62_VRP0p00_SEL_highest_z_avg,76,highest_z_avg,1.0,62,0.00,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682
662,Z1p00_RSI62_VRP0p00_SEL_highest_z_min,76,highest_z_min,1.0,62,0.00,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682
666,Z1p00_RSI62_VRP0p20_SEL_highest_z_avg,76,highest_z_avg,1.0,62,0.20,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682
667,Z1p00_RSI62_VRP0p20_SEL_highest_z_min,76,highest_z_min,1.0,62,0.20,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682
676,Z1p00_RSI62_VRPNONE_SEL_highest_z_avg,76,highest_z_avg,1.0,62,NONE,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682
677,Z1p00_RSI62_VRPNONE_SEL_highest_z_min,76,highest_z_min,1.0,62,NONE,"9,12,15,18,21,24,27,30,33",0.947368,0.947368,0.123289,2.64,0.104762,0.290682



Unique contract request plan preview


,contract_request_id,symbol,trade_date_str,expiration_str,strike_listed,right,mapped_unique_trades,mapped_tenors,mapped_leg_labels
0,SPY_20201231_20210115_386.0_C,SPY,2020-12-31,2021-01-15,386.0,C,1,9,short_call_1sd
1,SPY_20201231_20210115_410.0_C,SPY,2020-12-31,2021-01-15,410.0,C,1,9,long_call_3sd
2,SPY_20201231_20210129_397.0_C,SPY,2020-12-31,2021-01-29,397.0,C,1,27,short_call_1sd
3,SPY_20201231_20210129_440.0_C,SPY,2020-12-31,2021-01-29,440.0,C,1,27,long_call_3sd
4,SPY_20201231_20210205_400.0_C,SPY,2020-12-31,2021-02-05,400.0,C,1,33,short_call_1sd
5,SPY_20201231_20210205_450.0_C,SPY,2020-12-31,2021-02-05,450.0,C,1,33,long_call_3sd
6,SPY_20210106_20210115_389.0_C,SPY,2021-01-06,2021-01-15,389.0,C,1,9,short_call_1sd
7,SPY_20210106_20210115_420.0_C,SPY,2021-01-06,2021-01-15,420.0,C,1,9,long_call_3sd
8,SPY_20210111_20210122_392.0_C,SPY,2021-01-11,2021-01-22,392.0,C,1,9,short_call_1sd
9,SPY_20210111_20210122_420.0_C,SPY,2021-01-11,2021-01-22,420.0,C,1,9,long_call_3sd



Saved
Selected trades by rule:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_v1.parquet
Unique selected trade universe:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_long5_v1.parquet
Selected leg request plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_long5_v1.parquet
Unique contract request plan:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet
Rule pre-price summary:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_preprice_summary_long5_v1.parquet
Rule-year pre-price summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_preprice_summary_long5_v1.parqu

In [19]:
print("Selected long5 contract plan:")
print(SELECTED_UNIQUE_CONTRACT_PLAN_PATH)
print("Exists:", SELECTED_UNIQUE_CONTRACT_PLAN_PATH.exists())

print()
print("Long5 price cache:")
print(PRICE_CACHE_PATH)
print("Exists:", PRICE_CACHE_PATH.exists())

plan_check = pd.read_parquet(SELECTED_UNIQUE_CONTRACT_PLAN_PATH)
print()
print("Plan rows:", len(plan_check))
print(plan_check.head())

Selected long5 contract plan:
C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet
Exists: True

Long5 price cache:
C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_v1.parquet
Exists: False

Plan rows: 3437
             contract_request_id symbol right trade_date trade_date_str  \
0  SPY_20201231_20210115_386.0_C    SPY     C 2020-12-31     2020-12-31   
1  SPY_20201231_20210115_410.0_C    SPY     C 2020-12-31     2020-12-31   
2  SPY_20201231_20210129_397.0_C    SPY     C 2020-12-31     2020-12-31   
3  SPY_20201231_20210129_440.0_C    SPY     C 2020-12-31     2020-12-31   
4  SPY_20201231_20210205_400.0_C    SPY     C 2020-12-31     2020-12-31   

  expiration_date expiration_str expiration_yyyymmdd  strike_listed  \
0      2021-01-15     2021-01-15            20210115          386.0   
1      2021-01-15     2021-01-15       

In [20]:
# Cell 2E — Price selected long5 unique contract request plan with checkpointing
# Purpose:
#   Price only the selected-trade contract universe rebuilt in Cell 2D.1.
#
# Current strike convention:
#   short 1SD call: ceil raw strike to nearest $1
#   long 3SD call:  round raw strike to nearest $5
#
# Input:
#   call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet
#
# Output:
#   call_sleeve_corsi_call_selected_contract_price_cache_long5_v1.parquet
#
# First run:
#   MAX_REQUESTS_THIS_RUN = 250
#
# After this batch looks clean:
#   change MAX_REQUESTS_THIS_RUN = None and rerun to finish the full cache.

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json
import time
import traceback

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_UNIQUE_CONTRACT_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet"
)

PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_v1.parquet"
)

PRICE_CACHE_CSV_SNAPSHOT_PATH = (
    AUDIT_DIR / "02E_call_sleeve_corsi_call_selected_contract_price_cache_long5_latest.csv"
)

BASE_URL = "http://127.0.0.1:25503"
ENDPOINT = "/v3/option/history/greeks/eod"

# Runtime controls
CHECKPOINT_EVERY = 100
PROGRESS_EVERY = 25
REQUEST_SLEEP_SECONDS = 0.02

# Keep 250 for validation. Set to None after first batch looks clean.
MAX_REQUESTS_THIS_RUN = 250

# Usually False. If True, only already-priced rows are skipped; failed/missing rows are retried.
RETRY_FAILED_OR_MISSING = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_AUDIT_PATH = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_run_audit_{timestamp}.csv"
RUN_MANIFEST_PATH = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_manifest_{timestamp}.json"

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2E — Price selected long5 unique contract request plan")
print("=" * 100)
print(f"Selected unique contract plan: {SELECTED_UNIQUE_CONTRACT_PLAN_PATH}")
print(f"Price cache path:              {PRICE_CACHE_PATH}")
print(f"ThetaData base URL:            {BASE_URL}")
print(f"Endpoint:                      {ENDPOINT}")
print(f"Checkpoint every:              {CHECKPOINT_EVERY:,} attempts")
print(f"Max requests this run:         {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed/missing:          {RETRY_FAILED_OR_MISSING}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    # JSON fallback
    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None

                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break

                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)

                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    # CSV is expected for this endpoint with format=csv
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    # NDJSON fallback
    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})


def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None


def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "created": ["created"],
        "last_trade": ["last_trade"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "bid_size": ["bid_size"],
        "ask_size": ["ask_size"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close"],
        "volume": ["volume"],
        "count": ["count"],
        "underlying_price": ["underlying_price"],
        "underlying_timestamp": ["underlying_timestamp"],
        "implied_vol": ["implied_vol", "iv"],
        "iv_error": ["iv_error"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
        "rho": ["rho"],
        "epsilon": ["epsilon"],
        "lambda": ["lambda"],
        "omega": ["omega"],
        "vanna": ["vanna"],
        "charm": ["charm"],
        "vomma": ["vomma"],
        "veta": ["veta"],
        "vera": ["vera"],
        "speed": ["speed"],
        "zomma": ["zomma"],
        "color": ["color"],
        "ultima": ["ultima"],
        "d1": ["d1"],
        "d2": ["d2"],
        "dual_delta": ["dual_delta"],
        "dual_gamma": ["dual_gamma"],
    }

    rename_map = {}

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    numeric_cols = [
        "strike", "bid", "ask", "mid", "bid_size", "ask_size",
        "open", "high", "low", "close", "volume", "count",
        "underlying_price", "implied_vol", "iv_error",
        "delta", "theta", "vega", "gamma", "rho",
        "epsilon", "lambda", "omega", "vanna", "charm",
        "vomma", "veta", "vera", "speed", "zomma",
        "color", "ultima", "d1", "d2", "dual_delta", "dual_gamma",
    ]

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out


def atomic_write_parquet(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)


def atomic_write_csv(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def build_empty_cache():
    return pd.DataFrame(columns=[
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "endpoint",
        "status_code",
        "http_ok",
        "response_rows",
        "response_cols",
        "response_columns",
        "price_found",
        "price_status",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "timestamp",
        "created",
        "last_trade",
        "underlying_price",
        "underlying_timestamp",
        "implied_vol",
        "iv_error",
        "delta",
        "theta",
        "vega",
        "gamma",
        "rho",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "count",
        "raw_text_preview",
        "error",
        "request_url",
        "pulled_at",
    ])


def get_completed_ids(cache_df):
    if cache_df.empty:
        return set()

    if RETRY_FAILED_OR_MISSING:
        if "price_found" in cache_df.columns:
            return set(cache_df.loc[cache_df["price_found"] == True, "contract_request_id"].astype(str))
        return set()

    return set(cache_df["contract_request_id"].dropna().astype(str))


def price_contract(plan_row):
    symbol = str(plan_row["symbol"]).upper()
    right = normalize_right(plan_row["right"])
    strike = float(plan_row["strike_listed"])
    trade_date_str = str(plan_row["trade_date_str"])
    expiration_str = str(plan_row["expiration_str"])

    params = {
        "symbol": symbol,
        "right": right,
        "strike": strike,
        "expiration": expiration_str,
        "start_date": trade_date_str,
        "end_date": trade_date_str,
        "format": "csv",
    }

    url = BASE_URL.rstrip("/") + ENDPOINT
    pulled_at = datetime.now().isoformat(timespec="seconds")

    base_result = {
        "contract_request_id": str(plan_row["contract_request_id"]),
        "symbol": symbol,
        "right": right,
        "trade_date": pd.Timestamp(plan_row["trade_date"]),
        "trade_date_str": trade_date_str,
        "expiration_date": pd.Timestamp(plan_row["expiration_date"]),
        "expiration_str": expiration_str,
        "expiration_yyyymmdd": str(plan_row["expiration_yyyymmdd"]),
        "strike_listed": strike,
        "endpoint": ENDPOINT,
        "status_code": np.nan,
        "http_ok": False,
        "response_rows": 0,
        "response_cols": 0,
        "response_columns": "",
        "price_found": False,
        "price_status": "not_attempted",
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_size": np.nan,
        "ask_size": np.nan,
        "timestamp": None,
        "created": None,
        "last_trade": None,
        "underlying_price": np.nan,
        "underlying_timestamp": None,
        "implied_vol": np.nan,
        "iv_error": np.nan,
        "delta": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "gamma": np.nan,
        "rho": np.nan,
        "open": np.nan,
        "high": np.nan,
        "low": np.nan,
        "close": np.nan,
        "volume": np.nan,
        "count": np.nan,
        "raw_text_preview": "",
        "error": None,
        "request_url": "",
        "pulled_at": pulled_at,
    }

    try:
        resp = session.get(url, params=params, timeout=(2, 45))

        base_result["status_code"] = resp.status_code
        base_result["http_ok"] = bool(resp.ok)
        base_result["raw_text_preview"] = resp.text[:500] if hasattr(resp, "text") else ""
        base_result["request_url"] = resp.url

        raw_df = response_to_dataframe(resp)
        std_df = standardize_option_rows(raw_df)

        base_result["response_rows"] = int(len(raw_df))
        base_result["response_cols"] = int(len(raw_df.columns))
        base_result["response_columns"] = ", ".join(map(str, raw_df.columns))

        if not resp.ok:
            base_result["price_status"] = f"http_{resp.status_code}"
            return base_result

        if std_df.empty:
            base_result["price_status"] = "empty_response"
            return base_result

        if not {"bid", "ask"}.issubset(std_df.columns):
            base_result["price_status"] = "missing_bid_ask_columns"
            return base_result

        valid = std_df.loc[
            std_df["bid"].notna()
            & std_df["ask"].notna()
        ].copy()

        if valid.empty:
            base_result["price_status"] = "no_valid_bid_ask"
            return base_result

        # Prefer exact requested contract when identifiers exist.
        if {"strike", "right"}.issubset(valid.columns):
            exact_mask = (
                valid["strike"].round(4).eq(round(strike, 4))
                & valid["right"].eq(right)
            )

            if "expiration" in valid.columns:
                exact_mask = exact_mask & valid["expiration"].eq(expiration_str)

            exact = valid.loc[exact_mask].copy()

            if not exact.empty:
                valid = exact

        px = valid.iloc[-1]

        base_result["price_found"] = True
        base_result["price_status"] = "priced"

        for col in [
            "bid", "ask", "mid", "bid_size", "ask_size",
            "timestamp", "created", "last_trade",
            "underlying_price", "underlying_timestamp",
            "implied_vol", "iv_error",
            "delta", "theta", "vega", "gamma", "rho",
            "open", "high", "low", "close", "volume", "count",
        ]:
            if col in valid.columns:
                base_result[col] = px.get(col, base_result[col])

        return base_result

    except Exception as e:
        base_result["price_status"] = "exception"
        base_result["error"] = repr(e)
        base_result["raw_text_preview"] = traceback.format_exc()[:500]
        return base_result


def save_checkpoint(cache_df, run_audit_df=None):
    cache_df = cache_df.copy()

    if not cache_df.empty:
        cache_df["contract_request_id"] = cache_df["contract_request_id"].astype(str)
        cache_df["pulled_at"] = cache_df["pulled_at"].astype(str)

        cache_df = (
            cache_df
            .sort_values(["contract_request_id", "pulled_at"])
            .drop_duplicates("contract_request_id", keep="last")
            .sort_values(["trade_date_str", "expiration_str", "strike_listed", "right"])
            .reset_index(drop=True)
        )

    atomic_write_parquet(cache_df, PRICE_CACHE_PATH)
    atomic_write_csv(cache_df, PRICE_CACHE_CSV_SNAPSHOT_PATH)

    if run_audit_df is not None:
        atomic_write_csv(run_audit_df, RUN_AUDIT_PATH)

    return cache_df


# --------------------------------------------------------------------------------------------------
# Load selected long5 contract plan and existing long5 cache
# --------------------------------------------------------------------------------------------------

if not SELECTED_UNIQUE_CONTRACT_PLAN_PATH.exists():
    raise FileNotFoundError(f"Missing selected unique contract plan: {SELECTED_UNIQUE_CONTRACT_PLAN_PATH}")

plan = pd.read_parquet(SELECTED_UNIQUE_CONTRACT_PLAN_PATH).copy()

required_plan_cols = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "trade_date_str",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "strike_listed",
]

missing_plan_cols = [c for c in required_plan_cols if c not in plan.columns]
if missing_plan_cols:
    raise ValueError(f"Missing required columns from selected unique contract plan: {missing_plan_cols}")

plan["contract_request_id"] = plan["contract_request_id"].astype(str)
plan["trade_date"] = pd.to_datetime(plan["trade_date"], errors="coerce")
plan["expiration_date"] = pd.to_datetime(plan["expiration_date"], errors="coerce")
plan["strike_listed"] = pd.to_numeric(plan["strike_listed"], errors="coerce")

plan = (
    plan
    .dropna(subset=["trade_date", "expiration_date", "strike_listed"])
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

if PRICE_CACHE_PATH.exists():
    cache = pd.read_parquet(PRICE_CACHE_PATH).copy()

    if "contract_request_id" in cache.columns:
        cache["contract_request_id"] = cache["contract_request_id"].astype(str)
    else:
        cache = build_empty_cache()
else:
    cache = build_empty_cache()

completed_ids = get_completed_ids(cache)

remaining_full = plan.loc[
    ~plan["contract_request_id"].astype(str).isin(completed_ids)
].copy()

remaining = remaining_full.copy()

if MAX_REQUESTS_THIS_RUN is not None:
    remaining = remaining.head(int(MAX_REQUESTS_THIS_RUN)).copy()

print("=" * 100)
print("Loaded selected long5 contract plan/cache")
print("=" * 100)
print(f"Plan rows:                 {len(plan):,}")
print(f"Existing cache rows:       {len(cache):,}")
print(f"Completed/skipped IDs:     {len(completed_ids):,}")
print(f"Remaining full:            {len(remaining_full):,}")
print(f"Remaining this run:        {len(remaining):,}")
print()

if not cache.empty:
    existing_summary = (
        cache
        .groupby(["price_found", "price_status"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["price_found", "price_status"])
    )

    print("Existing cache status summary:")
    display(existing_summary)
    print()


# --------------------------------------------------------------------------------------------------
# Main pricing loop
# --------------------------------------------------------------------------------------------------

run_rows = []
new_rows = []
start_time = time.time()

try:
    if remaining.empty:
        print("No remaining selected long5 contract requests to price.")
    else:
        for i, (_, row) in enumerate(remaining.iterrows(), start=1):
            result = price_contract(row)
            new_rows.append(result)
            run_rows.append(result)

            if i % PROGRESS_EVERY == 0 or i == 1:
                elapsed = time.time() - start_time
                priced_count = sum(1 for r in run_rows if r.get("price_found") is True)
                miss_count = i - priced_count
                rate = i / elapsed if elapsed > 0 else np.nan
                remaining_count = len(remaining) - i
                eta_minutes = remaining_count / rate / 60 if rate and rate > 0 else np.nan

                print(
                    f"[{i:>5,}/{len(remaining):,}] "
                    f"priced={priced_count:,} "
                    f"missing_or_failed={miss_count:,} "
                    f"last_status={result['price_status']} "
                    f"last_http={result['status_code']} "
                    f"rate={rate:.2f}/sec "
                    f"ETA={eta_minutes:.1f} min"
                )

            if i % CHECKPOINT_EVERY == 0:
                new_df = pd.DataFrame(new_rows)

                if cache.empty:
                    cache = new_df.copy()
                else:
                    cache = pd.concat([cache, new_df], ignore_index=True)

                run_audit_df = pd.DataFrame(run_rows)
                cache = save_checkpoint(cache, run_audit_df=run_audit_df)
                new_rows = []

                checkpoint_summary = (
                    cache
                    .groupby(["price_found", "price_status"], dropna=False)
                    .size()
                    .reset_index(name="count")
                    .sort_values(["price_found", "price_status"])
                )

                print()
                print(f"Checkpoint saved after {i:,} attempts this run.")
                display(checkpoint_summary)
                print()

            if REQUEST_SLEEP_SECONDS and REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

except KeyboardInterrupt:
    print()
    print("KeyboardInterrupt received. Saving checkpoint before stopping...")

except Exception as e:
    print()
    print("Unexpected exception. Saving checkpoint before raising...")
    print(repr(e))
    print(traceback.format_exc())
    raise

finally:
    if new_rows:
        new_df = pd.DataFrame(new_rows)

        if cache.empty:
            cache = new_df.copy()
        else:
            cache = pd.concat([cache, new_df], ignore_index=True)

    run_audit_df = pd.DataFrame(run_rows)
    cache = save_checkpoint(cache, run_audit_df=run_audit_df)


# --------------------------------------------------------------------------------------------------
# Final diagnostics
# --------------------------------------------------------------------------------------------------

elapsed = time.time() - start_time
final_cache = pd.read_parquet(PRICE_CACHE_PATH)

status_summary = (
    final_cache
    .groupby(["price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["price_found", "price_status"])
)

status_code_summary = (
    final_cache
    .groupby(["status_code", "price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["status_code", "price_found", "price_status"])
)

date_coverage = (
    final_cache
    .groupby("trade_date_str")
    .agg(
        cached_contracts=("contract_request_id", "size"),
        priced_contracts=("price_found", "sum"),
    )
    .reset_index()
)

date_coverage["priced_rate"] = (
    date_coverage["priced_contracts"] / date_coverage["cached_contracts"]
)

missing_or_failed = final_cache.loc[
    final_cache["price_found"] != True
].copy()

missing_or_failed_preview = (
    missing_or_failed
    .sort_values(["trade_date_str", "expiration_str", "strike_listed"])
    .head(50)
)

# Save audit snapshots
status_summary_path = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_status_summary_{timestamp}.csv"
status_code_summary_path = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_status_code_summary_{timestamp}.csv"
date_coverage_path = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_date_coverage_{timestamp}.csv"
missing_preview_path = AUDIT_DIR / f"02E_long5_selected_contract_price_cache_missing_preview_{timestamp}.csv"

status_summary.to_csv(status_summary_path, index=False)
status_code_summary.to_csv(status_code_summary_path, index=False)
date_coverage.to_csv(date_coverage_path, index=False)
missing_or_failed_preview.to_csv(missing_preview_path, index=False)

manifest = {
    "cell": "2E_long5",
    "description": "Selected SPY long5 contract price cache using ThetaData v3 /v3/option/history/greeks/eod",
    "created_at": timestamp,
    "base_url": BASE_URL,
    "endpoint": ENDPOINT,
    "selected_unique_contract_plan_path": str(SELECTED_UNIQUE_CONTRACT_PLAN_PATH),
    "price_cache_path": str(PRICE_CACHE_PATH),
    "price_cache_csv_snapshot_path": str(PRICE_CACHE_CSV_SNAPSHOT_PATH),
    "run_audit_path": str(RUN_AUDIT_PATH),
    "status_summary_path": str(status_summary_path),
    "status_code_summary_path": str(status_code_summary_path),
    "date_coverage_path": str(date_coverage_path),
    "missing_preview_path": str(missing_preview_path),
    "runtime_seconds": elapsed,
    "plan_rows": int(len(plan)),
    "cache_rows_final": int(len(final_cache)),
    "run_attempts": int(len(run_audit_df)),
    "run_priced": int(run_audit_df["price_found"].sum()) if not run_audit_df.empty and "price_found" in run_audit_df.columns else 0,
    "final_priced": int(final_cache["price_found"].sum()) if "price_found" in final_cache.columns else 0,
    "final_missing_or_failed": int((final_cache["price_found"] != True).sum()) if "price_found" in final_cache.columns else int(len(final_cache)),
    "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
    "retry_failed_or_missing": RETRY_FAILED_OR_MISSING,
}

with open(RUN_MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Cell 2E long5 complete")
print("=" * 100)
print(f"Runtime minutes:       {elapsed / 60:.2f}")
print(f"Run attempts:          {len(run_audit_df):,}")
print(f"Final cache rows:      {len(final_cache):,}")
print(f"Final priced rows:     {int(final_cache['price_found'].sum()):,}")
print(f"Final missing/failed:  {int((final_cache['price_found'] != True).sum()):,}")
print()

print("Status summary:")
display(status_summary)

print()
print("Status-code summary:")
display(status_code_summary)

print()
print("Date coverage preview:")
display(date_coverage.head(30))

print()
print("Missing/failed preview:")
if missing_or_failed_preview.empty:
    print("No missing/failed rows in cache so far.")
else:
    display(
        missing_or_failed_preview[
            [
                "contract_request_id",
                "trade_date_str",
                "expiration_str",
                "strike_listed",
                "right",
                "status_code",
                "price_status",
                "raw_text_preview",
            ]
        ].head(50)
    )

print()
print("Saved:")
print(f"  Price cache parquet:       {PRICE_CACHE_PATH}")
print(f"  Price cache CSV snapshot:  {PRICE_CACHE_CSV_SNAPSHOT_PATH}")
print(f"  Run audit:                 {RUN_AUDIT_PATH}")
print(f"  Status summary:            {status_summary_path}")
print(f"  Status-code summary:       {status_code_summary_path}")
print(f"  Date coverage:             {date_coverage_path}")
print(f"  Missing preview:           {missing_preview_path}")
print(f"  Manifest:                  {RUN_MANIFEST_PATH}")

if len(final_cache) < len(plan):
    print()
    print("Result: Cache is partially complete. Review long5 first-batch coverage; then rerun with MAX_REQUESTS_THIS_RUN = None to finish.")
else:
    print()
    print("Result: Selected long5 contract cache completed. Next cell should join prices back to selected trades and compute real spread P&L.")

Cell 2E — Price selected long5 unique contract request plan
Selected unique contract plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_v1.parquet
Price cache path:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_v1.parquet
ThetaData base URL:            http://127.0.0.1:25503
Endpoint:                      /v3/option/history/greeks/eod
Checkpoint every:              100 attempts
Max requests this run:         250
Retry failed/missing:          False

Loaded selected long5 contract plan/cache
Plan rows:                 3,437
Existing cache rows:       0
Completed/skipped IDs:     0
Remaining full:            3,437
Remaining this run:        250

[    1/250] priced=1 missing_or_failed=0 last_status=priced last_http=200 rate=3.63/sec ETA=1.1 min
[   25/250] priced=24 missing_or_failed=1 last_status=priced last_http=200

,price_found,price_status,count
0,False,http_472,1
1,True,priced,99



[  125/250] priced=124 missing_or_failed=1 last_status=priced last_http=200 rate=8.85/sec ETA=0.2 min
[  150/250] priced=147 missing_or_failed=3 last_status=priced last_http=200 rate=8.60/sec ETA=0.2 min
[  175/250] priced=171 missing_or_failed=4 last_status=priced last_http=200 rate=8.77/sec ETA=0.1 min
[  200/250] priced=194 missing_or_failed=6 last_status=priced last_http=200 rate=8.86/sec ETA=0.1 min

Checkpoint saved after 200 attempts this run.


,price_found,price_status,count
0,False,http_472,6
1,True,priced,194



[  225/250] priced=215 missing_or_failed=10 last_status=priced last_http=200 rate=8.62/sec ETA=0.0 min
[  250/250] priced=231 missing_or_failed=19 last_status=priced last_http=200 rate=8.85/sec ETA=0.0 min

Cell 2E long5 complete
Runtime minutes:       0.47
Run attempts:          250
Final cache rows:      250
Final priced rows:     231
Final missing/failed:  19

Status summary:


...


Date coverage preview:


,trade_date_str,cached_contracts,priced_contracts,priced_rate
0,2020-12-31,6,5,0.833333
1,2021-01-06,2,2,1.000000
2,2021-01-11,2,2,1.000000
3,2021-01-19,4,4,1.000000
4,2021-01-25,8,8,1.000000
5,2021-01-26,10,10,1.000000
6,2021-02-16,10,10,1.000000
7,2021-02-17,10,10,1.000000
8,2021-02-18,6,6,1.000000
9,2021-02-19,4,4,1.000000



Missing/failed preview:


,contract_request_id,trade_date_str,expiration_str,strike_listed,right,status_code,price_status,raw_text_preview
5,SPY_20201231_20210205_450.0_C,2020-12-31,2021-02-05,450.0,C,472,http_472,No data found for your request
138,SPY_20211026_20211203_479.0_C,2021-10-26,2021-12-03,479.0,C,472,http_472,No data found for your request
139,SPY_20211026_20211203_525.0_C,2021-10-26,2021-12-03,525.0,C,472,http_472,No data found for your request
159,SPY_20211102_20211126_515.0_C,2021-11-02,2021-11-26,515.0,C,472,http_472,No data found for your request
179,SPY_20211104_20211203_525.0_C,2021-11-04,2021-12-03,525.0,C,472,http_472,No data found for your request
183,SPY_20211104_20211210_535.0_C,2021-11-04,2021-12-10,535.0,C,472,http_472,No data found for your request
202,SPY_20211108_20211210_535.0_C,2021-11-08,2021-12-10,535.0,C,472,http_472,No data found for your request
220,SPY_20211117_20211224_493.0_C,2021-11-17,2021-12-24,493.0,C,472,http_472,No data found for your request
221,SPY_20211117_20211224_540.0_C,2021-11-17,2021-12-24,540.0,C,472,http_472,No data found for your request
223,SPY_20211118_20211210_515.0_C,2021-11-18,2021-12-10,515.0,C,472,http_472,No data found for your request



Saved:
  Price cache parquet:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_v1.parquet
  Price cache CSV snapshot:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_call_sleeve_corsi_call_selected_contract_price_cache_long5_latest.csv
  Run audit:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_selected_contract_price_cache_run_audit_20260710_110943.csv
  Status summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_selected_contract_price_cache_status_summary_20260710_110943.csv
  Status-code summary:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_selected_contract_price_cache_status_code_summary_20260710_110943.csv
  Date coverage:             C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_selected_contract_price_cache_date_coverage_20260710_110943.csv
  Missing preview

In [21]:
# Cell 2D.2 — Rebuild selected-trade pricing universe with shared holiday-aware expiration logic
# Purpose:
#   Rebuild the selected-trade universe using the same holiday-aware expiration convention
#   that should be used across VIX-style measure construction, trade construction, and
#   future multi-ticker workflows.
#
# Current trade convention:
#   short 1SD call: ceil raw strike to nearest $1
#   long 3SD call:  round raw strike to nearest $5
#
# Expiration convention:
#   1. target_date = trade_date + tenor calendar days
#   2. nominal_expiration = first calendar Friday on or after target_date
#   3. if nominal Friday is a trading day, use it
#   4. if nominal Friday is a market holiday, use the prior trading day
#
# Calendar source:
#   Uses SPY EOD trading dates as the exchange-trading-day calendar for this notebook.
#   This is ticker-neutral logic conceptually: future ticker work should pass in the same
#   shared exchange trading calendar used by the VIX-style construction process.
#
# This creates fresh *_long5_cal_v1 files.

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import json
import math

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_TRADES_SOURCE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet"
)

SPY_EOD_PATH = (
    PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_eod_prices_v1.parquet"
)

SYMBOL = "SPY"
RIGHT = "C"
PRICING_ENDPOINT = "/v3/option/history/greeks/eod"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2D.2 — Rebuild selected-trade pricing universe with shared holiday-aware calendar logic")
print("=" * 100)
print(f"Input selected trades: {SELECTED_TRADES_SOURCE_PATH}")
print(f"Trading calendar/EOD:  {SPY_EOD_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def infer_date_col(df):
    for c in ["date", "trade_date", "timestamp", "datetime"]:
        if c in df.columns:
            return c
    raise ValueError(f"Could not infer date column from columns: {list(df.columns)}")

def infer_close_col(df):
    for c in ["spy_close", "close", "Close", "adj_close", "Adj Close", "last"]:
        if c in df.columns:
            return c
    raise ValueError(f"Could not infer close column from columns: {list(df.columns)}")

def ceil_to_increment(x, increment):
    if pd.isna(x):
        return np.nan
    return float(math.ceil(float(x) / float(increment)) * float(increment))

def round_to_nearest_increment_half_up(x, increment):
    if pd.isna(x):
        return np.nan
    return float(math.floor(float(x) / float(increment) + 0.5) * float(increment))

def first_friday_on_or_after(dt):
    d = pd.Timestamp(dt).normalize()
    days_forward = (4 - d.weekday()) % 7
    return d + pd.Timedelta(days=days_forward)

def make_trade_id(trade_date, tenor, expiration_date, short_strike, long_strike):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"T{int(tenor):02d}_"
        f"EXP{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"S{float(short_strike):.1f}_"
        f"L{float(long_strike):.1f}"
    )

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def build_exchange_calendar_from_eod(eod_path):
    eod = pd.read_parquet(eod_path).copy()
    date_col = infer_date_col(eod)
    close_col = infer_close_col(eod)

    prices = eod[[date_col, close_col]].copy()
    prices.columns = ["price_date", "underlying_close"]
    prices["price_date"] = pd.to_datetime(prices["price_date"], errors="coerce").dt.normalize()
    prices["underlying_close"] = pd.to_numeric(prices["underlying_close"], errors="coerce")

    prices = (
        prices
        .dropna(subset=["price_date", "underlying_close"])
        .sort_values("price_date")
        .drop_duplicates("price_date", keep="last")
        .reset_index(drop=True)
    )

    trading_days = pd.DatetimeIndex(prices["price_date"]).sort_values().unique()

    return prices, trading_days

def holiday_aware_friday_expiration(target_date, trading_days):
    """
    Ticker-neutral expiration utility.

    Pick the first Friday on or after target_date.
    If that Friday is not a trading day, use the prior trading day.
    """
    nominal_friday = first_friday_on_or_after(target_date)

    if nominal_friday in trading_days:
        actual_expiration = nominal_friday
        holiday_adjusted = False
    else:
        prior_days = trading_days[trading_days < nominal_friday]
        if len(prior_days) == 0:
            actual_expiration = pd.NaT
            holiday_adjusted = True
        else:
            actual_expiration = pd.Timestamp(prior_days[-1])
            holiday_adjusted = True

    return pd.Series({
        "nominal_friday_expiration": nominal_friday,
        "expiration_date": actual_expiration,
        "expiration_holiday_adjusted": holiday_adjusted,
    })

def summarize_rule(g):
    g = g.sort_values("trade_date").copy()

    intrinsic = g["terminal_spread_intrinsic_before_premium"].fillna(0.0)
    intrinsic_pct = g["terminal_intrinsic_pct_width"].fillna(0.0)

    rolling_20_intrinsic = intrinsic.rolling(20, min_periods=1).sum()
    rolling_20_intrinsic_pct = intrinsic_pct.rolling(20, min_periods=1).sum()

    year_counts = g.groupby("trade_year").size()

    return pd.Series({
        "selected_trades": len(g),
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "years_traded": ",".join(map(str, sorted(g["trade_year"].dropna().astype(int).unique()))),
        "n_years_traded": g["trade_year"].nunique(),
        "median_trades_per_year": year_counts.median() if not year_counts.empty else np.nan,
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "avg_effective_dte": g["effective_dte"].mean(),
        "avg_listed_width": g["listed_width"].mean(),
        "median_listed_width": g["listed_width"].median(),
        "short_call_otm_rate": g["expired_short_call_otm"].mean(),
        "zero_intrinsic_rate": g["expired_spread_zero_intrinsic"].mean(),
        "avg_terminal_intrinsic": intrinsic.mean(),
        "median_terminal_intrinsic": intrinsic.median(),
        "p95_terminal_intrinsic": intrinsic.quantile(0.95),
        "worst_terminal_intrinsic": intrinsic.max(),
        "avg_terminal_intrinsic_pct_width": intrinsic_pct.mean(),
        "p95_terminal_intrinsic_pct_width": intrinsic_pct.quantile(0.95),
        "worst_terminal_intrinsic_pct_width": intrinsic_pct.max(),
        "worst_20_trade_intrinsic_sum": rolling_20_intrinsic.max(),
        "worst_20_trade_intrinsic_pct_width_sum": rolling_20_intrinsic_pct.max(),
        "expiry_price_exact_rate": g["expiry_price_date_exact"].mean(),
        "holiday_adjusted_expiration_count": int(g["expiration_holiday_adjusted"].sum()),
        "holiday_adjusted_expiration_rate": g["expiration_holiday_adjusted"].mean(),
    })

# --------------------------------------------------------------------------------------------------
# Load selected trades and exchange calendar
# --------------------------------------------------------------------------------------------------

selected_trades = pd.read_parquet(SELECTED_TRADES_SOURCE_PATH).copy()

selected_trades["trade_date"] = pd.to_datetime(selected_trades["trade_date"], errors="coerce").dt.normalize()
selected_trades["tenor"] = selected_trades["tenor"].astype(int)

required_cols = [
    "trade_date",
    "tenor",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "rule_id",
    "selection_rule",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
]

missing = [c for c in required_cols if c not in selected_trades.columns]
if missing:
    raise ValueError(f"Missing required columns from selected trades: {missing}")

# Preserve old expiration for audit/comparison.
if "expiration_date" in selected_trades.columns:
    selected_trades["old_expiration_date"] = pd.to_datetime(
        selected_trades["expiration_date"], errors="coerce"
    ).dt.normalize()
else:
    selected_trades["old_expiration_date"] = pd.NaT

calendar_prices, trading_days = build_exchange_calendar_from_eod(SPY_EOD_PATH)

print("=" * 100)
print("Loaded source/calendar")
print("=" * 100)
print(f"Selected-trade rows:       {len(selected_trades):,}")
print(f"Calendar price rows:       {len(calendar_prices):,}")
print(f"Trading calendar range:    {trading_days.min().date()} to {trading_days.max().date()}")
print()

# --------------------------------------------------------------------------------------------------
# Apply shared holiday-aware expiration logic
# --------------------------------------------------------------------------------------------------

selected_trades["target_expiration_date"] = (
    selected_trades["trade_date"] + pd.to_timedelta(selected_trades["tenor"], unit="D")
)

expiration_info = selected_trades["target_expiration_date"].apply(
    lambda x: holiday_aware_friday_expiration(x, trading_days)
)

selected_trades = pd.concat(
    [
        selected_trades.drop(
            columns=[
                c for c in [
                    "nominal_friday_expiration",
                    "expiration_date",
                    "expiration_holiday_adjusted",
                ]
                if c in selected_trades.columns
            ]
        ),
        expiration_info,
    ],
    axis=1,
)

selected_trades["expiration_date"] = pd.to_datetime(
    selected_trades["expiration_date"], errors="coerce"
).dt.normalize()

selected_trades["effective_dte"] = (
    selected_trades["expiration_date"] - selected_trades["trade_date"]
).dt.days

selected_trades["expiration_slippage_days"] = (
    selected_trades["expiration_date"] - selected_trades["target_expiration_date"]
).dt.days

selected_trades["expiration_changed_vs_old"] = (
    selected_trades["old_expiration_date"].notna()
    & selected_trades["expiration_date"].notna()
    & (selected_trades["old_expiration_date"] != selected_trades["expiration_date"])
)

# --------------------------------------------------------------------------------------------------
# Apply strike convention
# --------------------------------------------------------------------------------------------------

selected_trades["short_strike"] = selected_trades["short_call_1sd_raw"].apply(
    lambda x: ceil_to_increment(x, 1.0)
)

selected_trades["long_strike"] = selected_trades["long_call_3sd_raw"].apply(
    lambda x: round_to_nearest_increment_half_up(x, 5.0)
)

# Safety: long must be above short.
bad_long = selected_trades["long_strike"] <= selected_trades["short_strike"]

selected_trades.loc[bad_long, "long_strike"] = selected_trades.loc[
    bad_long, "short_strike"
].apply(
    lambda x: ceil_to_increment(float(x) + 0.01, 5.0)
)

selected_trades["listed_width"] = selected_trades["long_strike"] - selected_trades["short_strike"]

selected_trades = selected_trades.loc[
    selected_trades["expiration_date"].notna()
    & selected_trades["short_strike"].notna()
    & selected_trades["long_strike"].notna()
    & (selected_trades["listed_width"] > 0)
].copy()

# --------------------------------------------------------------------------------------------------
# Attach expiration close from the same calendar/EOD source
# --------------------------------------------------------------------------------------------------

expiry_prices = calendar_prices.rename(
    columns={
        "price_date": "expiration_date",
        "underlying_close": "expiry_spy_close",
    }
)

selected_trades = selected_trades.drop(
    columns=[c for c in ["spy_price_date", "expiry_spy_close", "expiry_price_date_exact"] if c in selected_trades.columns]
)

selected_trades = selected_trades.merge(
    expiry_prices,
    on="expiration_date",
    how="left",
)

selected_trades["spy_price_date"] = selected_trades["expiration_date"]
selected_trades["expiry_price_date_exact"] = selected_trades["expiry_spy_close"].notna()

selected_trades["short_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["short_strike"],
    0.0,
)

selected_trades["long_call_terminal_intrinsic"] = np.maximum(
    selected_trades["expiry_spy_close"] - selected_trades["long_strike"],
    0.0,
)

selected_trades["terminal_spread_intrinsic_before_premium"] = (
    selected_trades["short_call_terminal_intrinsic"]
    - selected_trades["long_call_terminal_intrinsic"]
)

selected_trades["terminal_intrinsic_pct_width"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"]
    / selected_trades["listed_width"]
)

selected_trades["expired_short_call_otm"] = (
    selected_trades["expiry_spy_close"] <= selected_trades["short_strike"]
)

selected_trades["expired_spread_zero_intrinsic"] = (
    selected_trades["terminal_spread_intrinsic_before_premium"] <= 0
)

selected_trades["trade_year"] = selected_trades["trade_date"].dt.year

selected_trades["selected_trade_id"] = selected_trades.apply(
    lambda r: make_trade_id(
        r["trade_date"],
        r["tenor"],
        r["expiration_date"],
        r["short_strike"],
        r["long_strike"],
    ),
    axis=1,
)

# --------------------------------------------------------------------------------------------------
# Rebuild unique selected trade universe
# --------------------------------------------------------------------------------------------------

rule_usage = (
    selected_trades
    .groupby("selected_trade_id")
    .agg(
        selected_by_rule_count=("rule_id", "nunique"),
        selection_rules=("selection_rule", lambda x: ",".join(sorted(set(map(str, x))))),
        min_z_threshold=("z_threshold_shared", "min"),
        max_z_threshold=("z_threshold_shared", "max"),
        min_rsi_floor=("rsi_floor", "min"),
        max_rsi_floor=("rsi_floor", "max"),
        vrp_log_floor_labels=("vrp_log_floor_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

trade_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "vix_style_vol_pct",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "z_avg",
    "z_min",
    "rsi14",
    "target_expiration_date",
    "nominal_friday_expiration",
    "old_expiration_date",
    "expiration_date",
    "expiration_changed_vs_old",
    "expiration_holiday_adjusted",
    "effective_dte",
    "expiration_slippage_days",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "short_strike",
    "long_strike",
    "listed_width",
    "spy_price_date",
    "expiry_spy_close",
    "expiry_price_date_exact",
    "short_call_terminal_intrinsic",
    "long_call_terminal_intrinsic",
    "terminal_spread_intrinsic_before_premium",
    "terminal_intrinsic_pct_width",
    "expired_short_call_otm",
    "expired_spread_zero_intrinsic",
]

trade_cols = [c for c in trade_cols if c in selected_trades.columns]

unique_selected_trades = (
    selected_trades[trade_cols]
    .drop_duplicates("selected_trade_id")
    .merge(rule_usage, on="selected_trade_id", how="left")
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Rebuild selected option leg request plan
# --------------------------------------------------------------------------------------------------

short_legs = unique_selected_trades.copy()
short_legs["leg_label"] = "short_call_1sd"
short_legs["leg_side"] = "sell"
short_legs["price_field_for_entry"] = "bid"
short_legs["symbol"] = SYMBOL
short_legs["right"] = RIGHT
short_legs["strike_listed"] = short_legs["short_strike"]

long_legs = unique_selected_trades.copy()
long_legs["leg_label"] = "long_call_3sd"
long_legs["leg_side"] = "buy"
long_legs["price_field_for_entry"] = "ask"
long_legs["symbol"] = SYMBOL
long_legs["right"] = RIGHT
long_legs["strike_listed"] = long_legs["long_strike"]

selected_leg_plan = pd.concat([short_legs, long_legs], ignore_index=True)

selected_leg_plan["trade_date_str"] = selected_leg_plan["trade_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_str"] = selected_leg_plan["expiration_date"].dt.strftime("%Y-%m-%d")
selected_leg_plan["expiration_yyyymmdd"] = selected_leg_plan["expiration_date"].dt.strftime("%Y%m%d")
selected_leg_plan["endpoint"] = PRICING_ENDPOINT

selected_leg_plan["contract_request_id"] = selected_leg_plan.apply(
    lambda r: make_contract_request_id(
        r["symbol"],
        r["trade_date"],
        r["expiration_date"],
        r["strike_listed"],
        r["right"],
    ),
    axis=1,
)

contract_usage = (
    selected_leg_plan
    .groupby("contract_request_id")
    .agg(
        mapped_trade_legs=("selected_trade_id", "size"),
        mapped_unique_trades=("selected_trade_id", "nunique"),
        mapped_tenors=("tenor", lambda x: ",".join(map(str, sorted(set(x.astype(int)))))),
        mapped_leg_labels=("leg_label", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

unique_contract_request_plan = (
    selected_leg_plan[
        [
            "contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "strike_listed",
            "endpoint",
        ]
    ]
    .drop_duplicates()
    .merge(contract_usage, on="contract_request_id", how="left")
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Rule summaries
# --------------------------------------------------------------------------------------------------

rule_summary = (
    selected_trades
    .groupby(
        [
            "rule_id",
            "z_threshold_shared",
            "rsi_floor",
            "vrp_log_floor_label",
            "selection_rule",
        ],
        dropna=False,
    )
    .apply(summarize_rule, include_groups=False)
    .reset_index()
)

rule_year_summary = (
    selected_trades
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_trades=("selected_trade_id", "size"),
        short_call_otm_rate=("expired_short_call_otm", "mean"),
        zero_intrinsic_rate=("expired_spread_zero_intrinsic", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
        holiday_adjusted_expiration_count=("expiration_holiday_adjusted", "sum"),
    )
    .reset_index()
    .sort_values(["rule_id", "trade_year"])
)

# --------------------------------------------------------------------------------------------------
# Save long5_cal outputs
# --------------------------------------------------------------------------------------------------

selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet"
)

unique_selected_trades_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet"
)

selected_leg_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet"
)

unique_contract_request_plan_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_cal_v1.parquet"
)

rule_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_preprice_summary_long5_cal_v1.parquet"
)

rule_year_summary_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_preprice_summary_long5_cal_v1.parquet"
)

selected_trades.to_parquet(selected_trades_path, index=False)
unique_selected_trades.to_parquet(unique_selected_trades_path, index=False)
selected_leg_plan.to_parquet(selected_leg_plan_path, index=False)
unique_contract_request_plan.to_parquet(unique_contract_request_plan_path, index=False)
rule_summary.to_parquet(rule_summary_path, index=False)
rule_year_summary.to_parquet(rule_year_summary_path, index=False)

selected_trades_csv = AUDIT_DIR / f"02D2_rule_selected_trades_unpriced_long5_cal_{timestamp}.csv"
unique_selected_trades_csv = AUDIT_DIR / f"02D2_unique_selected_trade_universe_long5_cal_{timestamp}.csv"
selected_leg_plan_csv = AUDIT_DIR / f"02D2_selected_leg_request_plan_long5_cal_{timestamp}.csv"
unique_contract_request_plan_csv = AUDIT_DIR / f"02D2_selected_unique_contract_request_plan_long5_cal_{timestamp}.csv"
rule_summary_csv = AUDIT_DIR / f"02D2_rule_preprice_summary_long5_cal_{timestamp}.csv"
rule_year_summary_csv = AUDIT_DIR / f"02D2_rule_year_preprice_summary_long5_cal_{timestamp}.csv"

selected_trades.to_csv(selected_trades_csv, index=False)
unique_selected_trades.to_csv(unique_selected_trades_csv, index=False)
selected_leg_plan.to_csv(selected_leg_plan_csv, index=False)
unique_contract_request_plan.to_csv(unique_contract_request_plan_csv, index=False)
rule_summary.to_csv(rule_summary_csv, index=False)
rule_year_summary.to_csv(rule_year_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02D2_selected_trade_pricing_universe_long5_cal_manifest_{timestamp}.json"

manifest = {
    "cell": "2D.2",
    "description": "Rebuild selected-trade pricing universe with long 3SD strike rounded to nearest $5 and shared holiday-aware expiration logic",
    "created_at": timestamp,
    "input_selected_trades_source_path": str(SELECTED_TRADES_SOURCE_PATH),
    "trading_calendar_source_path": str(SPY_EOD_PATH),
    "pricing_endpoint_for_next_cell": PRICING_ENDPOINT,
    "strike_convention": {
        "short_call_1sd": "ceil raw strike to nearest $1",
        "long_call_3sd": "round raw strike to nearest $5 using half-up rounding",
    },
    "expiration_convention": {
        "target_date": "trade_date + tenor calendar days",
        "nominal_expiration": "first calendar Friday on or after target_date",
        "holiday_adjustment": "if nominal Friday is not a trading day, use prior trading day",
        "calendar_source": "same exchange trading calendar concept used by VIX-style construction; this notebook uses SPY EOD dates as calendar source",
    },
    "counts": {
        "rule_selected_rows": int(len(selected_trades)),
        "unique_selected_trades": int(len(unique_selected_trades)),
        "selected_leg_rows": int(len(selected_leg_plan)),
        "unique_contract_requests": int(len(unique_contract_request_plan)),
        "rule_count": int(rule_summary["rule_id"].nunique()),
        "expiration_changed_vs_old_rows": int(selected_trades["expiration_changed_vs_old"].sum()),
        "holiday_adjusted_expiration_rows": int(selected_trades["expiration_holiday_adjusted"].sum()),
    },
    "outputs": {
        "selected_trades_path": str(selected_trades_path),
        "unique_selected_trades_path": str(unique_selected_trades_path),
        "selected_leg_plan_path": str(selected_leg_plan_path),
        "unique_contract_request_plan_path": str(unique_contract_request_plan_path),
        "rule_summary_path": str(rule_summary_path),
        "rule_year_summary_path": str(rule_year_summary_path),
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Diagnostics
# --------------------------------------------------------------------------------------------------

summary_counts = pd.DataFrame([{
    "rule_selected_rows": len(selected_trades),
    "unique_selected_trades": len(unique_selected_trades),
    "selected_leg_rows": len(selected_leg_plan),
    "unique_contract_requests_to_price": len(unique_contract_request_plan),
    "rule_count": rule_summary["rule_id"].nunique(),
    "date_min": selected_trades["trade_date"].min().date(),
    "date_max": selected_trades["trade_date"].max().date(),
    "expiration_changed_vs_old_rows": int(selected_trades["expiration_changed_vs_old"].sum()),
    "holiday_adjusted_expiration_rows": int(selected_trades["expiration_holiday_adjusted"].sum()),
    "expiry_price_exact_rate": selected_trades["expiry_price_date_exact"].mean(),
}])

strike_check = pd.DataFrame([{
    "short_non_integer_count": int((selected_trades["short_strike"] % 1 != 0).sum()),
    "long_not_multiple_of_5_count": int((selected_trades["long_strike"] % 5 != 0).sum()),
    "min_width": selected_trades["listed_width"].min(),
    "median_width": selected_trades["listed_width"].median(),
    "max_width": selected_trades["listed_width"].max(),
}])

expiration_change_preview = (
    selected_trades.loc[selected_trades["expiration_changed_vs_old"]]
    [
        [
            "trade_date",
            "tenor",
            "target_expiration_date",
            "nominal_friday_expiration",
            "old_expiration_date",
            "expiration_date",
            "expiration_holiday_adjusted",
            "effective_dte",
        ]
    ]
    .drop_duplicates()
    .sort_values(["trade_date", "tenor"])
    .head(50)
)

holiday_adjusted_preview = (
    selected_trades.loc[selected_trades["expiration_holiday_adjusted"]]
    [
        [
            "trade_date",
            "tenor",
            "target_expiration_date",
            "nominal_friday_expiration",
            "expiration_date",
            "effective_dte",
        ]
    ]
    .drop_duplicates()
    .sort_values(["trade_date", "tenor"])
    .head(50)
)

print()
print("=" * 100)
print("Long5 calendar-consistent selected pricing universe summary")
print("=" * 100)
display(summary_counts)

print()
print("=" * 100)
print("Strike convention check")
print("=" * 100)
display(strike_check)

print()
print("=" * 100)
print("Expiration changes vs old naïve logic")
print("=" * 100)
if expiration_change_preview.empty:
    print("No expiration changes versus old selected-trade table.")
else:
    display(expiration_change_preview)

print()
print("=" * 100)
print("Holiday-adjusted expiration preview")
print("=" * 100)
if holiday_adjusted_preview.empty:
    print("No holiday-adjusted expirations found.")
else:
    display(holiday_adjusted_preview)

print()
print("=" * 100)
print("Unique contract request plan preview")
print("=" * 100)
display(
    unique_contract_request_plan[
        [
            "contract_request_id",
            "symbol",
            "trade_date_str",
            "expiration_str",
            "strike_listed",
            "right",
            "mapped_unique_trades",
            "mapped_tenors",
            "mapped_leg_labels",
        ]
    ].head(40)
)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Selected trades by rule:          {selected_trades_path}")
print(f"Unique selected trade universe:   {unique_selected_trades_path}")
print(f"Selected leg request plan:        {selected_leg_plan_path}")
print(f"Unique contract request plan:     {unique_contract_request_plan_path}")
print(f"Rule pre-price summary:           {rule_summary_path}")
print(f"Rule-year pre-price summary:      {rule_year_summary_path}")
print(f"Manifest:                         {manifest_path}")

print()
print("Result: long5 calendar-consistent selected-trade pricing universe rebuilt.")
print("Next: price the long5_cal unique contract plan with a fresh 2E cache.")

Cell 2D.2 — Rebuild selected-trade pricing universe with shared holiday-aware calendar logic
Input selected trades: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_v1.parquet
Trading calendar/EOD:  C:\Users\patri\vrp_project\data\processed\market_data\spy_eod_prices_v1.parquet

Loaded source/calendar
Selected-trade rows:       155,550
Calendar price rows:       2,140
Trading calendar range:    2018-01-02 to 2026-07-09


Long5 calendar-consistent selected pricing universe summary


,rule_selected_rows,unique_selected_trades,selected_leg_rows,unique_contract_requests_to_price,rule_count,date_min,date_max,expiration_changed_vs_old_rows,holiday_adjusted_expiration_rows,expiry_price_exact_rate
0,155550,1741,3482,3437,840,2020-12-31,2026-06-02,3808,3808,1.0



Strike convention check


,short_non_integer_count,long_not_multiple_of_5_count,min_width,median_width,max_width
0,0,0,14.0,31.0,87.0



Expiration changes vs old naïve logic


...


Holiday-adjusted expiration preview


,trade_date,tenor,target_expiration_date,nominal_friday_expiration,expiration_date,effective_dte
33,2021-11-17,33,2021-12-20,2021-12-24,2021-12-23,36
34,2021-11-18,33,2021-12-21,2021-12-24,2021-12-23,35
35,2021-11-19,30,2021-12-19,2021-12-24,2021-12-23,34
1847,2021-11-19,33,2021-12-22,2021-12-24,2021-12-23,34
116448,2021-11-22,30,2021-12-22,2021-12-24,2021-12-23,31
38,2021-11-24,24,2021-12-18,2021-12-24,2021-12-23,29
134270,2021-11-24,27,2021-12-21,2021-12-24,2021-12-23,29
36360,2022-03-18,24,2022-04-11,2022-04-15,2022-04-14,27
134271,2022-03-21,21,2022-04-11,2022-04-15,2022-04-14,24
116452,2022-03-22,24,2022-04-15,2022-04-15,2022-04-14,23



Unique contract request plan preview


,contract_request_id,symbol,trade_date_str,expiration_str,strike_listed,right,mapped_unique_trades,mapped_tenors,mapped_leg_labels
0,SPY_20201231_20210115_386.0_C,SPY,2020-12-31,2021-01-15,386.0,C,1,9,short_call_1sd
1,SPY_20201231_20210115_410.0_C,SPY,2020-12-31,2021-01-15,410.0,C,1,9,long_call_3sd
2,SPY_20201231_20210129_397.0_C,SPY,2020-12-31,2021-01-29,397.0,C,1,27,short_call_1sd
3,SPY_20201231_20210129_440.0_C,SPY,2020-12-31,2021-01-29,440.0,C,1,27,long_call_3sd
4,SPY_20201231_20210205_400.0_C,SPY,2020-12-31,2021-02-05,400.0,C,1,33,short_call_1sd
5,SPY_20201231_20210205_450.0_C,SPY,2020-12-31,2021-02-05,450.0,C,1,33,long_call_3sd
6,SPY_20210106_20210115_389.0_C,SPY,2021-01-06,2021-01-15,389.0,C,1,9,short_call_1sd
7,SPY_20210106_20210115_420.0_C,SPY,2021-01-06,2021-01-15,420.0,C,1,9,long_call_3sd
8,SPY_20210111_20210122_392.0_C,SPY,2021-01-11,2021-01-22,392.0,C,1,9,short_call_1sd
9,SPY_20210111_20210122_420.0_C,SPY,2021-01-11,2021-01-22,420.0,C,1,9,long_call_3sd



Saved
Selected trades by rule:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet
Unique selected trade universe:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet
Selected leg request plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet
Unique contract request plan:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_cal_v1.parquet
Rule pre-price summary:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_preprice_summary_long5_cal_v1.parquet
Rule-year pre-price summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_preprice_su

In [24]:
# Cell 2E — Price selected long5_cal unique contract request plan with checkpointing
# Purpose:
#   Price only the selected-trade contract universe rebuilt in Cell 2D.2.
#
# Current trade convention:
#   short 1SD call: ceil raw strike to nearest $1
#   long 3SD call:  round raw strike to nearest $5
#   expiration:      shared holiday-aware calendar logic
#
# Input:
#   call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_cal_v1.parquet
#
# Output:
#   call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet
#
# First run:
#   MAX_REQUESTS_THIS_RUN = None
#
# After this batch looks clean:
#   change MAX_REQUESTS_THIS_RUN = None and rerun to finish the full cache.

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
from datetime import datetime
import json
import time
import traceback

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_UNIQUE_CONTRACT_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_cal_v1.parquet"
)

PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet"
)

PRICE_CACHE_CSV_SNAPSHOT_PATH = (
    AUDIT_DIR / "02E_call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_latest.csv"
)

BASE_URL = "http://127.0.0.1:25503"
ENDPOINT = "/v3/option/history/greeks/eod"

# Runtime controls
CHECKPOINT_EVERY = 100
PROGRESS_EVERY = 25
REQUEST_SLEEP_SECONDS = 0.02

# Keep 250 for validation. Set to None after first batch looks clean.
MAX_REQUESTS_THIS_RUN = None

# Usually False. If True, only already-priced rows are skipped; failed/missing rows are retried.
RETRY_FAILED_OR_MISSING = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_AUDIT_PATH = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_run_audit_{timestamp}.csv"
RUN_MANIFEST_PATH = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_manifest_{timestamp}.json"

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2E — Price selected long5_cal unique contract request plan")
print("=" * 100)
print(f"Selected unique contract plan: {SELECTED_UNIQUE_CONTRACT_PLAN_PATH}")
print(f"Price cache path:              {PRICE_CACHE_PATH}")
print(f"ThetaData base URL:            {BASE_URL}")
print(f"Endpoint:                      {ENDPOINT}")
print(f"Checkpoint every:              {CHECKPOINT_EVERY:,} attempts")
print(f"Max requests this run:         {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed/missing:          {RETRY_FAILED_OR_MISSING}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    # JSON fallback
    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None

                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break

                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)

                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    # CSV is expected for this endpoint with format=csv
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    # NDJSON fallback
    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})


def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None


def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "created": ["created"],
        "last_trade": ["last_trade"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "bid_size": ["bid_size"],
        "ask_size": ["ask_size"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close"],
        "volume": ["volume"],
        "count": ["count"],
        "underlying_price": ["underlying_price"],
        "underlying_timestamp": ["underlying_timestamp"],
        "implied_vol": ["implied_vol", "iv"],
        "iv_error": ["iv_error"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
        "rho": ["rho"],
        "epsilon": ["epsilon"],
        "lambda": ["lambda"],
        "omega": ["omega"],
        "vanna": ["vanna"],
        "charm": ["charm"],
        "vomma": ["vomma"],
        "veta": ["veta"],
        "vera": ["vera"],
        "speed": ["speed"],
        "zomma": ["zomma"],
        "color": ["color"],
        "ultima": ["ultima"],
        "d1": ["d1"],
        "d2": ["d2"],
        "dual_delta": ["dual_delta"],
        "dual_gamma": ["dual_gamma"],
    }

    rename_map = {}

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    numeric_cols = [
        "strike", "bid", "ask", "mid", "bid_size", "ask_size",
        "open", "high", "low", "close", "volume", "count",
        "underlying_price", "implied_vol", "iv_error",
        "delta", "theta", "vega", "gamma", "rho",
        "epsilon", "lambda", "omega", "vanna", "charm",
        "vomma", "veta", "vera", "speed", "zomma",
        "color", "ultima", "d1", "d2", "dual_delta", "dual_gamma",
    ]

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out


def atomic_write_parquet(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)


def atomic_write_csv(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def build_empty_cache():
    return pd.DataFrame(columns=[
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "endpoint",
        "status_code",
        "http_ok",
        "response_rows",
        "response_cols",
        "response_columns",
        "price_found",
        "price_status",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "timestamp",
        "created",
        "last_trade",
        "underlying_price",
        "underlying_timestamp",
        "implied_vol",
        "iv_error",
        "delta",
        "theta",
        "vega",
        "gamma",
        "rho",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "count",
        "raw_text_preview",
        "error",
        "request_url",
        "pulled_at",
    ])


def get_completed_ids(cache_df):
    if cache_df.empty:
        return set()

    if RETRY_FAILED_OR_MISSING:
        if "price_found" in cache_df.columns:
            return set(cache_df.loc[cache_df["price_found"] == True, "contract_request_id"].astype(str))
        return set()

    return set(cache_df["contract_request_id"].dropna().astype(str))


def price_contract(plan_row):
    symbol = str(plan_row["symbol"]).upper()
    right = normalize_right(plan_row["right"])
    strike = float(plan_row["strike_listed"])
    trade_date_str = str(plan_row["trade_date_str"])
    expiration_str = str(plan_row["expiration_str"])

    params = {
        "symbol": symbol,
        "right": right,
        "strike": strike,
        "expiration": expiration_str,
        "start_date": trade_date_str,
        "end_date": trade_date_str,
        "format": "csv",
    }

    url = BASE_URL.rstrip("/") + ENDPOINT
    pulled_at = datetime.now().isoformat(timespec="seconds")

    base_result = {
        "contract_request_id": str(plan_row["contract_request_id"]),
        "symbol": symbol,
        "right": right,
        "trade_date": pd.Timestamp(plan_row["trade_date"]),
        "trade_date_str": trade_date_str,
        "expiration_date": pd.Timestamp(plan_row["expiration_date"]),
        "expiration_str": expiration_str,
        "expiration_yyyymmdd": str(plan_row["expiration_yyyymmdd"]),
        "strike_listed": strike,
        "endpoint": ENDPOINT,
        "status_code": np.nan,
        "http_ok": False,
        "response_rows": 0,
        "response_cols": 0,
        "response_columns": "",
        "price_found": False,
        "price_status": "not_attempted",
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_size": np.nan,
        "ask_size": np.nan,
        "timestamp": None,
        "created": None,
        "last_trade": None,
        "underlying_price": np.nan,
        "underlying_timestamp": None,
        "implied_vol": np.nan,
        "iv_error": np.nan,
        "delta": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "gamma": np.nan,
        "rho": np.nan,
        "open": np.nan,
        "high": np.nan,
        "low": np.nan,
        "close": np.nan,
        "volume": np.nan,
        "count": np.nan,
        "raw_text_preview": "",
        "error": None,
        "request_url": "",
        "pulled_at": pulled_at,
    }

    try:
        resp = session.get(url, params=params, timeout=(2, 45))

        base_result["status_code"] = resp.status_code
        base_result["http_ok"] = bool(resp.ok)
        base_result["raw_text_preview"] = resp.text[:500] if hasattr(resp, "text") else ""
        base_result["request_url"] = resp.url

        raw_df = response_to_dataframe(resp)
        std_df = standardize_option_rows(raw_df)

        base_result["response_rows"] = int(len(raw_df))
        base_result["response_cols"] = int(len(raw_df.columns))
        base_result["response_columns"] = ", ".join(map(str, raw_df.columns))

        if not resp.ok:
            base_result["price_status"] = f"http_{resp.status_code}"
            return base_result

        if std_df.empty:
            base_result["price_status"] = "empty_response"
            return base_result

        if not {"bid", "ask"}.issubset(std_df.columns):
            base_result["price_status"] = "missing_bid_ask_columns"
            return base_result

        valid = std_df.loc[
            std_df["bid"].notna()
            & std_df["ask"].notna()
        ].copy()

        if valid.empty:
            base_result["price_status"] = "no_valid_bid_ask"
            return base_result

        # Prefer exact requested contract when identifiers exist.
        if {"strike", "right"}.issubset(valid.columns):
            exact_mask = (
                valid["strike"].round(4).eq(round(strike, 4))
                & valid["right"].eq(right)
            )

            if "expiration" in valid.columns:
                exact_mask = exact_mask & valid["expiration"].eq(expiration_str)

            exact = valid.loc[exact_mask].copy()

            if not exact.empty:
                valid = exact

        px = valid.iloc[-1]

        base_result["price_found"] = True
        base_result["price_status"] = "priced"

        for col in [
            "bid", "ask", "mid", "bid_size", "ask_size",
            "timestamp", "created", "last_trade",
            "underlying_price", "underlying_timestamp",
            "implied_vol", "iv_error",
            "delta", "theta", "vega", "gamma", "rho",
            "open", "high", "low", "close", "volume", "count",
        ]:
            if col in valid.columns:
                base_result[col] = px.get(col, base_result[col])

        return base_result

    except Exception as e:
        base_result["price_status"] = "exception"
        base_result["error"] = repr(e)
        base_result["raw_text_preview"] = traceback.format_exc()[:500]
        return base_result


def save_checkpoint(cache_df, run_audit_df=None):
    cache_df = cache_df.copy()

    if not cache_df.empty:
        cache_df["contract_request_id"] = cache_df["contract_request_id"].astype(str)
        cache_df["pulled_at"] = cache_df["pulled_at"].astype(str)

        cache_df = (
            cache_df
            .sort_values(["contract_request_id", "pulled_at"])
            .drop_duplicates("contract_request_id", keep="last")
            .sort_values(["trade_date_str", "expiration_str", "strike_listed", "right"])
            .reset_index(drop=True)
        )

    atomic_write_parquet(cache_df, PRICE_CACHE_PATH)
    atomic_write_csv(cache_df, PRICE_CACHE_CSV_SNAPSHOT_PATH)

    if run_audit_df is not None:
        atomic_write_csv(run_audit_df, RUN_AUDIT_PATH)

    return cache_df


# --------------------------------------------------------------------------------------------------
# Load selected long5_cal contract plan and existing long5_cal cache
# --------------------------------------------------------------------------------------------------

if not SELECTED_UNIQUE_CONTRACT_PLAN_PATH.exists():
    raise FileNotFoundError(f"Missing selected unique contract plan: {SELECTED_UNIQUE_CONTRACT_PLAN_PATH}")

plan = pd.read_parquet(SELECTED_UNIQUE_CONTRACT_PLAN_PATH).copy()

required_plan_cols = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "trade_date_str",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "strike_listed",
]

missing_plan_cols = [c for c in required_plan_cols if c not in plan.columns]
if missing_plan_cols:
    raise ValueError(f"Missing required columns from selected unique contract plan: {missing_plan_cols}")

plan["contract_request_id"] = plan["contract_request_id"].astype(str)
plan["trade_date"] = pd.to_datetime(plan["trade_date"], errors="coerce")
plan["expiration_date"] = pd.to_datetime(plan["expiration_date"], errors="coerce")
plan["strike_listed"] = pd.to_numeric(plan["strike_listed"], errors="coerce")

plan = (
    plan
    .dropna(subset=["trade_date", "expiration_date", "strike_listed"])
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

if PRICE_CACHE_PATH.exists():
    cache = pd.read_parquet(PRICE_CACHE_PATH).copy()

    if "contract_request_id" in cache.columns:
        cache["contract_request_id"] = cache["contract_request_id"].astype(str)
    else:
        cache = build_empty_cache()
else:
    cache = build_empty_cache()

completed_ids = get_completed_ids(cache)

remaining_full = plan.loc[
    ~plan["contract_request_id"].astype(str).isin(completed_ids)
].copy()

remaining = remaining_full.copy()

if MAX_REQUESTS_THIS_RUN is not None:
    remaining = remaining.head(int(MAX_REQUESTS_THIS_RUN)).copy()

print("=" * 100)
print("Loaded selected long5_cal contract plan/cache")
print("=" * 100)
print(f"Plan rows:                 {len(plan):,}")
print(f"Existing cache rows:       {len(cache):,}")
print(f"Completed/skipped IDs:     {len(completed_ids):,}")
print(f"Remaining full:            {len(remaining_full):,}")
print(f"Remaining this run:        {len(remaining):,}")
print()

if not cache.empty:
    existing_summary = (
        cache
        .groupby(["price_found", "price_status"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["price_found", "price_status"])
    )

    print("Existing cache status summary:")
    display(existing_summary)
    print()


# --------------------------------------------------------------------------------------------------
# Main pricing loop
# --------------------------------------------------------------------------------------------------

run_rows = []
new_rows = []
start_time = time.time()

try:
    if remaining.empty:
        print("No remaining selected long5_cal contract requests to price.")
    else:
        for i, (_, row) in enumerate(remaining.iterrows(), start=1):
            result = price_contract(row)
            new_rows.append(result)
            run_rows.append(result)

            if i % PROGRESS_EVERY == 0 or i == 1:
                elapsed = time.time() - start_time
                priced_count = sum(1 for r in run_rows if r.get("price_found") is True)
                miss_count = i - priced_count
                rate = i / elapsed if elapsed > 0 else np.nan
                remaining_count = len(remaining) - i
                eta_minutes = remaining_count / rate / 60 if rate and rate > 0 else np.nan

                print(
                    f"[{i:>5,}/{len(remaining):,}] "
                    f"priced={priced_count:,} "
                    f"missing_or_failed={miss_count:,} "
                    f"last_status={result['price_status']} "
                    f"last_http={result['status_code']} "
                    f"rate={rate:.2f}/sec "
                    f"ETA={eta_minutes:.1f} min"
                )

            if i % CHECKPOINT_EVERY == 0:
                new_df = pd.DataFrame(new_rows)

                if cache.empty:
                    cache = new_df.copy()
                else:
                    cache = pd.concat([cache, new_df], ignore_index=True)

                run_audit_df = pd.DataFrame(run_rows)
                cache = save_checkpoint(cache, run_audit_df=run_audit_df)
                new_rows = []

                checkpoint_summary = (
                    cache
                    .groupby(["price_found", "price_status"], dropna=False)
                    .size()
                    .reset_index(name="count")
                    .sort_values(["price_found", "price_status"])
                )

                print()
                print(f"Checkpoint saved after {i:,} attempts this run.")
                display(checkpoint_summary)
                print()

            if REQUEST_SLEEP_SECONDS and REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

except KeyboardInterrupt:
    print()
    print("KeyboardInterrupt received. Saving checkpoint before stopping...")

except Exception as e:
    print()
    print("Unexpected exception. Saving checkpoint before raising...")
    print(repr(e))
    print(traceback.format_exc())
    raise

finally:
    if new_rows:
        new_df = pd.DataFrame(new_rows)

        if cache.empty:
            cache = new_df.copy()
        else:
            cache = pd.concat([cache, new_df], ignore_index=True)

    run_audit_df = pd.DataFrame(run_rows)
    cache = save_checkpoint(cache, run_audit_df=run_audit_df)


# --------------------------------------------------------------------------------------------------
# Final diagnostics
# --------------------------------------------------------------------------------------------------

elapsed = time.time() - start_time
final_cache = pd.read_parquet(PRICE_CACHE_PATH)

status_summary = (
    final_cache
    .groupby(["price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["price_found", "price_status"])
)

status_code_summary = (
    final_cache
    .groupby(["status_code", "price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["status_code", "price_found", "price_status"])
)

date_coverage = (
    final_cache
    .groupby("trade_date_str")
    .agg(
        cached_contracts=("contract_request_id", "size"),
        priced_contracts=("price_found", "sum"),
    )
    .reset_index()
)

date_coverage["priced_rate"] = (
    date_coverage["priced_contracts"] / date_coverage["cached_contracts"]
)

missing_or_failed = final_cache.loc[
    final_cache["price_found"] != True
].copy()

missing_or_failed_preview = (
    missing_or_failed
    .sort_values(["trade_date_str", "expiration_str", "strike_listed"])
    .head(50)
)

# Save audit snapshots
status_summary_path = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_status_summary_{timestamp}.csv"
status_code_summary_path = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_status_code_summary_{timestamp}.csv"
date_coverage_path = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_date_coverage_{timestamp}.csv"
missing_preview_path = AUDIT_DIR / f"02E_long5_cal_selected_contract_price_cache_missing_preview_{timestamp}.csv"

status_summary.to_csv(status_summary_path, index=False)
status_code_summary.to_csv(status_code_summary_path, index=False)
date_coverage.to_csv(date_coverage_path, index=False)
missing_or_failed_preview.to_csv(missing_preview_path, index=False)

manifest = {
    "cell": "2E_long5_cal",
    "description": "Selected SPY long5 calendar-consistent contract price cache using ThetaData v3 /v3/option/history/greeks/eod",
    "created_at": timestamp,
    "base_url": BASE_URL,
    "endpoint": ENDPOINT,
    "selected_unique_contract_plan_path": str(SELECTED_UNIQUE_CONTRACT_PLAN_PATH),
    "price_cache_path": str(PRICE_CACHE_PATH),
    "price_cache_csv_snapshot_path": str(PRICE_CACHE_CSV_SNAPSHOT_PATH),
    "run_audit_path": str(RUN_AUDIT_PATH),
    "status_summary_path": str(status_summary_path),
    "status_code_summary_path": str(status_code_summary_path),
    "date_coverage_path": str(date_coverage_path),
    "missing_preview_path": str(missing_preview_path),
    "runtime_seconds": elapsed,
    "plan_rows": int(len(plan)),
    "cache_rows_final": int(len(final_cache)),
    "run_attempts": int(len(run_audit_df)),
    "run_priced": int(run_audit_df["price_found"].sum()) if not run_audit_df.empty and "price_found" in run_audit_df.columns else 0,
    "final_priced": int(final_cache["price_found"].sum()) if "price_found" in final_cache.columns else 0,
    "final_missing_or_failed": int((final_cache["price_found"] != True).sum()) if "price_found" in final_cache.columns else int(len(final_cache)),
    "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
    "retry_failed_or_missing": RETRY_FAILED_OR_MISSING,
}

with open(RUN_MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Cell 2E long5_cal complete")
print("=" * 100)
print(f"Runtime minutes:       {elapsed / 60:.2f}")
print(f"Run attempts:          {len(run_audit_df):,}")
print(f"Final cache rows:      {len(final_cache):,}")
print(f"Final priced rows:     {int(final_cache['price_found'].sum()):,}")
print(f"Final missing/failed:  {int((final_cache['price_found'] != True).sum()):,}")
print()

print("Status summary:")
display(status_summary)

print()
print("Status-code summary:")
display(status_code_summary)

print()
print("Date coverage preview:")
display(date_coverage.head(30))

print()
print("Missing/failed preview:")
if missing_or_failed_preview.empty:
    print("No missing/failed rows in cache so far.")
else:
    display(
        missing_or_failed_preview[
            [
                "contract_request_id",
                "trade_date_str",
                "expiration_str",
                "strike_listed",
                "right",
                "status_code",
                "price_status",
                "raw_text_preview",
            ]
        ].head(50)
    )

print()
print("Saved:")
print(f"  Price cache parquet:       {PRICE_CACHE_PATH}")
print(f"  Price cache CSV snapshot:  {PRICE_CACHE_CSV_SNAPSHOT_PATH}")
print(f"  Run audit:                 {RUN_AUDIT_PATH}")
print(f"  Status summary:            {status_summary_path}")
print(f"  Status-code summary:       {status_code_summary_path}")
print(f"  Date coverage:             {date_coverage_path}")
print(f"  Missing preview:           {missing_preview_path}")
print(f"  Manifest:                  {RUN_MANIFEST_PATH}")

if len(final_cache) < len(plan):
    print()
    print("Result: Cache is partially complete. Review long5_cal first-batch coverage; then rerun with MAX_REQUESTS_THIS_RUN = None to finish.")
else:
    print()
    print("Result: Selected long5_cal contract cache completed. Next cell should join prices back to selected trades and compute real spread P&L.")

Cell 2E — Price selected long5_cal unique contract request plan
Selected unique contract plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_long5_cal_v1.parquet
Price cache path:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet
ThetaData base URL:            http://127.0.0.1:25503
Endpoint:                      /v3/option/history/greeks/eod
Checkpoint every:              100 attempts
Max requests this run:         None
Retry failed/missing:          False

Loaded selected long5_cal contract plan/cache
Plan rows:                 3,437
Existing cache rows:       250
Completed/skipped IDs:     250
Remaining full:            3,187
Remaining this run:        3,187

Existing cache status summary:


,price_found,price_status,count
0,False,http_472,12
1,True,priced,238



[    1/3,187] priced=1 missing_or_failed=0 last_status=priced last_http=200 rate=5.73/sec ETA=9.3 min
[   25/3,187] priced=24 missing_or_failed=1 last_status=priced last_http=200 rate=6.58/sec ETA=8.0 min
[   50/3,187] priced=49 missing_or_failed=1 last_status=priced last_http=200 rate=5.94/sec ETA=8.8 min
[   75/3,187] priced=71 missing_or_failed=4 last_status=priced last_http=200 rate=5.57/sec ETA=9.3 min
[  100/3,187] priced=93 missing_or_failed=7 last_status=priced last_http=200 rate=5.94/sec ETA=8.7 min

Checkpoint saved after 100 attempts this run.


,price_found,price_status,count
0,False,http_472,19
1,True,priced,331



[  125/3,187] priced=117 missing_or_failed=8 last_status=priced last_http=200 rate=5.89/sec ETA=8.7 min
[  150/3,187] priced=138 missing_or_failed=12 last_status=http_472 last_http=472 rate=6.11/sec ETA=8.3 min
[  175/3,187] priced=159 missing_or_failed=16 last_status=priced last_http=200 rate=6.42/sec ETA=7.8 min
[  200/3,187] priced=182 missing_or_failed=18 last_status=priced last_http=200 rate=6.34/sec ETA=7.9 min

Checkpoint saved after 200 attempts this run.


...


Date coverage preview:


,trade_date_str,cached_contracts,priced_contracts,priced_rate
0,2020-12-31,6,5,0.833333
1,2021-01-06,2,2,1.000000
2,2021-01-11,2,2,1.000000
3,2021-01-19,4,4,1.000000
4,2021-01-25,8,8,1.000000
5,2021-01-26,10,10,1.000000
6,2021-02-16,10,10,1.000000
7,2021-02-17,10,10,1.000000
8,2021-02-18,6,6,1.000000
9,2021-02-19,4,4,1.000000



Missing/failed preview:


,contract_request_id,trade_date_str,expiration_str,strike_listed,right,status_code,price_status,raw_text_preview
5,SPY_20201231_20210205_450.0_C,2020-12-31,2021-02-05,450.0,C,472,http_472,No data found for your request
138,SPY_20211026_20211203_479.0_C,2021-10-26,2021-12-03,479.0,C,472,http_472,No data found for your request
139,SPY_20211026_20211203_525.0_C,2021-10-26,2021-12-03,525.0,C,472,http_472,No data found for your request
159,SPY_20211102_20211126_515.0_C,2021-11-02,2021-11-26,515.0,C,472,http_472,No data found for your request
179,SPY_20211104_20211203_525.0_C,2021-11-04,2021-12-03,525.0,C,472,http_472,No data found for your request
183,SPY_20211104_20211210_535.0_C,2021-11-04,2021-12-10,535.0,C,472,http_472,No data found for your request
202,SPY_20211108_20211210_535.0_C,2021-11-08,2021-12-10,535.0,C,472,http_472,No data found for your request
223,SPY_20211118_20211210_515.0_C,2021-11-18,2021-12-10,515.0,C,472,http_472,No data found for your request
227,SPY_20211118_20211223_545.0_C,2021-11-18,2021-12-23,545.0,C,472,http_472,No data found for your request
233,SPY_20211119_20211210_525.0_C,2021-11-19,2021-12-10,525.0,C,472,http_472,No data found for your request



Saved:
  Price cache parquet:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet
  Price cache CSV snapshot:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_latest.csv
  Run audit:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_cal_selected_contract_price_cache_run_audit_20260710_113416.csv
  Status summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_cal_selected_contract_price_cache_status_summary_20260710_113416.csv
  Status-code summary:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_cal_selected_contract_price_cache_status_code_summary_20260710_113416.csv
  Date coverage:             C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E_long5_cal_selected_contract_price_cache_date_coverage_20260710_1134

In [25]:
# Cell 2F — Join long5_cal contract prices back to selected spreads and compute spread P&L
# Purpose:
#   Mechanical infrastructure step unaffected by the RSI repair.
#
# This cell:
#   - Joins the completed 2E long5_cal option price cache to the selected leg plan
#   - Builds spread-level pricing completeness
#   - Classifies missing short leg vs missing long leg
#   - Computes conservative entry credit for complete spreads
#   - Computes max loss, expiry P&L, and return on max loss
#   - Joins priced spread outcomes back to rule-selected trades
#
# This cell does NOT:
#   - Change RSI
#   - Change strikes
#   - Add fallback strike logic
#   - Rank final rules
#   - Select final parameters
#   - Promote anything to production

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_LEG_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet"
)

UNIQUE_SELECTED_TRADE_UNIVERSE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet"
)

RULE_SELECTED_TRADES_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet"
)

PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F — Join long5_cal prices to selected spreads and compute spread P&L")
print("=" * 100)
print(f"Selected leg plan:        {SELECTED_LEG_PLAN_PATH}")
print(f"Unique trade universe:    {UNIQUE_SELECTED_TRADE_UNIVERSE_PATH}")
print(f"Rule selected trades:     {RULE_SELECTED_TRADES_PATH}")
print(f"Price cache:              {PRICE_CACHE_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def bool_sum(x):
    return int(pd.Series(x).fillna(False).sum())

def make_status(row):
    short_ok = bool(row.get("short_price_found", False))
    long_ok = bool(row.get("long_price_found", False))

    if short_ok and long_ok:
        return "complete"
    if (not short_ok) and long_ok:
        return "missing_short_only"
    if short_ok and (not long_ok):
        return "missing_long_only"
    return "missing_both"

def summarize_rule_group(g):
    complete = g.loc[g["spread_pricing_status"].eq("complete")].copy()

    out = {
        "selected_rows": len(g),
        "unique_selected_trades": g["selected_trade_id"].nunique(),
        "complete_rows": len(complete),
        "complete_unique_trades": complete["selected_trade_id"].nunique(),
        "coverage_rate": len(complete) / len(g) if len(g) else np.nan,
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "missing_short_only_rows": int(g["spread_pricing_status"].eq("missing_short_only").sum()),
        "missing_long_only_rows": int(g["spread_pricing_status"].eq("missing_long_only").sum()),
        "missing_both_rows": int(g["spread_pricing_status"].eq("missing_both").sum()),
    }

    if complete.empty:
        out.update({
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_max_loss": np.nan,
            "win_rate": np.nan,
            "avg_expiry_pnl": np.nan,
            "median_expiry_pnl": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
            "avg_terminal_intrinsic": np.nan,
            "worst_terminal_intrinsic": np.nan,
        })
        return pd.Series(out)

    out.update({
        "avg_entry_credit": complete["entry_credit_conservative"].mean(),
        "median_entry_credit": complete["entry_credit_conservative"].median(),
        "avg_credit_pct_width": complete["entry_credit_pct_width"].mean(),
        "median_credit_pct_width": complete["entry_credit_pct_width"].median(),
        "avg_max_loss": complete["max_loss_conservative"].mean(),
        "win_rate": complete["expiry_pnl_conservative"].gt(0).mean(),
        "avg_expiry_pnl": complete["expiry_pnl_conservative"].mean(),
        "median_expiry_pnl": complete["expiry_pnl_conservative"].median(),
        "avg_return_on_max_loss": complete["return_on_max_loss_conservative"].mean(),
        "median_return_on_max_loss": complete["return_on_max_loss_conservative"].median(),
        "worst_return_on_max_loss": complete["return_on_max_loss_conservative"].min(),
        "p05_return_on_max_loss": complete["return_on_max_loss_conservative"].quantile(0.05),
        "p95_return_on_max_loss": complete["return_on_max_loss_conservative"].quantile(0.95),
        "avg_terminal_intrinsic": complete["terminal_spread_intrinsic_before_premium"].mean(),
        "worst_terminal_intrinsic": complete["terminal_spread_intrinsic_before_premium"].max(),
    })

    return pd.Series(out)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    SELECTED_LEG_PLAN_PATH,
    UNIQUE_SELECTED_TRADE_UNIVERSE_PATH,
    RULE_SELECTED_TRADES_PATH,
    PRICE_CACHE_PATH,
]:
    require_file(p)

leg_plan = pd.read_parquet(SELECTED_LEG_PLAN_PATH).copy()
unique_trades = pd.read_parquet(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH).copy()
rule_trades = pd.read_parquet(RULE_SELECTED_TRADES_PATH).copy()
price_cache = pd.read_parquet(PRICE_CACHE_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Leg plan rows:             {len(leg_plan):,}")
print(f"Unique selected trades:    {len(unique_trades):,}")
print(f"Rule selected rows:        {len(rule_trades):,}")
print(f"Price cache rows:          {len(price_cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Basic validation / cleanup
# --------------------------------------------------------------------------------------------------

required_leg_cols = [
    "selected_trade_id",
    "contract_request_id",
    "leg_label",
    "leg_side",
    "strike_listed",
    "trade_date",
    "expiration_date",
    "trade_date_str",
    "expiration_str",
]

required_trade_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
]

required_rule_cols = [
    "selected_trade_id",
    "rule_id",
    "selection_rule",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
]

required_cache_cols = [
    "contract_request_id",
    "price_found",
    "price_status",
    "bid",
    "ask",
    "mid",
]

missing_leg = [c for c in required_leg_cols if c not in leg_plan.columns]
missing_trade = [c for c in required_trade_cols if c not in unique_trades.columns]
missing_rule = [c for c in required_rule_cols if c not in rule_trades.columns]
missing_cache = [c for c in required_cache_cols if c not in price_cache.columns]

if missing_leg:
    raise ValueError(f"Missing required leg plan columns: {missing_leg}")
if missing_trade:
    raise ValueError(f"Missing required unique trade columns: {missing_trade}")
if missing_rule:
    raise ValueError(f"Missing required rule trade columns: {missing_rule}")
if missing_cache:
    raise ValueError(f"Missing required price cache columns: {missing_cache}")

for df in [leg_plan, unique_trades, rule_trades, price_cache]:
    if "selected_trade_id" in df.columns:
        df["selected_trade_id"] = df["selected_trade_id"].astype(str)
    if "contract_request_id" in df.columns:
        df["contract_request_id"] = df["contract_request_id"].astype(str)

for df in [leg_plan, unique_trades, rule_trades]:
    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce").dt.normalize()
    if "expiration_date" in df.columns:
        df["expiration_date"] = pd.to_datetime(df["expiration_date"], errors="coerce").dt.normalize()

price_cache["price_found"] = price_cache["price_found"].fillna(False).astype(bool)

# Deduplicate cache defensively.
if "pulled_at" in price_cache.columns:
    price_cache["pulled_at"] = price_cache["pulled_at"].astype(str)
    price_cache = (
        price_cache
        .sort_values(["contract_request_id", "pulled_at"])
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )
else:
    price_cache = (
        price_cache
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )

for col in ["bid", "ask", "mid", "strike_listed", "underlying_price", "implied_vol", "delta", "theta", "vega", "gamma"]:
    if col in price_cache.columns:
        price_cache[col] = pd.to_numeric(price_cache[col], errors="coerce")

print("=" * 100)
print("Post-cleanup counts")
print("=" * 100)
print(f"Deduped price cache rows:  {len(price_cache):,}")
print(f"Priced cache rows:         {int(price_cache['price_found'].sum()):,}")
print(f"Missing cache rows:        {int((~price_cache['price_found']).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Join cache to leg plan
# --------------------------------------------------------------------------------------------------

cache_cols = [
    "contract_request_id",
    "price_found",
    "price_status",
    "status_code",
    "bid",
    "ask",
    "mid",
    "bid_size",
    "ask_size",
    "underlying_price",
    "implied_vol",
    "delta",
    "theta",
    "vega",
    "gamma",
    "rho",
    "raw_text_preview",
    "pulled_at",
]

cache_cols = [c for c in cache_cols if c in price_cache.columns]

leg_prices = leg_plan.merge(
    price_cache[cache_cols],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_prices["price_found"] = leg_prices["price_found"].fillna(False).astype(bool)
leg_prices["price_status"] = leg_prices["price_status"].fillna("not_in_cache")

leg_prices["entry_leg_price_conservative"] = np.where(
    (leg_prices["leg_side"].eq("sell")) & (leg_prices["price_found"]),
    leg_prices["bid"],
    np.where(
        (leg_prices["leg_side"].eq("buy")) & (leg_prices["price_found"]),
        leg_prices["ask"],
        np.nan,
    )
)

leg_prices["entry_leg_price_mid"] = np.where(
    leg_prices["price_found"],
    leg_prices["mid"],
    np.nan,
)

leg_prices["entry_leg_cashflow_conservative"] = np.where(
    leg_prices["leg_side"].eq("sell"),
    leg_prices["entry_leg_price_conservative"],
    -leg_prices["entry_leg_price_conservative"],
)

leg_prices["entry_leg_cashflow_mid"] = np.where(
    leg_prices["leg_side"].eq("sell"),
    leg_prices["entry_leg_price_mid"],
    -leg_prices["entry_leg_price_mid"],
)

# --------------------------------------------------------------------------------------------------
# Build spread-level table from leg prices
# --------------------------------------------------------------------------------------------------

leg_core_cols = [
    "selected_trade_id",
    "leg_label",
    "contract_request_id",
    "strike_listed",
    "price_found",
    "price_status",
    "bid",
    "ask",
    "mid",
    "bid_size",
    "ask_size",
    "underlying_price",
    "implied_vol",
    "delta",
    "theta",
    "vega",
    "gamma",
    "rho",
    "entry_leg_price_conservative",
    "entry_leg_price_mid",
    "entry_leg_cashflow_conservative",
    "entry_leg_cashflow_mid",
]

leg_core_cols = [c for c in leg_core_cols if c in leg_prices.columns]

short_leg = (
    leg_prices.loc[leg_prices["leg_label"].eq("short_call_1sd"), leg_core_cols]
    .copy()
    .rename(columns={
        "contract_request_id": "short_contract_request_id",
        "strike_listed": "short_leg_strike",
        "price_found": "short_price_found",
        "price_status": "short_price_status",
        "bid": "short_bid",
        "ask": "short_ask",
        "mid": "short_mid",
        "bid_size": "short_bid_size",
        "ask_size": "short_ask_size",
        "underlying_price": "short_underlying_price",
        "implied_vol": "short_implied_vol",
        "delta": "short_delta",
        "theta": "short_theta",
        "vega": "short_vega",
        "gamma": "short_gamma",
        "rho": "short_rho",
        "entry_leg_price_conservative": "short_entry_price_conservative",
        "entry_leg_price_mid": "short_entry_price_mid",
        "entry_leg_cashflow_conservative": "short_entry_cashflow_conservative",
        "entry_leg_cashflow_mid": "short_entry_cashflow_mid",
    })
    .drop(columns=["leg_label"], errors="ignore")
)

long_leg = (
    leg_prices.loc[leg_prices["leg_label"].eq("long_call_3sd"), leg_core_cols]
    .copy()
    .rename(columns={
        "contract_request_id": "long_contract_request_id",
        "strike_listed": "long_leg_strike",
        "price_found": "long_price_found",
        "price_status": "long_price_status",
        "bid": "long_bid",
        "ask": "long_ask",
        "mid": "long_mid",
        "bid_size": "long_bid_size",
        "ask_size": "long_ask_size",
        "underlying_price": "long_underlying_price",
        "implied_vol": "long_implied_vol",
        "delta": "long_delta",
        "theta": "long_theta",
        "vega": "long_vega",
        "gamma": "long_gamma",
        "rho": "long_rho",
        "entry_leg_price_conservative": "long_entry_price_conservative",
        "entry_leg_price_mid": "long_entry_price_mid",
        "entry_leg_cashflow_conservative": "long_entry_cashflow_conservative",
        "entry_leg_cashflow_mid": "long_entry_cashflow_mid",
    })
    .drop(columns=["leg_label"], errors="ignore")
)

# One short and one long row per selected_trade_id expected.
short_dupes = short_leg["selected_trade_id"].duplicated().sum()
long_dupes = long_leg["selected_trade_id"].duplicated().sum()

if short_dupes:
    print(f"WARNING: duplicate short leg selected_trade_id rows found: {short_dupes:,}")
if long_dupes:
    print(f"WARNING: duplicate long leg selected_trade_id rows found: {long_dupes:,}")

short_leg = short_leg.drop_duplicates("selected_trade_id", keep="last")
long_leg = long_leg.drop_duplicates("selected_trade_id", keep="last")

spread_prices = unique_trades.merge(
    short_leg,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices = spread_prices.merge(
    long_leg,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices["short_price_found"] = spread_prices["short_price_found"].fillna(False).astype(bool)
spread_prices["long_price_found"] = spread_prices["long_price_found"].fillna(False).astype(bool)

spread_prices["spread_pricing_status"] = spread_prices.apply(make_status, axis=1)
spread_prices["spread_complete"] = spread_prices["spread_pricing_status"].eq("complete")

# Price and P&L calculations. Conservative means sell short at bid, buy long at ask.
spread_prices["entry_credit_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_bid"] - spread_prices["long_ask"],
    np.nan,
)

spread_prices["entry_credit_mid"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_mid"] - spread_prices["long_mid"],
    np.nan,
)

spread_prices["entry_credit_optimistic"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_ask"] - spread_prices["long_bid"],
    np.nan,
)

spread_prices["entry_credit_conservative_x100"] = spread_prices["entry_credit_conservative"] * 100.0
spread_prices["entry_credit_mid_x100"] = spread_prices["entry_credit_mid"] * 100.0

spread_prices["credit_pct_width"] = safe_divide(
    spread_prices["entry_credit_conservative"],
    spread_prices["listed_width"],
)

spread_prices["max_loss_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["listed_width"] - spread_prices["entry_credit_conservative"],
    np.nan,
)

spread_prices["max_loss_conservative_x100"] = spread_prices["max_loss_conservative"] * 100.0

spread_prices["expiry_pnl_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["entry_credit_conservative"] - spread_prices["terminal_spread_intrinsic_before_premium"],
    np.nan,
)

spread_prices["expiry_pnl_conservative_x100"] = spread_prices["expiry_pnl_conservative"] * 100.0

spread_prices["return_on_max_loss_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["max_loss_conservative"],
)

spread_prices["return_on_width_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["listed_width"],
)

spread_prices["trade_win_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["expiry_pnl_conservative"] > 0,
    np.nan,
)

spread_prices["invalid_credit_flag"] = (
    spread_prices["spread_complete"]
    & (
        spread_prices["entry_credit_conservative"].isna()
        | (spread_prices["entry_credit_conservative"] <= 0)
        | (spread_prices["max_loss_conservative"] <= 0)
    )
)

spread_prices["trade_year"] = pd.to_datetime(spread_prices["trade_date"]).dt.year

# --------------------------------------------------------------------------------------------------
# Join spread outcomes back to rule-selected trades
# --------------------------------------------------------------------------------------------------

spread_outcome_cols = [
    "selected_trade_id",
    "spread_pricing_status",
    "spread_complete",
    "short_price_found",
    "long_price_found",
    "short_price_status",
    "long_price_status",
    "short_contract_request_id",
    "long_contract_request_id",
    "short_bid",
    "short_ask",
    "short_mid",
    "long_bid",
    "long_ask",
    "long_mid",
    "entry_credit_conservative",
    "entry_credit_mid",
    "entry_credit_optimistic",
    "entry_credit_conservative_x100",
    "entry_credit_mid_x100",
    "credit_pct_width",
    "max_loss_conservative",
    "max_loss_conservative_x100",
    "expiry_pnl_conservative",
    "expiry_pnl_conservative_x100",
    "return_on_max_loss_conservative",
    "return_on_width_conservative",
    "trade_win_conservative",
    "invalid_credit_flag",
]

spread_outcome_cols = [c for c in spread_outcome_cols if c in spread_prices.columns]

# Avoid duplicate calculation cols in rule rows before merge.
rule_trades_priced = rule_trades.drop(
    columns=[c for c in spread_outcome_cols if c in rule_trades.columns and c != "selected_trade_id"],
    errors="ignore",
).merge(
    spread_prices[spread_outcome_cols],
    on="selected_trade_id",
    how="left",
    validate="many_to_one",
)

rule_trades_priced["spread_complete"] = rule_trades_priced["spread_complete"].fillna(False).astype(bool)
rule_trades_priced["spread_pricing_status"] = rule_trades_priced["spread_pricing_status"].fillna("not_in_spread_table")
rule_trades_priced["trade_year"] = pd.to_datetime(rule_trades_priced["trade_date"]).dt.year

# --------------------------------------------------------------------------------------------------
# Audits and summaries
# --------------------------------------------------------------------------------------------------

leg_price_summary = (
    leg_prices
    .groupby(["leg_label", "price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="leg_count")
    .sort_values(["leg_label", "price_found", "price_status"])
)

spread_status_summary = (
    spread_prices
    .groupby("spread_pricing_status", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        avg_tenor=("tenor", "mean"),
        avg_listed_width=("listed_width", "mean"),
    )
    .reset_index()
    .sort_values("unique_spreads", ascending=False)
)

spread_status_by_year = (
    spread_prices
    .groupby(["trade_year", "spread_pricing_status"], dropna=False)
    .size()
    .reset_index(name="unique_spreads")
    .sort_values(["trade_year", "spread_pricing_status"])
)

spread_status_by_tenor = (
    spread_prices
    .groupby(["tenor", "spread_pricing_status"], dropna=False)
    .size()
    .reset_index(name="unique_spreads")
    .sort_values(["tenor", "spread_pricing_status"])
)

complete_spreads = spread_prices.loc[spread_prices["spread_complete"]].copy()

complete_spread_summary = pd.DataFrame([{
    "unique_spreads_total": len(spread_prices),
    "unique_spreads_complete": len(complete_spreads),
    "complete_rate": len(complete_spreads) / len(spread_prices) if len(spread_prices) else np.nan,
    "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
    "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
    "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
    "invalid_credit_count": int(spread_prices["invalid_credit_flag"].sum()),
    "avg_entry_credit": complete_spreads["entry_credit_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_entry_credit": complete_spreads["entry_credit_conservative"].median() if not complete_spreads.empty else np.nan,
    "avg_credit_pct_width": complete_spreads["credit_pct_width"].mean() if not complete_spreads.empty else np.nan,
    "median_credit_pct_width": complete_spreads["credit_pct_width"].median() if not complete_spreads.empty else np.nan,
    "avg_max_loss": complete_spreads["max_loss_conservative"].mean() if not complete_spreads.empty else np.nan,
    "win_rate": complete_spreads["trade_win_conservative"].mean() if not complete_spreads.empty else np.nan,
    "avg_expiry_pnl": complete_spreads["expiry_pnl_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_expiry_pnl": complete_spreads["expiry_pnl_conservative"].median() if not complete_spreads.empty else np.nan,
    "avg_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].median() if not complete_spreads.empty else np.nan,
    "worst_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].min() if not complete_spreads.empty else np.nan,
}])

rule_group_cols = [
    "rule_id",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
    "selection_rule",
]

rule_group_cols = [c for c in rule_group_cols if c in rule_trades_priced.columns]

rule_priced_summary = (
    rule_trades_priced
    .groupby(rule_group_cols, dropna=False)
    .apply(summarize_rule_group)
    .reset_index()
    .sort_values(["rule_id"])
)

rule_year_priced_summary = (
    rule_trades_priced
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        complete_rows=("spread_complete", "sum"),
        unique_selected_trades=("selected_trade_id", "nunique"),
        avg_entry_credit=("entry_credit_conservative", "mean"),
        avg_credit_pct_width=("credit_pct_width", "mean"),
        win_rate=("trade_win_conservative", "mean"),
        avg_expiry_pnl=("expiry_pnl_conservative", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative", "min"),
    )
    .reset_index()
)

rule_year_priced_summary["coverage_rate"] = safe_divide(
    rule_year_priced_summary["complete_rows"],
    rule_year_priced_summary["selected_rows"],
)

missing_spread_preview = (
    spread_prices.loc[~spread_prices["spread_complete"]]
    [
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "long_strike",
            "listed_width",
            "spread_pricing_status",
            "short_contract_request_id",
            "short_price_status",
            "long_contract_request_id",
            "long_price_status",
        ]
    ]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(100)
)

complete_spread_preview = (
    spread_prices.loc[spread_prices["spread_complete"]]
    [
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "long_strike",
            "listed_width",
            "short_bid",
            "long_ask",
            "entry_credit_conservative",
            "credit_pct_width",
            "expiry_spy_close",
            "terminal_spread_intrinsic_before_premium",
            "expiry_pnl_conservative",
            "return_on_max_loss_conservative",
        ]
    ]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(100)
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

LEG_PRICES_JOINED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_prices_joined_long5_cal_v1.parquet"
)

UNIQUE_TRADES_PRICED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_long5_cal_v1.parquet"
)

RULE_TRADES_PRICED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_priced_long5_cal_v1.parquet"
)

SPREAD_STATUS_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_spread_pricing_status_summary_long5_cal_v1.parquet"
)

RULE_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_priced_summary_long5_cal_v1.parquet"
)

RULE_YEAR_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_priced_summary_long5_cal_v1.parquet"
)

leg_prices.to_parquet(LEG_PRICES_JOINED_PATH, index=False)
spread_prices.to_parquet(UNIQUE_TRADES_PRICED_PATH, index=False)
rule_trades_priced.to_parquet(RULE_TRADES_PRICED_PATH, index=False)
spread_status_summary.to_parquet(SPREAD_STATUS_SUMMARY_PATH, index=False)
rule_priced_summary.to_parquet(RULE_PRICED_SUMMARY_PATH, index=False)
rule_year_priced_summary.to_parquet(RULE_YEAR_PRICED_SUMMARY_PATH, index=False)

# Audit CSVs
leg_price_summary_csv = AUDIT_DIR / f"02F_leg_price_summary_long5_cal_{timestamp}.csv"
spread_status_summary_csv = AUDIT_DIR / f"02F_spread_status_summary_long5_cal_{timestamp}.csv"
spread_status_by_year_csv = AUDIT_DIR / f"02F_spread_status_by_year_long5_cal_{timestamp}.csv"
spread_status_by_tenor_csv = AUDIT_DIR / f"02F_spread_status_by_tenor_long5_cal_{timestamp}.csv"
complete_spread_summary_csv = AUDIT_DIR / f"02F_complete_spread_summary_long5_cal_{timestamp}.csv"
missing_spread_preview_csv = AUDIT_DIR / f"02F_missing_spread_preview_long5_cal_{timestamp}.csv"
complete_spread_preview_csv = AUDIT_DIR / f"02F_complete_spread_preview_long5_cal_{timestamp}.csv"
rule_priced_summary_csv = AUDIT_DIR / f"02F_rule_priced_summary_long5_cal_{timestamp}.csv"
rule_year_priced_summary_csv = AUDIT_DIR / f"02F_rule_year_priced_summary_long5_cal_{timestamp}.csv"

leg_price_summary.to_csv(leg_price_summary_csv, index=False)
spread_status_summary.to_csv(spread_status_summary_csv, index=False)
spread_status_by_year.to_csv(spread_status_by_year_csv, index=False)
spread_status_by_tenor.to_csv(spread_status_by_tenor_csv, index=False)
complete_spread_summary.to_csv(complete_spread_summary_csv, index=False)
missing_spread_preview.to_csv(missing_spread_preview_csv, index=False)
complete_spread_preview.to_csv(complete_spread_preview_csv, index=False)
rule_priced_summary.to_csv(rule_priced_summary_csv, index=False)
rule_year_priced_summary.to_csv(rule_year_priced_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02F_join_prices_spread_pnl_long5_cal_manifest_{timestamp}.json"

manifest = {
    "cell": "2F",
    "description": "Join long5_cal contract prices back to selected call spreads and compute spread P&L",
    "created_at": timestamp,
    "inputs": {
        "selected_leg_plan_path": str(SELECTED_LEG_PLAN_PATH),
        "unique_selected_trade_universe_path": str(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH),
        "rule_selected_trades_path": str(RULE_SELECTED_TRADES_PATH),
        "price_cache_path": str(PRICE_CACHE_PATH),
    },
    "outputs": {
        "leg_prices_joined_path": str(LEG_PRICES_JOINED_PATH),
        "unique_trades_priced_path": str(UNIQUE_TRADES_PRICED_PATH),
        "rule_trades_priced_path": str(RULE_TRADES_PRICED_PATH),
        "spread_status_summary_path": str(SPREAD_STATUS_SUMMARY_PATH),
        "rule_priced_summary_path": str(RULE_PRICED_SUMMARY_PATH),
        "rule_year_priced_summary_path": str(RULE_YEAR_PRICED_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "leg_plan_rows": int(len(leg_plan)),
        "unique_selected_trades": int(len(unique_trades)),
        "rule_selected_rows": int(len(rule_trades)),
        "price_cache_rows": int(len(price_cache)),
        "price_cache_priced_rows": int(price_cache["price_found"].sum()),
        "price_cache_missing_rows": int((~price_cache["price_found"]).sum()),
        "spread_rows": int(len(spread_prices)),
        "complete_spread_rows": int(spread_prices["spread_complete"].sum()),
        "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
        "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
        "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
        "rule_priced_rows": int(len(rule_trades_priced)),
        "rule_complete_rows": int(rule_trades_priced["spread_complete"].sum()),
    },
    "notes": {
        "rsi_status": "Current selected-trade universe is provisional pending RSI repair. This cell is a mechanical pricing/P&L infrastructure step only.",
        "pricing_convention": "Conservative entry credit = short bid - long ask.",
        "pnl_convention": "Expiry P&L = conservative entry credit - terminal spread intrinsic before premium.",
        "not_done_here": "No final rule ranking, no strike fallback, no parameter lock, no production promotion.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F complete — high-level spread completeness")
print("=" * 100)
display(complete_spread_summary)

print()
print("=" * 100)
print("Leg price summary")
print("=" * 100)
display(leg_price_summary)

print()
print("=" * 100)
print("Spread pricing status summary")
print("=" * 100)
display(spread_status_summary)

print()
print("=" * 100)
print("Spread pricing status by tenor")
print("=" * 100)
display(spread_status_by_tenor)

print()
print("=" * 100)
print("Spread pricing status by year")
print("=" * 100)
display(spread_status_by_year)

print()
print("=" * 100)
print("Complete spread preview")
print("=" * 100)
display(complete_spread_preview)

print()
print("=" * 100)
print("Missing spread preview")
print("=" * 100)
if missing_spread_preview.empty:
    print("No missing spreads.")
else:
    display(missing_spread_preview)

print()
print("=" * 100)
print("Rule priced summary preview — not final ranking")
print("=" * 100)
display(rule_priced_summary.head(30))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Leg prices joined:          {LEG_PRICES_JOINED_PATH}")
print(f"Unique trades priced:       {UNIQUE_TRADES_PRICED_PATH}")
print(f"Rule trades priced:         {RULE_TRADES_PRICED_PATH}")
print(f"Spread status summary:      {SPREAD_STATUS_SUMMARY_PATH}")
print(f"Rule priced summary:        {RULE_PRICED_SUMMARY_PATH}")
print(f"Rule-year priced summary:   {RULE_YEAR_PRICED_SUMMARY_PATH}")
print(f"Manifest:                   {manifest_path}")

print()
print("Result: 2F pricing/P&L infrastructure complete.")
print("Next unaffected step: review missing short-vs-long leg impact before deciding whether a listed-strike fallback is needed.")

Cell 2F — Join long5_cal prices to selected spreads and compute spread P&L
Selected leg plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet
Unique trade universe:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet
Rule selected trades:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet
Price cache:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet

Loaded inputs
Leg plan rows:             3,482
Unique selected trades:    1,741
Rule selected rows:        155,550
Price cache rows:          3,437

Post-cleanup counts
Deduped price cache rows:  3,437
Priced cache rows:         2,825
Missing cache rows:        612



KeyError: 'entry_credit_pct_width'

In [26]:
# Cell 2F — Join long5_cal contract prices back to selected spreads and compute spread P&L
# Purpose:
#   Mechanical infrastructure step unaffected by the RSI repair.
#
# This cell:
#   - Joins the completed 2E long5_cal option price cache to the selected leg plan
#   - Builds spread-level pricing completeness
#   - Classifies missing short leg vs missing long leg
#   - Computes conservative entry credit for complete spreads
#   - Computes max loss, expiry P&L, and return on max loss
#   - Joins priced spread outcomes back to rule-selected trades
#
# This cell does NOT:
#   - Change RSI
#   - Change strikes
#   - Add fallback strike logic
#   - Rank final rules
#   - Select final parameters
#   - Promote anything to production

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_LEG_PLAN_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet"
)

UNIQUE_SELECTED_TRADE_UNIVERSE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet"
)

RULE_SELECTED_TRADES_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet"
)

PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F — Join long5_cal prices to selected spreads and compute spread P&L")
print("=" * 100)
print(f"Selected leg plan:        {SELECTED_LEG_PLAN_PATH}")
print(f"Unique trade universe:    {UNIQUE_SELECTED_TRADE_UNIVERSE_PATH}")
print(f"Rule selected trades:     {RULE_SELECTED_TRADES_PATH}")
print(f"Price cache:              {PRICE_CACHE_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def make_status(row):
    short_ok = bool(row.get("short_price_found", False))
    long_ok = bool(row.get("long_price_found", False))

    if short_ok and long_ok:
        return "complete"
    if (not short_ok) and long_ok:
        return "missing_short_only"
    if short_ok and (not long_ok):
        return "missing_long_only"
    return "missing_both"

def summarize_rule_group(g):
    complete = g.loc[g["spread_pricing_status"].eq("complete")].copy()

    out = {
        "selected_rows": len(g),
        "unique_selected_trades": g["selected_trade_id"].nunique(),
        "complete_rows": len(complete),
        "complete_unique_trades": complete["selected_trade_id"].nunique(),
        "coverage_rate": len(complete) / len(g) if len(g) else np.nan,
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "missing_short_only_rows": int(g["spread_pricing_status"].eq("missing_short_only").sum()),
        "missing_long_only_rows": int(g["spread_pricing_status"].eq("missing_long_only").sum()),
        "missing_both_rows": int(g["spread_pricing_status"].eq("missing_both").sum()),
    }

    if complete.empty:
        out.update({
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_max_loss": np.nan,
            "win_rate": np.nan,
            "avg_expiry_pnl": np.nan,
            "median_expiry_pnl": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
            "avg_terminal_intrinsic": np.nan,
            "worst_terminal_intrinsic": np.nan,
        })
        return pd.Series(out)

    out.update({
        "avg_entry_credit": complete["entry_credit_conservative"].mean(),
        "median_entry_credit": complete["entry_credit_conservative"].median(),
        "avg_credit_pct_width": complete["credit_pct_width"].mean(),
        "median_credit_pct_width": complete["credit_pct_width"].median(),
        "avg_max_loss": complete["max_loss_conservative"].mean(),
        "win_rate": complete["expiry_pnl_conservative"].gt(0).mean(),
        "avg_expiry_pnl": complete["expiry_pnl_conservative"].mean(),
        "median_expiry_pnl": complete["expiry_pnl_conservative"].median(),
        "avg_return_on_max_loss": complete["return_on_max_loss_conservative"].mean(),
        "median_return_on_max_loss": complete["return_on_max_loss_conservative"].median(),
        "worst_return_on_max_loss": complete["return_on_max_loss_conservative"].min(),
        "p05_return_on_max_loss": complete["return_on_max_loss_conservative"].quantile(0.05),
        "p95_return_on_max_loss": complete["return_on_max_loss_conservative"].quantile(0.95),
        "avg_terminal_intrinsic": complete["terminal_spread_intrinsic_before_premium"].mean(),
        "worst_terminal_intrinsic": complete["terminal_spread_intrinsic_before_premium"].max(),
    })

    return pd.Series(out)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    SELECTED_LEG_PLAN_PATH,
    UNIQUE_SELECTED_TRADE_UNIVERSE_PATH,
    RULE_SELECTED_TRADES_PATH,
    PRICE_CACHE_PATH,
]:
    require_file(p)

leg_plan = pd.read_parquet(SELECTED_LEG_PLAN_PATH).copy()
unique_trades = pd.read_parquet(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH).copy()
rule_trades = pd.read_parquet(RULE_SELECTED_TRADES_PATH).copy()
price_cache = pd.read_parquet(PRICE_CACHE_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Leg plan rows:             {len(leg_plan):,}")
print(f"Unique selected trades:    {len(unique_trades):,}")
print(f"Rule selected rows:        {len(rule_trades):,}")
print(f"Price cache rows:          {len(price_cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Basic validation / cleanup
# --------------------------------------------------------------------------------------------------

required_leg_cols = [
    "selected_trade_id",
    "contract_request_id",
    "leg_label",
    "leg_side",
    "strike_listed",
    "trade_date",
    "expiration_date",
    "trade_date_str",
    "expiration_str",
]

required_trade_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
]

required_rule_cols = [
    "selected_trade_id",
    "rule_id",
    "selection_rule",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
]

required_cache_cols = [
    "contract_request_id",
    "price_found",
    "price_status",
    "bid",
    "ask",
    "mid",
]

missing_leg = [c for c in required_leg_cols if c not in leg_plan.columns]
missing_trade = [c for c in required_trade_cols if c not in unique_trades.columns]
missing_rule = [c for c in required_rule_cols if c not in rule_trades.columns]
missing_cache = [c for c in required_cache_cols if c not in price_cache.columns]

if missing_leg:
    raise ValueError(f"Missing required leg plan columns: {missing_leg}")
if missing_trade:
    raise ValueError(f"Missing required unique trade columns: {missing_trade}")
if missing_rule:
    raise ValueError(f"Missing required rule trade columns: {missing_rule}")
if missing_cache:
    raise ValueError(f"Missing required price cache columns: {missing_cache}")

for df in [leg_plan, unique_trades, rule_trades, price_cache]:
    if "selected_trade_id" in df.columns:
        df["selected_trade_id"] = df["selected_trade_id"].astype(str)
    if "contract_request_id" in df.columns:
        df["contract_request_id"] = df["contract_request_id"].astype(str)

for df in [leg_plan, unique_trades, rule_trades]:
    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce").dt.normalize()
    if "expiration_date" in df.columns:
        df["expiration_date"] = pd.to_datetime(df["expiration_date"], errors="coerce").dt.normalize()

price_cache["price_found"] = price_cache["price_found"].fillna(False).astype(bool)

if "pulled_at" in price_cache.columns:
    price_cache["pulled_at"] = price_cache["pulled_at"].astype(str)
    price_cache = (
        price_cache
        .sort_values(["contract_request_id", "pulled_at"])
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )
else:
    price_cache = (
        price_cache
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )

for col in [
    "bid", "ask", "mid", "strike_listed", "underlying_price",
    "implied_vol", "delta", "theta", "vega", "gamma", "rho",
]:
    if col in price_cache.columns:
        price_cache[col] = pd.to_numeric(price_cache[col], errors="coerce")

print("=" * 100)
print("Post-cleanup counts")
print("=" * 100)
print(f"Deduped price cache rows:  {len(price_cache):,}")
print(f"Priced cache rows:         {int(price_cache['price_found'].sum()):,}")
print(f"Missing cache rows:        {int((~price_cache['price_found']).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Join cache to leg plan
# --------------------------------------------------------------------------------------------------

cache_cols = [
    "contract_request_id",
    "price_found",
    "price_status",
    "status_code",
    "bid",
    "ask",
    "mid",
    "bid_size",
    "ask_size",
    "underlying_price",
    "implied_vol",
    "delta",
    "theta",
    "vega",
    "gamma",
    "rho",
    "raw_text_preview",
    "pulled_at",
]

cache_cols = [c for c in cache_cols if c in price_cache.columns]

leg_prices = leg_plan.merge(
    price_cache[cache_cols],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_prices["price_found"] = leg_prices["price_found"].fillna(False).astype(bool)
leg_prices["price_status"] = leg_prices["price_status"].fillna("not_in_cache")

leg_prices["entry_leg_price_conservative"] = np.where(
    (leg_prices["leg_side"].eq("sell")) & (leg_prices["price_found"]),
    leg_prices["bid"],
    np.where(
        (leg_prices["leg_side"].eq("buy")) & (leg_prices["price_found"]),
        leg_prices["ask"],
        np.nan,
    )
)

leg_prices["entry_leg_price_mid"] = np.where(
    leg_prices["price_found"],
    leg_prices["mid"],
    np.nan,
)

leg_prices["entry_leg_cashflow_conservative"] = np.where(
    leg_prices["leg_side"].eq("sell"),
    leg_prices["entry_leg_price_conservative"],
    -leg_prices["entry_leg_price_conservative"],
)

leg_prices["entry_leg_cashflow_mid"] = np.where(
    leg_prices["leg_side"].eq("sell"),
    leg_prices["entry_leg_price_mid"],
    -leg_prices["entry_leg_price_mid"],
)

# --------------------------------------------------------------------------------------------------
# Build spread-level table from leg prices
# --------------------------------------------------------------------------------------------------

leg_core_cols = [
    "selected_trade_id",
    "leg_label",
    "contract_request_id",
    "strike_listed",
    "price_found",
    "price_status",
    "bid",
    "ask",
    "mid",
    "bid_size",
    "ask_size",
    "underlying_price",
    "implied_vol",
    "delta",
    "theta",
    "vega",
    "gamma",
    "rho",
    "entry_leg_price_conservative",
    "entry_leg_price_mid",
    "entry_leg_cashflow_conservative",
    "entry_leg_cashflow_mid",
]

leg_core_cols = [c for c in leg_core_cols if c in leg_prices.columns]

short_leg = (
    leg_prices.loc[leg_prices["leg_label"].eq("short_call_1sd"), leg_core_cols]
    .copy()
    .rename(columns={
        "contract_request_id": "short_contract_request_id",
        "strike_listed": "short_leg_strike",
        "price_found": "short_price_found",
        "price_status": "short_price_status",
        "bid": "short_bid",
        "ask": "short_ask",
        "mid": "short_mid",
        "bid_size": "short_bid_size",
        "ask_size": "short_ask_size",
        "underlying_price": "short_underlying_price",
        "implied_vol": "short_implied_vol",
        "delta": "short_delta",
        "theta": "short_theta",
        "vega": "short_vega",
        "gamma": "short_gamma",
        "rho": "short_rho",
        "entry_leg_price_conservative": "short_entry_price_conservative",
        "entry_leg_price_mid": "short_entry_price_mid",
        "entry_leg_cashflow_conservative": "short_entry_cashflow_conservative",
        "entry_leg_cashflow_mid": "short_entry_cashflow_mid",
    })
    .drop(columns=["leg_label"], errors="ignore")
)

long_leg = (
    leg_prices.loc[leg_prices["leg_label"].eq("long_call_3sd"), leg_core_cols]
    .copy()
    .rename(columns={
        "contract_request_id": "long_contract_request_id",
        "strike_listed": "long_leg_strike",
        "price_found": "long_price_found",
        "price_status": "long_price_status",
        "bid": "long_bid",
        "ask": "long_ask",
        "mid": "long_mid",
        "bid_size": "long_bid_size",
        "ask_size": "long_ask_size",
        "underlying_price": "long_underlying_price",
        "implied_vol": "long_implied_vol",
        "delta": "long_delta",
        "theta": "long_theta",
        "vega": "long_vega",
        "gamma": "long_gamma",
        "rho": "long_rho",
        "entry_leg_price_conservative": "long_entry_price_conservative",
        "entry_leg_price_mid": "long_entry_price_mid",
        "entry_leg_cashflow_conservative": "long_entry_cashflow_conservative",
        "entry_leg_cashflow_mid": "long_entry_cashflow_mid",
    })
    .drop(columns=["leg_label"], errors="ignore")
)

short_dupes = short_leg["selected_trade_id"].duplicated().sum()
long_dupes = long_leg["selected_trade_id"].duplicated().sum()

if short_dupes:
    print(f"WARNING: duplicate short leg selected_trade_id rows found: {short_dupes:,}")
if long_dupes:
    print(f"WARNING: duplicate long leg selected_trade_id rows found: {long_dupes:,}")

short_leg = short_leg.drop_duplicates("selected_trade_id", keep="last")
long_leg = long_leg.drop_duplicates("selected_trade_id", keep="last")

spread_prices = unique_trades.merge(
    short_leg,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices = spread_prices.merge(
    long_leg,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices["short_price_found"] = spread_prices["short_price_found"].fillna(False).astype(bool)
spread_prices["long_price_found"] = spread_prices["long_price_found"].fillna(False).astype(bool)

spread_prices["spread_pricing_status"] = spread_prices.apply(make_status, axis=1)
spread_prices["spread_complete"] = spread_prices["spread_pricing_status"].eq("complete")

spread_prices["entry_credit_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_bid"] - spread_prices["long_ask"],
    np.nan,
)

spread_prices["entry_credit_mid"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_mid"] - spread_prices["long_mid"],
    np.nan,
)

spread_prices["entry_credit_optimistic"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_ask"] - spread_prices["long_bid"],
    np.nan,
)

spread_prices["entry_credit_conservative_x100"] = spread_prices["entry_credit_conservative"] * 100.0
spread_prices["entry_credit_mid_x100"] = spread_prices["entry_credit_mid"] * 100.0

spread_prices["credit_pct_width"] = safe_divide(
    spread_prices["entry_credit_conservative"],
    spread_prices["listed_width"],
)

spread_prices["max_loss_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["listed_width"] - spread_prices["entry_credit_conservative"],
    np.nan,
)

spread_prices["max_loss_conservative_x100"] = spread_prices["max_loss_conservative"] * 100.0

spread_prices["expiry_pnl_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["entry_credit_conservative"] - spread_prices["terminal_spread_intrinsic_before_premium"],
    np.nan,
)

spread_prices["expiry_pnl_conservative_x100"] = spread_prices["expiry_pnl_conservative"] * 100.0

spread_prices["return_on_max_loss_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["max_loss_conservative"],
)

spread_prices["return_on_width_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["listed_width"],
)

spread_prices["trade_win_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["expiry_pnl_conservative"] > 0,
    np.nan,
)

spread_prices["invalid_credit_flag"] = (
    spread_prices["spread_complete"]
    & (
        spread_prices["entry_credit_conservative"].isna()
        | (spread_prices["entry_credit_conservative"] <= 0)
        | (spread_prices["max_loss_conservative"] <= 0)
    )
)

spread_prices["trade_year"] = pd.to_datetime(spread_prices["trade_date"]).dt.year

# --------------------------------------------------------------------------------------------------
# Join spread outcomes back to rule-selected trades
# --------------------------------------------------------------------------------------------------

spread_outcome_cols = [
    "selected_trade_id",
    "spread_pricing_status",
    "spread_complete",
    "short_price_found",
    "long_price_found",
    "short_price_status",
    "long_price_status",
    "short_contract_request_id",
    "long_contract_request_id",
    "short_bid",
    "short_ask",
    "short_mid",
    "long_bid",
    "long_ask",
    "long_mid",
    "entry_credit_conservative",
    "entry_credit_mid",
    "entry_credit_optimistic",
    "entry_credit_conservative_x100",
    "entry_credit_mid_x100",
    "credit_pct_width",
    "max_loss_conservative",
    "max_loss_conservative_x100",
    "expiry_pnl_conservative",
    "expiry_pnl_conservative_x100",
    "return_on_max_loss_conservative",
    "return_on_width_conservative",
    "trade_win_conservative",
    "invalid_credit_flag",
]

spread_outcome_cols = [c for c in spread_outcome_cols if c in spread_prices.columns]

rule_trades_priced = rule_trades.drop(
    columns=[c for c in spread_outcome_cols if c in rule_trades.columns and c != "selected_trade_id"],
    errors="ignore",
).merge(
    spread_prices[spread_outcome_cols],
    on="selected_trade_id",
    how="left",
    validate="many_to_one",
)

rule_trades_priced["spread_complete"] = rule_trades_priced["spread_complete"].fillna(False).astype(bool)
rule_trades_priced["spread_pricing_status"] = rule_trades_priced["spread_pricing_status"].fillna("not_in_spread_table")
rule_trades_priced["trade_year"] = pd.to_datetime(rule_trades_priced["trade_date"]).dt.year

# --------------------------------------------------------------------------------------------------
# Audits and summaries
# --------------------------------------------------------------------------------------------------

leg_price_summary = (
    leg_prices
    .groupby(["leg_label", "price_found", "price_status"], dropna=False)
    .size()
    .reset_index(name="leg_count")
    .sort_values(["leg_label", "price_found", "price_status"])
)

spread_status_summary = (
    spread_prices
    .groupby("spread_pricing_status", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        avg_tenor=("tenor", "mean"),
        avg_listed_width=("listed_width", "mean"),
    )
    .reset_index()
    .sort_values("unique_spreads", ascending=False)
)

spread_status_by_year = (
    spread_prices
    .groupby(["trade_year", "spread_pricing_status"], dropna=False)
    .size()
    .reset_index(name="unique_spreads")
    .sort_values(["trade_year", "spread_pricing_status"])
)

spread_status_by_tenor = (
    spread_prices
    .groupby(["tenor", "spread_pricing_status"], dropna=False)
    .size()
    .reset_index(name="unique_spreads")
    .sort_values(["tenor", "spread_pricing_status"])
)

complete_spreads = spread_prices.loc[spread_prices["spread_complete"]].copy()

complete_spread_summary = pd.DataFrame([{
    "unique_spreads_total": len(spread_prices),
    "unique_spreads_complete": len(complete_spreads),
    "complete_rate": len(complete_spreads) / len(spread_prices) if len(spread_prices) else np.nan,
    "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
    "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
    "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
    "invalid_credit_count": int(spread_prices["invalid_credit_flag"].sum()),
    "avg_entry_credit": complete_spreads["entry_credit_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_entry_credit": complete_spreads["entry_credit_conservative"].median() if not complete_spreads.empty else np.nan,
    "avg_credit_pct_width": complete_spreads["credit_pct_width"].mean() if not complete_spreads.empty else np.nan,
    "median_credit_pct_width": complete_spreads["credit_pct_width"].median() if not complete_spreads.empty else np.nan,
    "avg_max_loss": complete_spreads["max_loss_conservative"].mean() if not complete_spreads.empty else np.nan,
    "win_rate": complete_spreads["trade_win_conservative"].mean() if not complete_spreads.empty else np.nan,
    "avg_expiry_pnl": complete_spreads["expiry_pnl_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_expiry_pnl": complete_spreads["expiry_pnl_conservative"].median() if not complete_spreads.empty else np.nan,
    "avg_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].mean() if not complete_spreads.empty else np.nan,
    "median_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].median() if not complete_spreads.empty else np.nan,
    "worst_return_on_max_loss": complete_spreads["return_on_max_loss_conservative"].min() if not complete_spreads.empty else np.nan,
}])

rule_group_cols = [
    "rule_id",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
    "selection_rule",
]

rule_group_cols = [c for c in rule_group_cols if c in rule_trades_priced.columns]

rule_priced_summary = (
    rule_trades_priced
    .groupby(rule_group_cols, dropna=False)
    .apply(summarize_rule_group)
    .reset_index()
    .sort_values(["rule_id"])
)

rule_year_priced_summary = (
    rule_trades_priced
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        complete_rows=("spread_complete", "sum"),
        unique_selected_trades=("selected_trade_id", "nunique"),
        avg_entry_credit=("entry_credit_conservative", "mean"),
        avg_credit_pct_width=("credit_pct_width", "mean"),
        win_rate=("trade_win_conservative", "mean"),
        avg_expiry_pnl=("expiry_pnl_conservative", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative", "min"),
    )
    .reset_index()
)

rule_year_priced_summary["coverage_rate"] = safe_divide(
    rule_year_priced_summary["complete_rows"],
    rule_year_priced_summary["selected_rows"],
)

missing_spread_preview_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "spread_pricing_status",
    "short_contract_request_id",
    "short_price_status",
    "long_contract_request_id",
    "long_price_status",
]

missing_spread_preview_cols = [c for c in missing_spread_preview_cols if c in spread_prices.columns]

missing_spread_preview = (
    spread_prices.loc[~spread_prices["spread_complete"], missing_spread_preview_cols]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(100)
)

complete_spread_preview_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "short_bid",
    "long_ask",
    "entry_credit_conservative",
    "credit_pct_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
    "expiry_pnl_conservative",
    "return_on_max_loss_conservative",
]

complete_spread_preview_cols = [c for c in complete_spread_preview_cols if c in spread_prices.columns]

complete_spread_preview = (
    spread_prices.loc[spread_prices["spread_complete"], complete_spread_preview_cols]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(100)
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

LEG_PRICES_JOINED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_leg_prices_joined_long5_cal_v1.parquet"
)

UNIQUE_TRADES_PRICED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_long5_cal_v1.parquet"
)

RULE_TRADES_PRICED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_priced_long5_cal_v1.parquet"
)

SPREAD_STATUS_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_spread_pricing_status_summary_long5_cal_v1.parquet"
)

RULE_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_priced_summary_long5_cal_v1.parquet"
)

RULE_YEAR_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_priced_summary_long5_cal_v1.parquet"
)

leg_prices.to_parquet(LEG_PRICES_JOINED_PATH, index=False)
spread_prices.to_parquet(UNIQUE_TRADES_PRICED_PATH, index=False)
rule_trades_priced.to_parquet(RULE_TRADES_PRICED_PATH, index=False)
spread_status_summary.to_parquet(SPREAD_STATUS_SUMMARY_PATH, index=False)
rule_priced_summary.to_parquet(RULE_PRICED_SUMMARY_PATH, index=False)
rule_year_priced_summary.to_parquet(RULE_YEAR_PRICED_SUMMARY_PATH, index=False)

leg_price_summary_csv = AUDIT_DIR / f"02F_leg_price_summary_long5_cal_{timestamp}.csv"
spread_status_summary_csv = AUDIT_DIR / f"02F_spread_status_summary_long5_cal_{timestamp}.csv"
spread_status_by_year_csv = AUDIT_DIR / f"02F_spread_status_by_year_long5_cal_{timestamp}.csv"
spread_status_by_tenor_csv = AUDIT_DIR / f"02F_spread_status_by_tenor_long5_cal_{timestamp}.csv"
complete_spread_summary_csv = AUDIT_DIR / f"02F_complete_spread_summary_long5_cal_{timestamp}.csv"
missing_spread_preview_csv = AUDIT_DIR / f"02F_missing_spread_preview_long5_cal_{timestamp}.csv"
complete_spread_preview_csv = AUDIT_DIR / f"02F_complete_spread_preview_long5_cal_{timestamp}.csv"
rule_priced_summary_csv = AUDIT_DIR / f"02F_rule_priced_summary_long5_cal_{timestamp}.csv"
rule_year_priced_summary_csv = AUDIT_DIR / f"02F_rule_year_priced_summary_long5_cal_{timestamp}.csv"

leg_price_summary.to_csv(leg_price_summary_csv, index=False)
spread_status_summary.to_csv(spread_status_summary_csv, index=False)
spread_status_by_year.to_csv(spread_status_by_year_csv, index=False)
spread_status_by_tenor.to_csv(spread_status_by_tenor_csv, index=False)
complete_spread_summary.to_csv(complete_spread_summary_csv, index=False)
missing_spread_preview.to_csv(missing_spread_preview_csv, index=False)
complete_spread_preview.to_csv(complete_spread_preview_csv, index=False)
rule_priced_summary.to_csv(rule_priced_summary_csv, index=False)
rule_year_priced_summary.to_csv(rule_year_priced_summary_csv, index=False)

manifest_path = AUDIT_DIR / f"02F_join_prices_spread_pnl_long5_cal_manifest_{timestamp}.json"

manifest = {
    "cell": "2F",
    "description": "Join long5_cal contract prices back to selected call spreads and compute spread P&L",
    "created_at": timestamp,
    "inputs": {
        "selected_leg_plan_path": str(SELECTED_LEG_PLAN_PATH),
        "unique_selected_trade_universe_path": str(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH),
        "rule_selected_trades_path": str(RULE_SELECTED_TRADES_PATH),
        "price_cache_path": str(PRICE_CACHE_PATH),
    },
    "outputs": {
        "leg_prices_joined_path": str(LEG_PRICES_JOINED_PATH),
        "unique_trades_priced_path": str(UNIQUE_TRADES_PRICED_PATH),
        "rule_trades_priced_path": str(RULE_TRADES_PRICED_PATH),
        "spread_status_summary_path": str(SPREAD_STATUS_SUMMARY_PATH),
        "rule_priced_summary_path": str(RULE_PRICED_SUMMARY_PATH),
        "rule_year_priced_summary_path": str(RULE_YEAR_PRICED_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "leg_plan_rows": int(len(leg_plan)),
        "unique_selected_trades": int(len(unique_trades)),
        "rule_selected_rows": int(len(rule_trades)),
        "price_cache_rows": int(len(price_cache)),
        "price_cache_priced_rows": int(price_cache["price_found"].sum()),
        "price_cache_missing_rows": int((~price_cache["price_found"]).sum()),
        "spread_rows": int(len(spread_prices)),
        "complete_spread_rows": int(spread_prices["spread_complete"].sum()),
        "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
        "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
        "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
        "rule_priced_rows": int(len(rule_trades_priced)),
        "rule_complete_rows": int(rule_trades_priced["spread_complete"].sum()),
    },
    "notes": {
        "rsi_status": "Current selected-trade universe is provisional pending RSI repair. This cell is a mechanical pricing/P&L infrastructure step only.",
        "pricing_convention": "Conservative entry credit = short bid - long ask.",
        "pnl_convention": "Expiry P&L = conservative entry credit - terminal spread intrinsic before premium.",
        "not_done_here": "No final rule ranking, no strike fallback, no parameter lock, no production promotion.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F complete — high-level spread completeness")
print("=" * 100)
display(complete_spread_summary)

print()
print("=" * 100)
print("Leg price summary")
print("=" * 100)
display(leg_price_summary)

print()
print("=" * 100)
print("Spread pricing status summary")
print("=" * 100)
display(spread_status_summary)

print()
print("=" * 100)
print("Spread pricing status by tenor")
print("=" * 100)
display(spread_status_by_tenor)

print()
print("=" * 100)
print("Spread pricing status by year")
print("=" * 100)
display(spread_status_by_year)

print()
print("=" * 100)
print("Complete spread preview")
print("=" * 100)
display(complete_spread_preview)

print()
print("=" * 100)
print("Missing spread preview")
print("=" * 100)
if missing_spread_preview.empty:
    print("No missing spreads.")
else:
    display(missing_spread_preview)

print()
print("=" * 100)
print("Rule priced summary preview — not final ranking")
print("=" * 100)
display(rule_priced_summary.head(30))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Leg prices joined:          {LEG_PRICES_JOINED_PATH}")
print(f"Unique trades priced:       {UNIQUE_TRADES_PRICED_PATH}")
print(f"Rule trades priced:         {RULE_TRADES_PRICED_PATH}")
print(f"Spread status summary:      {SPREAD_STATUS_SUMMARY_PATH}")
print(f"Rule priced summary:        {RULE_PRICED_SUMMARY_PATH}")
print(f"Rule-year priced summary:   {RULE_YEAR_PRICED_SUMMARY_PATH}")
print(f"Manifest:                   {manifest_path}")

print()
print("Result: 2F pricing/P&L infrastructure complete.")
print("Next unaffected step: review missing short-vs-long leg impact before deciding whether a listed-strike fallback is needed.")

Cell 2F — Join long5_cal prices to selected spreads and compute spread P&L
Selected leg plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_long5_cal_v1.parquet
Unique trade universe:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet
Rule selected trades:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet
Price cache:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet

Loaded inputs
Leg plan rows:             3,482
Unique selected trades:    1,741
Rule selected rows:        155,550
Price cache rows:          3,437

Post-cleanup counts
Deduped price cache rows:  3,437
Priced cache rows:         2,825
Missing cache rows:        612



C:\Users\patri\AppData\Local\Temp\ipykernel_6252\122836037.py:667: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_rule_group)



Cell 2F complete — high-level spread completeness


,unique_spreads_total,unique_spreads_complete,complete_rate,missing_short_only,missing_long_only,missing_both,invalid_credit_count,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,avg_max_loss,win_rate,avg_expiry_pnl,median_expiry_pnl,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss
0,1741,1193,0.685238,430,53,65,0,0.752347,0.72,0.023811,0.022778,32.993671,0.859179,0.039908,0.65,0.000858,0.01983,-0.516209



Leg price summary


...


Missing spread preview


,selected_trade_id,trade_date,tenor,expiration_date,short_strike,long_strike,listed_width,spread_pricing_status,short_contract_request_id,short_price_status,long_contract_request_id,long_price_status
2,SPY_CALL_20201231_T33_EXP20210205_S400.0_L450.0,2020-12-31,33,2021-02-05,400.0,450.0,50.0,missing_long_only,SPY_20201231_20210205_400.0_C,priced,SPY_20201231_20210205_450.0_C,http_472
69,SPY_CALL_20211026_T33_EXP20211203_S479.0_L525.0,2021-10-26,33,2021-12-03,479.0,525.0,46.0,missing_both,SPY_20211026_20211203_479.0_C,http_472,SPY_20211026_20211203_525.0_C,http_472
79,SPY_CALL_20211102_T21_EXP20211126_S479.0_L515.0,2021-11-02,21,2021-11-26,479.0,515.0,36.0,missing_long_only,SPY_20211102_20211126_479.0_C,priced,SPY_20211102_20211126_515.0_C,http_472
89,SPY_CALL_20211104_T27_EXP20211203_S487.0_L525.0,2021-11-04,27,2021-12-03,487.0,525.0,38.0,missing_long_only,SPY_20211104_20211203_487.0_C,priced,SPY_20211104_20211203_525.0_C,http_472
91,SPY_CALL_20211104_T33_EXP20211210_S490.0_L535.0,2021-11-04,33,2021-12-10,490.0,535.0,45.0,missing_long_only,SPY_20211104_20211210_490.0_C,priced,SPY_20211104_20211210_535.0_C,http_472
...,...,...,...,...,...,...,...,...,...,...,...,...
640,SPY_CALL_20240212_T33_EXP20240322_S523.0_L565.0,2024-02-12,33,2024-03-22,523.0,565.0,42.0,missing_short_only,SPY_20240212_20240322_523.0_C,http_472,SPY_20240212_20240322_565.0_C,priced
644,SPY_CALL_20240213_T33_EXP20240322_S519.0_L565.0,2024-02-13,33,2024-03-22,519.0,565.0,46.0,missing_short_only,SPY_20240213_20240322_519.0_C,http_472,SPY_20240213_20240322_565.0_C,priced
649,SPY_CALL_20240220_T30_EXP20240322_S519.0_L560.0,2024-02-20,30,2024-03-22,519.0,560.0,41.0,missing_short_only,SPY_20240220_20240322_519.0_C,http_472,SPY_20240220_20240322_560.0_C,priced
655,SPY_CALL_20240221_T30_EXP20240322_S519.0_L560.0,2024-02-21,30,2024-03-22,519.0,560.0,41.0,missing_short_only,SPY_20240221_20240322_519.0_C,http_472,SPY_20240221_20240322_560.0_C,priced



Rule priced summary preview — not final ranking


,rule_id,z_threshold_shared,rsi_floor,vrp_log_floor_label,selection_rule,selected_rows,unique_selected_trades,complete_rows,complete_unique_trades,coverage_rate,...,win_rate,avg_expiry_pnl,median_expiry_pnl,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,p95_return_on_max_loss,avg_terminal_intrinsic,worst_terminal_intrinsic
0,Z0p00_RSI55_VRP0p00_SEL_highest_vrp_log,0.0,55,0.00,highest_vrp_log,375,375,277,277,0.738667,...,0.862816,0.147545,0.600,0.003775,0.021450,-0.352771,-0.139780,0.042980,0.568195,10.65
1,Z0p00_RSI55_VRP0p00_SEL_highest_z_avg,0.0,55,0.00,highest_z_avg,375,375,245,245,0.653333,...,0.844898,0.066531,0.640,0.001251,0.019231,-0.396081,-0.149826,0.040210,0.635347,13.20
2,Z0p00_RSI55_VRP0p00_SEL_highest_z_min,0.0,55,0.00,highest_z_min,375,375,253,253,0.674667,...,0.865613,0.162451,0.620,0.004020,0.019628,-0.396081,-0.123820,0.041053,0.535178,13.20
3,Z0p00_RSI55_VRP0p00_SEL_longest_tenor,0.0,55,0.00,longest_tenor,375,375,184,184,0.490667,...,0.842391,0.014130,0.700,-0.000853,0.018279,-0.323449,-0.159343,0.038666,0.809728,13.85
4,Z0p00_RSI55_VRP0p00_SEL_shortest_tenor,0.0,55,0.00,shortest_tenor,375,375,328,328,0.874667,...,0.841463,-0.131921,0.530,-0.001774,0.021109,-0.352771,-0.189434,0.045854,0.808720,17.62
5,Z0p00_RSI55_VRP0p20_SEL_highest_vrp_log,0.0,55,0.20,highest_vrp_log,375,375,277,277,0.738667,...,0.862816,0.147545,0.600,0.003775,0.021450,-0.352771,-0.139780,0.042980,0.568195,10.65
6,Z0p00_RSI55_VRP0p20_SEL_highest_z_avg,0.0,55,0.20,highest_z_avg,375,375,245,245,0.653333,...,0.844898,0.066531,0.640,0.001251,0.019231,-0.396081,-0.149826,0.040210,0.635347,13.20
7,Z0p00_RSI55_VRP0p20_SEL_highest_z_min,0.0,55,0.20,highest_z_min,375,375,253,253,0.674667,...,0.865613,0.162451,0.620,0.004020,0.019628,-0.396081,-0.123820,0.041053,0.535178,13.20
8,Z0p00_RSI55_VRP0p20_SEL_longest_tenor,0.0,55,0.20,longest_tenor,375,375,184,184,0.490667,...,0.842391,0.014130,0.700,-0.000853,0.018279,-0.323449,-0.159343,0.038666,0.809728,13.85
9,Z0p00_RSI55_VRP0p20_SEL_shortest_tenor,0.0,55,0.20,shortest_tenor,375,375,328,328,0.874667,...,0.841463,-0.131921,0.530,-0.001774,0.021109,-0.352771,-0.189434,0.045854,0.808720,17.62



Saved
Leg prices joined:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_prices_joined_long5_cal_v1.parquet
Unique trades priced:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_long5_cal_v1.parquet
Rule trades priced:         C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_long5_cal_v1.parquet
Spread status summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_spread_pricing_status_summary_long5_cal_v1.parquet
Rule priced summary:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_priced_summary_long5_cal_v1.parquet
Rule-year priced summary:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_priced_summary_long5_cal_v1.parquet
Manifest:      

In [27]:
# Cell 2F.1 — Missing contract fallback diagnostic for listed-strike availability
# Purpose:
#   Mechanical infrastructure/audit step unaffected by the RSI repair.
#
# This cell:
#   - Isolates missing short and long option legs from Cell 2F
#   - Tests nearby listed strikes using the same ThetaData v3 EOD greeks endpoint
#   - Builds a fallback map showing whether missing legs can be recovered
#   - Computes hypothetical spread completeness/P&L after fallback
#
# This cell does NOT:
#   - Change the original selected trade universe
#   - Change the original 2F priced spread file
#   - Rank final rules
#   - Select final parameters
#   - Promote anything to production
#
# Fallback preference:
#   Short call fallback:
#       Prefer nearest listed strike AT OR ABOVE the target short strike,
#       while still below the long strike.
#       Reason: selling below target is more aggressive than intended.
#
#   Long call fallback:
#       Prefer nearest listed strike AT OR BELOW the target long strike,
#       while still above the short strike.
#       Reason: buying above target is less protective / more aggressive than intended.
#
# Candidate ranges:
#   Short missing legs: target + [0, +1, +2, +3, +4, +5, -1, -2, -3, -4, -5]
#   Long missing legs:  target + [0, -5, +5, -10, +10]
#
# Runtime note:
#   This may make several thousand ThetaData calls. It checkpoints and can resume.
#   To test only part of the diagnostic first, set MAX_CANDIDATE_REQUESTS_THIS_RUN = 1000.

from pathlib import Path
from datetime import datetime
from io import StringIO
import json
import time
import traceback

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

UNIQUE_TRADES_PRICED_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_long5_cal_v1.parquet"
)

ORIGINAL_PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet"
)

FALLBACK_CANDIDATES_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_missing_contract_fallback_candidates_long5_cal_v1.parquet"
)

FALLBACK_CANDIDATE_PRICE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_missing_contract_fallback_map_long5_cal_v1.parquet"
)

SPREADS_AFTER_FALLBACK_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet"
)

BASE_URL = "http://127.0.0.1:25503"
ENDPOINT = "/v3/option/history/greeks/eod"

SHORT_OFFSETS = [0, 1, 2, 3, 4, 5, -1, -2, -3, -4, -5]
LONG_OFFSETS = [0, -5, 5, -10, 10]

CHECKPOINT_EVERY = 200
PROGRESS_EVERY = 50
REQUEST_SLEEP_SECONDS = 0.02

# Set to 1000 for first test run if desired. None = run all remaining fallback candidates.
MAX_CANDIDATE_REQUESTS_THIS_RUN = None

RETRY_FAILED_OR_MISSING = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_AUDIT_PATH = AUDIT_DIR / f"02F1_fallback_candidate_price_run_audit_{timestamp}.csv"
MANIFEST_PATH = AUDIT_DIR / f"02F1_fallback_diagnostic_manifest_{timestamp}.json"

session = requests.Session()
session.trust_env = False

print("=" * 100)
print("Cell 2F.1 — Missing contract fallback diagnostic")
print("=" * 100)
print(f"Priced spread file:          {UNIQUE_TRADES_PRICED_PATH}")
print(f"Original price cache:        {ORIGINAL_PRICE_CACHE_PATH}")
print(f"Fallback candidates path:    {FALLBACK_CANDIDATES_PATH}")
print(f"Fallback price cache path:   {FALLBACK_CANDIDATE_PRICE_CACHE_PATH}")
print(f"Fallback map path:           {FALLBACK_MAP_PATH}")
print(f"Spreads after fallback path: {SPREADS_AFTER_FALLBACK_PATH}")
print(f"ThetaData endpoint:          {BASE_URL}{ENDPOINT}")
print(f"Max candidate requests:      {MAX_CANDIDATE_REQUESTS_THIS_RUN}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def normalize_right(x):
    s = str(x).upper()
    if s in {"CALL", "C"}:
        return "C"
    if s in {"PUT", "P"}:
        return "P"
    return s[:1] if s else None

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        obj = resp.json()

        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None

                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break

                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)

                return pd.DataFrame(data)

            if "data" in obj:
                data = obj.get("data")
                if isinstance(data, list):
                    return pd.DataFrame(data)
                if isinstance(data, dict):
                    return pd.json_normalize(data)

            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass

    try:
        return pd.read_json(StringIO(text), lines=True)
    except Exception:
        pass

    return pd.DataFrame({"raw_response": text.splitlines()})

def standardize_option_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    lower_to_col = {str(c).lower(): c for c in out.columns}

    mapping = {
        "symbol": ["symbol", "root", "underlying"],
        "expiration": ["expiration", "expiry", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type"],
        "timestamp": ["timestamp"],
        "created": ["created"],
        "last_trade": ["last_trade"],
        "bid": ["bid"],
        "ask": ["ask"],
        "mid": ["mid", "mark"],
        "bid_size": ["bid_size"],
        "ask_size": ["ask_size"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close"],
        "volume": ["volume"],
        "count": ["count"],
        "underlying_price": ["underlying_price"],
        "underlying_timestamp": ["underlying_timestamp"],
        "implied_vol": ["implied_vol", "iv"],
        "iv_error": ["iv_error"],
        "delta": ["delta"],
        "theta": ["theta"],
        "vega": ["vega"],
        "gamma": ["gamma"],
        "rho": ["rho"],
    }

    rename_map = {}

    for canonical, candidates in mapping.items():
        for cand in candidates:
            if cand in lower_to_col:
                rename_map[lower_to_col[cand]] = canonical
                break

    out = out.rename(columns=rename_map)

    numeric_cols = [
        "strike", "bid", "ask", "mid", "bid_size", "ask_size",
        "open", "high", "low", "close", "volume", "count",
        "underlying_price", "implied_vol", "iv_error",
        "delta", "theta", "vega", "gamma", "rho",
    ]

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    if "right" in out.columns:
        out["right"] = out["right"].apply(normalize_right)

    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    if "expiration" in out.columns:
        out["expiration"] = pd.to_datetime(out["expiration"], errors="coerce").dt.strftime("%Y-%m-%d")

    return out

def atomic_write_parquet(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def build_empty_candidate_cache():
    return pd.DataFrame(columns=[
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "endpoint",
        "status_code",
        "http_ok",
        "response_rows",
        "response_cols",
        "response_columns",
        "price_found",
        "price_status",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "timestamp",
        "created",
        "last_trade",
        "underlying_price",
        "underlying_timestamp",
        "implied_vol",
        "iv_error",
        "delta",
        "theta",
        "vega",
        "gamma",
        "rho",
        "raw_text_preview",
        "error",
        "request_url",
        "pulled_at",
    ])

def get_completed_ids(cache_df):
    if cache_df.empty:
        return set()

    if RETRY_FAILED_OR_MISSING:
        if "price_found" in cache_df.columns:
            return set(cache_df.loc[cache_df["price_found"] == True, "contract_request_id"].astype(str))
        return set()

    return set(cache_df["contract_request_id"].dropna().astype(str))

def price_contract(row):
    symbol = str(row["symbol"]).upper()
    right = normalize_right(row["right"])
    strike = float(row["strike_listed"])
    trade_date_str = str(row["trade_date_str"])
    expiration_str = str(row["expiration_str"])

    params = {
        "symbol": symbol,
        "right": right,
        "strike": strike,
        "expiration": expiration_str,
        "start_date": trade_date_str,
        "end_date": trade_date_str,
        "format": "csv",
    }

    url = BASE_URL.rstrip("/") + ENDPOINT
    pulled_at = datetime.now().isoformat(timespec="seconds")

    base_result = {
        "contract_request_id": str(row["contract_request_id"]),
        "symbol": symbol,
        "right": right,
        "trade_date": pd.Timestamp(row["trade_date"]),
        "trade_date_str": trade_date_str,
        "expiration_date": pd.Timestamp(row["expiration_date"]),
        "expiration_str": expiration_str,
        "expiration_yyyymmdd": str(row["expiration_yyyymmdd"]),
        "strike_listed": strike,
        "endpoint": ENDPOINT,
        "status_code": np.nan,
        "http_ok": False,
        "response_rows": 0,
        "response_cols": 0,
        "response_columns": "",
        "price_found": False,
        "price_status": "not_attempted",
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_size": np.nan,
        "ask_size": np.nan,
        "timestamp": None,
        "created": None,
        "last_trade": None,
        "underlying_price": np.nan,
        "underlying_timestamp": None,
        "implied_vol": np.nan,
        "iv_error": np.nan,
        "delta": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "gamma": np.nan,
        "rho": np.nan,
        "raw_text_preview": "",
        "error": None,
        "request_url": "",
        "pulled_at": pulled_at,
    }

    try:
        resp = session.get(url, params=params, timeout=(2, 45))

        base_result["status_code"] = resp.status_code
        base_result["http_ok"] = bool(resp.ok)
        base_result["raw_text_preview"] = resp.text[:500] if hasattr(resp, "text") else ""
        base_result["request_url"] = resp.url

        raw_df = response_to_dataframe(resp)
        std_df = standardize_option_rows(raw_df)

        base_result["response_rows"] = int(len(raw_df))
        base_result["response_cols"] = int(len(raw_df.columns))
        base_result["response_columns"] = ", ".join(map(str, raw_df.columns))

        if not resp.ok:
            base_result["price_status"] = f"http_{resp.status_code}"
            return base_result

        if std_df.empty:
            base_result["price_status"] = "empty_response"
            return base_result

        if not {"bid", "ask"}.issubset(std_df.columns):
            base_result["price_status"] = "missing_bid_ask_columns"
            return base_result

        valid = std_df.loc[
            std_df["bid"].notna()
            & std_df["ask"].notna()
        ].copy()

        if valid.empty:
            base_result["price_status"] = "no_valid_bid_ask"
            return base_result

        if {"strike", "right"}.issubset(valid.columns):
            exact_mask = (
                valid["strike"].round(4).eq(round(strike, 4))
                & valid["right"].eq(right)
            )

            if "expiration" in valid.columns:
                exact_mask = exact_mask & valid["expiration"].eq(expiration_str)

            exact = valid.loc[exact_mask].copy()

            if not exact.empty:
                valid = exact

        px = valid.iloc[-1]

        base_result["price_found"] = True
        base_result["price_status"] = "priced"

        for col in [
            "bid", "ask", "mid", "bid_size", "ask_size",
            "timestamp", "created", "last_trade",
            "underlying_price", "underlying_timestamp",
            "implied_vol", "iv_error",
            "delta", "theta", "vega", "gamma", "rho",
        ]:
            if col in valid.columns:
                base_result[col] = px.get(col, base_result[col])

        return base_result

    except Exception as e:
        base_result["price_status"] = "exception"
        base_result["error"] = repr(e)
        base_result["raw_text_preview"] = traceback.format_exc()[:500]
        return base_result

def save_candidate_cache(cache_df, run_audit_df=None):
    cache_df = cache_df.copy()

    if not cache_df.empty:
        cache_df["contract_request_id"] = cache_df["contract_request_id"].astype(str)
        cache_df["pulled_at"] = cache_df["pulled_at"].astype(str)

        cache_df = (
            cache_df
            .sort_values(["contract_request_id", "pulled_at"])
            .drop_duplicates("contract_request_id", keep="last")
            .sort_values(["trade_date_str", "expiration_str", "strike_listed", "right"])
            .reset_index(drop=True)
        )

    atomic_write_parquet(cache_df, FALLBACK_CANDIDATE_PRICE_CACHE_PATH)

    csv_snapshot = AUDIT_DIR / "02F1_fallback_candidate_contract_price_cache_latest.csv"
    atomic_write_csv(cache_df, csv_snapshot)

    if run_audit_df is not None:
        atomic_write_csv(run_audit_df, RUN_AUDIT_PATH)

    return cache_df

def choose_fallback_for_group(g):
    g = g.copy()
    leg_label = str(g["leg_label"].iloc[0])

    priced = g.loc[g["price_found"] == True].copy()

    if priced.empty:
        out = g.iloc[0].copy()
        out["fallback_found"] = False
        out["fallback_choice_type"] = "none"
        out["fallback_contract_request_id"] = None
        out["fallback_strike"] = np.nan
        out["fallback_bid"] = np.nan
        out["fallback_ask"] = np.nan
        out["fallback_mid"] = np.nan
        out["fallback_strike_slippage"] = np.nan
        out["fallback_abs_strike_slippage"] = np.nan
        out["fallback_price_status"] = None
        return out

    if leg_label == "short_call_1sd":
        preferred = priced.loc[
            (priced["candidate_strike"] >= priced["target_strike"])
            & (priced["candidate_strike"] < priced["long_strike"])
        ].copy()

        if not preferred.empty:
            preferred["sort_slippage"] = preferred["candidate_strike"] - preferred["target_strike"]
            chosen = preferred.sort_values(
                ["sort_slippage", "abs_strike_slippage", "candidate_strike"]
            ).iloc[0].copy()
            choice_type = "preferred_at_or_above_target"
        else:
            valid_any = priced.loc[
                priced["candidate_strike"] < priced["long_strike"]
            ].copy()

            if valid_any.empty:
                out = g.iloc[0].copy()
                out["fallback_found"] = False
                out["fallback_choice_type"] = "none_valid_below_long"
                out["fallback_contract_request_id"] = None
                out["fallback_strike"] = np.nan
                out["fallback_bid"] = np.nan
                out["fallback_ask"] = np.nan
                out["fallback_mid"] = np.nan
                out["fallback_strike_slippage"] = np.nan
                out["fallback_abs_strike_slippage"] = np.nan
                out["fallback_price_status"] = None
                return out

            chosen = valid_any.sort_values(
                ["abs_strike_slippage", "candidate_strike"]
            ).iloc[0].copy()
            choice_type = "nearest_any_valid"

    elif leg_label == "long_call_3sd":
        preferred = priced.loc[
            (priced["candidate_strike"] <= priced["target_strike"])
            & (priced["candidate_strike"] > priced["short_strike"])
        ].copy()

        if not preferred.empty:
            preferred["sort_slippage"] = preferred["target_strike"] - preferred["candidate_strike"]
            chosen = preferred.sort_values(
                ["sort_slippage", "abs_strike_slippage", "candidate_strike"],
                ascending=[True, True, False],
            ).iloc[0].copy()
            choice_type = "preferred_at_or_below_target"
        else:
            valid_any = priced.loc[
                priced["candidate_strike"] > priced["short_strike"]
            ].copy()

            if valid_any.empty:
                out = g.iloc[0].copy()
                out["fallback_found"] = False
                out["fallback_choice_type"] = "none_valid_above_short"
                out["fallback_contract_request_id"] = None
                out["fallback_strike"] = np.nan
                out["fallback_bid"] = np.nan
                out["fallback_ask"] = np.nan
                out["fallback_mid"] = np.nan
                out["fallback_strike_slippage"] = np.nan
                out["fallback_abs_strike_slippage"] = np.nan
                out["fallback_price_status"] = None
                return out

            chosen = valid_any.sort_values(
                ["abs_strike_slippage", "candidate_strike"]
            ).iloc[0].copy()
            choice_type = "nearest_any_valid"

    else:
        chosen = priced.sort_values(
            ["abs_strike_slippage", "candidate_strike"]
        ).iloc[0].copy()
        choice_type = "nearest_any_valid_unknown_leg"

    chosen["fallback_found"] = True
    chosen["fallback_choice_type"] = choice_type
    chosen["fallback_contract_request_id"] = chosen["candidate_contract_request_id"]
    chosen["fallback_strike"] = chosen["candidate_strike"]
    chosen["fallback_bid"] = chosen["bid"]
    chosen["fallback_ask"] = chosen["ask"]
    chosen["fallback_mid"] = chosen["mid"]
    chosen["fallback_strike_slippage"] = chosen["strike_slippage"]
    chosen["fallback_abs_strike_slippage"] = chosen["abs_strike_slippage"]
    chosen["fallback_price_status"] = chosen["price_status"]

    return chosen

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

require_file(UNIQUE_TRADES_PRICED_PATH)
require_file(ORIGINAL_PRICE_CACHE_PATH)

spreads = pd.read_parquet(UNIQUE_TRADES_PRICED_PATH).copy()
original_cache = pd.read_parquet(ORIGINAL_PRICE_CACHE_PATH).copy()

for c in ["trade_date", "expiration_date"]:
    if c in spreads.columns:
        spreads[c] = pd.to_datetime(spreads[c], errors="coerce").dt.normalize()
    if c in original_cache.columns:
        original_cache[c] = pd.to_datetime(original_cache[c], errors="coerce").dt.normalize()

for c in ["selected_trade_id", "short_contract_request_id", "long_contract_request_id"]:
    if c in spreads.columns:
        spreads[c] = spreads[c].astype(str)

original_cache["contract_request_id"] = original_cache["contract_request_id"].astype(str)
original_cache["price_found"] = original_cache["price_found"].fillna(False).astype(bool)

if "pulled_at" in original_cache.columns:
    original_cache["pulled_at"] = original_cache["pulled_at"].astype(str)
    original_cache = (
        original_cache
        .sort_values(["contract_request_id", "pulled_at"])
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )
else:
    original_cache = (
        original_cache
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Spread rows:             {len(spreads):,}")
print(f"Original cache rows:     {len(original_cache):,}")
print(f"Original priced rows:    {int(original_cache['price_found'].sum()):,}")
print(f"Original missing rows:   {int((~original_cache['price_found']).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Build missing-leg table
# --------------------------------------------------------------------------------------------------

missing_short = spreads.loc[
    spreads["short_price_found"] != True
].copy()

missing_short["leg_label"] = "short_call_1sd"
missing_short["target_strike"] = missing_short["short_strike"]
missing_short["original_contract_request_id"] = missing_short["short_contract_request_id"]

missing_long = spreads.loc[
    spreads["long_price_found"] != True
].copy()

missing_long["leg_label"] = "long_call_3sd"
missing_long["target_strike"] = missing_long["long_strike"]
missing_long["original_contract_request_id"] = missing_long["long_contract_request_id"]

missing_legs = pd.concat([missing_short, missing_long], ignore_index=True)

missing_legs["missing_leg_id"] = (
    missing_legs["selected_trade_id"].astype(str)
    + "__"
    + missing_legs["leg_label"].astype(str)
)

missing_legs["symbol"] = "SPY"
missing_legs["right"] = "C"
missing_legs["trade_date_str"] = missing_legs["trade_date"].dt.strftime("%Y-%m-%d")
missing_legs["expiration_str"] = missing_legs["expiration_date"].dt.strftime("%Y-%m-%d")
missing_legs["expiration_yyyymmdd"] = missing_legs["expiration_date"].dt.strftime("%Y%m%d")

print("=" * 100)
print("Missing legs")
print("=" * 100)
missing_leg_summary = (
    missing_legs
    .groupby("leg_label")
    .size()
    .reset_index(name="missing_legs")
)
display(missing_leg_summary)
print()

# --------------------------------------------------------------------------------------------------
# Generate fallback candidate contract list
# --------------------------------------------------------------------------------------------------

candidate_rows = []

for _, r in missing_legs.iterrows():
    leg_label = r["leg_label"]
    target = float(r["target_strike"])
    short_strike = float(r["short_strike"])
    long_strike = float(r["long_strike"])

    offsets = SHORT_OFFSETS if leg_label == "short_call_1sd" else LONG_OFFSETS

    for rank, offset in enumerate(offsets):
        candidate_strike = float(target + offset)

        if candidate_strike <= 0:
            continue

        # Basic structural validity.
        if leg_label == "short_call_1sd":
            structurally_valid = candidate_strike < long_strike
            preferred_side = candidate_strike >= target
        elif leg_label == "long_call_3sd":
            structurally_valid = candidate_strike > short_strike
            preferred_side = candidate_strike <= target
        else:
            structurally_valid = True
            preferred_side = False

        if not structurally_valid:
            continue

        candidate_contract_request_id = make_contract_request_id(
            symbol="SPY",
            trade_date=r["trade_date"],
            expiration_date=r["expiration_date"],
            strike=candidate_strike,
            right="C",
        )

        candidate_rows.append({
            "missing_leg_id": r["missing_leg_id"],
            "selected_trade_id": r["selected_trade_id"],
            "leg_label": leg_label,
            "symbol": "SPY",
            "right": "C",
            "trade_date": r["trade_date"],
            "trade_date_str": r["trade_date_str"],
            "expiration_date": r["expiration_date"],
            "expiration_str": r["expiration_str"],
            "expiration_yyyymmdd": r["expiration_yyyymmdd"],
            "tenor": r["tenor"],
            "target_strike": target,
            "candidate_strike": candidate_strike,
            "strike_slippage": candidate_strike - target,
            "abs_strike_slippage": abs(candidate_strike - target),
            "candidate_offset": offset,
            "candidate_rank": rank,
            "preferred_side": preferred_side,
            "short_strike": short_strike,
            "long_strike": long_strike,
            "listed_width_original": r["listed_width"],
            "original_contract_request_id": r["original_contract_request_id"],
            "candidate_contract_request_id": candidate_contract_request_id,
            "endpoint": ENDPOINT,
        })

fallback_candidates = pd.DataFrame(candidate_rows)

if fallback_candidates.empty:
    raise ValueError("No fallback candidates generated.")

fallback_candidates = (
    fallback_candidates
    .drop_duplicates(["missing_leg_id", "candidate_contract_request_id"])
    .sort_values(["leg_label", "trade_date", "expiration_date", "target_strike", "candidate_rank"])
    .reset_index(drop=True)
)

unique_candidate_contract_plan = (
    fallback_candidates[
        [
            "candidate_contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "candidate_strike",
            "endpoint",
        ]
    ]
    .drop_duplicates()
    .rename(columns={
        "candidate_contract_request_id": "contract_request_id",
        "candidate_strike": "strike_listed",
    })
    .sort_values(["trade_date", "expiration_date", "strike_listed"])
    .reset_index(drop=True)
)

atomic_write_parquet(fallback_candidates, FALLBACK_CANDIDATES_PATH)

fallback_candidates_csv = AUDIT_DIR / f"02F1_fallback_candidates_long5_cal_{timestamp}.csv"
fallback_candidates.to_csv(fallback_candidates_csv, index=False)

print("=" * 100)
print("Generated fallback candidates")
print("=" * 100)
print(f"Missing legs:                   {missing_legs['missing_leg_id'].nunique():,}")
print(f"Fallback candidate rows:        {len(fallback_candidates):,}")
print(f"Unique candidate contracts:     {len(unique_candidate_contract_plan):,}")
print()

candidate_summary = (
    fallback_candidates
    .groupby("leg_label")
    .agg(
        missing_legs=("missing_leg_id", "nunique"),
        candidate_rows=("candidate_contract_request_id", "size"),
        unique_candidate_contracts=("candidate_contract_request_id", "nunique"),
    )
    .reset_index()
)

display(candidate_summary)
print()

# --------------------------------------------------------------------------------------------------
# Load existing fallback candidate cache and determine remaining requests
# --------------------------------------------------------------------------------------------------

if FALLBACK_CANDIDATE_PRICE_CACHE_PATH.exists():
    fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_PRICE_CACHE_PATH).copy()

    if "contract_request_id" in fallback_cache.columns:
        fallback_cache["contract_request_id"] = fallback_cache["contract_request_id"].astype(str)
        fallback_cache["price_found"] = fallback_cache["price_found"].fillna(False).astype(bool)
    else:
        fallback_cache = build_empty_candidate_cache()
else:
    fallback_cache = build_empty_candidate_cache()

# Combine original selected-contract cache and fallback candidate cache for candidate lookup.
# Original cache already contains many nearby contracts; do not re-query those.
original_cache_for_lookup = original_cache.copy()
fallback_cache_for_lookup = fallback_cache.copy()

combined_cache = pd.concat(
    [
        original_cache_for_lookup,
        fallback_cache_for_lookup,
    ],
    ignore_index=True,
    sort=False,
)

if not combined_cache.empty:
    combined_cache["contract_request_id"] = combined_cache["contract_request_id"].astype(str)
    combined_cache["price_found"] = combined_cache["price_found"].fillna(False).astype(bool)

    if "pulled_at" in combined_cache.columns:
        combined_cache["pulled_at"] = combined_cache["pulled_at"].fillna("").astype(str)
        combined_cache = (
            combined_cache
            .sort_values(["contract_request_id", "pulled_at"])
            .drop_duplicates("contract_request_id", keep="last")
            .reset_index(drop=True)
        )
    else:
        combined_cache = (
            combined_cache
            .drop_duplicates("contract_request_id", keep="last")
            .reset_index(drop=True)
        )

completed_ids = get_completed_ids(combined_cache)

remaining_full = unique_candidate_contract_plan.loc[
    ~unique_candidate_contract_plan["contract_request_id"].astype(str).isin(completed_ids)
].copy()

remaining = remaining_full.copy()

if MAX_CANDIDATE_REQUESTS_THIS_RUN is not None:
    remaining = remaining.head(int(MAX_CANDIDATE_REQUESTS_THIS_RUN)).copy()

print("=" * 100)
print("Fallback price cache status")
print("=" * 100)
print(f"Original cache rows used for lookup:        {len(original_cache):,}")
print(f"Existing fallback cache rows:               {len(fallback_cache):,}")
print(f"Combined cache rows:                        {len(combined_cache):,}")
print(f"Unique candidate contracts:                 {len(unique_candidate_contract_plan):,}")
print(f"Already cached candidate contracts:         {len(completed_ids.intersection(set(unique_candidate_contract_plan['contract_request_id'].astype(str)))):,}")
print(f"Remaining candidate contracts full:         {len(remaining_full):,}")
print(f"Remaining candidate contracts this run:     {len(remaining):,}")
print()

if not fallback_cache.empty:
    existing_fallback_status = (
        fallback_cache
        .groupby(["price_found", "price_status"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["price_found", "price_status"])
    )
    print("Existing fallback cache status:")
    display(existing_fallback_status)
    print()

# --------------------------------------------------------------------------------------------------
# Price remaining fallback candidates
# --------------------------------------------------------------------------------------------------

run_rows = []
new_rows = []
start_time = time.time()

try:
    if remaining.empty:
        print("No remaining fallback candidate contracts to price.")
    else:
        for i, (_, row) in enumerate(remaining.iterrows(), start=1):
            result = price_contract(row)
            new_rows.append(result)
            run_rows.append(result)

            if i % PROGRESS_EVERY == 0 or i == 1:
                elapsed = time.time() - start_time
                priced_count = sum(1 for r in run_rows if r.get("price_found") is True)
                miss_count = i - priced_count
                rate = i / elapsed if elapsed > 0 else np.nan
                remaining_count = len(remaining) - i
                eta_minutes = remaining_count / rate / 60 if rate and rate > 0 else np.nan

                print(
                    f"[{i:>5,}/{len(remaining):,}] "
                    f"priced={priced_count:,} "
                    f"missing_or_failed={miss_count:,} "
                    f"last_status={result['price_status']} "
                    f"last_http={result['status_code']} "
                    f"rate={rate:.2f}/sec "
                    f"ETA={eta_minutes:.1f} min"
                )

            if i % CHECKPOINT_EVERY == 0:
                new_df = pd.DataFrame(new_rows)

                if fallback_cache.empty:
                    fallback_cache = new_df.copy()
                else:
                    fallback_cache = pd.concat([fallback_cache, new_df], ignore_index=True)

                run_audit_df = pd.DataFrame(run_rows)
                fallback_cache = save_candidate_cache(fallback_cache, run_audit_df=run_audit_df)
                new_rows = []

                checkpoint_summary = (
                    fallback_cache
                    .groupby(["price_found", "price_status"], dropna=False)
                    .size()
                    .reset_index(name="count")
                    .sort_values(["price_found", "price_status"])
                )

                print()
                print(f"Checkpoint saved after {i:,} candidate attempts this run.")
                display(checkpoint_summary)
                print()

            if REQUEST_SLEEP_SECONDS and REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

except KeyboardInterrupt:
    print()
    print("KeyboardInterrupt received. Saving fallback checkpoint before stopping...")

except Exception as e:
    print()
    print("Unexpected exception. Saving fallback checkpoint before raising...")
    print(repr(e))
    print(traceback.format_exc())
    raise

finally:
    if new_rows:
        new_df = pd.DataFrame(new_rows)

        if fallback_cache.empty:
            fallback_cache = new_df.copy()
        else:
            fallback_cache = pd.concat([fallback_cache, new_df], ignore_index=True)

    run_audit_df = pd.DataFrame(run_rows)
    fallback_cache = save_candidate_cache(fallback_cache, run_audit_df=run_audit_df)

elapsed = time.time() - start_time

# --------------------------------------------------------------------------------------------------
# Rebuild combined cache after pricing
# --------------------------------------------------------------------------------------------------

fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_PRICE_CACHE_PATH).copy()
fallback_cache["contract_request_id"] = fallback_cache["contract_request_id"].astype(str)
fallback_cache["price_found"] = fallback_cache["price_found"].fillna(False).astype(bool)

combined_cache = pd.concat(
    [
        original_cache,
        fallback_cache,
    ],
    ignore_index=True,
    sort=False,
)

combined_cache["contract_request_id"] = combined_cache["contract_request_id"].astype(str)
combined_cache["price_found"] = combined_cache["price_found"].fillna(False).astype(bool)

if "pulled_at" in combined_cache.columns:
    combined_cache["pulled_at"] = combined_cache["pulled_at"].fillna("").astype(str)
    combined_cache = (
        combined_cache
        .sort_values(["contract_request_id", "pulled_at"])
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )
else:
    combined_cache = (
        combined_cache
        .drop_duplicates("contract_request_id", keep="last")
        .reset_index(drop=True)
    )

# --------------------------------------------------------------------------------------------------
# Attach prices to fallback candidates and choose fallback per missing leg
# --------------------------------------------------------------------------------------------------

lookup_cols = [
    "contract_request_id",
    "price_found",
    "price_status",
    "status_code",
    "bid",
    "ask",
    "mid",
    "bid_size",
    "ask_size",
    "underlying_price",
    "implied_vol",
    "delta",
    "theta",
    "vega",
    "gamma",
    "rho",
    "raw_text_preview",
]

lookup_cols = [c for c in lookup_cols if c in combined_cache.columns]

candidates_priced = fallback_candidates.merge(
    combined_cache[lookup_cols],
    left_on="candidate_contract_request_id",
    right_on="contract_request_id",
    how="left",
    validate="many_to_one",
)

candidates_priced["price_found"] = candidates_priced["price_found"].fillna(False).astype(bool)
candidates_priced["price_status"] = candidates_priced["price_status"].fillna("not_queried_or_not_cached")

fallback_map_rows = []

for missing_leg_id, g in candidates_priced.groupby("missing_leg_id", dropna=False):
    fallback_map_rows.append(choose_fallback_for_group(g))

fallback_map = pd.DataFrame(fallback_map_rows)

fallback_map = (
    fallback_map
    .sort_values(["leg_label", "trade_date", "expiration_date", "target_strike"])
    .reset_index(drop=True)
)

atomic_write_parquet(fallback_map, FALLBACK_MAP_PATH)

fallback_map_csv = AUDIT_DIR / f"02F1_fallback_map_long5_cal_{timestamp}.csv"
fallback_map.to_csv(fallback_map_csv, index=False)

candidates_priced_path = (
    PROCESSED_DIR / "call_sleeve_corsi_call_missing_contract_fallback_candidates_priced_long5_cal_v1.parquet"
)
atomic_write_parquet(candidates_priced, candidates_priced_path)

candidates_priced_csv = AUDIT_DIR / f"02F1_fallback_candidates_priced_long5_cal_{timestamp}.csv"
candidates_priced.to_csv(candidates_priced_csv, index=False)

# --------------------------------------------------------------------------------------------------
# Compute hypothetical spread outcomes after fallback
# --------------------------------------------------------------------------------------------------

short_fallback = (
    fallback_map.loc[fallback_map["leg_label"].eq("short_call_1sd")]
    [
        [
            "selected_trade_id",
            "fallback_found",
            "fallback_choice_type",
            "fallback_contract_request_id",
            "fallback_strike",
            "fallback_bid",
            "fallback_ask",
            "fallback_mid",
            "fallback_strike_slippage",
            "fallback_abs_strike_slippage",
        ]
    ]
    .rename(columns={
        "fallback_found": "short_fallback_found",
        "fallback_choice_type": "short_fallback_choice_type",
        "fallback_contract_request_id": "short_fallback_contract_request_id",
        "fallback_strike": "short_fallback_strike",
        "fallback_bid": "short_fallback_bid",
        "fallback_ask": "short_fallback_ask",
        "fallback_mid": "short_fallback_mid",
        "fallback_strike_slippage": "short_fallback_strike_slippage",
        "fallback_abs_strike_slippage": "short_fallback_abs_strike_slippage",
    })
    .drop_duplicates("selected_trade_id", keep="last")
)

long_fallback = (
    fallback_map.loc[fallback_map["leg_label"].eq("long_call_3sd")]
    [
        [
            "selected_trade_id",
            "fallback_found",
            "fallback_choice_type",
            "fallback_contract_request_id",
            "fallback_strike",
            "fallback_bid",
            "fallback_ask",
            "fallback_mid",
            "fallback_strike_slippage",
            "fallback_abs_strike_slippage",
        ]
    ]
    .rename(columns={
        "fallback_found": "long_fallback_found",
        "fallback_choice_type": "long_fallback_choice_type",
        "fallback_contract_request_id": "long_fallback_contract_request_id",
        "fallback_strike": "long_fallback_strike",
        "fallback_bid": "long_fallback_bid",
        "fallback_ask": "long_fallback_ask",
        "fallback_mid": "long_fallback_mid",
        "fallback_strike_slippage": "long_fallback_strike_slippage",
        "fallback_abs_strike_slippage": "long_fallback_abs_strike_slippage",
    })
    .drop_duplicates("selected_trade_id", keep="last")
)

spreads_after = spreads.merge(
    short_fallback,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spreads_after = spreads_after.merge(
    long_fallback,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spreads_after["short_effective_price_found"] = (
    (spreads_after["short_price_found"] == True)
    | (spreads_after["short_fallback_found"] == True)
)

spreads_after["long_effective_price_found"] = (
    (spreads_after["long_price_found"] == True)
    | (spreads_after["long_fallback_found"] == True)
)

spreads_after["spread_complete_after_fallback"] = (
    spreads_after["short_effective_price_found"]
    & spreads_after["long_effective_price_found"]
)

spreads_after["short_effective_strike"] = np.where(
    spreads_after["short_price_found"] == True,
    spreads_after["short_strike"],
    spreads_after["short_fallback_strike"],
)

spreads_after["long_effective_strike"] = np.where(
    spreads_after["long_price_found"] == True,
    spreads_after["long_strike"],
    spreads_after["long_fallback_strike"],
)

spreads_after["short_effective_bid"] = np.where(
    spreads_after["short_price_found"] == True,
    spreads_after["short_bid"],
    spreads_after["short_fallback_bid"],
)

spreads_after["short_effective_ask"] = np.where(
    spreads_after["short_price_found"] == True,
    spreads_after["short_ask"],
    spreads_after["short_fallback_ask"],
)

spreads_after["short_effective_mid"] = np.where(
    spreads_after["short_price_found"] == True,
    spreads_after["short_mid"],
    spreads_after["short_fallback_mid"],
)

spreads_after["long_effective_bid"] = np.where(
    spreads_after["long_price_found"] == True,
    spreads_after["long_bid"],
    spreads_after["long_fallback_bid"],
)

spreads_after["long_effective_ask"] = np.where(
    spreads_after["long_price_found"] == True,
    spreads_after["long_ask"],
    spreads_after["long_fallback_ask"],
)

spreads_after["long_effective_mid"] = np.where(
    spreads_after["long_price_found"] == True,
    spreads_after["long_mid"],
    spreads_after["long_fallback_mid"],
)

spreads_after["effective_width_after_fallback"] = (
    spreads_after["long_effective_strike"] - spreads_after["short_effective_strike"]
)

spreads_after["effective_strike_structure_valid"] = (
    spreads_after["short_effective_strike"].notna()
    & spreads_after["long_effective_strike"].notna()
    & (spreads_after["effective_width_after_fallback"] > 0)
)

spreads_after["entry_credit_conservative_after_fallback"] = np.where(
    spreads_after["spread_complete_after_fallback"] & spreads_after["effective_strike_structure_valid"],
    spreads_after["short_effective_bid"] - spreads_after["long_effective_ask"],
    np.nan,
)

spreads_after["credit_pct_width_after_fallback"] = safe_divide(
    spreads_after["entry_credit_conservative_after_fallback"],
    spreads_after["effective_width_after_fallback"],
)

spreads_after["max_loss_conservative_after_fallback"] = np.where(
    spreads_after["spread_complete_after_fallback"] & spreads_after["effective_strike_structure_valid"],
    spreads_after["effective_width_after_fallback"] - spreads_after["entry_credit_conservative_after_fallback"],
    np.nan,
)

spreads_after["short_terminal_intrinsic_after_fallback"] = np.maximum(
    spreads_after["expiry_spy_close"] - spreads_after["short_effective_strike"],
    0.0,
)

spreads_after["long_terminal_intrinsic_after_fallback"] = np.maximum(
    spreads_after["expiry_spy_close"] - spreads_after["long_effective_strike"],
    0.0,
)

spreads_after["terminal_spread_intrinsic_after_fallback"] = (
    spreads_after["short_terminal_intrinsic_after_fallback"]
    - spreads_after["long_terminal_intrinsic_after_fallback"]
)

spreads_after["expiry_pnl_conservative_after_fallback"] = np.where(
    spreads_after["spread_complete_after_fallback"] & spreads_after["effective_strike_structure_valid"],
    spreads_after["entry_credit_conservative_after_fallback"]
    - spreads_after["terminal_spread_intrinsic_after_fallback"],
    np.nan,
)

spreads_after["return_on_max_loss_conservative_after_fallback"] = safe_divide(
    spreads_after["expiry_pnl_conservative_after_fallback"],
    spreads_after["max_loss_conservative_after_fallback"],
)

spreads_after["trade_win_conservative_after_fallback"] = np.where(
    spreads_after["spread_complete_after_fallback"],
    spreads_after["expiry_pnl_conservative_after_fallback"] > 0,
    np.nan,
)

spreads_after["newly_recovered_by_fallback"] = (
    (spreads_after["spread_complete"] != True)
    & (spreads_after["spread_complete_after_fallback"] == True)
)

spreads_after["still_incomplete_after_fallback"] = (
    spreads_after["spread_complete_after_fallback"] != True
)

atomic_write_parquet(spreads_after, SPREADS_AFTER_FALLBACK_PATH)

spreads_after_csv = AUDIT_DIR / f"02F1_spreads_after_fallback_long5_cal_{timestamp}.csv"
spreads_after.to_csv(spreads_after_csv, index=False)

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

fallback_leg_summary = (
    fallback_map
    .groupby(["leg_label", "fallback_found", "fallback_choice_type"], dropna=False)
    .agg(
        missing_legs=("missing_leg_id", "size"),
        avg_abs_slippage=("fallback_abs_strike_slippage", "mean"),
        median_abs_slippage=("fallback_abs_strike_slippage", "median"),
        avg_signed_slippage=("fallback_strike_slippage", "mean"),
    )
    .reset_index()
    .sort_values(["leg_label", "fallback_found", "fallback_choice_type"])
)

candidate_price_summary = (
    candidates_priced
    .groupby(["leg_label", "price_found", "price_status"], dropna=False)
    .agg(
        candidate_rows=("candidate_contract_request_id", "size"),
        unique_candidate_contracts=("candidate_contract_request_id", "nunique"),
    )
    .reset_index()
    .sort_values(["leg_label", "price_found", "price_status"])
)

spread_recovery_summary = pd.DataFrame([{
    "unique_spreads_total": len(spreads_after),
    "complete_before_fallback": int(spreads_after["spread_complete"].sum()),
    "complete_after_fallback": int(spreads_after["spread_complete_after_fallback"].sum()),
    "newly_recovered_by_fallback": int(spreads_after["newly_recovered_by_fallback"].sum()),
    "still_incomplete_after_fallback": int(spreads_after["still_incomplete_after_fallback"].sum()),
    "complete_rate_before": spreads_after["spread_complete"].mean(),
    "complete_rate_after": spreads_after["spread_complete_after_fallback"].mean(),
    "avg_entry_credit_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "entry_credit_conservative_after_fallback"
    ].mean(),
    "median_entry_credit_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "entry_credit_conservative_after_fallback"
    ].median(),
    "avg_credit_pct_width_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "credit_pct_width_after_fallback"
    ].mean(),
    "win_rate_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "trade_win_conservative_after_fallback"
    ].mean(),
    "avg_return_on_max_loss_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "return_on_max_loss_conservative_after_fallback"
    ].mean(),
    "median_return_on_max_loss_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "return_on_max_loss_conservative_after_fallback"
    ].median(),
    "worst_return_on_max_loss_after_fallback": spreads_after.loc[
        spreads_after["spread_complete_after_fallback"], "return_on_max_loss_conservative_after_fallback"
    ].min(),
}])

spread_recovery_by_tenor = (
    spreads_after
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        win_rate_after=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss_after=("return_on_max_loss_conservative_after_fallback", "mean"),
    )
    .reset_index()
)

spread_recovery_by_tenor["complete_rate_before"] = safe_divide(
    spread_recovery_by_tenor["complete_before"],
    spread_recovery_by_tenor["unique_spreads"],
)

spread_recovery_by_tenor["complete_rate_after"] = safe_divide(
    spread_recovery_by_tenor["complete_after"],
    spread_recovery_by_tenor["unique_spreads"],
)

spread_recovery_by_year = (
    spreads_after
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        win_rate_after=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss_after=("return_on_max_loss_conservative_after_fallback", "mean"),
    )
    .reset_index()
)

spread_recovery_by_year["complete_rate_before"] = safe_divide(
    spread_recovery_by_year["complete_before"],
    spread_recovery_by_year["unique_spreads"],
)

spread_recovery_by_year["complete_rate_after"] = safe_divide(
    spread_recovery_by_year["complete_after"],
    spread_recovery_by_year["unique_spreads"],
)

newly_recovered_preview = (
    spreads_after.loc[spreads_after["newly_recovered_by_fallback"]]
    [
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "short_effective_strike",
            "short_fallback_choice_type",
            "long_strike",
            "long_effective_strike",
            "long_fallback_choice_type",
            "listed_width",
            "effective_width_after_fallback",
            "entry_credit_conservative_after_fallback",
            "credit_pct_width_after_fallback",
            "expiry_spy_close",
            "terminal_spread_intrinsic_after_fallback",
            "expiry_pnl_conservative_after_fallback",
            "return_on_max_loss_conservative_after_fallback",
        ]
    ]
    .sort_values(["trade_date", "tenor"])
    .head(100)
)

still_missing_preview = (
    spreads_after.loc[spreads_after["still_incomplete_after_fallback"]]
    [
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "long_strike",
            "spread_pricing_status",
            "short_fallback_found",
            "short_fallback_choice_type",
            "long_fallback_found",
            "long_fallback_choice_type",
        ]
    ]
    .sort_values(["trade_date", "tenor"])
    .head(100)
)

# Save summaries
fallback_leg_summary_path = AUDIT_DIR / f"02F1_fallback_leg_summary_long5_cal_{timestamp}.csv"
candidate_price_summary_path = AUDIT_DIR / f"02F1_candidate_price_summary_long5_cal_{timestamp}.csv"
spread_recovery_summary_path = AUDIT_DIR / f"02F1_spread_recovery_summary_long5_cal_{timestamp}.csv"
spread_recovery_by_tenor_path = AUDIT_DIR / f"02F1_spread_recovery_by_tenor_long5_cal_{timestamp}.csv"
spread_recovery_by_year_path = AUDIT_DIR / f"02F1_spread_recovery_by_year_long5_cal_{timestamp}.csv"
newly_recovered_preview_path = AUDIT_DIR / f"02F1_newly_recovered_preview_long5_cal_{timestamp}.csv"
still_missing_preview_path = AUDIT_DIR / f"02F1_still_missing_preview_long5_cal_{timestamp}.csv"

fallback_leg_summary.to_csv(fallback_leg_summary_path, index=False)
candidate_price_summary.to_csv(candidate_price_summary_path, index=False)
spread_recovery_summary.to_csv(spread_recovery_summary_path, index=False)
spread_recovery_by_tenor.to_csv(spread_recovery_by_tenor_path, index=False)
spread_recovery_by_year.to_csv(spread_recovery_by_year_path, index=False)
newly_recovered_preview.to_csv(newly_recovered_preview_path, index=False)
still_missing_preview.to_csv(still_missing_preview_path, index=False)

manifest = {
    "cell": "2F.1",
    "description": "Fallback diagnostic for missing listed SPY call strikes after Cell 2F",
    "created_at": timestamp,
    "inputs": {
        "unique_trades_priced_path": str(UNIQUE_TRADES_PRICED_PATH),
        "original_price_cache_path": str(ORIGINAL_PRICE_CACHE_PATH),
    },
    "outputs": {
        "fallback_candidates_path": str(FALLBACK_CANDIDATES_PATH),
        "fallback_candidate_price_cache_path": str(FALLBACK_CANDIDATE_PRICE_CACHE_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
        "spreads_after_fallback_path": str(SPREADS_AFTER_FALLBACK_PATH),
        "manifest_path": str(MANIFEST_PATH),
    },
    "runtime_seconds": elapsed,
    "candidate_generation": {
        "short_offsets": SHORT_OFFSETS,
        "long_offsets": LONG_OFFSETS,
        "short_preference": "nearest listed strike at or above target and below long strike",
        "long_preference": "nearest listed strike at or below target and above short strike",
    },
    "counts": {
        "missing_legs": int(missing_legs["missing_leg_id"].nunique()),
        "fallback_candidate_rows": int(len(fallback_candidates)),
        "unique_candidate_contracts": int(len(unique_candidate_contract_plan)),
        "fallback_cache_rows": int(len(fallback_cache)),
        "run_candidate_attempts": int(len(run_audit_df)),
        "fallback_legs_recovered": int(fallback_map["fallback_found"].sum()),
        "complete_before_fallback": int(spreads_after["spread_complete"].sum()),
        "complete_after_fallback": int(spreads_after["spread_complete_after_fallback"].sum()),
        "newly_recovered_by_fallback": int(spreads_after["newly_recovered_by_fallback"].sum()),
        "still_incomplete_after_fallback": int(spreads_after["still_incomplete_after_fallback"].sum()),
    },
    "notes": {
        "rsi_status": "Current selected-trade universe is provisional pending RSI repair.",
        "not_done_here": "No final ranking, no permanent strike-convention change, no production promotion.",
    },
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.1 complete — fallback recovery summary")
print("=" * 100)
display(spread_recovery_summary)

print()
print("=" * 100)
print("Fallback leg summary")
print("=" * 100)
display(fallback_leg_summary)

print()
print("=" * 100)
print("Candidate price summary")
print("=" * 100)
display(candidate_price_summary)

print()
print("=" * 100)
print("Spread recovery by tenor")
print("=" * 100)
display(spread_recovery_by_tenor)

print()
print("=" * 100)
print("Spread recovery by year")
print("=" * 100)
display(spread_recovery_by_year)

print()
print("=" * 100)
print("Newly recovered preview")
print("=" * 100)
if newly_recovered_preview.empty:
    print("No newly recovered spreads.")
else:
    display(newly_recovered_preview)

print()
print("=" * 100)
print("Still incomplete preview")
print("=" * 100)
if still_missing_preview.empty:
    print("No still-incomplete spreads after fallback.")
else:
    display(still_missing_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Fallback candidates:          {FALLBACK_CANDIDATES_PATH}")
print(f"Fallback candidate prices:    {FALLBACK_CANDIDATE_PRICE_CACHE_PATH}")
print(f"Fallback map:                 {FALLBACK_MAP_PATH}")
print(f"Spreads after fallback:       {SPREADS_AFTER_FALLBACK_PATH}")
print(f"Manifest:                     {MANIFEST_PATH}")

print()
print("Result: 2F.1 fallback diagnostic complete.")
print("Next unaffected step: decide whether fallback convention is acceptable mechanically, but do not lock final rules until RSI is repaired.")

Cell 2F.1 — Missing contract fallback diagnostic
Priced spread file:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_long5_cal_v1.parquet
Original price cache:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet
Fallback candidates path:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_candidates_long5_cal_v1.parquet
Fallback price cache path:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet
Fallback map path:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_map_long5_cal_v1.parquet
Spreads after fallback path: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call

,leg_label,missing_legs
0,long_call_3sd,118
1,short_call_1sd,495



Generated fallback candidates
Missing legs:                   613
Fallback candidate rows:        6,035
Unique candidate contracts:     5,380



,leg_label,missing_legs,candidate_rows,unique_candidate_contracts
0,long_call_3sd,118,590,563
1,short_call_1sd,495,5445,4817



Fallback price cache status
Original cache rows used for lookup:        3,437
Existing fallback cache rows:               0
Combined cache rows:                        3,437
Unique candidate contracts:                 5,380
Already cached candidate contracts:         702
Remaining candidate contracts full:         4,678
Remaining candidate contracts this run:     4,678



...


Newly recovered preview


,selected_trade_id,trade_date,tenor,expiration_date,short_strike,short_effective_strike,short_fallback_choice_type,long_strike,long_effective_strike,long_fallback_choice_type,listed_width,effective_width_after_fallback,entry_credit_conservative_after_fallback,credit_pct_width_after_fallback,expiry_spy_close,terminal_spread_intrinsic_after_fallback,expiry_pnl_conservative_after_fallback,return_on_max_loss_conservative_after_fallback
2,SPY_CALL_20201231_T33_EXP20210205_S400.0_L450.0,2020-12-31,33,2021-02-05,400.0,400.0,NaN,450.0,445.0,preferred_at_or_below_target,50.0,45.0,0.51,0.011333,387.71,0.00,0.51,0.011463
69,SPY_CALL_20211026_T33_EXP20211203_S479.0_L525.0,2021-10-26,33,2021-12-03,479.0,480.0,preferred_at_or_above_target,525.0,520.0,preferred_at_or_below_target,46.0,40.0,0.27,0.006750,453.42,0.00,0.27,0.006796
79,SPY_CALL_20211102_T21_EXP20211126_S479.0_L515.0,2021-11-02,21,2021-11-26,479.0,479.0,NaN,515.0,510.0,preferred_at_or_below_target,36.0,31.0,0.21,0.006774,458.97,0.00,0.21,0.006820
89,SPY_CALL_20211104_T27_EXP20211203_S487.0_L525.0,2021-11-04,27,2021-12-03,487.0,487.0,NaN,525.0,520.0,preferred_at_or_below_target,38.0,33.0,0.25,0.007576,453.42,0.00,0.25,0.007634
91,SPY_CALL_20211104_T33_EXP20211210_S490.0_L535.0,2021-11-04,33,2021-12-10,490.0,490.0,NaN,535.0,530.0,preferred_at_or_below_target,45.0,40.0,0.30,0.007500,470.74,0.00,0.30,0.007557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,SPY_CALL_20240212_T33_EXP20240322_S523.0_L565.0,2024-02-12,33,2024-03-22,523.0,524.0,preferred_at_or_above_target,565.0,565.0,NaN,42.0,41.0,0.91,0.022195,521.21,0.00,0.91,0.022699
644,SPY_CALL_20240213_T33_EXP20240322_S519.0_L565.0,2024-02-13,33,2024-03-22,519.0,520.0,preferred_at_or_above_target,565.0,565.0,NaN,46.0,45.0,0.68,0.015111,521.21,1.21,-0.53,-0.011958
649,SPY_CALL_20240220_T30_EXP20240322_S519.0_L560.0,2024-02-20,30,2024-03-22,519.0,520.0,preferred_at_or_above_target,560.0,560.0,NaN,41.0,40.0,0.67,0.016750,521.21,1.21,-0.54,-0.013730
655,SPY_CALL_20240221_T30_EXP20240322_S519.0_L560.0,2024-02-21,30,2024-03-22,519.0,520.0,preferred_at_or_above_target,560.0,560.0,NaN,41.0,40.0,0.65,0.016250,521.21,1.21,-0.56,-0.014231



Still incomplete preview


,selected_trade_id,trade_date,tenor,expiration_date,short_strike,long_strike,spread_pricing_status,short_fallback_found,short_fallback_choice_type,long_fallback_found,long_fallback_choice_type
1050,SPY_CALL_20241003_T33_EXP20241108_S605.0_L675.0,2024-10-03,33,2024-11-08,605.0,675.0,missing_long_only,NaN,NaN,False,none
1056,SPY_CALL_20241004_T33_EXP20241108_S607.0_L675.0,2024-10-04,33,2024-11-08,607.0,675.0,missing_both,True,preferred_at_or_above_target,False,none
1058,SPY_CALL_20241008_T30_EXP20241108_S609.0_L680.0,2024-10-08,30,2024-11-08,609.0,680.0,missing_both,True,preferred_at_or_above_target,False,none
1062,SPY_CALL_20241009_T30_EXP20241108_S612.0_L680.0,2024-10-09,30,2024-11-08,612.0,680.0,missing_both,True,preferred_at_or_above_target,False,none
1073,SPY_CALL_20241011_T27_EXP20241108_S612.0_L675.0,2024-10-11,27,2024-11-08,612.0,675.0,missing_both,True,preferred_at_or_above_target,False,none
1078,SPY_CALL_20241014_T24_EXP20241108_S614.0_L675.0,2024-10-14,24,2024-11-08,614.0,675.0,missing_both,True,preferred_at_or_above_target,False,none
1080,SPY_CALL_20241014_T33_EXP20241122_S619.0_L690.0,2024-10-14,33,2024-11-22,619.0,690.0,missing_both,True,preferred_at_or_above_target,False,none
1085,SPY_CALL_20241015_T33_EXP20241122_S616.0_L690.0,2024-10-15,33,2024-11-22,616.0,690.0,missing_both,True,preferred_at_or_above_target,False,none
1091,SPY_CALL_20241016_T33_EXP20241122_S617.0_L685.0,2024-10-16,33,2024-11-22,617.0,685.0,missing_both,True,preferred_at_or_above_target,False,none
1098,SPY_CALL_20241017_T33_EXP20241122_S616.0_L685.0,2024-10-17,33,2024-11-22,616.0,685.0,missing_both,True,preferred_at_or_above_target,False,none



Saved
Fallback candidates:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_candidates_long5_cal_v1.parquet
Fallback candidate prices:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet
Fallback map:                 C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_map_long5_cal_v1.parquet
Spreads after fallback:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet
Manifest:                     C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F1_fallback_diagnostic_manifest_20260710_124551.json

Result: 2F.1 fallback diagnostic complete.
Next unaffected step: decide whether fallback convention is acceptable mechanically, but do not lo

In [28]:
# Cell 2F.2 — Build fallback-adjusted rule outcome tables
# Purpose:
#   Mechanical infrastructure step unaffected by the RSI repair.
#
# This cell:
#   - Loads rule-selected trades
#   - Loads fallback-adjusted unique spread outcomes from Cell 2F.1
#   - Joins fallback-adjusted spread outcomes back to every rule-selected row
#   - Computes rule-level, rule-year, and rule-tenor outcome summaries
#   - Saves reusable priced-with-fallback rule tables
#
# This cell does NOT:
#   - Change RSI
#   - Rebuild the selected universe
#   - Rank/select final parameters
#   - Lock the fallback convention permanently
#   - Promote anything to production

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RULE_SELECTED_TRADES_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet"
)

SPREADS_AFTER_FALLBACK_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F.2 — Build fallback-adjusted rule outcome tables")
print("=" * 100)
print(f"Rule selected trades:       {RULE_SELECTED_TRADES_PATH}")
print(f"Spreads after fallback:     {SPREADS_AFTER_FALLBACK_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def summarize_rule_group(g):
    complete = g.loc[g["spread_complete_after_fallback"] == True].copy()

    out = {
        "selected_rows": len(g),
        "unique_selected_trades": g["selected_trade_id"].nunique(),
        "complete_before_rows": int(g["spread_complete"].fillna(False).sum()),
        "complete_after_rows": int(g["spread_complete_after_fallback"].fillna(False).sum()),
        "newly_recovered_rows": int(g["newly_recovered_by_fallback"].fillna(False).sum()),
        "still_incomplete_rows": int(g["still_incomplete_after_fallback"].fillna(False).sum()),
        "coverage_before": g["spread_complete"].fillna(False).mean(),
        "coverage_after": g["spread_complete_after_fallback"].fillna(False).mean(),
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "tenors_selected": ",".join(map(str, sorted(g["tenor"].dropna().astype(int).unique()))),
        "dominant_tenor": g["tenor"].mode().iloc[0] if not g["tenor"].mode().empty else np.nan,
        "avg_original_width": g["listed_width"].mean(),
        "avg_effective_width": complete["effective_width_after_fallback"].mean() if not complete.empty else np.nan,
        "fallback_short_used_rows": int(
            ((g["short_price_found"] != True) & (g["short_fallback_found"] == True)).sum()
        ),
        "fallback_long_used_rows": int(
            ((g["long_price_found"] != True) & (g["long_fallback_found"] == True)).sum()
        ),
    }

    if complete.empty:
        out.update({
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_max_loss": np.nan,
            "median_max_loss": np.nan,
            "win_rate": np.nan,
            "avg_expiry_pnl": np.nan,
            "median_expiry_pnl": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p01_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
            "avg_terminal_intrinsic": np.nan,
            "worst_terminal_intrinsic": np.nan,
        })
        return pd.Series(out)

    out.update({
        "avg_entry_credit": complete["entry_credit_conservative_after_fallback"].mean(),
        "median_entry_credit": complete["entry_credit_conservative_after_fallback"].median(),
        "avg_credit_pct_width": complete["credit_pct_width_after_fallback"].mean(),
        "median_credit_pct_width": complete["credit_pct_width_after_fallback"].median(),
        "avg_max_loss": complete["max_loss_conservative_after_fallback"].mean(),
        "median_max_loss": complete["max_loss_conservative_after_fallback"].median(),
        "win_rate": complete["trade_win_conservative_after_fallback"].mean(),
        "avg_expiry_pnl": complete["expiry_pnl_conservative_after_fallback"].mean(),
        "median_expiry_pnl": complete["expiry_pnl_conservative_after_fallback"].median(),
        "avg_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].mean(),
        "median_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].median(),
        "worst_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].min(),
        "p01_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].quantile(0.01),
        "p05_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].quantile(0.05),
        "p95_return_on_max_loss": complete["return_on_max_loss_conservative_after_fallback"].quantile(0.95),
        "avg_terminal_intrinsic": complete["terminal_spread_intrinsic_after_fallback"].mean(),
        "worst_terminal_intrinsic": complete["terminal_spread_intrinsic_after_fallback"].max(),
    })

    return pd.Series(out)

def summarize_group_simple(df, group_cols):
    out = (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            selected_rows=("selected_trade_id", "size"),
            unique_selected_trades=("selected_trade_id", "nunique"),
            complete_before_rows=("spread_complete", "sum"),
            complete_after_rows=("spread_complete_after_fallback", "sum"),
            newly_recovered_rows=("newly_recovered_by_fallback", "sum"),
            still_incomplete_rows=("still_incomplete_after_fallback", "sum"),
            avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
            median_entry_credit=("entry_credit_conservative_after_fallback", "median"),
            avg_credit_pct_width=("credit_pct_width_after_fallback", "mean"),
            avg_effective_width=("effective_width_after_fallback", "mean"),
            win_rate=("trade_win_conservative_after_fallback", "mean"),
            avg_expiry_pnl=("expiry_pnl_conservative_after_fallback", "mean"),
            median_expiry_pnl=("expiry_pnl_conservative_after_fallback", "median"),
            avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
            median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
            worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
            p05_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", lambda x: x.quantile(0.05)),
            p95_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", lambda x: x.quantile(0.95)),
        )
        .reset_index()
    )

    out["coverage_before"] = safe_divide(out["complete_before_rows"], out["selected_rows"])
    out["coverage_after"] = safe_divide(out["complete_after_rows"], out["selected_rows"])

    return out

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

require_file(RULE_SELECTED_TRADES_PATH)
require_file(SPREADS_AFTER_FALLBACK_PATH)

rule_trades = pd.read_parquet(RULE_SELECTED_TRADES_PATH).copy()
spreads_after = pd.read_parquet(SPREADS_AFTER_FALLBACK_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Rule selected rows:        {len(rule_trades):,}")
print(f"Unique spread rows:        {len(spreads_after):,}")
print()

# --------------------------------------------------------------------------------------------------
# Validation / cleanup
# --------------------------------------------------------------------------------------------------

required_rule_cols = [
    "selected_trade_id",
    "rule_id",
    "selection_rule",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
    "trade_date",
    "tenor",
]

required_spread_cols = [
    "selected_trade_id",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_price_found",
    "long_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "short_fallback_choice_type",
    "long_fallback_choice_type",
    "short_strike",
    "long_strike",
    "short_effective_strike",
    "long_effective_strike",
    "listed_width",
    "effective_width_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

missing_rule = [c for c in required_rule_cols if c not in rule_trades.columns]
missing_spread = [c for c in required_spread_cols if c not in spreads_after.columns]

if missing_rule:
    raise ValueError(f"Missing required rule columns: {missing_rule}")
if missing_spread:
    raise ValueError(f"Missing required spread-after-fallback columns: {missing_spread}")

rule_trades["selected_trade_id"] = rule_trades["selected_trade_id"].astype(str)
spreads_after["selected_trade_id"] = spreads_after["selected_trade_id"].astype(str)

for df in [rule_trades, spreads_after]:
    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce").dt.normalize()
    if "expiration_date" in df.columns:
        df["expiration_date"] = pd.to_datetime(df["expiration_date"], errors="coerce").dt.normalize()

bool_cols = [
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_price_found",
    "long_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "trade_win_conservative_after_fallback",
]

for c in bool_cols:
    if c in spreads_after.columns:
        spreads_after[c] = spreads_after[c].fillna(False).astype(bool)

if "trade_year" not in rule_trades.columns:
    rule_trades["trade_year"] = pd.to_datetime(rule_trades["trade_date"]).dt.year

if "trade_year" not in spreads_after.columns:
    spreads_after["trade_year"] = pd.to_datetime(spreads_after["trade_date"]).dt.year

# Defensive de-dupe.
dupe_spreads = spreads_after["selected_trade_id"].duplicated().sum()
if dupe_spreads:
    print(f"WARNING: duplicate selected_trade_id rows in spreads_after: {dupe_spreads:,}")
    spreads_after = spreads_after.drop_duplicates("selected_trade_id", keep="last").reset_index(drop=True)

# --------------------------------------------------------------------------------------------------
# Build spread outcome columns and join to rule rows
# --------------------------------------------------------------------------------------------------

spread_outcome_cols = [
    "selected_trade_id",
    "spread_pricing_status",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_price_found",
    "long_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "short_fallback_choice_type",
    "long_fallback_choice_type",
    "short_contract_request_id",
    "long_contract_request_id",
    "short_fallback_contract_request_id",
    "long_fallback_contract_request_id",
    "short_strike",
    "long_strike",
    "short_effective_strike",
    "long_effective_strike",
    "listed_width",
    "effective_width_after_fallback",
    "short_bid",
    "short_ask",
    "short_mid",
    "long_bid",
    "long_ask",
    "long_mid",
    "short_fallback_bid",
    "short_fallback_ask",
    "short_fallback_mid",
    "long_fallback_bid",
    "long_fallback_ask",
    "long_fallback_mid",
    "short_effective_bid",
    "short_effective_ask",
    "short_effective_mid",
    "long_effective_bid",
    "long_effective_ask",
    "long_effective_mid",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative",
    "return_on_width_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
    "expiry_spy_close",
]

spread_outcome_cols = [c for c in spread_outcome_cols if c in spreads_after.columns]

drop_existing = [
    c for c in spread_outcome_cols
    if c in rule_trades.columns and c != "selected_trade_id"
]

rule_trades_with_fallback = (
    rule_trades
    .drop(columns=drop_existing, errors="ignore")
    .merge(
        spreads_after[spread_outcome_cols],
        on="selected_trade_id",
        how="left",
        validate="many_to_one",
    )
)

for c in [
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_price_found",
    "long_price_found",
    "short_fallback_found",
    "long_fallback_found",
]:
    if c in rule_trades_with_fallback.columns:
        rule_trades_with_fallback[c] = rule_trades_with_fallback[c].fillna(False).astype(bool)

rule_trades_with_fallback["trade_year"] = pd.to_datetime(
    rule_trades_with_fallback["trade_date"], errors="coerce"
).dt.year

# --------------------------------------------------------------------------------------------------
# Global audit
# --------------------------------------------------------------------------------------------------

global_summary = pd.DataFrame([{
    "rule_selected_rows": len(rule_trades_with_fallback),
    "unique_selected_trades": rule_trades_with_fallback["selected_trade_id"].nunique(),
    "unique_spreads_total": len(spreads_after),
    "unique_spreads_complete_before": int(spreads_after["spread_complete"].sum()),
    "unique_spreads_complete_after": int(spreads_after["spread_complete_after_fallback"].sum()),
    "unique_spreads_newly_recovered": int(spreads_after["newly_recovered_by_fallback"].sum()),
    "unique_spreads_still_incomplete": int(spreads_after["still_incomplete_after_fallback"].sum()),
    "unique_complete_rate_before": spreads_after["spread_complete"].mean(),
    "unique_complete_rate_after": spreads_after["spread_complete_after_fallback"].mean(),
    "rule_rows_complete_before": int(rule_trades_with_fallback["spread_complete"].sum()),
    "rule_rows_complete_after": int(rule_trades_with_fallback["spread_complete_after_fallback"].sum()),
    "rule_rows_newly_recovered": int(rule_trades_with_fallback["newly_recovered_by_fallback"].sum()),
    "rule_rows_still_incomplete": int(rule_trades_with_fallback["still_incomplete_after_fallback"].sum()),
    "rule_row_coverage_before": rule_trades_with_fallback["spread_complete"].mean(),
    "rule_row_coverage_after": rule_trades_with_fallback["spread_complete_after_fallback"].mean(),
}])

# --------------------------------------------------------------------------------------------------
# Rule summaries
# --------------------------------------------------------------------------------------------------

rule_group_cols = [
    "rule_id",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
    "selection_rule",
]

rule_group_cols = [c for c in rule_group_cols if c in rule_trades_with_fallback.columns]

rule_fallback_summary = (
    rule_trades_with_fallback
    .groupby(rule_group_cols, dropna=False)
    .apply(summarize_rule_group)
    .reset_index()
    .sort_values(["rule_id"])
)

rule_year_fallback_summary = summarize_group_simple(
    rule_trades_with_fallback,
    ["rule_id", "trade_year"],
)

rule_tenor_fallback_summary = summarize_group_simple(
    rule_trades_with_fallback,
    ["rule_id", "tenor"],
)

rule_selection_fallback_summary = summarize_group_simple(
    rule_trades_with_fallback,
    ["selection_rule"],
).sort_values(["selection_rule"])

fallback_usage_by_rule = (
    rule_trades_with_fallback
    .groupby(rule_group_cols, dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        short_fallback_rows=("short_fallback_found", "sum"),
        long_fallback_rows=("long_fallback_found", "sum"),
        newly_recovered_rows=("newly_recovered_by_fallback", "sum"),
        still_incomplete_rows=("still_incomplete_after_fallback", "sum"),
    )
    .reset_index()
)

fallback_usage_by_rule["short_fallback_rate"] = safe_divide(
    fallback_usage_by_rule["short_fallback_rows"],
    fallback_usage_by_rule["selected_rows"],
)

fallback_usage_by_rule["long_fallback_rate"] = safe_divide(
    fallback_usage_by_rule["long_fallback_rows"],
    fallback_usage_by_rule["selected_rows"],
)

fallback_usage_by_rule["newly_recovered_rate"] = safe_divide(
    fallback_usage_by_rule["newly_recovered_rows"],
    fallback_usage_by_rule["selected_rows"],
)

# Preview only, not final ranking.
preview_cols = [
    "rule_id",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor_label",
    "selection_rule",
    "selected_rows",
    "complete_after_rows",
    "coverage_after",
    "newly_recovered_rows",
    "still_incomplete_rows",
    "win_rate",
    "avg_entry_credit",
    "median_entry_credit",
    "avg_credit_pct_width",
    "avg_return_on_max_loss",
    "median_return_on_max_loss",
    "worst_return_on_max_loss",
    "p05_return_on_max_loss",
]

preview_cols = [c for c in preview_cols if c in rule_fallback_summary.columns]

rule_preview_not_final = (
    rule_fallback_summary[preview_cols]
    .sort_values(
        ["coverage_after", "selected_rows", "win_rate", "avg_return_on_max_loss"],
        ascending=[False, False, False, False],
    )
    .head(50)
)

still_incomplete_rule_preview = (
    rule_trades_with_fallback.loc[
        rule_trades_with_fallback["still_incomplete_after_fallback"] == True
    ]
    .groupby(rule_group_cols, dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        still_incomplete_rows=("still_incomplete_after_fallback", "sum"),
    )
    .reset_index()
)

still_incomplete_rule_preview["still_incomplete_rate"] = safe_divide(
    still_incomplete_rule_preview["still_incomplete_rows"],
    still_incomplete_rule_preview["selected_rows"],
)

still_incomplete_rule_preview = still_incomplete_rule_preview.sort_values(
    ["still_incomplete_rows", "still_incomplete_rate"],
    ascending=[False, False],
).head(50)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

RULE_TRADES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_long5_cal_v1.parquet"
)

RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_fallback_summary_long5_cal_v1.parquet"
)

RULE_YEAR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_fallback_summary_long5_cal_v1.parquet"
)

RULE_TENOR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_tenor_fallback_summary_long5_cal_v1.parquet"
)

SELECTION_RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selection_rule_fallback_summary_long5_cal_v1.parquet"
)

FALLBACK_USAGE_BY_RULE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_fallback_usage_by_rule_long5_cal_v1.parquet"
)

rule_trades_with_fallback.to_parquet(RULE_TRADES_WITH_FALLBACK_PATH, index=False)
rule_fallback_summary.to_parquet(RULE_FALLBACK_SUMMARY_PATH, index=False)
rule_year_fallback_summary.to_parquet(RULE_YEAR_FALLBACK_SUMMARY_PATH, index=False)
rule_tenor_fallback_summary.to_parquet(RULE_TENOR_FALLBACK_SUMMARY_PATH, index=False)
rule_selection_fallback_summary.to_parquet(SELECTION_RULE_FALLBACK_SUMMARY_PATH, index=False)
fallback_usage_by_rule.to_parquet(FALLBACK_USAGE_BY_RULE_PATH, index=False)

# Audit CSVs
global_summary_csv = AUDIT_DIR / f"02F2_global_fallback_rule_summary_{timestamp}.csv"
rule_fallback_summary_csv = AUDIT_DIR / f"02F2_rule_fallback_summary_{timestamp}.csv"
rule_year_fallback_summary_csv = AUDIT_DIR / f"02F2_rule_year_fallback_summary_{timestamp}.csv"
rule_tenor_fallback_summary_csv = AUDIT_DIR / f"02F2_rule_tenor_fallback_summary_{timestamp}.csv"
selection_rule_fallback_summary_csv = AUDIT_DIR / f"02F2_selection_rule_fallback_summary_{timestamp}.csv"
fallback_usage_by_rule_csv = AUDIT_DIR / f"02F2_fallback_usage_by_rule_{timestamp}.csv"
rule_preview_not_final_csv = AUDIT_DIR / f"02F2_rule_preview_not_final_{timestamp}.csv"
still_incomplete_rule_preview_csv = AUDIT_DIR / f"02F2_still_incomplete_rule_preview_{timestamp}.csv"

global_summary.to_csv(global_summary_csv, index=False)
rule_fallback_summary.to_csv(rule_fallback_summary_csv, index=False)
rule_year_fallback_summary.to_csv(rule_year_fallback_summary_csv, index=False)
rule_tenor_fallback_summary.to_csv(rule_tenor_fallback_summary_csv, index=False)
rule_selection_fallback_summary.to_csv(selection_rule_fallback_summary_csv, index=False)
fallback_usage_by_rule.to_csv(fallback_usage_by_rule_csv, index=False)
rule_preview_not_final.to_csv(rule_preview_not_final_csv, index=False)
still_incomplete_rule_preview.to_csv(still_incomplete_rule_preview_csv, index=False)

manifest_path = AUDIT_DIR / f"02F2_fallback_adjusted_rule_outcomes_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.2",
    "description": "Join fallback-adjusted spread outcomes back to rule-selected trades and summarize rule outcomes",
    "created_at": timestamp,
    "inputs": {
        "rule_selected_trades_path": str(RULE_SELECTED_TRADES_PATH),
        "spreads_after_fallback_path": str(SPREADS_AFTER_FALLBACK_PATH),
    },
    "outputs": {
        "rule_trades_with_fallback_path": str(RULE_TRADES_WITH_FALLBACK_PATH),
        "rule_fallback_summary_path": str(RULE_FALLBACK_SUMMARY_PATH),
        "rule_year_fallback_summary_path": str(RULE_YEAR_FALLBACK_SUMMARY_PATH),
        "rule_tenor_fallback_summary_path": str(RULE_TENOR_FALLBACK_SUMMARY_PATH),
        "selection_rule_fallback_summary_path": str(SELECTION_RULE_FALLBACK_SUMMARY_PATH),
        "fallback_usage_by_rule_path": str(FALLBACK_USAGE_BY_RULE_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "rule_selected_rows": int(len(rule_trades_with_fallback)),
        "unique_selected_trades": int(rule_trades_with_fallback["selected_trade_id"].nunique()),
        "unique_spreads_total": int(len(spreads_after)),
        "unique_spreads_complete_before": int(spreads_after["spread_complete"].sum()),
        "unique_spreads_complete_after": int(spreads_after["spread_complete_after_fallback"].sum()),
        "rule_rows_complete_before": int(rule_trades_with_fallback["spread_complete"].sum()),
        "rule_rows_complete_after": int(rule_trades_with_fallback["spread_complete_after_fallback"].sum()),
        "rule_count": int(rule_fallback_summary["rule_id"].nunique()) if "rule_id" in rule_fallback_summary.columns else int(len(rule_fallback_summary)),
    },
    "notes": {
        "rsi_status": "Current selected-trade universe is provisional pending RSI repair.",
        "ranking_status": "Rule preview is not final ranking and should not be used to select locked parameters.",
        "fallback_status": "Fallback convention tested mechanically but not permanently locked in production.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.2 complete — global fallback-adjusted rule coverage")
print("=" * 100)
display(global_summary)

print()
print("=" * 100)
print("Selection-rule summary — infrastructure preview only")
print("=" * 100)
display(rule_selection_fallback_summary)

print()
print("=" * 100)
print("Rule preview — NOT FINAL RANKING")
print("=" * 100)
display(rule_preview_not_final)

print()
print("=" * 100)
print("Fallback usage by rule preview")
print("=" * 100)
display(fallback_usage_by_rule.head(30))

print()
print("=" * 100)
print("Still-incomplete rule preview")
print("=" * 100)
if still_incomplete_rule_preview.empty:
    print("No still-incomplete rule rows after fallback.")
else:
    display(still_incomplete_rule_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Rule trades with fallback:       {RULE_TRADES_WITH_FALLBACK_PATH}")
print(f"Rule fallback summary:           {RULE_FALLBACK_SUMMARY_PATH}")
print(f"Rule-year fallback summary:      {RULE_YEAR_FALLBACK_SUMMARY_PATH}")
print(f"Rule-tenor fallback summary:     {RULE_TENOR_FALLBACK_SUMMARY_PATH}")
print(f"Selection-rule fallback summary: {SELECTION_RULE_FALLBACK_SUMMARY_PATH}")
print(f"Fallback usage by rule:          {FALLBACK_USAGE_BY_RULE_PATH}")
print(f"Manifest:                        {manifest_path}")

print()
print("Result: 2F.2 fallback-adjusted rule outcome infrastructure complete.")
print("Next unaffected step: mechanical sanity checks on fallback-adjusted outputs; final rule selection remains frozen until RSI repair.")

Cell 2F.2 — Build fallback-adjusted rule outcome tables
Rule selected trades:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet
Spreads after fallback:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet

Loaded inputs
Rule selected rows:        155,550
Unique spread rows:        1,741



C:\Users\patri\AppData\Local\Temp\ipykernel_6252\3163670296.py:267: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spreads_after[c] = spreads_after[c].fillna(False).astype(bool)
C:\Users\patri\AppData\Local\Temp\ipykernel_6252\3163670296.py:267: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spreads_after[c] = spreads_after[c].fillna(False).astype(bool)
C:\Users\patri\AppData\Local\Temp\ipykernel_6252\3163670296.py:412: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of panda


Cell 2F.2 complete — global fallback-adjusted rule coverage


,rule_selected_rows,unique_selected_trades,unique_spreads_total,unique_spreads_complete_before,unique_spreads_complete_after,unique_spreads_newly_recovered,unique_spreads_still_incomplete,unique_complete_rate_before,unique_complete_rate_after,rule_rows_complete_before,rule_rows_complete_after,rule_rows_newly_recovered,rule_rows_still_incomplete,rule_row_coverage_before,rule_row_coverage_after
0,155550,1741,1741,1193,1703,510,38,0.685238,0.978173,103386,150942,47556,4608,0.664648,0.970376



Selection-rule summary — infrastructure preview only


...


Fallback usage by rule preview


,rule_id,z_threshold_shared,rsi_floor,vrp_log_floor_label,selection_rule,selected_rows,short_fallback_rows,long_fallback_rows,newly_recovered_rows,still_incomplete_rows,short_fallback_rate,long_fallback_rate,newly_recovered_rate
0,Z0p00_RSI55_VRP0p00_SEL_highest_vrp_log,0.0,55,0.00,highest_vrp_log,375,94,11,84,14,0.250667,0.029333,0.224000
1,Z0p00_RSI55_VRP0p00_SEL_highest_z_avg,0.0,55,0.00,highest_z_avg,375,121,19,113,17,0.322667,0.050667,0.301333
2,Z0p00_RSI55_VRP0p00_SEL_highest_z_min,0.0,55,0.00,highest_z_min,375,115,15,107,15,0.306667,0.040000,0.285333
3,Z0p00_RSI55_VRP0p00_SEL_longest_tenor,0.0,55,0.00,longest_tenor,375,160,46,164,27,0.426667,0.122667,0.437333
4,Z0p00_RSI55_VRP0p00_SEL_shortest_tenor,0.0,55,0.00,shortest_tenor,375,46,1,47,0,0.122667,0.002667,0.125333
5,Z0p00_RSI55_VRP0p20_SEL_highest_vrp_log,0.0,55,0.20,highest_vrp_log,375,94,11,84,14,0.250667,0.029333,0.224000
6,Z0p00_RSI55_VRP0p20_SEL_highest_z_avg,0.0,55,0.20,highest_z_avg,375,121,19,113,17,0.322667,0.050667,0.301333
7,Z0p00_RSI55_VRP0p20_SEL_highest_z_min,0.0,55,0.20,highest_z_min,375,115,15,107,15,0.306667,0.040000,0.285333
8,Z0p00_RSI55_VRP0p20_SEL_longest_tenor,0.0,55,0.20,longest_tenor,375,160,46,164,27,0.426667,0.122667,0.437333
9,Z0p00_RSI55_VRP0p20_SEL_shortest_tenor,0.0,55,0.20,shortest_tenor,375,46,1,47,0,0.122667,0.002667,0.125333



Still-incomplete rule preview


,rule_id,z_threshold_shared,rsi_floor,vrp_log_floor_label,selection_rule,selected_rows,still_incomplete_rows,still_incomplete_rate
583,Zm0p25_RSI55_VRP0p00_SEL_longest_tenor,-0.25,55,0.00,longest_tenor,30,30,1.0
587,Zm0p25_RSI55_VRP0p20_SEL_longest_tenor,-0.25,55,0.20,longest_tenor,30,30,1.0
595,Zm0p25_RSI55_VRPNONE_SEL_longest_tenor,-0.25,55,NONE,longest_tenor,30,30,1.0
591,Zm0p25_RSI55_VRP0p40_SEL_longest_tenor,-0.25,55,0.40,longest_tenor,29,29,1.0
3,Z0p00_RSI55_VRP0p00_SEL_longest_tenor,0.00,55,0.00,longest_tenor,27,27,1.0
7,Z0p00_RSI55_VRP0p20_SEL_longest_tenor,0.00,55,0.20,longest_tenor,27,27,1.0
11,Z0p00_RSI55_VRP0p40_SEL_longest_tenor,0.00,55,0.40,longest_tenor,27,27,1.0
15,Z0p00_RSI55_VRPNONE_SEL_longest_tenor,0.00,55,NONE,longest_tenor,27,27,1.0
99,Z0p10_RSI55_VRP0p00_SEL_longest_tenor,0.10,55,0.00,longest_tenor,25,25,1.0
103,Z0p10_RSI55_VRP0p20_SEL_longest_tenor,0.10,55,0.20,longest_tenor,25,25,1.0



Saved
Rule trades with fallback:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_long5_cal_v1.parquet
Rule fallback summary:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_fallback_summary_long5_cal_v1.parquet
Rule-year fallback summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_fallback_summary_long5_cal_v1.parquet
Rule-tenor fallback summary:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_tenor_fallback_summary_long5_cal_v1.parquet
Selection-rule fallback summary: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selection_rule_fallback_summary_long5_cal_v1.parquet
Fallback usage by rule:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_usage_by_rule_

In [29]:
# Cell 2F.3 — Mechanical sanity checks on fallback-adjusted pricing/P&L outputs
# Purpose:
#   Mechanical validation step unaffected by RSI repair.
#
# This cell:
#   - Validates fallback-adjusted unique spread outcomes
#   - Validates fallback-adjusted rule-selected outcome table
#   - Checks pricing/P&L arithmetic consistency
#   - Checks strike structure validity
#   - Checks coverage reconciliation
#   - Saves sanity-check reports
#
# This cell does NOT:
#   - Change RSI
#   - Rebuild selected trades
#   - Change strike/fallback logic
#   - Rank final rules
#   - Select parameters
#   - Promote anything to production

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SPREADS_AFTER_FALLBACK_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet"
)

RULE_TRADES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_long5_cal_v1.parquet"
)

RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_fallback_summary_long5_cal_v1.parquet"
)

RULE_YEAR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_year_fallback_summary_long5_cal_v1.parquet"
)

RULE_TENOR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_tenor_fallback_summary_long5_cal_v1.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

TOL = 1e-8

print("=" * 100)
print("Cell 2F.3 — Mechanical sanity checks on fallback-adjusted outputs")
print("=" * 100)
print(f"Spreads after fallback:       {SPREADS_AFTER_FALLBACK_PATH}")
print(f"Rule trades with fallback:    {RULE_TRADES_WITH_FALLBACK_PATH}")
print(f"Rule fallback summary:        {RULE_FALLBACK_SUMMARY_PATH}")
print(f"Rule-year fallback summary:   {RULE_YEAR_FALLBACK_SUMMARY_PATH}")
print(f"Rule-tenor fallback summary:  {RULE_TENOR_FALLBACK_SUMMARY_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def bool_series(s):
    return s.fillna(False).astype(bool)

def check_result(name, passed, severity="hard", details=""):
    return {
        "check_name": name,
        "severity": severity,
        "passed": bool(passed),
        "details": details,
    }

def make_preview(df, mask, cols=None, n=50):
    if cols is None:
        cols = list(df.columns)
    cols = [c for c in cols if c in df.columns]
    return df.loc[mask, cols].head(n).copy()

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    SPREADS_AFTER_FALLBACK_PATH,
    RULE_TRADES_WITH_FALLBACK_PATH,
    RULE_FALLBACK_SUMMARY_PATH,
    RULE_YEAR_FALLBACK_SUMMARY_PATH,
    RULE_TENOR_FALLBACK_SUMMARY_PATH,
]:
    require_file(p)

spreads = pd.read_parquet(SPREADS_AFTER_FALLBACK_PATH).copy()
rule_rows = pd.read_parquet(RULE_TRADES_WITH_FALLBACK_PATH).copy()
rule_summary = pd.read_parquet(RULE_FALLBACK_SUMMARY_PATH).copy()
rule_year_summary = pd.read_parquet(RULE_YEAR_FALLBACK_SUMMARY_PATH).copy()
rule_tenor_summary = pd.read_parquet(RULE_TENOR_FALLBACK_SUMMARY_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Unique spread rows:       {len(spreads):,}")
print(f"Rule-selected rows:       {len(rule_rows):,}")
print(f"Rule summary rows:        {len(rule_summary):,}")
print(f"Rule-year summary rows:   {len(rule_year_summary):,}")
print(f"Rule-tenor summary rows:  {len(rule_tenor_summary):,}")
print()

# --------------------------------------------------------------------------------------------------
# Required columns
# --------------------------------------------------------------------------------------------------

required_spread_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "short_effective_strike",
    "long_effective_strike",
    "listed_width",
    "effective_width_after_fallback",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_spy_close",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

required_rule_cols = [
    "selected_trade_id",
    "rule_id",
    "trade_date",
    "tenor",
    "selection_rule",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "entry_credit_conservative_after_fallback",
    "max_loss_conservative_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

missing_spread_cols = [c for c in required_spread_cols if c not in spreads.columns]
missing_rule_cols = [c for c in required_rule_cols if c not in rule_rows.columns]

if missing_spread_cols:
    raise ValueError(f"Missing required spread columns: {missing_spread_cols}")
if missing_rule_cols:
    raise ValueError(f"Missing required rule-row columns: {missing_rule_cols}")

# --------------------------------------------------------------------------------------------------
# Normalize types
# --------------------------------------------------------------------------------------------------

for df in [spreads, rule_rows]:
    df["selected_trade_id"] = df["selected_trade_id"].astype(str)

    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "trade_win_conservative_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c])
    if c in rule_rows.columns:
        rule_rows[c] = bool_series(rule_rows[c])

for c in [
    "short_fallback_found",
    "long_fallback_found",
    "short_price_found",
    "long_price_found",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c])
    if c in rule_rows.columns:
        rule_rows[c] = bool_series(rule_rows[c])

num_cols = [
    "short_strike",
    "long_strike",
    "short_effective_strike",
    "long_effective_strike",
    "listed_width",
    "effective_width_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_spy_close",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
]

for df in [spreads, rule_rows]:
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

# --------------------------------------------------------------------------------------------------
# Derived fields for checks
# --------------------------------------------------------------------------------------------------

spreads["trade_year"] = pd.to_datetime(spreads["trade_date"]).dt.year
rule_rows["trade_year"] = pd.to_datetime(rule_rows["trade_date"]).dt.year

complete = spreads["spread_complete_after_fallback"]

spreads["calc_effective_width"] = (
    spreads["long_effective_strike"] - spreads["short_effective_strike"]
)

spreads["calc_credit_pct_width"] = safe_divide(
    spreads["entry_credit_conservative_after_fallback"],
    spreads["effective_width_after_fallback"],
)

spreads["calc_max_loss"] = (
    spreads["effective_width_after_fallback"]
    - spreads["entry_credit_conservative_after_fallback"]
)

spreads["calc_short_terminal_intrinsic"] = np.maximum(
    spreads["expiry_spy_close"] - spreads["short_effective_strike"],
    0.0,
)

spreads["calc_long_terminal_intrinsic"] = np.maximum(
    spreads["expiry_spy_close"] - spreads["long_effective_strike"],
    0.0,
)

spreads["calc_terminal_spread_intrinsic"] = (
    spreads["calc_short_terminal_intrinsic"]
    - spreads["calc_long_terminal_intrinsic"]
)

spreads["calc_expiry_pnl"] = (
    spreads["entry_credit_conservative_after_fallback"]
    - spreads["terminal_spread_intrinsic_after_fallback"]
)

spreads["calc_return_on_max_loss"] = safe_divide(
    spreads["expiry_pnl_conservative_after_fallback"],
    spreads["max_loss_conservative_after_fallback"],
)

spreads["calc_win"] = spreads["expiry_pnl_conservative_after_fallback"] > 0

# Only complete rows should have complete pricing fields.
complete_spreads = spreads.loc[complete].copy()
incomplete_spreads = spreads.loc[~complete].copy()

# --------------------------------------------------------------------------------------------------
# Mechanical checks
# --------------------------------------------------------------------------------------------------

checks = []

# Basic shape / uniqueness
checks.append(check_result(
    "unique_spreads_selected_trade_id_unique",
    spreads["selected_trade_id"].is_unique,
    "hard",
    f"duplicate selected_trade_id count = {int(spreads['selected_trade_id'].duplicated().sum())}",
))

checks.append(check_result(
    "rule_rows_have_no_missing_selected_trade_id",
    rule_rows["selected_trade_id"].notna().all(),
    "hard",
    f"missing selected_trade_id rows = {int(rule_rows['selected_trade_id'].isna().sum())}",
))

checks.append(check_result(
    "all_rule_rows_map_to_unique_spreads",
    rule_rows["selected_trade_id"].isin(set(spreads["selected_trade_id"])).all(),
    "hard",
    f"unmapped rule rows = {int((~rule_rows['selected_trade_id'].isin(set(spreads['selected_trade_id']))).sum())}",
))

# Coverage consistency
unique_complete_after = int(spreads["spread_complete_after_fallback"].sum())
rule_complete_after = int(rule_rows["spread_complete_after_fallback"].sum())

checks.append(check_result(
    "unique_spread_complete_after_positive",
    unique_complete_after > 0,
    "hard",
    f"unique complete after fallback = {unique_complete_after:,}",
))

checks.append(check_result(
    "rule_row_complete_after_positive",
    rule_complete_after > 0,
    "hard",
    f"rule complete after fallback = {rule_complete_after:,}",
))

checks.append(check_result(
    "newly_recovered_only_was_incomplete_before",
    (
        spreads.loc[spreads["newly_recovered_by_fallback"], "spread_complete"].eq(False).all()
        and spreads.loc[spreads["newly_recovered_by_fallback"], "spread_complete_after_fallback"].eq(True).all()
    ),
    "hard",
    f"newly recovered count = {int(spreads['newly_recovered_by_fallback'].sum()):,}",
))

checks.append(check_result(
    "still_incomplete_consistent",
    (
        spreads["still_incomplete_after_fallback"].eq(~spreads["spread_complete_after_fallback"]).all()
    ),
    "hard",
    f"still incomplete count = {int(spreads['still_incomplete_after_fallback'].sum()):,}",
))

# Strike structure
strike_valid_mask = (
    complete
    & spreads["short_effective_strike"].notna()
    & spreads["long_effective_strike"].notna()
    & (spreads["long_effective_strike"] > spreads["short_effective_strike"])
)

checks.append(check_result(
    "complete_spreads_have_valid_effective_strike_structure",
    bool(strike_valid_mask.loc[complete].all()) if complete.any() else False,
    "hard",
    f"bad complete strike rows = {int((complete & ~strike_valid_mask).sum())}",
))

width_diff = (
    spreads["effective_width_after_fallback"]
    - spreads["calc_effective_width"]
).abs()

checks.append(check_result(
    "effective_width_matches_effective_strikes",
    bool((width_diff.loc[complete] <= TOL).all()) if complete.any() else False,
    "hard",
    f"max width diff complete = {width_diff.loc[complete].max() if complete.any() else np.nan}",
))

checks.append(check_result(
    "effective_width_positive_for_complete_spreads",
    bool((spreads.loc[complete, "effective_width_after_fallback"] > 0).all()) if complete.any() else False,
    "hard",
    f"nonpositive complete widths = {int((complete & (spreads['effective_width_after_fallback'] <= 0)).sum())}",
))

# Credit / max loss
checks.append(check_result(
    "entry_credit_positive_for_complete_spreads",
    bool((spreads.loc[complete, "entry_credit_conservative_after_fallback"] > 0).all()) if complete.any() else False,
    "hard",
    f"nonpositive complete credits = {int((complete & (spreads['entry_credit_conservative_after_fallback'] <= 0)).sum())}",
))

checks.append(check_result(
    "entry_credit_less_than_width_for_complete_spreads",
    bool((
        spreads.loc[complete, "entry_credit_conservative_after_fallback"]
        < spreads.loc[complete, "effective_width_after_fallback"]
    ).all()) if complete.any() else False,
    "hard",
    f"credit >= width rows = {int((complete & (spreads['entry_credit_conservative_after_fallback'] >= spreads['effective_width_after_fallback'])).sum())}",
))

max_loss_diff = (
    spreads["max_loss_conservative_after_fallback"]
    - spreads["calc_max_loss"]
).abs()

checks.append(check_result(
    "max_loss_matches_width_minus_credit",
    bool((max_loss_diff.loc[complete] <= TOL).all()) if complete.any() else False,
    "hard",
    f"max max_loss diff complete = {max_loss_diff.loc[complete].max() if complete.any() else np.nan}",
))

checks.append(check_result(
    "max_loss_positive_for_complete_spreads",
    bool((spreads.loc[complete, "max_loss_conservative_after_fallback"] > 0).all()) if complete.any() else False,
    "hard",
    f"nonpositive complete max_loss rows = {int((complete & (spreads['max_loss_conservative_after_fallback'] <= 0)).sum())}",
))

# Intrinsic / P&L math
intrinsic = spreads["terminal_spread_intrinsic_after_fallback"]
width = spreads["effective_width_after_fallback"]

checks.append(check_result(
    "terminal_intrinsic_nonnegative_for_complete_spreads",
    bool((intrinsic.loc[complete] >= -TOL).all()) if complete.any() else False,
    "hard",
    f"negative terminal intrinsic rows = {int((complete & (intrinsic < -TOL)).sum())}",
))

checks.append(check_result(
    "terminal_intrinsic_bounded_by_width_for_complete_spreads",
    bool((intrinsic.loc[complete] <= width.loc[complete] + TOL).all()) if complete.any() else False,
    "hard",
    f"intrinsic > width rows = {int((complete & (intrinsic > width + TOL)).sum())}",
))

terminal_intrinsic_diff = (
    spreads["terminal_spread_intrinsic_after_fallback"]
    - spreads["calc_terminal_spread_intrinsic"]
).abs()

checks.append(check_result(
    "terminal_intrinsic_matches_effective_strike_calculation",
    bool((terminal_intrinsic_diff.loc[complete] <= TOL).all()) if complete.any() else False,
    "hard",
    f"max terminal intrinsic diff complete = {terminal_intrinsic_diff.loc[complete].max() if complete.any() else np.nan}",
))

pnl_diff = (
    spreads["expiry_pnl_conservative_after_fallback"]
    - spreads["calc_expiry_pnl"]
).abs()

checks.append(check_result(
    "expiry_pnl_matches_credit_minus_intrinsic",
    bool((pnl_diff.loc[complete] <= TOL).all()) if complete.any() else False,
    "hard",
    f"max pnl diff complete = {pnl_diff.loc[complete].max() if complete.any() else np.nan}",
))

roml_diff = (
    spreads["return_on_max_loss_conservative_after_fallback"]
    - spreads["calc_return_on_max_loss"]
).abs()

checks.append(check_result(
    "return_on_max_loss_matches_pnl_div_max_loss",
    bool((roml_diff.loc[complete] <= TOL).all()) if complete.any() else False,
    "hard",
    f"max return diff complete = {roml_diff.loc[complete].max() if complete.any() else np.nan}",
))

# For a short call spread held to expiry:
# PnL should be no less than -max_loss and no greater than entry credit.
pnl = spreads["expiry_pnl_conservative_after_fallback"]
credit = spreads["entry_credit_conservative_after_fallback"]
max_loss = spreads["max_loss_conservative_after_fallback"]

checks.append(check_result(
    "expiry_pnl_bounded_by_credit_and_max_loss",
    bool((
        (pnl.loc[complete] <= credit.loc[complete] + TOL)
        & (pnl.loc[complete] >= -max_loss.loc[complete] - TOL)
    ).all()) if complete.any() else False,
    "hard",
    (
        f"pnl > credit rows = {int((complete & (pnl > credit + TOL)).sum())}; "
        f"pnl < -max_loss rows = {int((complete & (pnl < -max_loss - TOL)).sum())}"
    ),
))

checks.append(check_result(
    "return_on_max_loss_not_below_minus_one",
    bool((spreads.loc[complete, "return_on_max_loss_conservative_after_fallback"] >= -1 - TOL).all()) if complete.any() else False,
    "hard",
    f"return < -1 rows = {int((complete & (spreads['return_on_max_loss_conservative_after_fallback'] < -1 - TOL)).sum())}",
))

win_mismatch = (
    complete
    & (
        spreads["trade_win_conservative_after_fallback"]
        != spreads["calc_win"]
    )
)

checks.append(check_result(
    "win_flag_matches_positive_pnl",
    int(win_mismatch.sum()) == 0,
    "hard",
    f"win flag mismatch rows = {int(win_mismatch.sum())}",
))

# Missing/incomplete field checks.
complete_required_px_cols = [
    "short_effective_bid",
    "long_effective_ask",
    "entry_credit_conservative_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
]

complete_missing_field_counts = {}

for c in complete_required_px_cols:
    if c in spreads.columns:
        complete_missing_field_counts[c] = int(spreads.loc[complete, c].isna().sum())

checks.append(check_result(
    "complete_spreads_have_no_missing_required_pricing_fields",
    all(v == 0 for v in complete_missing_field_counts.values()),
    "hard",
    json.dumps(complete_missing_field_counts),
))

# Rule summary reconciliation.
summary_complete_sum = None
if "complete_after_rows" in rule_summary.columns:
    summary_complete_sum = int(rule_summary["complete_after_rows"].sum())

    checks.append(check_result(
        "rule_summary_complete_after_reconciles_to_rule_rows",
        summary_complete_sum == rule_complete_after,
        "hard",
        f"rule_summary complete_after sum = {summary_complete_sum:,}; rule rows complete_after = {rule_complete_after:,}",
    ))

if "selected_rows" in rule_summary.columns:
    summary_selected_sum = int(rule_summary["selected_rows"].sum())

    checks.append(check_result(
        "rule_summary_selected_rows_reconciles_to_rule_rows",
        summary_selected_sum == len(rule_rows),
        "hard",
        f"rule_summary selected_rows sum = {summary_selected_sum:,}; rule_rows = {len(rule_rows):,}",
    ))

# Coverage thresholds are sanity warnings, not hard failures.
checks.append(check_result(
    "unique_spread_coverage_after_fallback_above_95pct",
    spreads["spread_complete_after_fallback"].mean() >= 0.95,
    "soft",
    f"unique spread coverage after fallback = {spreads['spread_complete_after_fallback'].mean():.4%}",
))

checks.append(check_result(
    "rule_row_coverage_after_fallback_above_95pct",
    rule_rows["spread_complete_after_fallback"].mean() >= 0.95,
    "soft",
    f"rule-row coverage after fallback = {rule_rows['spread_complete_after_fallback'].mean():.4%}",
))

checks_df = pd.DataFrame(checks)

# --------------------------------------------------------------------------------------------------
# Summary tables
# --------------------------------------------------------------------------------------------------

global_sanity_summary = pd.DataFrame([{
    "unique_spreads_total": len(spreads),
    "unique_spreads_complete_before": int(spreads["spread_complete"].sum()),
    "unique_spreads_complete_after": int(spreads["spread_complete_after_fallback"].sum()),
    "unique_spreads_newly_recovered": int(spreads["newly_recovered_by_fallback"].sum()),
    "unique_spreads_still_incomplete": int(spreads["still_incomplete_after_fallback"].sum()),
    "unique_coverage_before": spreads["spread_complete"].mean(),
    "unique_coverage_after": spreads["spread_complete_after_fallback"].mean(),

    "rule_rows_total": len(rule_rows),
    "rule_rows_complete_before": int(rule_rows["spread_complete"].sum()),
    "rule_rows_complete_after": int(rule_rows["spread_complete_after_fallback"].sum()),
    "rule_rows_newly_recovered": int(rule_rows["newly_recovered_by_fallback"].sum()),
    "rule_rows_still_incomplete": int(rule_rows["still_incomplete_after_fallback"].sum()),
    "rule_row_coverage_before": rule_rows["spread_complete"].mean(),
    "rule_row_coverage_after": rule_rows["spread_complete_after_fallback"].mean(),

    "hard_checks_total": int(checks_df["severity"].eq("hard").sum()),
    "hard_checks_failed": int((checks_df["severity"].eq("hard") & ~checks_df["passed"]).sum()),
    "soft_checks_total": int(checks_df["severity"].eq("soft").sum()),
    "soft_checks_failed": int((checks_df["severity"].eq("soft") & ~checks_df["passed"]).sum()),

    "complete_avg_entry_credit": complete_spreads["entry_credit_conservative_after_fallback"].mean() if not complete_spreads.empty else np.nan,
    "complete_median_entry_credit": complete_spreads["entry_credit_conservative_after_fallback"].median() if not complete_spreads.empty else np.nan,
    "complete_avg_credit_pct_width": complete_spreads["credit_pct_width_after_fallback"].mean() if not complete_spreads.empty else np.nan,
    "complete_win_rate": complete_spreads["trade_win_conservative_after_fallback"].mean() if not complete_spreads.empty else np.nan,
    "complete_avg_return_on_max_loss": complete_spreads["return_on_max_loss_conservative_after_fallback"].mean() if not complete_spreads.empty else np.nan,
    "complete_median_return_on_max_loss": complete_spreads["return_on_max_loss_conservative_after_fallback"].median() if not complete_spreads.empty else np.nan,
    "complete_worst_return_on_max_loss": complete_spreads["return_on_max_loss_conservative_after_fallback"].min() if not complete_spreads.empty else np.nan,
}])

coverage_by_tenor = (
    spreads
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

coverage_by_tenor["coverage_before"] = safe_divide(
    coverage_by_tenor["complete_before"],
    coverage_by_tenor["unique_spreads"],
)

coverage_by_tenor["coverage_after"] = safe_divide(
    coverage_by_tenor["complete_after"],
    coverage_by_tenor["unique_spreads"],
)

coverage_by_year = (
    spreads
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

coverage_by_year["coverage_before"] = safe_divide(
    coverage_by_year["complete_before"],
    coverage_by_year["unique_spreads"],
)

coverage_by_year["coverage_after"] = safe_divide(
    coverage_by_year["complete_after"],
    coverage_by_year["unique_spreads"],
)

fallback_usage_unique = pd.DataFrame([{
    "unique_spreads_total": len(spreads),
    "short_fallback_used": int(((spreads["short_price_found"] != True) & (spreads.get("short_fallback_found", False) == True)).sum()) if "short_fallback_found" in spreads.columns else np.nan,
    "long_fallback_used": int(((spreads["long_price_found"] != True) & (spreads.get("long_fallback_found", False) == True)).sum()) if "long_fallback_found" in spreads.columns else np.nan,
    "newly_recovered": int(spreads["newly_recovered_by_fallback"].sum()),
    "still_incomplete": int(spreads["still_incomplete_after_fallback"].sum()),
}])

# --------------------------------------------------------------------------------------------------
# Failure previews
# --------------------------------------------------------------------------------------------------

preview_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "short_effective_strike",
    "long_effective_strike",
    "listed_width",
    "effective_width_after_fallback",
    "entry_credit_conservative_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_spy_close",
    "terminal_spread_intrinsic_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
]

bad_strike_preview = make_preview(
    spreads,
    complete & ~strike_valid_mask,
    preview_cols,
)

bad_width_preview = make_preview(
    spreads,
    complete & (width_diff > TOL),
    preview_cols + ["calc_effective_width"],
)

bad_credit_preview = make_preview(
    spreads,
    complete & (
        (spreads["entry_credit_conservative_after_fallback"] <= 0)
        | (spreads["entry_credit_conservative_after_fallback"] >= spreads["effective_width_after_fallback"])
    ),
    preview_cols,
)

bad_intrinsic_preview = make_preview(
    spreads,
    complete & (
        (intrinsic < -TOL)
        | (intrinsic > width + TOL)
        | (terminal_intrinsic_diff > TOL)
    ),
    preview_cols + ["calc_terminal_spread_intrinsic"],
)

bad_pnl_preview = make_preview(
    spreads,
    complete & (
        (pnl_diff > TOL)
        | (pnl > credit + TOL)
        | (pnl < -max_loss - TOL)
        | (roml_diff > TOL)
    ),
    preview_cols + ["calc_expiry_pnl", "calc_return_on_max_loss"],
)

still_incomplete_preview = (
    spreads.loc[spreads["still_incomplete_after_fallback"]]
    [
        [
            "selected_trade_id",
            "trade_date",
            "tenor",
            "expiration_date",
            "short_strike",
            "long_strike",
            "short_effective_strike",
            "long_effective_strike",
            "spread_pricing_status",
            "short_fallback_found",
            "short_fallback_choice_type",
            "long_fallback_found",
            "long_fallback_choice_type",
        ]
    ]
    .sort_values(["trade_date", "tenor"])
    .head(100)
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

CHECKS_PATH = AUDIT_DIR / f"02F3_mechanical_sanity_checks_{timestamp}.csv"
GLOBAL_SUMMARY_PATH = AUDIT_DIR / f"02F3_global_sanity_summary_{timestamp}.csv"
COVERAGE_BY_TENOR_PATH = AUDIT_DIR / f"02F3_coverage_by_tenor_{timestamp}.csv"
COVERAGE_BY_YEAR_PATH = AUDIT_DIR / f"02F3_coverage_by_year_{timestamp}.csv"
FALLBACK_USAGE_UNIQUE_PATH = AUDIT_DIR / f"02F3_fallback_usage_unique_{timestamp}.csv"

BAD_STRIKE_PREVIEW_PATH = AUDIT_DIR / f"02F3_bad_strike_preview_{timestamp}.csv"
BAD_WIDTH_PREVIEW_PATH = AUDIT_DIR / f"02F3_bad_width_preview_{timestamp}.csv"
BAD_CREDIT_PREVIEW_PATH = AUDIT_DIR / f"02F3_bad_credit_preview_{timestamp}.csv"
BAD_INTRINSIC_PREVIEW_PATH = AUDIT_DIR / f"02F3_bad_intrinsic_preview_{timestamp}.csv"
BAD_PNL_PREVIEW_PATH = AUDIT_DIR / f"02F3_bad_pnl_preview_{timestamp}.csv"
STILL_INCOMPLETE_PREVIEW_PATH = AUDIT_DIR / f"02F3_still_incomplete_preview_{timestamp}.csv"

checks_df.to_csv(CHECKS_PATH, index=False)
global_sanity_summary.to_csv(GLOBAL_SUMMARY_PATH, index=False)
coverage_by_tenor.to_csv(COVERAGE_BY_TENOR_PATH, index=False)
coverage_by_year.to_csv(COVERAGE_BY_YEAR_PATH, index=False)
fallback_usage_unique.to_csv(FALLBACK_USAGE_UNIQUE_PATH, index=False)

bad_strike_preview.to_csv(BAD_STRIKE_PREVIEW_PATH, index=False)
bad_width_preview.to_csv(BAD_WIDTH_PREVIEW_PATH, index=False)
bad_credit_preview.to_csv(BAD_CREDIT_PREVIEW_PATH, index=False)
bad_intrinsic_preview.to_csv(BAD_INTRINSIC_PREVIEW_PATH, index=False)
bad_pnl_preview.to_csv(BAD_PNL_PREVIEW_PATH, index=False)
still_incomplete_preview.to_csv(STILL_INCOMPLETE_PREVIEW_PATH, index=False)

# Also save a compact latest pointer for quick inspection.
LATEST_CHECKS_PATH = AUDIT_DIR / "02F3_mechanical_sanity_checks_latest.csv"
LATEST_GLOBAL_SUMMARY_PATH = AUDIT_DIR / "02F3_global_sanity_summary_latest.csv"

checks_df.to_csv(LATEST_CHECKS_PATH, index=False)
global_sanity_summary.to_csv(LATEST_GLOBAL_SUMMARY_PATH, index=False)

manifest_path = AUDIT_DIR / f"02F3_mechanical_sanity_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.3",
    "description": "Mechanical sanity checks for fallback-adjusted call sleeve pricing/P&L infrastructure",
    "created_at": timestamp,
    "inputs": {
        "spreads_after_fallback_path": str(SPREADS_AFTER_FALLBACK_PATH),
        "rule_trades_with_fallback_path": str(RULE_TRADES_WITH_FALLBACK_PATH),
        "rule_fallback_summary_path": str(RULE_FALLBACK_SUMMARY_PATH),
        "rule_year_fallback_summary_path": str(RULE_YEAR_FALLBACK_SUMMARY_PATH),
        "rule_tenor_fallback_summary_path": str(RULE_TENOR_FALLBACK_SUMMARY_PATH),
    },
    "outputs": {
        "checks_path": str(CHECKS_PATH),
        "global_summary_path": str(GLOBAL_SUMMARY_PATH),
        "coverage_by_tenor_path": str(COVERAGE_BY_TENOR_PATH),
        "coverage_by_year_path": str(COVERAGE_BY_YEAR_PATH),
        "still_incomplete_preview_path": str(STILL_INCOMPLETE_PREVIEW_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "unique_spreads_total": int(len(spreads)),
        "unique_spreads_complete_after": int(spreads["spread_complete_after_fallback"].sum()),
        "rule_rows_total": int(len(rule_rows)),
        "rule_rows_complete_after": int(rule_rows["spread_complete_after_fallback"].sum()),
        "hard_checks_total": int(checks_df["severity"].eq("hard").sum()),
        "hard_checks_failed": int((checks_df["severity"].eq("hard") & ~checks_df["passed"]).sum()),
        "soft_checks_total": int(checks_df["severity"].eq("soft").sum()),
        "soft_checks_failed": int((checks_df["severity"].eq("soft") & ~checks_df["passed"]).sum()),
    },
    "notes": {
        "rsi_status": "Current selected-trade universe remains provisional pending RSI repair.",
        "ranking_status": "No rule ranking or parameter selection performed.",
        "fallback_status": "Fallback-adjusted infrastructure sanity-checked mechanically only.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

hard_failed = checks_df.loc[
    checks_df["severity"].eq("hard") & ~checks_df["passed"]
].copy()

soft_failed = checks_df.loc[
    checks_df["severity"].eq("soft") & ~checks_df["passed"]
].copy()

print()
print("=" * 100)
print("Cell 2F.3 complete — global sanity summary")
print("=" * 100)
display(global_sanity_summary)

print()
print("=" * 100)
print("Mechanical checks")
print("=" * 100)
display(checks_df)

print()
print("=" * 100)
print("Coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Hard-check failures")
print("=" * 100)
if hard_failed.empty:
    print("No hard-check failures.")
else:
    display(hard_failed)

print()
print("=" * 100)
print("Soft-check failures")
print("=" * 100)
if soft_failed.empty:
    print("No soft-check failures.")
else:
    display(soft_failed)

print()
print("=" * 100)
print("Still incomplete preview")
print("=" * 100)
if still_incomplete_preview.empty:
    print("No still-incomplete spreads after fallback.")
else:
    display(still_incomplete_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Checks:                    {CHECKS_PATH}")
print(f"Global summary:            {GLOBAL_SUMMARY_PATH}")
print(f"Coverage by tenor:         {COVERAGE_BY_TENOR_PATH}")
print(f"Coverage by year:          {COVERAGE_BY_YEAR_PATH}")
print(f"Still incomplete preview:  {STILL_INCOMPLETE_PREVIEW_PATH}")
print(f"Manifest:                  {manifest_path}")

print()
if hard_failed.empty:
    print("Result: 2F.3 mechanical sanity checks PASS.")
    print("Call-sleeve pricing/P&L infrastructure is mechanically usable, but final ranking remains frozen until RSI repair.")
else:
    print("Result: 2F.3 mechanical sanity checks FAIL.")
    print("Fix the hard-check failures before using these outputs for any downstream research.")

Cell 2F.3 — Mechanical sanity checks on fallback-adjusted outputs
Spreads after fallback:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_long5_cal_v1.parquet
Rule trades with fallback:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_long5_cal_v1.parquet
Rule fallback summary:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_fallback_summary_long5_cal_v1.parquet
Rule-year fallback summary:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_fallback_summary_long5_cal_v1.parquet
Rule-tenor fallback summary:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_tenor_fallback_summary_long5_cal_v1.parquet

Loaded inputs
Unique spread rows:       1,741
Rule-selected rows:       155,5

C:\Users\patri\AppData\Local\Temp\ipykernel_6252\2605577730.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return s.fillna(False).astype(bool)
C:\Users\patri\AppData\Local\Temp\ipykernel_6252\2605577730.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return s.fillna(False).astype(bool)


,unique_spreads_total,unique_spreads_complete_before,unique_spreads_complete_after,unique_spreads_newly_recovered,unique_spreads_still_incomplete,unique_coverage_before,unique_coverage_after,rule_rows_total,rule_rows_complete_before,rule_rows_complete_after,...,hard_checks_failed,soft_checks_total,soft_checks_failed,complete_avg_entry_credit,complete_median_entry_credit,complete_avg_credit_pct_width,complete_win_rate,complete_avg_return_on_max_loss,complete_median_return_on_max_loss,complete_worst_return_on_max_loss
0,1741,1193,1703,510,38,0.685238,0.978173,155550,103386,150942,...,0,2,0,0.744134,0.71,0.021718,0.881973,0.003576,0.017812,-0.516209



Mechanical checks


,check_name,severity,passed,details
0,unique_spreads_selected_trade_id_unique,hard,True,duplicate selected_trade_id count = 0
1,rule_rows_have_no_missing_selected_trade_id,hard,True,missing selected_trade_id rows = 0
2,all_rule_rows_map_to_unique_spreads,hard,True,unmapped rule rows = 0
3,unique_spread_complete_after_positive,hard,True,"unique complete after fallback = 1,703"
4,rule_row_complete_after_positive,hard,True,"rule complete after fallback = 150,942"
5,newly_recovered_only_was_incomplete_before,hard,True,newly recovered count = 510
6,still_incomplete_consistent,hard,True,still incomplete count = 38
7,complete_spreads_have_valid_effective_strike_s...,hard,True,bad complete strike rows = 0
8,effective_width_matches_effective_strikes,hard,True,max width diff complete = 0.0
9,effective_width_positive_for_complete_spreads,hard,True,nonpositive complete widths = 0


...


Coverage by year


,trade_year,unique_spreads,complete_before,complete_after,newly_recovered,still_incomplete,avg_effective_width,avg_entry_credit,win_rate,avg_return_on_max_loss,worst_return_on_max_loss,coverage_before,coverage_after
0,2020,3,2,3,1,0,37.333333,0.560000,1.000000,0.017724,0.010338,0.666667,1.000000
1,2021,133,122,133,11,0,30.947368,0.281729,0.984962,0.007823,-0.169440,0.917293,1.000000
2,2022,156,117,156,39,0,40.224359,1.052244,0.923077,0.015445,-0.432467,0.750000,1.000000
3,2023,283,248,283,35,0,27.293286,0.733675,0.795053,-0.007055,-0.516209,0.876325,1.000000
4,2024,617,402,603,201,14,33.983416,0.739453,0.862237,0.005296,-0.396081,0.651540,0.977310
5,2025,401,205,379,174,22,43.300792,0.743245,0.905237,0.010723,-0.388111,0.511222,0.945137
6,2026,148,97,146,49,2,55.472603,0.881849,0.702703,-0.018312,-0.263600,0.655405,0.986486



Hard-check failures
No hard-check failures.

Soft-check failures
No soft-check failures.

Still incomplete preview


,selected_trade_id,trade_date,tenor,expiration_date,short_strike,long_strike,short_effective_strike,long_effective_strike,spread_pricing_status,short_fallback_found,short_fallback_choice_type,long_fallback_found,long_fallback_choice_type
1050,SPY_CALL_20241003_T33_EXP20241108_S605.0_L675.0,2024-10-03,33,2024-11-08,605.0,675.0,605.0,NaN,missing_long_only,False,None,False,none
1056,SPY_CALL_20241004_T33_EXP20241108_S607.0_L675.0,2024-10-04,33,2024-11-08,607.0,675.0,610.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1058,SPY_CALL_20241008_T30_EXP20241108_S609.0_L680.0,2024-10-08,30,2024-11-08,609.0,680.0,610.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1062,SPY_CALL_20241009_T30_EXP20241108_S612.0_L680.0,2024-10-09,30,2024-11-08,612.0,680.0,615.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1073,SPY_CALL_20241011_T27_EXP20241108_S612.0_L675.0,2024-10-11,27,2024-11-08,612.0,675.0,615.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1078,SPY_CALL_20241014_T24_EXP20241108_S614.0_L675.0,2024-10-14,24,2024-11-08,614.0,675.0,615.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1080,SPY_CALL_20241014_T33_EXP20241122_S619.0_L690.0,2024-10-14,33,2024-11-22,619.0,690.0,620.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1085,SPY_CALL_20241015_T33_EXP20241122_S616.0_L690.0,2024-10-15,33,2024-11-22,616.0,690.0,620.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1091,SPY_CALL_20241016_T33_EXP20241122_S617.0_L685.0,2024-10-16,33,2024-11-22,617.0,685.0,620.0,NaN,missing_both,True,preferred_at_or_above_target,False,none
1098,SPY_CALL_20241017_T33_EXP20241122_S616.0_L685.0,2024-10-17,33,2024-11-22,616.0,685.0,620.0,NaN,missing_both,True,preferred_at_or_above_target,False,none



Saved
Checks:                    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_mechanical_sanity_checks_20260710_130301.csv
Global summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_global_sanity_summary_20260710_130301.csv
Coverage by tenor:         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_coverage_by_tenor_20260710_130301.csv
Coverage by year:          C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_coverage_by_year_20260710_130301.csv
Still incomplete preview:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_still_incomplete_preview_20260710_130301.csv
Manifest:                  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F3_mechanical_sanity_manifest_20260710_130301.json

Result: 2F.3 mechanical sanity checks PASS.
Call-sleeve pricing/P&L infrastructure is mechanically usable, but final ranking remains frozen until RSI repair.


In [2]:
# --------------------------------------------------------------------------------------------------
# Column normalization
# --------------------------------------------------------------------------------------------------

trade_date_col = pick_col(signal_raw, ["trade_date", "date"], "trade date")

tenor_col = pick_col(
    signal_raw,
    ["tenor", "dte", "target_tenor", "tenor_days"],
    "tenor",
)

spy_close_col = pick_col(
    signal_raw,
    ["spy_close", "underlying_close", "spy_eod_close", "close", "underlying_price"],
    "SPY close",
)

vrp_log_col = pick_col(
    signal_raw,
    [
        "model_vrp_log_final",
        "corsi_vrp_log",
        "model_vrp_log",
        "vrp_log",
        "model_vrp",
        "source_model_vrp_log",
    ],
    "VRP log",
)

z3m_col = pick_col(
    signal_raw,
    [
        "z_3m_final",
        "corsi_vrp_z_3m",
        "model_vrp_z_3m",
        "model_vrp_zscore_3m",
        "vrp_z_3m",
        "z_3m",
        "z3m",
        "model_vrp_z_63",
        "model_vrp_z_63d",
        "source_model_vrp_z_3m",
    ],
    "3M VRP z-score",
)

z1y_col = pick_col(
    signal_raw,
    [
        "z_1y_final",
        "corsi_vrp_z_1y",
        "model_vrp_z_1y",
        "model_vrp_zscore_1y",
        "vrp_z_1y",
        "z_1y",
        "z1y",
        "model_vrp_z_252",
        "model_vrp_z_252d",
        "source_model_vrp_z_1y",
    ],
    "1Y VRP z-score",
)

rsi_col = pick_col(
    signal_raw,
    [
        "rsi14_final",
        "spy_wilder_rsi14",
        "rsi14",
        "wilder_rsi14",
        "RSI14",
    ],
    "repaired RSI14",
)

rv21d_col = pick_col(
    signal_raw,
    [
        "rv21d_vol_pct_final",
        "rv21d_vol_pct",
        "RV21D",
        "rv21d",
        "realized_vol_21d_pct",
    ],
    "RV21D vol pct",
    required=False,
)

forecast_var_col = pick_col(
    signal_raw,
    [
        "forecast_variance_final",
        "forecast_variance",
        "forecast_variance_candidate",
        "model_forecast_variance",
        "corsi_forecast_variance",
    ],
    "forecast variance",
    required=False,
)

forecast_vol_col = pick_col(
    signal_raw,
    [
        "forecast_vol_final",
        "forecast_vol_pct",
        "candidate_forecast_vol_pct",
        "model_forecast_vol_pct",
        "forecast_volatility_pct",
    ],
    "forecast vol pct",
    required=False,
)

vix_vol_col = pick_col(
    signal_raw,
    [
        "vix_style_vol_final",
        "vix_style_vol_pct",
        "vix_style_vol",
        "vix_style_volatility_pct",
        "implied_vol_pct",
        "implied_volatility_pct",
        "tenor_implied_vol_pct",
    ],
    "VIX-style vol pct",
    required=False,
)

implied_var_col = pick_col(
    signal_raw,
    [
        "implied_variance_final",
        "implied_variance",
        "vix_style_implied_variance",
        "implied_variance_value",
        "tenor_implied_variance",
        "model_implied_variance",
    ],
    "implied variance",
    required=False,
)

if vix_vol_col is None and implied_var_col is None:
    raise ValueError(
        "Could not find either VIX-style vol pct or implied variance. "
        "Need one of those to construct call strikes."
    )

print("=" * 100)
print("Resolved source columns")
print("=" * 100)

resolved_cols = pd.DataFrame(
    [
        ("trade_date", trade_date_col),
        ("tenor", tenor_col),
        ("spy_close", spy_close_col),
        ("vix_style_vol_pct", vix_vol_col),
        ("implied_variance", implied_var_col),
        ("forecast_variance", forecast_var_col),
        ("forecast_vol_pct", forecast_vol_col),
        ("corsi_vrp_log", vrp_log_col),
        ("corsi_vrp_z_3m", z3m_col),
        ("corsi_vrp_z_1y", z1y_col),
        ("rsi14", rsi_col),
        ("rv21d_vol_pct", rv21d_col),
    ],
    columns=["normalized_field", "source_column"],
)

display(resolved_cols)
print()

Resolved source columns


,normalized_field,source_column
0,trade_date,trade_date
1,tenor,tenor
2,spy_close,spy_close
3,vix_style_vol_pct,vix_style_vol_final
4,implied_variance,implied_variance_final
5,forecast_variance,forecast_variance_final
6,forecast_vol_pct,forecast_vol_final
7,corsi_vrp_log,model_vrp_log_final
8,corsi_vrp_z_3m,z_3m_final
9,corsi_vrp_z_1y,z_1y_final


In [3]:
# Cell 2D.3 — Rebuild call-sleeve selected-trade pricing universe from repaired RSI source
# Purpose:
#   Rebuild the short-call Corsi selected-trade universe using the repaired Wilder RSI signal base.
#
# This cell:
#   - Loads the repaired RSI/Corsi signal base
#   - Recreates the call-sleeve parameter-grid selected trades
#   - Uses the same call-sleeve research grid as the provisional run
#   - Uses holiday-aware expiration logic
#   - Uses short call = ceil 1SD call strike to nearest $1
#   - Uses long call = nearest $5 around 3SD target, half-up, forced above short
#   - Builds selected-trade, unique-trade, leg-plan, and unique-contract request-plan outputs
#   - Saves old-vs-repaired universe comparison audits if old provisional outputs exist
#
# This cell does NOT:
#   - Price contracts
#   - Rank final rules
#   - Apply Hybrid v2 put-spread thresholds/sizing/selector
#   - Overwrite old provisional 2D/2F outputs

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REPAIRED_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_repaired_wilder_rsi_signal"
    / "vrp_repaired_wilder_rsi_signal_base_v1.parquet"
)

SPY_EOD_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "market_data"
    / "spy_eod_prices_v1.parquet"
)

OLD_RULE_SELECTED_TRADES_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_rule_selected_trades_unpriced_long5_cal_v1.parquet"
)

OLD_UNIQUE_SELECTED_TRADE_UNIVERSE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_unique_selected_trade_universe_long5_cal_v1.parquet"
)

SUFFIX = "repaired_rsi_long5_cal_v1"

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

Z_THRESHOLDS = [-0.25, 0.00, 0.10, 0.25, 0.50, 0.75, 1.00]
RSI_FLOORS = [55, 58, 60, 62, 65, 68]
VRP_LOG_FLOORS = [None, 0.00, 0.20, 0.40]
SELECTION_RULES = [
    "highest_z_avg",
    "highest_z_min",
    "highest_vrp_log",
    "shortest_tenor",
    "longest_tenor",
]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2D.3 — Rebuild call-sleeve selected-trade universe from repaired RSI source")
print("=" * 100)
print(f"Repaired signal source: {REPAIRED_SIGNAL_PATH}")
print(f"SPY EOD calendar:       {SPY_EOD_PATH}")
print(f"Output suffix:          {SUFFIX}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def pick_col(df, candidates, label, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Could not find required column for {label}. "
            f"Tried: {candidates}\nAvailable columns:\n{list(df.columns)}"
        )
    return None

def normalize_vol_pct(s):
    s = pd.to_numeric(s, errors="coerce")
    med = s.dropna().median()

    # Treat values like 0.18 as decimal vol and values like 18.0 as percent vol.
    if pd.notna(med) and med > 0 and med < 2.0:
        return s * 100.0

    return s

def ceil_to_increment(x, inc=1.0):
    return np.ceil(np.asarray(x, dtype=float) / inc) * inc

def round_half_up_to_increment(x, inc=5.0):
    arr = np.asarray(x, dtype=float)
    return np.floor(arr / inc + 0.5) * inc

def fmt_num_for_id(x):
    if x is None:
        return "NONE"
    sign = "m" if float(x) < 0 else ""
    return sign + f"{abs(float(x)):.2f}".replace(".", "p")

def make_rule_id(z, rsi_floor, vrp_floor, selection_rule):
    vrp_label = "NONE" if vrp_floor is None else fmt_num_for_id(vrp_floor)
    return (
        f"Z{fmt_num_for_id(z)}"
        f"_RSI{int(rsi_floor)}"
        f"_VRP{vrp_label}"
        f"_SEL_{selection_rule}"
    )

def make_trade_id(row):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(row['trade_date']).strftime('%Y%m%d')}"
        f"_T{int(row['tenor'])}"
        f"_EXP{pd.Timestamp(row['expiration_date']).strftime('%Y%m%d')}"
        f"_S{float(row['short_strike']):.1f}"
        f"_L{float(row['long_strike']):.1f}"
    )

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def tenor_bucket(tenor):
    t = int(tenor)
    if t <= 18:
        return "Front"
    if t <= 24:
        return "Middle"
    return "Back"

def first_friday_on_or_after(date_value):
    d = pd.Timestamp(date_value).normalize()
    days_ahead = (4 - d.weekday()) % 7
    return d + pd.Timedelta(days=days_ahead)

def holiday_adjusted_expiration(trade_date, tenor, trading_calendar):
    trade_date = pd.Timestamp(trade_date).normalize()
    target_date = trade_date + pd.Timedelta(days=int(tenor))
    nominal_friday = first_friday_on_or_after(target_date)

    if len(trading_calendar) == 0:
        return pd.NaT

    max_calendar_date = pd.Timestamp(trading_calendar.max()).normalize()

    # Important:
    # If the nominal Friday is beyond available EOD history, do not incorrectly use the last available EOD date.
    # This prevents unexpired future trades from being assigned a false Thursday expiration.
    if nominal_friday > max_calendar_date:
        return pd.NaT

    idx = trading_calendar.searchsorted(nominal_friday, side="right") - 1
    if idx < 0:
        return pd.NaT

    candidate = pd.Timestamp(trading_calendar[idx]).normalize()

    # For normal Friday expiry candidate == nominal Friday.
    # For exchange-holiday Friday expiry, candidate should be the prior trading day.
    if candidate < nominal_friday - pd.Timedelta(days=3):
        return pd.NaT

    if candidate <= trade_date:
        return pd.NaT

    return candidate

def choose_selected_by_rule(qualifying, selection_rule):
    q = qualifying.copy()

    q["z_avg_score"] = (q["corsi_vrp_z_3m"] + q["corsi_vrp_z_1y"]) / 2.0
    q["z_min_score"] = q[["corsi_vrp_z_3m", "corsi_vrp_z_1y"]].min(axis=1)

    if selection_rule == "highest_z_avg":
        sort_cols = [
            "trade_date",
            "z_avg_score",
            "z_min_score",
            "corsi_vrp_log",
            "rsi14",
            "tenor",
        ]
        ascending = [True, False, False, False, False, True]

    elif selection_rule == "highest_z_min":
        sort_cols = [
            "trade_date",
            "z_min_score",
            "z_avg_score",
            "corsi_vrp_log",
            "rsi14",
            "tenor",
        ]
        ascending = [True, False, False, False, False, True]

    elif selection_rule == "highest_vrp_log":
        sort_cols = [
            "trade_date",
            "corsi_vrp_log",
            "z_avg_score",
            "z_min_score",
            "rsi14",
            "tenor",
        ]
        ascending = [True, False, False, False, False, True]

    elif selection_rule == "shortest_tenor":
        sort_cols = [
            "trade_date",
            "tenor",
            "z_avg_score",
            "z_min_score",
            "corsi_vrp_log",
            "rsi14",
        ]
        ascending = [True, True, False, False, False, False]

    elif selection_rule == "longest_tenor":
        sort_cols = [
            "trade_date",
            "tenor",
            "z_avg_score",
            "z_min_score",
            "corsi_vrp_log",
            "rsi14",
        ]
        ascending = [True, False, False, False, False, False]

    else:
        raise ValueError(f"Unknown selection_rule: {selection_rule}")

    selected = (
        q.sort_values(sort_cols, ascending=ascending, kind="mergesort")
        .drop_duplicates("trade_date", keep="first")
        .reset_index(drop=True)
    )

    if selection_rule == "highest_z_avg":
        selected["selection_score"] = selected["z_avg_score"]
    elif selection_rule == "highest_z_min":
        selected["selection_score"] = selected["z_min_score"]
    elif selection_rule == "highest_vrp_log":
        selected["selection_score"] = selected["corsi_vrp_log"]
    elif selection_rule == "shortest_tenor":
        selected["selection_score"] = -selected["tenor"].astype(float)
    elif selection_rule == "longest_tenor":
        selected["selection_score"] = selected["tenor"].astype(float)

    return selected

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

# --------------------------------------------------------------------------------------------------
# Load data
# --------------------------------------------------------------------------------------------------

require_file(REPAIRED_SIGNAL_PATH)
require_file(SPY_EOD_PATH)

signal_raw = pd.read_parquet(REPAIRED_SIGNAL_PATH).copy()
spy_eod = pd.read_parquet(SPY_EOD_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Repaired signal rows: {len(signal_raw):,}")
print(f"SPY EOD rows:         {len(spy_eod):,}")
print()

# --------------------------------------------------------------------------------------------------
# Column normalization
# --------------------------------------------------------------------------------------------------

trade_date_col = pick_col(signal_raw, ["trade_date", "date"], "trade date")

tenor_col = pick_col(
    signal_raw,
    ["tenor", "dte", "target_tenor", "tenor_days"],
    "tenor",
)

spy_close_col = pick_col(
    signal_raw,
    ["spy_close", "underlying_close", "spy_eod_close", "close", "underlying_price"],
    "SPY close",
)

vrp_log_col = pick_col(
    signal_raw,
    [
        "model_vrp_log_final",
        "corsi_vrp_log",
        "model_vrp_log",
        "vrp_log",
        "model_vrp",
        "source_model_vrp_log",
    ],
    "VRP log",
)

z3m_col = pick_col(
    signal_raw,
    [
        "z_3m_final",
        "corsi_vrp_z_3m",
        "model_vrp_z_3m",
        "model_vrp_zscore_3m",
        "vrp_z_3m",
        "z_3m",
        "z3m",
        "model_vrp_z_63",
        "model_vrp_z_63d",
        "source_model_vrp_z_3m",
    ],
    "3M VRP z-score",
)

z1y_col = pick_col(
    signal_raw,
    [
        "z_1y_final",
        "corsi_vrp_z_1y",
        "model_vrp_z_1y",
        "model_vrp_zscore_1y",
        "vrp_z_1y",
        "z_1y",
        "z1y",
        "model_vrp_z_252",
        "model_vrp_z_252d",
        "source_model_vrp_z_1y",
    ],
    "1Y VRP z-score",
)

rsi_col = pick_col(
    signal_raw,
    [
        "rsi14_final",
        "spy_wilder_rsi14",
        "rsi14",
        "wilder_rsi14",
        "RSI14",
    ],
    "repaired RSI14",
)

rv21d_col = pick_col(
    signal_raw,
    [
        "rv21d_vol_pct_final",
        "rv21d_vol_pct",
        "RV21D",
        "rv21d",
        "realized_vol_21d_pct",
    ],
    "RV21D vol pct",
    required=False,
)

forecast_var_col = pick_col(
    signal_raw,
    [
        "forecast_variance_final",
        "forecast_variance",
        "forecast_variance_candidate",
        "model_forecast_variance",
        "corsi_forecast_variance",
    ],
    "forecast variance",
    required=False,
)

forecast_vol_col = pick_col(
    signal_raw,
    [
        "forecast_vol_final",
        "forecast_vol_pct",
        "candidate_forecast_vol_pct",
        "model_forecast_vol_pct",
        "forecast_volatility_pct",
    ],
    "forecast vol pct",
    required=False,
)

vix_vol_col = pick_col(
    signal_raw,
    [
        "vix_style_vol_final",
        "vix_style_vol_pct",
        "vix_style_vol",
        "vix_style_volatility_pct",
        "implied_vol_pct",
        "implied_volatility_pct",
        "tenor_implied_vol_pct",
    ],
    "VIX-style vol pct",
    required=False,
)

implied_var_col = pick_col(
    signal_raw,
    [
        "implied_variance_final",
        "implied_variance",
        "vix_style_implied_variance",
        "implied_variance_value",
        "tenor_implied_variance",
        "model_implied_variance",
    ],
    "implied variance",
    required=False,
)

if vix_vol_col is None and implied_var_col is None:
    raise ValueError(
        "Could not find either VIX-style vol pct or implied variance. "
        "Need one of those to construct call strikes."
    )

print("=" * 100)
print("Resolved source columns")
print("=" * 100)

resolved_cols = pd.DataFrame(
    [
        ("trade_date", trade_date_col),
        ("tenor", tenor_col),
        ("spy_close", spy_close_col),
        ("vix_style_vol_pct", vix_vol_col),
        ("implied_variance", implied_var_col),
        ("forecast_variance", forecast_var_col),
        ("forecast_vol_pct", forecast_vol_col),
        ("corsi_vrp_log", vrp_log_col),
        ("corsi_vrp_z_3m", z3m_col),
        ("corsi_vrp_z_1y", z1y_col),
        ("rsi14", rsi_col),
        ("rv21d_vol_pct", rv21d_col),
    ],
    columns=["normalized_field", "source_column"],
)

display(resolved_cols)
print()

# --------------------------------------------------------------------------------------------------
# Build normalized call candidate base
# --------------------------------------------------------------------------------------------------

base = pd.DataFrame()

base["trade_date"] = pd.to_datetime(signal_raw[trade_date_col], errors="coerce").dt.normalize()
base["tenor"] = pd.to_numeric(signal_raw[tenor_col], errors="coerce").astype("Int64")
base["tenor_bucket"] = base["tenor"].apply(lambda x: tenor_bucket(x) if pd.notna(x) else None)
base["spy_close"] = pd.to_numeric(signal_raw[spy_close_col], errors="coerce")

if vix_vol_col is not None:
    base["vix_style_vol_pct"] = normalize_vol_pct(signal_raw[vix_vol_col])
else:
    base["vix_style_vol_pct"] = 100.0 * np.sqrt(pd.to_numeric(signal_raw[implied_var_col], errors="coerce"))

if implied_var_col is not None:
    base["implied_variance"] = pd.to_numeric(signal_raw[implied_var_col], errors="coerce")
else:
    base["implied_variance"] = (base["vix_style_vol_pct"] / 100.0) ** 2

if forecast_var_col is not None:
    base["forecast_variance"] = pd.to_numeric(signal_raw[forecast_var_col], errors="coerce")
elif forecast_vol_col is not None:
    fv = normalize_vol_pct(signal_raw[forecast_vol_col])
    base["forecast_variance"] = (fv / 100.0) ** 2
else:
    base["forecast_variance"] = np.nan

if forecast_vol_col is not None:
    base["forecast_vol_pct"] = normalize_vol_pct(signal_raw[forecast_vol_col])
else:
    base["forecast_vol_pct"] = 100.0 * np.sqrt(base["forecast_variance"])

base["corsi_vrp_log"] = pd.to_numeric(signal_raw[vrp_log_col], errors="coerce")
base["corsi_vrp_z_3m"] = pd.to_numeric(signal_raw[z3m_col], errors="coerce")
base["corsi_vrp_z_1y"] = pd.to_numeric(signal_raw[z1y_col], errors="coerce")
base["rsi14"] = pd.to_numeric(signal_raw[rsi_col], errors="coerce")

if rv21d_col is not None:
    base["rv21d_vol_pct"] = pd.to_numeric(signal_raw[rv21d_col], errors="coerce")
else:
    base["rv21d_vol_pct"] = np.nan

# Optional RSI integrity audit if both columns are present.
if "rsi14_final" in signal_raw.columns and "spy_wilder_rsi14" in signal_raw.columns:
    rsi_diff = (
        pd.to_numeric(signal_raw["rsi14_final"], errors="coerce")
        - pd.to_numeric(signal_raw["spy_wilder_rsi14"], errors="coerce")
    ).abs()

    print("=" * 100)
    print("RSI source integrity check")
    print("=" * 100)
    print(f"Max abs(rsi14_final - spy_wilder_rsi14): {rsi_diff.max()}")
    print(f"Rows with diff > 1e-10:                  {int((rsi_diff > 1e-10).sum()):,}")
    print()

base = base.loc[base["tenor"].isin(TARGET_TENORS)].copy()

tenor_float = pd.to_numeric(base["tenor"], errors="coerce").astype(float)

base["call_1sd_move_pct"] = (
    base["vix_style_vol_pct"] / 100.0 * np.sqrt(tenor_float / 365.0)
)

base["short_call_1sd_raw"] = base["spy_close"] * (1.0 + base["call_1sd_move_pct"])
base["long_call_3sd_raw"] = base["spy_close"] * (1.0 + 3.0 * base["call_1sd_move_pct"])

base["short_strike"] = ceil_to_increment(base["short_call_1sd_raw"], 1.0)
base["long_strike"] = round_half_up_to_increment(base["long_call_3sd_raw"], 5.0)

base["long_strike"] = np.where(
    base["long_strike"] <= base["short_strike"],
    np.ceil((base["short_strike"] + 1.0) / 5.0) * 5.0,
    base["long_strike"],
)

base["listed_width"] = base["long_strike"] - base["short_strike"]

# --------------------------------------------------------------------------------------------------
# Expiration calendar and expiry close
# --------------------------------------------------------------------------------------------------

spy_date_col = pick_col(spy_eod, ["trade_date", "date"], "SPY EOD date")
spy_close_eod_col = pick_col(spy_eod, ["close", "spy_close", "Close", "underlying_close"], "SPY EOD close")

spy_eod_clean = spy_eod[[spy_date_col, spy_close_eod_col]].copy()
spy_eod_clean = spy_eod_clean.rename(columns={spy_date_col: "trade_date", spy_close_eod_col: "spy_eod_close"})
spy_eod_clean["trade_date"] = pd.to_datetime(spy_eod_clean["trade_date"], errors="coerce").dt.normalize()
spy_eod_clean["spy_eod_close"] = pd.to_numeric(spy_eod_clean["spy_eod_close"], errors="coerce")

spy_eod_clean = (
    spy_eod_clean
    .dropna(subset=["trade_date"])
    .drop_duplicates("trade_date", keep="last")
    .sort_values("trade_date")
    .reset_index(drop=True)
)

trading_calendar = pd.DatetimeIndex(spy_eod_clean["trade_date"].unique()).sort_values()
expiry_close_map = spy_eod_clean.set_index("trade_date")["spy_eod_close"].to_dict()

base["expiration_date"] = [
    holiday_adjusted_expiration(td, ten, trading_calendar)
    for td, ten in zip(base["trade_date"], base["tenor"])
]

base["effective_dte"] = (
    pd.to_datetime(base["expiration_date"]) - pd.to_datetime(base["trade_date"])
).dt.days

base["expiry_spy_close"] = base["expiration_date"].map(expiry_close_map)

base["terminal_short_call_intrinsic"] = np.maximum(
    base["expiry_spy_close"] - base["short_strike"],
    0.0,
)

base["terminal_long_call_intrinsic"] = np.maximum(
    base["expiry_spy_close"] - base["long_strike"],
    0.0,
)

base["terminal_spread_intrinsic_before_premium"] = (
    base["terminal_short_call_intrinsic"] - base["terminal_long_call_intrinsic"]
)

base["candidate_inputs_complete"] = (
    base[
        [
            "trade_date",
            "tenor",
            "spy_close",
            "vix_style_vol_pct",
            "implied_variance",
            "forecast_variance",
            "corsi_vrp_log",
            "corsi_vrp_z_3m",
            "corsi_vrp_z_1y",
            "rsi14",
            "rv21d_vol_pct",
            "short_strike",
            "long_strike",
            "listed_width",
            "expiration_date",
            "effective_dte",
        ]
    ]
    .notna()
    .all(axis=1)
    & (base["spy_close"] > 0)
    & (base["vix_style_vol_pct"] > 0)
    & (base["implied_variance"] > 0)
    & (base["forecast_variance"] > 0)
    & (base["listed_width"] > 0)
    & (base["effective_dte"] > 0)
)

base["outcome_available"] = base["expiry_spy_close"].notna()
base["research_eligible"] = base["candidate_inputs_complete"] & base["outcome_available"]
base["trade_year"] = base["trade_date"].dt.year

print("=" * 100)
print("Normalized repaired call candidate base")
print("=" * 100)
print(f"Base rows after tenor filter:          {len(base):,}")
print(f"Date range:                           {base['trade_date'].min()} to {base['trade_date'].max()}")
print(f"Tenors:                               {sorted(base['tenor'].dropna().astype(int).unique().tolist())}")
print(f"Candidate inputs complete rows:       {int(base['candidate_inputs_complete'].sum()):,}")
print(f"Outcome available rows:               {int(base['outcome_available'].sum()):,}")
print(f"Research eligible rows:               {int(base['research_eligible'].sum()):,}")
print()

eligibility_by_tenor = (
    base
    .groupby("tenor", dropna=False)
    .agg(
        rows=("trade_date", "size"),
        complete_rows=("candidate_inputs_complete", "sum"),
        outcome_available_rows=("outcome_available", "sum"),
        research_eligible_rows=("research_eligible", "sum"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
    )
    .reset_index()
)

display(eligibility_by_tenor)
print()

# --------------------------------------------------------------------------------------------------
# Generate parameter rules
# --------------------------------------------------------------------------------------------------

rule_rows = []

for z in Z_THRESHOLDS:
    for rsi_floor in RSI_FLOORS:
        for vrp_floor in VRP_LOG_FLOORS:
            for selection_rule in SELECTION_RULES:
                rule_rows.append({
                    "rule_id": make_rule_id(z, rsi_floor, vrp_floor, selection_rule),
                    "z_threshold_shared": float(z),
                    "rsi_floor": int(rsi_floor),
                    "vrp_log_floor": np.nan if vrp_floor is None else float(vrp_floor),
                    "vrp_log_floor_label": "NONE" if vrp_floor is None else f"{float(vrp_floor):.2f}",
                    "selection_rule": selection_rule,
                })

rule_meta = pd.DataFrame(rule_rows)

if rule_meta["rule_id"].duplicated().any():
    dupes = rule_meta.loc[rule_meta["rule_id"].duplicated(), "rule_id"].tolist()
    raise ValueError(f"Duplicate rule_id values generated: {dupes[:10]}")

print("=" * 100)
print("Parameter grid")
print("=" * 100)
print(f"Rule count: {len(rule_meta):,}")
display(rule_meta.head(20))
print()

# --------------------------------------------------------------------------------------------------
# Select trades by rule
# --------------------------------------------------------------------------------------------------

eligible = base.loc[base["research_eligible"]].copy()

selected_by_rule = []

for _, rule in rule_meta.iterrows():
    z = float(rule["z_threshold_shared"])
    rsi_floor = int(rule["rsi_floor"])
    vrp_floor_label = rule["vrp_log_floor_label"]
    selection_rule = rule["selection_rule"]

    q = eligible.loc[
        (eligible["corsi_vrp_z_3m"] > z)
        & (eligible["corsi_vrp_z_1y"] > z)
        & (eligible["rsi14"] > rsi_floor)
    ].copy()

    if vrp_floor_label != "NONE":
        q = q.loc[q["corsi_vrp_log"] > float(rule["vrp_log_floor"])].copy()

    if q.empty:
        continue

    selected = choose_selected_by_rule(q, selection_rule)

    selected["rule_id"] = rule["rule_id"]
    selected["z_threshold_shared"] = z
    selected["rsi_floor"] = rsi_floor
    selected["vrp_log_floor"] = rule["vrp_log_floor"]
    selected["vrp_log_floor_label"] = vrp_floor_label
    selected["selection_rule"] = selection_rule

    selected_by_rule.append(selected)

if selected_by_rule:
    rule_selected_trades = pd.concat(selected_by_rule, ignore_index=True)
else:
    rule_selected_trades = pd.DataFrame()

if rule_selected_trades.empty:
    raise ValueError("No selected trades generated from repaired RSI source. Check thresholds/source columns.")

rule_selected_trades["selected_trade_id"] = rule_selected_trades.apply(make_trade_id, axis=1)

rule_selected_trades["expiration_str"] = pd.to_datetime(
    rule_selected_trades["expiration_date"]
).dt.strftime("%Y-%m-%d")

rule_selected_trades["trade_date_str"] = pd.to_datetime(
    rule_selected_trades["trade_date"]
).dt.strftime("%Y-%m-%d")

rule_selected_trades["expiration_yyyymmdd"] = pd.to_datetime(
    rule_selected_trades["expiration_date"]
).dt.strftime("%Y%m%d")

front_cols = [
    "rule_id",
    "z_threshold_shared",
    "rsi_floor",
    "vrp_log_floor",
    "vrp_log_floor_label",
    "selection_rule",
    "selected_trade_id",
    "trade_date",
    "trade_date_str",
    "trade_year",
    "tenor",
    "tenor_bucket",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "effective_dte",
    "spy_close",
    "vix_style_vol_pct",
    "implied_variance",
    "forecast_variance",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "rv21d_vol_pct",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_short_call_intrinsic",
    "terminal_long_call_intrinsic",
    "terminal_spread_intrinsic_before_premium",
    "z_avg_score",
    "z_min_score",
    "selection_score",
]

front_cols = [c for c in front_cols if c in rule_selected_trades.columns]
other_cols = [c for c in rule_selected_trades.columns if c not in front_cols]

rule_selected_trades = rule_selected_trades[front_cols + other_cols].copy()

# --------------------------------------------------------------------------------------------------
# Unique selected-trade universe
# --------------------------------------------------------------------------------------------------

unique_trade_cols = [
    "selected_trade_id",
    "trade_date",
    "trade_date_str",
    "trade_year",
    "tenor",
    "tenor_bucket",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "effective_dte",
    "spy_close",
    "vix_style_vol_pct",
    "implied_variance",
    "forecast_variance",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "rv21d_vol_pct",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_short_call_intrinsic",
    "terminal_long_call_intrinsic",
    "terminal_spread_intrinsic_before_premium",
]

unique_trade_cols = [c for c in unique_trade_cols if c in rule_selected_trades.columns]

unique_selected_trades = (
    rule_selected_trades[unique_trade_cols]
    .drop_duplicates("selected_trade_id")
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .reset_index(drop=True)
)

id_counts = rule_selected_trades.groupby("selected_trade_id").size().rename("rule_use_count").reset_index()
unique_selected_trades = unique_selected_trades.merge(id_counts, on="selected_trade_id", how="left")

# --------------------------------------------------------------------------------------------------
# Build leg request plan
# --------------------------------------------------------------------------------------------------

leg_rows = []

for _, row in unique_selected_trades.iterrows():
    for leg_label, leg_side, strike_col in [
        ("short_call_1sd", "sell", "short_strike"),
        ("long_call_3sd", "buy", "long_strike"),
    ]:
        strike = float(row[strike_col])

        contract_request_id = make_contract_request_id(
            symbol="SPY",
            trade_date=row["trade_date"],
            expiration_date=row["expiration_date"],
            strike=strike,
            right="C",
        )

        leg_rows.append({
            "selected_trade_id": row["selected_trade_id"],
            "leg_label": leg_label,
            "leg_side": leg_side,
            "symbol": "SPY",
            "right": "C",
            "trade_date": row["trade_date"],
            "trade_date_str": row["trade_date_str"],
            "expiration_date": row["expiration_date"],
            "expiration_str": row["expiration_str"],
            "expiration_yyyymmdd": row["expiration_yyyymmdd"],
            "tenor": int(row["tenor"]),
            "tenor_bucket": row["tenor_bucket"],
            "strike_listed": strike,
            "contract_request_id": contract_request_id,
            "underlying_close": row["spy_close"],
            "vix_style_vol_pct": row["vix_style_vol_pct"],
            "short_strike": row["short_strike"],
            "long_strike": row["long_strike"],
            "listed_width": row["listed_width"],
        })

selected_leg_request_plan = pd.DataFrame(leg_rows)

unique_contract_request_plan = (
    selected_leg_request_plan[
        [
            "contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "strike_listed",
        ]
    ]
    .drop_duplicates("contract_request_id")
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

rule_count_summary = (
    rule_selected_trades
    .groupby("rule_id", dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        unique_selected_trades=("selected_trade_id", "nunique"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        avg_tenor=("tenor", "mean"),
        median_tenor=("tenor", "median"),
        avg_vrp_log=("corsi_vrp_log", "mean"),
        avg_z_3m=("corsi_vrp_z_3m", "mean"),
        avg_z_1y=("corsi_vrp_z_1y", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_width=("listed_width", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
)

tenors_by_rule = (
    rule_selected_trades
    .groupby("rule_id", dropna=False)["tenor"]
    .apply(lambda x: ",".join(map(str, sorted(pd.Series(x).dropna().astype(int).unique()))))
    .rename("tenors_selected")
    .reset_index()
)

rule_preprice_summary = (
    rule_meta
    .merge(rule_count_summary, on="rule_id", how="left")
    .merge(tenors_by_rule, on="rule_id", how="left")
)

for c in ["selected_rows", "unique_selected_trades"]:
    if c in rule_preprice_summary.columns:
        rule_preprice_summary[c] = rule_preprice_summary[c].fillna(0).astype(int)

rule_preprice_summary["tenors_selected"] = rule_preprice_summary["tenors_selected"].fillna("")

rule_year_preprice_summary = (
    rule_selected_trades
    .groupby(["rule_id", "trade_year"], dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        unique_selected_trades=("selected_trade_id", "nunique"),
        avg_tenor=("tenor", "mean"),
        avg_vrp_log=("corsi_vrp_log", "mean"),
        avg_z_3m=("corsi_vrp_z_3m", "mean"),
        avg_z_1y=("corsi_vrp_z_1y", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_width=("listed_width", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
    .sort_values(["rule_id", "trade_year"])
)

selection_rule_summary = (
    rule_selected_trades
    .groupby("selection_rule", dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        unique_selected_trades=("selected_trade_id", "nunique"),
        avg_tenor=("tenor", "mean"),
        avg_width=("listed_width", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
    .sort_values("selection_rule")
)

tenor_summary = (
    unique_selected_trades
    .groupby("tenor", dropna=False)
    .agg(
        unique_selected_trades=("selected_trade_id", "size"),
        avg_width=("listed_width", "mean"),
        avg_vix_style_vol_pct=("vix_style_vol_pct", "mean"),
        avg_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "mean"),
        worst_terminal_intrinsic=("terminal_spread_intrinsic_before_premium", "max"),
    )
    .reset_index()
    .sort_values("tenor")
)

# --------------------------------------------------------------------------------------------------
# Old-vs-repaired comparison
# --------------------------------------------------------------------------------------------------

comparison_outputs = {}

if OLD_UNIQUE_SELECTED_TRADE_UNIVERSE_PATH.exists():
    old_unique = pd.read_parquet(OLD_UNIQUE_SELECTED_TRADE_UNIVERSE_PATH).copy()
    old_unique["selected_trade_id"] = old_unique["selected_trade_id"].astype(str)

    new_ids = set(unique_selected_trades["selected_trade_id"].astype(str))
    old_ids = set(old_unique["selected_trade_id"].astype(str))

    unique_comparison_summary = pd.DataFrame([{
        "old_unique_selected_trades": len(old_ids),
        "new_unique_selected_trades": len(new_ids),
        "overlap_unique_selected_trades": len(old_ids & new_ids),
        "added_unique_selected_trades": len(new_ids - old_ids),
        "removed_unique_selected_trades": len(old_ids - new_ids),
        "old_only_pct": len(old_ids - new_ids) / len(old_ids) if old_ids else np.nan,
        "new_only_pct": len(new_ids - old_ids) / len(new_ids) if new_ids else np.nan,
    }])

    added_unique = unique_selected_trades.loc[
        unique_selected_trades["selected_trade_id"].astype(str).isin(new_ids - old_ids)
    ].copy()

    removed_unique = old_unique.loc[
        old_unique["selected_trade_id"].astype(str).isin(old_ids - new_ids)
    ].copy()

    comparison_outputs["unique_comparison_summary"] = unique_comparison_summary
    comparison_outputs["added_unique"] = added_unique
    comparison_outputs["removed_unique"] = removed_unique

else:
    unique_comparison_summary = pd.DataFrame([{
        "old_unique_selected_trades": np.nan,
        "new_unique_selected_trades": unique_selected_trades["selected_trade_id"].nunique(),
        "overlap_unique_selected_trades": np.nan,
        "added_unique_selected_trades": np.nan,
        "removed_unique_selected_trades": np.nan,
        "old_only_pct": np.nan,
        "new_only_pct": np.nan,
    }])
    comparison_outputs["unique_comparison_summary"] = unique_comparison_summary

if OLD_RULE_SELECTED_TRADES_PATH.exists():
    old_rule = pd.read_parquet(OLD_RULE_SELECTED_TRADES_PATH).copy()
    old_rule["selected_trade_id"] = old_rule["selected_trade_id"].astype(str)
    old_rule["trade_date"] = pd.to_datetime(old_rule["trade_date"], errors="coerce").dt.normalize()

    old_rule_comp = old_rule[
        ["rule_id", "trade_date", "selected_trade_id", "tenor"]
    ].rename(columns={
        "selected_trade_id": "old_selected_trade_id",
        "tenor": "old_tenor",
    })

    new_rule_comp = rule_selected_trades[
        ["rule_id", "trade_date", "selected_trade_id", "tenor"]
    ].rename(columns={
        "selected_trade_id": "new_selected_trade_id",
        "tenor": "new_tenor",
    })

    rule_date_comparison = old_rule_comp.merge(
        new_rule_comp,
        on=["rule_id", "trade_date"],
        how="outer",
        indicator=True,
    )

    rule_date_comparison["comparison_status"] = np.select(
        [
            rule_date_comparison["_merge"].eq("left_only"),
            rule_date_comparison["_merge"].eq("right_only"),
            (
                rule_date_comparison["_merge"].eq("both")
                & rule_date_comparison["old_selected_trade_id"].eq(rule_date_comparison["new_selected_trade_id"])
            ),
            (
                rule_date_comparison["_merge"].eq("both")
                & ~rule_date_comparison["old_selected_trade_id"].eq(rule_date_comparison["new_selected_trade_id"])
            ),
        ],
        [
            "old_only_rule_date",
            "new_only_rule_date",
            "same_selected_trade",
            "changed_selected_trade",
        ],
        default="unknown",
    )

    rule_date_comparison_summary = (
        rule_date_comparison
        .groupby("comparison_status", dropna=False)
        .size()
        .reset_index(name="rule_date_rows")
        .sort_values("comparison_status")
    )

    comparison_outputs["rule_date_comparison"] = rule_date_comparison
    comparison_outputs["rule_date_comparison_summary"] = rule_date_comparison_summary

else:
    comparison_outputs["rule_date_comparison_summary"] = pd.DataFrame([{
        "comparison_status": "old_rule_file_missing",
        "rule_date_rows": np.nan,
    }])

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

RULE_SELECTED_TRADES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_unpriced_{SUFFIX}.parquet"
)

UNIQUE_SELECTED_TRADE_UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_{SUFFIX}.parquet"
)

SELECTED_LEG_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_leg_request_plan_{SUFFIX}.parquet"
)

UNIQUE_CONTRACT_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_unique_contract_request_plan_{SUFFIX}.parquet"
)

RULE_PREPRICE_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_preprice_summary_{SUFFIX}.parquet"
)

RULE_YEAR_PREPRICE_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_year_preprice_summary_{SUFFIX}.parquet"
)

BASE_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_{SUFFIX}.parquet"
)

atomic_write_parquet(base, BASE_CANDIDATE_PANEL_PATH)
atomic_write_parquet(rule_selected_trades, RULE_SELECTED_TRADES_PATH)
atomic_write_parquet(unique_selected_trades, UNIQUE_SELECTED_TRADE_UNIVERSE_PATH)
atomic_write_parquet(selected_leg_request_plan, SELECTED_LEG_REQUEST_PLAN_PATH)
atomic_write_parquet(unique_contract_request_plan, UNIQUE_CONTRACT_REQUEST_PLAN_PATH)
atomic_write_parquet(rule_preprice_summary, RULE_PREPRICE_SUMMARY_PATH)
atomic_write_parquet(rule_year_preprice_summary, RULE_YEAR_PREPRICE_SUMMARY_PATH)

# Audit CSVs
base_summary_csv = AUDIT_DIR / f"02D3_base_eligibility_by_tenor_{SUFFIX}_{timestamp}.csv"
rule_preprice_summary_csv = AUDIT_DIR / f"02D3_rule_preprice_summary_{SUFFIX}_{timestamp}.csv"
rule_year_preprice_summary_csv = AUDIT_DIR / f"02D3_rule_year_preprice_summary_{SUFFIX}_{timestamp}.csv"
selection_rule_summary_csv = AUDIT_DIR / f"02D3_selection_rule_summary_{SUFFIX}_{timestamp}.csv"
tenor_summary_csv = AUDIT_DIR / f"02D3_unique_trade_tenor_summary_{SUFFIX}_{timestamp}.csv"
contract_plan_csv = AUDIT_DIR / f"02D3_unique_contract_request_plan_{SUFFIX}_{timestamp}.csv"

atomic_write_csv(eligibility_by_tenor, base_summary_csv)
atomic_write_csv(rule_preprice_summary, rule_preprice_summary_csv)
atomic_write_csv(rule_year_preprice_summary, rule_year_preprice_summary_csv)
atomic_write_csv(selection_rule_summary, selection_rule_summary_csv)
atomic_write_csv(tenor_summary, tenor_summary_csv)
atomic_write_csv(unique_contract_request_plan, contract_plan_csv)

comparison_paths = {}

if "unique_comparison_summary" in comparison_outputs:
    p = AUDIT_DIR / f"02D3_old_vs_repaired_unique_summary_{SUFFIX}_{timestamp}.csv"
    atomic_write_csv(comparison_outputs["unique_comparison_summary"], p)
    comparison_paths["unique_comparison_summary"] = str(p)

if "added_unique" in comparison_outputs:
    p = AUDIT_DIR / f"02D3_old_vs_repaired_added_unique_{SUFFIX}_{timestamp}.csv"
    atomic_write_csv(comparison_outputs["added_unique"], p)
    comparison_paths["added_unique"] = str(p)

if "removed_unique" in comparison_outputs:
    p = AUDIT_DIR / f"02D3_old_vs_repaired_removed_unique_{SUFFIX}_{timestamp}.csv"
    atomic_write_csv(comparison_outputs["removed_unique"], p)
    comparison_paths["removed_unique"] = str(p)

if "rule_date_comparison" in comparison_outputs:
    p = AUDIT_DIR / f"02D3_old_vs_repaired_rule_date_comparison_{SUFFIX}_{timestamp}.csv"
    atomic_write_csv(comparison_outputs["rule_date_comparison"], p)
    comparison_paths["rule_date_comparison"] = str(p)

if "rule_date_comparison_summary" in comparison_outputs:
    p = AUDIT_DIR / f"02D3_old_vs_repaired_rule_date_comparison_summary_{SUFFIX}_{timestamp}.csv"
    atomic_write_csv(comparison_outputs["rule_date_comparison_summary"], p)
    comparison_paths["rule_date_comparison_summary"] = str(p)

manifest_path = AUDIT_DIR / f"02D3_repaired_rsi_selected_trade_pricing_universe_manifest_{timestamp}.json"

manifest = {
    "cell": "2D.3",
    "description": "Rebuild call-sleeve selected-trade pricing universe using repaired Wilder RSI source",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "repaired_signal_path": str(REPAIRED_SIGNAL_PATH),
        "spy_eod_path": str(SPY_EOD_PATH),
        "old_rule_selected_trades_path": str(OLD_RULE_SELECTED_TRADES_PATH),
        "old_unique_selected_trade_universe_path": str(OLD_UNIQUE_SELECTED_TRADE_UNIVERSE_PATH),
    },
    "resolved_columns": {
        row["normalized_field"]: row["source_column"]
        for _, row in resolved_cols.iterrows()
    },
    "parameter_grid": {
        "target_tenors": TARGET_TENORS,
        "z_thresholds": Z_THRESHOLDS,
        "rsi_floors": RSI_FLOORS,
        "vrp_log_floors": ["NONE" if x is None else x for x in VRP_LOG_FLOORS],
        "selection_rules": SELECTION_RULES,
        "rule_count": int(len(rule_meta)),
    },
    "strike_and_expiration_conventions": {
        "short_call": "ceil raw 1SD call strike to nearest $1",
        "long_call": "round raw 3SD call strike to nearest $5 half-up; if <= short, push to next $5 above short",
        "expiration": "target = trade_date + tenor; first Friday on/after target; if Friday holiday use prior SPY trading day; if nominal Friday beyond EOD calendar, return NaT",
        "terminal_intrinsic": "max(expiry_close - short_strike, 0) - max(expiry_close - long_strike, 0)",
    },
    "outputs": {
        "base_candidate_panel_path": str(BASE_CANDIDATE_PANEL_PATH),
        "rule_selected_trades_path": str(RULE_SELECTED_TRADES_PATH),
        "unique_selected_trade_universe_path": str(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH),
        "selected_leg_request_plan_path": str(SELECTED_LEG_REQUEST_PLAN_PATH),
        "unique_contract_request_plan_path": str(UNIQUE_CONTRACT_REQUEST_PLAN_PATH),
        "rule_preprice_summary_path": str(RULE_PREPRICE_SUMMARY_PATH),
        "rule_year_preprice_summary_path": str(RULE_YEAR_PREPRICE_SUMMARY_PATH),
        "comparison_paths": comparison_paths,
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "source_signal_rows": int(len(signal_raw)),
        "base_rows_after_tenor_filter": int(len(base)),
        "candidate_inputs_complete_rows": int(base["candidate_inputs_complete"].sum()),
        "outcome_available_rows": int(base["outcome_available"].sum()),
        "research_eligible_rows": int(base["research_eligible"].sum()),
        "rule_selected_rows": int(len(rule_selected_trades)),
        "unique_selected_trades": int(len(unique_selected_trades)),
        "selected_leg_rows": int(len(selected_leg_request_plan)),
        "unique_contract_requests": int(len(unique_contract_request_plan)),
        "rule_count": int(len(rule_meta)),
    },
    "notes": {
        "rsi_status": "Uses repaired Wilder RSI signal base.",
        "ranking_status": "No final rule ranking performed.",
        "hybrid_v2_status": "Hybrid v2 put-spread thresholds/sizing/selector are not applied to this call-sleeve research universe.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2D.3 complete — repaired RSI selected-trade universe")
print("=" * 100)
print(f"Rule-selected rows:             {len(rule_selected_trades):,}")
print(f"Unique selected trades:         {len(unique_selected_trades):,}")
print(f"Selected leg rows:              {len(selected_leg_request_plan):,}")
print(f"Unique contract requests:       {len(unique_contract_request_plan):,}")
print(f"Rule count:                     {len(rule_meta):,}")
print(f"Selected date range:            {rule_selected_trades['trade_date'].min()} to {rule_selected_trades['trade_date'].max()}")
print()

print("=" * 100)
print("Selection-rule summary")
print("=" * 100)
display(selection_rule_summary)

print()
print("=" * 100)
print("Unique selected trades by tenor")
print("=" * 100)
display(tenor_summary)

print()
print("=" * 100)
print("Rule preprice summary preview")
print("=" * 100)
display(rule_preprice_summary.sort_values(["selected_rows", "rule_id"], ascending=[False, True]).head(30))

print()
print("=" * 100)
print("Old vs repaired unique selected-trade comparison")
print("=" * 100)
display(comparison_outputs["unique_comparison_summary"])

print()
print("=" * 100)
print("Old vs repaired rule/date comparison")
print("=" * 100)
display(comparison_outputs["rule_date_comparison_summary"])

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Base candidate panel:       {BASE_CANDIDATE_PANEL_PATH}")
print(f"Rule selected trades:       {RULE_SELECTED_TRADES_PATH}")
print(f"Unique trade universe:      {UNIQUE_SELECTED_TRADE_UNIVERSE_PATH}")
print(f"Selected leg plan:          {SELECTED_LEG_REQUEST_PLAN_PATH}")
print(f"Unique contract plan:       {UNIQUE_CONTRACT_REQUEST_PLAN_PATH}")
print(f"Rule preprice summary:      {RULE_PREPRICE_SUMMARY_PATH}")
print(f"Rule-year summary:          {RULE_YEAR_PREPRICE_SUMMARY_PATH}")
print(f"Manifest:                   {manifest_path}")

print()
print("Result: 2D.3 repaired-RSI selected universe complete.")
print("Next step: price the repaired-RSI unique contract request plan, reusing existing cache where contract IDs overlap.")

Cell 2D.3 — Rebuild call-sleeve selected-trade universe from repaired RSI source
Repaired signal source: C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet
SPY EOD calendar:       C:\Users\patri\vrp_project\data\processed\market_data\spy_eod_prices_v1.parquet
Output suffix:          repaired_rsi_long5_cal_v1

Loaded inputs
Repaired signal rows: 14,724
SPY EOD rows:         2,141

Resolved source columns


,normalized_field,source_column
0,trade_date,trade_date
1,tenor,tenor
2,spy_close,spy_close
3,vix_style_vol_pct,vix_style_vol_final
4,implied_variance,implied_variance_final
5,forecast_variance,forecast_variance_final
6,forecast_vol_pct,forecast_vol_final
7,corsi_vrp_log,model_vrp_log_final
8,corsi_vrp_z_3m,z_3m_final
9,corsi_vrp_z_1y,z_1y_final



RSI source integrity check
Max abs(rsi14_final - spy_wilder_rsi14): 0.0
Rows with diff > 1e-10:                  0

Normalized repaired call candidate base
Base rows after tenor filter:          14,724
Date range:                           2020-01-02 00:00:00 to 2026-07-09 00:00:00
Tenors:                               [9, 12, 15, 18, 21, 24, 27, 30, 33]
Candidate inputs complete rows:       12,345
Outcome available rows:               14,613
Research eligible rows:               12,345



,tenor,rows,complete_rows,outcome_available_rows,research_eligible_rows,date_min,date_max
0,9,1636,1380,1632,1380,2020-01-02,2026-07-09
1,12,1636,1377,1629,1377,2020-01-02,2026-07-09
2,15,1636,1376,1628,1376,2020-01-02,2026-07-09
3,18,1636,1373,1625,1373,2020-01-02,2026-07-09
4,21,1636,1372,1624,1372,2020-01-02,2026-07-09
5,24,1636,1370,1622,1370,2020-01-02,2026-07-09
6,27,1636,1368,1620,1368,2020-01-02,2026-07-09
7,30,1636,1366,1618,1366,2020-01-02,2026-07-09
8,33,1636,1363,1615,1363,2020-01-02,2026-07-09



Parameter grid
Rule count: 840


...


Old vs repaired unique selected-trade comparison


,old_unique_selected_trades,new_unique_selected_trades,overlap_unique_selected_trades,added_unique_selected_trades,removed_unique_selected_trades,old_only_pct,new_only_pct
0,1741,1734,1352,382,389,0.223435,0.2203



Old vs repaired rule/date comparison


,comparison_status,rule_date_rows
0,changed_selected_trade,59699
1,new_only_rule_date,2695
2,old_only_rule_date,3125
3,same_selected_trade,92726



Saved
Base candidate panel:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet
Rule selected trades:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_repaired_rsi_long5_cal_v1.parquet
Unique trade universe:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_repaired_rsi_long5_cal_v1.parquet
Selected leg plan:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_repaired_rsi_long5_cal_v1.parquet
Unique contract plan:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_repaired_rsi_long5_cal_v1.parquet
Rule preprice summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\cal

In [4]:
# Cell 2E.3 — Price repaired-RSI selected call-sleeve contracts
# Purpose:
#   Price the repaired-RSI unique selected-contract request plan from Cell 2D.3.
#
# This cell:
#   - Loads the repaired-RSI unique contract request plan
#   - Reuses exact contract matches from:
#       1) existing repaired-RSI cache, if present
#       2) old provisional long5_cal primary cache
#       3) old provisional long5_cal fallback candidate cache
#   - Queries ThetaData only for contracts not already cached
#   - Saves a repaired-RSI contract price cache
#   - Saves audit summaries
#
# This cell does NOT:
#   - Join prices back to spreads
#   - Run fallback mapping
#   - Rank rules
#   - Apply Hybrid v2 put-spread sizing/selector

from pathlib import Path
from datetime import datetime, timezone
import json
import time

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

REPAIRED_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_repaired_wilder_rsi_signal"
    / "vrp_repaired_wilder_rsi_signal_base_v1.parquet"
)

REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_unique_contract_request_plan_{SUFFIX}.parquet"
)

REPAIRED_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_contract_price_cache_{SUFFIX}.parquet"
)

# Existing old caches for exact contract reuse only.
OLD_PRIMARY_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet"
)

OLD_FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet"
)

THETADATA_BASE_URL = "http://127.0.0.1:25503"
THETADATA_ENDPOINT = "/v3/option/history/greeks/eod"
THETADATA_URL = f"{THETADATA_BASE_URL}{THETADATA_ENDPOINT}"

# Set to None to attempt all remaining contracts.
# Set to 250 for a test batch.
MAX_REQUESTS_THIS_RUN = None

REQUEST_TIMEOUT_SECONDS = 20
REQUEST_SLEEP_SECONDS = 0.03
SAVE_EVERY_N_REQUESTS = 25

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2E.3 — Price repaired-RSI selected call-sleeve contracts")
print("=" * 100)
print(f"Request plan:          {REQUEST_PLAN_PATH}")
print(f"Repaired cache output: {REPAIRED_CACHE_PATH}")
print(f"ThetaData endpoint:    {THETADATA_URL}")
print(f"Max requests this run: {MAX_REQUESTS_THIS_RUN}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def yyyymmdd(x):
    return pd.Timestamp(x).strftime("%Y%m%d")

def safe_numeric(x):
    return pd.to_numeric(x, errors="coerce")

def theta_strike_value(strike):
    # ThetaData option strike convention is usually strike * 1000.
    return int(round(float(strike) * 1000))

def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_bool_series(s):
    if s is None:
        return pd.Series(False)
    if s.dtype == bool:
        return s.fillna(False)
    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
    )

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def parse_thetadata_payload_to_df(payload):
    if not isinstance(payload, dict):
        return pd.DataFrame()

    header = payload.get("header", None)
    rows = payload.get("response", None)

    if rows is None:
        rows = payload.get("data", None)

    if isinstance(rows, dict):
        if "data" in rows:
            rows = rows.get("data")
        elif "response" in rows:
            rows = rows.get("response")

    if rows is None:
        return pd.DataFrame()

    if isinstance(header, dict):
        cols = (
            header.get("format")
            or header.get("columns")
            or header.get("fields")
            or header.get("header")
        )
    elif isinstance(header, list):
        cols = header
    else:
        cols = None

    if isinstance(rows, list) and len(rows) == 0:
        return pd.DataFrame()

    try:
        if cols is not None and isinstance(rows, list):
            df = pd.DataFrame(rows, columns=cols)
        else:
            df = pd.DataFrame(rows)
    except Exception:
        df = pd.DataFrame(rows)

    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def extract_quote_fields(df):
    if df is None or df.empty:
        return {}

    d = df.copy()
    d.columns = [str(c).strip().lower() for c in d.columns]

    # If multiple rows come back, use the last row after ms_of_day sorting if possible.
    if "ms_of_day" in d.columns:
        d["ms_of_day"] = pd.to_numeric(d["ms_of_day"], errors="coerce")
        d = d.sort_values("ms_of_day")

    row = d.iloc[-1]

    bid_col = first_existing_col(d, ["bid", "bid_price", "best_bid"])
    ask_col = first_existing_col(d, ["ask", "ask_price", "best_ask"])

    underlying_col = first_existing_col(
        d,
        ["underlying_price", "underlying", "underlying_close", "root_price"],
    )

    iv_col = first_existing_col(
        d,
        ["implied_vol", "iv", "implied_volatility", "implied_volatility_mid"],
    )

    delta_col = first_existing_col(d, ["delta"])
    gamma_col = first_existing_col(d, ["gamma"])
    theta_col = first_existing_col(d, ["theta"])
    vega_col = first_existing_col(d, ["vega"])
    rho_col = first_existing_col(d, ["rho"])

    fields = {
        "bid": safe_numeric(row[bid_col]) if bid_col else np.nan,
        "ask": safe_numeric(row[ask_col]) if ask_col else np.nan,
        "underlying_price": safe_numeric(row[underlying_col]) if underlying_col else np.nan,
        "implied_vol": safe_numeric(row[iv_col]) if iv_col else np.nan,
        "delta": safe_numeric(row[delta_col]) if delta_col else np.nan,
        "gamma": safe_numeric(row[gamma_col]) if gamma_col else np.nan,
        "theta": safe_numeric(row[theta_col]) if theta_col else np.nan,
        "vega": safe_numeric(row[vega_col]) if vega_col else np.nan,
        "rho": safe_numeric(row[rho_col]) if rho_col else np.nan,
        "raw_columns": ",".join(d.columns),
        "raw_row_count": int(len(d)),
    }

    fields["mid"] = (
        (fields["bid"] + fields["ask"]) / 2.0
        if pd.notna(fields["bid"]) and pd.notna(fields["ask"])
        else np.nan
    )

    fields["bid_ask_spread"] = (
        fields["ask"] - fields["bid"]
        if pd.notna(fields["bid"]) and pd.notna(fields["ask"])
        else np.nan
    )

    return fields

STANDARD_CACHE_COLUMNS = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "trade_date_str",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "strike_listed",
    "theta_strike",
    "status",
    "status_code",
    "price_found",
    "bid",
    "ask",
    "mid",
    "bid_ask_spread",
    "underlying_price",
    "implied_vol",
    "delta",
    "gamma",
    "theta",
    "vega",
    "rho",
    "raw_columns",
    "raw_row_count",
    "query_url",
    "query_params_json",
    "source_cache",
    "reused_from_cache",
    "request_timestamp_utc",
]

def normalize_cache_schema(cache_df, source_cache_name):
    if cache_df is None or cache_df.empty:
        return pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

    df = cache_df.copy()

    # Reconstruct contract_request_id if needed and possible.
    if "contract_request_id" not in df.columns:
        needed = ["symbol", "trade_date", "expiration_date", "strike_listed", "right"]
        if all(c in df.columns for c in needed):
            df["contract_request_id"] = [
                f"{sym}_{yyyymmdd(td)}_{yyyymmdd(exp)}_{float(strike):.1f}_{right}"
                for sym, td, exp, strike, right in zip(
                    df["symbol"],
                    df["trade_date"],
                    df["expiration_date"],
                    df["strike_listed"],
                    df["right"],
                )
            ]
        else:
            raise ValueError(f"Cache {source_cache_name} has no contract_request_id and cannot reconstruct it.")

    # Common aliases.
    rename_map = {}

    if "request_status" in df.columns and "status" not in df.columns:
        rename_map["request_status"] = "status"

    if "http_status_code" in df.columns and "status_code" not in df.columns:
        rename_map["http_status_code"] = "status_code"

    if "strike" in df.columns and "strike_listed" not in df.columns:
        rename_map["strike"] = "strike_listed"

    if "exp" in df.columns and "expiration_yyyymmdd" not in df.columns:
        rename_map["exp"] = "expiration_yyyymmdd"

    if rename_map:
        df = df.rename(columns=rename_map)

    # If price_found missing, infer it.
    if "price_found" not in df.columns:
        if "status" in df.columns:
            df["price_found"] = df["status"].astype(str).str.lower().eq("priced")
        elif "bid" in df.columns and "ask" in df.columns:
            df["price_found"] = df["bid"].notna() & df["ask"].notna()
        else:
            df["price_found"] = False

    # Dates.
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

    if "trade_date_str" not in df.columns and "trade_date" in df.columns:
        df["trade_date_str"] = pd.to_datetime(df["trade_date"]).dt.strftime("%Y-%m-%d")

    if "expiration_str" not in df.columns and "expiration_date" in df.columns:
        df["expiration_str"] = pd.to_datetime(df["expiration_date"]).dt.strftime("%Y-%m-%d")

    if "expiration_yyyymmdd" not in df.columns and "expiration_date" in df.columns:
        df["expiration_yyyymmdd"] = pd.to_datetime(df["expiration_date"]).dt.strftime("%Y%m%d")

    if "theta_strike" not in df.columns and "strike_listed" in df.columns:
        df["theta_strike"] = df["strike_listed"].apply(lambda x: theta_strike_value(x) if pd.notna(x) else np.nan)

    # Numeric fields.
    for c in [
        "strike_listed",
        "theta_strike",
        "status_code",
        "bid",
        "ask",
        "mid",
        "bid_ask_spread",
        "underlying_price",
        "implied_vol",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "raw_row_count",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["price_found"] = normalize_bool_series(df["price_found"])
    df["source_cache"] = source_cache_name
    df["reused_from_cache"] = True

    if "request_timestamp_utc" not in df.columns:
        df["request_timestamp_utc"] = pd.NaT

    for c in STANDARD_CACHE_COLUMNS:
        if c not in df.columns:
            df[c] = np.nan

    return df[STANDARD_CACHE_COLUMNS].copy()

def query_contract(row, session):
    symbol = str(row["symbol"])
    right = str(row["right"])
    trade_date = pd.Timestamp(row["trade_date"])
    expiration_date = pd.Timestamp(row["expiration_date"])
    strike = float(row["strike_listed"])
    theta_strike = theta_strike_value(strike)

    params = {
        "root": symbol,
        "exp": yyyymmdd(expiration_date),
        "right": right,
        "strike": theta_strike,
        "start_date": yyyymmdd(trade_date),
        "end_date": yyyymmdd(trade_date),
    }

    query_params_json = json.dumps(params, sort_keys=True)

    out = {
        "contract_request_id": row["contract_request_id"],
        "symbol": symbol,
        "right": right,
        "trade_date": trade_date,
        "trade_date_str": trade_date.strftime("%Y-%m-%d"),
        "expiration_date": expiration_date,
        "expiration_str": expiration_date.strftime("%Y-%m-%d"),
        "expiration_yyyymmdd": expiration_date.strftime("%Y%m%d"),
        "strike_listed": strike,
        "theta_strike": theta_strike,
        "status": None,
        "status_code": np.nan,
        "price_found": False,
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_ask_spread": np.nan,
        "underlying_price": np.nan,
        "implied_vol": np.nan,
        "delta": np.nan,
        "gamma": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "rho": np.nan,
        "raw_columns": "",
        "raw_row_count": 0,
        "query_url": THETADATA_URL,
        "query_params_json": query_params_json,
        "source_cache": "thetadata_query",
        "reused_from_cache": False,
        "request_timestamp_utc": now_utc_iso(),
    }

    try:
        response = session.get(
            THETADATA_URL,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        out["status_code"] = int(response.status_code)

        if response.status_code != 200:
            out["status"] = f"http_{response.status_code}"
            return out

        try:
            payload = response.json()
        except Exception:
            out["status"] = "json_parse_error"
            return out

        df = parse_thetadata_payload_to_df(payload)

        if df.empty:
            out["status"] = "no_rows"
            return out

        quote_fields = extract_quote_fields(df)
        out.update(quote_fields)

        if (
            pd.notna(out["bid"])
            and pd.notna(out["ask"])
            and out["bid"] >= 0
            and out["ask"] >= 0
        ):
            out["status"] = "priced"
            out["price_found"] = True
        else:
            out["status"] = "missing_bid_ask"
            out["price_found"] = False

        return out

    except requests.exceptions.Timeout:
        out["status"] = "timeout"
        return out

    except requests.exceptions.ConnectionError:
        out["status"] = "connection_error"
        return out

    except Exception as e:
        out["status"] = f"exception_{type(e).__name__}"
        out["raw_columns"] = str(e)[:500]
        return out

def save_cache(cache_df):
    cache_df = cache_df.copy()

    for c in ["trade_date", "expiration_date"]:
        if c in cache_df.columns:
            cache_df[c] = pd.to_datetime(cache_df[c], errors="coerce").dt.normalize()

    if "price_found" in cache_df.columns:
        cache_df["price_found"] = normalize_bool_series(cache_df["price_found"])

    cache_df = cache_df[STANDARD_CACHE_COLUMNS].copy()
    atomic_write_parquet(cache_df, REPAIRED_CACHE_PATH)

# --------------------------------------------------------------------------------------------------
# Load request plan
# --------------------------------------------------------------------------------------------------

require_file(REQUEST_PLAN_PATH)

plan = pd.read_parquet(REQUEST_PLAN_PATH).copy()

required_plan_cols = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "expiration_date",
    "strike_listed",
]

missing_plan_cols = [c for c in required_plan_cols if c not in plan.columns]
if missing_plan_cols:
    raise ValueError(f"Request plan missing required columns: {missing_plan_cols}")

plan["contract_request_id"] = plan["contract_request_id"].astype(str)
plan["trade_date"] = pd.to_datetime(plan["trade_date"], errors="coerce").dt.normalize()
plan["expiration_date"] = pd.to_datetime(plan["expiration_date"], errors="coerce").dt.normalize()
plan["strike_listed"] = pd.to_numeric(plan["strike_listed"], errors="coerce")
plan["symbol"] = plan["symbol"].astype(str)
plan["right"] = plan["right"].astype(str)

if "trade_date_str" not in plan.columns:
    plan["trade_date_str"] = plan["trade_date"].dt.strftime("%Y-%m-%d")
if "expiration_str" not in plan.columns:
    plan["expiration_str"] = plan["expiration_date"].dt.strftime("%Y-%m-%d")
if "expiration_yyyymmdd" not in plan.columns:
    plan["expiration_yyyymmdd"] = plan["expiration_date"].dt.strftime("%Y%m%d")

plan = (
    plan
    .drop_duplicates("contract_request_id")
    .sort_values(["trade_date", "expiration_date", "strike_listed", "right"])
    .reset_index(drop=True)
)

if plan["contract_request_id"].duplicated().any():
    raise ValueError("Duplicate contract_request_id values remain in request plan.")

print("=" * 100)
print("Loaded repaired-RSI request plan")
print("=" * 100)
print(f"Plan rows:              {len(plan):,}")
print(f"Trade date range:       {plan['trade_date'].min()} to {plan['trade_date'].max()}")
print(f"Expiration date range:  {plan['expiration_date'].min()} to {plan['expiration_date'].max()}")
print()

# --------------------------------------------------------------------------------------------------
# Optional model audit for same Corsi denominator/source
# --------------------------------------------------------------------------------------------------

if REPAIRED_SIGNAL_PATH.exists():
    signal_raw = pd.read_parquet(REPAIRED_SIGNAL_PATH).copy()

    model_audit_cols = [
        c for c in [
            "final_signal_version",
            "model_spec",
            "model_source",
            "fit_status_candidate",
            "forecast_repair_scope",
            "rsi_formula_version",
        ]
        if c in signal_raw.columns
    ]

    print("=" * 100)
    print("Repaired signal source model audit")
    print("=" * 100)

    if model_audit_cols:
        model_audit = (
            signal_raw[model_audit_cols]
            .drop_duplicates()
            .sort_values(model_audit_cols)
            .reset_index(drop=True)
        )
        display(model_audit)

        if "model_spec" in signal_raw.columns:
            model_specs = set(signal_raw["model_spec"].dropna().astype(str).unique())
            print("model_spec values:", model_specs)

        if "model_source" in signal_raw.columns:
            model_sources = set(signal_raw["model_source"].dropna().astype(str).unique())
            print("model_source values:", model_sources)

        if "rsi_formula_version" in signal_raw.columns:
            rsi_versions = set(signal_raw["rsi_formula_version"].dropna().astype(str).unique())
            print("rsi_formula_version values:", rsi_versions)
    else:
        print("No model audit columns found in repaired signal file.")

    print()

# --------------------------------------------------------------------------------------------------
# Load and merge caches
# --------------------------------------------------------------------------------------------------

cache_frames = []

if REPAIRED_CACHE_PATH.exists():
    existing_repaired_cache = pd.read_parquet(REPAIRED_CACHE_PATH)
    cache_frames.append(normalize_cache_schema(existing_repaired_cache, "existing_repaired_cache"))
    print(f"Existing repaired cache rows loaded: {len(existing_repaired_cache):,}")
else:
    print("Existing repaired cache rows loaded: 0")

if OLD_PRIMARY_CACHE_PATH.exists():
    old_primary_cache = pd.read_parquet(OLD_PRIMARY_CACHE_PATH)
    old_primary_cache_norm = normalize_cache_schema(old_primary_cache, "old_primary_long5_cal_cache")
    old_primary_cache_norm = old_primary_cache_norm.loc[
        old_primary_cache_norm["contract_request_id"].isin(set(plan["contract_request_id"]))
    ].copy()
    cache_frames.append(old_primary_cache_norm)
    print(f"Reusable old primary cache rows loaded: {len(old_primary_cache_norm):,}")
else:
    print("Reusable old primary cache rows loaded: 0")

if OLD_FALLBACK_CANDIDATE_CACHE_PATH.exists():
    old_fallback_cache = pd.read_parquet(OLD_FALLBACK_CANDIDATE_CACHE_PATH)
    old_fallback_cache_norm = normalize_cache_schema(old_fallback_cache, "old_fallback_candidate_cache")
    old_fallback_cache_norm = old_fallback_cache_norm.loc[
        old_fallback_cache_norm["contract_request_id"].isin(set(plan["contract_request_id"]))
    ].copy()
    cache_frames.append(old_fallback_cache_norm)
    print(f"Reusable old fallback candidate cache rows loaded: {len(old_fallback_cache_norm):,}")
else:
    print("Reusable old fallback candidate cache rows loaded: 0")

if cache_frames:
    cache = pd.concat(cache_frames, ignore_index=True)
else:
    cache = pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

# Keep best available row per contract_request_id:
#   prefer price_found rows, then existing repaired cache, then old primary, then old fallback.
source_priority = {
    "existing_repaired_cache": 0,
    "old_primary_long5_cal_cache": 1,
    "old_fallback_candidate_cache": 2,
    "thetadata_query": 3,
}

if not cache.empty:
    cache["source_priority"] = cache["source_cache"].map(source_priority).fillna(9).astype(int)
    cache["price_found_sort"] = normalize_bool_series(cache["price_found"]).astype(int)

    cache = (
        cache
        .sort_values(
            ["contract_request_id", "price_found_sort", "source_priority"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
        .reset_index(drop=True)
    )

    cache = cache.loc[cache["contract_request_id"].isin(set(plan["contract_request_id"]))].copy()
    save_cache(cache)

cached_ids = set(cache["contract_request_id"].astype(str)) if not cache.empty else set()
remaining = plan.loc[~plan["contract_request_id"].astype(str).isin(cached_ids)].copy()

if MAX_REQUESTS_THIS_RUN is not None:
    run_batch = remaining.head(int(MAX_REQUESTS_THIS_RUN)).copy()
else:
    run_batch = remaining.copy()

print()
print("=" * 100)
print("Cache / remaining summary before ThetaData requests")
print("=" * 100)
print(f"Plan rows:                  {len(plan):,}")
print(f"Cache rows after reuse:     {len(cache):,}")
print(f"Remaining uncached rows:    {len(remaining):,}")
print(f"Rows to query this run:     {len(run_batch):,}")

if not cache.empty:
    cache_status_summary_pre = (
        cache
        .groupby(["source_cache", "status", "price_found"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["source_cache", "status", "price_found"])
    )
    display(cache_status_summary_pre)

print()

# --------------------------------------------------------------------------------------------------
# Query ThetaData for remaining contracts
# --------------------------------------------------------------------------------------------------

new_rows = []

if len(run_batch) > 0:
    session = requests.Session()

    print("=" * 100)
    print("Querying ThetaData")
    print("=" * 100)

    for i, (_, row) in enumerate(run_batch.iterrows(), start=1):
        result = query_contract(row, session)
        new_rows.append(result)

        if i <= 5 or i % 50 == 0 or i == len(run_batch):
            print(
                f"{i:>6,}/{len(run_batch):,} "
                f"{result['contract_request_id']} "
                f"status={result['status']} "
                f"price_found={result['price_found']}"
            )

        if SAVE_EVERY_N_REQUESTS and i % SAVE_EVERY_N_REQUESTS == 0:
            new_df_partial = pd.DataFrame(new_rows)
            new_df_partial = normalize_cache_schema(new_df_partial, "thetadata_query")

            combined_partial = pd.concat([cache, new_df_partial], ignore_index=True)

            combined_partial["source_priority"] = combined_partial["source_cache"].map(source_priority).fillna(9).astype(int)
            combined_partial["price_found_sort"] = normalize_bool_series(combined_partial["price_found"]).astype(int)

            combined_partial = (
                combined_partial
                .sort_values(
                    ["contract_request_id", "price_found_sort", "source_priority"],
                    ascending=[True, False, True],
                    kind="mergesort",
                )
                .drop_duplicates("contract_request_id", keep="first")
                .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
                .reset_index(drop=True)
            )

            save_cache(combined_partial)

        time.sleep(REQUEST_SLEEP_SECONDS)

    new_df = pd.DataFrame(new_rows)
    new_df = normalize_cache_schema(new_df, "thetadata_query")

    cache = pd.concat([cache, new_df], ignore_index=True)

    cache["source_priority"] = cache["source_cache"].map(source_priority).fillna(9).astype(int)
    cache["price_found_sort"] = normalize_bool_series(cache["price_found"]).astype(int)

    cache = (
        cache
        .sort_values(
            ["contract_request_id", "price_found_sort", "source_priority"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
        .reset_index(drop=True)
    )

    save_cache(cache)

else:
    print("=" * 100)
    print("No ThetaData requests needed; all contracts are already cached.")
    print("=" * 100)
    new_df = pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

# --------------------------------------------------------------------------------------------------
# Final summaries and audits
# --------------------------------------------------------------------------------------------------

cache = pd.read_parquet(REPAIRED_CACHE_PATH).copy()
cache = normalize_cache_schema(cache, "final_repaired_cache")
cache = cache.loc[cache["contract_request_id"].isin(set(plan["contract_request_id"]))].copy()

plan_with_cache = plan.merge(
    cache[
        [
            "contract_request_id",
            "source_cache",
            "status",
            "status_code",
            "price_found",
            "bid",
            "ask",
            "mid",
            "bid_ask_spread",
            "underlying_price",
            "implied_vol",
        ]
    ],
    on="contract_request_id",
    how="left",
)

plan_with_cache["price_found"] = normalize_bool_series(plan_with_cache["price_found"])
plan_with_cache["cache_row_found"] = plan_with_cache["status"].notna()

final_status_summary = (
    plan_with_cache
    .groupby(["cache_row_found", "price_found", "status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["cache_row_found", "price_found", "status"])
)

status_code_summary = (
    plan_with_cache
    .groupby(["status_code", "status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["status_code", "status"])
)

source_cache_summary = (
    plan_with_cache
    .groupby(["source_cache", "price_found", "status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source_cache", "price_found", "status"])
)

date_coverage = (
    plan_with_cache
    .groupby("trade_date", dropna=False)
    .agg(
        contract_requests=("contract_request_id", "size"),
        cached_rows=("cache_row_found", "sum"),
        priced_rows=("price_found", "sum"),
        missing_or_failed_rows=("price_found", lambda x: int((~normalize_bool_series(x)).sum())),
    )
    .reset_index()
)

date_coverage["priced_rate"] = np.where(
    date_coverage["contract_requests"] > 0,
    date_coverage["priced_rows"] / date_coverage["contract_requests"],
    np.nan,
)

missing_preview = (
    plan_with_cache
    .loc[~plan_with_cache["price_found"]]
    .sort_values(["trade_date", "expiration_date", "strike_listed"])
    .head(100)
    .copy()
)

latest_cache_csv = AUDIT_DIR / f"02E3_call_sleeve_corsi_call_selected_contract_price_cache_{SUFFIX}_latest.csv"
run_audit_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_run_audit_{timestamp}.csv"
status_summary_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_status_summary_{timestamp}.csv"
status_code_summary_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_status_code_summary_{timestamp}.csv"
source_cache_summary_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_source_summary_{timestamp}.csv"
date_coverage_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_date_coverage_{timestamp}.csv"
missing_preview_csv = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_missing_preview_{timestamp}.csv"

atomic_write_csv(cache, latest_cache_csv)
atomic_write_csv(new_df, run_audit_csv)
atomic_write_csv(final_status_summary, status_summary_csv)
atomic_write_csv(status_code_summary, status_code_summary_csv)
atomic_write_csv(source_cache_summary, source_cache_summary_csv)
atomic_write_csv(date_coverage, date_coverage_csv)
atomic_write_csv(missing_preview, missing_preview_csv)

manifest_path = AUDIT_DIR / f"02E3_{SUFFIX}_selected_contract_price_cache_manifest_{timestamp}.json"

manifest = {
    "cell": "2E.3",
    "description": "Price repaired-RSI selected call-sleeve contracts using ThetaData v3 EOD greeks endpoint with cache reuse",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "theta_endpoint": THETADATA_URL,
    "inputs": {
        "request_plan_path": str(REQUEST_PLAN_PATH),
        "repaired_signal_path": str(REPAIRED_SIGNAL_PATH),
        "existing_repaired_cache_path": str(REPAIRED_CACHE_PATH),
        "old_primary_cache_path": str(OLD_PRIMARY_CACHE_PATH),
        "old_fallback_candidate_cache_path": str(OLD_FALLBACK_CANDIDATE_CACHE_PATH),
    },
    "outputs": {
        "repaired_cache_path": str(REPAIRED_CACHE_PATH),
        "latest_cache_csv": str(latest_cache_csv),
        "run_audit_csv": str(run_audit_csv),
        "status_summary_csv": str(status_summary_csv),
        "status_code_summary_csv": str(status_code_summary_csv),
        "source_cache_summary_csv": str(source_cache_summary_csv),
        "date_coverage_csv": str(date_coverage_csv),
        "missing_preview_csv": str(missing_preview_csv),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "plan_rows": int(len(plan)),
        "cache_rows_final": int(len(cache)),
        "run_attempts": int(len(run_batch)),
        "new_rows_from_this_run": int(len(new_df)),
        "final_priced_rows": int(plan_with_cache["price_found"].sum()),
        "final_missing_or_failed_rows": int((~plan_with_cache["price_found"]).sum()),
        "cache_row_found_rows": int(plan_with_cache["cache_row_found"].sum()),
        "uncached_rows_after_run": int((~plan_with_cache["cache_row_found"]).sum()),
    },
    "config": {
        "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
        "request_timeout_seconds": REQUEST_TIMEOUT_SECONDS,
        "request_sleep_seconds": REQUEST_SLEEP_SECONDS,
        "save_every_n_requests": SAVE_EVERY_N_REQUESTS,
        "theta_strike_convention": "strike_listed * 1000",
    },
    "notes": {
        "ranking_status": "No final rule ranking performed.",
        "next_step": "Join repaired-RSI prices back to unique spreads and rule-selected rows.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Cell 2E.3 complete — repaired-RSI contract price cache")
print("=" * 100)
print(f"Plan rows:                       {len(plan):,}")
print(f"Final cache rows:                {len(cache):,}")
print(f"Run attempts this cell:          {len(run_batch):,}")
print(f"Final priced rows:               {int(plan_with_cache['price_found'].sum()):,}")
print(f"Final missing/failed rows:       {int((~plan_with_cache['price_found']).sum()):,}")
print(f"Cache row found rows:            {int(plan_with_cache['cache_row_found'].sum()):,}")
print(f"Uncached rows after run:         {int((~plan_with_cache['cache_row_found']).sum()):,}")
print(f"Final contract pricing rate:     {plan_with_cache['price_found'].mean():.2%}")
print()

print("=" * 100)
print("Final status summary")
print("=" * 100)
display(final_status_summary)

print()
print("=" * 100)
print("Source cache summary")
print("=" * 100)
display(source_cache_summary)

print()
print("=" * 100)
print("Status-code summary")
print("=" * 100)
display(status_code_summary)

print()
print("=" * 100)
print("Missing / failed preview")
print("=" * 100)
if missing_preview.empty:
    print("No missing/failed contracts.")
else:
    display(missing_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Repaired cache parquet:       {REPAIRED_CACHE_PATH}")
print(f"Latest cache CSV:             {latest_cache_csv}")
print(f"Run audit CSV:                {run_audit_csv}")
print(f"Status summary CSV:           {status_summary_csv}")
print(f"Source summary CSV:           {source_cache_summary_csv}")
print(f"Date coverage CSV:            {date_coverage_csv}")
print(f"Missing preview CSV:          {missing_preview_csv}")
print(f"Manifest:                     {manifest_path}")

print()
print("Result: 2E.3 repaired-RSI contract pricing complete.")
print("Next step: 2F.4 join repaired-RSI prices back to selected spreads and compute spread P&L.")

Cell 2E.3 — Price repaired-RSI selected call-sleeve contracts
Request plan:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_unique_contract_request_plan_repaired_rsi_long5_cal_v1.parquet
Repaired cache output: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet
ThetaData endpoint:    http://127.0.0.1:25503/v3/option/history/greeks/eod
Max requests this run: None

Loaded repaired-RSI request plan
Plan rows:              3,423
Trade date range:       2020-12-31 00:00:00 to 2026-06-02 00:00:00
Expiration date range:  2021-01-15 00:00:00 to 2026-07-10 00:00:00

Repaired signal source model audit


,final_signal_version,model_spec,model_source,fit_status_candidate,forecast_repair_scope,rsi_formula_version
0,vrp_final_corsi_signal_panel_v1,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,RESEARCH_FRONT,wilder_rsi14_spy_close_v2_long_warmup
1,vrp_final_corsi_signal_panel_v1,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,RESEARCH_MIDDLE,wilder_rsi14_spy_close_v2_long_warmup
2,vrp_final_corsi_signal_panel_v1,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,wilder_rsi14_spy_close_v2_long_warmup
3,None,None,None,None,None,wilder_rsi14_spy_close_v2_long_warmup


model_spec values: {'unified_fds_no_min_return'}
model_source values: {'unified_fds_no_min_return_oos_refit'}
rsi_formula_version values: {'wilder_rsi14_spy_close_v2_long_warmup'}

Existing repaired cache rows loaded: 0
Reusable old primary cache rows loaded: 3,389
Reusable old fallback candidate cache rows loaded: 0

Cache / remaining summary before ThetaData requests
Plan rows:                  3,423
Cache rows after reuse:     3,389
Remaining uncached rows:    34
Rows to query this run:     34


,source_cache,status,price_found,rows
0,old_primary_long5_cal_cache,NaN,False,598
1,old_primary_long5_cal_cache,NaN,True,2791



Querying ThetaData
     1/34 SPY_20210617_20210702_433.0_C status=http_410 price_found=False
     2/34 SPY_20210617_20210702_435.0_C status=http_410 price_found=False
     3/34 SPY_20210617_20210702_455.0_C status=http_410 price_found=False
     4/34 SPY_20210617_20210702_460.0_C status=http_410 price_found=False
     5/34 SPY_20230209_20230224_422.0_C status=http_410 price_found=False
    34/34 SPY_20260602_20260710_870.0_C status=http_410 price_found=False

Cell 2E.3 complete — repaired-RSI contract price cache
Plan rows:                       3,423
Final cache rows:                3,423
Run attempts this cell:          34
Final priced rows:               2,791
Final missing/failed rows:       632
Cache row found rows:            34
Uncached rows after run:         3,389
Final contract pricing rate:     81.54%

Final status summary


...


Status-code summary


,status_code,status,rows
0,200,NaN,2791
1,410,http_410,34
2,472,NaN,598



Missing / failed preview


,contract_request_id,symbol,right,trade_date,trade_date_str,expiration_date,expiration_str,expiration_yyyymmdd,strike_listed,source_cache,status,status_code,price_found,bid,ask,mid,bid_ask_spread,underlying_price,implied_vol,cache_row_found
5,SPY_20201231_20210205_450.0_C,SPY,C,2020-12-31,2020-12-31,2021-02-05,2021-02-05,20210205,450.0,final_repaired_cache,None,472,False,NaN,NaN,NaN,NaN,NaN,NaN,False
82,SPY_20210617_20210702_433.0_C,SPY,C,2021-06-17,2021-06-17,2021-07-02,2021-07-02,20210702,433.0,final_repaired_cache,http_410,410,False,NaN,NaN,NaN,NaN,NaN,NaN,True
83,SPY_20210617_20210702_435.0_C,SPY,C,2021-06-17,2021-06-17,2021-07-02,2021-07-02,20210702,435.0,final_repaired_cache,http_410,410,False,NaN,NaN,NaN,NaN,NaN,NaN,True
84,SPY_20210617_20210702_455.0_C,SPY,C,2021-06-17,2021-06-17,2021-07-02,2021-07-02,20210702,455.0,final_repaired_cache,http_410,410,False,NaN,NaN,NaN,NaN,NaN,NaN,True
85,SPY_20210617_20210702_460.0_C,SPY,C,2021-06-17,2021-06-17,2021-07-02,2021-07-02,20210702,460.0,final_repaired_cache,http_410,410,False,NaN,NaN,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1106,SPY_20231219_20240112_489.0_C,SPY,C,2023-12-19,2023-12-19,2024-01-12,2024-01-12,20240112,489.0,final_repaired_cache,None,472,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1112,SPY_20231219_20240126_493.0_C,SPY,C,2023-12-19,2023-12-19,2024-01-26,2024-01-26,20240126,493.0,final_repaired_cache,None,472,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1123,SPY_20240116_20240209_491.0_C,SPY,C,2024-01-16,2024-01-16,2024-02-09,2024-02-09,20240209,491.0,final_repaired_cache,None,472,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1151,SPY_20240118_20240223_497.0_C,SPY,C,2024-01-18,2024-01-18,2024-02-23,2024-02-23,20240223,497.0,final_repaired_cache,None,472,False,NaN,NaN,NaN,NaN,NaN,NaN,False



Saved
Repaired cache parquet:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet
Latest cache CSV:             C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E3_call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1_latest.csv
Run audit CSV:                C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E3_repaired_rsi_long5_cal_v1_selected_contract_price_cache_run_audit_20260712_131958.csv
Status summary CSV:           C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E3_repaired_rsi_long5_cal_v1_selected_contract_price_cache_status_summary_20260712_131958.csv
Source summary CSV:           C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02E3_repaired_rsi_long5_cal_v1_selected_contract_price_cache_source_summary_20260712_131958.csv
Date coverage CSV:            C:\Users\patri\vrp_project\data\audit\call_sleeve

In [5]:
# Cell 2F.4 — Join repaired-RSI primary contract prices back to spreads and compute spread P&L
# Purpose:
#   Primary-price join for the repaired-RSI call-sleeve universe.
#
# This cell:
#   - Loads repaired-RSI leg request plan
#   - Loads repaired-RSI unique selected trade universe
#   - Loads repaired-RSI rule-selected trades
#   - Loads repaired-RSI primary contract price cache
#   - Joins bid/ask prices to short and long call legs
#   - Computes conservative entry credit, max loss, expiry P&L, and return on max loss
#   - Saves spread-level and rule-level primary-priced outputs
#
# This cell does NOT:
#   - Run listed-strike fallback
#   - Rank final rules
#   - Select final parameters
#   - Apply Hybrid v2 put-spread sizing/selector

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

LEG_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_leg_request_plan_{SUFFIX}.parquet"
)

UNIQUE_SELECTED_TRADE_UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_{SUFFIX}.parquet"
)

RULE_SELECTED_TRADES_UNPRICED_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_unpriced_{SUFFIX}.parquet"
)

CONTRACT_PRICE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_contract_price_cache_{SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F.4 — Join repaired-RSI primary contract prices back to spreads and compute spread P&L")
print("=" * 100)
print(f"Leg request plan:        {LEG_REQUEST_PLAN_PATH}")
print(f"Unique trade universe:   {UNIQUE_SELECTED_TRADE_UNIVERSE_PATH}")
print(f"Rule selected unpriced:  {RULE_SELECTED_TRADES_UNPRICED_PATH}")
print(f"Contract price cache:    {CONTRACT_PRICE_CACHE_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def bool_series(s):
    if s is None:
        return pd.Series(False)
    if s.dtype == bool:
        return s.fillna(False)
    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
    )

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    LEG_REQUEST_PLAN_PATH,
    UNIQUE_SELECTED_TRADE_UNIVERSE_PATH,
    RULE_SELECTED_TRADES_UNPRICED_PATH,
    CONTRACT_PRICE_CACHE_PATH,
]:
    require_file(p)

legs = pd.read_parquet(LEG_REQUEST_PLAN_PATH).copy()
unique_trades = pd.read_parquet(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH).copy()
rule_trades = pd.read_parquet(RULE_SELECTED_TRADES_UNPRICED_PATH).copy()
cache = pd.read_parquet(CONTRACT_PRICE_CACHE_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Leg request rows:        {len(legs):,}")
print(f"Unique spread rows:      {len(unique_trades):,}")
print(f"Rule selected rows:      {len(rule_trades):,}")
print(f"Contract cache rows:     {len(cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Required columns and normalization
# --------------------------------------------------------------------------------------------------

required_leg_cols = [
    "selected_trade_id",
    "leg_label",
    "contract_request_id",
    "trade_date",
    "expiration_date",
    "strike_listed",
]

required_unique_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
]

required_rule_cols = [
    "rule_id",
    "selected_trade_id",
    "trade_date",
    "tenor",
    "selection_rule",
]

required_cache_cols = [
    "contract_request_id",
    "price_found",
    "bid",
    "ask",
]

missing_leg_cols = [c for c in required_leg_cols if c not in legs.columns]
missing_unique_cols = [c for c in required_unique_cols if c not in unique_trades.columns]
missing_rule_cols = [c for c in required_rule_cols if c not in rule_trades.columns]
missing_cache_cols = [c for c in required_cache_cols if c not in cache.columns]

if missing_leg_cols:
    raise ValueError(f"Leg request plan missing columns: {missing_leg_cols}")
if missing_unique_cols:
    raise ValueError(f"Unique trade universe missing columns: {missing_unique_cols}")
if missing_rule_cols:
    raise ValueError(f"Rule selected trades missing columns: {missing_rule_cols}")
if missing_cache_cols:
    raise ValueError(f"Contract cache missing columns: {missing_cache_cols}")

for df in [legs, unique_trades, rule_trades, cache]:
    if "selected_trade_id" in df.columns:
        df["selected_trade_id"] = df["selected_trade_id"].astype(str)
    if "contract_request_id" in df.columns:
        df["contract_request_id"] = df["contract_request_id"].astype(str)

for df in [legs, unique_trades, rule_trades, cache]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "strike_listed",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
    "bid",
    "ask",
    "mid",
    "bid_ask_spread",
    "underlying_price",
    "implied_vol",
    "status_code",
]:
    for df in [legs, unique_trades, rule_trades, cache]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

cache["price_found"] = bool_series(cache["price_found"])

if "status" not in cache.columns:
    cache["status"] = np.nan
if "status_code" not in cache.columns:
    cache["status_code"] = np.nan
if "source_cache" not in cache.columns:
    cache["source_cache"] = np.nan

# If duplicate cache rows exist, prefer price_found rows.
cache = (
    cache
    .assign(price_found_sort=cache["price_found"].astype(int))
    .sort_values(["contract_request_id", "price_found_sort"], ascending=[True, False], kind="mergesort")
    .drop_duplicates("contract_request_id", keep="first")
    .drop(columns=["price_found_sort"], errors="ignore")
    .reset_index(drop=True)
)

if unique_trades["selected_trade_id"].duplicated().any():
    dup_count = int(unique_trades["selected_trade_id"].duplicated().sum())
    raise ValueError(f"Unique selected trade universe has duplicate selected_trade_id rows: {dup_count}")

# --------------------------------------------------------------------------------------------------
# Join cache to legs
# --------------------------------------------------------------------------------------------------

price_cols = [
    "contract_request_id",
    "price_found",
    "status",
    "status_code",
    "source_cache",
    "bid",
    "ask",
    "mid",
    "bid_ask_spread",
    "underlying_price",
    "implied_vol",
]

price_cols = [c for c in price_cols if c in cache.columns]

leg_prices = legs.merge(
    cache[price_cols],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_prices["price_found"] = bool_series(leg_prices["price_found"])
leg_prices["cache_row_found"] = leg_prices["bid"].notna() | leg_prices["ask"].notna() | leg_prices["status"].notna() | leg_prices["status_code"].notna()

leg_prices["leg_price_found"] = (
    leg_prices["price_found"]
    & leg_prices["bid"].notna()
    & leg_prices["ask"].notna()
    & (leg_prices["bid"] >= 0)
    & (leg_prices["ask"] >= 0)
)

leg_prices["leg_pricing_status"] = np.select(
    [
        leg_prices["leg_price_found"],
        ~leg_prices["cache_row_found"],
        leg_prices["status"].notna(),
        leg_prices["status_code"].notna(),
    ],
    [
        "priced",
        "no_cache_row",
        leg_prices["status"].astype(str),
        "status_code_" + leg_prices["status_code"].astype("Int64").astype(str),
    ],
    default="missing_price",
)

# --------------------------------------------------------------------------------------------------
# Pivot to spread-level prices
# --------------------------------------------------------------------------------------------------

short_legs = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("short_call_1sd")]
    .copy()
    .rename(columns={
        "contract_request_id": "short_contract_request_id",
        "strike_listed": "short_leg_strike_listed",
        "price_found": "short_price_found",
        "leg_price_found": "short_leg_price_found",
        "leg_pricing_status": "short_leg_pricing_status",
        "status": "short_status",
        "status_code": "short_status_code",
        "source_cache": "short_source_cache",
        "bid": "short_bid",
        "ask": "short_ask",
        "mid": "short_mid",
        "bid_ask_spread": "short_bid_ask_spread",
        "underlying_price": "short_underlying_price",
        "implied_vol": "short_implied_vol",
    })
)

long_legs = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("long_call_3sd")]
    .copy()
    .rename(columns={
        "contract_request_id": "long_contract_request_id",
        "strike_listed": "long_leg_strike_listed",
        "price_found": "long_price_found",
        "leg_price_found": "long_leg_price_found",
        "leg_pricing_status": "long_leg_pricing_status",
        "status": "long_status",
        "status_code": "long_status_code",
        "source_cache": "long_source_cache",
        "bid": "long_bid",
        "ask": "long_ask",
        "mid": "long_mid",
        "bid_ask_spread": "long_bid_ask_spread",
        "underlying_price": "long_underlying_price",
        "implied_vol": "long_implied_vol",
    })
)

short_keep_cols = [
    "selected_trade_id",
    "short_contract_request_id",
    "short_leg_strike_listed",
    "short_price_found",
    "short_leg_price_found",
    "short_leg_pricing_status",
    "short_status",
    "short_status_code",
    "short_source_cache",
    "short_bid",
    "short_ask",
    "short_mid",
    "short_bid_ask_spread",
    "short_underlying_price",
    "short_implied_vol",
]

long_keep_cols = [
    "selected_trade_id",
    "long_contract_request_id",
    "long_leg_strike_listed",
    "long_price_found",
    "long_leg_price_found",
    "long_leg_pricing_status",
    "long_status",
    "long_status_code",
    "long_source_cache",
    "long_bid",
    "long_ask",
    "long_mid",
    "long_bid_ask_spread",
    "long_underlying_price",
    "long_implied_vol",
]

short_keep_cols = [c for c in short_keep_cols if c in short_legs.columns]
long_keep_cols = [c for c in long_keep_cols if c in long_legs.columns]

if short_legs["selected_trade_id"].duplicated().any():
    raise ValueError("Duplicate short legs by selected_trade_id.")
if long_legs["selected_trade_id"].duplicated().any():
    raise ValueError("Duplicate long legs by selected_trade_id.")

spread_prices = unique_trades.merge(
    short_legs[short_keep_cols],
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices = spread_prices.merge(
    long_legs[long_keep_cols],
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_prices["short_leg_price_found"] = bool_series(spread_prices["short_leg_price_found"])
spread_prices["long_leg_price_found"] = bool_series(spread_prices["long_leg_price_found"])

spread_prices["spread_complete"] = (
    spread_prices["short_leg_price_found"]
    & spread_prices["long_leg_price_found"]
)

spread_prices["missing_short"] = ~spread_prices["short_leg_price_found"]
spread_prices["missing_long"] = ~spread_prices["long_leg_price_found"]

spread_prices["spread_pricing_status"] = np.select(
    [
        spread_prices["spread_complete"],
        spread_prices["missing_short"] & ~spread_prices["missing_long"],
        ~spread_prices["missing_short"] & spread_prices["missing_long"],
        spread_prices["missing_short"] & spread_prices["missing_long"],
    ],
    [
        "complete",
        "missing_short_only",
        "missing_long_only",
        "missing_both",
    ],
    default="unknown",
)

# --------------------------------------------------------------------------------------------------
# Compute spread P&L
# --------------------------------------------------------------------------------------------------

spread_prices["entry_credit_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_bid"] - spread_prices["long_ask"],
    np.nan,
)

spread_prices["entry_credit_mid"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["short_mid"] - spread_prices["long_mid"],
    np.nan,
)

spread_prices["credit_pct_width"] = safe_divide(
    spread_prices["entry_credit_conservative"],
    spread_prices["listed_width"],
)

spread_prices["max_loss_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["listed_width"] - spread_prices["entry_credit_conservative"],
    np.nan,
)

spread_prices["expiry_pnl_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["entry_credit_conservative"] - spread_prices["terminal_spread_intrinsic_before_premium"],
    np.nan,
)

spread_prices["return_on_max_loss_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["max_loss_conservative"],
)

spread_prices["return_on_width_conservative"] = safe_divide(
    spread_prices["expiry_pnl_conservative"],
    spread_prices["listed_width"],
)

spread_prices["trade_win_conservative"] = np.where(
    spread_prices["spread_complete"],
    spread_prices["expiry_pnl_conservative"] > 0,
    np.nan,
)

spread_prices["invalid_credit_flag"] = (
    spread_prices["spread_complete"]
    & (
        spread_prices["entry_credit_conservative"].isna()
        | (spread_prices["entry_credit_conservative"] <= 0)
        | (spread_prices["entry_credit_conservative"] >= spread_prices["listed_width"])
    )
)

spread_prices["invalid_max_loss_flag"] = (
    spread_prices["spread_complete"]
    & (
        spread_prices["max_loss_conservative"].isna()
        | (spread_prices["max_loss_conservative"] <= 0)
    )
)

# --------------------------------------------------------------------------------------------------
# Join spread prices back to rule-selected rows
# --------------------------------------------------------------------------------------------------

spread_outcome_cols = [
    "selected_trade_id",
    "spread_complete",
    "spread_pricing_status",
    "missing_short",
    "missing_long",
    "short_contract_request_id",
    "long_contract_request_id",
    "short_price_found",
    "long_price_found",
    "short_leg_price_found",
    "long_leg_price_found",
    "short_leg_pricing_status",
    "long_leg_pricing_status",
    "short_status",
    "long_status",
    "short_status_code",
    "long_status_code",
    "short_bid",
    "short_ask",
    "short_mid",
    "long_bid",
    "long_ask",
    "long_mid",
    "entry_credit_conservative",
    "entry_credit_mid",
    "credit_pct_width",
    "max_loss_conservative",
    "terminal_spread_intrinsic_before_premium",
    "expiry_pnl_conservative",
    "return_on_max_loss_conservative",
    "return_on_width_conservative",
    "trade_win_conservative",
    "invalid_credit_flag",
    "invalid_max_loss_flag",
]

spread_outcome_cols = [c for c in spread_outcome_cols if c in spread_prices.columns]

rule_trades_priced = rule_trades.merge(
    spread_prices[spread_outcome_cols],
    on="selected_trade_id",
    how="left",
    validate="many_to_one",
)

rule_trades_priced["spread_complete"] = bool_series(rule_trades_priced["spread_complete"])
rule_trades_priced["invalid_credit_flag"] = bool_series(rule_trades_priced["invalid_credit_flag"])
rule_trades_priced["invalid_max_loss_flag"] = bool_series(rule_trades_priced["invalid_max_loss_flag"])

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

global_summary = pd.DataFrame([{
    "unique_spreads": int(len(spread_prices)),
    "complete_spreads": int(spread_prices["spread_complete"].sum()),
    "complete_rate": float(spread_prices["spread_complete"].mean()),
    "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
    "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
    "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
    "invalid_credit_count": int(spread_prices["invalid_credit_flag"].sum()),
    "invalid_max_loss_count": int(spread_prices["invalid_max_loss_flag"].sum()),
    "avg_entry_credit": float(spread_prices.loc[spread_prices["spread_complete"], "entry_credit_conservative"].mean()),
    "median_entry_credit": float(spread_prices.loc[spread_prices["spread_complete"], "entry_credit_conservative"].median()),
    "avg_credit_pct_width": float(spread_prices.loc[spread_prices["spread_complete"], "credit_pct_width"].mean()),
    "median_credit_pct_width": float(spread_prices.loc[spread_prices["spread_complete"], "credit_pct_width"].median()),
    "avg_max_loss": float(spread_prices.loc[spread_prices["spread_complete"], "max_loss_conservative"].mean()),
    "win_rate": float(spread_prices.loc[spread_prices["spread_complete"], "trade_win_conservative"].mean()),
    "avg_expiry_pnl": float(spread_prices.loc[spread_prices["spread_complete"], "expiry_pnl_conservative"].mean()),
    "median_expiry_pnl": float(spread_prices.loc[spread_prices["spread_complete"], "expiry_pnl_conservative"].median()),
    "avg_return_on_max_loss": float(spread_prices.loc[spread_prices["spread_complete"], "return_on_max_loss_conservative"].mean()),
    "median_return_on_max_loss": float(spread_prices.loc[spread_prices["spread_complete"], "return_on_max_loss_conservative"].median()),
    "worst_return_on_max_loss": float(spread_prices.loc[spread_prices["spread_complete"], "return_on_max_loss_conservative"].min()),
}])

leg_price_summary = (
    leg_prices
    .groupby(["leg_label", "leg_price_found", "leg_pricing_status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["leg_label", "leg_price_found", "leg_pricing_status"])
)

spread_status_summary = (
    spread_prices
    .groupby("spread_pricing_status", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        avg_tenor=("tenor", "mean"),
        avg_width=("listed_width", "mean"),
        avg_vix_style_vol_pct=("vix_style_vol_pct", "mean"),
        avg_entry_credit=("entry_credit_conservative", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative", "mean"),
    )
    .reset_index()
    .sort_values("spread_pricing_status")
)

tenor_pricing_summary = (
    spread_prices
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_spreads=("spread_complete", "sum"),
        missing_short=("missing_short", "sum"),
        missing_long=("missing_long", "sum"),
        avg_width=("listed_width", "mean"),
        avg_entry_credit=("entry_credit_conservative", "mean"),
        win_rate=("trade_win_conservative", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative", "min"),
    )
    .reset_index()
)

tenor_pricing_summary["complete_rate"] = safe_divide(
    tenor_pricing_summary["complete_spreads"],
    tenor_pricing_summary["unique_spreads"],
)

year_pricing_summary = (
    spread_prices
    .assign(trade_year=pd.to_datetime(spread_prices["trade_date"]).dt.year)
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_spreads=("spread_complete", "sum"),
        missing_short=("missing_short", "sum"),
        missing_long=("missing_long", "sum"),
        avg_width=("listed_width", "mean"),
        avg_entry_credit=("entry_credit_conservative", "mean"),
        win_rate=("trade_win_conservative", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative", "min"),
    )
    .reset_index()
)

year_pricing_summary["complete_rate"] = safe_divide(
    year_pricing_summary["complete_spreads"],
    year_pricing_summary["unique_spreads"],
)

def summarize_rule_group(g):
    complete = g["spread_complete"].fillna(False).astype(bool)
    complete_g = g.loc[complete]

    out = {
        "selected_rows": len(g),
        "complete_rows": int(complete.sum()),
        "coverage": float(complete.mean()) if len(g) else np.nan,
        "missing_rows": int((~complete).sum()),
        "invalid_credit_rows": int(g["invalid_credit_flag"].fillna(False).sum()) if "invalid_credit_flag" in g.columns else 0,
        "invalid_max_loss_rows": int(g["invalid_max_loss_flag"].fillna(False).sum()) if "invalid_max_loss_flag" in g.columns else 0,
    }

    for c in [
        "z_threshold_shared",
        "rsi_floor",
        "vrp_log_floor",
        "vrp_log_floor_label",
        "selection_rule",
    ]:
        if c in g.columns:
            out[c] = g[c].iloc[0]

    if len(complete_g) > 0:
        out.update({
            "win_rate": float(complete_g["trade_win_conservative"].mean()),
            "avg_entry_credit": float(complete_g["entry_credit_conservative"].mean()),
            "median_entry_credit": float(complete_g["entry_credit_conservative"].median()),
            "avg_credit_pct_width": float(complete_g["credit_pct_width"].mean()),
            "avg_expiry_pnl": float(complete_g["expiry_pnl_conservative"].mean()),
            "median_expiry_pnl": float(complete_g["expiry_pnl_conservative"].median()),
            "avg_return_on_max_loss": float(complete_g["return_on_max_loss_conservative"].mean()),
            "median_return_on_max_loss": float(complete_g["return_on_max_loss_conservative"].median()),
            "worst_return_on_max_loss": float(complete_g["return_on_max_loss_conservative"].min()),
            "p05_return_on_max_loss": float(complete_g["return_on_max_loss_conservative"].quantile(0.05)),
            "p95_return_on_max_loss": float(complete_g["return_on_max_loss_conservative"].quantile(0.95)),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "avg_expiry_pnl": np.nan,
            "median_expiry_pnl": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
        })

    return pd.Series(out)

rule_priced_summary = (
    rule_trades_priced
    .groupby("rule_id", dropna=False)
    .apply(summarize_rule_group)
    .reset_index()
)

rule_year_priced_summary = (
    rule_trades_priced
    .assign(trade_year=pd.to_datetime(rule_trades_priced["trade_date"]).dt.year)
    .groupby(["rule_id", "trade_year"], dropna=False)
    .apply(summarize_rule_group)
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

LEG_PRICES_JOINED_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_leg_prices_joined_{SUFFIX}.parquet"
)

UNIQUE_TRADE_UNIVERSE_PRICED_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_{SUFFIX}.parquet"
)

RULE_SELECTED_TRADES_PRICED_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_priced_{SUFFIX}.parquet"
)

SPREAD_STATUS_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_spread_pricing_status_summary_{SUFFIX}.parquet"
)

RULE_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_priced_summary_{SUFFIX}.parquet"
)

RULE_YEAR_PRICED_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_year_priced_summary_{SUFFIX}.parquet"
)

atomic_write_parquet(leg_prices, LEG_PRICES_JOINED_PATH)
atomic_write_parquet(spread_prices, UNIQUE_TRADE_UNIVERSE_PRICED_PATH)
atomic_write_parquet(rule_trades_priced, RULE_SELECTED_TRADES_PRICED_PATH)
atomic_write_parquet(spread_status_summary, SPREAD_STATUS_SUMMARY_PATH)
atomic_write_parquet(rule_priced_summary, RULE_PRICED_SUMMARY_PATH)
atomic_write_parquet(rule_year_priced_summary, RULE_YEAR_PRICED_SUMMARY_PATH)

# Audit CSVs
global_summary_csv = AUDIT_DIR / f"02F4_global_primary_pricing_summary_{SUFFIX}_{timestamp}.csv"
leg_summary_csv = AUDIT_DIR / f"02F4_leg_primary_pricing_summary_{SUFFIX}_{timestamp}.csv"
spread_status_csv = AUDIT_DIR / f"02F4_spread_primary_pricing_status_summary_{SUFFIX}_{timestamp}.csv"
tenor_summary_csv = AUDIT_DIR / f"02F4_tenor_primary_pricing_summary_{SUFFIX}_{timestamp}.csv"
year_summary_csv = AUDIT_DIR / f"02F4_year_primary_pricing_summary_{SUFFIX}_{timestamp}.csv"
rule_summary_csv = AUDIT_DIR / f"02F4_rule_primary_priced_summary_{SUFFIX}_{timestamp}.csv"
rule_year_summary_csv = AUDIT_DIR / f"02F4_rule_year_primary_priced_summary_{SUFFIX}_{timestamp}.csv"

missing_spreads_preview_csv = AUDIT_DIR / f"02F4_missing_spreads_preview_{SUFFIX}_{timestamp}.csv"

missing_spreads_preview = (
    spread_prices
    .loc[~spread_prices["spread_complete"]]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(200)
    .copy()
)

atomic_write_csv(global_summary, global_summary_csv)
atomic_write_csv(leg_price_summary, leg_summary_csv)
atomic_write_csv(spread_status_summary, spread_status_csv)
atomic_write_csv(tenor_pricing_summary, tenor_summary_csv)
atomic_write_csv(year_pricing_summary, year_summary_csv)
atomic_write_csv(rule_priced_summary, rule_summary_csv)
atomic_write_csv(rule_year_priced_summary, rule_year_summary_csv)
atomic_write_csv(missing_spreads_preview, missing_spreads_preview_csv)

manifest_path = AUDIT_DIR / f"02F4_join_primary_prices_spread_pnl_{SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.4",
    "description": "Join repaired-RSI primary contract prices back to call spreads and compute primary spread P&L",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "leg_request_plan_path": str(LEG_REQUEST_PLAN_PATH),
        "unique_selected_trade_universe_path": str(UNIQUE_SELECTED_TRADE_UNIVERSE_PATH),
        "rule_selected_trades_unpriced_path": str(RULE_SELECTED_TRADES_UNPRICED_PATH),
        "contract_price_cache_path": str(CONTRACT_PRICE_CACHE_PATH),
    },
    "outputs": {
        "leg_prices_joined_path": str(LEG_PRICES_JOINED_PATH),
        "unique_trade_universe_priced_path": str(UNIQUE_TRADE_UNIVERSE_PRICED_PATH),
        "rule_selected_trades_priced_path": str(RULE_SELECTED_TRADES_PRICED_PATH),
        "spread_status_summary_path": str(SPREAD_STATUS_SUMMARY_PATH),
        "rule_priced_summary_path": str(RULE_PRICED_SUMMARY_PATH),
        "rule_year_priced_summary_path": str(RULE_YEAR_PRICED_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "leg_request_rows": int(len(legs)),
        "unique_spread_rows": int(len(spread_prices)),
        "rule_selected_rows": int(len(rule_trades_priced)),
        "complete_spreads": int(spread_prices["spread_complete"].sum()),
        "missing_short_only": int(spread_prices["spread_pricing_status"].eq("missing_short_only").sum()),
        "missing_long_only": int(spread_prices["spread_pricing_status"].eq("missing_long_only").sum()),
        "missing_both": int(spread_prices["spread_pricing_status"].eq("missing_both").sum()),
        "invalid_credit_count": int(spread_prices["invalid_credit_flag"].sum()),
        "invalid_max_loss_count": int(spread_prices["invalid_max_loss_flag"].sum()),
    },
    "notes": {
        "pricing_status": "Primary exact-contract pricing only; listed-strike fallback has not been applied.",
        "ranking_status": "No final rule ranking performed.",
        "next_step": "Run repaired-RSI fallback diagnostic to recover missing legs/spreads.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.4 complete — repaired-RSI primary spread P&L")
print("=" * 100)
display(global_summary)

print()
print("=" * 100)
print("Leg price summary")
print("=" * 100)
display(leg_price_summary)

print()
print("=" * 100)
print("Spread pricing status summary")
print("=" * 100)
display(spread_status_summary)

print()
print("=" * 100)
print("Tenor pricing summary")
print("=" * 100)
display(tenor_pricing_summary)

print()
print("=" * 100)
print("Year pricing summary")
print("=" * 100)
display(year_pricing_summary)

print()
print("=" * 100)
print("Rule priced summary preview — NOT FINAL RANKING")
print("=" * 100)
display(
    rule_priced_summary
    .sort_values(["complete_rows", "avg_return_on_max_loss"], ascending=[False, False])
    .head(40)
)

print()
print("=" * 100)
print("Missing spread preview")
print("=" * 100)
if missing_spreads_preview.empty:
    print("No missing spreads.")
else:
    display(missing_spreads_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Leg prices joined:          {LEG_PRICES_JOINED_PATH}")
print(f"Unique spreads priced:      {UNIQUE_TRADE_UNIVERSE_PRICED_PATH}")
print(f"Rule selected priced:       {RULE_SELECTED_TRADES_PRICED_PATH}")
print(f"Spread status summary:      {SPREAD_STATUS_SUMMARY_PATH}")
print(f"Rule priced summary:        {RULE_PRICED_SUMMARY_PATH}")
print(f"Rule-year priced summary:   {RULE_YEAR_PRICED_SUMMARY_PATH}")
print(f"Manifest:                   {manifest_path}")

print()
print("Result: 2F.4 repaired-RSI primary spread P&L complete.")
print("Next step: 2F.5 repaired-RSI listed-strike fallback diagnostic.")

Cell 2F.4 — Join repaired-RSI primary contract prices back to spreads and compute spread P&L
Leg request plan:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_request_plan_repaired_rsi_long5_cal_v1.parquet
Unique trade universe:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_repaired_rsi_long5_cal_v1.parquet
Rule selected unpriced:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_repaired_rsi_long5_cal_v1.parquet
Contract price cache:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet

Loaded inputs
Leg request rows:        3,468
Unique spread rows:      1,734
Rule selected rows:      155,120
Contract cache rows:     3,423



C:\Users\patri\AppData\Local\Temp\ipykernel_14668\764429813.py:704: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_rule_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\764429813.py:712: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_rule_group)



Cell 2F.4 complete — repaired-RSI primary spread P&L


,unique_spreads,complete_spreads,complete_rate,missing_short_only,missing_long_only,missing_both,invalid_credit_count,invalid_max_loss_count,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,avg_max_loss,win_rate,avg_expiry_pnl,median_expiry_pnl,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss
0,1734,1178,0.679354,426,53,77,0,0,0.750942,0.72,0.023827,0.022778,32.889975,0.862479,0.072818,0.65,0.001681,0.019908,-0.516209



Leg price summary


...


Rule priced summary preview — NOT FINAL RANKING


,rule_id,selected_rows,complete_rows,coverage,missing_rows,invalid_credit_rows,invalid_max_loss_rows,z_threshold_shared,rsi_floor,vrp_log_floor,...,avg_entry_credit,median_entry_credit,avg_credit_pct_width,avg_expiry_pnl,median_expiry_pnl,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,p95_return_on_max_loss
724,Zm0p25_RSI55_VRP0p00_SEL_shortest_tenor,453,399,0.880795,54,0,0,-0.25,55,0.0,...,0.677820,0.650,0.026490,0.012682,0.540,0.001762,0.021314,-0.516209,-0.177819,0.049462
739,Zm0p25_RSI55_VRPNONE_SEL_shortest_tenor,453,399,0.880795,54,0,0,-0.25,55,NaN,...,0.677820,0.650,0.026490,0.012682,0.540,0.001762,0.021314,-0.516209,-0.177819,0.049462
729,Zm0p25_RSI55_VRP0p20_SEL_shortest_tenor,452,398,0.880531,54,0,0,-0.25,55,0.2,...,0.676759,0.645,0.026482,0.009950,0.540,0.001689,0.021314,-0.516209,-0.177963,0.049502
734,Zm0p25_RSI55_VRP0p40_SEL_shortest_tenor,408,362,0.887255,46,0,0,-0.25,55,0.4,...,0.674503,0.630,0.026307,0.025967,0.535,0.003038,0.021396,-0.372422,-0.177126,0.049329
744,Zm0p25_RSI58_VRP0p00_SEL_shortest_tenor,385,337,0.875325,48,0,0,-0.25,58,0.0,...,0.650059,0.620,0.026341,0.071484,0.520,0.004214,0.021566,-0.372422,-0.165501,0.049541
759,Zm0p25_RSI58_VRPNONE_SEL_shortest_tenor,385,337,0.875325,48,0,0,-0.25,58,NaN,...,0.650059,0.620,0.026341,0.071484,0.520,0.004214,0.021566,-0.372422,-0.165501,0.049541
749,Zm0p25_RSI58_VRP0p20_SEL_shortest_tenor,384,336,0.875000,48,0,0,-0.25,58,0.2,...,0.648720,0.620,0.026331,0.068423,0.520,0.004135,0.021536,-0.372422,-0.165747,0.049581
720,Zm0p25_RSI55_VRP0p00_SEL_highest_vrp_log,453,335,0.739514,118,0,0,-0.25,55,0.0,...,0.711015,0.680,0.025652,0.070149,0.590,0.000677,0.021314,-0.516209,-0.158463,0.043653
735,Zm0p25_RSI55_VRPNONE_SEL_highest_vrp_log,453,335,0.739514,118,0,0,-0.25,55,NaN,...,0.711015,0.680,0.025652,0.070149,0.590,0.000677,0.021314,-0.516209,-0.158463,0.043653
725,Zm0p25_RSI55_VRP0p20_SEL_highest_vrp_log,452,334,0.738938,118,0,0,-0.25,55,0.2,...,0.709850,0.680,0.025640,0.067066,0.590,0.000587,0.021314,-0.516209,-0.158895,0.043660



Missing spread preview


,selected_trade_id,trade_date,trade_date_str,trade_year,tenor,tenor_bucket,expiration_date,expiration_str,expiration_yyyymmdd,effective_dte,...,entry_credit_conservative,entry_credit_mid,credit_pct_width,max_loss_conservative,expiry_pnl_conservative,return_on_max_loss_conservative,return_on_width_conservative,trade_win_conservative,invalid_credit_flag,invalid_max_loss_flag
2,SPY_CALL_20201231_T33_EXP20210205_S400.0_L450.0,2020-12-31,2020-12-31,2020,33,Back,2021-02-05,2021-02-05,20210205,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
41,SPY_CALL_20210617_T9_EXP20210702_S433.0_L455.0,2021-06-17,2021-06-17,2021,9,Front,2021-07-02,2021-07-02,20210702,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
42,SPY_CALL_20210617_T12_EXP20210702_S435.0_L460.0,2021-06-17,2021-06-17,2021,12,Front,2021-07-02,2021-07-02,20210702,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
71,SPY_CALL_20211026_T33_EXP20211203_S479.0_L525.0,2021-10-26,2021-10-26,2021,33,Back,2021-12-03,2021-12-03,20211203,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
81,SPY_CALL_20211102_T21_EXP20211126_S479.0_L515.0,2021-11-02,2021-11-02,2021,21,Middle,2021-11-26,2021-11-26,20211126,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
973,SPY_CALL_20240822_T33_EXP20240927_S586.0_L645.0,2024-08-22,2024-08-22,2024,33,Back,2024-09-27,2024-09-27,20240927,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
975,SPY_CALL_20240823_T27_EXP20240920_S587.0_L635.0,2024-08-23,2024-08-23,2024,27,Back,2024-09-20,2024-09-20,20240920,28.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
976,SPY_CALL_20240823_T30_EXP20240927_S588.0_L640.0,2024-08-23,2024-08-23,2024,30,Back,2024-09-27,2024-09-27,20240927,35.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
977,SPY_CALL_20240823_T33_EXP20240927_S589.0_L640.0,2024-08-23,2024-08-23,2024,33,Back,2024-09-27,2024-09-27,20240927,35.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False



Saved
Leg prices joined:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_leg_prices_joined_repaired_rsi_long5_cal_v1.parquet
Unique spreads priced:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_repaired_rsi_long5_cal_v1.parquet
Rule selected priced:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_repaired_rsi_long5_cal_v1.parquet
Spread status summary:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_spread_pricing_status_summary_repaired_rsi_long5_cal_v1.parquet
Rule priced summary:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_priced_summary_repaired_rsi_long5_cal_v1.parquet
Rule-year priced summary:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_c

In [6]:
# Cell 2F.5 — Repaired-RSI listed-strike fallback diagnostic
# Purpose:
#   Recover incomplete repaired-RSI call spreads by searching nearby listed strikes.
#
# This cell:
#   - Loads repaired-RSI primary-priced unique spread universe from 2F.4
#   - Loads repaired-RSI primary contract price cache from 2E.3
#   - Generates fallback candidate contracts for missing short/long legs
#   - Reuses old fallback candidate cache where exact candidate contract IDs overlap
#   - Queries ThetaData only for not-yet-cached fallback candidates
#   - Chooses effective fallback legs using conservative strike rules
#   - Recomputes spread P&L with effective strikes/prices
#   - Saves fallback candidate cache, fallback map, and spread universe priced with fallback
#
# Fallback convention:
#   - Missing short call:
#       choose nearest priced listed call strike at or above target,
#       while remaining below the long strike.
#       If no at/above candidate exists, allow lower-strike diagnostic fallback.
#   - Missing long call:
#       choose nearest priced listed call strike at or below target,
#       while remaining above the effective short strike.
#       If no at/below candidate exists, allow higher-strike diagnostic fallback.
#
# This cell does NOT:
#   - Rank final rules
#   - Select final parameters
#   - Apply Hybrid v2 put-spread sizing/selector

from pathlib import Path
from datetime import datetime, timezone
import json
import time

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

PRIMARY_PRICED_SPREADS_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_{SUFFIX}.parquet"
)

PRIMARY_CONTRACT_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selected_contract_price_cache_{SUFFIX}.parquet"
)

FALLBACK_CANDIDATES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_missing_contract_fallback_candidates_{SUFFIX}.parquet"
)

FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_fallback_candidate_contract_price_cache_{SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_missing_contract_fallback_map_{SUFFIX}.parquet"
)

SPREADS_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_{SUFFIX}.parquet"
)

# Old provisional fallback cache for exact candidate-contract reuse.
OLD_FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet"
)

THETADATA_BASE_URL = "http://127.0.0.1:25503"

# Primary endpoint used previously.
THETADATA_URLS = [
    f"{THETADATA_BASE_URL}/v3/option/history/greeks/eod",
]

# Set to None to attempt all remaining uncached fallback candidates.
# Set to e.g. 250 for a test batch.
MAX_REQUESTS_THIS_RUN = None

REQUEST_TIMEOUT_SECONDS = 20
REQUEST_SLEEP_SECONDS = 0.03
SAVE_EVERY_N_REQUESTS = 25

SHORT_OFFSETS = [0, 1, 2, 3, 4, 5, -1, -2, -3, -4, -5]
LONG_OFFSETS = [0, -5, 5, -10, 10, -15, 15]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F.5 — Repaired-RSI listed-strike fallback diagnostic")
print("=" * 100)
print(f"Primary priced spreads:       {PRIMARY_PRICED_SPREADS_PATH}")
print(f"Primary contract cache:       {PRIMARY_CONTRACT_CACHE_PATH}")
print(f"Fallback candidate cache out: {FALLBACK_CANDIDATE_CACHE_PATH}")
print(f"Old fallback cache reuse:     {OLD_FALLBACK_CANDIDATE_CACHE_PATH}")
print(f"Max requests this run:        {MAX_REQUESTS_THIS_RUN}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def yyyymmdd(x):
    return pd.Timestamp(x).strftime("%Y%m%d")

def theta_strike_value(strike):
    return int(round(float(strike) * 1000))

def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index)
    if isinstance(s, bool):
        return pd.Series(s, index=index)
    if getattr(s, "dtype", None) == bool:
        return s.fillna(False)
    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
    )

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def make_contract_request_id(symbol, trade_date, expiration_date, strike, right):
    return (
        f"{symbol}_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_"
        f"{right}"
    )

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def parse_thetadata_payload_to_df(payload):
    if not isinstance(payload, dict):
        return pd.DataFrame()

    header = payload.get("header", None)
    rows = payload.get("response", None)

    if rows is None:
        rows = payload.get("data", None)

    if isinstance(rows, dict):
        if "data" in rows:
            rows = rows.get("data")
        elif "response" in rows:
            rows = rows.get("response")

    if rows is None:
        return pd.DataFrame()

    if isinstance(header, dict):
        cols = (
            header.get("format")
            or header.get("columns")
            or header.get("fields")
            or header.get("header")
        )
    elif isinstance(header, list):
        cols = header
    else:
        cols = None

    if isinstance(rows, list) and len(rows) == 0:
        return pd.DataFrame()

    try:
        if cols is not None and isinstance(rows, list):
            df = pd.DataFrame(rows, columns=cols)
        else:
            df = pd.DataFrame(rows)
    except Exception:
        df = pd.DataFrame(rows)

    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def extract_quote_fields(df):
    if df is None or df.empty:
        return {}

    d = df.copy()
    d.columns = [str(c).strip().lower() for c in d.columns]

    if "ms_of_day" in d.columns:
        d["ms_of_day"] = pd.to_numeric(d["ms_of_day"], errors="coerce")
        d = d.sort_values("ms_of_day")

    row = d.iloc[-1]

    bid_col = first_existing_col(d, ["bid", "bid_price", "best_bid"])
    ask_col = first_existing_col(d, ["ask", "ask_price", "best_ask"])
    underlying_col = first_existing_col(d, ["underlying_price", "underlying", "underlying_close", "root_price"])
    iv_col = first_existing_col(d, ["implied_vol", "iv", "implied_volatility", "implied_volatility_mid"])
    delta_col = first_existing_col(d, ["delta"])
    gamma_col = first_existing_col(d, ["gamma"])
    theta_col = first_existing_col(d, ["theta"])
    vega_col = first_existing_col(d, ["vega"])
    rho_col = first_existing_col(d, ["rho"])

    fields = {
        "bid": pd.to_numeric(row[bid_col], errors="coerce") if bid_col else np.nan,
        "ask": pd.to_numeric(row[ask_col], errors="coerce") if ask_col else np.nan,
        "underlying_price": pd.to_numeric(row[underlying_col], errors="coerce") if underlying_col else np.nan,
        "implied_vol": pd.to_numeric(row[iv_col], errors="coerce") if iv_col else np.nan,
        "delta": pd.to_numeric(row[delta_col], errors="coerce") if delta_col else np.nan,
        "gamma": pd.to_numeric(row[gamma_col], errors="coerce") if gamma_col else np.nan,
        "theta": pd.to_numeric(row[theta_col], errors="coerce") if theta_col else np.nan,
        "vega": pd.to_numeric(row[vega_col], errors="coerce") if vega_col else np.nan,
        "rho": pd.to_numeric(row[rho_col], errors="coerce") if rho_col else np.nan,
        "raw_columns": ",".join(d.columns),
        "raw_row_count": int(len(d)),
    }

    fields["mid"] = (
        (fields["bid"] + fields["ask"]) / 2.0
        if pd.notna(fields["bid"]) and pd.notna(fields["ask"])
        else np.nan
    )

    fields["bid_ask_spread"] = (
        fields["ask"] - fields["bid"]
        if pd.notna(fields["bid"]) and pd.notna(fields["ask"])
        else np.nan
    )

    return fields

STANDARD_CACHE_COLUMNS = [
    "contract_request_id",
    "symbol",
    "right",
    "trade_date",
    "trade_date_str",
    "expiration_date",
    "expiration_str",
    "expiration_yyyymmdd",
    "strike_listed",
    "theta_strike",
    "status",
    "status_code",
    "price_found",
    "bid",
    "ask",
    "mid",
    "bid_ask_spread",
    "underlying_price",
    "implied_vol",
    "delta",
    "gamma",
    "theta",
    "vega",
    "rho",
    "raw_columns",
    "raw_row_count",
    "query_url",
    "query_params_json",
    "source_cache",
    "reused_from_cache",
    "request_timestamp_utc",
]

def normalize_cache_schema(cache_df, source_cache_name):
    if cache_df is None or cache_df.empty:
        return pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

    df = cache_df.copy()

    rename_map = {}

    aliases = {
        "contract_request_id": [
            "contract_request_id",
            "candidate_contract_request_id",
            "fallback_contract_request_id",
        ],
        "strike_listed": [
            "strike_listed",
            "candidate_strike",
            "fallback_strike",
            "strike",
        ],
        "status": [
            "status",
            "request_status",
        ],
        "status_code": [
            "status_code",
            "http_status_code",
        ],
    }

    for target, candidates in aliases.items():
        if target not in df.columns:
            src = first_existing_col(df, candidates)
            if src is not None:
                rename_map[src] = target

    if rename_map:
        df = df.rename(columns=rename_map)

    if "contract_request_id" not in df.columns:
        needed = ["symbol", "trade_date", "expiration_date", "strike_listed", "right"]
        if all(c in df.columns for c in needed):
            df["contract_request_id"] = [
                make_contract_request_id(sym, td, exp, strike, right)
                for sym, td, exp, strike, right in zip(
                    df["symbol"],
                    df["trade_date"],
                    df["expiration_date"],
                    df["strike_listed"],
                    df["right"],
                )
            ]
        else:
            raise ValueError(f"Cache {source_cache_name} has no contract_request_id and cannot reconstruct it.")

    if "price_found" not in df.columns:
        if "status" in df.columns:
            df["price_found"] = df["status"].astype(str).str.lower().eq("priced")
        elif "bid" in df.columns and "ask" in df.columns:
            df["price_found"] = df["bid"].notna() & df["ask"].notna()
        else:
            df["price_found"] = False

    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

    if "trade_date_str" not in df.columns and "trade_date" in df.columns:
        df["trade_date_str"] = pd.to_datetime(df["trade_date"]).dt.strftime("%Y-%m-%d")

    if "expiration_str" not in df.columns and "expiration_date" in df.columns:
        df["expiration_str"] = pd.to_datetime(df["expiration_date"]).dt.strftime("%Y-%m-%d")

    if "expiration_yyyymmdd" not in df.columns and "expiration_date" in df.columns:
        df["expiration_yyyymmdd"] = pd.to_datetime(df["expiration_date"]).dt.strftime("%Y%m%d")

    if "theta_strike" not in df.columns and "strike_listed" in df.columns:
        df["theta_strike"] = df["strike_listed"].apply(lambda x: theta_strike_value(x) if pd.notna(x) else np.nan)

    for c in [
        "strike_listed",
        "theta_strike",
        "status_code",
        "bid",
        "ask",
        "mid",
        "bid_ask_spread",
        "underlying_price",
        "implied_vol",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "raw_row_count",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["contract_request_id"] = df["contract_request_id"].astype(str)
    df["price_found"] = bool_series(df["price_found"], index=df.index)
    df["source_cache"] = source_cache_name
    df["reused_from_cache"] = True

    if "request_timestamp_utc" not in df.columns:
        df["request_timestamp_utc"] = pd.NaT

    for c in STANDARD_CACHE_COLUMNS:
        if c not in df.columns:
            df[c] = np.nan

    return df[STANDARD_CACHE_COLUMNS].copy()

def query_contract(row, session):
    symbol = str(row["symbol"])
    right = str(row["right"])
    trade_date = pd.Timestamp(row["trade_date"])
    expiration_date = pd.Timestamp(row["expiration_date"])
    strike = float(row["strike_listed"])
    theta_strike = theta_strike_value(strike)

    params = {
        "root": symbol,
        "exp": yyyymmdd(expiration_date),
        "right": right,
        "strike": theta_strike,
        "start_date": yyyymmdd(trade_date),
        "end_date": yyyymmdd(trade_date),
    }

    base_out = {
        "contract_request_id": row["contract_request_id"],
        "symbol": symbol,
        "right": right,
        "trade_date": trade_date,
        "trade_date_str": trade_date.strftime("%Y-%m-%d"),
        "expiration_date": expiration_date,
        "expiration_str": expiration_date.strftime("%Y-%m-%d"),
        "expiration_yyyymmdd": expiration_date.strftime("%Y%m%d"),
        "strike_listed": strike,
        "theta_strike": theta_strike,
        "status": None,
        "status_code": np.nan,
        "price_found": False,
        "bid": np.nan,
        "ask": np.nan,
        "mid": np.nan,
        "bid_ask_spread": np.nan,
        "underlying_price": np.nan,
        "implied_vol": np.nan,
        "delta": np.nan,
        "gamma": np.nan,
        "theta": np.nan,
        "vega": np.nan,
        "rho": np.nan,
        "raw_columns": "",
        "raw_row_count": 0,
        "query_url": "",
        "query_params_json": json.dumps(params, sort_keys=True),
        "source_cache": "thetadata_query",
        "reused_from_cache": False,
        "request_timestamp_utc": now_utc_iso(),
    }

    last_out = None

    for url in THETADATA_URLS:
        out = dict(base_out)
        out["query_url"] = url

        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            out["status_code"] = int(response.status_code)

            if response.status_code != 200:
                out["status"] = f"http_{response.status_code}"
                last_out = out
                continue

            try:
                payload = response.json()
            except Exception:
                out["status"] = "json_parse_error"
                last_out = out
                continue

            df = parse_thetadata_payload_to_df(payload)

            if df.empty:
                out["status"] = "no_rows"
                last_out = out
                continue

            quote_fields = extract_quote_fields(df)
            out.update(quote_fields)

            if (
                pd.notna(out["bid"])
                and pd.notna(out["ask"])
                and out["bid"] >= 0
                and out["ask"] >= 0
            ):
                out["status"] = "priced"
                out["price_found"] = True
            else:
                out["status"] = "missing_bid_ask"
                out["price_found"] = False

            return out

        except requests.exceptions.Timeout:
            out["status"] = "timeout"
            last_out = out

        except requests.exceptions.ConnectionError:
            out["status"] = "connection_error"
            last_out = out

        except Exception as e:
            out["status"] = f"exception_{type(e).__name__}"
            out["raw_columns"] = str(e)[:500]
            last_out = out

    return last_out if last_out is not None else base_out

def save_cache(cache_df):
    cache_df = cache_df.copy()

    for c in ["trade_date", "expiration_date"]:
        if c in cache_df.columns:
            cache_df[c] = pd.to_datetime(cache_df[c], errors="coerce").dt.normalize()

    if "price_found" in cache_df.columns:
        cache_df["price_found"] = bool_series(cache_df["price_found"], index=cache_df.index)

    for c in STANDARD_CACHE_COLUMNS:
        if c not in cache_df.columns:
            cache_df[c] = np.nan

    cache_df = cache_df[STANDARD_CACHE_COLUMNS].copy()
    atomic_write_parquet(cache_df, FALLBACK_CANDIDATE_CACHE_PATH)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

require_file(PRIMARY_PRICED_SPREADS_PATH)
require_file(PRIMARY_CONTRACT_CACHE_PATH)

spreads = pd.read_parquet(PRIMARY_PRICED_SPREADS_PATH).copy()
primary_cache = pd.read_parquet(PRIMARY_CONTRACT_CACHE_PATH).copy()

for c in ["trade_date", "expiration_date"]:
    if c in spreads.columns:
        spreads[c] = pd.to_datetime(spreads[c], errors="coerce").dt.normalize()
    if c in primary_cache.columns:
        primary_cache[c] = pd.to_datetime(primary_cache[c], errors="coerce").dt.normalize()

spreads["selected_trade_id"] = spreads["selected_trade_id"].astype(str)

for c in [
    "spread_complete",
    "short_leg_price_found",
    "long_leg_price_found",
    "short_price_found",
    "long_price_found",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "terminal_spread_intrinsic_before_premium",
    "short_bid",
    "short_ask",
    "short_mid",
    "long_bid",
    "long_ask",
    "long_mid",
    "vix_style_vol_pct",
]:
    if c in spreads.columns:
        spreads[c] = pd.to_numeric(spreads[c], errors="coerce")

primary_cache_norm = normalize_cache_schema(primary_cache, "primary_repaired_cache")

print("=" * 100)
print("Loaded primary-priced repaired-RSI spreads")
print("=" * 100)
print(f"Unique spread rows:       {len(spreads):,}")
print(f"Primary complete spreads: {int(spreads['spread_complete'].sum()):,}")
print(f"Incomplete spreads:       {int((~spreads['spread_complete']).sum()):,}")
print(f"Primary cache rows:       {len(primary_cache_norm):,}")
print()

# --------------------------------------------------------------------------------------------------
# Generate fallback candidates
# --------------------------------------------------------------------------------------------------

candidate_rows = []

for _, row in spreads.loc[~spreads["spread_complete"]].iterrows():
    selected_trade_id = row["selected_trade_id"]
    trade_date = row["trade_date"]
    expiration_date = row["expiration_date"]
    tenor = int(row["tenor"])
    short_target = float(row["short_strike"])
    long_target = float(row["long_strike"])

    if not bool(row.get("short_leg_price_found", False)):
        for offset in SHORT_OFFSETS:
            candidate_strike = short_target + float(offset)

            if candidate_strike <= 0:
                continue

            # Keep candidates below the original long strike to preserve spread structure.
            if candidate_strike >= long_target:
                continue

            candidate_rows.append({
                "selected_trade_id": selected_trade_id,
                "leg_label": "short_call_1sd",
                "leg_side": "sell",
                "symbol": "SPY",
                "right": "C",
                "trade_date": trade_date,
                "expiration_date": expiration_date,
                "expiration_yyyymmdd": pd.Timestamp(expiration_date).strftime("%Y%m%d"),
                "tenor": tenor,
                "target_strike": short_target,
                "other_leg_target_strike": long_target,
                "candidate_strike": candidate_strike,
                "candidate_offset": float(offset),
                "preferred_direction": "at_or_above_target",
                "preferred_side_flag": candidate_strike >= short_target,
                "abs_strike_slippage": abs(candidate_strike - short_target),
                "signed_strike_slippage": candidate_strike - short_target,
                "contract_request_id": make_contract_request_id("SPY", trade_date, expiration_date, candidate_strike, "C"),
            })

    if not bool(row.get("long_leg_price_found", False)):
        for offset in LONG_OFFSETS:
            candidate_strike = long_target + float(offset)

            if candidate_strike <= 0:
                continue

            # Keep candidates above the original short strike at candidate-generation stage.
            if candidate_strike <= short_target:
                continue

            candidate_rows.append({
                "selected_trade_id": selected_trade_id,
                "leg_label": "long_call_3sd",
                "leg_side": "buy",
                "symbol": "SPY",
                "right": "C",
                "trade_date": trade_date,
                "expiration_date": expiration_date,
                "expiration_yyyymmdd": pd.Timestamp(expiration_date).strftime("%Y%m%d"),
                "tenor": tenor,
                "target_strike": long_target,
                "other_leg_target_strike": short_target,
                "candidate_strike": candidate_strike,
                "candidate_offset": float(offset),
                "preferred_direction": "at_or_below_target",
                "preferred_side_flag": candidate_strike <= long_target,
                "abs_strike_slippage": abs(candidate_strike - long_target),
                "signed_strike_slippage": candidate_strike - long_target,
                "contract_request_id": make_contract_request_id("SPY", trade_date, expiration_date, candidate_strike, "C"),
            })

fallback_candidates = pd.DataFrame(candidate_rows)

if fallback_candidates.empty:
    print("No fallback candidates needed; all spreads are already complete.")
else:
    fallback_candidates["trade_date_str"] = pd.to_datetime(fallback_candidates["trade_date"]).dt.strftime("%Y-%m-%d")
    fallback_candidates["expiration_str"] = pd.to_datetime(fallback_candidates["expiration_date"]).dt.strftime("%Y-%m-%d")
    fallback_candidates["strike_listed"] = fallback_candidates["candidate_strike"]
    fallback_candidates["theta_strike"] = fallback_candidates["strike_listed"].apply(theta_strike_value)

fallback_unique_contracts = (
    fallback_candidates[
        [
            "contract_request_id",
            "symbol",
            "right",
            "trade_date",
            "trade_date_str",
            "expiration_date",
            "expiration_str",
            "expiration_yyyymmdd",
            "strike_listed",
            "theta_strike",
        ]
    ]
    .drop_duplicates("contract_request_id")
    .sort_values(["trade_date", "expiration_date", "strike_listed"])
    .reset_index(drop=True)
    if not fallback_candidates.empty
    else pd.DataFrame(columns=[
        "contract_request_id",
        "symbol",
        "right",
        "trade_date",
        "trade_date_str",
        "expiration_date",
        "expiration_str",
        "expiration_yyyymmdd",
        "strike_listed",
        "theta_strike",
    ])
)

missing_leg_summary = (
    pd.DataFrame([
        {
            "leg_label": "short_call_1sd",
            "missing_legs": int((~spreads["short_leg_price_found"]).sum()),
        },
        {
            "leg_label": "long_call_3sd",
            "missing_legs": int((~spreads["long_leg_price_found"]).sum()),
        },
    ])
)

fallback_candidate_summary = (
    fallback_candidates
    .groupby("leg_label", dropna=False)
    .agg(
        fallback_candidate_rows=("contract_request_id", "size"),
        unique_candidate_contracts=("contract_request_id", "nunique"),
        missing_spreads=("selected_trade_id", "nunique"),
    )
    .reset_index()
    if not fallback_candidates.empty
    else pd.DataFrame()
)

print("=" * 100)
print("Generated fallback candidates")
print("=" * 100)
display(missing_leg_summary)
print(f"Fallback candidate rows:        {len(fallback_candidates):,}")
print(f"Unique fallback contracts:      {len(fallback_unique_contracts):,}")
if not fallback_candidate_summary.empty:
    display(fallback_candidate_summary)
print()

atomic_write_parquet(fallback_candidates, FALLBACK_CANDIDATES_PATH)

# --------------------------------------------------------------------------------------------------
# Load/reuse fallback candidate caches
# --------------------------------------------------------------------------------------------------

cache_frames = []

# Existing repaired fallback cache.
if FALLBACK_CANDIDATE_CACHE_PATH.exists():
    existing_fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_CACHE_PATH)
    existing_norm = normalize_cache_schema(existing_fallback_cache, "existing_repaired_fallback_cache")
    cache_frames.append(existing_norm)
    print(f"Existing repaired fallback cache rows loaded: {len(existing_norm):,}")
else:
    print("Existing repaired fallback cache rows loaded: 0")

# Primary cache may already contain some candidate contracts.
primary_candidate_overlap = primary_cache_norm.loc[
    primary_cache_norm["contract_request_id"].isin(set(fallback_unique_contracts["contract_request_id"]))
].copy()
primary_candidate_overlap["source_cache"] = "primary_repaired_cache_overlap"
primary_candidate_overlap["reused_from_cache"] = True
cache_frames.append(primary_candidate_overlap)
print(f"Primary-cache overlap candidate rows loaded: {len(primary_candidate_overlap):,}")

# Old provisional fallback cache.
if OLD_FALLBACK_CANDIDATE_CACHE_PATH.exists():
    old_fallback_cache = pd.read_parquet(OLD_FALLBACK_CANDIDATE_CACHE_PATH)
    old_fallback_norm = normalize_cache_schema(old_fallback_cache, "old_long5_cal_fallback_cache")
    old_fallback_norm = old_fallback_norm.loc[
        old_fallback_norm["contract_request_id"].isin(set(fallback_unique_contracts["contract_request_id"]))
    ].copy()
    cache_frames.append(old_fallback_norm)
    print(f"Old fallback-cache overlap candidate rows loaded: {len(old_fallback_norm):,}")
else:
    print("Old fallback-cache overlap candidate rows loaded: 0")

if cache_frames:
    fallback_cache = pd.concat(cache_frames, ignore_index=True)
else:
    fallback_cache = pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

source_priority = {
    "existing_repaired_fallback_cache": 0,
    "primary_repaired_cache_overlap": 1,
    "old_long5_cal_fallback_cache": 2,
    "thetadata_query": 3,
}

if not fallback_cache.empty:
    fallback_cache["source_priority"] = fallback_cache["source_cache"].map(source_priority).fillna(9).astype(int)
    fallback_cache["price_found_sort"] = bool_series(fallback_cache["price_found"], index=fallback_cache.index).astype(int)

    fallback_cache = (
        fallback_cache
        .sort_values(
            ["contract_request_id", "price_found_sort", "source_priority"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
        .reset_index(drop=True)
    )

    fallback_cache = fallback_cache.loc[
        fallback_cache["contract_request_id"].isin(set(fallback_unique_contracts["contract_request_id"]))
    ].copy()

    save_cache(fallback_cache)

cached_ids = set(fallback_cache["contract_request_id"].astype(str)) if not fallback_cache.empty else set()
remaining = fallback_unique_contracts.loc[
    ~fallback_unique_contracts["contract_request_id"].astype(str).isin(cached_ids)
].copy()

if MAX_REQUESTS_THIS_RUN is not None:
    run_batch = remaining.head(int(MAX_REQUESTS_THIS_RUN)).copy()
else:
    run_batch = remaining.copy()

print()
print("=" * 100)
print("Fallback cache / remaining summary before ThetaData requests")
print("=" * 100)
print(f"Unique fallback contracts:      {len(fallback_unique_contracts):,}")
print(f"Fallback cache rows after reuse:{len(fallback_cache):,}")
print(f"Remaining uncached rows:        {len(remaining):,}")
print(f"Rows to query this run:         {len(run_batch):,}")

if not fallback_cache.empty:
    pre_cache_summary = (
        fallback_cache
        .groupby(["source_cache", "status", "price_found"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["source_cache", "status", "price_found"])
    )
    display(pre_cache_summary)

print()

# --------------------------------------------------------------------------------------------------
# Query remaining fallback candidates
# --------------------------------------------------------------------------------------------------

new_rows = []

if len(run_batch) > 0:
    session = requests.Session()

    print("=" * 100)
    print("Querying ThetaData for fallback candidates")
    print("=" * 100)

    for i, (_, row) in enumerate(run_batch.iterrows(), start=1):
        result = query_contract(row, session)
        new_rows.append(result)

        if i <= 5 or i % 50 == 0 or i == len(run_batch):
            print(
                f"{i:>6,}/{len(run_batch):,} "
                f"{result['contract_request_id']} "
                f"status={result['status']} "
                f"price_found={result['price_found']}"
            )

        if SAVE_EVERY_N_REQUESTS and i % SAVE_EVERY_N_REQUESTS == 0:
            new_df_partial = normalize_cache_schema(pd.DataFrame(new_rows), "thetadata_query")
            combined_partial = pd.concat([fallback_cache, new_df_partial], ignore_index=True)

            combined_partial["source_priority"] = combined_partial["source_cache"].map(source_priority).fillna(9).astype(int)
            combined_partial["price_found_sort"] = bool_series(combined_partial["price_found"], index=combined_partial.index).astype(int)

            combined_partial = (
                combined_partial
                .sort_values(
                    ["contract_request_id", "price_found_sort", "source_priority"],
                    ascending=[True, False, True],
                    kind="mergesort",
                )
                .drop_duplicates("contract_request_id", keep="first")
                .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
                .reset_index(drop=True)
            )

            save_cache(combined_partial)

        time.sleep(REQUEST_SLEEP_SECONDS)

    new_df = normalize_cache_schema(pd.DataFrame(new_rows), "thetadata_query")

    fallback_cache = pd.concat([fallback_cache, new_df], ignore_index=True)

    fallback_cache["source_priority"] = fallback_cache["source_cache"].map(source_priority).fillna(9).astype(int)
    fallback_cache["price_found_sort"] = bool_series(fallback_cache["price_found"], index=fallback_cache.index).astype(int)

    fallback_cache = (
        fallback_cache
        .sort_values(
            ["contract_request_id", "price_found_sort", "source_priority"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["source_priority", "price_found_sort"], errors="ignore")
        .reset_index(drop=True)
    )

    save_cache(fallback_cache)

else:
    print("=" * 100)
    print("No new ThetaData fallback requests needed.")
    print("=" * 100)
    new_df = pd.DataFrame(columns=STANDARD_CACHE_COLUMNS)

# Reload final fallback cache.
fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_CACHE_PATH).copy()
fallback_cache = normalize_cache_schema(fallback_cache, "final_repaired_fallback_cache")

# --------------------------------------------------------------------------------------------------
# Join fallback candidate prices
# --------------------------------------------------------------------------------------------------

candidate_price_cols = [
    "contract_request_id",
    "source_cache",
    "status",
    "status_code",
    "price_found",
    "bid",
    "ask",
    "mid",
    "bid_ask_spread",
    "underlying_price",
    "implied_vol",
]

fallback_candidates_priced = fallback_candidates.merge(
    fallback_cache[candidate_price_cols],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

fallback_candidates_priced["price_found"] = bool_series(
    fallback_candidates_priced["price_found"],
    index=fallback_candidates_priced.index,
)

fallback_candidates_priced["candidate_price_found"] = (
    fallback_candidates_priced["price_found"]
    & fallback_candidates_priced["bid"].notna()
    & fallback_candidates_priced["ask"].notna()
    & (fallback_candidates_priced["bid"] >= 0)
    & (fallback_candidates_priced["ask"] >= 0)
)

# --------------------------------------------------------------------------------------------------
# Choose fallbacks and rebuild effective spread prices
# --------------------------------------------------------------------------------------------------

candidate_groups = {
    (sid, leg): g.copy()
    for (sid, leg), g in fallback_candidates_priced.groupby(["selected_trade_id", "leg_label"], dropna=False)
}

effective_rows = []
fallback_map_rows = []

for _, row in spreads.iterrows():
    out = row.to_dict()

    sid = row["selected_trade_id"]

    # Original leg status/prices.
    short_found_primary = bool(row.get("short_leg_price_found", False))
    long_found_primary = bool(row.get("long_leg_price_found", False))

    short_effective_strike = row["short_strike"] if short_found_primary else np.nan
    short_effective_bid = row["short_bid"] if short_found_primary else np.nan
    short_effective_ask = row["short_ask"] if short_found_primary else np.nan
    short_effective_mid = row["short_mid"] if short_found_primary else np.nan
    short_effective_source = "primary" if short_found_primary else None
    short_fallback_found = False
    short_fallback_choice_type = None
    short_signed_slippage = np.nan
    short_abs_slippage = np.nan
    short_contract_effective = row.get("short_contract_request_id", None) if short_found_primary else None

    long_effective_strike = row["long_strike"] if long_found_primary else np.nan
    long_effective_bid = row["long_bid"] if long_found_primary else np.nan
    long_effective_ask = row["long_ask"] if long_found_primary else np.nan
    long_effective_mid = row["long_mid"] if long_found_primary else np.nan
    long_effective_source = "primary" if long_found_primary else None
    long_fallback_found = False
    long_fallback_choice_type = None
    long_signed_slippage = np.nan
    long_abs_slippage = np.nan
    long_contract_effective = row.get("long_contract_request_id", None) if long_found_primary else None

    # Short fallback first, constrained below original/effective long target.
    if not short_found_primary:
        g = candidate_groups.get((sid, "short_call_1sd"), pd.DataFrame()).copy()

        if not g.empty:
            g = g.loc[g["candidate_price_found"]].copy()
            g = g.loc[g["candidate_strike"] < row["long_strike"]].copy()

            if not g.empty:
                g["choice_side_rank"] = np.where(g["candidate_strike"] >= row["short_strike"], 0, 1)
                g["choice_abs_rank"] = (g["candidate_strike"] - row["short_strike"]).abs()
                g["choice_strike_rank"] = g["candidate_strike"]

                g = g.sort_values(
                    ["choice_side_rank", "choice_abs_rank", "choice_strike_rank"],
                    ascending=[True, True, True],
                    kind="mergesort",
                )

                chosen = g.iloc[0]

                short_effective_strike = float(chosen["candidate_strike"])
                short_effective_bid = float(chosen["bid"])
                short_effective_ask = float(chosen["ask"])
                short_effective_mid = float(chosen["mid"]) if pd.notna(chosen["mid"]) else np.nan
                short_effective_source = "fallback"
                short_fallback_found = True
                short_signed_slippage = float(chosen["signed_strike_slippage"])
                short_abs_slippage = float(chosen["abs_strike_slippage"])
                short_contract_effective = chosen["contract_request_id"]
                short_fallback_choice_type = (
                    "preferred_at_or_above_target"
                    if short_effective_strike >= row["short_strike"]
                    else "diagnostic_below_target"
                )

                fallback_map_rows.append({
                    "selected_trade_id": sid,
                    "leg_label": "short_call_1sd",
                    "target_strike": row["short_strike"],
                    "effective_strike": short_effective_strike,
                    "signed_strike_slippage": short_signed_slippage,
                    "abs_strike_slippage": short_abs_slippage,
                    "choice_type": short_fallback_choice_type,
                    "contract_request_id": short_contract_effective,
                    "bid": short_effective_bid,
                    "ask": short_effective_ask,
                    "mid": short_effective_mid,
                    "source_cache": chosen.get("source_cache", None),
                    "status": chosen.get("status", None),
                    "status_code": chosen.get("status_code", None),
                })

    # Long fallback second, constrained above effective short.
    effective_short_for_constraint = short_effective_strike if pd.notna(short_effective_strike) else row["short_strike"]

    if not long_found_primary:
        g = candidate_groups.get((sid, "long_call_3sd"), pd.DataFrame()).copy()

        if not g.empty:
            g = g.loc[g["candidate_price_found"]].copy()
            g = g.loc[g["candidate_strike"] > effective_short_for_constraint].copy()

            if not g.empty:
                g["choice_side_rank"] = np.where(g["candidate_strike"] <= row["long_strike"], 0, 1)
                g["choice_abs_rank"] = (g["candidate_strike"] - row["long_strike"]).abs()

                # For long candidates, prefer at/below target. If still tied, prefer lower strike.
                g["choice_strike_rank"] = g["candidate_strike"]

                g = g.sort_values(
                    ["choice_side_rank", "choice_abs_rank", "choice_strike_rank"],
                    ascending=[True, True, True],
                    kind="mergesort",
                )

                chosen = g.iloc[0]

                long_effective_strike = float(chosen["candidate_strike"])
                long_effective_bid = float(chosen["bid"])
                long_effective_ask = float(chosen["ask"])
                long_effective_mid = float(chosen["mid"]) if pd.notna(chosen["mid"]) else np.nan
                long_effective_source = "fallback"
                long_fallback_found = True
                long_signed_slippage = float(chosen["signed_strike_slippage"])
                long_abs_slippage = float(chosen["abs_strike_slippage"])
                long_contract_effective = chosen["contract_request_id"]
                long_fallback_choice_type = (
                    "preferred_at_or_below_target"
                    if long_effective_strike <= row["long_strike"]
                    else "diagnostic_above_target"
                )

                fallback_map_rows.append({
                    "selected_trade_id": sid,
                    "leg_label": "long_call_3sd",
                    "target_strike": row["long_strike"],
                    "effective_strike": long_effective_strike,
                    "signed_strike_slippage": long_signed_slippage,
                    "abs_strike_slippage": long_abs_slippage,
                    "choice_type": long_fallback_choice_type,
                    "contract_request_id": long_contract_effective,
                    "bid": long_effective_bid,
                    "ask": long_effective_ask,
                    "mid": long_effective_mid,
                    "source_cache": chosen.get("source_cache", None),
                    "status": chosen.get("status", None),
                    "status_code": chosen.get("status_code", None),
                })

    spread_complete_after = (
        pd.notna(short_effective_strike)
        and pd.notna(long_effective_strike)
        and pd.notna(short_effective_bid)
        and pd.notna(long_effective_ask)
        and short_effective_bid >= 0
        and long_effective_ask >= 0
        and long_effective_strike > short_effective_strike
    )

    effective_width = (
        long_effective_strike - short_effective_strike
        if pd.notna(long_effective_strike) and pd.notna(short_effective_strike)
        else np.nan
    )

    terminal_intrinsic_after = (
        max(row["expiry_spy_close"] - short_effective_strike, 0.0)
        - max(row["expiry_spy_close"] - long_effective_strike, 0.0)
        if spread_complete_after and pd.notna(row["expiry_spy_close"])
        else np.nan
    )

    entry_credit_after = (
        short_effective_bid - long_effective_ask
        if spread_complete_after
        else np.nan
    )

    max_loss_after = (
        effective_width - entry_credit_after
        if spread_complete_after
        else np.nan
    )

    expiry_pnl_after = (
        entry_credit_after - terminal_intrinsic_after
        if spread_complete_after
        else np.nan
    )

    return_on_max_loss_after = (
        expiry_pnl_after / max_loss_after
        if spread_complete_after and pd.notna(max_loss_after) and max_loss_after != 0
        else np.nan
    )

    credit_pct_width_after = (
        entry_credit_after / effective_width
        if spread_complete_after and pd.notna(effective_width) and effective_width != 0
        else np.nan
    )

    out.update({
        "short_effective_strike": short_effective_strike,
        "long_effective_strike": long_effective_strike,
        "short_effective_bid": short_effective_bid,
        "short_effective_ask": short_effective_ask,
        "short_effective_mid": short_effective_mid,
        "long_effective_bid": long_effective_bid,
        "long_effective_ask": long_effective_ask,
        "long_effective_mid": long_effective_mid,
        "short_effective_source": short_effective_source,
        "long_effective_source": long_effective_source,
        "short_fallback_found": short_fallback_found,
        "long_fallback_found": long_fallback_found,
        "short_fallback_choice_type": short_fallback_choice_type,
        "long_fallback_choice_type": long_fallback_choice_type,
        "short_signed_strike_slippage": short_signed_slippage,
        "long_signed_strike_slippage": long_signed_slippage,
        "short_abs_strike_slippage": short_abs_slippage,
        "long_abs_strike_slippage": long_abs_slippage,
        "short_effective_contract_request_id": short_contract_effective,
        "long_effective_contract_request_id": long_contract_effective,
        "spread_complete_after_fallback": bool(spread_complete_after),
        "newly_recovered_by_fallback": bool((not bool(row["spread_complete"])) and spread_complete_after),
        "still_incomplete_after_fallback": bool(not spread_complete_after),
        "effective_width_after_fallback": effective_width,
        "terminal_spread_intrinsic_after_fallback": terminal_intrinsic_after,
        "entry_credit_conservative_after_fallback": entry_credit_after,
        "credit_pct_width_after_fallback": credit_pct_width_after,
        "max_loss_conservative_after_fallback": max_loss_after,
        "expiry_pnl_conservative_after_fallback": expiry_pnl_after,
        "return_on_max_loss_conservative_after_fallback": return_on_max_loss_after,
        "return_on_width_conservative_after_fallback": (
            expiry_pnl_after / effective_width
            if spread_complete_after and pd.notna(effective_width) and effective_width != 0
            else np.nan
        ),
        "trade_win_conservative_after_fallback": (
            expiry_pnl_after > 0
            if spread_complete_after and pd.notna(expiry_pnl_after)
            else np.nan
        ),
    })

    effective_rows.append(out)

spreads_after = pd.DataFrame(effective_rows)
fallback_map = pd.DataFrame(fallback_map_rows)

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

complete_before = bool_series(spreads_after["spread_complete"], index=spreads_after.index)
complete_after = bool_series(spreads_after["spread_complete_after_fallback"], index=spreads_after.index)

recovery_summary = pd.DataFrame([{
    "unique_spreads_total": int(len(spreads_after)),
    "complete_before_fallback": int(complete_before.sum()),
    "complete_after_fallback": int(complete_after.sum()),
    "newly_recovered_by_fallback": int(spreads_after["newly_recovered_by_fallback"].sum()),
    "still_incomplete_after_fallback": int(spreads_after["still_incomplete_after_fallback"].sum()),
    "complete_rate_before": float(complete_before.mean()),
    "complete_rate_after": float(complete_after.mean()),
    "avg_entry_credit_after_fallback": float(spreads_after.loc[complete_after, "entry_credit_conservative_after_fallback"].mean()),
    "median_entry_credit_after_fallback": float(spreads_after.loc[complete_after, "entry_credit_conservative_after_fallback"].median()),
    "avg_credit_pct_width_after_fallback": float(spreads_after.loc[complete_after, "credit_pct_width_after_fallback"].mean()),
    "win_rate_after_fallback": float(spreads_after.loc[complete_after, "trade_win_conservative_after_fallback"].mean()),
    "avg_return_on_max_loss_after_fallback": float(spreads_after.loc[complete_after, "return_on_max_loss_conservative_after_fallback"].mean()),
    "median_return_on_max_loss_after_fallback": float(spreads_after.loc[complete_after, "return_on_max_loss_conservative_after_fallback"].median()),
    "worst_return_on_max_loss_after_fallback": float(spreads_after.loc[complete_after, "return_on_max_loss_conservative_after_fallback"].min()),
}])

fallback_leg_summary = (
    fallback_map
    .groupby(["leg_label", "choice_type"], dropna=False)
    .agg(
        found_rows=("selected_trade_id", "size"),
        avg_abs_slippage=("abs_strike_slippage", "mean"),
        median_abs_slippage=("abs_strike_slippage", "median"),
        avg_signed_slippage=("signed_strike_slippage", "mean"),
    )
    .reset_index()
    if not fallback_map.empty
    else pd.DataFrame()
)

candidate_price_summary = (
    fallback_candidates_priced
    .groupby(["leg_label", "candidate_price_found", "status"], dropna=False)
    .agg(
        rows=("contract_request_id", "size"),
        unique_contracts=("contract_request_id", "nunique"),
    )
    .reset_index()
    .sort_values(["leg_label", "candidate_price_found", "status"])
    if not fallback_candidates_priced.empty
    else pd.DataFrame()
)

recovery_by_tenor = (
    spreads_after
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

recovery_by_tenor["complete_rate_before"] = safe_divide(
    recovery_by_tenor["complete_before"],
    recovery_by_tenor["unique_spreads"],
)

recovery_by_tenor["complete_rate_after"] = safe_divide(
    recovery_by_tenor["complete_after"],
    recovery_by_tenor["unique_spreads"],
)

spreads_after["trade_year"] = pd.to_datetime(spreads_after["trade_date"]).dt.year

recovery_by_year = (
    spreads_after
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

recovery_by_year["complete_rate_before"] = safe_divide(
    recovery_by_year["complete_before"],
    recovery_by_year["unique_spreads"],
)

recovery_by_year["complete_rate_after"] = safe_divide(
    recovery_by_year["complete_after"],
    recovery_by_year["unique_spreads"],
)

still_incomplete_preview = (
    spreads_after
    .loc[spreads_after["still_incomplete_after_fallback"]]
    .sort_values(["trade_date", "tenor"])
    .head(100)
    .copy()
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

atomic_write_parquet(fallback_candidates, FALLBACK_CANDIDATES_PATH)
atomic_write_parquet(fallback_cache, FALLBACK_CANDIDATE_CACHE_PATH)
atomic_write_parquet(fallback_map, FALLBACK_MAP_PATH)
atomic_write_parquet(spreads_after, SPREADS_WITH_FALLBACK_PATH)

recovery_summary_csv = AUDIT_DIR / f"02F5_recovery_summary_{SUFFIX}_{timestamp}.csv"
fallback_leg_summary_csv = AUDIT_DIR / f"02F5_fallback_leg_summary_{SUFFIX}_{timestamp}.csv"
candidate_price_summary_csv = AUDIT_DIR / f"02F5_candidate_price_summary_{SUFFIX}_{timestamp}.csv"
recovery_by_tenor_csv = AUDIT_DIR / f"02F5_recovery_by_tenor_{SUFFIX}_{timestamp}.csv"
recovery_by_year_csv = AUDIT_DIR / f"02F5_recovery_by_year_{SUFFIX}_{timestamp}.csv"
still_incomplete_preview_csv = AUDIT_DIR / f"02F5_still_incomplete_preview_{SUFFIX}_{timestamp}.csv"
new_query_audit_csv = AUDIT_DIR / f"02F5_new_fallback_query_audit_{SUFFIX}_{timestamp}.csv"

atomic_write_csv(recovery_summary, recovery_summary_csv)
atomic_write_csv(fallback_leg_summary, fallback_leg_summary_csv)
atomic_write_csv(candidate_price_summary, candidate_price_summary_csv)
atomic_write_csv(recovery_by_tenor, recovery_by_tenor_csv)
atomic_write_csv(recovery_by_year, recovery_by_year_csv)
atomic_write_csv(still_incomplete_preview, still_incomplete_preview_csv)
atomic_write_csv(new_df, new_query_audit_csv)

manifest_path = AUDIT_DIR / f"02F5_fallback_diagnostic_{SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.5",
    "description": "Repaired-RSI listed-strike fallback diagnostic for incomplete call-sleeve spreads",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "primary_priced_spreads_path": str(PRIMARY_PRICED_SPREADS_PATH),
        "primary_contract_cache_path": str(PRIMARY_CONTRACT_CACHE_PATH),
        "old_fallback_candidate_cache_path": str(OLD_FALLBACK_CANDIDATE_CACHE_PATH),
    },
    "outputs": {
        "fallback_candidates_path": str(FALLBACK_CANDIDATES_PATH),
        "fallback_candidate_cache_path": str(FALLBACK_CANDIDATE_CACHE_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
        "spreads_with_fallback_path": str(SPREADS_WITH_FALLBACK_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "unique_spreads_total": int(len(spreads_after)),
        "complete_before_fallback": int(complete_before.sum()),
        "complete_after_fallback": int(complete_after.sum()),
        "newly_recovered_by_fallback": int(spreads_after["newly_recovered_by_fallback"].sum()),
        "still_incomplete_after_fallback": int(spreads_after["still_incomplete_after_fallback"].sum()),
        "fallback_candidate_rows": int(len(fallback_candidates)),
        "unique_fallback_contracts": int(len(fallback_unique_contracts)),
        "fallback_cache_rows": int(len(fallback_cache)),
        "new_query_attempts": int(len(new_df)),
    },
    "fallback_convention": {
        "short_call": "nearest priced listed strike at or above target, below long strike; lower diagnostic fallback only if no preferred candidate",
        "long_call": "nearest priced listed strike at or below target, above effective short strike; higher diagnostic fallback only if no preferred candidate",
    },
    "notes": {
        "ranking_status": "No final rule ranking performed.",
        "next_step": "Build repaired-RSI fallback-adjusted rule outcome tables.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.5 complete — repaired-RSI fallback diagnostic")
print("=" * 100)
display(recovery_summary)

print()
print("=" * 100)
print("Fallback candidate price summary")
print("=" * 100)
display(candidate_price_summary)

print()
print("=" * 100)
print("Fallback leg summary")
print("=" * 100)
if fallback_leg_summary.empty:
    print("No fallback legs selected.")
else:
    display(fallback_leg_summary)

print()
print("=" * 100)
print("Recovery by tenor")
print("=" * 100)
display(recovery_by_tenor)

print()
print("=" * 100)
print("Recovery by year")
print("=" * 100)
display(recovery_by_year)

print()
print("=" * 100)
print("Still incomplete preview")
print("=" * 100)
if still_incomplete_preview.empty:
    print("No still-incomplete spreads after fallback.")
else:
    display(still_incomplete_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Fallback candidates:       {FALLBACK_CANDIDATES_PATH}")
print(f"Fallback candidate cache:  {FALLBACK_CANDIDATE_CACHE_PATH}")
print(f"Fallback map:              {FALLBACK_MAP_PATH}")
print(f"Spreads with fallback:     {SPREADS_WITH_FALLBACK_PATH}")
print(f"Manifest:                  {manifest_path}")

print()
print("Result: 2F.5 repaired-RSI fallback diagnostic complete.")
print("Next step: 2F.6 build repaired-RSI fallback-adjusted rule outcome tables.")

Cell 2F.5 — Repaired-RSI listed-strike fallback diagnostic
Primary priced spreads:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_repaired_rsi_long5_cal_v1.parquet
Primary contract cache:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet
Fallback candidate cache out: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_candidate_contract_price_cache_repaired_rsi_long5_cal_v1.parquet
Old fallback cache reuse:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet
Max requests this run:        None

Loaded primary-priced repaired-RSI spreads
Unique spread rows:       1,734
Primary complete spreads: 1,178
Incomplete spreads:       556
Primary cache rows:       3,423



,leg_label,missing_legs
0,short_call_1sd,503
1,long_call_3sd,130


Fallback candidate rows:        6,443
Unique fallback contracts:      5,745


,leg_label,fallback_candidate_rows,unique_candidate_contracts,missing_spreads
0,long_call_3sd,910,851,130
1,short_call_1sd,5533,4895,503



Existing repaired fallback cache rows loaded: 0
Primary-cache overlap candidate rows loaded: 722
Old fallback-cache overlap candidate rows loaded: 4,587

Fallback cache / remaining summary before ThetaData requests
Unique fallback contracts:      5,745
Fallback cache rows after reuse:5,309
Remaining uncached rows:        436
Rows to query this run:         436


...


Recovery by year


,trade_year,unique_spreads,complete_before,complete_after,newly_recovered,still_incomplete,avg_effective_width,avg_entry_credit,win_rate,avg_return_on_max_loss,worst_return_on_max_loss,complete_rate_before,complete_rate_after
0,2020,3,2,3,1,0,37.333333,0.560000,1.0,0.017724,0.010338,0.666667,1.000000
1,2021,135,122,133,11,2,30.947368,0.281729,0.984962,0.007823,-0.169440,0.903704,0.985185
2,2022,146,109,146,37,0,39.972603,1.065548,0.938356,0.016507,-0.432467,0.746575,1.000000
3,2023,286,248,283,35,3,27.293286,0.733675,0.795053,-0.007055,-0.516209,0.867133,0.989510
4,2024,614,399,602,203,12,34.079734,0.737392,0.88206,0.005144,-0.396081,0.649837,0.980456
5,2025,399,201,373,172,26,43.198391,0.744906,0.97319,0.014032,-0.332471,0.503759,0.934837
6,2026,151,97,146,49,5,55.472603,0.881849,0.712329,-0.018312,-0.263600,0.642384,0.966887



Still incomplete preview


,selected_trade_id,trade_date,trade_date_str,trade_year,tenor,tenor_bucket,expiration_date,expiration_str,expiration_yyyymmdd,effective_dte,...,still_incomplete_after_fallback,effective_width_after_fallback,terminal_spread_intrinsic_after_fallback,entry_credit_conservative_after_fallback,credit_pct_width_after_fallback,max_loss_conservative_after_fallback,expiry_pnl_conservative_after_fallback,return_on_max_loss_conservative_after_fallback,return_on_width_conservative_after_fallback,trade_win_conservative_after_fallback
41,SPY_CALL_20210617_T9_EXP20210702_S433.0_L455.0,2021-06-17,2021-06-17,2021,9,Front,2021-07-02,2021-07-02,20210702,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,SPY_CALL_20210617_T12_EXP20210702_S435.0_L460.0,2021-06-17,2021-06-17,2021,12,Front,2021-07-02,2021-07-02,20210702,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,SPY_CALL_20230209_T9_EXP20230224_S422.0_L450.0,2023-02-09,2023-02-09,2023,9,Front,2023-02-24,2023-02-24,20230224,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
342,SPY_CALL_20230209_T12_EXP20230224_S424.0_L455.0,2023-02-09,2023-02-09,2023,12,Front,2023-02-24,2023-02-24,20230224,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
343,SPY_CALL_20230209_T18_EXP20230303_S426.0_L465.0,2023-02-09,2023-02-09,2023,18,Front,2023-03-03,2023-03-03,20230303,22.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1042,SPY_CALL_20241003_T33_EXP20241108_S605.0_L675.0,2024-10-03,2024-10-03,2024,33,Back,2024-11-08,2024-11-08,20241108,36.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1050,SPY_CALL_20241008_T30_EXP20241108_S609.0_L680.0,2024-10-08,2024-10-08,2024,30,Back,2024-11-08,2024-11-08,20241108,31.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1054,SPY_CALL_20241009_T30_EXP20241108_S612.0_L680.0,2024-10-09,2024-10-09,2024,30,Back,2024-11-08,2024-11-08,20241108,30.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1065,SPY_CALL_20241011_T27_EXP20241108_S612.0_L675.0,2024-10-11,2024-10-11,2024,27,Back,2024-11-08,2024-11-08,20241108,28.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1070,SPY_CALL_20241014_T24_EXP20241108_S614.0_L675.0,2024-10-14,2024-10-14,2024,24,Middle,2024-11-08,2024-11-08,20241108,25.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved
Fallback candidates:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_candidates_repaired_rsi_long5_cal_v1.parquet
Fallback candidate cache:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_fallback_candidate_contract_price_cache_repaired_rsi_long5_cal_v1.parquet
Fallback map:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_map_repaired_rsi_long5_cal_v1.parquet
Spreads with fallback:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Manifest:                  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F5_fallback_diagnostic_repaired_rsi_long5_cal_v1_manifest_20260712_134008.json

Result: 2F.5 repaired-RSI fallback diagnostic complete.
Next step: 2F.6 build 

In [8]:
# Cell 2F.6 — Build repaired-RSI fallback-adjusted rule outcome tables
# Fixed version:
#   - Avoids duplicate grouping-column collisions, e.g. selection_rule already exists.
#   - Keeps same input/output filenames as prior 2F.6.
#
# This cell:
#   - Loads repaired-RSI rule-selected trades from 2D.3
#   - Loads repaired-RSI unique selected spreads with fallback-adjusted P&L from 2F.5
#   - Joins fallback-adjusted outcomes back to all rule-selected rows
#   - Builds rule-level, rule-year, rule-tenor, selection-rule, and fallback-usage summaries
#   - Saves all repaired-RSI fallback-adjusted outputs
#
# This cell does NOT:
#   - Run final mechanical sanity checks
#   - Rank final rules
#   - Select final parameters
#   - Apply Hybrid v2 put-spread sizing/selector

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

RULE_SELECTED_TRADES_UNPRICED_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_unpriced_{SUFFIX}.parquet"
)

SPREADS_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_{SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_missing_contract_fallback_map_{SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2F.6 — Build repaired-RSI fallback-adjusted rule outcome tables")
print("=" * 100)
print(f"Rule selected unpriced:  {RULE_SELECTED_TRADES_UNPRICED_PATH}")
print(f"Spreads with fallback:   {SPREADS_WITH_FALLBACK_PATH}")
print(f"Fallback map:            {FALLBACK_MAP_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def first_value(g, col):
    if col not in g.columns:
        return np.nan
    vals = g[col].dropna()
    if len(vals) == 0:
        return np.nan
    return vals.iloc[0]

def summarize_group(g):
    before = bool_series(g["spread_complete"], index=g.index) if "spread_complete" in g.columns else pd.Series(False, index=g.index)
    after = bool_series(g["spread_complete_after_fallback"], index=g.index)
    recovered = bool_series(g["newly_recovered_by_fallback"], index=g.index)
    still_incomplete = bool_series(g["still_incomplete_after_fallback"], index=g.index)

    short_fallback = (
        bool_series(g["short_fallback_found"], index=g.index)
        if "short_fallback_found" in g.columns
        else pd.Series(False, index=g.index)
    )

    long_fallback = (
        bool_series(g["long_fallback_found"], index=g.index)
        if "long_fallback_found" in g.columns
        else pd.Series(False, index=g.index)
    )

    complete_g = g.loc[after].copy()

    out = {
        "selected_rows": int(len(g)),
        "complete_before_fallback_rows": int(before.sum()),
        "complete_after_fallback_rows": int(after.sum()),
        "newly_recovered_by_fallback_rows": int(recovered.sum()),
        "still_incomplete_after_fallback_rows": int(still_incomplete.sum()),
        "coverage_before_fallback": float(before.mean()) if len(g) else np.nan,
        "coverage_after_fallback": float(after.mean()) if len(g) else np.nan,
        "fallback_recovery_rate_on_incomplete": (
            float(recovered.sum() / (~before).sum()) if int((~before).sum()) > 0 else np.nan
        ),
        "short_fallback_rows": int(short_fallback.sum()),
        "long_fallback_rows": int(long_fallback.sum()),
        "any_fallback_rows": int((short_fallback | long_fallback).sum()),
        "primary_complete_rows": int(before.sum()),
        "fallback_complete_rows": int(after.sum() - before.sum()),
    }

    # Include rule metadata when present. group_apply_summary() will remove any duplicated grouping columns.
    meta_cols = [
        "z_threshold_shared",
        "rsi_floor",
        "vrp_log_floor",
        "vrp_log_floor_label",
        "selection_rule",
    ]

    for c in meta_cols:
        if c in g.columns:
            out[c] = first_value(g, c)

    if len(complete_g) > 0:
        r = pd.to_numeric(complete_g["return_on_max_loss_conservative_after_fallback"], errors="coerce")

        out.update({
            "win_rate": float(complete_g["trade_win_conservative_after_fallback"].mean()),
            "avg_entry_credit": float(complete_g["entry_credit_conservative_after_fallback"].mean()),
            "median_entry_credit": float(complete_g["entry_credit_conservative_after_fallback"].median()),
            "avg_credit_pct_width": float(complete_g["credit_pct_width_after_fallback"].mean()),
            "median_credit_pct_width": float(complete_g["credit_pct_width_after_fallback"].median()),
            "avg_effective_width": float(complete_g["effective_width_after_fallback"].mean()),
            "median_effective_width": float(complete_g["effective_width_after_fallback"].median()),
            "avg_max_loss": float(complete_g["max_loss_conservative_after_fallback"].mean()),
            "median_max_loss": float(complete_g["max_loss_conservative_after_fallback"].median()),
            "avg_expiry_pnl": float(complete_g["expiry_pnl_conservative_after_fallback"].mean()),
            "median_expiry_pnl": float(complete_g["expiry_pnl_conservative_after_fallback"].median()),
            "avg_return_on_max_loss": float(r.mean()),
            "median_return_on_max_loss": float(r.median()),
            "worst_return_on_max_loss": float(r.min()),
            "best_return_on_max_loss": float(r.max()),
            "p01_return_on_max_loss": float(r.quantile(0.01)),
            "p05_return_on_max_loss": float(r.quantile(0.05)),
            "p10_return_on_max_loss": float(r.quantile(0.10)),
            "p90_return_on_max_loss": float(r.quantile(0.90)),
            "p95_return_on_max_loss": float(r.quantile(0.95)),
            "p99_return_on_max_loss": float(r.quantile(0.99)),
            "avg_terminal_intrinsic": float(complete_g["terminal_spread_intrinsic_after_fallback"].mean()),
            "worst_terminal_intrinsic": float(complete_g["terminal_spread_intrinsic_after_fallback"].max()),
            "avg_short_strike_slippage": (
                float(complete_g["short_signed_strike_slippage"].mean())
                if "short_signed_strike_slippage" in complete_g.columns
                else np.nan
            ),
            "avg_long_strike_slippage": (
                float(complete_g["long_signed_strike_slippage"].mean())
                if "long_signed_strike_slippage" in complete_g.columns
                else np.nan
            ),
        })

    else:
        out.update({
            "win_rate": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_effective_width": np.nan,
            "median_effective_width": np.nan,
            "avg_max_loss": np.nan,
            "median_max_loss": np.nan,
            "avg_expiry_pnl": np.nan,
            "median_expiry_pnl": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "best_return_on_max_loss": np.nan,
            "p01_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p10_return_on_max_loss": np.nan,
            "p90_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
            "p99_return_on_max_loss": np.nan,
            "avg_terminal_intrinsic": np.nan,
            "worst_terminal_intrinsic": np.nan,
            "avg_short_strike_slippage": np.nan,
            "avg_long_strike_slippage": np.nan,
        })

    return pd.Series(out)

def group_apply_summary(df, group_cols):
    if len(df) == 0:
        return pd.DataFrame()

    out = (
        df
        .groupby(group_cols, dropna=False)
        .apply(summarize_group)
    )

    # Critical fix:
    # If summarize_group also returns a grouping column, reset_index() would fail with:
    # ValueError: cannot insert selection_rule, already exists.
    for c in group_cols:
        if c in out.columns:
            out = out.drop(columns=[c])

    return out.reset_index()

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

require_file(RULE_SELECTED_TRADES_UNPRICED_PATH)
require_file(SPREADS_WITH_FALLBACK_PATH)

rule_trades = pd.read_parquet(RULE_SELECTED_TRADES_UNPRICED_PATH).copy()
spreads = pd.read_parquet(SPREADS_WITH_FALLBACK_PATH).copy()

if FALLBACK_MAP_PATH.exists():
    fallback_map = pd.read_parquet(FALLBACK_MAP_PATH).copy()
else:
    fallback_map = pd.DataFrame()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Rule-selected rows:       {len(rule_trades):,}")
print(f"Unique spreads rows:      {len(spreads):,}")
print(f"Fallback map rows:        {len(fallback_map):,}")
print()

# --------------------------------------------------------------------------------------------------
# Normalize columns
# --------------------------------------------------------------------------------------------------

required_rule_cols = [
    "rule_id",
    "selected_trade_id",
    "trade_date",
    "tenor",
    "selection_rule",
]

required_spread_cols = [
    "selected_trade_id",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

missing_rule_cols = [c for c in required_rule_cols if c not in rule_trades.columns]
missing_spread_cols = [c for c in required_spread_cols if c not in spreads.columns]

if missing_rule_cols:
    raise ValueError(f"Rule selected trades missing required columns: {missing_rule_cols}")
if missing_spread_cols:
    raise ValueError(f"Spreads with fallback missing required columns: {missing_spread_cols}")

rule_trades["selected_trade_id"] = rule_trades["selected_trade_id"].astype(str)
spreads["selected_trade_id"] = spreads["selected_trade_id"].astype(str)

for df in [rule_trades, spreads, fallback_map]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "spread_complete",
    "short_leg_price_found",
    "long_leg_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "trade_win_conservative_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "tenor",
    "effective_dte",
    "short_strike",
    "long_strike",
    "listed_width",
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "short_signed_strike_slippage",
    "long_signed_strike_slippage",
    "short_abs_strike_slippage",
    "long_abs_strike_slippage",
]:
    if c in spreads.columns:
        spreads[c] = pd.to_numeric(spreads[c], errors="coerce")

if spreads["selected_trade_id"].duplicated().any():
    dup_count = int(spreads["selected_trade_id"].duplicated().sum())
    raise ValueError(f"Spreads with fallback has duplicate selected_trade_id rows: {dup_count}")

# --------------------------------------------------------------------------------------------------
# Build spread outcome slice and join to rule rows
# --------------------------------------------------------------------------------------------------

spread_cols_keep = [
    "selected_trade_id",

    # primary status
    "spread_complete",
    "spread_pricing_status",
    "missing_short",
    "missing_long",
    "short_leg_price_found",
    "long_leg_price_found",

    # fallback status
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_fallback_found",
    "long_fallback_found",
    "short_fallback_choice_type",
    "long_fallback_choice_type",

    # effective contract/strike details
    "short_effective_contract_request_id",
    "long_effective_contract_request_id",
    "short_effective_source",
    "long_effective_source",
    "short_effective_strike",
    "long_effective_strike",
    "short_effective_bid",
    "short_effective_ask",
    "short_effective_mid",
    "long_effective_bid",
    "long_effective_ask",
    "long_effective_mid",
    "short_signed_strike_slippage",
    "long_signed_strike_slippage",
    "short_abs_strike_slippage",
    "long_abs_strike_slippage",

    # fallback-adjusted economics
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "trade_win_conservative_after_fallback",

    # original primary economics
    "entry_credit_conservative",
    "credit_pct_width",
    "max_loss_conservative",
    "expiry_pnl_conservative",
    "return_on_max_loss_conservative",
    "trade_win_conservative",
]

spread_cols_keep = [c for c in spread_cols_keep if c in spreads.columns]

rule_trades_with_fallback = rule_trades.merge(
    spreads[spread_cols_keep],
    on="selected_trade_id",
    how="left",
    validate="many_to_one",
)

for c in [
    "spread_complete",
    "short_leg_price_found",
    "long_leg_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "trade_win_conservative_after_fallback",
]:
    if c in rule_trades_with_fallback.columns:
        rule_trades_with_fallback[c] = bool_series(rule_trades_with_fallback[c], index=rule_trades_with_fallback.index)

rule_trades_with_fallback["rule_trade_outcome_available"] = rule_trades_with_fallback["spread_complete_after_fallback"]
rule_trades_with_fallback["trade_year"] = pd.to_datetime(rule_trades_with_fallback["trade_date"]).dt.year

# --------------------------------------------------------------------------------------------------
# Global summary
# --------------------------------------------------------------------------------------------------

unique_complete_before = bool_series(spreads["spread_complete"], index=spreads.index)
unique_complete_after = bool_series(spreads["spread_complete_after_fallback"], index=spreads.index)

rule_complete_before = bool_series(rule_trades_with_fallback["spread_complete"], index=rule_trades_with_fallback.index)
rule_complete_after = bool_series(rule_trades_with_fallback["spread_complete_after_fallback"], index=rule_trades_with_fallback.index)

global_summary = pd.DataFrame([{
    "rule_selected_rows": int(len(rule_trades_with_fallback)),
    "unique_selected_trades": int(spreads["selected_trade_id"].nunique()),
    "unique_spreads_total": int(len(spreads)),
    "unique_spreads_complete_before": int(unique_complete_before.sum()),
    "unique_spreads_complete_after": int(unique_complete_after.sum()),
    "unique_spreads_newly_recovered": int(bool_series(spreads["newly_recovered_by_fallback"], index=spreads.index).sum()),
    "unique_spreads_still_incomplete": int(bool_series(spreads["still_incomplete_after_fallback"], index=spreads.index).sum()),
    "unique_complete_rate_before": float(unique_complete_before.mean()),
    "unique_complete_rate_after": float(unique_complete_after.mean()),
    "rule_rows_complete_before": int(rule_complete_before.sum()),
    "rule_rows_complete_after": int(rule_complete_after.sum()),
    "rule_rows_newly_recovered": int(bool_series(rule_trades_with_fallback["newly_recovered_by_fallback"], index=rule_trades_with_fallback.index).sum()),
    "rule_rows_still_incomplete": int(bool_series(rule_trades_with_fallback["still_incomplete_after_fallback"], index=rule_trades_with_fallback.index).sum()),
    "rule_row_coverage_before": float(rule_complete_before.mean()),
    "rule_row_coverage_after": float(rule_complete_after.mean()),
    "win_rate_after_fallback": float(
        rule_trades_with_fallback.loc[rule_complete_after, "trade_win_conservative_after_fallback"].mean()
    ),
    "avg_return_on_max_loss_after_fallback": float(
        rule_trades_with_fallback.loc[rule_complete_after, "return_on_max_loss_conservative_after_fallback"].mean()
    ),
    "median_return_on_max_loss_after_fallback": float(
        rule_trades_with_fallback.loc[rule_complete_after, "return_on_max_loss_conservative_after_fallback"].median()
    ),
    "worst_return_on_max_loss_after_fallback": float(
        rule_trades_with_fallback.loc[rule_complete_after, "return_on_max_loss_conservative_after_fallback"].min()
    ),
}])

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("Building grouped fallback-adjusted summaries")
print("=" * 100)

rule_fallback_summary = group_apply_summary(
    rule_trades_with_fallback,
    ["rule_id"],
)

rule_year_fallback_summary = group_apply_summary(
    rule_trades_with_fallback,
    ["rule_id", "trade_year"],
)

rule_tenor_fallback_summary = group_apply_summary(
    rule_trades_with_fallback,
    ["rule_id", "tenor"],
)

selection_rule_fallback_summary = group_apply_summary(
    rule_trades_with_fallback,
    ["selection_rule"],
)

fallback_usage_by_rule = (
    rule_trades_with_fallback
    .assign(
        short_fallback_used=bool_series(rule_trades_with_fallback.get("short_fallback_found"), index=rule_trades_with_fallback.index),
        long_fallback_used=bool_series(rule_trades_with_fallback.get("long_fallback_found"), index=rule_trades_with_fallback.index),
    )
    .assign(
        any_fallback_used=lambda x: x["short_fallback_used"] | x["long_fallback_used"]
    )
    .groupby("rule_id", dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        complete_after_fallback_rows=("spread_complete_after_fallback", "sum"),
        short_fallback_rows=("short_fallback_used", "sum"),
        long_fallback_rows=("long_fallback_used", "sum"),
        any_fallback_rows=("any_fallback_used", "sum"),
        primary_complete_rows=("spread_complete", "sum"),
        newly_recovered_rows=("newly_recovered_by_fallback", "sum"),
        still_incomplete_rows=("still_incomplete_after_fallback", "sum"),
        avg_short_signed_slippage=("short_signed_strike_slippage", "mean"),
        avg_long_signed_slippage=("long_signed_strike_slippage", "mean"),
        avg_short_abs_slippage=("short_abs_strike_slippage", "mean"),
        avg_long_abs_slippage=("long_abs_strike_slippage", "mean"),
    )
    .reset_index()
)

fallback_usage_by_rule["coverage_after_fallback"] = safe_divide(
    fallback_usage_by_rule["complete_after_fallback_rows"],
    fallback_usage_by_rule["selected_rows"],
)

fallback_usage_by_rule["fallback_usage_rate"] = safe_divide(
    fallback_usage_by_rule["any_fallback_rows"],
    fallback_usage_by_rule["selected_rows"],
)

fallback_usage_by_rule["fallback_share_of_complete"] = safe_divide(
    fallback_usage_by_rule["any_fallback_rows"],
    fallback_usage_by_rule["complete_after_fallback_rows"],
)

unique_tenor_summary = (
    spreads
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

unique_tenor_summary["complete_rate_before"] = safe_divide(
    unique_tenor_summary["complete_before"],
    unique_tenor_summary["unique_spreads"],
)

unique_tenor_summary["complete_rate_after"] = safe_divide(
    unique_tenor_summary["complete_after"],
    unique_tenor_summary["unique_spreads"],
)

unique_year_summary = (
    spreads
    .assign(trade_year=pd.to_datetime(spreads["trade_date"]).dt.year)
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

unique_year_summary["complete_rate_before"] = safe_divide(
    unique_year_summary["complete_before"],
    unique_year_summary["unique_spreads"],
)

unique_year_summary["complete_rate_after"] = safe_divide(
    unique_year_summary["complete_after"],
    unique_year_summary["unique_spreads"],
)

rule_preview = (
    rule_fallback_summary
    .sort_values(
        [
            "coverage_after_fallback",
            "win_rate",
            "avg_return_on_max_loss",
            "selected_rows",
        ],
        ascending=[False, False, False, False],
    )
    .head(40)
    .copy()
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

RULE_SELECTED_TRADES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_{SUFFIX}.parquet"
)

RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_fallback_summary_{SUFFIX}.parquet"
)

RULE_YEAR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_year_fallback_summary_{SUFFIX}.parquet"
)

RULE_TENOR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_tenor_fallback_summary_{SUFFIX}.parquet"
)

SELECTION_RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selection_rule_fallback_summary_{SUFFIX}.parquet"
)

FALLBACK_USAGE_BY_RULE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_fallback_usage_by_rule_{SUFFIX}.parquet"
)

UNIQUE_TENOR_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_tenor_fallback_summary_{SUFFIX}.parquet"
)

UNIQUE_YEAR_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_year_fallback_summary_{SUFFIX}.parquet"
)

atomic_write_parquet(rule_trades_with_fallback, RULE_SELECTED_TRADES_WITH_FALLBACK_PATH)
atomic_write_parquet(rule_fallback_summary, RULE_FALLBACK_SUMMARY_PATH)
atomic_write_parquet(rule_year_fallback_summary, RULE_YEAR_FALLBACK_SUMMARY_PATH)
atomic_write_parquet(rule_tenor_fallback_summary, RULE_TENOR_FALLBACK_SUMMARY_PATH)
atomic_write_parquet(selection_rule_fallback_summary, SELECTION_RULE_FALLBACK_SUMMARY_PATH)
atomic_write_parquet(fallback_usage_by_rule, FALLBACK_USAGE_BY_RULE_PATH)
atomic_write_parquet(unique_tenor_summary, UNIQUE_TENOR_SUMMARY_PATH)
atomic_write_parquet(unique_year_summary, UNIQUE_YEAR_SUMMARY_PATH)

# Audit CSVs
global_summary_csv = AUDIT_DIR / f"02F6_global_fallback_adjusted_summary_{SUFFIX}_{timestamp}.csv"
rule_summary_csv = AUDIT_DIR / f"02F6_rule_fallback_summary_{SUFFIX}_{timestamp}.csv"
rule_year_summary_csv = AUDIT_DIR / f"02F6_rule_year_fallback_summary_{SUFFIX}_{timestamp}.csv"
rule_tenor_summary_csv = AUDIT_DIR / f"02F6_rule_tenor_fallback_summary_{SUFFIX}_{timestamp}.csv"
selection_rule_summary_csv = AUDIT_DIR / f"02F6_selection_rule_fallback_summary_{SUFFIX}_{timestamp}.csv"
fallback_usage_csv = AUDIT_DIR / f"02F6_fallback_usage_by_rule_{SUFFIX}_{timestamp}.csv"
unique_tenor_csv = AUDIT_DIR / f"02F6_unique_tenor_fallback_summary_{SUFFIX}_{timestamp}.csv"
unique_year_csv = AUDIT_DIR / f"02F6_unique_year_fallback_summary_{SUFFIX}_{timestamp}.csv"
rule_preview_csv = AUDIT_DIR / f"02F6_rule_preview_not_final_ranking_{SUFFIX}_{timestamp}.csv"

atomic_write_csv(global_summary, global_summary_csv)
atomic_write_csv(rule_fallback_summary, rule_summary_csv)
atomic_write_csv(rule_year_fallback_summary, rule_year_summary_csv)
atomic_write_csv(rule_tenor_fallback_summary, rule_tenor_summary_csv)
atomic_write_csv(selection_rule_fallback_summary, selection_rule_summary_csv)
atomic_write_csv(fallback_usage_by_rule, fallback_usage_csv)
atomic_write_csv(unique_tenor_summary, unique_tenor_csv)
atomic_write_csv(unique_year_summary, unique_year_csv)
atomic_write_csv(rule_preview, rule_preview_csv)

manifest_path = AUDIT_DIR / f"02F6_fallback_adjusted_rule_outcomes_{SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.6",
    "description": "Build repaired-RSI fallback-adjusted rule outcome tables for call-sleeve research",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "rule_selected_trades_unpriced_path": str(RULE_SELECTED_TRADES_UNPRICED_PATH),
        "spreads_with_fallback_path": str(SPREADS_WITH_FALLBACK_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
    },
    "outputs": {
        "rule_selected_trades_with_fallback_path": str(RULE_SELECTED_TRADES_WITH_FALLBACK_PATH),
        "rule_fallback_summary_path": str(RULE_FALLBACK_SUMMARY_PATH),
        "rule_year_fallback_summary_path": str(RULE_YEAR_FALLBACK_SUMMARY_PATH),
        "rule_tenor_fallback_summary_path": str(RULE_TENOR_FALLBACK_SUMMARY_PATH),
        "selection_rule_fallback_summary_path": str(SELECTION_RULE_FALLBACK_SUMMARY_PATH),
        "fallback_usage_by_rule_path": str(FALLBACK_USAGE_BY_RULE_PATH),
        "unique_tenor_summary_path": str(UNIQUE_TENOR_SUMMARY_PATH),
        "unique_year_summary_path": str(UNIQUE_YEAR_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "rule_selected_rows": int(len(rule_trades_with_fallback)),
        "unique_spreads_total": int(len(spreads)),
        "unique_spreads_complete_before": int(unique_complete_before.sum()),
        "unique_spreads_complete_after": int(unique_complete_after.sum()),
        "rule_rows_complete_before": int(rule_complete_before.sum()),
        "rule_rows_complete_after": int(rule_complete_after.sum()),
        "rule_count": int(rule_trades_with_fallback["rule_id"].nunique()),
        "rule_summary_rows": int(len(rule_fallback_summary)),
        "rule_year_summary_rows": int(len(rule_year_fallback_summary)),
        "rule_tenor_summary_rows": int(len(rule_tenor_fallback_summary)),
        "selection_rule_summary_rows": int(len(selection_rule_fallback_summary)),
    },
    "notes": {
        "ranking_status": "No final rule ranking performed.",
        "next_step": "Run repaired-RSI mechanical sanity checks before ranking candidate rules.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.6 complete — repaired-RSI fallback-adjusted rule outcomes")
print("=" * 100)
display(global_summary)

print()
print("=" * 100)
print("Selection-rule fallback summary")
print("=" * 100)
display(selection_rule_fallback_summary.sort_values("selection_rule"))

print()
print("=" * 100)
print("Unique spread fallback summary by tenor")
print("=" * 100)
display(unique_tenor_summary)

print()
print("=" * 100)
print("Unique spread fallback summary by year")
print("=" * 100)
display(unique_year_summary)

print()
print("=" * 100)
print("Rule fallback summary preview — NOT FINAL RANKING")
print("=" * 100)
display(rule_preview)

print()
print("=" * 100)
print("Fallback usage by rule preview")
print("=" * 100)
display(
    fallback_usage_by_rule
    .sort_values(["fallback_usage_rate", "selected_rows"], ascending=[False, False])
    .head(30)
)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Rule selected with fallback:       {RULE_SELECTED_TRADES_WITH_FALLBACK_PATH}")
print(f"Rule fallback summary:             {RULE_FALLBACK_SUMMARY_PATH}")
print(f"Rule-year fallback summary:        {RULE_YEAR_FALLBACK_SUMMARY_PATH}")
print(f"Rule-tenor fallback summary:       {RULE_TENOR_FALLBACK_SUMMARY_PATH}")
print(f"Selection-rule fallback summary:   {SELECTION_RULE_FALLBACK_SUMMARY_PATH}")
print(f"Fallback usage by rule:            {FALLBACK_USAGE_BY_RULE_PATH}")
print(f"Unique tenor summary:              {UNIQUE_TENOR_SUMMARY_PATH}")
print(f"Unique year summary:               {UNIQUE_YEAR_SUMMARY_PATH}")
print(f"Manifest:                          {manifest_path}")

print()
print("Result: 2F.6 repaired-RSI fallback-adjusted rule outcomes complete.")
print("Next step: 2F.7 repaired-RSI mechanical sanity checks before ranking.")

Cell 2F.6 — Build repaired-RSI fallback-adjusted rule outcome tables
Rule selected unpriced:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_unpriced_repaired_rsi_long5_cal_v1.parquet
Spreads with fallback:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Fallback map:            C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_map_repaired_rsi_long5_cal_v1.parquet

Loaded inputs
Rule-selected rows:       155,120
Unique spreads rows:      1,734
Fallback map rows:        568



C:\Users\patri\AppData\Local\Temp\ipykernel_14668\2862194310.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)


Building grouped fallback-adjusted summaries


C:\Users\patri\AppData\Local\Temp\ipykernel_14668\2862194310.py:252: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\2862194310.py:252: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\2862194310.py:252: DeprecationWarning: DataFrameGroupBy.apply operated on the grou


Cell 2F.6 complete — repaired-RSI fallback-adjusted rule outcomes


...


Rule fallback summary preview — NOT FINAL RANKING


,rule_id,selected_rows,complete_before_fallback_rows,complete_after_fallback_rows,newly_recovered_by_fallback_rows,still_incomplete_after_fallback_rows,coverage_before_fallback,coverage_after_fallback,fallback_recovery_rate_on_incomplete,short_fallback_rows,...,p01_return_on_max_loss,p05_return_on_max_loss,p10_return_on_max_loss,p90_return_on_max_loss,p95_return_on_max_loss,p99_return_on_max_loss,avg_terminal_intrinsic,worst_terminal_intrinsic,avg_short_strike_slippage,avg_long_strike_slippage
584,Z0p75_RSI68_VRP0p00_SEL_shortest_tenor,55,41,55,14,0,0.745455,1.0,1.0,14,...,-0.138985,-0.011616,0.011303,0.035840,0.041363,0.049951,0.158727,3.33,2.285714,-10.0
589,Z0p75_RSI68_VRP0p20_SEL_shortest_tenor,55,41,55,14,0,0.745455,1.0,1.0,14,...,-0.138985,-0.011616,0.011303,0.035840,0.041363,0.049951,0.158727,3.33,2.285714,-10.0
599,Z0p75_RSI68_VRPNONE_SEL_shortest_tenor,55,41,55,14,0,0.745455,1.0,1.0,14,...,-0.138985,-0.011616,0.011303,0.035840,0.041363,0.049951,0.158727,3.33,2.285714,-10.0
594,Z0p75_RSI68_VRP0p40_SEL_shortest_tenor,53,39,53,14,0,0.735849,1.0,1.0,14,...,-0.139129,-0.019550,0.009891,0.036055,0.041406,0.050165,0.164717,3.33,2.500000,-10.0
664,Z1p00_RSI62_VRP0p00_SEL_shortest_tenor,76,52,76,24,0,0.684211,1.0,1.0,24,...,-0.139973,-0.064396,0.009749,0.036369,0.041341,0.045667,0.209079,4.17,2.541667,NaN
669,Z1p00_RSI62_VRP0p20_SEL_shortest_tenor,76,52,76,24,0,0.684211,1.0,1.0,24,...,-0.139973,-0.064396,0.009749,0.036369,0.041341,0.045667,0.209079,4.17,2.541667,NaN
679,Z1p00_RSI62_VRPNONE_SEL_shortest_tenor,76,52,76,24,0,0.684211,1.0,1.0,24,...,-0.139973,-0.064396,0.009749,0.036369,0.041341,0.045667,0.209079,4.17,2.541667,NaN
704,Z1p00_RSI68_VRP0p00_SEL_shortest_tenor,45,30,45,15,0,0.666667,1.0,1.0,15,...,-0.141165,-0.047455,0.010021,0.036914,0.041580,0.046474,0.194000,3.33,2.333333,NaN
709,Z1p00_RSI68_VRP0p20_SEL_shortest_tenor,45,30,45,15,0,0.666667,1.0,1.0,15,...,-0.141165,-0.047455,0.010021,0.036914,0.041580,0.046474,0.194000,3.33,2.333333,NaN
719,Z1p00_RSI68_VRPNONE_SEL_shortest_tenor,45,30,45,15,0,0.666667,1.0,1.0,15,...,-0.141165,-0.047455,0.010021,0.036914,0.041580,0.046474,0.194000,3.33,2.333333,NaN



Fallback usage by rule preview


,rule_id,selected_rows,complete_after_fallback_rows,short_fallback_rows,long_fallback_rows,any_fallback_rows,primary_complete_rows,newly_recovered_rows,still_incomplete_rows,avg_short_signed_slippage,avg_long_signed_slippage,avg_short_abs_slippage,avg_long_abs_slippage,coverage_after_fallback,fallback_usage_rate,fallback_share_of_complete
703,Z1p00_RSI68_VRP0p00_SEL_longest_tenor,45,44,29,12,36,9,35,1,2.689655,-5.416667,2.689655,5.416667,0.977778,0.800000,0.818182
708,Z1p00_RSI68_VRP0p20_SEL_longest_tenor,45,44,29,12,36,9,35,1,2.689655,-5.416667,2.689655,5.416667,0.977778,0.800000,0.818182
718,Z1p00_RSI68_VRPNONE_SEL_longest_tenor,45,44,29,12,36,9,35,1,2.689655,-5.416667,2.689655,5.416667,0.977778,0.800000,0.818182
713,Z1p00_RSI68_VRP0p40_SEL_longest_tenor,43,42,27,12,34,9,33,1,2.888889,-5.416667,2.888889,5.416667,0.976744,0.790698,0.809524
683,Z1p00_RSI65_VRP0p00_SEL_longest_tenor,57,56,33,16,42,15,41,1,2.666667,-5.312500,2.666667,5.312500,0.982456,0.736842,0.750000
688,Z1p00_RSI65_VRP0p20_SEL_longest_tenor,57,56,33,16,42,15,41,1,2.666667,-5.312500,2.666667,5.312500,0.982456,0.736842,0.750000
698,Z1p00_RSI65_VRPNONE_SEL_longest_tenor,57,56,33,16,42,15,41,1,2.666667,-5.312500,2.666667,5.312500,0.982456,0.736842,0.750000
693,Z1p00_RSI65_VRP0p40_SEL_longest_tenor,54,53,30,16,39,15,38,1,2.833333,-5.312500,2.833333,5.312500,0.981481,0.722222,0.735849
633,Z1p00_RSI58_VRP0p40_SEL_longest_tenor,85,80,48,25,60,25,55,5,2.791667,-6.000000,2.791667,6.000000,0.941176,0.705882,0.750000
623,Z1p00_RSI58_VRP0p00_SEL_longest_tenor,88,83,50,25,62,26,57,5,2.680000,-6.000000,2.680000,6.000000,0.943182,0.704545,0.746988



Saved
Rule selected with fallback:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Rule fallback summary:             C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_fallback_summary_repaired_rsi_long5_cal_v1.parquet
Rule-year fallback summary:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_year_fallback_summary_repaired_rsi_long5_cal_v1.parquet
Rule-tenor fallback summary:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_tenor_fallback_summary_repaired_rsi_long5_cal_v1.parquet
Selection-rule fallback summary:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_selection_rule_fallback_summary_repaired_rsi_long5_cal_v1.parquet
Fallback usage by rule:            C:\Users\patri\vrp_project\data\

In [9]:
# Cell 2F.7 — Repaired-RSI mechanical sanity checks before ranking
# Purpose:
#   Validate repaired-RSI fallback-adjusted call-sleeve outputs before rule ranking.
#
# This cell checks:
#   1. selected_trade_id uniqueness and mapping
#   2. every rule row maps to one unique spread
#   3. effective long strike > effective short strike
#   4. effective width equals effective long - effective short
#   5. credit > 0 and max loss > 0 for complete rows
#   6. terminal intrinsic bounded by effective width
#   7. P&L = credit - terminal intrinsic
#   8. return = P&L / max loss
#   9. win flag matches P&L > 0
#  10. fallback direction / structure checks
#  11. coverage by tenor/year/selection rule/rule
#  12. still-incomplete spread list
#
# This cell does NOT:
#   - Rank rules
#   - Select final parameters
#   - Apply Hybrid v2 put-spread sizing/selector

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

SPREADS_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_{SUFFIX}.parquet"
)

RULE_SELECTED_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_{SUFFIX}.parquet"
)

RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_fallback_summary_{SUFFIX}.parquet"
)

RULE_YEAR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_year_fallback_summary_{SUFFIX}.parquet"
)

RULE_TENOR_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_rule_tenor_fallback_summary_{SUFFIX}.parquet"
)

SELECTION_RULE_FALLBACK_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_selection_rule_fallback_summary_{SUFFIX}.parquet"
)

FALLBACK_USAGE_BY_RULE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_fallback_usage_by_rule_{SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_missing_contract_fallback_map_{SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Keep this False so the cell saves diagnostics even if something fails.
# If hard checks fail, do not proceed to ranking.
RAISE_ON_HARD_FAIL = False

TOL = 1e-8

print("=" * 100)
print("Cell 2F.7 — Repaired-RSI mechanical sanity checks before ranking")
print("=" * 100)
print(f"Spreads with fallback:       {SPREADS_WITH_FALLBACK_PATH}")
print(f"Rule rows with fallback:     {RULE_SELECTED_WITH_FALLBACK_PATH}")
print(f"Rule fallback summary:       {RULE_FALLBACK_SUMMARY_PATH}")
print(f"Fallback map:                {FALLBACK_MAP_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def near(a, b, tol=TOL):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return (a - b).abs() <= tol

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def add_check(rows, severity, check_name, passed, fail_count, detail):
    rows.append({
        "severity": severity,
        "check_name": check_name,
        "passed": bool(passed),
        "fail_count": int(fail_count) if pd.notna(fail_count) else np.nan,
        "detail": detail,
    })

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    SPREADS_WITH_FALLBACK_PATH,
    RULE_SELECTED_WITH_FALLBACK_PATH,
    RULE_FALLBACK_SUMMARY_PATH,
    RULE_YEAR_FALLBACK_SUMMARY_PATH,
    RULE_TENOR_FALLBACK_SUMMARY_PATH,
    SELECTION_RULE_FALLBACK_SUMMARY_PATH,
    FALLBACK_USAGE_BY_RULE_PATH,
]:
    require_file(p)

spreads = pd.read_parquet(SPREADS_WITH_FALLBACK_PATH).copy()
rule_rows = pd.read_parquet(RULE_SELECTED_WITH_FALLBACK_PATH).copy()
rule_summary = pd.read_parquet(RULE_FALLBACK_SUMMARY_PATH).copy()
rule_year_summary = pd.read_parquet(RULE_YEAR_FALLBACK_SUMMARY_PATH).copy()
rule_tenor_summary = pd.read_parquet(RULE_TENOR_FALLBACK_SUMMARY_PATH).copy()
selection_rule_summary = pd.read_parquet(SELECTION_RULE_FALLBACK_SUMMARY_PATH).copy()
fallback_usage_by_rule = pd.read_parquet(FALLBACK_USAGE_BY_RULE_PATH).copy()

if FALLBACK_MAP_PATH.exists():
    fallback_map = pd.read_parquet(FALLBACK_MAP_PATH).copy()
else:
    fallback_map = pd.DataFrame()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Unique spread rows:              {len(spreads):,}")
print(f"Rule-selected rows:              {len(rule_rows):,}")
print(f"Rule summary rows:               {len(rule_summary):,}")
print(f"Rule-year summary rows:          {len(rule_year_summary):,}")
print(f"Rule-tenor summary rows:         {len(rule_tenor_summary):,}")
print(f"Selection-rule summary rows:     {len(selection_rule_summary):,}")
print(f"Fallback usage rows:             {len(fallback_usage_by_rule):,}")
print(f"Fallback map rows:               {len(fallback_map):,}")
print()

# --------------------------------------------------------------------------------------------------
# Normalize
# --------------------------------------------------------------------------------------------------

for df in [spreads, rule_rows, rule_summary, rule_year_summary, rule_tenor_summary, selection_rule_summary, fallback_usage_by_rule, fallback_map]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for df in [spreads, rule_rows, fallback_map]:
    if "selected_trade_id" in df.columns:
        df["selected_trade_id"] = df["selected_trade_id"].astype(str)

for c in [
    "spread_complete",
    "short_leg_price_found",
    "long_leg_price_found",
    "short_fallback_found",
    "long_fallback_found",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "trade_win_conservative_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)
    if c in rule_rows.columns:
        rule_rows[c] = bool_series(rule_rows[c], index=rule_rows.index)

numeric_cols = [
    "tenor",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "short_signed_strike_slippage",
    "long_signed_strike_slippage",
    "short_abs_strike_slippage",
    "long_abs_strike_slippage",
]

for df in [spreads, rule_rows]:
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

# --------------------------------------------------------------------------------------------------
# Required column checks
# --------------------------------------------------------------------------------------------------

required_spread_cols = [
    "selected_trade_id",
    "trade_date",
    "tenor",
    "expiration_date",
    "short_strike",
    "long_strike",
    "listed_width",
    "expiry_spy_close",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

required_rule_cols = [
    "rule_id",
    "selected_trade_id",
    "trade_date",
    "tenor",
    "selection_rule",
    "spread_complete_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

missing_spread_cols = [c for c in required_spread_cols if c not in spreads.columns]
missing_rule_cols = [c for c in required_rule_cols if c not in rule_rows.columns]

if missing_spread_cols:
    raise ValueError(f"Spreads file missing required columns: {missing_spread_cols}")

if missing_rule_cols:
    raise ValueError(f"Rule rows file missing required columns: {missing_rule_cols}")

# --------------------------------------------------------------------------------------------------
# Mechanical checks
# --------------------------------------------------------------------------------------------------

checks = []

complete_spreads = spreads.loc[spreads["spread_complete_after_fallback"]].copy()
incomplete_spreads = spreads.loc[~spreads["spread_complete_after_fallback"]].copy()

complete_rule_rows = rule_rows.loc[rule_rows["spread_complete_after_fallback"]].copy()

# 1. selected_trade_id uniqueness and mapping
dup_spread_ids = int(spreads["selected_trade_id"].duplicated().sum())
add_check(
    checks,
    "HARD",
    "unique_spread_selected_trade_id",
    dup_spread_ids == 0,
    dup_spread_ids,
    "Unique spread table must have no duplicate selected_trade_id rows.",
)

spread_id_set = set(spreads["selected_trade_id"])
rule_id_set = set(rule_rows["selected_trade_id"])
missing_rule_to_spread = int((~rule_rows["selected_trade_id"].isin(spread_id_set)).sum())
add_check(
    checks,
    "HARD",
    "every_rule_row_maps_to_spread",
    missing_rule_to_spread == 0,
    missing_rule_to_spread,
    "Every rule-selected row must map to exactly one unique spread.",
)

spread_ids_unused_by_rules = len(spread_id_set - rule_id_set)
add_check(
    checks,
    "SOFT",
    "all_unique_spreads_used_by_some_rule",
    spread_ids_unused_by_rules == 0,
    spread_ids_unused_by_rules,
    "A unique spread not selected by any rule is not fatal but should be understood.",
)

# 2. outcome availability for complete rows
economic_cols = [
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

missing_economics = int(complete_spreads[economic_cols].isna().any(axis=1).sum())
add_check(
    checks,
    "HARD",
    "complete_spreads_have_all_economic_fields",
    missing_economics == 0,
    missing_economics,
    "Complete spreads must have all fallback-adjusted economics populated.",
)

# 3. effective long strike > effective short strike
bad_strike_order = int(
    (
        complete_spreads["long_effective_strike"]
        <= complete_spreads["short_effective_strike"]
    ).sum()
)
add_check(
    checks,
    "HARD",
    "effective_long_strike_above_short_strike",
    bad_strike_order == 0,
    bad_strike_order,
    "Complete call spreads must have effective long strike > effective short strike.",
)

# 4. effective width math
bad_effective_width = int(
    (~near(
        complete_spreads["effective_width_after_fallback"],
        complete_spreads["long_effective_strike"] - complete_spreads["short_effective_strike"],
    )).sum()
)
add_check(
    checks,
    "HARD",
    "effective_width_matches_effective_strikes",
    bad_effective_width == 0,
    bad_effective_width,
    "Effective width must equal long_effective_strike - short_effective_strike.",
)

# 5. credit and max loss sanity
bad_credit = int(
    (
        (complete_spreads["entry_credit_conservative_after_fallback"] <= 0)
        | (complete_spreads["entry_credit_conservative_after_fallback"] >= complete_spreads["effective_width_after_fallback"])
        | complete_spreads["entry_credit_conservative_after_fallback"].isna()
    ).sum()
)
add_check(
    checks,
    "HARD",
    "entry_credit_positive_and_below_width",
    bad_credit == 0,
    bad_credit,
    "Entry credit must be > 0 and < effective width.",
)

bad_max_loss = int(
    (
        (complete_spreads["max_loss_conservative_after_fallback"] <= 0)
        | complete_spreads["max_loss_conservative_after_fallback"].isna()
    ).sum()
)
add_check(
    checks,
    "HARD",
    "max_loss_positive",
    bad_max_loss == 0,
    bad_max_loss,
    "Max loss must be positive for complete spreads.",
)

bad_max_loss_formula = int(
    (~near(
        complete_spreads["max_loss_conservative_after_fallback"],
        complete_spreads["effective_width_after_fallback"] - complete_spreads["entry_credit_conservative_after_fallback"],
    )).sum()
)
add_check(
    checks,
    "HARD",
    "max_loss_equals_width_minus_credit",
    bad_max_loss_formula == 0,
    bad_max_loss_formula,
    "Max loss must equal effective width minus conservative entry credit.",
)

# 6. terminal intrinsic bounds and recomputation
terminal_recomputed = (
    np.maximum(complete_spreads["expiry_spy_close"] - complete_spreads["short_effective_strike"], 0.0)
    - np.maximum(complete_spreads["expiry_spy_close"] - complete_spreads["long_effective_strike"], 0.0)
)

bad_terminal_formula = int(
    (~near(
        complete_spreads["terminal_spread_intrinsic_after_fallback"],
        terminal_recomputed,
    )).sum()
)
add_check(
    checks,
    "HARD",
    "terminal_intrinsic_formula",
    bad_terminal_formula == 0,
    bad_terminal_formula,
    "Terminal spread intrinsic must equal short call intrinsic minus long call intrinsic.",
)

bad_terminal_bounds = int(
    (
        (complete_spreads["terminal_spread_intrinsic_after_fallback"] < -TOL)
        | (complete_spreads["terminal_spread_intrinsic_after_fallback"] > complete_spreads["effective_width_after_fallback"] + TOL)
    ).sum()
)
add_check(
    checks,
    "HARD",
    "terminal_intrinsic_bounded_by_width",
    bad_terminal_bounds == 0,
    bad_terminal_bounds,
    "Terminal intrinsic must be between zero and effective width.",
)

# 7. P&L math
bad_pnl_formula = int(
    (~near(
        complete_spreads["expiry_pnl_conservative_after_fallback"],
        complete_spreads["entry_credit_conservative_after_fallback"] - complete_spreads["terminal_spread_intrinsic_after_fallback"],
    )).sum()
)
add_check(
    checks,
    "HARD",
    "pnl_equals_credit_minus_terminal_intrinsic",
    bad_pnl_formula == 0,
    bad_pnl_formula,
    "Expiry P&L must equal credit minus terminal spread intrinsic.",
)

# 8. return math
bad_return_formula = int(
    (~near(
        complete_spreads["return_on_max_loss_conservative_after_fallback"],
        complete_spreads["expiry_pnl_conservative_after_fallback"] / complete_spreads["max_loss_conservative_after_fallback"],
    )).sum()
)
add_check(
    checks,
    "HARD",
    "return_on_max_loss_formula",
    bad_return_formula == 0,
    bad_return_formula,
    "Return on max loss must equal P&L divided by max loss.",
)

bad_return_width_formula = int(
    (~near(
        complete_spreads["return_on_width_conservative_after_fallback"],
        complete_spreads["expiry_pnl_conservative_after_fallback"] / complete_spreads["effective_width_after_fallback"],
    )).sum()
)
add_check(
    checks,
    "HARD",
    "return_on_width_formula",
    bad_return_width_formula == 0,
    bad_return_width_formula,
    "Return on width must equal P&L divided by effective width.",
)

# 9. win flag
bad_win_flag = int(
    (
        complete_spreads["trade_win_conservative_after_fallback"].astype(bool)
        != (complete_spreads["expiry_pnl_conservative_after_fallback"] > 0)
    ).sum()
)
add_check(
    checks,
    "HARD",
    "win_flag_matches_positive_pnl",
    bad_win_flag == 0,
    bad_win_flag,
    "Trade win flag must match P&L > 0.",
)

# 10. fallback logic / direction / structure checks
short_fb = complete_spreads.loc[bool_series(complete_spreads.get("short_fallback_found"), index=complete_spreads.index)].copy()
long_fb = complete_spreads.loc[bool_series(complete_spreads.get("long_fallback_found"), index=complete_spreads.index)].copy()

short_pref_viol = 0
if len(short_fb) > 0 and "short_fallback_choice_type" in short_fb.columns:
    short_pref = short_fb.loc[short_fb["short_fallback_choice_type"].eq("preferred_at_or_above_target")]
    short_pref_viol = int((short_pref["short_effective_strike"] < short_pref["short_strike"] - TOL).sum())

add_check(
    checks,
    "HARD",
    "short_fallback_preferred_direction_valid",
    short_pref_viol == 0,
    short_pref_viol,
    "Short fallback marked preferred_at_or_above_target must have effective strike >= target short strike.",
)

long_pref_viol = 0
if len(long_fb) > 0 and "long_fallback_choice_type" in long_fb.columns:
    long_pref = long_fb.loc[long_fb["long_fallback_choice_type"].eq("preferred_at_or_below_target")]
    long_pref_viol = int((long_pref["long_effective_strike"] > long_pref["long_strike"] + TOL).sum())

add_check(
    checks,
    "HARD",
    "long_fallback_preferred_direction_valid",
    long_pref_viol == 0,
    long_pref_viol,
    "Long fallback marked preferred_at_or_below_target must have effective strike <= target long strike.",
)

diagnostic_short_count = 0
if "short_fallback_choice_type" in complete_spreads.columns:
    diagnostic_short_count = int(complete_spreads["short_fallback_choice_type"].eq("diagnostic_below_target").sum())

add_check(
    checks,
    "SOFT",
    "no_short_diagnostic_below_target_fallbacks",
    diagnostic_short_count == 0,
    diagnostic_short_count,
    "Diagnostic short fallback below target is more aggressive; should be zero or reviewed.",
)

diagnostic_long_count = 0
if "long_fallback_choice_type" in complete_spreads.columns:
    diagnostic_long_count = int(complete_spreads["long_fallback_choice_type"].eq("diagnostic_above_target").sum())

add_check(
    checks,
    "SOFT",
    "no_long_diagnostic_above_target_fallbacks",
    diagnostic_long_count == 0,
    diagnostic_long_count,
    "Diagnostic long fallback above target is less protective; should be zero or reviewed.",
)

bad_fallback_structure = int(
    (
        (complete_spreads["long_effective_strike"] <= complete_spreads["short_effective_strike"])
        | (complete_spreads["effective_width_after_fallback"] <= 0)
    ).sum()
)
add_check(
    checks,
    "HARD",
    "fallback_preserves_valid_spread_structure",
    bad_fallback_structure == 0,
    bad_fallback_structure,
    "Fallback must preserve valid vertical call spread structure.",
)

# 11. flag identities
bad_recovered_flag = int(
    (
        bool_series(spreads["newly_recovered_by_fallback"], index=spreads.index)
        != (
            (~bool_series(spreads["spread_complete"], index=spreads.index))
            & bool_series(spreads["spread_complete_after_fallback"], index=spreads.index)
        )
    ).sum()
)
add_check(
    checks,
    "HARD",
    "newly_recovered_flag_identity",
    bad_recovered_flag == 0,
    bad_recovered_flag,
    "newly_recovered_by_fallback must equal not primary-complete and fallback-complete.",
)

bad_still_incomplete_flag = int(
    (
        bool_series(spreads["still_incomplete_after_fallback"], index=spreads.index)
        != (~bool_series(spreads["spread_complete_after_fallback"], index=spreads.index))
    ).sum()
)
add_check(
    checks,
    "HARD",
    "still_incomplete_flag_identity",
    bad_still_incomplete_flag == 0,
    bad_still_incomplete_flag,
    "still_incomplete_after_fallback must equal not spread_complete_after_fallback.",
)

# 12. summaries should reconcile to detail rows
rule_complete_after_from_detail = (
    rule_rows
    .groupby("rule_id", dropna=False)["spread_complete_after_fallback"]
    .sum()
    .reset_index(name="detail_complete_after")
)

if "complete_after_fallback_rows" in rule_summary.columns:
    rule_recon = rule_summary[["rule_id", "complete_after_fallback_rows"]].merge(
        rule_complete_after_from_detail,
        on="rule_id",
        how="outer",
    )
    rule_recon["diff"] = (
        pd.to_numeric(rule_recon["complete_after_fallback_rows"], errors="coerce")
        - pd.to_numeric(rule_recon["detail_complete_after"], errors="coerce")
    )
    bad_rule_summary_recon = int(rule_recon["diff"].fillna(999999).ne(0).sum())
else:
    rule_recon = pd.DataFrame()
    bad_rule_summary_recon = 0

add_check(
    checks,
    "HARD",
    "rule_summary_reconciles_to_detail",
    bad_rule_summary_recon == 0,
    bad_rule_summary_recon,
    "Rule summary complete_after_fallback_rows must reconcile to rule detail rows.",
)

# --------------------------------------------------------------------------------------------------
# Coverage summaries
# --------------------------------------------------------------------------------------------------

coverage_by_tenor = (
    spreads
    .groupby("tenor", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

coverage_by_tenor["complete_rate_before"] = safe_divide(
    coverage_by_tenor["complete_before"],
    coverage_by_tenor["unique_spreads"],
)

coverage_by_tenor["complete_rate_after"] = safe_divide(
    coverage_by_tenor["complete_after"],
    coverage_by_tenor["unique_spreads"],
)

spreads["trade_year"] = pd.to_datetime(spreads["trade_date"]).dt.year

coverage_by_year = (
    spreads
    .groupby("trade_year", dropna=False)
    .agg(
        unique_spreads=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        avg_effective_width=("effective_width_after_fallback", "mean"),
        avg_entry_credit=("entry_credit_conservative_after_fallback", "mean"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

coverage_by_year["complete_rate_before"] = safe_divide(
    coverage_by_year["complete_before"],
    coverage_by_year["unique_spreads"],
)

coverage_by_year["complete_rate_after"] = safe_divide(
    coverage_by_year["complete_after"],
    coverage_by_year["unique_spreads"],
)

coverage_by_selection_rule = (
    rule_rows
    .groupby("selection_rule", dropna=False)
    .agg(
        selected_rows=("selected_trade_id", "size"),
        complete_before=("spread_complete", "sum"),
        complete_after=("spread_complete_after_fallback", "sum"),
        newly_recovered=("newly_recovered_by_fallback", "sum"),
        still_incomplete=("still_incomplete_after_fallback", "sum"),
        win_rate=("trade_win_conservative_after_fallback", "mean"),
        avg_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "mean"),
        median_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "median"),
        worst_return_on_max_loss=("return_on_max_loss_conservative_after_fallback", "min"),
    )
    .reset_index()
)

coverage_by_selection_rule["coverage_before"] = safe_divide(
    coverage_by_selection_rule["complete_before"],
    coverage_by_selection_rule["selected_rows"],
)

coverage_by_selection_rule["coverage_after"] = safe_divide(
    coverage_by_selection_rule["complete_after"],
    coverage_by_selection_rule["selected_rows"],
)

rule_coverage_extremes = (
    rule_summary
    .sort_values(
        ["coverage_after_fallback", "selected_rows"],
        ascending=[True, False],
    )
    .head(50)
    .copy()
)

still_incomplete_preview = (
    spreads
    .loc[~spreads["spread_complete_after_fallback"]]
    .sort_values(["trade_date", "tenor", "short_strike", "long_strike"])
    .head(200)
    .copy()
)

fallback_direction_violations = []

if len(short_fb) > 0 and "short_fallback_choice_type" in short_fb.columns:
    fallback_direction_violations.append(
        short_fb.loc[
            short_fb["short_fallback_choice_type"].eq("preferred_at_or_above_target")
            & (short_fb["short_effective_strike"] < short_fb["short_strike"] - TOL)
        ].assign(violation_type="short_preferred_below_target")
    )

if len(long_fb) > 0 and "long_fallback_choice_type" in long_fb.columns:
    fallback_direction_violations.append(
        long_fb.loc[
            long_fb["long_fallback_choice_type"].eq("preferred_at_or_below_target")
            & (long_fb["long_effective_strike"] > long_fb["long_strike"] + TOL)
        ].assign(violation_type="long_preferred_above_target")
    )

if fallback_direction_violations:
    fallback_direction_violations = pd.concat(fallback_direction_violations, ignore_index=True)
else:
    fallback_direction_violations = pd.DataFrame()

# --------------------------------------------------------------------------------------------------
# Global sanity summary
# --------------------------------------------------------------------------------------------------

checks_df = pd.DataFrame(checks)

hard_failed = checks_df.loc[
    checks_df["severity"].eq("HARD") & (~checks_df["passed"])
].copy()

soft_failed = checks_df.loc[
    checks_df["severity"].eq("SOFT") & (~checks_df["passed"])
].copy()

complete_mask = spreads["spread_complete_after_fallback"]

global_sanity_summary = pd.DataFrame([{
    "unique_spread_rows": int(len(spreads)),
    "rule_selected_rows": int(len(rule_rows)),
    "rule_summary_rows": int(len(rule_summary)),
    "rule_year_summary_rows": int(len(rule_year_summary)),
    "rule_tenor_summary_rows": int(len(rule_tenor_summary)),
    "selection_rule_summary_rows": int(len(selection_rule_summary)),
    "fallback_usage_rows": int(len(fallback_usage_by_rule)),
    "fallback_map_rows": int(len(fallback_map)),
    "hard_checks_total": int(checks_df["severity"].eq("HARD").sum()),
    "hard_checks_failed": int(len(hard_failed)),
    "soft_checks_total": int(checks_df["severity"].eq("SOFT").sum()),
    "soft_checks_failed": int(len(soft_failed)),
    "unique_complete_before": int(spreads["spread_complete"].sum()),
    "unique_complete_after": int(spreads["spread_complete_after_fallback"].sum()),
    "unique_still_incomplete": int((~spreads["spread_complete_after_fallback"]).sum()),
    "unique_coverage_before": float(spreads["spread_complete"].mean()),
    "unique_coverage_after": float(spreads["spread_complete_after_fallback"].mean()),
    "rule_rows_complete_before": int(rule_rows["spread_complete"].sum()),
    "rule_rows_complete_after": int(rule_rows["spread_complete_after_fallback"].sum()),
    "rule_rows_still_incomplete": int((~rule_rows["spread_complete_after_fallback"]).sum()),
    "rule_row_coverage_before": float(rule_rows["spread_complete"].mean()),
    "rule_row_coverage_after": float(rule_rows["spread_complete_after_fallback"].mean()),
    "complete_avg_entry_credit": float(spreads.loc[complete_mask, "entry_credit_conservative_after_fallback"].mean()),
    "complete_median_entry_credit": float(spreads.loc[complete_mask, "entry_credit_conservative_after_fallback"].median()),
    "complete_avg_credit_pct_width": float(spreads.loc[complete_mask, "credit_pct_width_after_fallback"].mean()),
    "complete_win_rate": float(spreads.loc[complete_mask, "trade_win_conservative_after_fallback"].mean()),
    "complete_avg_return_on_max_loss": float(spreads.loc[complete_mask, "return_on_max_loss_conservative_after_fallback"].mean()),
    "complete_median_return_on_max_loss": float(spreads.loc[complete_mask, "return_on_max_loss_conservative_after_fallback"].median()),
    "complete_worst_return_on_max_loss": float(spreads.loc[complete_mask, "return_on_max_loss_conservative_after_fallback"].min()),
}])

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

checks_csv = AUDIT_DIR / f"02F7_mechanical_sanity_checks_{SUFFIX}_{timestamp}.csv"
global_summary_csv = AUDIT_DIR / f"02F7_global_sanity_summary_{SUFFIX}_{timestamp}.csv"
coverage_by_tenor_csv = AUDIT_DIR / f"02F7_coverage_by_tenor_{SUFFIX}_{timestamp}.csv"
coverage_by_year_csv = AUDIT_DIR / f"02F7_coverage_by_year_{SUFFIX}_{timestamp}.csv"
coverage_by_selection_rule_csv = AUDIT_DIR / f"02F7_coverage_by_selection_rule_{SUFFIX}_{timestamp}.csv"
rule_coverage_extremes_csv = AUDIT_DIR / f"02F7_rule_coverage_extremes_{SUFFIX}_{timestamp}.csv"
still_incomplete_preview_csv = AUDIT_DIR / f"02F7_still_incomplete_preview_{SUFFIX}_{timestamp}.csv"
fallback_direction_violations_csv = AUDIT_DIR / f"02F7_fallback_direction_violations_{SUFFIX}_{timestamp}.csv"

atomic_write_csv(checks_df, checks_csv)
atomic_write_csv(global_sanity_summary, global_summary_csv)
atomic_write_csv(coverage_by_tenor, coverage_by_tenor_csv)
atomic_write_csv(coverage_by_year, coverage_by_year_csv)
atomic_write_csv(coverage_by_selection_rule, coverage_by_selection_rule_csv)
atomic_write_csv(rule_coverage_extremes, rule_coverage_extremes_csv)
atomic_write_csv(still_incomplete_preview, still_incomplete_preview_csv)
atomic_write_csv(fallback_direction_violations, fallback_direction_violations_csv)

manifest_path = AUDIT_DIR / f"02F7_mechanical_sanity_{SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2F.7",
    "description": "Mechanical sanity checks for repaired-RSI fallback-adjusted call-sleeve outputs before ranking",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "spreads_with_fallback_path": str(SPREADS_WITH_FALLBACK_PATH),
        "rule_selected_with_fallback_path": str(RULE_SELECTED_WITH_FALLBACK_PATH),
        "rule_fallback_summary_path": str(RULE_FALLBACK_SUMMARY_PATH),
        "rule_year_fallback_summary_path": str(RULE_YEAR_FALLBACK_SUMMARY_PATH),
        "rule_tenor_fallback_summary_path": str(RULE_TENOR_FALLBACK_SUMMARY_PATH),
        "selection_rule_fallback_summary_path": str(SELECTION_RULE_FALLBACK_SUMMARY_PATH),
        "fallback_usage_by_rule_path": str(FALLBACK_USAGE_BY_RULE_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
    },
    "outputs": {
        "checks_csv": str(checks_csv),
        "global_summary_csv": str(global_summary_csv),
        "coverage_by_tenor_csv": str(coverage_by_tenor_csv),
        "coverage_by_year_csv": str(coverage_by_year_csv),
        "coverage_by_selection_rule_csv": str(coverage_by_selection_rule_csv),
        "rule_coverage_extremes_csv": str(rule_coverage_extremes_csv),
        "still_incomplete_preview_csv": str(still_incomplete_preview_csv),
        "fallback_direction_violations_csv": str(fallback_direction_violations_csv),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "unique_spread_rows": int(len(spreads)),
        "rule_selected_rows": int(len(rule_rows)),
        "hard_checks_failed": int(len(hard_failed)),
        "soft_checks_failed": int(len(soft_failed)),
        "unique_complete_after": int(spreads["spread_complete_after_fallback"].sum()),
        "unique_still_incomplete": int((~spreads["spread_complete_after_fallback"]).sum()),
        "rule_rows_complete_after": int(rule_rows["spread_complete_after_fallback"].sum()),
        "rule_rows_still_incomplete": int((~rule_rows["spread_complete_after_fallback"]).sum()),
    },
    "notes": {
        "ranking_status": "No final rule ranking performed.",
        "next_step_if_passed": "Rank repaired-RSI fallback-adjusted call-sleeve rules.",
        "next_step_if_failed": "Investigate hard-check failures before ranking.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2F.7 complete — repaired-RSI mechanical sanity checks")
print("=" * 100)
display(global_sanity_summary)

print()
print("=" * 100)
print("Hard / soft check results")
print("=" * 100)
display(checks_df)

print()
print("=" * 100)
print("Failed hard checks")
print("=" * 100)
if hard_failed.empty:
    print("No failed hard checks.")
else:
    display(hard_failed)

print()
print("=" * 100)
print("Failed soft checks")
print("=" * 100)
if soft_failed.empty:
    print("No failed soft checks.")
else:
    display(soft_failed)

print()
print("=" * 100)
print("Coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Coverage by selection rule")
print("=" * 100)
display(coverage_by_selection_rule)

print()
print("=" * 100)
print("Worst rule coverage preview")
print("=" * 100)
display(rule_coverage_extremes)

print()
print("=" * 100)
print("Still-incomplete spread preview")
print("=" * 100)
if still_incomplete_preview.empty:
    print("No still-incomplete spreads.")
else:
    display(still_incomplete_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Checks CSV:                         {checks_csv}")
print(f"Global summary CSV:                 {global_summary_csv}")
print(f"Coverage by tenor CSV:              {coverage_by_tenor_csv}")
print(f"Coverage by year CSV:               {coverage_by_year_csv}")
print(f"Coverage by selection rule CSV:     {coverage_by_selection_rule_csv}")
print(f"Rule coverage extremes CSV:         {rule_coverage_extremes_csv}")
print(f"Still incomplete preview CSV:       {still_incomplete_preview_csv}")
print(f"Fallback direction violations CSV:  {fallback_direction_violations_csv}")
print(f"Manifest:                           {manifest_path}")

if len(hard_failed) == 0:
    print()
    print("Result: 2F.7 PASSED hard mechanical checks.")
    print("Next step: rank repaired-RSI fallback-adjusted call-sleeve rules.")
else:
    print()
    print("Result: 2F.7 FAILED hard mechanical checks. Do not rank rules until these are fixed.")

if RAISE_ON_HARD_FAIL and len(hard_failed) > 0:
    raise AssertionError(f"2F.7 hard mechanical checks failed: {len(hard_failed)}")

Cell 2F.7 — Repaired-RSI mechanical sanity checks before ranking
Spreads with fallback:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Rule rows with fallback:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_selected_trades_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Rule fallback summary:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_rule_fallback_summary_repaired_rsi_long5_cal_v1.parquet
Fallback map:                C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_missing_contract_fallback_map_repaired_rsi_long5_cal_v1.parquet

Loaded inputs
Unique spread rows:              1,734
Rule-selected rows:              155,120
Rule summary rows:               840
Rule-year summary rows:          5,680
Rule-tenor summary row

C:\Users\patri\AppData\Local\Temp\ipykernel_14668\1188734623.py:122: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)



Cell 2F.7 complete — repaired-RSI mechanical sanity checks


,unique_spread_rows,rule_selected_rows,rule_summary_rows,rule_year_summary_rows,rule_tenor_summary_rows,selection_rule_summary_rows,fallback_usage_rows,fallback_map_rows,hard_checks_total,hard_checks_failed,...,rule_rows_still_incomplete,rule_row_coverage_before,rule_row_coverage_after,complete_avg_entry_credit,complete_median_entry_credit,complete_avg_credit_pct_width,complete_win_rate,complete_avg_return_on_max_loss,complete_median_return_on_max_loss,complete_worst_return_on_max_loss
0,1734,155120,840,5680,7423,5,840,568,20,0,...,4258,0.668386,0.97255,0.743096,0.71,0.021711,0.886121,0.004249,0.017832,-0.516209



Hard / soft check results


...


Worst rule coverage preview


,rule_id,selected_rows,complete_before_fallback_rows,complete_after_fallback_rows,newly_recovered_by_fallback_rows,still_incomplete_after_fallback_rows,coverage_before_fallback,coverage_after_fallback,fallback_recovery_rate_on_incomplete,short_fallback_rows,...,p01_return_on_max_loss,p05_return_on_max_loss,p10_return_on_max_loss,p90_return_on_max_loss,p95_return_on_max_loss,p99_return_on_max_loss,avg_terminal_intrinsic,worst_terminal_intrinsic,avg_short_strike_slippage,avg_long_strike_slippage
613,Z1p00_RSI55_VRP0p40_SEL_longest_tenor,103,31,91,60,12,0.300971,0.883495,0.833333,56,...,-0.154603,-0.022705,0.008234,0.030004,0.035489,0.043248,0.289670,10.48,2.717391,-6.296296
603,Z1p00_RSI55_VRP0p00_SEL_longest_tenor,106,32,94,62,12,0.301887,0.886792,0.837838,58,...,-0.153930,-0.013755,0.008465,0.030651,0.035401,0.043092,0.280426,10.48,2.604167,-6.296296
608,Z1p00_RSI55_VRP0p20_SEL_longest_tenor,106,32,94,62,12,0.301887,0.886792,0.837838,58,...,-0.153930,-0.013755,0.008465,0.030651,0.035401,0.043092,0.280426,10.48,2.604167,-6.296296
618,Z1p00_RSI55_VRPNONE_SEL_longest_tenor,106,32,94,62,12,0.301887,0.886792,0.837838,58,...,-0.153930,-0.013755,0.008465,0.030651,0.035401,0.043092,0.280426,10.48,2.604167,-6.296296
493,Z0p75_RSI55_VRP0p40_SEL_longest_tenor,150,64,136,72,14,0.426667,0.906667,0.837209,67,...,-0.157617,0.005074,0.008132,0.032969,0.036150,0.041783,0.274338,15.10,2.690909,-6.911765
373,Z0p50_RSI55_VRP0p40_SEL_longest_tenor,215,107,195,88,20,0.497674,0.906977,0.814815,84,...,-0.189400,-0.072287,0.005838,0.032063,0.036212,0.043344,0.408718,13.20,2.441176,-7.027027
483,Z0p75_RSI55_VRP0p00_SEL_longest_tenor,154,65,140,75,14,0.422078,0.909091,0.842697,70,...,-0.155654,0.005637,0.008186,0.033267,0.035959,0.041190,0.266500,15.10,2.655172,-6.911765
488,Z0p75_RSI55_VRP0p20_SEL_longest_tenor,154,65,140,75,14,0.422078,0.909091,0.842697,70,...,-0.155654,0.005637,0.008186,0.033267,0.035959,0.041190,0.266500,15.10,2.655172,-6.911765
498,Z0p75_RSI55_VRPNONE_SEL_longest_tenor,154,65,140,75,14,0.422078,0.909091,0.842697,70,...,-0.155654,0.005637,0.008186,0.033267,0.035959,0.041190,0.266500,15.10,2.655172,-6.911765
363,Z0p50_RSI55_VRP0p00_SEL_longest_tenor,221,104,201,97,20,0.470588,0.909502,0.829060,93,...,-0.179039,-0.071023,0.005506,0.031232,0.035197,0.041249,0.402537,13.20,2.467532,-7.027027



Still-incomplete spread preview


,selected_trade_id,trade_date,trade_date_str,trade_year,tenor,tenor_bucket,expiration_date,expiration_str,expiration_yyyymmdd,effective_dte,...,still_incomplete_after_fallback,effective_width_after_fallback,terminal_spread_intrinsic_after_fallback,entry_credit_conservative_after_fallback,credit_pct_width_after_fallback,max_loss_conservative_after_fallback,expiry_pnl_conservative_after_fallback,return_on_max_loss_conservative_after_fallback,return_on_width_conservative_after_fallback,trade_win_conservative_after_fallback
41,SPY_CALL_20210617_T9_EXP20210702_S433.0_L455.0,2021-06-17,2021-06-17,2021,9,Front,2021-07-02,2021-07-02,20210702,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
42,SPY_CALL_20210617_T12_EXP20210702_S435.0_L460.0,2021-06-17,2021-06-17,2021,12,Front,2021-07-02,2021-07-02,20210702,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
341,SPY_CALL_20230209_T9_EXP20230224_S422.0_L450.0,2023-02-09,2023-02-09,2023,9,Front,2023-02-24,2023-02-24,20230224,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
342,SPY_CALL_20230209_T12_EXP20230224_S424.0_L455.0,2023-02-09,2023-02-09,2023,12,Front,2023-02-24,2023-02-24,20230224,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
343,SPY_CALL_20230209_T18_EXP20230303_S426.0_L465.0,2023-02-09,2023-02-09,2023,18,Front,2023-03-03,2023-03-03,20230303,22.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1042,SPY_CALL_20241003_T33_EXP20241108_S605.0_L675.0,2024-10-03,2024-10-03,2024,33,Back,2024-11-08,2024-11-08,20241108,36.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1050,SPY_CALL_20241008_T30_EXP20241108_S609.0_L680.0,2024-10-08,2024-10-08,2024,30,Back,2024-11-08,2024-11-08,20241108,31.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1054,SPY_CALL_20241009_T30_EXP20241108_S612.0_L680.0,2024-10-09,2024-10-09,2024,30,Back,2024-11-08,2024-11-08,20241108,30.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1065,SPY_CALL_20241011_T27_EXP20241108_S612.0_L675.0,2024-10-11,2024-10-11,2024,27,Back,2024-11-08,2024-11-08,20241108,28.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1070,SPY_CALL_20241014_T24_EXP20241108_S614.0_L675.0,2024-10-14,2024-10-14,2024,24,Middle,2024-11-08,2024-11-08,20241108,25.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False



Saved
Checks CSV:                         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_mechanical_sanity_checks_repaired_rsi_long5_cal_v1_20260712_152806.csv
Global summary CSV:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_global_sanity_summary_repaired_rsi_long5_cal_v1_20260712_152806.csv
Coverage by tenor CSV:              C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_coverage_by_tenor_repaired_rsi_long5_cal_v1_20260712_152806.csv
Coverage by year CSV:               C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_coverage_by_year_repaired_rsi_long5_cal_v1_20260712_152806.csv
Coverage by selection rule CSV:     C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_coverage_by_selection_rule_repaired_rsi_long5_cal_v1_20260712_152806.csv
Rule coverage extremes CSV:         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02F7_rule_coverage_extremes_repaired_rsi_long5_cal_v1_20260

In [10]:
# Cell 2G.0 — DTE diagnostic for fixed parameters: z > 0.0 and RSI > 58
# Purpose:
#   See how each target tenor / DTE performs under the same signal filter:
#
#       z_3m > 0.0
#       z_1y > 0.0
#       RSI > 58
#
#   This is a DTE-only diagnostic:
#       9, 12, 15, 18, 21, 24, 27, 30, 33
#
# This cell does NOT:
#   - Use rule selection
#   - Rank final rules
#   - Select final parameters
#   - Apply Hybrid v2 put-spread sizing/selector
#
# Important:
#   Pricing is available only where the DTE candidate overlaps the already-priced repaired-RSI
#   selected-trade universe. The cell reports that coverage explicitly.

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "repaired_rsi_long5_cal_v1"

BASE_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_{SUFFIX}.parquet"
)

SPREADS_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_{SUFFIX}.parquet"
)

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

Z_THRESHOLD = 0.0
RSI_FLOOR = 58.0

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.0 — DTE diagnostic: z_3m > 0.0, z_1y > 0.0, RSI > 58")
print("=" * 100)
print(f"Base candidate panel:       {BASE_CANDIDATE_PANEL_PATH}")
print(f"Spreads with fallback P&L:  {SPREADS_WITH_FALLBACK_PATH}")
print(f"Target tenors:              {TARGET_TENORS}")
print(f"Z threshold:                > {Z_THRESHOLD}")
print(f"RSI floor:                  > {RSI_FLOOR}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def safe_divide(num, den):
    return np.where(
        pd.notna(num) & pd.notna(den) & (den != 0),
        num / den,
        np.nan,
    )

def first_existing_col(df, candidates, required=True, label=None):
    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise ValueError(
            f"Could not find required column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def make_selected_trade_id(trade_date, tenor, expiration_date, short_strike, long_strike):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"T{int(tenor)}_"
        f"EXP{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"S{float(short_strike):.1f}_"
        f"L{float(long_strike):.1f}"
    )

def summarize_group(g):
    matched = bool_series(g["matched_to_priced_universe"], index=g.index)
    complete = bool_series(g["spread_complete_after_fallback"], index=g.index)

    complete_g = g.loc[complete].copy()

    out = {
        "candidate_rows": int(len(g)),
        "matched_to_priced_universe": int(matched.sum()),
        "complete_priced_rows": int(complete.sum()),
        "unmatched_rows": int((~matched).sum()),
        "incomplete_after_fallback_rows": int((matched & ~complete).sum()),
        "match_rate_to_priced_universe": float(matched.mean()) if len(g) else np.nan,
        "complete_rate_vs_candidates": float(complete.mean()) if len(g) else np.nan,
        "complete_rate_vs_matched": float(complete.sum() / matched.sum()) if int(matched.sum()) > 0 else np.nan,
    }

    if len(complete_g) > 0:
        r = pd.to_numeric(complete_g["return_on_max_loss_conservative_after_fallback"], errors="coerce")

        out.update({
            "win_rate": float(complete_g["trade_win_conservative_after_fallback"].mean()),
            "avg_return_on_max_loss": float(r.mean()),
            "median_return_on_max_loss": float(r.median()),
            "p01_return_on_max_loss": float(r.quantile(0.01)),
            "p05_return_on_max_loss": float(r.quantile(0.05)),
            "p10_return_on_max_loss": float(r.quantile(0.10)),
            "p90_return_on_max_loss": float(r.quantile(0.90)),
            "p95_return_on_max_loss": float(r.quantile(0.95)),
            "best_return_on_max_loss": float(r.max()),
            "worst_return_on_max_loss": float(r.min()),
            "avg_entry_credit": float(complete_g["entry_credit_conservative_after_fallback"].mean()),
            "median_entry_credit": float(complete_g["entry_credit_conservative_after_fallback"].median()),
            "avg_credit_pct_width": float(complete_g["credit_pct_width_after_fallback"].mean()),
            "median_credit_pct_width": float(complete_g["credit_pct_width_after_fallback"].median()),
            "avg_effective_width": float(complete_g["effective_width_after_fallback"].mean()),
            "median_effective_width": float(complete_g["effective_width_after_fallback"].median()),
            "avg_terminal_intrinsic": float(complete_g["terminal_spread_intrinsic_after_fallback"].mean()),
            "worst_terminal_intrinsic": float(complete_g["terminal_spread_intrinsic_after_fallback"].max()),
            "short_fallback_rows": int(bool_series(complete_g.get("short_fallback_found"), index=complete_g.index).sum()),
            "long_fallback_rows": int(bool_series(complete_g.get("long_fallback_found"), index=complete_g.index).sum()),
            "any_fallback_rows": int(
                (
                    bool_series(complete_g.get("short_fallback_found"), index=complete_g.index)
                    | bool_series(complete_g.get("long_fallback_found"), index=complete_g.index)
                ).sum()
            ),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "p01_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p10_return_on_max_loss": np.nan,
            "p90_return_on_max_loss": np.nan,
            "p95_return_on_max_loss": np.nan,
            "best_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_effective_width": np.nan,
            "median_effective_width": np.nan,
            "avg_terminal_intrinsic": np.nan,
            "worst_terminal_intrinsic": np.nan,
            "short_fallback_rows": 0,
            "long_fallback_rows": 0,
            "any_fallback_rows": 0,
        })

    out["fallback_usage_rate_on_complete"] = (
        out["any_fallback_rows"] / out["complete_priced_rows"]
        if out["complete_priced_rows"] > 0
        else np.nan
    )

    return pd.Series(out)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

require_file(BASE_CANDIDATE_PANEL_PATH)
require_file(SPREADS_WITH_FALLBACK_PATH)

base = pd.read_parquet(BASE_CANDIDATE_PANEL_PATH).copy()
spreads = pd.read_parquet(SPREADS_WITH_FALLBACK_PATH).copy()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Base candidate rows:        {len(base):,}")
print(f"Priced spread rows:         {len(spreads):,}")
print()

# --------------------------------------------------------------------------------------------------
# Resolve columns
# --------------------------------------------------------------------------------------------------

trade_date_col = first_existing_col(base, ["trade_date"], label="trade_date")
tenor_col = first_existing_col(base, ["tenor"], label="tenor")
expiration_col = first_existing_col(base, ["expiration_date", "expiration"], label="expiration_date")

z3_col = first_existing_col(
    base,
    ["corsi_vrp_z_3m", "z_3m_final", "model_vrp_z_3m", "vrp_z_3m"],
    label="3m z-score",
)

z1_col = first_existing_col(
    base,
    ["corsi_vrp_z_1y", "z_1y_final", "model_vrp_z_1y", "vrp_z_1y"],
    label="1y z-score",
)

rsi_col = first_existing_col(
    base,
    ["rsi14", "rsi14_final", "spy_wilder_rsi14"],
    label="RSI14",
)

short_col = first_existing_col(
    base,
    ["short_strike", "short_call_1sd_strike", "short_call_strike", "short_call_1sd_listed"],
    label="short call listed strike",
)

long_col = first_existing_col(
    base,
    ["long_strike", "long_call_3sd_strike", "long_call_strike", "long_call_3sd_listed"],
    label="long call listed strike",
)

candidate_complete_col = first_existing_col(
    base,
    ["research_eligible", "candidate_inputs_complete"],
    required=False,
    label="candidate completeness",
)

print("=" * 100)
print("Resolved base columns")
print("=" * 100)
print(f"trade_date:          {trade_date_col}")
print(f"tenor:               {tenor_col}")
print(f"expiration_date:     {expiration_col}")
print(f"z_3m:                {z3_col}")
print(f"z_1y:                {z1_col}")
print(f"rsi14:               {rsi_col}")
print(f"short strike:        {short_col}")
print(f"long strike:         {long_col}")
print(f"candidate complete:  {candidate_complete_col}")
print()

# --------------------------------------------------------------------------------------------------
# Normalize base and build candidate IDs
# --------------------------------------------------------------------------------------------------

base = base.copy()

base["trade_date"] = pd.to_datetime(base[trade_date_col], errors="coerce").dt.normalize()
base["expiration_date"] = pd.to_datetime(base[expiration_col], errors="coerce").dt.normalize()
base["tenor"] = pd.to_numeric(base[tenor_col], errors="coerce").astype("Int64")

base["z_3m"] = pd.to_numeric(base[z3_col], errors="coerce")
base["z_1y"] = pd.to_numeric(base[z1_col], errors="coerce")
base["rsi14"] = pd.to_numeric(base[rsi_col], errors="coerce")
base["short_strike"] = pd.to_numeric(base[short_col], errors="coerce")
base["long_strike"] = pd.to_numeric(base[long_col], errors="coerce")

if candidate_complete_col is not None:
    base["candidate_complete"] = bool_series(base[candidate_complete_col], index=base.index)
else:
    base["candidate_complete"] = (
        base["trade_date"].notna()
        & base["expiration_date"].notna()
        & base["tenor"].notna()
        & base["z_3m"].notna()
        & base["z_1y"].notna()
        & base["rsi14"].notna()
        & base["short_strike"].notna()
        & base["long_strike"].notna()
    )

if "selected_trade_id" in base.columns:
    base["candidate_trade_id"] = base["selected_trade_id"].astype(str)
else:
    base["candidate_trade_id"] = [
        make_selected_trade_id(td, tenor, exp, s, l)
        if pd.notna(td) and pd.notna(tenor) and pd.notna(exp) and pd.notna(s) and pd.notna(l)
        else None
        for td, tenor, exp, s, l in zip(
            base["trade_date"],
            base["tenor"],
            base["expiration_date"],
            base["short_strike"],
            base["long_strike"],
        )
    ]

base["trade_year"] = base["trade_date"].dt.year

# --------------------------------------------------------------------------------------------------
# Filter fixed parameter set
# --------------------------------------------------------------------------------------------------

diagnostic_base = base.loc[
    base["candidate_complete"]
    & base["tenor"].isin(TARGET_TENORS)
    & (base["z_3m"] > Z_THRESHOLD)
    & (base["z_1y"] > Z_THRESHOLD)
    & (base["rsi14"] > RSI_FLOOR)
].copy()

diagnostic_base = diagnostic_base.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

print("=" * 100)
print("Filtered DTE diagnostic base")
print("=" * 100)
print(f"Filtered candidate rows: {len(diagnostic_base):,}")
print(f"Date range:              {diagnostic_base['trade_date'].min()} to {diagnostic_base['trade_date'].max()}")
print()
display(
    diagnostic_base
    .groupby("tenor", dropna=False)
    .agg(
        candidate_rows=("candidate_trade_id", "size"),
        first_date=("trade_date", "min"),
        last_date=("trade_date", "max"),
        avg_z_3m=("z_3m", "mean"),
        avg_z_1y=("z_1y", "mean"),
        avg_rsi14=("rsi14", "mean"),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Normalize spread outcomes and join
# --------------------------------------------------------------------------------------------------

spreads = spreads.copy()

spreads["selected_trade_id"] = spreads["selected_trade_id"].astype(str)

for c in ["trade_date", "expiration_date"]:
    if c in spreads.columns:
        spreads[c] = pd.to_datetime(spreads[c], errors="coerce").dt.normalize()

for c in [
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_fallback_found",
    "long_fallback_found",
    "trade_win_conservative_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

needed_spread_cols = [
    "selected_trade_id",
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_fallback_found",
    "long_fallback_found",
    "short_fallback_choice_type",
    "long_fallback_choice_type",
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
    "trade_win_conservative_after_fallback",
]

needed_spread_cols = [c for c in needed_spread_cols if c in spreads.columns]

dte_diag = diagnostic_base.merge(
    spreads[needed_spread_cols],
    left_on="candidate_trade_id",
    right_on="selected_trade_id",
    how="left",
    validate="many_to_one",
)

dte_diag["matched_to_priced_universe"] = dte_diag["selected_trade_id"].notna()

for c in [
    "spread_complete",
    "spread_complete_after_fallback",
    "newly_recovered_by_fallback",
    "still_incomplete_after_fallback",
    "short_fallback_found",
    "long_fallback_found",
    "trade_win_conservative_after_fallback",
]:
    if c in dte_diag.columns:
        dte_diag[c] = bool_series(dte_diag[c], index=dte_diag.index)
    else:
        dte_diag[c] = False

for c in [
    "short_effective_strike",
    "long_effective_strike",
    "effective_width_after_fallback",
    "terminal_spread_intrinsic_after_fallback",
    "entry_credit_conservative_after_fallback",
    "credit_pct_width_after_fallback",
    "max_loss_conservative_after_fallback",
    "expiry_pnl_conservative_after_fallback",
    "return_on_max_loss_conservative_after_fallback",
    "return_on_width_conservative_after_fallback",
]:
    if c in dte_diag.columns:
        dte_diag[c] = pd.to_numeric(dte_diag[c], errors="coerce")

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

summary_by_tenor = (
    dte_diag
    .groupby("tenor", dropna=False)
    .apply(summarize_group)
    .reset_index()
    .sort_values("tenor")
)

summary_by_tenor_year = (
    dte_diag
    .groupby(["tenor", "trade_year"], dropna=False)
    .apply(summarize_group)
    .reset_index()
    .sort_values(["tenor", "trade_year"])
)

summary_by_year = (
    dte_diag
    .groupby("trade_year", dropna=False)
    .apply(summarize_group)
    .reset_index()
    .sort_values("trade_year")
)

overall_summary = pd.DataFrame([summarize_group(dte_diag)])

unmatched_preview = (
    dte_diag
    .loc[~dte_diag["matched_to_priced_universe"]]
    .sort_values(["trade_date", "tenor"])
    .head(200)
    .copy()
)

still_incomplete_preview = (
    dte_diag
    .loc[dte_diag["matched_to_priced_universe"] & ~dte_diag["spread_complete_after_fallback"]]
    .sort_values(["trade_date", "tenor"])
    .head(200)
    .copy()
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

OUT_DETAIL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_detail_{SUFFIX}.parquet"
)

OUT_TENOR_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_tenor_{SUFFIX}.parquet"
)

OUT_TENOR_YEAR_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_tenor_year_{SUFFIX}.parquet"
)

OUT_YEAR_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_year_{SUFFIX}.parquet"
)

atomic_write_parquet(dte_diag, OUT_DETAIL_PATH)
atomic_write_parquet(summary_by_tenor, OUT_TENOR_SUMMARY_PATH)
atomic_write_parquet(summary_by_tenor_year, OUT_TENOR_YEAR_SUMMARY_PATH)
atomic_write_parquet(summary_by_year, OUT_YEAR_SUMMARY_PATH)

overall_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_overall_{SUFFIX}_{timestamp}.csv"
tenor_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_by_tenor_{SUFFIX}_{timestamp}.csv"
tenor_year_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_by_tenor_year_{SUFFIX}_{timestamp}.csv"
year_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_by_year_{SUFFIX}_{timestamp}.csv"
unmatched_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_unmatched_preview_{SUFFIX}_{timestamp}.csv"
still_incomplete_csv = AUDIT_DIR / f"02G0_dte_diag_z0_rsi58_still_incomplete_preview_{SUFFIX}_{timestamp}.csv"

atomic_write_csv(overall_summary, overall_csv)
atomic_write_csv(summary_by_tenor, tenor_csv)
atomic_write_csv(summary_by_tenor_year, tenor_year_csv)
atomic_write_csv(summary_by_year, year_csv)
atomic_write_csv(unmatched_preview, unmatched_csv)
atomic_write_csv(still_incomplete_preview, still_incomplete_csv)

manifest_path = AUDIT_DIR / f"02G0_dte_diagnostic_z0_rsi58_{SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.0",
    "description": "DTE diagnostic for fixed filter z_3m > 0.0, z_1y > 0.0, RSI > 58 across target tenors",
    "created_at": timestamp,
    "suffix": SUFFIX,
    "inputs": {
        "base_candidate_panel_path": str(BASE_CANDIDATE_PANEL_PATH),
        "spreads_with_fallback_path": str(SPREADS_WITH_FALLBACK_PATH),
    },
    "filter": {
        "target_tenors": TARGET_TENORS,
        "z_3m": f"> {Z_THRESHOLD}",
        "z_1y": f"> {Z_THRESHOLD}",
        "rsi14": f"> {RSI_FLOOR}",
        "vrp_log_floor": None,
    },
    "outputs": {
        "detail_path": str(OUT_DETAIL_PATH),
        "summary_by_tenor_path": str(OUT_TENOR_SUMMARY_PATH),
        "summary_by_tenor_year_path": str(OUT_TENOR_YEAR_SUMMARY_PATH),
        "summary_by_year_path": str(OUT_YEAR_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "counts": {
        "filtered_candidate_rows": int(len(dte_diag)),
        "matched_to_priced_universe": int(dte_diag["matched_to_priced_universe"].sum()),
        "complete_after_fallback": int(dte_diag["spread_complete_after_fallback"].sum()),
        "unmatched_rows": int((~dte_diag["matched_to_priced_universe"]).sum()),
        "matched_but_incomplete_rows": int(
            (dte_diag["matched_to_priced_universe"] & ~dte_diag["spread_complete_after_fallback"]).sum()
        ),
    },
    "notes": {
        "diagnostic_type": "DTE-only fixed-parameter diagnostic; no selection rule used.",
        "pricing_scope": "Performance is shown where the all-DTE candidate overlaps already-priced repaired-RSI selected-trade universe.",
        "next_step_if_unmatched_large": "Build and price missing all-DTE candidates for a fully pure DTE comparison.",
    },
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.0 complete — DTE diagnostic: z > 0.0 and RSI > 58")
print("=" * 100)

print()
print("Overall summary")
display(overall_summary)

print()
print("=" * 100)
print("Summary by tenor / DTE")
print("=" * 100)
display(summary_by_tenor)

print()
print("=" * 100)
print("Summary by year")
print("=" * 100)
display(summary_by_year)

print()
print("=" * 100)
print("Summary by tenor/year")
print("=" * 100)
display(summary_by_tenor_year)

print()
print("=" * 100)
print("Unmatched-to-priced-universe preview")
print("=" * 100)
if unmatched_preview.empty:
    print("No unmatched rows. All filtered DTE candidates are available in priced universe.")
else:
    display(unmatched_preview)

print()
print("=" * 100)
print("Matched but still incomplete after fallback preview")
print("=" * 100)
if still_incomplete_preview.empty:
    print("No matched-but-incomplete rows.")
else:
    display(still_incomplete_preview)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Detail:                  {OUT_DETAIL_PATH}")
print(f"Summary by tenor:         {OUT_TENOR_SUMMARY_PATH}")
print(f"Summary by tenor/year:    {OUT_TENOR_YEAR_SUMMARY_PATH}")
print(f"Summary by year:          {OUT_YEAR_SUMMARY_PATH}")
print(f"Manifest:                 {manifest_path}")

print()
print("Result: 2G.0 DTE diagnostic complete.")

Cell 2G.0 — DTE diagnostic: z_3m > 0.0, z_1y > 0.0, RSI > 58
Base candidate panel:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet
Spreads with fallback P&L:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_unique_selected_trade_universe_priced_with_fallback_repaired_rsi_long5_cal_v1.parquet
Target tenors:              [9, 12, 15, 18, 21, 24, 27, 30, 33]
Z threshold:                > 0.0
RSI floor:                  > 58.0

Loaded inputs
Base candidate rows:        14,724
Priced spread rows:         1,734

Resolved base columns
trade_date:          trade_date
tenor:               tenor
expiration_date:     expiration_date
z_3m:                corsi_vrp_z_3m
z_1y:                corsi_vrp_z_1y
rsi14:               rsi14
short strike:        short_strike
long strike:         long_strike
candidate complete:  research_eligible

Filtered DTE di

,tenor,candidate_rows,first_date,last_date,avg_z_3m,avg_z_1y,avg_rsi14
0,9,249,2020-12-31,2026-05-14,0.936604,0.845740,66.530945
1,12,241,2020-12-31,2026-05-14,0.913457,0.863338,66.349828
2,15,231,2021-01-25,2026-06-02,0.889416,0.886317,66.730031
3,18,223,2020-12-31,2026-06-02,0.917289,0.921846,66.716384
4,21,214,2020-12-31,2026-06-02,0.948404,0.989429,66.913643
5,24,205,2020-12-31,2026-06-02,0.988641,1.033183,66.913950
6,27,200,2020-12-31,2026-06-02,0.988811,1.071823,66.883999
7,30,191,2020-12-31,2026-06-02,0.997715,1.084636,66.683953
8,33,187,2020-12-31,2026-06-02,0.989196,1.104065,66.778345


C:\Users\patri\AppData\Local\Temp\ipykernel_14668\3322908869.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\3322908869.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\3322908869.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future


Cell 2G.0 complete — DTE diagnostic: z > 0.0 and RSI > 58

Overall summary


C:\Users\patri\AppData\Local\Temp\ipykernel_14668\3322908869.py:487: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_14668\3322908869.py:495: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_group)


...


Unmatched-to-priced-universe preview


,trade_date,tenor,tenor_bucket,spy_close,vix_style_vol_pct,implied_variance,forecast_variance,forecast_vol_pct,corsi_vrp_log,corsi_vrp_z_3m,...,effective_width_after_fallback,terminal_spread_intrinsic_after_fallback,entry_credit_conservative_after_fallback,credit_pct_width_after_fallback,max_loss_conservative_after_fallback,expiry_pnl_conservative_after_fallback,return_on_max_loss_conservative_after_fallback,return_on_width_conservative_after_fallback,trade_win_conservative_after_fallback,matched_to_priced_universe
1,2020-12-31,12,Front,373.88,17.751876,0.031513,0.008140,9.022023,1.353645,0.572945,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,2020-12-31,18,Front,373.88,19.587186,0.038366,0.009290,9.638721,1.418174,0.753331,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,2020-12-31,21,Middle,373.88,21.298720,0.045364,0.009940,9.970143,1.518104,1.486967,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,2020-12-31,24,Middle,373.88,22.000447,0.048402,0.010268,10.133068,1.550517,1.761020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
6,2020-12-31,30,Back,373.88,22.573620,0.050957,0.010965,10.471598,1.536231,1.457979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,2024-03-25,21,Middle,519.77,12.706544,0.016146,0.010611,10.301150,0.419723,0.864306,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
678,2024-03-25,30,Back,519.77,13.165562,0.017333,0.012649,11.246599,0.315077,1.039480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
687,2024-03-26,30,Back,518.81,13.383900,0.017913,0.013061,11.428621,0.315863,1.016417,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
693,2024-03-28,24,Middle,523.07,12.753775,0.016266,0.011960,10.936117,0.307513,0.358432,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False



Matched but still incomplete after fallback preview


,trade_date,tenor,tenor_bucket,spy_close,vix_style_vol_pct,implied_variance,forecast_variance,forecast_vol_pct,corsi_vrp_log,corsi_vrp_z_3m,...,effective_width_after_fallback,terminal_spread_intrinsic_after_fallback,entry_credit_conservative_after_fallback,credit_pct_width_after_fallback,max_loss_conservative_after_fallback,expiry_pnl_conservative_after_fallback,return_on_max_loss_conservative_after_fallback,return_on_width_conservative_after_fallback,trade_win_conservative_after_fallback,matched_to_priced_universe
1046,2024-10-08,30,Back,573.17,21.453076,0.046023,0.018431,13.575991,0.915130,2.514771,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1055,2024-10-09,30,Back,577.14,20.880061,0.043598,0.017984,13.410494,0.885514,2.274200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1071,2024-10-11,27,Back,579.58,20.279973,0.041128,0.017020,13.046089,0.882291,2.058550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1079,2024-10-14,24,Middle,584.32,19.718129,0.038880,0.015069,12.275641,0.947843,2.353106,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1082,2024-10-14,33,Back,584.32,19.735170,0.038948,0.016200,12.727827,0.877223,1.850027,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1091,2024-10-15,33,Back,579.78,20.607302,0.042466,0.017192,13.111865,0.904256,1.865788,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1099,2024-10-16,33,Back,582.30,19.627784,0.038525,0.017278,13.144605,0.801869,1.434259,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1107,2024-10-17,33,Back,582.35,19.183568,0.036801,0.017119,13.083965,0.765333,1.259402,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1141,2024-10-28,33,Back,580.83,19.539475,0.038179,0.019568,13.988729,0.668370,0.725473,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1150,2024-10-29,33,Back,581.77,19.529818,0.038141,0.018780,13.704092,0.708496,0.844758,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True



Saved
Detail:                  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_detail_repaired_rsi_long5_cal_v1.parquet
Summary by tenor:         C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_tenor_repaired_rsi_long5_cal_v1.parquet
Summary by tenor/year:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_tenor_year_repaired_rsi_long5_cal_v1.parquet
Summary by year:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_z0_rsi58_summary_by_year_repaired_rsi_long5_cal_v1.parquet
Manifest:                 C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G0_dte_diagnostic_z0_rsi58_repaired_rsi_long5_cal_v1_manifest_20260712_185030.json

Result: 2G.0 DTE diagnostic complete.


In [12]:
# Cell 2G.DATA.1 — Build full historical 1SD / 3SD call contract request plan
# Purpose:
#   Build a reusable data-layer request plan for SPY call premiums across all historical dates
#   and target tenors.
#
# For every trade_date x target_tenor row with the required market inputs:
#   - use SPY close
#   - use the VIX-style vol measure for that tenor
#   - calculate 1SD upside call strike
#   - calculate 3SD upside call strike
#   - map the holiday-aware expiration date
#   - build unique SPY call contract request IDs
#
# This cell has no trade-signal filters and makes no API calls.

from pathlib import Path
from datetime import datetime
import json
import math

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SPY_EOD_PATH = PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_eod_prices_v1.parquet"

# Leave as None to auto-select the widest usable panel from the candidates below.
SOURCE_PANEL_OVERRIDE = None

SOURCE_PANEL_CANDIDATES = [
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet",
    PROCESSED_DIR / "call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet",
    PROJECT_ROOT / "data" / "processed" / "vrp_repaired_wilder_rsi_signal" / "vrp_repaired_wilder_rsi_signal_base_v1.parquet",
    PROJECT_ROOT / "data" / "processed" / "vrp_final_signal" / "vrp_final_corsi_signal_base_panel_v1.parquet",
]

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

# Current call-sleeve strike convention.
SHORT_STRIKE_INCREMENT = 1.0
LONG_STRIKE_INCREMENT = 5.0

# Use tenor itself in the SD move:
#   move = vix_style_vol_pct / 100 * sqrt(tenor / 365)
DAYS_IN_YEAR = 365.0

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Existing caches to reuse for coverage diagnostics only.
CACHE_SOURCES = [
    {
        "source_name": "repaired_primary_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet",
        "priority": 1,
    },
    {
        "source_name": "repaired_fallback_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_repaired_rsi_long5_cal_v1.parquet",
        "priority": 2,
    },
    {
        "source_name": "old_long5_cal_primary_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet",
        "priority": 3,
    },
    {
        "source_name": "old_long5_cal_fallback_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet",
        "priority": 4,
    },
    {
        "source_name": "fullhist_primary_cache_existing",
        "path": PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_primary_contract_price_cache_{OUT_SUFFIX}.parquet",
        "priority": 0,
    },
]

print("=" * 100)
print("Cell 2G.DATA.1 — Full historical 1SD / 3SD call contract request plan")
print("=" * 100)
print(f"Target tenors: {TARGET_TENORS}")
print(f"SPY EOD path:  {SPY_EOD_PATH}")
print("No trade-signal filters are applied.")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def first_existing_col(df, candidates, required=True, label=None):
    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise ValueError(
            f"Could not find required column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def ceil_to_increment(x, increment):
    return np.ceil(np.asarray(x, dtype=float) / increment) * increment

def round_half_up_to_increment(x, increment):
    return np.floor(np.asarray(x, dtype=float) / increment + 0.5) * increment

def make_trade_id(trade_date, tenor, expiration_date, short_strike, long_strike):
    return (
        f"SPY_CALL_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"T{int(tenor)}_"
        f"EXP{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"S{float(short_strike):.1f}_"
        f"L{float(long_strike):.1f}"
    )

def make_contract_request_id(trade_date, expiration_date, strike):
    return (
        f"SPY_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_C"
    )

def first_friday_on_or_after(date_value):
    d = pd.Timestamp(date_value).normalize()
    days_to_friday = (4 - d.weekday()) % 7
    return d + pd.Timedelta(days=int(days_to_friday))

def map_expiration(trade_date, tenor, trading_days):
    trade_date = pd.Timestamp(trade_date).normalize()

    if pd.isna(trade_date) or pd.isna(tenor):
        return pd.NaT

    target_date = trade_date + pd.Timedelta(days=int(tenor))
    nominal_friday = first_friday_on_or_after(target_date)

    eligible = trading_days[(trading_days <= nominal_friday) & (trading_days > trade_date)]

    if len(eligible) == 0:
        return pd.NaT

    exp_date = eligible[-1]

    # Friday-holiday adjustment should normally land within 0-3 calendar days before nominal Friday.
    if (nominal_friday - exp_date).days > 3:
        return pd.NaT

    return exp_date

def normalize_cache(df, source_name, priority):
    out = df.copy()

    if "contract_request_id" not in out.columns:
        raise ValueError(f"{source_name} missing contract_request_id.")

    out["contract_request_id"] = out["contract_request_id"].astype(str)
    out["cache_source"] = source_name
    out["cache_priority"] = int(priority)

    for c in ["bid", "ask", "mid", "price"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan

    if "price_found" in out.columns:
        out["price_found"] = bool_series(out["price_found"], index=out.index)
    else:
        out["price_found"] = out["bid"].notna() & out["ask"].notna()

    out["bid_usable"] = out["bid"].notna() & (out["bid"] >= 0)
    out["ask_usable"] = out["ask"].notna() & (out["ask"] >= 0)
    out["both_sides_usable"] = out["bid_usable"] & out["ask_usable"]

    if "status" not in out.columns:
        out["status"] = np.nan

    if "status_code" not in out.columns:
        out["status_code"] = np.nan

    keep_cols = [
        "contract_request_id",
        "cache_source",
        "cache_priority",
        "price_found",
        "bid_usable",
        "ask_usable",
        "both_sides_usable",
        "bid",
        "ask",
        "mid",
        "status",
        "status_code",
    ]

    return out[keep_cols].copy()

def inspect_source_panel(path):
    path = Path(path)

    if not path.exists():
        return {
            "path": str(path),
            "exists": False,
            "usable": False,
            "reason": "file_not_found",
        }

    try:
        df = pd.read_parquet(path)
    except Exception as e:
        return {
            "path": str(path),
            "exists": True,
            "usable": False,
            "reason": f"read_error: {e}",
        }

    trade_col = first_existing_col(df, ["trade_date", "date"], required=False)
    tenor_col = first_existing_col(df, ["tenor", "dte", "target_tenor"], required=False)
    vol_col = first_existing_col(
        df,
        [
            "vix_style_vol_pct",
            "vix_style_vol_final",
            "vix_style_vol",
            "vix_vol_pct",
            "implied_vol_pct",
        ],
        required=False,
    )
    spy_col = first_existing_col(df, ["spy_close", "underlying_close", "close"], required=False)

    if trade_col is None or tenor_col is None or vol_col is None:
        return {
            "path": str(path),
            "exists": True,
            "usable": False,
            "reason": "missing trade_date/tenor/vix_style_vol columns",
            "rows": int(len(df)),
            "columns": list(df.columns),
        }

    tmp = df[[trade_col, tenor_col, vol_col] + ([spy_col] if spy_col is not None else [])].copy()
    tmp["trade_date"] = pd.to_datetime(tmp[trade_col], errors="coerce").dt.normalize()
    tmp["tenor"] = pd.to_numeric(tmp[tenor_col], errors="coerce")
    tmp["vol"] = pd.to_numeric(tmp[vol_col], errors="coerce")

    tmp = tmp.loc[
        tmp["trade_date"].notna()
        & tmp["tenor"].isin(TARGET_TENORS)
        & tmp["vol"].notna()
    ].copy()

    if tmp.empty:
        return {
            "path": str(path),
            "exists": True,
            "usable": False,
            "reason": "no usable target-tenor rows",
            "rows": int(len(df)),
        }

    return {
        "path": str(path),
        "exists": True,
        "usable": True,
        "reason": "usable",
        "rows": int(len(df)),
        "target_tenor_rows": int(len(tmp)),
        "first_date": tmp["trade_date"].min(),
        "last_date": tmp["trade_date"].max(),
        "date_span_days": int((tmp["trade_date"].max() - tmp["trade_date"].min()).days),
        "trade_col": trade_col,
        "tenor_col": tenor_col,
        "vol_col": vol_col,
        "spy_col": spy_col,
    }

# --------------------------------------------------------------------------------------------------
# Load SPY EOD / trading calendar
# --------------------------------------------------------------------------------------------------

require_file(SPY_EOD_PATH)

spy = pd.read_parquet(SPY_EOD_PATH).copy()

spy_date_col = first_existing_col(spy, ["date", "trade_date"], label="SPY EOD date")
spy_close_col = first_existing_col(spy, ["close", "spy_close", "adj_close"], label="SPY EOD close")

spy = spy[[spy_date_col, spy_close_col]].copy()
spy.columns = ["trade_date", "spy_close_eod"]
spy["trade_date"] = pd.to_datetime(spy["trade_date"], errors="coerce").dt.normalize()
spy["spy_close_eod"] = pd.to_numeric(spy["spy_close_eod"], errors="coerce")
spy = spy.dropna(subset=["trade_date", "spy_close_eod"]).drop_duplicates("trade_date")
spy = spy.sort_values("trade_date").reset_index(drop=True)

trading_days = pd.DatetimeIndex(spy["trade_date"].sort_values().unique())

print("=" * 100)
print("Loaded SPY EOD calendar")
print("=" * 100)
print(f"SPY EOD rows: {len(spy):,}")
print(f"Date range:   {spy['trade_date'].min()} to {spy['trade_date'].max()}")
print()

# --------------------------------------------------------------------------------------------------
# Select source panel
# --------------------------------------------------------------------------------------------------

if SOURCE_PANEL_OVERRIDE is not None:
    source_paths = [Path(SOURCE_PANEL_OVERRIDE)]
else:
    source_paths = [Path(p) for p in SOURCE_PANEL_CANDIDATES]

source_inspections = pd.DataFrame([inspect_source_panel(p) for p in source_paths])

print("=" * 100)
print("Source panel inspection")
print("=" * 100)
display(source_inspections.drop(columns=["columns"], errors="ignore"))

usable_sources = source_inspections.loc[source_inspections["usable"] == True].copy()

if usable_sources.empty:
    raise ValueError("No usable source panel found. Check SOURCE_PANEL_CANDIDATES or set SOURCE_PANEL_OVERRIDE.")

usable_sources = usable_sources.sort_values(
    ["date_span_days", "target_tenor_rows"],
    ascending=[False, False],
).reset_index(drop=True)

selected_source = usable_sources.iloc[0].to_dict()
SOURCE_PANEL_PATH = Path(selected_source["path"])

print()
print("=" * 100)
print("Selected source panel")
print("=" * 100)
print(f"Path:       {SOURCE_PANEL_PATH}")
print(f"Rows:       {selected_source.get('rows'):,}")
print(f"Target rows:{selected_source.get('target_tenor_rows'):,}")
print(f"Date range: {selected_source.get('first_date')} to {selected_source.get('last_date')}")
print(f"Trade col:  {selected_source.get('trade_col')}")
print(f"Tenor col:  {selected_source.get('tenor_col')}")
print(f"Vol col:    {selected_source.get('vol_col')}")
print(f"SPY col:    {selected_source.get('spy_col')}")
print()

# --------------------------------------------------------------------------------------------------
# Build canonical full historical tenor panel
# --------------------------------------------------------------------------------------------------

raw = pd.read_parquet(SOURCE_PANEL_PATH).copy()

trade_col = selected_source["trade_col"]
tenor_col = selected_source["tenor_col"]
vol_col = selected_source["vol_col"]
spy_col = selected_source.get("spy_col")

cols = [trade_col, tenor_col, vol_col]
if spy_col is not None and isinstance(spy_col, str):
    cols.append(spy_col)

panel = raw[cols].copy()
panel = panel.rename(
    columns={
        trade_col: "trade_date",
        tenor_col: "tenor",
        vol_col: "vix_style_vol_input",
    }
)

if spy_col is not None and isinstance(spy_col, str):
    panel = panel.rename(columns={spy_col: "spy_close_source"})

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")
panel["vix_style_vol_input"] = pd.to_numeric(panel["vix_style_vol_input"], errors="coerce")

panel = panel.loc[
    panel["trade_date"].notna()
    & panel["tenor"].isin(TARGET_TENORS)
    & panel["vix_style_vol_input"].notna()
].copy()

# Normalize vol to percent units.
# If source is decimal vol, convert to percent; if already percent, keep.
median_vol = panel["vix_style_vol_input"].median()
if pd.notna(median_vol) and median_vol <= 1.5:
    panel["vix_style_vol_pct"] = panel["vix_style_vol_input"] * 100.0
    vol_unit_interpretation = "input appeared decimal; converted to percent"
else:
    panel["vix_style_vol_pct"] = panel["vix_style_vol_input"]
    vol_unit_interpretation = "input appeared percent; kept as percent"

# Join canonical SPY close.
panel = panel.merge(spy, on="trade_date", how="left", validate="many_to_one")

if "spy_close_source" in panel.columns:
    panel["spy_close_source"] = pd.to_numeric(panel["spy_close_source"], errors="coerce")
    panel["spy_close"] = panel["spy_close_source"].where(panel["spy_close_source"].notna(), panel["spy_close_eod"])
else:
    panel["spy_close"] = panel["spy_close_eod"]

# Drop duplicate date/tenor rows if any.
before_dedupe = len(panel)
panel = (
    panel
    .sort_values(["trade_date", "tenor"])
    .drop_duplicates(["trade_date", "tenor"], keep="last")
    .reset_index(drop=True)
)
duplicate_rows_removed = before_dedupe - len(panel)

# Expiration mapping.
panel["expiration_date"] = [
    map_expiration(td, tenor, trading_days)
    for td, tenor in zip(panel["trade_date"], panel["tenor"])
]

# Expiration close for later P&L.
expiry_close = spy.rename(columns={"trade_date": "expiration_date", "spy_close_eod": "expiry_spy_close"})
panel = panel.merge(expiry_close, on="expiration_date", how="left", validate="many_to_one")

# Mechanical rows for contract request planning.
panel["has_required_market_inputs"] = (
    panel["trade_date"].notna()
    & panel["tenor"].notna()
    & panel["spy_close"].notna()
    & panel["vix_style_vol_pct"].notna()
    & (panel["vix_style_vol_pct"] > 0)
    & panel["expiration_date"].notna()
)

universe = panel.loc[panel["has_required_market_inputs"]].copy()

# Strike construction.
universe["sd_move"] = (
    universe["vix_style_vol_pct"] / 100.0
    * np.sqrt(universe["tenor"].astype(float) / DAYS_IN_YEAR)
)

universe["short_call_1sd_raw"] = universe["spy_close"] * (1.0 + universe["sd_move"])
universe["long_call_3sd_raw"] = universe["spy_close"] * (1.0 + 3.0 * universe["sd_move"])

universe["short_strike"] = ceil_to_increment(universe["short_call_1sd_raw"], SHORT_STRIKE_INCREMENT)
universe["long_strike"] = round_half_up_to_increment(universe["long_call_3sd_raw"], LONG_STRIKE_INCREMENT)

# Ensure long strike is strictly above short strike.
bad_width = universe["long_strike"] <= universe["short_strike"]
if bad_width.any():
    universe.loc[bad_width, "long_strike"] = (
        np.floor(universe.loc[bad_width, "short_strike"] / LONG_STRIKE_INCREMENT + 1.0)
        * LONG_STRIKE_INCREMENT
    )

universe["width"] = universe["long_strike"] - universe["short_strike"]
universe["outcome_available"] = universe["expiry_spy_close"].notna()

universe["trade_year"] = universe["trade_date"].dt.year
universe["trade_date_str"] = universe["trade_date"].dt.strftime("%Y-%m-%d")
universe["expiration_str"] = universe["expiration_date"].dt.strftime("%Y-%m-%d")

universe["selected_trade_id"] = [
    make_trade_id(td, tenor, exp, s, l)
    for td, tenor, exp, s, l in zip(
        universe["trade_date"],
        universe["tenor"],
        universe["expiration_date"],
        universe["short_strike"],
        universe["long_strike"],
    )
]

universe["short_contract_request_id"] = [
    make_contract_request_id(td, exp, s)
    for td, exp, s in zip(
        universe["trade_date"],
        universe["expiration_date"],
        universe["short_strike"],
    )
]

universe["long_contract_request_id"] = [
    make_contract_request_id(td, exp, l)
    for td, exp, l in zip(
        universe["trade_date"],
        universe["expiration_date"],
        universe["long_strike"],
    )
]

dup_trade_ids = int(universe["selected_trade_id"].duplicated().sum())
if dup_trade_ids:
    raise ValueError(f"Duplicate selected_trade_id rows found: {dup_trade_ids}")

print("=" * 100)
print("Full historical 1SD / 3SD universe")
print("=" * 100)
print(f"Raw source rows:              {len(raw):,}")
print(f"Target-tenor panel rows:      {len(panel):,}")
print(f"Duplicate date/tenor removed: {duplicate_rows_removed:,}")
print(f"Request-universe rows:        {len(universe):,}")
print(f"Rows with outcome available:  {int(universe['outcome_available'].sum()):,}")
print(f"Date range:                   {universe['trade_date'].min()} to {universe['trade_date'].max()}")
print(f"Vol unit handling:            {vol_unit_interpretation}")
print()

date_span_years = (universe["trade_date"].max() - universe["trade_date"].min()).days / 365.25
if date_span_years < 7.0:
    print("WARNING: Selected source covers less than ~7 years.")
    print("If you expected a longer sample, point SOURCE_PANEL_OVERRIDE to the older VIX-style tenor panel.")
    print()

universe_by_tenor = (
    universe
    .groupby("tenor", dropna=False)
    .agg(
        rows=("selected_trade_id", "size"),
        first_date=("trade_date", "min"),
        last_date=("trade_date", "max"),
        outcome_available=("outcome_available", "sum"),
        avg_spy_close=("spy_close", "mean"),
        avg_vix_style_vol_pct=("vix_style_vol_pct", "mean"),
        avg_short_strike=("short_strike", "mean"),
        avg_long_strike=("long_strike", "mean"),
        avg_width=("width", "mean"),
    )
    .reset_index()
)

display(universe_by_tenor)

# --------------------------------------------------------------------------------------------------
# Build leg request plan and unique contract request plan
# --------------------------------------------------------------------------------------------------

short_legs = universe.copy()
short_legs["leg_label"] = "short_call_1sd"
short_legs["side"] = "sell"
short_legs["needed_price_field"] = "bid"
short_legs["contract_request_id"] = short_legs["short_contract_request_id"]
short_legs["strike"] = short_legs["short_strike"]

long_legs = universe.copy()
long_legs["leg_label"] = "long_call_3sd"
long_legs["side"] = "buy"
long_legs["needed_price_field"] = "ask"
long_legs["contract_request_id"] = long_legs["long_contract_request_id"]
long_legs["strike"] = long_legs["long_strike"]

leg_cols = [
    "selected_trade_id",
    "trade_date",
    "trade_date_str",
    "trade_year",
    "tenor",
    "expiration_date",
    "expiration_str",
    "outcome_available",
    "leg_label",
    "side",
    "needed_price_field",
    "contract_request_id",
    "strike",
    "short_strike",
    "long_strike",
    "spy_close",
    "vix_style_vol_pct",
    "sd_move",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "width",
]

leg_request_plan = pd.concat(
    [short_legs[leg_cols], long_legs[leg_cols]],
    ignore_index=True,
).sort_values(["trade_date", "tenor", "leg_label"]).reset_index(drop=True)

unique_contract_request_plan = (
    leg_request_plan
    .groupby("contract_request_id", dropna=False)
    .agg(
        trade_date=("trade_date", "first"),
        expiration_date=("expiration_date", "first"),
        strike=("strike", "first"),
        leg_use_count=("selected_trade_id", "size"),
        used_as_short=("leg_label", lambda x: int((x == "short_call_1sd").sum())),
        used_as_long=("leg_label", lambda x: int((x == "long_call_3sd").sum())),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
    )
    .reset_index()
    .sort_values(["trade_date", "expiration_date", "strike"])
    .reset_index(drop=True)
)

print("=" * 100)
print("Contract request plan")
print("=" * 100)
print(f"Spread rows:              {len(universe):,}")
print(f"Leg rows before dedupe:   {len(leg_request_plan):,}")
print(f"Unique contract requests: {len(unique_contract_request_plan):,}")
print(f"Unique short contracts:   {int((unique_contract_request_plan['used_as_short'] > 0).sum()):,}")
print(f"Unique long contracts:    {int((unique_contract_request_plan['used_as_long'] > 0).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Existing cache overlap diagnostics
# --------------------------------------------------------------------------------------------------

cache_frames = []
cache_load_rows = []

for src in CACHE_SOURCES:
    path = Path(src["path"])
    source_name = src["source_name"]
    priority = int(src["priority"])

    if path.exists():
        raw_cache = pd.read_parquet(path).copy()
        norm = normalize_cache(raw_cache, source_name, priority)
        cache_frames.append(norm)

        cache_load_rows.append({
            "cache_source": source_name,
            "path": str(path),
            "exists": True,
            "raw_rows": int(len(raw_cache)),
            "unique_contracts": int(norm["contract_request_id"].nunique()),
            "bid_usable_rows": int(norm["bid_usable"].sum()),
            "ask_usable_rows": int(norm["ask_usable"].sum()),
            "both_sides_usable_rows": int(norm["both_sides_usable"].sum()),
        })
    else:
        cache_load_rows.append({
            "cache_source": source_name,
            "path": str(path),
            "exists": False,
            "raw_rows": 0,
            "unique_contracts": 0,
            "bid_usable_rows": 0,
            "ask_usable_rows": 0,
            "both_sides_usable_rows": 0,
        })

cache_load_summary = pd.DataFrame(cache_load_rows)

if cache_frames:
    combined_cache_raw = pd.concat(cache_frames, ignore_index=True)
else:
    combined_cache_raw = pd.DataFrame(
        columns=[
            "contract_request_id",
            "cache_source",
            "cache_priority",
            "price_found",
            "bid_usable",
            "ask_usable",
            "both_sides_usable",
            "bid",
            "ask",
            "mid",
            "status",
            "status_code",
        ]
    )

combined_cache_raw["contract_request_id"] = combined_cache_raw["contract_request_id"].astype(str)
combined_cache_raw["best_sort"] = np.where(combined_cache_raw["both_sides_usable"], 0, 1)

combined_cache_best = (
    combined_cache_raw
    .sort_values(["contract_request_id", "best_sort", "cache_priority"])
    .drop_duplicates("contract_request_id", keep="first")
    .drop(columns=["best_sort"])
    .reset_index(drop=True)
)

contract_cache_overlap = unique_contract_request_plan.merge(
    combined_cache_best,
    on="contract_request_id",
    how="left",
    validate="one_to_one",
)

contract_cache_overlap["found_in_any_cache"] = contract_cache_overlap["cache_source"].notna()
contract_cache_overlap["bid_usable"] = bool_series(contract_cache_overlap.get("bid_usable"), index=contract_cache_overlap.index)
contract_cache_overlap["ask_usable"] = bool_series(contract_cache_overlap.get("ask_usable"), index=contract_cache_overlap.index)
contract_cache_overlap["both_sides_usable"] = bool_series(contract_cache_overlap.get("both_sides_usable"), index=contract_cache_overlap.index)

contract_cache_overlap["bid_needed"] = contract_cache_overlap["used_as_short"] > 0
contract_cache_overlap["ask_needed"] = contract_cache_overlap["used_as_long"] > 0

contract_cache_overlap["sufficient_for_all_uses"] = (
    (~contract_cache_overlap["bid_needed"] | contract_cache_overlap["bid_usable"])
    & (~contract_cache_overlap["ask_needed"] | contract_cache_overlap["ask_usable"])
)

contract_cache_overlap["needs_exact_primary_query"] = ~contract_cache_overlap["sufficient_for_all_uses"]

contract_overlap_summary = pd.DataFrame([{
    "unique_contract_requests": int(len(contract_cache_overlap)),
    "found_in_any_cache": int(contract_cache_overlap["found_in_any_cache"].sum()),
    "sufficient_from_existing_cache": int(contract_cache_overlap["sufficient_for_all_uses"].sum()),
    "needs_exact_primary_query": int(contract_cache_overlap["needs_exact_primary_query"].sum()),
    "cache_hit_rate_any": float(contract_cache_overlap["found_in_any_cache"].mean()),
    "sufficient_cache_hit_rate": float(contract_cache_overlap["sufficient_for_all_uses"].mean()),
}])

cache_source_overlap_summary = (
    contract_cache_overlap
    .assign(cache_source=contract_cache_overlap["cache_source"].fillna("not_in_cache"))
    .groupby("cache_source", dropna=False)
    .agg(
        contracts=("contract_request_id", "size"),
        sufficient_for_all_uses=("sufficient_for_all_uses", "sum"),
        needs_exact_primary_query=("needs_exact_primary_query", "sum"),
    )
    .reset_index()
    .sort_values(["sufficient_for_all_uses", "contracts"], ascending=[False, False])
)

leg_cache_coverage = leg_request_plan.merge(
    contract_cache_overlap[
        [
            "contract_request_id",
            "found_in_any_cache",
            "cache_source",
            "bid_usable",
            "ask_usable",
            "bid",
            "ask",
            "mid",
            "sufficient_for_all_uses",
            "needs_exact_primary_query",
        ]
    ],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_cache_coverage["leg_price_usable"] = np.where(
    leg_cache_coverage["needed_price_field"].eq("bid"),
    leg_cache_coverage["bid_usable"],
    leg_cache_coverage["ask_usable"],
).astype(bool)

leg_summary_by_label = (
    leg_cache_coverage
    .groupby("leg_label", dropna=False)
    .agg(
        leg_rows=("contract_request_id", "size"),
        found_in_any_cache=("found_in_any_cache", "sum"),
        usable_leg_price=("leg_price_usable", "sum"),
        needs_exact_primary_query=("needs_exact_primary_query", "sum"),
    )
    .reset_index()
)

leg_summary_by_label["usable_leg_price_rate"] = (
    leg_summary_by_label["usable_leg_price"] / leg_summary_by_label["leg_rows"]
)

spread_leg_pivot = (
    leg_cache_coverage
    .pivot_table(
        index="selected_trade_id",
        columns="leg_label",
        values="leg_price_usable",
        aggfunc="first",
    )
    .reset_index()
)

spread_leg_pivot.columns.name = None

spread_cache_coverage = universe[
    [
        "selected_trade_id",
        "trade_date",
        "trade_year",
        "tenor",
        "expiration_date",
        "outcome_available",
        "spy_close",
        "expiry_spy_close",
        "vix_style_vol_pct",
        "short_strike",
        "long_strike",
        "width",
    ]
].merge(
    spread_leg_pivot,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

for c in ["short_call_1sd", "long_call_3sd"]:
    if c not in spread_cache_coverage.columns:
        spread_cache_coverage[c] = False
    spread_cache_coverage[c] = bool_series(spread_cache_coverage[c], index=spread_cache_coverage.index)

spread_cache_coverage["exact_primary_complete_from_cache"] = (
    spread_cache_coverage["short_call_1sd"]
    & spread_cache_coverage["long_call_3sd"]
)

spread_overlap_summary = pd.DataFrame([{
    "spread_rows": int(len(spread_cache_coverage)),
    "exact_primary_complete_from_existing_cache": int(spread_cache_coverage["exact_primary_complete_from_cache"].sum()),
    "needs_any_exact_primary_leg_query": int((~spread_cache_coverage["exact_primary_complete_from_cache"]).sum()),
    "exact_primary_complete_cache_rate": float(spread_cache_coverage["exact_primary_complete_from_cache"].mean()),
    "short_leg_usable_from_cache": int(spread_cache_coverage["short_call_1sd"].sum()),
    "long_leg_usable_from_cache": int(spread_cache_coverage["long_call_3sd"].sum()),
}])

spread_overlap_by_tenor = (
    spread_cache_coverage
    .groupby("tenor", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        outcome_available=("outcome_available", "sum"),
        exact_primary_complete_from_existing_cache=("exact_primary_complete_from_cache", "sum"),
        short_leg_usable_from_cache=("short_call_1sd", "sum"),
        long_leg_usable_from_cache=("long_call_3sd", "sum"),
    )
    .reset_index()
)

spread_overlap_by_tenor["exact_primary_complete_cache_rate"] = (
    spread_overlap_by_tenor["exact_primary_complete_from_existing_cache"]
    / spread_overlap_by_tenor["spread_rows"]
)

spread_overlap_by_year = (
    spread_cache_coverage
    .groupby("trade_year", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        outcome_available=("outcome_available", "sum"),
        exact_primary_complete_from_existing_cache=("exact_primary_complete_from_cache", "sum"),
        short_leg_usable_from_cache=("short_call_1sd", "sum"),
        long_leg_usable_from_cache=("long_call_3sd", "sum"),
    )
    .reset_index()
)

spread_overlap_by_year["exact_primary_complete_cache_rate"] = (
    spread_overlap_by_year["exact_primary_complete_from_existing_cache"]
    / spread_overlap_by_year["spread_rows"]
)

contracts_needing_exact_primary_query = (
    contract_cache_overlap
    .loc[contract_cache_overlap["needs_exact_primary_query"]]
    .sort_values(["trade_date", "expiration_date", "strike"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

OUT_UNIVERSE_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
OUT_LEG_REQUEST_PLAN_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_{OUT_SUFFIX}.parquet"
OUT_UNIQUE_CONTRACT_REQUEST_PLAN_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_unique_contract_request_plan_{OUT_SUFFIX}.parquet"
OUT_CONTRACT_CACHE_OVERLAP_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_{OUT_SUFFIX}.parquet"
OUT_LEG_CACHE_COVERAGE_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_cache_coverage_{OUT_SUFFIX}.parquet"
OUT_SPREAD_CACHE_COVERAGE_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_cache_coverage_{OUT_SUFFIX}.parquet"
OUT_CONTRACTS_NEEDING_QUERY_PATH = PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_contracts_needing_exact_primary_query_{OUT_SUFFIX}.parquet"

atomic_write_parquet(universe, OUT_UNIVERSE_PATH)
atomic_write_parquet(leg_request_plan, OUT_LEG_REQUEST_PLAN_PATH)
atomic_write_parquet(unique_contract_request_plan, OUT_UNIQUE_CONTRACT_REQUEST_PLAN_PATH)
atomic_write_parquet(contract_cache_overlap, OUT_CONTRACT_CACHE_OVERLAP_PATH)
atomic_write_parquet(leg_cache_coverage, OUT_LEG_CACHE_COVERAGE_PATH)
atomic_write_parquet(spread_cache_coverage, OUT_SPREAD_CACHE_COVERAGE_PATH)
atomic_write_parquet(contracts_needing_exact_primary_query, OUT_CONTRACTS_NEEDING_QUERY_PATH)

# Audit CSVs.
source_inspection_csv = AUDIT_DIR / f"02G_DATA_1_source_panel_inspection_{OUT_SUFFIX}_{timestamp}.csv"
universe_by_tenor_csv = AUDIT_DIR / f"02G_DATA_1_universe_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
cache_load_summary_csv = AUDIT_DIR / f"02G_DATA_1_cache_load_summary_{OUT_SUFFIX}_{timestamp}.csv"
contract_overlap_summary_csv = AUDIT_DIR / f"02G_DATA_1_contract_overlap_summary_{OUT_SUFFIX}_{timestamp}.csv"
cache_source_overlap_summary_csv = AUDIT_DIR / f"02G_DATA_1_cache_source_overlap_summary_{OUT_SUFFIX}_{timestamp}.csv"
leg_summary_by_label_csv = AUDIT_DIR / f"02G_DATA_1_leg_summary_by_label_{OUT_SUFFIX}_{timestamp}.csv"
spread_overlap_summary_csv = AUDIT_DIR / f"02G_DATA_1_spread_overlap_summary_{OUT_SUFFIX}_{timestamp}.csv"
spread_overlap_by_tenor_csv = AUDIT_DIR / f"02G_DATA_1_spread_overlap_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
spread_overlap_by_year_csv = AUDIT_DIR / f"02G_DATA_1_spread_overlap_by_year_{OUT_SUFFIX}_{timestamp}.csv"
contracts_needing_query_preview_csv = AUDIT_DIR / f"02G_DATA_1_contracts_needing_exact_primary_query_preview_{OUT_SUFFIX}_{timestamp}.csv"

atomic_write_csv(source_inspections.drop(columns=["columns"], errors="ignore"), source_inspection_csv)
atomic_write_csv(universe_by_tenor, universe_by_tenor_csv)
atomic_write_csv(cache_load_summary, cache_load_summary_csv)
atomic_write_csv(contract_overlap_summary, contract_overlap_summary_csv)
atomic_write_csv(cache_source_overlap_summary, cache_source_overlap_summary_csv)
atomic_write_csv(leg_summary_by_label, leg_summary_by_label_csv)
atomic_write_csv(spread_overlap_summary, spread_overlap_summary_csv)
atomic_write_csv(spread_overlap_by_tenor, spread_overlap_by_tenor_csv)
atomic_write_csv(spread_overlap_by_year, spread_overlap_by_year_csv)
atomic_write_csv(contracts_needing_exact_primary_query.head(500), contracts_needing_query_preview_csv)

manifest_path = AUDIT_DIR / f"02G_DATA_1_full_history_1sd3sd_request_plan_{OUT_SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.DATA.1",
    "description": "Build full historical 1SD/3SD SPY call contract request plan from VIX-style tenor measures",
    "created_at": timestamp,
    "target_tenors": TARGET_TENORS,
    "strike_convention": {
        "short_call_1sd": f"ceil to {SHORT_STRIKE_INCREMENT}",
        "long_call_3sd": f"round half-up to {LONG_STRIKE_INCREMENT}; force above short strike",
        "sd_move": "vix_style_vol_pct / 100 * sqrt(tenor / 365)",
    },
    "source_panel": {
        "selected_path": str(SOURCE_PANEL_PATH),
        "trade_col": selected_source.get("trade_col"),
        "tenor_col": selected_source.get("tenor_col"),
        "vol_col": selected_source.get("vol_col"),
        "spy_col": selected_source.get("spy_col"),
        "vol_unit_interpretation": vol_unit_interpretation,
    },
    "counts": {
        "target_tenor_panel_rows": int(len(panel)),
        "request_universe_rows": int(len(universe)),
        "rows_with_outcome_available": int(universe["outcome_available"].sum()),
        "leg_rows": int(len(leg_request_plan)),
        "unique_contract_requests": int(len(unique_contract_request_plan)),
        "unique_contracts_sufficient_from_existing_cache": int(contract_cache_overlap["sufficient_for_all_uses"].sum()),
        "unique_contracts_needing_exact_primary_query": int(contract_cache_overlap["needs_exact_primary_query"].sum()),
        "spreads_exact_primary_complete_from_existing_cache": int(spread_cache_coverage["exact_primary_complete_from_cache"].sum()),
        "spreads_needing_any_exact_primary_leg_query": int((~spread_cache_coverage["exact_primary_complete_from_cache"]).sum()),
    },
    "outputs": {
        "universe_path": str(OUT_UNIVERSE_PATH),
        "leg_request_plan_path": str(OUT_LEG_REQUEST_PLAN_PATH),
        "unique_contract_request_plan_path": str(OUT_UNIQUE_CONTRACT_REQUEST_PLAN_PATH),
        "contract_cache_overlap_path": str(OUT_CONTRACT_CACHE_OVERLAP_PATH),
        "leg_cache_coverage_path": str(OUT_LEG_CACHE_COVERAGE_PATH),
        "spread_cache_coverage_path": str(OUT_SPREAD_CACHE_COVERAGE_PATH),
        "contracts_needing_exact_primary_query_path": str(OUT_CONTRACTS_NEEDING_QUERY_PATH),
        "manifest_path": str(manifest_path),
    },
    "api_calls": "None",
    "next_step": "2G.DATA.2 should query ThetaData for contracts_needing_exact_primary_query.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display final diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.1 complete — full historical 1SD / 3SD request plan")
print("=" * 100)

print()
print("Universe by tenor")
display(universe_by_tenor)

print()
print("=" * 100)
print("Cache load summary")
print("=" * 100)
display(cache_load_summary)

print()
print("=" * 100)
print("Unique contract overlap summary")
print("=" * 100)
display(contract_overlap_summary)

print()
print("=" * 100)
print("Overlap by cache source")
print("=" * 100)
display(cache_source_overlap_summary)

print()
print("=" * 100)
print("Leg summary by label")
print("=" * 100)
display(leg_summary_by_label)

print()
print("=" * 100)
print("Spread exact-primary completeness from existing caches")
print("=" * 100)
display(spread_overlap_summary)

print()
print("=" * 100)
print("Spread cache coverage by tenor")
print("=" * 100)
display(spread_overlap_by_tenor)

print()
print("=" * 100)
print("Spread cache coverage by year")
print("=" * 100)
display(spread_overlap_by_year)

print()
print("=" * 100)
print("Contracts needing exact primary ThetaData query preview")
print("=" * 100)
if contracts_needing_exact_primary_query.empty:
    print("No missing exact primary contracts. Existing caches are sufficient.")
else:
    display(contracts_needing_exact_primary_query.head(100))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Universe:                         {OUT_UNIVERSE_PATH}")
print(f"Leg request plan:                 {OUT_LEG_REQUEST_PLAN_PATH}")
print(f"Unique contract request plan:     {OUT_UNIQUE_CONTRACT_REQUEST_PLAN_PATH}")
print(f"Contract cache overlap:           {OUT_CONTRACT_CACHE_OVERLAP_PATH}")
print(f"Leg cache coverage:               {OUT_LEG_CACHE_COVERAGE_PATH}")
print(f"Spread cache coverage:            {OUT_SPREAD_CACHE_COVERAGE_PATH}")
print(f"Contracts needing query:          {OUT_CONTRACTS_NEEDING_QUERY_PATH}")
print(f"Manifest:                         {manifest_path}")

print()
print("Result: 2G.DATA.1 data-only request plan complete.")
print("Next step: 2G.DATA.2 exact primary ThetaData pull for missing contracts.")

Cell 2G.DATA.1 — Full historical 1SD / 3SD call contract request plan
Target tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
SPY EOD path:  C:\Users\patri\vrp_project\data\processed\market_data\spy_eod_prices_v1.parquet
No trade-signal filters are applied.

Loaded SPY EOD calendar
SPY EOD rows: 2,141
Date range:   2018-01-02 00:00:00 to 2026-07-10 00:00:00

Source panel inspection


,path,exists,usable,reason,rows,target_tenor_rows,first_date,last_date,date_span_days,trade_col,tenor_col,vol_col,spy_col
0,C:\Users\patri\vrp_project\data\processed\call...,True,True,usable,14715,14715,2020-01-02,2026-07-08,2379,trade_date,tenor,vix_style_vol_pct,spy_close
1,C:\Users\patri\vrp_project\data\processed\call...,True,True,usable,14724,14724,2020-01-02,2026-07-09,2380,trade_date,tenor,vix_style_vol_pct,spy_close
2,C:\Users\patri\vrp_project\data\processed\vrp_...,True,True,usable,14724,14724,2020-01-02,2026-07-09,2380,trade_date,tenor,vix_style_vol_final,spy_close
3,C:\Users\patri\vrp_project\data\processed\vrp_...,True,True,usable,14724,14724,2020-01-02,2026-07-09,2380,trade_date,tenor,vix_style_vol_final,spy_close



Selected source panel
Path:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet
Rows:       14,724
Target rows:14,724
Date range: 2020-01-02 00:00:00 to 2026-07-09 00:00:00
Trade col:  trade_date
Tenor col:  tenor
Vol col:    vix_style_vol_pct
SPY col:    spy_close

Full historical 1SD / 3SD universe
Raw source rows:              14,724
Target-tenor panel rows:      14,724
Duplicate date/tenor removed: 0
Request-universe rows:        14,613
Rows with outcome available:  14,613
Date range:                   2020-01-02 00:00:00 to 2026-07-01 00:00:00
Vol unit handling:            input appeared percent; kept as percent

If you expected a longer sample, point SOURCE_PANEL_OVERRIDE to the older VIX-style tenor panel.



,tenor,rows,first_date,last_date,outcome_available,avg_spy_close,avg_vix_style_vol_pct,avg_short_strike,avg_long_strike,avg_width
0,9,1632,2020-01-02,2026-07-01,1632,475.962952,19.913988,490.583946,518.345588,27.761642
1,12,1629,2020-01-02,2026-06-26,1629,475.468390,20.095014,492.450583,524.904850,32.454266
2,15,1628,2020-01-02,2026-06-25,1628,475.312664,20.233482,494.377764,531.022727,36.644963
3,18,1625,2020-01-02,2026-06-22,1625,474.835629,20.377938,495.830154,536.323077,40.492923
4,21,1624,2020-01-02,2026-06-18,1624,474.669647,20.507520,497.473522,541.530172,44.056650
5,24,1622,2020-01-02,2026-06-16,1622,474.337736,20.634159,498.838471,546.294698,47.456227
6,27,1620,2020-01-02,2026-06-12,1620,473.994227,20.767012,500.114815,550.873457,50.758642
7,30,1618,2020-01-02,2026-06-10,1618,473.665722,20.888092,501.311496,555.098888,53.787392
8,33,1615,2020-01-02,2026-06-05,1615,473.182314,21.010384,502.295975,559.024768,56.728793


Contract request plan
Spread rows:              14,613
Leg rows before dedupe:   29,226
Unique contract requests: 28,523
Unique short contracts:   14,568
Unique long contracts:    13,955



...


Spread cache coverage by year


,trade_year,spread_rows,outcome_available,exact_primary_complete_from_existing_cache,short_leg_usable_from_cache,long_leg_usable_from_cache,exact_primary_complete_cache_rate
0,2020,2277,2277,3,4,5,0.001318
1,2021,2268,2268,122,132,126,0.053792
2,2022,2259,2259,118,132,156,0.052236
3,2023,2250,2250,251,254,306,0.111556
4,2024,2268,2268,406,443,620,0.179012
5,2025,2250,2250,207,253,348,0.092000
6,2026,1041,1041,97,99,146,0.093180



Contracts needing exact primary ThetaData query preview


,contract_request_id,trade_date,expiration_date,strike,leg_use_count,used_as_short,used_as_long,first_trade_date,last_trade_date,cache_source,...,bid,ask,mid,status,status_code,found_in_any_cache,bid_needed,ask_needed,sufficient_for_all_uses,needs_exact_primary_query
0,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,1,1,0,2020-01-02,2020-01-02,NaN,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,True
1,SPY_20200102_20200117_332.0_C,2020-01-02,2020-01-17,332.0,1,1,0,2020-01-02,2020-01-02,NaN,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,True
2,SPY_20200102_20200117_333.0_C,2020-01-02,2020-01-17,333.0,1,1,0,2020-01-02,2020-01-02,NaN,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,True
3,SPY_20200102_20200117_340.0_C,2020-01-02,2020-01-17,340.0,1,0,1,2020-01-02,2020-01-02,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,False,True
4,SPY_20200102_20200117_345.0_C,2020-01-02,2020-01-17,345.0,2,0,2,2020-01-02,2020-01-02,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,SPY_20200110_20200124_345.0_C,2020-01-10,2020-01-24,345.0,1,0,1,2020-01-10,2020-01-10,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,False,True
96,SPY_20200110_20200131_334.0_C,2020-01-10,2020-01-31,334.0,1,1,0,2020-01-10,2020-01-10,NaN,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,True
97,SPY_20200110_20200131_335.0_C,2020-01-10,2020-01-31,335.0,2,2,0,2020-01-10,2020-01-10,NaN,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,True
98,SPY_20200110_20200131_350.0_C,2020-01-10,2020-01-31,350.0,2,0,2,2020-01-10,2020-01-10,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,False,True



Saved
Universe:                         C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_universe_fullhist_1sd3sd_long5_cal_v1.parquet
Leg request plan:                 C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Unique contract request plan:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_unique_contract_request_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Contract cache overlap:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_fullhist_1sd3sd_long5_cal_v1.parquet
Leg cache coverage:               C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_cache_coverage_fullhist_1sd3sd_long5_cal_v1.parquet
Spread cach

In [13]:
# Cell 2G.DATA.2 — Exact primary ThetaData pull for full-history 1SD / 3SD call contracts
# Purpose:
#   Pull missing exact SPY call bid/ask premiums for the full historical 1SD / 3SD call
#   contract request plan built in 2G.DATA.1.
#
# This is a DATA LAYER cell:
#   - No RSI
#   - No z-score
#   - No VRP filter
#   - No signal logic
#   - No selection rule
#
# It:
#   - Loads the full-history contract cache overlap from 2G.DATA.1
#   - Seeds a full-history primary cache from all existing caches
#   - Probes ThetaData endpoint/strike-param format
#   - Queries missing exact primary contracts only
#   - Saves a resumable full-history primary contract price cache
#   - Recomputes exact-primary leg/spread coverage
#
# If ThetaData endpoint/query format is broken, this cell stops after the probe instead of
# hammering 25k requests.

from pathlib import Path
from datetime import datetime
from io import StringIO
import json
import time

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

LEG_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_{OUT_SUFFIX}.parquet"
)

UNIQUE_CONTRACT_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_unique_contract_request_plan_{OUT_SUFFIX}.parquet"
)

CONTRACT_CACHE_OVERLAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_{OUT_SUFFIX}.parquet"
)

CONTRACTS_NEEDING_QUERY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_contracts_needing_exact_primary_query_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_CONTRACT_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_LEG_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_leg_prices_joined_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_SPREAD_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_prices_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_coverage_summary_{OUT_SUFFIX}.parquet"
)

# ThetaData local terminal.
THETADATA_BASE_URL = "http://127.0.0.1:25503"

# We probe candidates because prior new requests were returning http_410.
# The first one that produces usable bid/ask wins.
THETADATA_ENDPOINT_CANDIDATES = [
    "/v3/option/history/greeks/eod",
    "/v3/option/history/quote/eod",
    "/v3/option/history/eod",
]

# ThetaData strike param often uses strike * 1000. We probe both.
STRIKE_PARAM_FORMATS_TO_TRY = ["thousandths", "float"]

ROOT = "SPY"
RIGHT = "C"

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 0.02

PROBE_N_CONTRACTS = 20
REQUIRE_PROBE_SUCCESS = True

# None = query all remaining missing contracts.
# To smoke-test manually, set to e.g. 500.
MAX_REQUESTS_THIS_RUN = None

SAVE_EVERY_N_ATTEMPTS = 100
PROGRESS_EVERY_N_ATTEMPTS = 100

# Resume behavior:
#   False = do not retry contracts already attempted in the full-history primary cache.
#   True  = retry previous failures.
RETRY_FAILED_CONTRACTS = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.DATA.2 — Exact primary ThetaData pull for full-history 1SD / 3SD call contracts")
print("=" * 100)
print(f"Universe:                  {UNIVERSE_PATH}")
print(f"Leg request plan:          {LEG_REQUEST_PLAN_PATH}")
print(f"Contract cache overlap:    {CONTRACT_CACHE_OVERLAP_PATH}")
print(f"Contracts needing query:   {CONTRACTS_NEEDING_QUERY_PATH}")
print(f"Output primary cache:      {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
print(f"Max requests this run:     {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed contracts:    {RETRY_FAILED_CONTRACTS}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def yyyymmdd(x):
    return pd.Timestamp(x).strftime("%Y%m%d")

def theta_strike_param(strike, fmt):
    strike = float(strike)

    if fmt == "thousandths":
        return int(round(strike * 1000))

    if fmt == "float":
        if abs(strike - round(strike)) < 1e-9:
            return int(round(strike))
        return strike

    raise ValueError(f"Unknown strike format: {fmt}")

def extract_first_existing(row, candidates):
    for c in candidates:
        if c in row.index:
            val = row[c]
            if pd.notna(val):
                return val
    return np.nan

def parse_thetadata_response(resp):
    """
    Robust parser for ThetaData local terminal responses.
    Handles JSON-ish and CSV-ish responses.
    """
    text = (resp.text or "").strip()

    if resp.status_code != 200 or not text:
        return pd.DataFrame()

    # JSON-like response.
    try:
        data = resp.json()

        if isinstance(data, dict):
            if "header" in data and "response" in data:
                return pd.DataFrame(data["response"], columns=data["header"])

            if "data" in data and isinstance(data["data"], list):
                return pd.DataFrame(data["data"])

            if "response" in data and isinstance(data["response"], list):
                return pd.DataFrame(data["response"])

            return pd.DataFrame([data])

        if isinstance(data, list):
            return pd.DataFrame(data)

    except Exception:
        pass

    # CSV-like response.
    try:
        df = pd.read_csv(StringIO(text))
        if df.empty:
            return pd.DataFrame()
        return df
    except Exception:
        return pd.DataFrame()

def normalize_response_columns(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

def extract_quote_from_parsed_df(parsed):
    """
    Return bid/ask/mid/greeks from the last parsed row.
    """
    if parsed.empty:
        return {
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
        }

    parsed = normalize_response_columns(parsed)
    row = parsed.iloc[-1]

    bid = pd.to_numeric(
        extract_first_existing(row, ["bid", "bid_price", "close_bid", "eod_bid"]),
        errors="coerce",
    )

    ask = pd.to_numeric(
        extract_first_existing(row, ["ask", "ask_price", "close_ask", "eod_ask"]),
        errors="coerce",
    )

    mid = pd.to_numeric(
        extract_first_existing(row, ["mid", "mid_price", "mark", "price", "close", "eod_price"]),
        errors="coerce",
    )

    if pd.isna(mid) and pd.notna(bid) and pd.notna(ask):
        mid = (bid + ask) / 2.0

    iv = pd.to_numeric(
        extract_first_existing(row, ["iv", "implied_vol", "implied_volatility"]),
        errors="coerce",
    )

    delta = pd.to_numeric(extract_first_existing(row, ["delta"]), errors="coerce")
    gamma = pd.to_numeric(extract_first_existing(row, ["gamma"]), errors="coerce")
    theta = pd.to_numeric(extract_first_existing(row, ["theta"]), errors="coerce")
    vega = pd.to_numeric(extract_first_existing(row, ["vega"]), errors="coerce")
    rho = pd.to_numeric(extract_first_existing(row, ["rho"]), errors="coerce")

    price_found = (
        pd.notna(bid)
        and pd.notna(ask)
        and float(bid) >= 0
        and float(ask) >= 0
    )

    return {
        "price_found": bool(price_found),
        "bid": float(bid) if pd.notna(bid) else np.nan,
        "ask": float(ask) if pd.notna(ask) else np.nan,
        "mid": float(mid) if pd.notna(mid) else np.nan,
        "iv": float(iv) if pd.notna(iv) else np.nan,
        "delta": float(delta) if pd.notna(delta) else np.nan,
        "gamma": float(gamma) if pd.notna(gamma) else np.nan,
        "theta": float(theta) if pd.notna(theta) else np.nan,
        "vega": float(vega) if pd.notna(vega) else np.nan,
        "rho": float(rho) if pd.notna(rho) else np.nan,
    }

def query_contract_once(session, endpoint, strike_format, trade_date, expiration_date, strike):
    url = THETADATA_BASE_URL + endpoint

    params = {
        "root": ROOT,
        "exp": yyyymmdd(expiration_date),
        "right": RIGHT,
        "strike": theta_strike_param(strike, strike_format),
        "start_date": yyyymmdd(trade_date),
        "end_date": yyyymmdd(trade_date),
    }

    try:
        resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        parsed = parse_thetadata_response(resp)
        quote = extract_quote_from_parsed_df(parsed)

        return {
            **quote,
            "request_ok": True,
            "endpoint": endpoint,
            "strike_param_format": strike_format,
            "http_status_code": int(resp.status_code),
            "request_url": resp.url,
            "response_preview": (resp.text or "")[:300],
            "parsed_rows": int(len(parsed)),
            "error_message": "",
        }

    except Exception as e:
        return {
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
            "request_ok": False,
            "endpoint": endpoint,
            "strike_param_format": strike_format,
            "http_status_code": np.nan,
            "request_url": "",
            "response_preview": "",
            "parsed_rows": 0,
            "error_message": repr(e),
        }

def query_contract_best(session, contract_row, endpoint, strike_format):
    result = query_contract_once(
        session=session,
        endpoint=endpoint,
        strike_format=strike_format,
        trade_date=contract_row["trade_date"],
        expiration_date=contract_row["expiration_date"],
        strike=contract_row["strike"],
    )

    return {
        "contract_request_id": contract_row["contract_request_id"],
        "trade_date": pd.Timestamp(contract_row["trade_date"]).normalize(),
        "expiration_date": pd.Timestamp(contract_row["expiration_date"]).normalize(),
        "strike": float(contract_row["strike"]),
        "root": ROOT,
        "right": RIGHT,
        "price_found": bool(result["price_found"]),
        "bid": result["bid"],
        "ask": result["ask"],
        "mid": result["mid"],
        "iv": result["iv"],
        "delta": result["delta"],
        "gamma": result["gamma"],
        "theta": result["theta"],
        "vega": result["vega"],
        "rho": result["rho"],
        "status": "priced" if result["price_found"] else "no_price",
        "status_code": result["http_status_code"],
        "http_status_code": result["http_status_code"],
        "request_ok": bool(result["request_ok"]),
        "endpoint": result["endpoint"],
        "strike_param_format": result["strike_param_format"],
        "request_url": result["request_url"],
        "response_preview": result["response_preview"],
        "parsed_rows": int(result["parsed_rows"]),
        "error_message": result["error_message"],
        "query_attempted": True,
        "cache_source": "thetadata_exact_primary_fullhist",
        "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

def build_seed_cache_from_overlap(overlap):
    """
    One row per requested contract, seeded from existing exact contract cache overlap.
    Rows without existing data remain present but unpriced.
    """
    out = overlap.copy()

    for c in ["trade_date", "expiration_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.normalize()

    if "price_found" not in out.columns:
        out["price_found"] = False

    out["price_found"] = bool_series(out["price_found"], index=out.index)

    for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"]:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    for c in ["status", "status_code", "cache_source"]:
        if c not in out.columns:
            out[c] = np.nan

    out["root"] = ROOT
    out["right"] = RIGHT
    out["http_status_code"] = out["status_code"]
    out["request_ok"] = np.nan
    out["endpoint"] = np.nan
    out["strike_param_format"] = np.nan
    out["request_url"] = np.nan
    out["response_preview"] = np.nan
    out["parsed_rows"] = np.nan
    out["error_message"] = ""
    out["query_attempted"] = False
    out["queried_at"] = np.nan

    # Preserve original existing-cache source in this column.
    out["seed_cache_source"] = out["cache_source"]

    keep = [
        "contract_request_id",
        "trade_date",
        "expiration_date",
        "strike",
        "root",
        "right",
        "price_found",
        "bid",
        "ask",
        "mid",
        "iv",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "status",
        "status_code",
        "http_status_code",
        "request_ok",
        "endpoint",
        "strike_param_format",
        "request_url",
        "response_preview",
        "parsed_rows",
        "error_message",
        "query_attempted",
        "cache_source",
        "seed_cache_source",
        "queried_at",
    ]

    for c in keep:
        if c not in out.columns:
            out[c] = np.nan

    return out[keep].copy()

def combine_cache_rows(seed_cache, prior_cache, new_query_rows):
    frames = []

    if seed_cache is not None and len(seed_cache):
        frames.append(seed_cache.copy())

    if prior_cache is not None and len(prior_cache):
        frames.append(prior_cache.copy())

    if new_query_rows is not None and len(new_query_rows):
        frames.append(new_query_rows.copy())

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)

    for c in ["trade_date", "expiration_date"]:
        if c in combined.columns:
            combined[c] = pd.to_datetime(combined[c], errors="coerce").dt.normalize()

    for c in ["price_found", "query_attempted"]:
        combined[c] = bool_series(combined.get(c), index=combined.index)

    for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho", "strike"]:
        if c in combined.columns:
            combined[c] = pd.to_numeric(combined[c], errors="coerce")

    combined["bid_usable"] = combined["bid"].notna() & (combined["bid"] >= 0)
    combined["ask_usable"] = combined["ask"].notna() & (combined["ask"] >= 0)
    combined["both_sides_usable"] = combined["bid_usable"] & combined["ask_usable"]

    # Prefer:
    #   0 queried successful row
    #   1 seeded successful row
    #   2 queried failed row
    #   3 seeded missing row
    combined["row_preference"] = np.select(
        [
            combined["both_sides_usable"] & combined["query_attempted"],
            combined["both_sides_usable"] & ~combined["query_attempted"],
            ~combined["both_sides_usable"] & combined["query_attempted"],
        ],
        [0, 1, 2],
        default=3,
    )

    combined = (
        combined
        .sort_values(["contract_request_id", "row_preference"])
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["row_preference"])
        .reset_index(drop=True)
    )

    return combined

def summarize_primary_cache(cache):
    tmp = cache.copy()
    tmp["price_found"] = bool_series(tmp.get("price_found"), index=tmp.index)
    tmp["query_attempted"] = bool_series(tmp.get("query_attempted"), index=tmp.index)
    tmp["bid_usable"] = tmp["bid"].notna() & (tmp["bid"] >= 0)
    tmp["ask_usable"] = tmp["ask"].notna() & (tmp["ask"] >= 0)
    tmp["both_sides_usable"] = tmp["bid_usable"] & tmp["ask_usable"]

    return pd.DataFrame([{
        "contracts": int(len(tmp)),
        "both_sides_usable": int(tmp["both_sides_usable"].sum()),
        "price_found": int(tmp["price_found"].sum()),
        "query_attempted": int(tmp["query_attempted"].sum()),
        "query_attempted_and_priced": int((tmp["query_attempted"] & tmp["both_sides_usable"]).sum()),
        "query_attempted_not_priced": int((tmp["query_attempted"] & ~tmp["both_sides_usable"]).sum()),
        "still_missing_both_sides": int((~tmp["both_sides_usable"]).sum()),
        "usable_rate": float(tmp["both_sides_usable"].mean()) if len(tmp) else np.nan,
    }])

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    LEG_REQUEST_PLAN_PATH,
    UNIQUE_CONTRACT_REQUEST_PLAN_PATH,
    CONTRACT_CACHE_OVERLAP_PATH,
    CONTRACTS_NEEDING_QUERY_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
leg_plan = pd.read_parquet(LEG_REQUEST_PLAN_PATH).copy()
contract_plan = pd.read_parquet(UNIQUE_CONTRACT_REQUEST_PLAN_PATH).copy()
contract_overlap = pd.read_parquet(CONTRACT_CACHE_OVERLAP_PATH).copy()
contracts_needing_query = pd.read_parquet(CONTRACTS_NEEDING_QUERY_PATH).copy()

for df in [universe, leg_plan, contract_plan, contract_overlap, contracts_needing_query]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Universe spread rows:              {len(universe):,}")
print(f"Leg plan rows:                     {len(leg_plan):,}")
print(f"Unique contract plan rows:         {len(contract_plan):,}")
print(f"Contract overlap rows:             {len(contract_overlap):,}")
print(f"Contracts marked needing query:    {len(contracts_needing_query):,}")
print()

# --------------------------------------------------------------------------------------------------
# Seed cache and resume previous output if any
# --------------------------------------------------------------------------------------------------

seed_cache = build_seed_cache_from_overlap(contract_overlap)

if OUT_PRIMARY_CONTRACT_CACHE_PATH.exists():
    prior_cache = pd.read_parquet(OUT_PRIMARY_CONTRACT_CACHE_PATH).copy()
    print(f"Existing full-history primary cache found: {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
    print(f"Existing rows: {len(prior_cache):,}")
else:
    prior_cache = pd.DataFrame()
    print("No existing full-history primary cache found. Starting fresh from seed overlap.")

combined_cache_pre = combine_cache_rows(seed_cache, prior_cache, pd.DataFrame())

cache_pre_summary = summarize_primary_cache(combined_cache_pre)

print()
print("=" * 100)
print("Pre-query primary cache summary")
print("=" * 100)
display(cache_pre_summary)

# Determine query list from all contracts still not sufficient.
combined_cache_pre_small = combined_cache_pre[
    [
        "contract_request_id",
        "both_sides_usable",
        "query_attempted",
    ]
].copy()

query_candidates = contract_plan.merge(
    combined_cache_pre_small,
    on="contract_request_id",
    how="left",
    validate="one_to_one",
)

query_candidates["both_sides_usable"] = bool_series(query_candidates["both_sides_usable"], index=query_candidates.index)
query_candidates["query_attempted"] = bool_series(query_candidates["query_attempted"], index=query_candidates.index)

if RETRY_FAILED_CONTRACTS:
    to_query = query_candidates.loc[~query_candidates["both_sides_usable"]].copy()
else:
    to_query = query_candidates.loc[
        ~query_candidates["both_sides_usable"]
        & ~query_candidates["query_attempted"]
    ].copy()

to_query = to_query.sort_values(["trade_date", "expiration_date", "strike"]).reset_index(drop=True)

if MAX_REQUESTS_THIS_RUN is not None:
    to_query_run = to_query.head(int(MAX_REQUESTS_THIS_RUN)).copy()
else:
    to_query_run = to_query.copy()

print()
print("=" * 100)
print("Query plan")
print("=" * 100)
print(f"Contracts still missing both sides:  {(~query_candidates['both_sides_usable']).sum():,}")
print(f"Eligible to query this run:          {len(to_query):,}")
print(f"Actually querying this run:          {len(to_query_run):,}")
print()

if to_query_run.empty:
    print("Nothing to query. Proceeding directly to coverage rebuild.")

# --------------------------------------------------------------------------------------------------
# Probe endpoint / strike format
# --------------------------------------------------------------------------------------------------

selected_endpoint = None
selected_strike_format = None
probe_results_df = pd.DataFrame()

if not to_query_run.empty:
    probe_contracts = to_query_run.head(PROBE_N_CONTRACTS).copy()

    probe_rows = []

    print("=" * 100)
    print("ThetaData endpoint probe")
    print("=" * 100)
    print(f"Probe contracts: {len(probe_contracts):,}")
    print()

    with requests.Session() as session:
        for endpoint in THETADATA_ENDPOINT_CANDIDATES:
            for strike_fmt in STRIKE_PARAM_FORMATS_TO_TRY:
                successes = 0
                http_200 = 0
                http_410 = 0
                parsed_rows_total = 0

                for _, row in probe_contracts.iterrows():
                    result = query_contract_once(
                        session=session,
                        endpoint=endpoint,
                        strike_format=strike_fmt,
                        trade_date=row["trade_date"],
                        expiration_date=row["expiration_date"],
                        strike=row["strike"],
                    )

                    if result["http_status_code"] == 200:
                        http_200 += 1

                    if result["http_status_code"] == 410:
                        http_410 += 1

                    parsed_rows_total += int(result["parsed_rows"])

                    if result["price_found"]:
                        successes += 1

                    probe_rows.append({
                        "endpoint": endpoint,
                        "strike_param_format": strike_fmt,
                        "contract_request_id": row["contract_request_id"],
                        "trade_date": row["trade_date"],
                        "expiration_date": row["expiration_date"],
                        "strike": row["strike"],
                        "price_found": result["price_found"],
                        "bid": result["bid"],
                        "ask": result["ask"],
                        "http_status_code": result["http_status_code"],
                        "parsed_rows": result["parsed_rows"],
                        "request_url": result["request_url"],
                        "response_preview": result["response_preview"],
                        "error_message": result["error_message"],
                    })

                    time.sleep(REQUEST_SLEEP_SECONDS)

                print(
                    f"Probe {endpoint} / {strike_fmt}: "
                    f"successes={successes}, http_200={http_200}, http_410={http_410}, parsed_rows={parsed_rows_total}"
                )

    probe_results_df = pd.DataFrame(probe_rows)

    probe_combo_summary = (
        probe_results_df
        .groupby(["endpoint", "strike_param_format"], dropna=False)
        .agg(
            probe_contracts=("contract_request_id", "size"),
            price_found=("price_found", "sum"),
            http_200=("http_status_code", lambda x: int((x == 200).sum())),
            http_410=("http_status_code", lambda x: int((x == 410).sum())),
            parsed_rows=("parsed_rows", "sum"),
        )
        .reset_index()
        .sort_values(["price_found", "http_200", "parsed_rows"], ascending=[False, False, False])
    )

    print()
    print("Probe combo summary")
    display(probe_combo_summary)

    if not probe_combo_summary.empty and int(probe_combo_summary.iloc[0]["price_found"]) > 0:
        selected_endpoint = probe_combo_summary.iloc[0]["endpoint"]
        selected_strike_format = probe_combo_summary.iloc[0]["strike_param_format"]

        print()
        print(f"Selected endpoint:            {selected_endpoint}")
        print(f"Selected strike param format: {selected_strike_format}")

    elif REQUIRE_PROBE_SUCCESS:
        probe_path = AUDIT_DIR / f"02G_DATA_2_thetadata_probe_results_{OUT_SUFFIX}_{timestamp}.csv"
        atomic_write_csv(probe_results_df, probe_path)

        print()
        print("=" * 100)
        print("STOPPING: ThetaData probe found no usable bid/ask rows")
        print("=" * 100)
        print(f"Probe audit saved: {probe_path}")
        print("No bulk query was attempted.")
        print("This likely means the current ThetaData endpoint/query format needs to be fixed.")
        raise RuntimeError("ThetaData probe failed: no endpoint/strike-format combination returned usable bid/ask data.")

    else:
        selected_endpoint = THETADATA_ENDPOINT_CANDIDATES[0]
        selected_strike_format = STRIKE_PARAM_FORMATS_TO_TRY[0]
        print()
        print("WARNING: Probe found no successful bid/ask rows, but REQUIRE_PROBE_SUCCESS=False.")
        print(f"Continuing with {selected_endpoint} / {selected_strike_format}.")

# --------------------------------------------------------------------------------------------------
# Main query loop
# --------------------------------------------------------------------------------------------------

new_query_rows = []

if not to_query_run.empty:
    print()
    print("=" * 100)
    print("Starting main ThetaData query loop")
    print("=" * 100)

    start_time = time.time()

    with requests.Session() as session:
        for i, (_, row) in enumerate(to_query_run.iterrows(), start=1):
            out_row = query_contract_best(
                session=session,
                contract_row=row,
                endpoint=selected_endpoint,
                strike_format=selected_strike_format,
            )

            new_query_rows.append(out_row)

            if (i % PROGRESS_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                elapsed = time.time() - start_time
                priced_so_far = sum(1 for r in new_query_rows if r["price_found"])
                http_410_so_far = sum(1 for r in new_query_rows if r["http_status_code"] == 410)

                print(
                    f"Progress {i:,}/{len(to_query_run):,} | "
                    f"priced={priced_so_far:,} | "
                    f"http_410={http_410_so_far:,} | "
                    f"elapsed={elapsed/60:.1f} min"
                )

            if (i % SAVE_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                new_query_df_tmp = pd.DataFrame(new_query_rows)
                combined_tmp = combine_cache_rows(seed_cache, prior_cache, new_query_df_tmp)
                atomic_write_parquet(combined_tmp, OUT_PRIMARY_CONTRACT_CACHE_PATH)

            time.sleep(REQUEST_SLEEP_SECONDS)

new_query_df = pd.DataFrame(new_query_rows)

# Final cache combine / save.
primary_contract_cache = combine_cache_rows(seed_cache, prior_cache, new_query_df)
atomic_write_parquet(primary_contract_cache, OUT_PRIMARY_CONTRACT_CACHE_PATH)

print()
print("=" * 100)
print("Post-query primary contract cache summary")
print("=" * 100)
cache_post_summary = summarize_primary_cache(primary_contract_cache)
display(cache_post_summary)

# --------------------------------------------------------------------------------------------------
# Rebuild exact-primary leg and spread coverage
# --------------------------------------------------------------------------------------------------

cache_for_join = primary_contract_cache.copy()

for c in ["trade_date", "expiration_date"]:
    if c in cache_for_join.columns:
        cache_for_join[c] = pd.to_datetime(cache_for_join[c], errors="coerce").dt.normalize()

cache_for_join["bid"] = pd.to_numeric(cache_for_join["bid"], errors="coerce")
cache_for_join["ask"] = pd.to_numeric(cache_for_join["ask"], errors="coerce")
cache_for_join["mid"] = pd.to_numeric(cache_for_join["mid"], errors="coerce")
cache_for_join["bid_usable"] = cache_for_join["bid"].notna() & (cache_for_join["bid"] >= 0)
cache_for_join["ask_usable"] = cache_for_join["ask"].notna() & (cache_for_join["ask"] >= 0)

leg_prices = leg_plan.merge(
    cache_for_join[
        [
            "contract_request_id",
            "price_found",
            "bid",
            "ask",
            "mid",
            "iv",
            "delta",
            "gamma",
            "theta",
            "vega",
            "rho",
            "bid_usable",
            "ask_usable",
            "status",
            "status_code",
            "http_status_code",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
            "endpoint",
            "strike_param_format",
        ]
    ],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_prices["bid_usable"] = bool_series(leg_prices["bid_usable"], index=leg_prices.index)
leg_prices["ask_usable"] = bool_series(leg_prices["ask_usable"], index=leg_prices.index)

leg_prices["leg_price_usable"] = np.where(
    leg_prices["needed_price_field"].eq("bid"),
    leg_prices["bid_usable"],
    leg_prices["ask_usable"],
).astype(bool)

# Pivot leg economics into spread-level exact-primary table.
spread_base_cols = [
    "selected_trade_id",
    "trade_date",
    "trade_year",
    "tenor",
    "expiration_date",
    "expiration_str",
    "outcome_available",
    "short_strike",
    "long_strike",
    "spy_close",
    "vix_style_vol_pct",
    "sd_move",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "width",
]

spread_base = universe[spread_base_cols + ["expiry_spy_close"]].drop_duplicates("selected_trade_id").copy()

short_leg = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("short_call_1sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "strike",
            "bid",
            "ask",
            "mid",
            "leg_price_usable",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "short_contract_request_id",
            "strike": "short_effective_strike",
            "bid": "short_bid",
            "ask": "short_ask",
            "mid": "short_mid",
            "leg_price_usable": "short_leg_price_usable",
            "query_attempted": "short_query_attempted",
            "cache_source": "short_cache_source",
            "seed_cache_source": "short_seed_cache_source",
        }
    )
)

long_leg = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("long_call_3sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "strike",
            "bid",
            "ask",
            "mid",
            "leg_price_usable",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "long_contract_request_id",
            "strike": "long_effective_strike",
            "bid": "long_bid",
            "ask": "long_ask",
            "mid": "long_mid",
            "leg_price_usable": "long_leg_price_usable",
            "query_attempted": "long_query_attempted",
            "cache_source": "long_cache_source",
            "seed_cache_source": "long_seed_cache_source",
        }
    )
)

spread_prices = (
    spread_base
    .merge(short_leg, on="selected_trade_id", how="left", validate="one_to_one")
    .merge(long_leg, on="selected_trade_id", how="left", validate="one_to_one")
)

spread_prices["short_leg_price_usable"] = bool_series(spread_prices["short_leg_price_usable"], index=spread_prices.index)
spread_prices["long_leg_price_usable"] = bool_series(spread_prices["long_leg_price_usable"], index=spread_prices.index)

spread_prices["exact_primary_complete"] = (
    spread_prices["short_leg_price_usable"]
    & spread_prices["long_leg_price_usable"]
)

spread_prices["entry_credit_exact_primary"] = spread_prices["short_bid"] - spread_prices["long_ask"]
spread_prices["effective_width_exact_primary"] = spread_prices["long_effective_strike"] - spread_prices["short_effective_strike"]
spread_prices["max_loss_exact_primary"] = (
    spread_prices["effective_width_exact_primary"]
    - spread_prices["entry_credit_exact_primary"]
)

spread_prices["terminal_spread_intrinsic_exact_primary"] = np.maximum(
    spread_prices["expiry_spy_close"] - spread_prices["short_effective_strike"],
    0.0,
) - np.maximum(
    spread_prices["expiry_spy_close"] - spread_prices["long_effective_strike"],
    0.0,
)

spread_prices.loc[~spread_prices["outcome_available"], "terminal_spread_intrinsic_exact_primary"] = np.nan

spread_prices["expiry_pnl_exact_primary"] = (
    spread_prices["entry_credit_exact_primary"]
    - spread_prices["terminal_spread_intrinsic_exact_primary"]
)

spread_prices["return_on_max_loss_exact_primary"] = (
    spread_prices["expiry_pnl_exact_primary"]
    / spread_prices["max_loss_exact_primary"]
)

spread_prices["trade_win_exact_primary"] = spread_prices["expiry_pnl_exact_primary"] > 0

# Valid complete economics.
spread_prices["exact_primary_pnl_complete"] = (
    spread_prices["exact_primary_complete"]
    & spread_prices["outcome_available"]
    & spread_prices["entry_credit_exact_primary"].notna()
    & (spread_prices["entry_credit_exact_primary"] > 0)
    & spread_prices["max_loss_exact_primary"].notna()
    & (spread_prices["max_loss_exact_primary"] > 0)
    & spread_prices["terminal_spread_intrinsic_exact_primary"].notna()
    & spread_prices["return_on_max_loss_exact_primary"].notna()
)

def summarize_spreads(df):
    complete = df.loc[df["exact_primary_pnl_complete"]].copy()

    out = {
        "spread_rows": int(len(df)),
        "exact_primary_complete": int(df["exact_primary_complete"].sum()),
        "exact_primary_pnl_complete": int(df["exact_primary_pnl_complete"].sum()),
        "missing_or_invalid_pnl": int((~df["exact_primary_pnl_complete"]).sum()),
        "short_leg_usable": int(df["short_leg_price_usable"].sum()),
        "long_leg_usable": int(df["long_leg_price_usable"].sum()),
        "exact_primary_complete_rate": float(df["exact_primary_complete"].mean()) if len(df) else np.nan,
        "exact_primary_pnl_complete_rate": float(df["exact_primary_pnl_complete"].mean()) if len(df) else np.nan,
    }

    if len(complete):
        out.update({
            "win_rate": float(complete["trade_win_exact_primary"].mean()),
            "avg_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].mean()),
            "median_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].median()),
            "worst_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].min()),
            "p05_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].quantile(0.05)),
            "avg_entry_credit": float(complete["entry_credit_exact_primary"].mean()),
            "median_entry_credit": float(complete["entry_credit_exact_primary"].median()),
            "avg_credit_pct_width": float((complete["entry_credit_exact_primary"] / complete["effective_width_exact_primary"]).mean()),
            "median_credit_pct_width": float((complete["entry_credit_exact_primary"] / complete["effective_width_exact_primary"]).median()),
            "avg_width": float(complete["effective_width_exact_primary"].mean()),
            "median_width": float(complete["effective_width_exact_primary"].median()),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_width": np.nan,
            "median_width": np.nan,
        })

    return pd.Series(out)

coverage_summary = pd.DataFrame([summarize_spreads(spread_prices)])

coverage_by_tenor = (
    spread_prices
    .groupby("tenor", dropna=False)
    .apply(summarize_spreads)
    .reset_index()
)

coverage_by_year = (
    spread_prices
    .groupby("trade_year", dropna=False)
    .apply(summarize_spreads)
    .reset_index()
)

# Save joined outputs.
atomic_write_parquet(primary_contract_cache, OUT_PRIMARY_CONTRACT_CACHE_PATH)
atomic_write_parquet(leg_prices, OUT_PRIMARY_LEG_PRICES_PATH)
atomic_write_parquet(spread_prices, OUT_PRIMARY_SPREAD_PRICES_PATH)
atomic_write_parquet(coverage_summary, OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH)

# Audit CSVs.
probe_csv = AUDIT_DIR / f"02G_DATA_2_thetadata_probe_results_{OUT_SUFFIX}_{timestamp}.csv"
new_query_csv = AUDIT_DIR / f"02G_DATA_2_new_query_results_preview_{OUT_SUFFIX}_{timestamp}.csv"
cache_summary_csv = AUDIT_DIR / f"02G_DATA_2_primary_cache_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_summary_csv = AUDIT_DIR / f"02G_DATA_2_primary_spread_coverage_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_tenor_csv = AUDIT_DIR / f"02G_DATA_2_primary_spread_coverage_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_year_csv = AUDIT_DIR / f"02G_DATA_2_primary_spread_coverage_by_year_{OUT_SUFFIX}_{timestamp}.csv"
missing_contracts_preview_csv = AUDIT_DIR / f"02G_DATA_2_missing_contracts_after_primary_preview_{OUT_SUFFIX}_{timestamp}.csv"

if not probe_results_df.empty:
    atomic_write_csv(probe_results_df, probe_csv)

if not new_query_df.empty:
    atomic_write_csv(new_query_df.head(1000), new_query_csv)

atomic_write_csv(cache_post_summary, cache_summary_csv)
atomic_write_csv(coverage_summary, coverage_summary_csv)
atomic_write_csv(coverage_by_tenor, coverage_by_tenor_csv)
atomic_write_csv(coverage_by_year, coverage_by_year_csv)

missing_after = primary_contract_cache.loc[
    ~(primary_contract_cache["bid_usable"] & primary_contract_cache["ask_usable"])
].copy()

atomic_write_csv(missing_after.head(1000), missing_contracts_preview_csv)

manifest_path = AUDIT_DIR / f"02G_DATA_2_exact_primary_thetadata_pull_{OUT_SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.DATA.2",
    "description": "Exact primary ThetaData pull for full-history 1SD/3SD SPY call contracts",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "universe_path": str(UNIVERSE_PATH),
        "leg_request_plan_path": str(LEG_REQUEST_PLAN_PATH),
        "unique_contract_request_plan_path": str(UNIQUE_CONTRACT_REQUEST_PLAN_PATH),
        "contract_cache_overlap_path": str(CONTRACT_CACHE_OVERLAP_PATH),
        "contracts_needing_query_path": str(CONTRACTS_NEEDING_QUERY_PATH),
    },
    "thetadata": {
        "base_url": THETADATA_BASE_URL,
        "endpoint_candidates": THETADATA_ENDPOINT_CANDIDATES,
        "strike_param_formats_tried": STRIKE_PARAM_FORMATS_TO_TRY,
        "selected_endpoint": selected_endpoint,
        "selected_strike_param_format": selected_strike_format,
        "root": ROOT,
        "right": RIGHT,
        "probe_n_contracts": PROBE_N_CONTRACTS,
        "require_probe_success": REQUIRE_PROBE_SUCCESS,
        "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
        "retry_failed_contracts": RETRY_FAILED_CONTRACTS,
    },
    "counts": {
        "universe_spreads": int(len(universe)),
        "unique_contracts": int(len(contract_plan)),
        "contracts_missing_pre_query": int((~query_candidates["both_sides_usable"]).sum()),
        "contracts_eligible_to_query_this_run": int(len(to_query)),
        "contracts_queried_this_run": int(len(new_query_df)),
        "contracts_priced_this_run": int(new_query_df["price_found"].sum()) if not new_query_df.empty else 0,
        "contracts_usable_post_query": int((primary_contract_cache["bid_usable"] & primary_contract_cache["ask_usable"]).sum()),
        "contracts_still_missing_post_query": int(len(missing_after)),
        "spreads_exact_primary_complete": int(spread_prices["exact_primary_complete"].sum()),
        "spreads_exact_primary_pnl_complete": int(spread_prices["exact_primary_pnl_complete"].sum()),
    },
    "outputs": {
        "primary_contract_cache_path": str(OUT_PRIMARY_CONTRACT_CACHE_PATH),
        "primary_leg_prices_path": str(OUT_PRIMARY_LEG_PRICES_PATH),
        "primary_spread_prices_path": str(OUT_PRIMARY_SPREAD_PRICES_PATH),
        "primary_spread_coverage_summary_path": str(OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "next_step": "2G.DATA.3 fallback listed-strike search for legs still missing exact primary premiums.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display final diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.2 complete — exact primary ThetaData pull")
print("=" * 100)

print()
print("Primary contract cache summary")
display(cache_post_summary)

print()
print("=" * 100)
print("Exact-primary spread coverage summary")
print("=" * 100)
display(coverage_summary)

print()
print("=" * 100)
print("Exact-primary coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Exact-primary coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Missing contracts after exact-primary pull preview")
print("=" * 100)
if missing_after.empty:
    print("No missing contracts after exact-primary pull.")
else:
    display(missing_after.head(100))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Primary contract cache:          {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
print(f"Primary leg prices:              {OUT_PRIMARY_LEG_PRICES_PATH}")
print(f"Primary spread prices:           {OUT_PRIMARY_SPREAD_PRICES_PATH}")
print(f"Coverage summary:                {OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH}")
print(f"Manifest:                        {manifest_path}")

print()
print("Result: 2G.DATA.2 exact primary pull complete.")
print("Next step: 2G.DATA.3 fallback listed-strike search for still-missing legs.")

Cell 2G.DATA.2 — Exact primary ThetaData pull for full-history 1SD / 3SD call contracts
Universe:                  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_universe_fullhist_1sd3sd_long5_cal_v1.parquet
Leg request plan:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Contract cache overlap:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_fullhist_1sd3sd_long5_cal_v1.parquet
Contracts needing query:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_contracts_needing_exact_primary_query_fullhist_1sd3sd_long5_cal_v1.parquet
Output primary cache:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_cont

C:\Users\patri\AppData\Local\Temp\ipykernel_14668\4121431288.py:173: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)


,contracts,both_sides_usable,price_found,query_attempted,query_attempted_and_priced,query_attempted_not_priced,still_missing_both_sides,usable_rate
0,28523,2913,2913,0,0,0,25610,0.102128



Query plan
Contracts still missing both sides:  25,610
Eligible to query this run:          25,610
Actually querying this run:          25,610

ThetaData endpoint probe
Probe contracts: 20

Probe /v3/option/history/greeks/eod / thousandths: successes=0, http_200=0, http_410=20, parsed_rows=0
Probe /v3/option/history/greeks/eod / float: successes=0, http_200=0, http_410=20, parsed_rows=0
Probe /v3/option/history/quote/eod / thousandths: successes=0, http_200=0, http_410=20, parsed_rows=0
Probe /v3/option/history/quote/eod / float: successes=0, http_200=0, http_410=20, parsed_rows=0
Probe /v3/option/history/eod / thousandths: successes=0, http_200=0, http_410=20, parsed_rows=0
Probe /v3/option/history/eod / float: successes=0, http_200=0, http_410=20, parsed_rows=0

Probe combo summary


,endpoint,strike_param_format,probe_contracts,price_found,http_200,http_410,parsed_rows
0,/v3/option/history/eod,float,20,0,0,20,0
1,/v3/option/history/eod,thousandths,20,0,0,20,0
2,/v3/option/history/greeks/eod,float,20,0,0,20,0
3,/v3/option/history/greeks/eod,thousandths,20,0,0,20,0
4,/v3/option/history/quote/eod,float,20,0,0,20,0
5,/v3/option/history/quote/eod,thousandths,20,0,0,20,0



STOPPING: ThetaData probe found no usable bid/ask rows
Probe audit saved: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_2_thetadata_probe_results_fullhist_1sd3sd_long5_cal_v1_20260712_210102.csv
No bulk query was attempted.
This likely means the current ThetaData endpoint/query format needs to be fixed.


RuntimeError: ThetaData probe failed: no endpoint/strike-format combination returned usable bid/ask data.

In [14]:
# ThetaData v3 corrected smoke test — do not bulk query
# Purpose:
#   Confirm the correct v3 endpoint/parameter format before rerunning 2G.DATA.2.

from pathlib import Path
import pandas as pd
import requests
from io import StringIO

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

CONTRACTS_NEEDING_QUERY_PATH = (
    PROCESSED_DIR
    / f"call_sleeve_corsi_call_full_history_1sd3sd_contracts_needing_exact_primary_query_{OUT_SUFFIX}.parquet"
)

THETADATA_BASE_URL = "http://127.0.0.1:25503"
ENDPOINTS = [
    "/v3/option/history/eod",
    "/v3/option/history/greeks/eod",
]

def parse_response(resp):
    text = (resp.text or "").strip()

    if resp.status_code != 200 or not text:
        return pd.DataFrame()

    try:
        data = resp.json()
        if isinstance(data, dict):
            if "header" in data and "response" in data:
                return pd.DataFrame(data["response"], columns=data["header"])
            if "response" in data and isinstance(data["response"], list):
                return pd.DataFrame(data["response"])
            if "data" in data and isinstance(data["data"], list):
                return pd.DataFrame(data["data"])
            return pd.DataFrame([data])
        if isinstance(data, list):
            return pd.DataFrame(data)
    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        return pd.DataFrame()

contracts = pd.read_parquet(CONTRACTS_NEEDING_QUERY_PATH).copy()
contracts["trade_date"] = pd.to_datetime(contracts["trade_date"]).dt.normalize()
contracts["expiration_date"] = pd.to_datetime(contracts["expiration_date"]).dt.normalize()
contracts["strike"] = pd.to_numeric(contracts["strike"], errors="coerce")

# Use a few contracts from different dates so one bad old/illiquid contract does not fool us.
probe = (
    contracts
    .dropna(subset=["trade_date", "expiration_date", "strike"])
    .sort_values(["trade_date", "expiration_date", "strike"])
    .iloc[[0, 10, 50, 100, 500]]
    .copy()
)

rows = []

with requests.Session() as session:
    for endpoint in ENDPOINTS:
        for _, r in probe.iterrows():
            params = {
                "symbol": "SPY",
                "expiration": r["expiration_date"].strftime("%Y%m%d"),
                "strike": f"{float(r['strike']):.3f}",
                "right": "call",
                "start_date": r["trade_date"].strftime("%Y%m%d"),
                "end_date": r["trade_date"].strftime("%Y%m%d"),
                "format": "json",
            }

            url = THETADATA_BASE_URL + endpoint
            resp = session.get(url, params=params, timeout=30)
            parsed = parse_response(resp)

            parsed_cols = [str(c).lower() for c in parsed.columns] if not parsed.empty else []
            bid = None
            ask = None

            if not parsed.empty:
                last = parsed.copy()
                last.columns = parsed_cols
                row = last.iloc[-1]

                if "bid" in last.columns:
                    bid = row["bid"]
                if "ask" in last.columns:
                    ask = row["ask"]

            rows.append({
                "endpoint": endpoint,
                "contract_request_id": r["contract_request_id"],
                "trade_date": r["trade_date"].strftime("%Y-%m-%d"),
                "expiration": r["expiration_date"].strftime("%Y-%m-%d"),
                "strike": float(r["strike"]),
                "http_status_code": resp.status_code,
                "parsed_rows": len(parsed),
                "columns": ",".join(parsed_cols[:20]),
                "bid": bid,
                "ask": ask,
                "request_url": resp.url,
                "response_preview": (resp.text or "")[:250],
            })

smoke = pd.DataFrame(rows)
display(smoke)

print()
print("HTTP status counts")
display(smoke.groupby(["endpoint", "http_status_code"]).size().reset_index(name="count"))

print()
print("Rows with bid/ask")
display(smoke.loc[smoke["bid"].notna() | smoke["ask"].notna()])

,endpoint,contract_request_id,trade_date,expiration,strike,http_status_code,parsed_rows,columns,bid,ask,request_url,response_preview
0,/v3/option/history/eod,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.35, 0.35]","[0.36, 0.36]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1921,1921]..."
1,/v3/option/history/eod,SPY_20200102_20200207_337.0_C,2020-01-02,2020-02-07,337.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.33, 0.33]","[0.35, 0.35]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2046,2046]..."
2,/v3/option/history/eod,SPY_20200107_20200131_332.0_C,2020-01-07,2020-01-31,332.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.28, 0.28]","[0.29, 0.29]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[356,356],""..."
3,/v3/option/history/eod,SPY_20200110_20200207_336.0_C,2020-01-10,2020-02-07,336.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.25, 0.25]","[0.26, 0.26]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2106,2106]..."
4,/v3/option/history/eod,SPY_20200219_20200228_345.0_C,2020-02-19,2020-02-28,345.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.16, 0.16]","[0.17, 0.17]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[6120,6120]..."
5,/v3/option/history/greeks/eod,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.35],[0.36],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[1921],""dual_delt..."
6,/v3/option/history/greeks/eod,SPY_20200102_20200207_337.0_C,2020-01-02,2020-02-07,337.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.33],[0.35],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[2046],""dual_delt..."
7,/v3/option/history/greeks/eod,SPY_20200107_20200131_332.0_C,2020-01-07,2020-01-31,332.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.28],[0.29],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[356],""dual_delta..."
8,/v3/option/history/greeks/eod,SPY_20200110_20200207_336.0_C,2020-01-10,2020-02-07,336.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.25],[0.26],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[2106],""dual_delt..."
9,/v3/option/history/greeks/eod,SPY_20200219_20200228_345.0_C,2020-02-19,2020-02-28,345.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.16],[0.17],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[6120],""dual_delt..."



HTTP status counts


,endpoint,http_status_code,count
0,/v3/option/history/eod,200,5
1,/v3/option/history/greeks/eod,200,5



Rows with bid/ask


,endpoint,contract_request_id,trade_date,expiration,strike,http_status_code,parsed_rows,columns,bid,ask,request_url,response_preview
0,/v3/option/history/eod,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.35, 0.35]","[0.36, 0.36]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1921,1921]..."
1,/v3/option/history/eod,SPY_20200102_20200207_337.0_C,2020-01-02,2020-02-07,337.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.33, 0.33]","[0.35, 0.35]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2046,2046]..."
2,/v3/option/history/eod,SPY_20200107_20200131_332.0_C,2020-01-07,2020-01-31,332.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.28, 0.28]","[0.29, 0.29]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[356,356],""..."
3,/v3/option/history/eod,SPY_20200110_20200207_336.0_C,2020-01-10,2020-02-07,336.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.25, 0.25]","[0.26, 0.26]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2106,2106]..."
4,/v3/option/history/eod,SPY_20200219_20200228_345.0_C,2020-02-19,2020-02-28,345.0,200,1,"symbol,ask_size,last_trade,created,ask_conditi...","[0.16, 0.16]","[0.17, 0.17]",http://127.0.0.1:25503/v3/option/history/eod?s...,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[6120,6120]..."
5,/v3/option/history/greeks/eod,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.35],[0.36],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[1921],""dual_delt..."
6,/v3/option/history/greeks/eod,SPY_20200102_20200207_337.0_C,2020-01-02,2020-02-07,337.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.33],[0.35],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[2046],""dual_delt..."
7,/v3/option/history/greeks/eod,SPY_20200107_20200131_332.0_C,2020-01-07,2020-01-31,332.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.28],[0.29],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[356],""dual_delta..."
8,/v3/option/history/greeks/eod,SPY_20200110_20200207_336.0_C,2020-01-10,2020-02-07,336.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.25],[0.26],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[2106],""dual_delt..."
9,/v3/option/history/greeks/eod,SPY_20200219_20200228_345.0_C,2020-02-19,2020-02-28,345.0,200,1,"symbol,ask_size,dual_delta,color,zomma,delta,i...",[0.16],[0.17],http://127.0.0.1:25503/v3/option/history/greek...,"{""symbol"":[""SPY""],""ask_size"":[6120],""dual_delt..."


In [1]:
# Cell 2G.DATA.2 — PATCHED exact primary ThetaData pull for full-history 1SD / 3SD call contracts
# Purpose:
#   Pull missing exact SPY call bid/ask premiums for the full historical 1SD / 3SD call
#   request plan built in 2G.DATA.1.
#
# Corrected ThetaData v3 schema:
#   endpoint   = /v3/option/history/eod
#   symbol     = SPY
#   expiration = YYYYMMDD
#   strike     = 331.000
#   right      = call
#   start_date = YYYYMMDD
#   end_date   = YYYYMMDD
#   format     = json
#
# This is a DATA LAYER cell:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule

from pathlib import Path
from datetime import datetime
from io import StringIO
import json
import time

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

LEG_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_{OUT_SUFFIX}.parquet"
)

UNIQUE_CONTRACT_REQUEST_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_unique_contract_request_plan_{OUT_SUFFIX}.parquet"
)

CONTRACT_CACHE_OVERLAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_CONTRACT_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_LEG_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_leg_prices_joined_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_SPREAD_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_prices_{OUT_SUFFIX}.parquet"
)

OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_coverage_summary_{OUT_SUFFIX}.parquet"
)

THETADATA_BASE_URL = "http://127.0.0.1:25503"
THETADATA_ENDPOINT = "/v3/option/history/eod"
THETADATA_URL = THETADATA_BASE_URL + THETADATA_ENDPOINT

SYMBOL = "SPY"
RIGHT = "call"

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 0.01

PROBE_N_CONTRACTS = 10

# None = query all missing contracts.
# For a controlled partial run, set this to e.g. 1000.
MAX_REQUESTS_THIS_RUN = None

SAVE_EVERY_N_ATTEMPTS = 250
PROGRESS_EVERY_N_ATTEMPTS = 250

# False = resume safely and do not retry previously attempted failures.
# True = retry all still-missing rows even if already attempted.
RETRY_FAILED_CONTRACTS = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.DATA.2 — PATCHED exact primary ThetaData pull")
print("=" * 100)
print(f"Universe:               {UNIVERSE_PATH}")
print(f"Leg request plan:       {LEG_REQUEST_PLAN_PATH}")
print(f"Contract overlap:       {CONTRACT_CACHE_OVERLAP_PATH}")
print(f"Output primary cache:   {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
print(f"ThetaData URL:          {THETADATA_URL}")
print(f"Max requests this run:  {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed contracts: {RETRY_FAILED_CONTRACTS}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def yyyymmdd(x):
    return pd.Timestamp(x).strftime("%Y%m%d")

def extract_first_existing(row, candidates):
    for c in candidates:
        if c in row.index:
            val = row[c]
            if pd.notna(val):
                return val
    return np.nan

def json_dict_is_column_oriented(data):
    if not isinstance(data, dict):
        return False

    if not data:
        return False

    list_lengths = []
    scalar_count = 0

    for v in data.values():
        if isinstance(v, list):
            list_lengths.append(len(v))
        else:
            scalar_count += 1

    if not list_lengths:
        return False

    # ThetaData eod response can be {"bid": [..], "ask": [..], ...}
    # Allow scalar fields too, but require at least one list field.
    return len(set(list_lengths)) == 1

def parse_thetadata_response(resp):
    """
    Robust parser for ThetaData local terminal responses.

    Handles:
      1. {"header": [...], "response": [[...], ...]}
      2. {"bid": [0.35, 0.35], "ask": [0.36, 0.36], ...}
      3. [{"bid": 0.35, "ask": 0.36, ...}]
      4. CSV-like text
    """
    text = (resp.text or "").strip()

    if resp.status_code != 200 or not text:
        return pd.DataFrame()

    try:
        data = resp.json()

        if isinstance(data, dict):
            if "header" in data and "response" in data:
                return pd.DataFrame(data["response"], columns=data["header"])

            if "data" in data and isinstance(data["data"], list):
                return pd.DataFrame(data["data"])

            if "response" in data and isinstance(data["response"], list):
                return pd.DataFrame(data["response"])

            if json_dict_is_column_oriented(data):
                return pd.DataFrame(data)

            return pd.DataFrame([data])

        if isinstance(data, list):
            return pd.DataFrame(data)

    except Exception:
        pass

    try:
        df = pd.read_csv(StringIO(text))
        return df
    except Exception:
        return pd.DataFrame()

def normalize_response_columns(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

def extract_quote_from_parsed_df(parsed):
    if parsed.empty:
        return {
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
        }

    parsed = normalize_response_columns(parsed)

    # Use the last row returned by ThetaData.
    row = parsed.iloc[-1]

    bid = pd.to_numeric(
        extract_first_existing(row, ["bid", "bid_price", "close_bid", "eod_bid"]),
        errors="coerce",
    )

    ask = pd.to_numeric(
        extract_first_existing(row, ["ask", "ask_price", "close_ask", "eod_ask"]),
        errors="coerce",
    )

    mid = pd.to_numeric(
        extract_first_existing(row, ["mid", "mid_price", "mark", "price", "close", "eod_price"]),
        errors="coerce",
    )

    if pd.isna(mid) and pd.notna(bid) and pd.notna(ask):
        mid = (bid + ask) / 2.0

    iv = pd.to_numeric(
        extract_first_existing(row, ["iv", "implied_vol", "implied_volatility"]),
        errors="coerce",
    )

    delta = pd.to_numeric(extract_first_existing(row, ["delta"]), errors="coerce")
    gamma = pd.to_numeric(extract_first_existing(row, ["gamma"]), errors="coerce")
    theta = pd.to_numeric(extract_first_existing(row, ["theta"]), errors="coerce")
    vega = pd.to_numeric(extract_first_existing(row, ["vega"]), errors="coerce")
    rho = pd.to_numeric(extract_first_existing(row, ["rho"]), errors="coerce")

    price_found = (
        pd.notna(bid)
        and pd.notna(ask)
        and float(bid) >= 0
        and float(ask) >= 0
    )

    return {
        "price_found": bool(price_found),
        "bid": float(bid) if pd.notna(bid) else np.nan,
        "ask": float(ask) if pd.notna(ask) else np.nan,
        "mid": float(mid) if pd.notna(mid) else np.nan,
        "iv": float(iv) if pd.notna(iv) else np.nan,
        "delta": float(delta) if pd.notna(delta) else np.nan,
        "gamma": float(gamma) if pd.notna(gamma) else np.nan,
        "theta": float(theta) if pd.notna(theta) else np.nan,
        "vega": float(vega) if pd.notna(vega) else np.nan,
        "rho": float(rho) if pd.notna(rho) else np.nan,
    }

def query_contract(session, contract_row):
    trade_date = pd.Timestamp(contract_row["trade_date"]).normalize()
    expiration_date = pd.Timestamp(contract_row["expiration_date"]).normalize()
    strike = float(contract_row["strike"])

    params = {
        "symbol": SYMBOL,
        "expiration": yyyymmdd(expiration_date),
        "strike": f"{strike:.3f}",
        "right": RIGHT,
        "start_date": yyyymmdd(trade_date),
        "end_date": yyyymmdd(trade_date),
        "format": "json",
    }

    try:
        resp = session.get(
            THETADATA_URL,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        parsed = parse_thetadata_response(resp)
        quote = extract_quote_from_parsed_df(parsed)

        return {
            "contract_request_id": contract_row["contract_request_id"],
            "trade_date": trade_date,
            "expiration_date": expiration_date,
            "strike": strike,
            "root": SYMBOL,
            "right": RIGHT,
            "price_found": bool(quote["price_found"]),
            "bid": quote["bid"],
            "ask": quote["ask"],
            "mid": quote["mid"],
            "iv": quote["iv"],
            "delta": quote["delta"],
            "gamma": quote["gamma"],
            "theta": quote["theta"],
            "vega": quote["vega"],
            "rho": quote["rho"],
            "status": "priced" if quote["price_found"] else "no_price",
            "status_code": int(resp.status_code),
            "http_status_code": int(resp.status_code),
            "request_ok": True,
            "endpoint": THETADATA_ENDPOINT,
            "request_url": resp.url,
            "response_preview": (resp.text or "")[:300],
            "parsed_rows": int(len(parsed)),
            "error_message": "",
            "query_attempted": True,
            "cache_source": "thetadata_exact_primary_fullhist",
            "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        return {
            "contract_request_id": contract_row["contract_request_id"],
            "trade_date": trade_date,
            "expiration_date": expiration_date,
            "strike": strike,
            "root": SYMBOL,
            "right": RIGHT,
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
            "status": "request_error",
            "status_code": np.nan,
            "http_status_code": np.nan,
            "request_ok": False,
            "endpoint": THETADATA_ENDPOINT,
            "request_url": "",
            "response_preview": "",
            "parsed_rows": 0,
            "error_message": repr(e),
            "query_attempted": True,
            "cache_source": "thetadata_exact_primary_fullhist",
            "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

def build_seed_cache_from_overlap(overlap):
    out = overlap.copy()

    for c in ["trade_date", "expiration_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.normalize()

    for c in ["price_found"]:
        if c not in out.columns:
            out[c] = False

    out["price_found"] = bool_series(out["price_found"], index=out.index)

    for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho", "strike"]:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    for c in ["status", "status_code", "cache_source"]:
        if c not in out.columns:
            out[c] = np.nan

    out["root"] = SYMBOL
    out["right"] = RIGHT
    out["http_status_code"] = out["status_code"]
    out["request_ok"] = np.nan
    out["endpoint"] = np.nan
    out["request_url"] = np.nan
    out["response_preview"] = np.nan
    out["parsed_rows"] = np.nan
    out["error_message"] = ""
    out["query_attempted"] = False
    out["queried_at"] = np.nan
    out["seed_cache_source"] = out["cache_source"]

    keep = [
        "contract_request_id",
        "trade_date",
        "expiration_date",
        "strike",
        "root",
        "right",
        "price_found",
        "bid",
        "ask",
        "mid",
        "iv",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "status",
        "status_code",
        "http_status_code",
        "request_ok",
        "endpoint",
        "request_url",
        "response_preview",
        "parsed_rows",
        "error_message",
        "query_attempted",
        "cache_source",
        "seed_cache_source",
        "queried_at",
    ]

    for c in keep:
        if c not in out.columns:
            out[c] = np.nan

    return out[keep].copy()

def combine_cache_rows(seed_cache, prior_cache, new_query_rows):
    frames = []

    if seed_cache is not None and len(seed_cache):
        frames.append(seed_cache.copy())

    if prior_cache is not None and len(prior_cache):
        frames.append(prior_cache.copy())

    if new_query_rows is not None and len(new_query_rows):
        frames.append(new_query_rows.copy())

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)

    for c in ["trade_date", "expiration_date"]:
        if c in combined.columns:
            combined[c] = pd.to_datetime(combined[c], errors="coerce").dt.normalize()

    combined["price_found"] = bool_series(combined.get("price_found"), index=combined.index)
    combined["query_attempted"] = bool_series(combined.get("query_attempted"), index=combined.index)

    for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho", "strike"]:
        if c in combined.columns:
            combined[c] = pd.to_numeric(combined[c], errors="coerce")

    combined["bid_usable"] = combined["bid"].notna() & (combined["bid"] >= 0)
    combined["ask_usable"] = combined["ask"].notna() & (combined["ask"] >= 0)
    combined["both_sides_usable"] = combined["bid_usable"] & combined["ask_usable"]

    combined["row_preference"] = np.select(
        [
            combined["both_sides_usable"] & combined["query_attempted"],
            combined["both_sides_usable"] & ~combined["query_attempted"],
            ~combined["both_sides_usable"] & combined["query_attempted"],
        ],
        [0, 1, 2],
        default=3,
    )

    combined = (
        combined
        .sort_values(["contract_request_id", "row_preference"])
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["row_preference"])
        .reset_index(drop=True)
    )

    return combined

def summarize_primary_cache(cache):
    tmp = cache.copy()

    tmp["price_found"] = bool_series(tmp.get("price_found"), index=tmp.index)
    tmp["query_attempted"] = bool_series(tmp.get("query_attempted"), index=tmp.index)
    tmp["bid_usable"] = tmp["bid"].notna() & (tmp["bid"] >= 0)
    tmp["ask_usable"] = tmp["ask"].notna() & (tmp["ask"] >= 0)
    tmp["both_sides_usable"] = tmp["bid_usable"] & tmp["ask_usable"]

    return pd.DataFrame([{
        "contracts": int(len(tmp)),
        "both_sides_usable": int(tmp["both_sides_usable"].sum()),
        "price_found": int(tmp["price_found"].sum()),
        "query_attempted": int(tmp["query_attempted"].sum()),
        "query_attempted_and_priced": int((tmp["query_attempted"] & tmp["both_sides_usable"]).sum()),
        "query_attempted_not_priced": int((tmp["query_attempted"] & ~tmp["both_sides_usable"]).sum()),
        "still_missing_both_sides": int((~tmp["both_sides_usable"]).sum()),
        "usable_rate": float(tmp["both_sides_usable"].mean()) if len(tmp) else np.nan,
    }])

def summarize_spreads(df):
    complete = df.loc[df["exact_primary_pnl_complete"]].copy()

    out = {
        "spread_rows": int(len(df)),
        "exact_primary_complete": int(df["exact_primary_complete"].sum()),
        "exact_primary_pnl_complete": int(df["exact_primary_pnl_complete"].sum()),
        "missing_or_invalid_pnl": int((~df["exact_primary_pnl_complete"]).sum()),
        "short_leg_usable": int(df["short_leg_price_usable"].sum()),
        "long_leg_usable": int(df["long_leg_price_usable"].sum()),
        "exact_primary_complete_rate": float(df["exact_primary_complete"].mean()) if len(df) else np.nan,
        "exact_primary_pnl_complete_rate": float(df["exact_primary_pnl_complete"].mean()) if len(df) else np.nan,
    }

    if len(complete):
        out.update({
            "win_rate": float(complete["trade_win_exact_primary"].mean()),
            "avg_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].mean()),
            "median_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].median()),
            "worst_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].min()),
            "p05_return_on_max_loss": float(complete["return_on_max_loss_exact_primary"].quantile(0.05)),
            "avg_entry_credit": float(complete["entry_credit_exact_primary"].mean()),
            "median_entry_credit": float(complete["entry_credit_exact_primary"].median()),
            "avg_credit_pct_width": float((complete["entry_credit_exact_primary"] / complete["effective_width_exact_primary"]).mean()),
            "median_credit_pct_width": float((complete["entry_credit_exact_primary"] / complete["effective_width_exact_primary"]).median()),
            "avg_width": float(complete["effective_width_exact_primary"].mean()),
            "median_width": float(complete["effective_width_exact_primary"].median()),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_width": np.nan,
            "median_width": np.nan,
        })

    return pd.Series(out)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    LEG_REQUEST_PLAN_PATH,
    UNIQUE_CONTRACT_REQUEST_PLAN_PATH,
    CONTRACT_CACHE_OVERLAP_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
leg_plan = pd.read_parquet(LEG_REQUEST_PLAN_PATH).copy()
contract_plan = pd.read_parquet(UNIQUE_CONTRACT_REQUEST_PLAN_PATH).copy()
contract_overlap = pd.read_parquet(CONTRACT_CACHE_OVERLAP_PATH).copy()

for df in [universe, leg_plan, contract_plan, contract_overlap]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Universe spread rows:      {len(universe):,}")
print(f"Leg plan rows:             {len(leg_plan):,}")
print(f"Unique contract rows:      {len(contract_plan):,}")
print(f"Contract overlap rows:     {len(contract_overlap):,}")
print()

# --------------------------------------------------------------------------------------------------
# Seed cache / resume
# --------------------------------------------------------------------------------------------------

seed_cache = build_seed_cache_from_overlap(contract_overlap)

if OUT_PRIMARY_CONTRACT_CACHE_PATH.exists():
    prior_cache = pd.read_parquet(OUT_PRIMARY_CONTRACT_CACHE_PATH).copy()
    print(f"Existing full-history primary cache found: {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
    print(f"Existing rows: {len(prior_cache):,}")
else:
    prior_cache = pd.DataFrame()
    print("No existing full-history primary cache found. Starting from seed overlap.")

combined_cache_pre = combine_cache_rows(seed_cache, prior_cache, pd.DataFrame())
cache_pre_summary = summarize_primary_cache(combined_cache_pre)

print()
print("=" * 100)
print("Pre-query primary cache summary")
print("=" * 100)
display(cache_pre_summary)

query_candidates = contract_plan.merge(
    combined_cache_pre[
        [
            "contract_request_id",
            "both_sides_usable",
            "query_attempted",
        ]
    ],
    on="contract_request_id",
    how="left",
    validate="one_to_one",
)

query_candidates["both_sides_usable"] = bool_series(query_candidates["both_sides_usable"], index=query_candidates.index)
query_candidates["query_attempted"] = bool_series(query_candidates["query_attempted"], index=query_candidates.index)

if RETRY_FAILED_CONTRACTS:
    to_query = query_candidates.loc[~query_candidates["both_sides_usable"]].copy()
else:
    to_query = query_candidates.loc[
        ~query_candidates["both_sides_usable"]
        & ~query_candidates["query_attempted"]
    ].copy()

to_query = to_query.sort_values(["trade_date", "expiration_date", "strike"]).reset_index(drop=True)

if MAX_REQUESTS_THIS_RUN is not None:
    to_query_run = to_query.head(int(MAX_REQUESTS_THIS_RUN)).copy()
else:
    to_query_run = to_query.copy()

print()
print("=" * 100)
print("Query plan")
print("=" * 100)
print(f"Contracts still missing both sides: {(~query_candidates['both_sides_usable']).sum():,}")
print(f"Eligible to query this run:         {len(to_query):,}")
print(f"Actually querying this run:         {len(to_query_run):,}")
print()

# --------------------------------------------------------------------------------------------------
# Probe corrected endpoint
# --------------------------------------------------------------------------------------------------

probe_results_df = pd.DataFrame()

if not to_query_run.empty:
    probe_contracts = to_query_run.head(PROBE_N_CONTRACTS).copy()
    probe_rows = []

    print("=" * 100)
    print("Corrected ThetaData endpoint probe")
    print("=" * 100)

    with requests.Session() as session:
        for _, row in probe_contracts.iterrows():
            result = query_contract(session, row)
            probe_rows.append(result)
            time.sleep(REQUEST_SLEEP_SECONDS)

    probe_results_df = pd.DataFrame(probe_rows)

    probe_success = int(probe_results_df["price_found"].sum())
    probe_http_200 = int((probe_results_df["http_status_code"] == 200).sum())

    print(f"Probe contracts: {len(probe_results_df):,}")
    print(f"Probe HTTP 200:  {probe_http_200:,}")
    print(f"Probe priced:    {probe_success:,}")

    display(
        probe_results_df[
            [
                "contract_request_id",
                "trade_date",
                "expiration_date",
                "strike",
                "price_found",
                "bid",
                "ask",
                "http_status_code",
                "parsed_rows",
                "response_preview",
            ]
        ]
    )

    if probe_success == 0:
        probe_path = AUDIT_DIR / f"02G_DATA_2_PATCHED_probe_failed_{OUT_SUFFIX}_{timestamp}.csv"
        atomic_write_csv(probe_results_df, probe_path)

        print()
        print("=" * 100)
        print("STOPPING: corrected probe returned zero priced contracts")
        print("=" * 100)
        print(f"Probe audit saved: {probe_path}")
        raise RuntimeError("Corrected ThetaData probe failed: zero priced contracts.")

# --------------------------------------------------------------------------------------------------
# Main query loop
# --------------------------------------------------------------------------------------------------

new_query_rows = []

if not to_query_run.empty:
    print()
    print("=" * 100)
    print("Starting main corrected ThetaData query loop")
    print("=" * 100)

    start_time = time.time()

    with requests.Session() as session:
        for i, (_, row) in enumerate(to_query_run.iterrows(), start=1):
            out_row = query_contract(session, row)
            new_query_rows.append(out_row)

            if (i % PROGRESS_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                elapsed = time.time() - start_time
                priced_so_far = sum(1 for r in new_query_rows if r["price_found"])
                http_200_so_far = sum(1 for r in new_query_rows if r["http_status_code"] == 200)
                not_priced_so_far = i - priced_so_far

                print(
                    f"Progress {i:,}/{len(to_query_run):,} | "
                    f"http_200={http_200_so_far:,} | "
                    f"priced={priced_so_far:,} | "
                    f"not_priced={not_priced_so_far:,} | "
                    f"elapsed={elapsed/60:.1f} min"
                )

            if (i % SAVE_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                new_query_df_tmp = pd.DataFrame(new_query_rows)
                combined_tmp = combine_cache_rows(seed_cache, prior_cache, new_query_df_tmp)
                atomic_write_parquet(combined_tmp, OUT_PRIMARY_CONTRACT_CACHE_PATH)

            time.sleep(REQUEST_SLEEP_SECONDS)

new_query_df = pd.DataFrame(new_query_rows)

primary_contract_cache = combine_cache_rows(seed_cache, prior_cache, new_query_df)
atomic_write_parquet(primary_contract_cache, OUT_PRIMARY_CONTRACT_CACHE_PATH)

# --------------------------------------------------------------------------------------------------
# Rebuild exact-primary leg/spread coverage
# --------------------------------------------------------------------------------------------------

cache_for_join = primary_contract_cache.copy()

for c in ["trade_date", "expiration_date"]:
    if c in cache_for_join.columns:
        cache_for_join[c] = pd.to_datetime(cache_for_join[c], errors="coerce").dt.normalize()

for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"]:
    if c in cache_for_join.columns:
        cache_for_join[c] = pd.to_numeric(cache_for_join[c], errors="coerce")
    else:
        cache_for_join[c] = np.nan

cache_for_join["bid_usable"] = cache_for_join["bid"].notna() & (cache_for_join["bid"] >= 0)
cache_for_join["ask_usable"] = cache_for_join["ask"].notna() & (cache_for_join["ask"] >= 0)

leg_prices = leg_plan.merge(
    cache_for_join[
        [
            "contract_request_id",
            "price_found",
            "bid",
            "ask",
            "mid",
            "iv",
            "delta",
            "gamma",
            "theta",
            "vega",
            "rho",
            "bid_usable",
            "ask_usable",
            "status",
            "status_code",
            "http_status_code",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
            "endpoint",
        ]
    ],
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

leg_prices["bid_usable"] = bool_series(leg_prices["bid_usable"], index=leg_prices.index)
leg_prices["ask_usable"] = bool_series(leg_prices["ask_usable"], index=leg_prices.index)

leg_prices["leg_price_usable"] = np.where(
    leg_prices["needed_price_field"].eq("bid"),
    leg_prices["bid_usable"],
    leg_prices["ask_usable"],
).astype(bool)

spread_base_cols = [
    "selected_trade_id",
    "trade_date",
    "trade_year",
    "tenor",
    "expiration_date",
    "expiration_str",
    "outcome_available",
    "short_strike",
    "long_strike",
    "spy_close",
    "expiry_spy_close",
    "vix_style_vol_pct",
    "sd_move",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "width",
]

spread_base = universe[spread_base_cols].drop_duplicates("selected_trade_id").copy()

short_leg = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("short_call_1sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "strike",
            "bid",
            "ask",
            "mid",
            "leg_price_usable",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "short_contract_request_id",
            "strike": "short_effective_strike",
            "bid": "short_bid",
            "ask": "short_ask",
            "mid": "short_mid",
            "leg_price_usable": "short_leg_price_usable",
            "query_attempted": "short_query_attempted",
            "cache_source": "short_cache_source",
            "seed_cache_source": "short_seed_cache_source",
        }
    )
)

long_leg = (
    leg_prices
    .loc[leg_prices["leg_label"].eq("long_call_3sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "strike",
            "bid",
            "ask",
            "mid",
            "leg_price_usable",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "long_contract_request_id",
            "strike": "long_effective_strike",
            "bid": "long_bid",
            "ask": "long_ask",
            "mid": "long_mid",
            "leg_price_usable": "long_leg_price_usable",
            "query_attempted": "long_query_attempted",
            "cache_source": "long_cache_source",
            "seed_cache_source": "long_seed_cache_source",
        }
    )
)

spread_prices = (
    spread_base
    .merge(short_leg, on="selected_trade_id", how="left", validate="one_to_one")
    .merge(long_leg, on="selected_trade_id", how="left", validate="one_to_one")
)

spread_prices["short_leg_price_usable"] = bool_series(spread_prices["short_leg_price_usable"], index=spread_prices.index)
spread_prices["long_leg_price_usable"] = bool_series(spread_prices["long_leg_price_usable"], index=spread_prices.index)

spread_prices["exact_primary_complete"] = (
    spread_prices["short_leg_price_usable"]
    & spread_prices["long_leg_price_usable"]
)

spread_prices["entry_credit_exact_primary"] = spread_prices["short_bid"] - spread_prices["long_ask"]
spread_prices["effective_width_exact_primary"] = spread_prices["long_effective_strike"] - spread_prices["short_effective_strike"]
spread_prices["max_loss_exact_primary"] = (
    spread_prices["effective_width_exact_primary"]
    - spread_prices["entry_credit_exact_primary"]
)

spread_prices["terminal_spread_intrinsic_exact_primary"] = np.maximum(
    spread_prices["expiry_spy_close"] - spread_prices["short_effective_strike"],
    0.0,
) - np.maximum(
    spread_prices["expiry_spy_close"] - spread_prices["long_effective_strike"],
    0.0,
)

spread_prices.loc[~spread_prices["outcome_available"], "terminal_spread_intrinsic_exact_primary"] = np.nan

spread_prices["expiry_pnl_exact_primary"] = (
    spread_prices["entry_credit_exact_primary"]
    - spread_prices["terminal_spread_intrinsic_exact_primary"]
)

spread_prices["return_on_max_loss_exact_primary"] = (
    spread_prices["expiry_pnl_exact_primary"]
    / spread_prices["max_loss_exact_primary"]
)

spread_prices["trade_win_exact_primary"] = spread_prices["expiry_pnl_exact_primary"] > 0

spread_prices["exact_primary_pnl_complete"] = (
    spread_prices["exact_primary_complete"]
    & spread_prices["outcome_available"]
    & spread_prices["entry_credit_exact_primary"].notna()
    & (spread_prices["entry_credit_exact_primary"] > 0)
    & spread_prices["max_loss_exact_primary"].notna()
    & (spread_prices["max_loss_exact_primary"] > 0)
    & spread_prices["terminal_spread_intrinsic_exact_primary"].notna()
    & spread_prices["return_on_max_loss_exact_primary"].notna()
)

coverage_summary = pd.DataFrame([summarize_spreads(spread_prices)])

coverage_by_tenor = (
    spread_prices
    .groupby("tenor", dropna=False)
    .apply(summarize_spreads)
    .reset_index()
)

coverage_by_year = (
    spread_prices
    .groupby("trade_year", dropna=False)
    .apply(summarize_spreads)
    .reset_index()
)

cache_post_summary = summarize_primary_cache(primary_contract_cache)

# --------------------------------------------------------------------------------------------------
# Save outputs / audits
# --------------------------------------------------------------------------------------------------

atomic_write_parquet(primary_contract_cache, OUT_PRIMARY_CONTRACT_CACHE_PATH)
atomic_write_parquet(leg_prices, OUT_PRIMARY_LEG_PRICES_PATH)
atomic_write_parquet(spread_prices, OUT_PRIMARY_SPREAD_PRICES_PATH)
atomic_write_parquet(coverage_summary, OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH)

probe_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_probe_results_{OUT_SUFFIX}_{timestamp}.csv"
new_query_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_new_query_results_preview_{OUT_SUFFIX}_{timestamp}.csv"
cache_summary_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_primary_cache_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_summary_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_primary_spread_coverage_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_tenor_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_primary_spread_coverage_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_year_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_primary_spread_coverage_by_year_{OUT_SUFFIX}_{timestamp}.csv"
missing_contracts_preview_csv = AUDIT_DIR / f"02G_DATA_2_PATCHED_missing_contracts_after_primary_preview_{OUT_SUFFIX}_{timestamp}.csv"

if not probe_results_df.empty:
    atomic_write_csv(probe_results_df, probe_csv)

if not new_query_df.empty:
    atomic_write_csv(new_query_df.head(1000), new_query_csv)

atomic_write_csv(cache_post_summary, cache_summary_csv)
atomic_write_csv(coverage_summary, coverage_summary_csv)
atomic_write_csv(coverage_by_tenor, coverage_by_tenor_csv)
atomic_write_csv(coverage_by_year, coverage_by_year_csv)

missing_after = primary_contract_cache.loc[
    ~(primary_contract_cache["bid_usable"] & primary_contract_cache["ask_usable"])
].copy()

atomic_write_csv(missing_after.head(1000), missing_contracts_preview_csv)

manifest_path = AUDIT_DIR / f"02G_DATA_2_PATCHED_exact_primary_thetadata_pull_{OUT_SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.DATA.2_PATCHED",
    "description": "Corrected exact primary ThetaData pull for full-history 1SD/3SD SPY call contracts",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "universe_path": str(UNIVERSE_PATH),
        "leg_request_plan_path": str(LEG_REQUEST_PLAN_PATH),
        "unique_contract_request_plan_path": str(UNIQUE_CONTRACT_REQUEST_PLAN_PATH),
        "contract_cache_overlap_path": str(CONTRACT_CACHE_OVERLAP_PATH),
    },
    "thetadata": {
        "base_url": THETADATA_BASE_URL,
        "endpoint": THETADATA_ENDPOINT,
        "symbol": SYMBOL,
        "right": RIGHT,
        "request_schema": {
            "symbol": "SPY",
            "expiration": "YYYYMMDD",
            "strike": "dollar format, e.g. 331.000",
            "right": "call",
            "start_date": "YYYYMMDD",
            "end_date": "YYYYMMDD",
            "format": "json",
        },
        "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
        "retry_failed_contracts": RETRY_FAILED_CONTRACTS,
    },
    "counts": {
        "universe_spreads": int(len(universe)),
        "unique_contracts": int(len(contract_plan)),
        "contracts_missing_pre_query": int((~query_candidates["both_sides_usable"]).sum()),
        "contracts_eligible_to_query_this_run": int(len(to_query)),
        "contracts_queried_this_run": int(len(new_query_df)),
        "contracts_priced_this_run": int(new_query_df["price_found"].sum()) if not new_query_df.empty else 0,
        "contracts_usable_post_query": int((primary_contract_cache["bid_usable"] & primary_contract_cache["ask_usable"]).sum()),
        "contracts_still_missing_post_query": int(len(missing_after)),
        "spreads_exact_primary_complete": int(spread_prices["exact_primary_complete"].sum()),
        "spreads_exact_primary_pnl_complete": int(spread_prices["exact_primary_pnl_complete"].sum()),
    },
    "outputs": {
        "primary_contract_cache_path": str(OUT_PRIMARY_CONTRACT_CACHE_PATH),
        "primary_leg_prices_path": str(OUT_PRIMARY_LEG_PRICES_PATH),
        "primary_spread_prices_path": str(OUT_PRIMARY_SPREAD_PRICES_PATH),
        "primary_spread_coverage_summary_path": str(OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "next_step": "2G.DATA.3 fallback listed-strike search for still-missing legs, if exact-primary coverage is not sufficient.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display final diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.2 PATCHED complete — exact primary ThetaData pull")
print("=" * 100)

print()
print("Primary contract cache summary")
display(cache_post_summary)

print()
print("=" * 100)
print("Exact-primary spread coverage summary")
print("=" * 100)
display(coverage_summary)

print()
print("=" * 100)
print("Exact-primary coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Exact-primary coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Missing contracts after exact-primary pull preview")
print("=" * 100)
if missing_after.empty:
    print("No missing contracts after exact-primary pull.")
else:
    display(missing_after.head(100))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Primary contract cache:          {OUT_PRIMARY_CONTRACT_CACHE_PATH}")
print(f"Primary leg prices:              {OUT_PRIMARY_LEG_PRICES_PATH}")
print(f"Primary spread prices:           {OUT_PRIMARY_SPREAD_PRICES_PATH}")
print(f"Coverage summary:                {OUT_PRIMARY_SPREAD_COVERAGE_SUMMARY_PATH}")
print(f"Manifest:                        {manifest_path}")

print()
print("Result: 2G.DATA.2 PATCHED exact primary pull complete.")
print("Next step: 2G.DATA.3 fallback listed-strike search for still-missing legs if needed.")

Cell 2G.DATA.2 — PATCHED exact primary ThetaData pull
Universe:               C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_universe_fullhist_1sd3sd_long5_cal_v1.parquet
Leg request plan:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_request_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Contract overlap:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_contract_cache_overlap_fullhist_1sd3sd_long5_cal_v1.parquet
Output primary cache:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_fullhist_1sd3sd_long5_cal_v1.parquet
ThetaData URL:          http://127.0.0.1:25503/v3/option/history/eod
Max requests this run:  None
Retry failed contracts: False

Loaded inputs
Universe spread rows:      14,613
Leg plan rows:         

C:\Users\patri\AppData\Local\Temp\ipykernel_8092\358235541.py:156: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)


,contracts,both_sides_usable,price_found,query_attempted,query_attempted_and_priced,query_attempted_not_priced,still_missing_both_sides,usable_rate
0,28523,2913,2913,0,0,0,25610,0.102128



Query plan
Contracts still missing both sides: 25,610
Eligible to query this run:         25,610
Actually querying this run:         25,610

Corrected ThetaData endpoint probe
Probe contracts: 10
Probe HTTP 200:  10
Probe priced:    10


,contract_request_id,trade_date,expiration_date,strike,price_found,bid,ask,http_status_code,parsed_rows,response_preview
0,SPY_20200102_20200117_331.0_C,2020-01-02,2020-01-17,331.0,True,0.35,0.36,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1921,1921]..."
1,SPY_20200102_20200117_332.0_C,2020-01-02,2020-01-17,332.0,True,0.23,0.24,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1913,1913]..."
2,SPY_20200102_20200117_333.0_C,2020-01-02,2020-01-17,333.0,True,0.15,0.16,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2628,2628]..."
3,SPY_20200102_20200117_340.0_C,2020-01-02,2020-01-17,340.0,True,0.01,0.02,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2395,2395]..."
4,SPY_20200102_20200117_345.0_C,2020-01-02,2020-01-17,345.0,True,0.00,0.01,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1625,1625]..."
5,SPY_20200102_20200124_334.0_C,2020-01-02,2020-01-24,334.0,True,0.21,0.22,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[489,489],""..."
6,SPY_20200102_20200124_350.0_C,2020-01-02,2020-01-24,350.0,True,0.00,0.01,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1957,1957]..."
7,SPY_20200102_20200131_335.0_C,2020-01-02,2020-01-31,335.0,True,0.31,0.33,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1469,1469]..."
8,SPY_20200102_20200131_336.0_C,2020-01-02,2020-01-31,336.0,True,0.23,0.25,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[1525,1525]..."
9,SPY_20200102_20200131_355.0_C,2020-01-02,2020-01-31,355.0,True,0.00,0.01,200,2,"{""symbol"":[""SPY"",""SPY""],""ask_size"":[2087,2087]..."


...


Exact-primary coverage by year


,trade_year,spread_rows,exact_primary_complete,exact_primary_pnl_complete,missing_or_invalid_pnl,short_leg_usable,long_leg_usable,exact_primary_complete_rate,exact_primary_pnl_complete_rate,win_rate,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,avg_width,median_width
0,2020,2277.0,1801.0,1801.0,476.0,2184.0,1811.0,0.790953,0.790953,0.907829,0.001280,0.013057,-0.523503,-0.093159,0.560683,0.53,0.015669,0.013659,38.019989,37.0
1,2021,2268.0,1985.0,1985.0,283.0,2189.0,2045.0,0.875220,0.875220,0.905793,-0.005941,0.007634,-0.538002,-0.121756,0.313426,0.29,0.009703,0.008293,35.669018,35.0
2,2022,2259.0,2008.0,2008.0,251.0,2096.0,2153.0,0.888889,0.888889,0.902888,0.007671,0.020339,-0.566478,-0.095867,1.045125,1.02,0.023446,0.022478,47.618028,47.0
3,2023,2250.0,1928.0,1928.0,322.0,1931.0,2245.0,0.856889,0.856889,0.801867,-0.012581,0.021356,-0.667509,-0.224328,0.778750,0.75,0.025631,0.023810,32.050311,32.0
4,2024,2268.0,1560.0,1556.0,712.0,1598.0,2183.0,0.687831,0.686067,0.794987,-0.006786,0.020151,-0.509074,-0.183288,0.826131,0.81,0.025515,0.024329,35.555270,34.0
5,2025,2250.0,1419.0,1415.0,835.0,1517.0,2033.0,0.630667,0.628889,0.937102,0.007163,0.015677,-0.435404,-0.048044,0.844403,0.81,0.018125,0.016333,50.460071,49.0
6,2026,1041.0,861.0,859.0,182.0,861.0,1041.0,0.827089,0.825168,0.805588,-0.025821,0.010886,-0.668007,-0.273218,0.780489,0.74,0.014749,0.013200,60.012806,58.0



Missing contracts after exact-primary pull preview


,contract_request_id,trade_date,expiration_date,strike,root,right,price_found,bid,ask,mid,...,response_preview,parsed_rows,error_message,query_attempted,cache_source,seed_cache_source,queried_at,bid_usable,ask_usable,both_sides_usable
12,SPY_20200102_20200207_360.0_C,2020-01-02,2020-02-07,360.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:10:46,False,False,False
25,SPY_20200103_20200131_360.0_C,2020-01-03,2020-01-31,360.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:10:47,False,False,False
28,SPY_20200103_20200207_360.0_C,2020-01-03,2020-02-07,360.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:10:47,False,False,False
29,SPY_20200103_20200207_365.0_C,2020-01-03,2020-02-07,365.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:10:47,False,False,False
42,SPY_20200106_20200207_360.0_C,2020-01-06,2020-02-07,360.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:10:47,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
689,SPY_20200305_20200403_395.0_C,2020-03-05,2020-04-03,395.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:11:43,False,False,False
690,SPY_20200305_20200403_400.0_C,2020-03-05,2020-04-03,400.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:11:43,False,False,False
691,SPY_20200305_20200409_337.0_C,2020-03-05,2020-04-09,337.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:11:43,False,False,False
692,SPY_20200305_20200409_338.0_C,2020-03-05,2020-04-09,338.0,SPY,call,False,NaN,NaN,NaN,...,No data found for your request,0.0,,True,thetadata_exact_primary_fullhist,NaN,2026-07-13 19:11:43,False,False,False



Saved
Primary contract cache:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_fullhist_1sd3sd_long5_cal_v1.parquet
Primary leg prices:              C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_leg_prices_joined_fullhist_1sd3sd_long5_cal_v1.parquet
Primary spread prices:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_prices_fullhist_1sd3sd_long5_cal_v1.parquet
Coverage summary:                C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_coverage_summary_fullhist_1sd3sd_long5_cal_v1.parquet
Manifest:                        C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_2_PATCHED_exact_primary_thetadata_pull_fullhist_1sd3sd_long5_cal_v1_manifest_20

In [2]:
# Cell 2G.DATA.3 — Fallback listed-strike recovery for full-history 1SD / 3SD call data
# Purpose:
#   Recover missing exact-primary SPY call legs using conservative nearby-strike fallbacks.
#
# This is a DATA LAYER cell:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule
#
# Conservative fallback direction for call spreads:
#   Missing short 1SD call:
#       move UP from target short strike, because selling a higher call collects less premium.
#       must remain below the long target strike.
#
#   Missing long 3SD call:
#       move DOWN from target long strike, because buying a lower call costs more.
#       must remain above the short target strike.
#
# Output:
#   A full-history spread P&L table with exact-primary + fallback-adjusted leg prices.

from pathlib import Path
from datetime import datetime
from io import StringIO
import json
import time

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

PRIMARY_LEG_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_leg_prices_joined_{OUT_SUFFIX}.parquet"
)

PRIMARY_SPREAD_PRICES_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_prices_{OUT_SUFFIX}.parquet"
)

FULLHIST_PRIMARY_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_{OUT_SUFFIX}.parquet"
)

OUT_FALLBACK_CANDIDATE_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_plan_{OUT_SUFFIX}.parquet"
)

OUT_FALLBACK_UNIQUE_CONTRACT_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_unique_contract_plan_{OUT_SUFFIX}.parquet"
)

OUT_FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_{OUT_SUFFIX}.parquet"
)

OUT_FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_{OUT_SUFFIX}.parquet"
)

OUT_LEG_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

OUT_SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

OUT_FALLBACK_COVERAGE_SUMMARY_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_coverage_summary_{OUT_SUFFIX}.parquet"
)

# Historical caches that may already contain some candidate fallback quotes.
CACHE_SOURCES = [
    {
        "source_name": "fullhist_fallback_cache_existing",
        "path": OUT_FALLBACK_CANDIDATE_CACHE_PATH,
        "priority": 0,
    },
    {
        "source_name": "fullhist_primary_cache",
        "path": FULLHIST_PRIMARY_CACHE_PATH,
        "priority": 1,
    },
    {
        "source_name": "repaired_fallback_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_repaired_rsi_long5_cal_v1.parquet",
        "priority": 2,
    },
    {
        "source_name": "old_long5_cal_fallback_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_fallback_candidate_contract_price_cache_long5_cal_v1.parquet",
        "priority": 3,
    },
    {
        "source_name": "repaired_primary_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_repaired_rsi_long5_cal_v1.parquet",
        "priority": 4,
    },
    {
        "source_name": "old_long5_cal_primary_cache",
        "path": PROCESSED_DIR / "call_sleeve_corsi_call_selected_contract_price_cache_long5_cal_v1.parquet",
        "priority": 5,
    },
]

# Conservative fallback search widths.
# Short leg target is rounded to $1, so search $1 increments upward.
MAX_SHORT_UP_DOLLARS = 25
SHORT_STEP_DOLLARS = 1

# Long leg target is rounded to $5, so search $5 increments downward.
# Wider value because far-OTM 3SD long calls may have no exact strike/quote.
MAX_LONG_DOWN_DOLLARS = 75
LONG_STEP_DOLLARS = 5

# Reporting flags only. These do not exclude rows.
LARGE_FALLBACK_DOLLARS_FLAG = 10
LARGE_FALLBACK_WIDTH_PCT_FLAG = 0.25

THETADATA_BASE_URL = "http://127.0.0.1:25503"
THETADATA_ENDPOINT = "/v3/option/history/eod"
THETADATA_URL = THETADATA_BASE_URL + THETADATA_ENDPOINT

SYMBOL = "SPY"
RIGHT = "call"

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 0.01

PROBE_N_CONTRACTS = 10

# None = query all missing fallback candidates.
# For a controlled partial run, set to e.g. 10000.
MAX_REQUESTS_THIS_RUN = None

SAVE_EVERY_N_ATTEMPTS = 500
PROGRESS_EVERY_N_ATTEMPTS = 500

# False = do not retry previously attempted failed fallback candidates.
# True = retry all still-missing fallback candidates.
RETRY_FAILED_FALLBACK_CONTRACTS = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.DATA.3 — Fallback listed-strike recovery for full-history 1SD / 3SD calls")
print("=" * 100)
print(f"Primary leg prices:        {PRIMARY_LEG_PRICES_PATH}")
print(f"Primary spread prices:     {PRIMARY_SPREAD_PRICES_PATH}")
print(f"Fullhist primary cache:    {FULLHIST_PRIMARY_CACHE_PATH}")
print(f"ThetaData URL:             {THETADATA_URL}")
print(f"Short fallback:            +0 to +{MAX_SHORT_UP_DOLLARS}, ${SHORT_STEP_DOLLARS} increments")
print(f"Long fallback:             0 to -{MAX_LONG_DOWN_DOLLARS}, ${LONG_STEP_DOLLARS} increments")
print(f"Max requests this run:     {MAX_REQUESTS_THIS_RUN}")
print(f"Retry failed fallback:     {RETRY_FAILED_FALLBACK_CONTRACTS}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def yyyymmdd(x):
    return pd.Timestamp(x).strftime("%Y%m%d")

def make_contract_request_id(trade_date, expiration_date, strike):
    return (
        f"SPY_"
        f"{pd.Timestamp(trade_date).strftime('%Y%m%d')}_"
        f"{pd.Timestamp(expiration_date).strftime('%Y%m%d')}_"
        f"{float(strike):.1f}_C"
    )

def extract_first_existing(row, candidates):
    for c in candidates:
        if c in row.index:
            val = row[c]
            if pd.notna(val):
                return val
    return np.nan

def json_dict_is_column_oriented(data):
    if not isinstance(data, dict) or not data:
        return False

    list_lengths = []
    for v in data.values():
        if isinstance(v, list):
            list_lengths.append(len(v))

    if not list_lengths:
        return False

    return len(set(list_lengths)) == 1

def parse_thetadata_response(resp):
    text = (resp.text or "").strip()

    if resp.status_code != 200 or not text:
        return pd.DataFrame()

    try:
        data = resp.json()

        if isinstance(data, dict):
            if "header" in data and "response" in data:
                return pd.DataFrame(data["response"], columns=data["header"])

            if "data" in data and isinstance(data["data"], list):
                return pd.DataFrame(data["data"])

            if "response" in data and isinstance(data["response"], list):
                return pd.DataFrame(data["response"])

            if json_dict_is_column_oriented(data):
                return pd.DataFrame(data)

            return pd.DataFrame([data])

        if isinstance(data, list):
            return pd.DataFrame(data)

    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        return pd.DataFrame()

def normalize_response_columns(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

def extract_quote_from_parsed_df(parsed):
    if parsed.empty:
        return {
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
        }

    parsed = normalize_response_columns(parsed)
    row = parsed.iloc[-1]

    bid = pd.to_numeric(
        extract_first_existing(row, ["bid", "bid_price", "close_bid", "eod_bid"]),
        errors="coerce",
    )

    ask = pd.to_numeric(
        extract_first_existing(row, ["ask", "ask_price", "close_ask", "eod_ask"]),
        errors="coerce",
    )

    mid = pd.to_numeric(
        extract_first_existing(row, ["mid", "mid_price", "mark", "price", "close", "eod_price"]),
        errors="coerce",
    )

    if pd.isna(mid) and pd.notna(bid) and pd.notna(ask):
        mid = (bid + ask) / 2.0

    iv = pd.to_numeric(
        extract_first_existing(row, ["iv", "implied_vol", "implied_volatility"]),
        errors="coerce",
    )

    delta = pd.to_numeric(extract_first_existing(row, ["delta"]), errors="coerce")
    gamma = pd.to_numeric(extract_first_existing(row, ["gamma"]), errors="coerce")
    theta = pd.to_numeric(extract_first_existing(row, ["theta"]), errors="coerce")
    vega = pd.to_numeric(extract_first_existing(row, ["vega"]), errors="coerce")
    rho = pd.to_numeric(extract_first_existing(row, ["rho"]), errors="coerce")

    price_found = (
        pd.notna(bid)
        and pd.notna(ask)
        and float(bid) >= 0
        and float(ask) >= 0
    )

    return {
        "price_found": bool(price_found),
        "bid": float(bid) if pd.notna(bid) else np.nan,
        "ask": float(ask) if pd.notna(ask) else np.nan,
        "mid": float(mid) if pd.notna(mid) else np.nan,
        "iv": float(iv) if pd.notna(iv) else np.nan,
        "delta": float(delta) if pd.notna(delta) else np.nan,
        "gamma": float(gamma) if pd.notna(gamma) else np.nan,
        "theta": float(theta) if pd.notna(theta) else np.nan,
        "vega": float(vega) if pd.notna(vega) else np.nan,
        "rho": float(rho) if pd.notna(rho) else np.nan,
    }

def query_contract(session, contract_row):
    trade_date = pd.Timestamp(contract_row["trade_date"]).normalize()
    expiration_date = pd.Timestamp(contract_row["expiration_date"]).normalize()
    strike = float(contract_row["strike"])

    params = {
        "symbol": SYMBOL,
        "expiration": yyyymmdd(expiration_date),
        "strike": f"{strike:.3f}",
        "right": RIGHT,
        "start_date": yyyymmdd(trade_date),
        "end_date": yyyymmdd(trade_date),
        "format": "json",
    }

    try:
        resp = session.get(
            THETADATA_URL,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        parsed = parse_thetadata_response(resp)
        quote = extract_quote_from_parsed_df(parsed)

        return {
            "contract_request_id": contract_row["contract_request_id"],
            "trade_date": trade_date,
            "expiration_date": expiration_date,
            "strike": strike,
            "root": SYMBOL,
            "right": RIGHT,
            "price_found": bool(quote["price_found"]),
            "bid": quote["bid"],
            "ask": quote["ask"],
            "mid": quote["mid"],
            "iv": quote["iv"],
            "delta": quote["delta"],
            "gamma": quote["gamma"],
            "theta": quote["theta"],
            "vega": quote["vega"],
            "rho": quote["rho"],
            "status": "priced" if quote["price_found"] else "no_price",
            "status_code": int(resp.status_code),
            "http_status_code": int(resp.status_code),
            "request_ok": True,
            "endpoint": THETADATA_ENDPOINT,
            "request_url": resp.url,
            "response_preview": (resp.text or "")[:300],
            "parsed_rows": int(len(parsed)),
            "error_message": "",
            "query_attempted": True,
            "cache_source": "thetadata_fallback_fullhist",
            "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        return {
            "contract_request_id": contract_row["contract_request_id"],
            "trade_date": trade_date,
            "expiration_date": expiration_date,
            "strike": strike,
            "root": SYMBOL,
            "right": RIGHT,
            "price_found": False,
            "bid": np.nan,
            "ask": np.nan,
            "mid": np.nan,
            "iv": np.nan,
            "delta": np.nan,
            "gamma": np.nan,
            "theta": np.nan,
            "vega": np.nan,
            "rho": np.nan,
            "status": "request_error",
            "status_code": np.nan,
            "http_status_code": np.nan,
            "request_ok": False,
            "endpoint": THETADATA_ENDPOINT,
            "request_url": "",
            "response_preview": "",
            "parsed_rows": 0,
            "error_message": repr(e),
            "query_attempted": True,
            "cache_source": "thetadata_fallback_fullhist",
            "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

def normalize_cache(df, source_name, priority):
    out = df.copy()

    if "contract_request_id" not in out.columns:
        raise ValueError(f"{source_name} missing contract_request_id.")

    out["contract_request_id"] = out["contract_request_id"].astype(str)
    out["cache_source"] = out.get("cache_source", source_name)
    out["seed_cache_source"] = source_name
    out["cache_priority"] = int(priority)

    for c in ["trade_date", "expiration_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
        else:
            out[c] = pd.NaT

    for c in ["strike", "bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan

    if "price_found" in out.columns:
        out["price_found"] = bool_series(out["price_found"], index=out.index)
    else:
        out["price_found"] = out["bid"].notna() & out["ask"].notna()

    if "query_attempted" in out.columns:
        out["query_attempted"] = bool_series(out["query_attempted"], index=out.index)
    else:
        out["query_attempted"] = False

    for c in ["status", "status_code", "http_status_code", "request_ok", "endpoint", "request_url", "response_preview", "parsed_rows", "error_message", "queried_at"]:
        if c not in out.columns:
            out[c] = np.nan

    out["bid_usable"] = out["bid"].notna() & (out["bid"] >= 0)
    out["ask_usable"] = out["ask"].notna() & (out["ask"] >= 0)
    out["both_sides_usable"] = out["bid_usable"] & out["ask_usable"]

    keep = [
        "contract_request_id",
        "trade_date",
        "expiration_date",
        "strike",
        "root",
        "right",
        "price_found",
        "bid",
        "ask",
        "mid",
        "iv",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "status",
        "status_code",
        "http_status_code",
        "request_ok",
        "endpoint",
        "request_url",
        "response_preview",
        "parsed_rows",
        "error_message",
        "query_attempted",
        "cache_source",
        "seed_cache_source",
        "cache_priority",
        "queried_at",
        "bid_usable",
        "ask_usable",
        "both_sides_usable",
    ]

    for c in keep:
        if c not in out.columns:
            out[c] = np.nan

    return out[keep].copy()

def combine_cache_rows(seed_cache, prior_cache, new_query_rows):
    frames = []

    if seed_cache is not None and len(seed_cache):
        frames.append(seed_cache.copy())

    if prior_cache is not None and len(prior_cache):
        frames.append(prior_cache.copy())

    if new_query_rows is not None and len(new_query_rows):
        frames.append(new_query_rows.copy())

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)

    for c in ["trade_date", "expiration_date"]:
        if c in combined.columns:
            combined[c] = pd.to_datetime(combined[c], errors="coerce").dt.normalize()

    for c in ["price_found", "query_attempted"]:
        combined[c] = bool_series(combined.get(c), index=combined.index)

    for c in ["strike", "bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"]:
        if c in combined.columns:
            combined[c] = pd.to_numeric(combined[c], errors="coerce")
        else:
            combined[c] = np.nan

    if "cache_priority" not in combined.columns:
        combined["cache_priority"] = 99

    combined["bid_usable"] = combined["bid"].notna() & (combined["bid"] >= 0)
    combined["ask_usable"] = combined["ask"].notna() & (combined["ask"] >= 0)
    combined["both_sides_usable"] = combined["bid_usable"] & combined["ask_usable"]

    combined["row_preference"] = np.select(
        [
            combined["both_sides_usable"] & combined["query_attempted"],
            combined["both_sides_usable"] & ~combined["query_attempted"],
            ~combined["both_sides_usable"] & combined["query_attempted"],
        ],
        [0, 1, 2],
        default=3,
    )

    combined = (
        combined
        .sort_values(["contract_request_id", "row_preference", "cache_priority"])
        .drop_duplicates("contract_request_id", keep="first")
        .drop(columns=["row_preference"])
        .reset_index(drop=True)
    )

    return combined

def candidate_strikes_for_leg(row):
    leg_label = row["leg_label"]
    target_strike = float(row["strike"])
    short_target = float(row["short_strike"])
    long_target = float(row["long_strike"])

    strikes = []

    if leg_label == "short_call_1sd":
        offsets = list(range(0, MAX_SHORT_UP_DOLLARS + 1, SHORT_STEP_DOLLARS))

        for rank, off in enumerate(offsets):
            cand = target_strike + off

            if cand < long_target:
                strikes.append((rank, float(cand), float(off)))

    elif leg_label == "long_call_3sd":
        offsets = list(range(0, MAX_LONG_DOWN_DOLLARS + 1, LONG_STEP_DOLLARS))

        for rank, off in enumerate(offsets):
            cand = target_strike - off

            if cand > short_target:
                strikes.append((rank, float(cand), float(-off)))

    else:
        raise ValueError(f"Unknown leg_label: {leg_label}")

    # De-duplicate while preserving rank order.
    seen = set()
    out = []
    for rank, strike, signed_slip in strikes:
        key = round(strike, 6)
        if key in seen:
            continue
        seen.add(key)
        out.append((rank, strike, signed_slip))

    return out

def summarize_spreads(df, label=None):
    complete = df.loc[df["fallback_pnl_complete"]].copy()

    out = {
        "label": label if label is not None else "all",
        "spread_rows": int(len(df)),
        "primary_exact_complete": int(df["exact_primary_complete"].sum()) if "exact_primary_complete" in df.columns else np.nan,
        "complete_after_fallback": int(df["complete_after_fallback"].sum()),
        "pnl_complete_after_fallback": int(df["fallback_pnl_complete"].sum()),
        "newly_recovered_after_fallback": int((df["complete_after_fallback"] & ~df["exact_primary_complete"]).sum()),
        "still_incomplete_after_fallback": int((~df["complete_after_fallback"]).sum()),
        "complete_rate_after_fallback": float(df["complete_after_fallback"].mean()) if len(df) else np.nan,
        "pnl_complete_rate_after_fallback": float(df["fallback_pnl_complete"].mean()) if len(df) else np.nan,
        "fallback_used_any": int(df["fallback_used_any"].sum()),
        "fallback_used_rate": float(df["fallback_used_any"].mean()) if len(df) else np.nan,
        "large_fallback_dollar_flag_any": int(df["large_fallback_dollar_flag_any"].sum()),
        "large_fallback_width_pct_flag_any": int(df["large_fallback_width_pct_flag_any"].sum()),
        "invalid_effective_width": int((df["effective_width_after_fallback"] <= 0).sum()),
    }

    if len(complete):
        out.update({
            "win_rate": float(complete["trade_win_after_fallback"].mean()),
            "avg_return_on_max_loss": float(complete["return_on_max_loss_after_fallback"].mean()),
            "median_return_on_max_loss": float(complete["return_on_max_loss_after_fallback"].median()),
            "worst_return_on_max_loss": float(complete["return_on_max_loss_after_fallback"].min()),
            "p05_return_on_max_loss": float(complete["return_on_max_loss_after_fallback"].quantile(0.05)),
            "avg_entry_credit": float(complete["entry_credit_after_fallback"].mean()),
            "median_entry_credit": float(complete["entry_credit_after_fallback"].median()),
            "avg_credit_pct_width": float((complete["entry_credit_after_fallback"] / complete["effective_width_after_fallback"]).mean()),
            "median_credit_pct_width": float((complete["entry_credit_after_fallback"] / complete["effective_width_after_fallback"]).median()),
            "avg_effective_width": float(complete["effective_width_after_fallback"].mean()),
            "median_effective_width": float(complete["effective_width_after_fallback"].median()),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_effective_width": np.nan,
            "median_effective_width": np.nan,
        })

    return pd.Series(out)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    PRIMARY_LEG_PRICES_PATH,
    PRIMARY_SPREAD_PRICES_PATH,
    FULLHIST_PRIMARY_CACHE_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
primary_leg_prices = pd.read_parquet(PRIMARY_LEG_PRICES_PATH).copy()
primary_spread_prices = pd.read_parquet(PRIMARY_SPREAD_PRICES_PATH).copy()

for df in [universe, primary_leg_prices, primary_spread_prices]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

primary_leg_prices["leg_price_usable"] = bool_series(primary_leg_prices["leg_price_usable"], index=primary_leg_prices.index)
primary_spread_prices["exact_primary_complete"] = bool_series(primary_spread_prices["exact_primary_complete"], index=primary_spread_prices.index)

print("=" * 100)
print("Loaded primary exact data")
print("=" * 100)
print(f"Universe rows:                   {len(universe):,}")
print(f"Primary leg rows:                {len(primary_leg_prices):,}")
print(f"Primary spread rows:             {len(primary_spread_prices):,}")
print(f"Primary exact complete spreads:  {int(primary_spread_prices['exact_primary_complete'].sum()):,}")
print(f"Missing primary legs:            {int((~primary_leg_prices['leg_price_usable']).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Build fallback candidate plan
# --------------------------------------------------------------------------------------------------

missing_legs = primary_leg_prices.loc[~primary_leg_prices["leg_price_usable"]].copy()

candidate_rows = []

for _, row in missing_legs.iterrows():
    candidates = candidate_strikes_for_leg(row)

    for candidate_rank, candidate_strike, signed_slip in candidates:
        candidate_rows.append({
            "selected_trade_id": row["selected_trade_id"],
            "trade_date": row["trade_date"],
            "trade_year": row.get("trade_year", pd.Timestamp(row["trade_date"]).year),
            "tenor": row["tenor"],
            "expiration_date": row["expiration_date"],
            "expiration_str": row.get("expiration_str", pd.Timestamp(row["expiration_date"]).strftime("%Y-%m-%d")),
            "leg_label": row["leg_label"],
            "side": row["side"],
            "needed_price_field": row["needed_price_field"],
            "target_strike": float(row["strike"]),
            "candidate_strike": float(candidate_strike),
            "candidate_rank": int(candidate_rank),
            "signed_strike_slip": float(signed_slip),
            "abs_strike_slip": float(abs(signed_slip)),
            "short_strike": float(row["short_strike"]),
            "long_strike": float(row["long_strike"]),
            "original_width": float(row["long_strike"] - row["short_strike"]),
            "contract_request_id": make_contract_request_id(
                row["trade_date"],
                row["expiration_date"],
                candidate_strike,
            ),
        })

fallback_candidate_plan = pd.DataFrame(candidate_rows)

if fallback_candidate_plan.empty:
    raise RuntimeError("No fallback candidates were generated. Check missing legs and fallback ranges.")

fallback_candidate_plan["slip_pct_original_width"] = (
    fallback_candidate_plan["abs_strike_slip"] / fallback_candidate_plan["original_width"].replace(0, np.nan)
)

fallback_candidate_plan["direction_valid"] = np.where(
    fallback_candidate_plan["leg_label"].eq("short_call_1sd"),
    (fallback_candidate_plan["candidate_strike"] >= fallback_candidate_plan["target_strike"])
    & (fallback_candidate_plan["candidate_strike"] < fallback_candidate_plan["long_strike"]),
    (fallback_candidate_plan["candidate_strike"] <= fallback_candidate_plan["target_strike"])
    & (fallback_candidate_plan["candidate_strike"] > fallback_candidate_plan["short_strike"]),
)

fallback_candidate_plan = fallback_candidate_plan.loc[fallback_candidate_plan["direction_valid"]].copy()

fallback_unique_contract_plan = (
    fallback_candidate_plan
    .groupby("contract_request_id", dropna=False)
    .agg(
        trade_date=("trade_date", "first"),
        expiration_date=("expiration_date", "first"),
        strike=("candidate_strike", "first"),
        candidate_use_count=("selected_trade_id", "size"),
        used_as_short=("leg_label", lambda x: int((x == "short_call_1sd").sum())),
        used_as_long=("leg_label", lambda x: int((x == "long_call_3sd").sum())),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
    )
    .reset_index()
    .sort_values(["trade_date", "expiration_date", "strike"])
    .reset_index(drop=True)
)

print("=" * 100)
print("Fallback candidate plan")
print("=" * 100)
print(f"Missing primary legs:              {len(missing_legs):,}")
print(f"Fallback candidate leg rows:       {len(fallback_candidate_plan):,}")
print(f"Unique fallback candidate contracts:{len(fallback_unique_contract_plan):,}")
print()

fallback_candidate_by_leg = (
    fallback_candidate_plan
    .groupby("leg_label", dropna=False)
    .agg(
        missing_legs_with_candidates=("selected_trade_id", "nunique"),
        candidate_rows=("contract_request_id", "size"),
        unique_candidate_contracts=("contract_request_id", "nunique"),
        avg_candidates_per_missing_leg=("contract_request_id", lambda x: len(x)),
        max_abs_slip=("abs_strike_slip", "max"),
    )
    .reset_index()
)

display(fallback_candidate_by_leg)

# --------------------------------------------------------------------------------------------------
# Build fallback candidate cache from existing sources
# --------------------------------------------------------------------------------------------------

cache_frames = []
cache_load_rows = []

for src in CACHE_SOURCES:
    path = Path(src["path"])
    source_name = src["source_name"]
    priority = int(src["priority"])

    if path.exists():
        raw_cache = pd.read_parquet(path).copy()
        norm = normalize_cache(raw_cache, source_name, priority)
        cache_frames.append(norm)

        cache_load_rows.append({
            "cache_source": source_name,
            "path": str(path),
            "exists": True,
            "raw_rows": int(len(raw_cache)),
            "unique_contracts": int(norm["contract_request_id"].nunique()),
            "both_sides_usable": int(norm["both_sides_usable"].sum()),
            "query_attempted": int(norm["query_attempted"].sum()),
        })
    else:
        cache_load_rows.append({
            "cache_source": source_name,
            "path": str(path),
            "exists": False,
            "raw_rows": 0,
            "unique_contracts": 0,
            "both_sides_usable": 0,
            "query_attempted": 0,
        })

cache_load_summary = pd.DataFrame(cache_load_rows)

if cache_frames:
    existing_cache_raw = pd.concat(cache_frames, ignore_index=True, sort=False)
else:
    existing_cache_raw = pd.DataFrame()

existing_cache_best = combine_cache_rows(existing_cache_raw, pd.DataFrame(), pd.DataFrame())

candidate_cache_pre = fallback_unique_contract_plan.merge(
    existing_cache_best,
    on="contract_request_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_cache"),
)

# Fill canonical fields where no cache existed.
candidate_cache_pre["trade_date"] = candidate_cache_pre["trade_date"].where(
    candidate_cache_pre["trade_date"].notna(),
    candidate_cache_pre.get("trade_date_cache"),
)

candidate_cache_pre["expiration_date"] = candidate_cache_pre["expiration_date"].where(
    candidate_cache_pre["expiration_date"].notna(),
    candidate_cache_pre.get("expiration_date_cache"),
)

candidate_cache_pre["strike"] = candidate_cache_pre["strike"].where(
    candidate_cache_pre["strike"].notna(),
    candidate_cache_pre.get("strike_cache"),
)

for c in ["bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"]:
    if c not in candidate_cache_pre.columns:
        candidate_cache_pre[c] = np.nan
    candidate_cache_pre[c] = pd.to_numeric(candidate_cache_pre[c], errors="coerce")

candidate_cache_pre["bid_usable"] = candidate_cache_pre["bid"].notna() & (candidate_cache_pre["bid"] >= 0)
candidate_cache_pre["ask_usable"] = candidate_cache_pre["ask"].notna() & (candidate_cache_pre["ask"] >= 0)
candidate_cache_pre["both_sides_usable"] = candidate_cache_pre["bid_usable"] & candidate_cache_pre["ask_usable"]

candidate_cache_pre["query_attempted"] = bool_series(candidate_cache_pre.get("query_attempted"), index=candidate_cache_pre.index)

candidate_cache_pre["bid_needed"] = candidate_cache_pre["used_as_short"] > 0
candidate_cache_pre["ask_needed"] = candidate_cache_pre["used_as_long"] > 0

candidate_cache_pre["sufficient_for_all_uses"] = (
    (~candidate_cache_pre["bid_needed"] | candidate_cache_pre["bid_usable"])
    & (~candidate_cache_pre["ask_needed"] | candidate_cache_pre["ask_usable"])
)

candidate_cache_pre["needs_fallback_query"] = ~candidate_cache_pre["sufficient_for_all_uses"]

print()
print("=" * 100)
print("Existing cache coverage for fallback candidates")
print("=" * 100)
display(cache_load_summary)

fallback_cache_pre_summary = pd.DataFrame([{
    "unique_fallback_candidate_contracts": int(len(candidate_cache_pre)),
    "sufficient_from_existing_cache": int(candidate_cache_pre["sufficient_for_all_uses"].sum()),
    "needs_fallback_query": int(candidate_cache_pre["needs_fallback_query"].sum()),
    "sufficient_rate": float(candidate_cache_pre["sufficient_for_all_uses"].mean()),
}])

display(fallback_cache_pre_summary)

# --------------------------------------------------------------------------------------------------
# Query missing fallback candidates
# --------------------------------------------------------------------------------------------------

if RETRY_FAILED_FALLBACK_CONTRACTS:
    to_query = candidate_cache_pre.loc[~candidate_cache_pre["sufficient_for_all_uses"]].copy()
else:
    to_query = candidate_cache_pre.loc[
        ~candidate_cache_pre["sufficient_for_all_uses"]
        & ~candidate_cache_pre["query_attempted"]
    ].copy()

to_query = to_query.sort_values(["trade_date", "expiration_date", "strike"]).reset_index(drop=True)

if MAX_REQUESTS_THIS_RUN is not None:
    to_query_run = to_query.head(int(MAX_REQUESTS_THIS_RUN)).copy()
else:
    to_query_run = to_query.copy()

print()
print("=" * 100)
print("Fallback query plan")
print("=" * 100)
print(f"Fallback contracts insufficient: {int((~candidate_cache_pre['sufficient_for_all_uses']).sum()):,}")
print(f"Eligible to query this run:      {len(to_query):,}")
print(f"Actually querying this run:      {len(to_query_run):,}")
print()

probe_results_df = pd.DataFrame()

if not to_query_run.empty:
    probe_contracts = to_query_run.head(PROBE_N_CONTRACTS).copy()
    probe_rows = []

    print("=" * 100)
    print("Fallback candidate ThetaData probe")
    print("=" * 100)

    with requests.Session() as session:
        for _, row in probe_contracts.iterrows():
            result = query_contract(session, row)
            probe_rows.append(result)
            time.sleep(REQUEST_SLEEP_SECONDS)

    probe_results_df = pd.DataFrame(probe_rows)

    probe_success = int(probe_results_df["price_found"].sum())
    probe_http_200 = int((probe_results_df["http_status_code"] == 200).sum())

    print(f"Probe contracts: {len(probe_results_df):,}")
    print(f"Probe HTTP 200:  {probe_http_200:,}")
    print(f"Probe priced:    {probe_success:,}")

    display(
        probe_results_df[
            [
                "contract_request_id",
                "trade_date",
                "expiration_date",
                "strike",
                "price_found",
                "bid",
                "ask",
                "http_status_code",
                "parsed_rows",
                "response_preview",
            ]
        ]
    )

new_query_rows = []

if not to_query_run.empty:
    print()
    print("=" * 100)
    print("Starting fallback candidate query loop")
    print("=" * 100)

    start_time = time.time()

    with requests.Session() as session:
        for i, (_, row) in enumerate(to_query_run.iterrows(), start=1):
            out_row = query_contract(session, row)
            new_query_rows.append(out_row)

            if (i % PROGRESS_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                elapsed = time.time() - start_time
                priced_so_far = sum(1 for r in new_query_rows if r["price_found"])
                http_200_so_far = sum(1 for r in new_query_rows if r["http_status_code"] == 200)
                not_priced_so_far = i - priced_so_far

                print(
                    f"Progress {i:,}/{len(to_query_run):,} | "
                    f"http_200={http_200_so_far:,} | "
                    f"priced={priced_so_far:,} | "
                    f"not_priced={not_priced_so_far:,} | "
                    f"elapsed={elapsed/60:.1f} min"
                )

            if (i % SAVE_EVERY_N_ATTEMPTS == 0) or (i == len(to_query_run)):
                new_query_df_tmp = pd.DataFrame(new_query_rows)
                fallback_candidate_cache_tmp = combine_cache_rows(
                    candidate_cache_pre,
                    pd.DataFrame(),
                    new_query_df_tmp,
                )
                atomic_write_parquet(fallback_candidate_cache_tmp, OUT_FALLBACK_CANDIDATE_CACHE_PATH)

            time.sleep(REQUEST_SLEEP_SECONDS)

new_query_df = pd.DataFrame(new_query_rows)

fallback_candidate_cache = combine_cache_rows(
    candidate_cache_pre,
    pd.DataFrame(),
    new_query_df,
)

atomic_write_parquet(fallback_candidate_cache, OUT_FALLBACK_CANDIDATE_CACHE_PATH)

# --------------------------------------------------------------------------------------------------
# Select nearest conservative fallback for each missing leg
# --------------------------------------------------------------------------------------------------

cache_for_selection = fallback_candidate_cache[
    [
        "contract_request_id",
        "price_found",
        "bid",
        "ask",
        "mid",
        "iv",
        "delta",
        "gamma",
        "theta",
        "vega",
        "rho",
        "bid_usable",
        "ask_usable",
        "status",
        "status_code",
        "http_status_code",
        "query_attempted",
        "cache_source",
        "seed_cache_source",
        "endpoint",
    ]
].copy()

fallback_candidates_priced = fallback_candidate_plan.merge(
    cache_for_selection,
    on="contract_request_id",
    how="left",
    validate="many_to_one",
)

fallback_candidates_priced["bid_usable"] = bool_series(fallback_candidates_priced["bid_usable"], index=fallback_candidates_priced.index)
fallback_candidates_priced["ask_usable"] = bool_series(fallback_candidates_priced["ask_usable"], index=fallback_candidates_priced.index)

fallback_candidates_priced["candidate_price_usable"] = np.where(
    fallback_candidates_priced["needed_price_field"].eq("bid"),
    fallback_candidates_priced["bid_usable"],
    fallback_candidates_priced["ask_usable"],
).astype(bool)

selected_fallback = (
    fallback_candidates_priced
    .loc[fallback_candidates_priced["candidate_price_usable"]]
    .sort_values(["selected_trade_id", "leg_label", "candidate_rank", "abs_strike_slip"])
    .drop_duplicates(["selected_trade_id", "leg_label"], keep="first")
    .reset_index(drop=True)
)

fallback_map = missing_legs[
    [
        "selected_trade_id",
        "trade_date",
        "trade_year",
        "tenor",
        "expiration_date",
        "leg_label",
        "side",
        "needed_price_field",
        "strike",
        "short_strike",
        "long_strike",
        "width",
    ]
].rename(columns={"strike": "target_strike", "width": "original_width"}).copy()

fallback_map = fallback_map.merge(
    selected_fallback[
        [
            "selected_trade_id",
            "leg_label",
            "contract_request_id",
            "candidate_strike",
            "candidate_rank",
            "signed_strike_slip",
            "abs_strike_slip",
            "slip_pct_original_width",
            "bid",
            "ask",
            "mid",
            "iv",
            "delta",
            "gamma",
            "theta",
            "vega",
            "rho",
            "status",
            "status_code",
            "http_status_code",
            "query_attempted",
            "cache_source",
            "seed_cache_source",
        ]
    ],
    on=["selected_trade_id", "leg_label"],
    how="left",
    validate="one_to_one",
)

fallback_map["fallback_found"] = fallback_map["candidate_strike"].notna()

fallback_map["fallback_direction_valid"] = np.where(
    fallback_map["leg_label"].eq("short_call_1sd"),
    (fallback_map["candidate_strike"] >= fallback_map["target_strike"])
    & (fallback_map["candidate_strike"] < fallback_map["long_strike"]),
    (fallback_map["candidate_strike"] <= fallback_map["target_strike"])
    & (fallback_map["candidate_strike"] > fallback_map["short_strike"]),
)

fallback_map.loc[~fallback_map["fallback_found"], "fallback_direction_valid"] = False

fallback_map["large_fallback_dollar_flag"] = fallback_map["abs_strike_slip"] > LARGE_FALLBACK_DOLLARS_FLAG
fallback_map["large_fallback_width_pct_flag"] = fallback_map["slip_pct_original_width"] > LARGE_FALLBACK_WIDTH_PCT_FLAG

atomic_write_parquet(fallback_candidate_plan, OUT_FALLBACK_CANDIDATE_PLAN_PATH)
atomic_write_parquet(fallback_unique_contract_plan, OUT_FALLBACK_UNIQUE_CONTRACT_PLAN_PATH)
atomic_write_parquet(fallback_map, OUT_FALLBACK_MAP_PATH)

# --------------------------------------------------------------------------------------------------
# Build final leg prices with exact primary + fallback
# --------------------------------------------------------------------------------------------------

final_leg_prices = primary_leg_prices.copy()

final_leg_prices["primary_leg_price_usable"] = bool_series(final_leg_prices["leg_price_usable"], index=final_leg_prices.index)
final_leg_prices["target_strike"] = pd.to_numeric(final_leg_prices["strike"], errors="coerce")
final_leg_prices["effective_strike"] = final_leg_prices["target_strike"]
final_leg_prices["final_bid"] = pd.to_numeric(final_leg_prices["bid"], errors="coerce")
final_leg_prices["final_ask"] = pd.to_numeric(final_leg_prices["ask"], errors="coerce")
final_leg_prices["final_mid"] = pd.to_numeric(final_leg_prices["mid"], errors="coerce")
final_leg_prices["final_iv"] = pd.to_numeric(final_leg_prices.get("iv"), errors="coerce")
final_leg_prices["final_delta"] = pd.to_numeric(final_leg_prices.get("delta"), errors="coerce")
final_leg_prices["final_gamma"] = pd.to_numeric(final_leg_prices.get("gamma"), errors="coerce")
final_leg_prices["final_theta"] = pd.to_numeric(final_leg_prices.get("theta"), errors="coerce")
final_leg_prices["final_vega"] = pd.to_numeric(final_leg_prices.get("vega"), errors="coerce")
final_leg_prices["fallback_used"] = False
final_leg_prices["fallback_found"] = False
final_leg_prices["fallback_contract_request_id"] = np.nan
final_leg_prices["fallback_candidate_rank"] = np.nan
final_leg_prices["signed_strike_slip"] = 0.0
final_leg_prices["abs_strike_slip"] = 0.0
final_leg_prices["slip_pct_original_width"] = 0.0
final_leg_prices["large_fallback_dollar_flag"] = False
final_leg_prices["large_fallback_width_pct_flag"] = False
final_leg_prices["final_price_source"] = np.where(
    final_leg_prices["primary_leg_price_usable"],
    "exact_primary",
    "missing",
)

fallback_for_join = fallback_map.loc[fallback_map["fallback_found"]].copy()

fallback_for_join = fallback_for_join.rename(
    columns={
        "contract_request_id": "fallback_contract_request_id",
        "candidate_strike": "fallback_effective_strike",
        "candidate_rank": "fallback_candidate_rank",
        "bid": "fallback_bid",
        "ask": "fallback_ask",
        "mid": "fallback_mid",
        "iv": "fallback_iv",
        "delta": "fallback_delta",
        "gamma": "fallback_gamma",
        "theta": "fallback_theta",
        "vega": "fallback_vega",
    }
)

final_leg_prices = final_leg_prices.merge(
    fallback_for_join[
        [
            "selected_trade_id",
            "leg_label",
            "fallback_contract_request_id",
            "fallback_effective_strike",
            "fallback_candidate_rank",
            "signed_strike_slip",
            "abs_strike_slip",
            "slip_pct_original_width",
            "fallback_bid",
            "fallback_ask",
            "fallback_mid",
            "fallback_iv",
            "fallback_delta",
            "fallback_gamma",
            "fallback_theta",
            "fallback_vega",
            "fallback_direction_valid",
            "large_fallback_dollar_flag",
            "large_fallback_width_pct_flag",
            "cache_source",
            "seed_cache_source",
        ]
    ],
    on=["selected_trade_id", "leg_label"],
    how="left",
    suffixes=("", "_fallback"),
    validate="many_to_one",
)

use_fallback = (
    ~final_leg_prices["primary_leg_price_usable"]
    & final_leg_prices["fallback_effective_strike"].notna()
    & bool_series(final_leg_prices["fallback_direction_valid"], index=final_leg_prices.index)
)

final_leg_prices.loc[use_fallback, "fallback_used"] = True
final_leg_prices.loc[use_fallback, "fallback_found"] = True
final_leg_prices.loc[use_fallback, "effective_strike"] = final_leg_prices.loc[use_fallback, "fallback_effective_strike"]
final_leg_prices.loc[use_fallback, "final_bid"] = final_leg_prices.loc[use_fallback, "fallback_bid"]
final_leg_prices.loc[use_fallback, "final_ask"] = final_leg_prices.loc[use_fallback, "fallback_ask"]
final_leg_prices.loc[use_fallback, "final_mid"] = final_leg_prices.loc[use_fallback, "fallback_mid"]
final_leg_prices.loc[use_fallback, "final_iv"] = final_leg_prices.loc[use_fallback, "fallback_iv"]
final_leg_prices.loc[use_fallback, "final_delta"] = final_leg_prices.loc[use_fallback, "fallback_delta"]
final_leg_prices.loc[use_fallback, "final_gamma"] = final_leg_prices.loc[use_fallback, "fallback_gamma"]
final_leg_prices.loc[use_fallback, "final_theta"] = final_leg_prices.loc[use_fallback, "fallback_theta"]
final_leg_prices.loc[use_fallback, "final_vega"] = final_leg_prices.loc[use_fallback, "fallback_vega"]
final_leg_prices.loc[use_fallback, "final_price_source"] = "fallback_conservative"

# Ensure exact-primary rows do not inherit fallback slippage columns.
for col in ["signed_strike_slip", "abs_strike_slip", "slip_pct_original_width"]:
    fallback_col = f"{col}_fallback"
    if fallback_col in final_leg_prices.columns:
        final_leg_prices[col] = np.where(
            final_leg_prices["fallback_used"],
            final_leg_prices[fallback_col],
            final_leg_prices[col],
        )

for col in ["large_fallback_dollar_flag", "large_fallback_width_pct_flag"]:
    fallback_col = f"{col}_fallback"
    if fallback_col in final_leg_prices.columns:
        final_leg_prices[col] = np.where(
            final_leg_prices["fallback_used"],
            final_leg_prices[fallback_col],
            final_leg_prices[col],
        )

final_leg_prices["final_bid_usable"] = final_leg_prices["final_bid"].notna() & (final_leg_prices["final_bid"] >= 0)
final_leg_prices["final_ask_usable"] = final_leg_prices["final_ask"].notna() & (final_leg_prices["final_ask"] >= 0)

final_leg_prices["final_leg_price_usable"] = np.where(
    final_leg_prices["needed_price_field"].eq("bid"),
    final_leg_prices["final_bid_usable"],
    final_leg_prices["final_ask_usable"],
).astype(bool)

# --------------------------------------------------------------------------------------------------
# Build final spread prices with fallback
# --------------------------------------------------------------------------------------------------

spread_base_cols = [
    "selected_trade_id",
    "trade_date",
    "trade_year",
    "tenor",
    "expiration_date",
    "expiration_str",
    "outcome_available",
    "short_strike",
    "long_strike",
    "spy_close",
    "expiry_spy_close",
    "vix_style_vol_pct",
    "sd_move",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "width",
]

spread_base = universe[spread_base_cols].drop_duplicates("selected_trade_id").copy()

exact_status = primary_spread_prices[
    [
        "selected_trade_id",
        "exact_primary_complete",
        "exact_primary_pnl_complete",
    ]
].copy()

spread_base = spread_base.merge(
    exact_status,
    on="selected_trade_id",
    how="left",
    validate="one_to_one",
)

spread_base["exact_primary_complete"] = bool_series(spread_base["exact_primary_complete"], index=spread_base.index)
spread_base["exact_primary_pnl_complete"] = bool_series(spread_base["exact_primary_pnl_complete"], index=spread_base.index)

short_final = (
    final_leg_prices
    .loc[final_leg_prices["leg_label"].eq("short_call_1sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "fallback_contract_request_id",
            "target_strike",
            "effective_strike",
            "final_bid",
            "final_ask",
            "final_mid",
            "final_leg_price_usable",
            "fallback_used",
            "fallback_found",
            "signed_strike_slip",
            "abs_strike_slip",
            "slip_pct_original_width",
            "large_fallback_dollar_flag",
            "large_fallback_width_pct_flag",
            "final_price_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "short_exact_contract_request_id",
            "fallback_contract_request_id": "short_fallback_contract_request_id",
            "target_strike": "short_target_strike",
            "effective_strike": "short_effective_strike",
            "final_bid": "short_bid_final",
            "final_ask": "short_ask_final",
            "final_mid": "short_mid_final",
            "final_leg_price_usable": "short_final_leg_price_usable",
            "fallback_used": "short_fallback_used",
            "fallback_found": "short_fallback_found",
            "signed_strike_slip": "short_signed_strike_slip",
            "abs_strike_slip": "short_abs_strike_slip",
            "slip_pct_original_width": "short_slip_pct_original_width",
            "large_fallback_dollar_flag": "short_large_fallback_dollar_flag",
            "large_fallback_width_pct_flag": "short_large_fallback_width_pct_flag",
            "final_price_source": "short_final_price_source",
        }
    )
)

long_final = (
    final_leg_prices
    .loc[final_leg_prices["leg_label"].eq("long_call_3sd")]
    [
        [
            "selected_trade_id",
            "contract_request_id",
            "fallback_contract_request_id",
            "target_strike",
            "effective_strike",
            "final_bid",
            "final_ask",
            "final_mid",
            "final_leg_price_usable",
            "fallback_used",
            "fallback_found",
            "signed_strike_slip",
            "abs_strike_slip",
            "slip_pct_original_width",
            "large_fallback_dollar_flag",
            "large_fallback_width_pct_flag",
            "final_price_source",
        ]
    ]
    .rename(
        columns={
            "contract_request_id": "long_exact_contract_request_id",
            "fallback_contract_request_id": "long_fallback_contract_request_id",
            "target_strike": "long_target_strike",
            "effective_strike": "long_effective_strike",
            "final_bid": "long_bid_final",
            "final_ask": "long_ask_final",
            "final_mid": "long_mid_final",
            "final_leg_price_usable": "long_final_leg_price_usable",
            "fallback_used": "long_fallback_used",
            "fallback_found": "long_fallback_found",
            "signed_strike_slip": "long_signed_strike_slip",
            "abs_strike_slip": "long_abs_strike_slip",
            "slip_pct_original_width": "long_slip_pct_original_width",
            "large_fallback_dollar_flag": "long_large_fallback_dollar_flag",
            "large_fallback_width_pct_flag": "long_large_fallback_width_pct_flag",
            "final_price_source": "long_final_price_source",
        }
    )
)

spread_with_fallback = (
    spread_base
    .merge(short_final, on="selected_trade_id", how="left", validate="one_to_one")
    .merge(long_final, on="selected_trade_id", how="left", validate="one_to_one")
)

for c in [
    "short_final_leg_price_usable",
    "long_final_leg_price_usable",
    "short_fallback_used",
    "long_fallback_used",
    "short_large_fallback_dollar_flag",
    "long_large_fallback_dollar_flag",
    "short_large_fallback_width_pct_flag",
    "long_large_fallback_width_pct_flag",
]:
    spread_with_fallback[c] = bool_series(spread_with_fallback[c], index=spread_with_fallback.index)

spread_with_fallback["complete_after_fallback"] = (
    spread_with_fallback["short_final_leg_price_usable"]
    & spread_with_fallback["long_final_leg_price_usable"]
)

spread_with_fallback["fallback_used_any"] = (
    spread_with_fallback["short_fallback_used"]
    | spread_with_fallback["long_fallback_used"]
)

spread_with_fallback["large_fallback_dollar_flag_any"] = (
    spread_with_fallback["short_large_fallback_dollar_flag"]
    | spread_with_fallback["long_large_fallback_dollar_flag"]
)

spread_with_fallback["large_fallback_width_pct_flag_any"] = (
    spread_with_fallback["short_large_fallback_width_pct_flag"]
    | spread_with_fallback["long_large_fallback_width_pct_flag"]
)

spread_with_fallback["effective_width_after_fallback"] = (
    spread_with_fallback["long_effective_strike"]
    - spread_with_fallback["short_effective_strike"]
)

spread_with_fallback["entry_credit_after_fallback"] = (
    spread_with_fallback["short_bid_final"]
    - spread_with_fallback["long_ask_final"]
)

spread_with_fallback["max_loss_after_fallback"] = (
    spread_with_fallback["effective_width_after_fallback"]
    - spread_with_fallback["entry_credit_after_fallback"]
)

spread_with_fallback["terminal_spread_intrinsic_after_fallback"] = np.maximum(
    spread_with_fallback["expiry_spy_close"] - spread_with_fallback["short_effective_strike"],
    0.0,
) - np.maximum(
    spread_with_fallback["expiry_spy_close"] - spread_with_fallback["long_effective_strike"],
    0.0,
)

spread_with_fallback.loc[
    ~spread_with_fallback["outcome_available"],
    "terminal_spread_intrinsic_after_fallback",
] = np.nan

spread_with_fallback["expiry_pnl_after_fallback"] = (
    spread_with_fallback["entry_credit_after_fallback"]
    - spread_with_fallback["terminal_spread_intrinsic_after_fallback"]
)

spread_with_fallback["return_on_max_loss_after_fallback"] = (
    spread_with_fallback["expiry_pnl_after_fallback"]
    / spread_with_fallback["max_loss_after_fallback"]
)

spread_with_fallback["trade_win_after_fallback"] = spread_with_fallback["expiry_pnl_after_fallback"] > 0

spread_with_fallback["fallback_pnl_complete"] = (
    spread_with_fallback["complete_after_fallback"]
    & spread_with_fallback["outcome_available"]
    & spread_with_fallback["entry_credit_after_fallback"].notna()
    & (spread_with_fallback["entry_credit_after_fallback"] > 0)
    & spread_with_fallback["max_loss_after_fallback"].notna()
    & (spread_with_fallback["max_loss_after_fallback"] > 0)
    & spread_with_fallback["effective_width_after_fallback"].notna()
    & (spread_with_fallback["effective_width_after_fallback"] > 0)
    & spread_with_fallback["terminal_spread_intrinsic_after_fallback"].notna()
    & spread_with_fallback["return_on_max_loss_after_fallback"].notna()
)

# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

fallback_coverage_summary = pd.DataFrame([summarize_spreads(spread_with_fallback)])

coverage_by_tenor = pd.DataFrame([
    summarize_spreads(g, label=tenor)
    for tenor, g in spread_with_fallback.groupby("tenor", dropna=False)
]).rename(columns={"label": "tenor"})

coverage_by_year = pd.DataFrame([
    summarize_spreads(g, label=year)
    for year, g in spread_with_fallback.groupby("trade_year", dropna=False)
]).rename(columns={"label": "trade_year"})

fallback_leg_summary = (
    final_leg_prices
    .groupby("leg_label", dropna=False)
    .agg(
        leg_rows=("selected_trade_id", "size"),
        primary_usable=("primary_leg_price_usable", "sum"),
        fallback_used=("fallback_used", "sum"),
        final_usable=("final_leg_price_usable", "sum"),
        still_missing=("final_leg_price_usable", lambda x: int((~x).sum())),
        avg_abs_slip_used=("abs_strike_slip", lambda x: float(x[final_leg_prices.loc[x.index, "fallback_used"]].mean()) if final_leg_prices.loc[x.index, "fallback_used"].any() else np.nan),
        median_abs_slip_used=("abs_strike_slip", lambda x: float(x[final_leg_prices.loc[x.index, "fallback_used"]].median()) if final_leg_prices.loc[x.index, "fallback_used"].any() else np.nan),
        max_abs_slip_used=("abs_strike_slip", lambda x: float(x[final_leg_prices.loc[x.index, "fallback_used"]].max()) if final_leg_prices.loc[x.index, "fallback_used"].any() else np.nan),
        large_dollar_flags=("large_fallback_dollar_flag", "sum"),
        large_width_pct_flags=("large_fallback_width_pct_flag", "sum"),
    )
    .reset_index()
)

fallback_map_summary = (
    fallback_map
    .groupby("leg_label", dropna=False)
    .agg(
        missing_primary_legs=("selected_trade_id", "size"),
        fallback_found=("fallback_found", "sum"),
        fallback_not_found=("fallback_found", lambda x: int((~x).sum())),
        direction_valid=("fallback_direction_valid", "sum"),
        avg_abs_slip=("abs_strike_slip", "mean"),
        median_abs_slip=("abs_strike_slip", "median"),
        max_abs_slip=("abs_strike_slip", "max"),
        avg_slip_pct_width=("slip_pct_original_width", "mean"),
        large_dollar_flags=("large_fallback_dollar_flag", "sum"),
        large_width_pct_flags=("large_fallback_width_pct_flag", "sum"),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

atomic_write_parquet(final_leg_prices, OUT_LEG_PRICES_WITH_FALLBACK_PATH)
atomic_write_parquet(spread_with_fallback, OUT_SPREAD_PRICES_WITH_FALLBACK_PATH)
atomic_write_parquet(fallback_coverage_summary, OUT_FALLBACK_COVERAGE_SUMMARY_PATH)

audit_paths = {
    "fallback_candidate_by_leg": AUDIT_DIR / f"02G_DATA_3_fallback_candidate_by_leg_{OUT_SUFFIX}_{timestamp}.csv",
    "cache_load_summary": AUDIT_DIR / f"02G_DATA_3_cache_load_summary_{OUT_SUFFIX}_{timestamp}.csv",
    "fallback_cache_pre_summary": AUDIT_DIR / f"02G_DATA_3_fallback_cache_pre_summary_{OUT_SUFFIX}_{timestamp}.csv",
    "probe_results": AUDIT_DIR / f"02G_DATA_3_probe_results_{OUT_SUFFIX}_{timestamp}.csv",
    "new_query_preview": AUDIT_DIR / f"02G_DATA_3_new_query_results_preview_{OUT_SUFFIX}_{timestamp}.csv",
    "fallback_map_summary": AUDIT_DIR / f"02G_DATA_3_fallback_map_summary_{OUT_SUFFIX}_{timestamp}.csv",
    "fallback_leg_summary": AUDIT_DIR / f"02G_DATA_3_fallback_leg_summary_{OUT_SUFFIX}_{timestamp}.csv",
    "coverage_summary": AUDIT_DIR / f"02G_DATA_3_coverage_summary_{OUT_SUFFIX}_{timestamp}.csv",
    "coverage_by_tenor": AUDIT_DIR / f"02G_DATA_3_coverage_by_tenor_{OUT_SUFFIX}_{timestamp}.csv",
    "coverage_by_year": AUDIT_DIR / f"02G_DATA_3_coverage_by_year_{OUT_SUFFIX}_{timestamp}.csv",
    "large_fallback_preview": AUDIT_DIR / f"02G_DATA_3_large_fallback_preview_{OUT_SUFFIX}_{timestamp}.csv",
    "still_incomplete_preview": AUDIT_DIR / f"02G_DATA_3_still_incomplete_preview_{OUT_SUFFIX}_{timestamp}.csv",
}

atomic_write_csv(fallback_candidate_by_leg, audit_paths["fallback_candidate_by_leg"])
atomic_write_csv(cache_load_summary, audit_paths["cache_load_summary"])
atomic_write_csv(fallback_cache_pre_summary, audit_paths["fallback_cache_pre_summary"])

if not probe_results_df.empty:
    atomic_write_csv(probe_results_df, audit_paths["probe_results"])

if not new_query_df.empty:
    atomic_write_csv(new_query_df.head(1000), audit_paths["new_query_preview"])

atomic_write_csv(fallback_map_summary, audit_paths["fallback_map_summary"])
atomic_write_csv(fallback_leg_summary, audit_paths["fallback_leg_summary"])
atomic_write_csv(fallback_coverage_summary, audit_paths["coverage_summary"])
atomic_write_csv(coverage_by_tenor, audit_paths["coverage_by_tenor"])
atomic_write_csv(coverage_by_year, audit_paths["coverage_by_year"])

large_fallback_preview = spread_with_fallback.loc[
    spread_with_fallback["large_fallback_dollar_flag_any"]
    | spread_with_fallback["large_fallback_width_pct_flag_any"]
].copy()

still_incomplete_preview = spread_with_fallback.loc[
    ~spread_with_fallback["complete_after_fallback"]
].copy()

atomic_write_csv(large_fallback_preview.head(1000), audit_paths["large_fallback_preview"])
atomic_write_csv(still_incomplete_preview.head(1000), audit_paths["still_incomplete_preview"])

manifest_path = AUDIT_DIR / f"02G_DATA_3_fallback_recovery_{OUT_SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.DATA.3",
    "description": "Conservative fallback listed-strike recovery for full-history 1SD/3SD SPY call data",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "fallback_rules": {
        "short_call_1sd": {
            "direction": "at_or_above_target",
            "reason": "selling higher strike is conservative because premium received is lower",
            "max_up_dollars": MAX_SHORT_UP_DOLLARS,
            "step_dollars": SHORT_STEP_DOLLARS,
            "constraint": "candidate_strike < original long target strike",
        },
        "long_call_3sd": {
            "direction": "at_or_below_target",
            "reason": "buying lower strike is conservative because premium paid is higher",
            "max_down_dollars": MAX_LONG_DOWN_DOLLARS,
            "step_dollars": LONG_STEP_DOLLARS,
            "constraint": "candidate_strike > original short target strike",
        },
        "large_fallback_flags": {
            "dollar_flag_gt": LARGE_FALLBACK_DOLLARS_FLAG,
            "width_pct_flag_gt": LARGE_FALLBACK_WIDTH_PCT_FLAG,
            "flags_are_report_only": True,
        },
    },
    "thetadata": {
        "base_url": THETADATA_BASE_URL,
        "endpoint": THETADATA_ENDPOINT,
        "symbol": SYMBOL,
        "right": RIGHT,
        "max_requests_this_run": MAX_REQUESTS_THIS_RUN,
        "retry_failed_fallback_contracts": RETRY_FAILED_FALLBACK_CONTRACTS,
    },
    "counts": {
        "primary_spread_rows": int(len(primary_spread_prices)),
        "primary_exact_complete_spreads": int(primary_spread_prices["exact_primary_complete"].sum()),
        "missing_primary_legs": int(len(missing_legs)),
        "fallback_candidate_rows": int(len(fallback_candidate_plan)),
        "unique_fallback_candidate_contracts": int(len(fallback_unique_contract_plan)),
        "fallback_contracts_queried_this_run": int(len(new_query_df)),
        "fallback_contracts_priced_this_run": int(new_query_df["price_found"].sum()) if not new_query_df.empty else 0,
        "fallback_map_found": int(fallback_map["fallback_found"].sum()),
        "complete_after_fallback": int(spread_with_fallback["complete_after_fallback"].sum()),
        "pnl_complete_after_fallback": int(spread_with_fallback["fallback_pnl_complete"].sum()),
        "newly_recovered_after_fallback": int((spread_with_fallback["complete_after_fallback"] & ~spread_with_fallback["exact_primary_complete"]).sum()),
        "still_incomplete_after_fallback": int((~spread_with_fallback["complete_after_fallback"]).sum()),
        "large_fallback_dollar_flag_any": int(spread_with_fallback["large_fallback_dollar_flag_any"].sum()),
        "large_fallback_width_pct_flag_any": int(spread_with_fallback["large_fallback_width_pct_flag_any"].sum()),
    },
    "outputs": {
        "fallback_candidate_plan_path": str(OUT_FALLBACK_CANDIDATE_PLAN_PATH),
        "fallback_unique_contract_plan_path": str(OUT_FALLBACK_UNIQUE_CONTRACT_PLAN_PATH),
        "fallback_candidate_cache_path": str(OUT_FALLBACK_CANDIDATE_CACHE_PATH),
        "fallback_map_path": str(OUT_FALLBACK_MAP_PATH),
        "leg_prices_with_fallback_path": str(OUT_LEG_PRICES_WITH_FALLBACK_PATH),
        "spread_prices_with_fallback_path": str(OUT_SPREAD_PRICES_WITH_FALLBACK_PATH),
        "fallback_coverage_summary_path": str(OUT_FALLBACK_COVERAGE_SUMMARY_PATH),
        "manifest_path": str(manifest_path),
    },
    "next_step": "2G.DATA.4 mechanical sanity checks and then signal-filter diagnostics can be run from the fallback-adjusted full-history panel.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display final diagnostics
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.3 complete — fallback listed-strike recovery")
print("=" * 100)

print()
print("Fallback candidate by leg")
display(fallback_candidate_by_leg)

print()
print("=" * 100)
print("Fallback map summary")
print("=" * 100)
display(fallback_map_summary)

print()
print("=" * 100)
print("Fallback leg summary")
print("=" * 100)
display(fallback_leg_summary)

print()
print("=" * 100)
print("Coverage summary after fallback")
print("=" * 100)
display(fallback_coverage_summary)

print()
print("=" * 100)
print("Coverage by tenor after fallback")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Coverage by year after fallback")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Large fallback preview")
print("=" * 100)
if large_fallback_preview.empty:
    print("No large fallback flags.")
else:
    display(large_fallback_preview.head(100))

print()
print("=" * 100)
print("Still incomplete after fallback preview")
print("=" * 100)
if still_incomplete_preview.empty:
    print("No incomplete spreads after fallback.")
else:
    display(still_incomplete_preview.head(100))

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Fallback candidate plan:       {OUT_FALLBACK_CANDIDATE_PLAN_PATH}")
print(f"Fallback unique contract plan: {OUT_FALLBACK_UNIQUE_CONTRACT_PLAN_PATH}")
print(f"Fallback candidate cache:      {OUT_FALLBACK_CANDIDATE_CACHE_PATH}")
print(f"Fallback map:                  {OUT_FALLBACK_MAP_PATH}")
print(f"Leg prices with fallback:      {OUT_LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Spread prices with fallback:   {OUT_SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Coverage summary:              {OUT_FALLBACK_COVERAGE_SUMMARY_PATH}")
print(f"Manifest:                      {manifest_path}")

print()
print("Result: 2G.DATA.3 fallback recovery complete.")
print("Next step: 2G.DATA.4 mechanical sanity checks, then run signal diagnostics from the full fallback-adjusted panel.")

Cell 2G.DATA.3 — Fallback listed-strike recovery for full-history 1SD / 3SD calls
Primary leg prices:        C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_leg_prices_joined_fullhist_1sd3sd_long5_cal_v1.parquet
Primary spread prices:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_spread_prices_fullhist_1sd3sd_long5_cal_v1.parquet
Fullhist primary cache:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_fullhist_1sd3sd_long5_cal_v1.parquet
ThetaData URL:             http://127.0.0.1:25503/v3/option/history/eod
Short fallback:            +0 to +25, $1 increments
Long fallback:             0 to -75, $5 increments
Max requests this run:     None
Retry failed fallback:     False

Loaded primary exact data
Universe rows:                   14,613
Primary leg rows:       

,leg_label,missing_legs_with_candidates,candidate_rows,unique_candidate_contracts,avg_candidates_per_missing_leg,max_abs_slip
0,long_call_3sd,1102,13213,9982,13213,75.0
1,short_call_1sd,2237,58045,42125,58045,25.0



Existing cache coverage for fallback candidates


C:\Users\patri\AppData\Local\Temp\ipykernel_8092\1148186283.py:218: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False)


,cache_source,path,exists,raw_rows,unique_contracts,both_sides_usable,query_attempted
0,fullhist_fallback_cache_existing,C:\Users\patri\vrp_project\data\processed\call...,False,0,0,0,0
1,fullhist_primary_cache,C:\Users\patri\vrp_project\data\processed\call...,True,28523,28523,25221,25610
2,repaired_fallback_cache,C:\Users\patri\vrp_project\data\processed\call...,True,5745,5745,1656,0
3,old_long5_cal_fallback_cache,C:\Users\patri\vrp_project\data\processed\call...,True,4678,4678,1587,0
4,repaired_primary_cache,C:\Users\patri\vrp_project\data\processed\call...,True,3423,3423,2791,0
5,old_long5_cal_primary_cache,C:\Users\patri\vrp_project\data\processed\call...,True,3437,3437,2825,0


...


Coverage by year after fallback


,trade_year,spread_rows,primary_exact_complete,complete_after_fallback,pnl_complete_after_fallback,newly_recovered_after_fallback,still_incomplete_after_fallback,complete_rate_after_fallback,pnl_complete_rate_after_fallback,fallback_used_any,...,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,avg_effective_width,median_effective_width
0,2020.0,2277.0,1801.0,2277.0,2277.0,476.0,0.0,1.0,1.000000,476.0,...,0.002984,0.012793,-0.523503,-0.074938,0.549671,0.52,0.015051,0.013250,38.583663,38.0
1,2021.0,2268.0,1985.0,2268.0,2266.0,283.0,0.0,1.0,0.999118,283.0,...,-0.005750,0.007446,-0.538002,-0.108538,0.309051,0.29,0.009370,0.008020,36.412621,36.0
2,2022.0,2259.0,2008.0,2259.0,2259.0,251.0,0.0,1.0,1.000000,251.0,...,0.007105,0.020160,-0.566478,-0.099851,1.039380,1.00,0.023192,0.022319,47.888446,47.0
3,2023.0,2250.0,1928.0,2250.0,2250.0,322.0,0.0,1.0,1.000000,322.0,...,-0.010733,0.020942,-0.667509,-0.214254,0.766502,0.74,0.024903,0.023333,32.416000,32.0
4,2024.0,2268.0,1560.0,2268.0,2262.0,708.0,0.0,1.0,0.997354,708.0,...,-0.003102,0.017442,-0.509074,-0.155836,0.760986,0.74,0.022500,0.021183,37.480106,35.0
5,2025.0,2250.0,1419.0,2250.0,2245.0,831.0,0.0,1.0,0.997778,831.0,...,0.009239,0.014617,-0.435404,0.001269,0.788633,0.76,0.016706,0.015091,50.613808,50.0
6,2026.0,1041.0,861.0,1041.0,1039.0,180.0,0.0,1.0,0.998079,180.0,...,-0.026184,0.010526,-0.668007,-0.282451,0.773850,0.74,0.014358,0.012769,60.963426,60.0



Large fallback preview


,selected_trade_id,trade_date,trade_year,tenor,expiration_date,expiration_str,outcome_available,short_strike,long_strike,spy_close,...,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,effective_width_after_fallback,entry_credit_after_fallback,max_loss_after_fallback,terminal_spread_intrinsic_after_fallback,expiry_pnl_after_fallback,return_on_max_loss_after_fallback,trade_win_after_fallback,fallback_pnl_complete
17,SPY_CALL_20200103_T33_EXP20200207_S337.0_L365.0,2020-01-03,2020,33,2020-02-07,2020-02-07,True,337.0,365.0,322.5189,...,False,True,18.0,0.14,17.86,0.0,0.14,0.007839,True,True
26,SPY_CALL_20200106_T33_EXP20200214_S338.0_L365.0,2020-01-06,2020,33,2020-02-14,2020-02-14,True,338.0,365.0,323.6400,...,False,True,17.0,0.22,16.78,0.0,0.22,0.013111,True,True
35,SPY_CALL_20200107_T33_EXP20200214_S337.0_L365.0,2020-01-07,2020,33,2020-02-14,2020-02-14,True,337.0,365.0,322.7300,...,False,True,18.0,0.20,17.80,0.6,-0.40,-0.022472,False,True
43,SPY_CALL_20200108_T30_EXP20200207_S338.0_L365.0,2020-01-08,2020,30,2020-02-07,2020-02-07,True,338.0,365.0,324.4500,...,False,True,17.0,0.15,16.85,0.0,0.15,0.008902,True,True
44,SPY_CALL_20200108_T33_EXP20200214_S338.0_L365.0,2020-01-08,2020,33,2020-02-14,2020-02-14,True,338.0,365.0,324.4500,...,False,True,17.0,0.30,16.70,0.0,0.30,0.017964,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
520,SPY_CALL_20200325_T30_EXP20200424_S293.0_L385.0,2020-03-25,2020,30,2020-04-24,2020-04-24,True,293.0,385.0,246.7900,...,True,True,52.0,1.33,50.67,0.0,1.33,0.026248,True,True
521,SPY_CALL_20200325_T33_EXP20200501_S295.0_L390.0,2020-03-25,2020,33,2020-05-01,2020-05-01,True,295.0,390.0,246.7900,...,True,True,55.0,1.51,53.49,0.0,1.51,0.028230,True,True
527,SPY_CALL_20200326_T24_EXP20200424_S304.0_L385.0,2020-03-26,2020,24,2020-04-24,2020-04-24,True,304.0,385.0,262.0841,...,True,True,41.0,0.93,40.07,0.0,0.93,0.023209,True,True
528,SPY_CALL_20200326_T27_EXP20200424_S306.0_L395.0,2020-03-26,2020,27,2020-04-24,2020-04-24,True,306.0,395.0,262.0841,...,True,True,39.0,0.77,38.23,0.0,0.77,0.020141,True,True



Still incomplete after fallback preview
No incomplete spreads after fallback.

Saved
Fallback candidate plan:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback unique contract plan: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_unique_contract_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback candidate cache:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback map:                  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_fullhist_1sd3sd_long5_cal_v1.parquet
Leg prices with fallback:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi

In [1]:
# Cell 2G.DATA.4 — Mechanical sanity checks for full-history 1SD / 3SD call data
# Purpose:
#   Validate the full-history SPY 1SD / 3SD call-spread data panel after exact-primary
#   pricing and conservative fallback recovery.
#
# This is a DATA LAYER cell:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule
#
# It checks:
#   - row uniqueness and joins
#   - two legs per spread
#   - final leg price completeness
#   - fallback direction validity
#   - strike structure
#   - width / credit / max-loss formulas
#   - expiration intrinsic formula
#   - P&L / return formulas
#   - large fallback flags
#   - explanation for non-P&L-complete rows

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

LEG_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_{OUT_SUFFIX}.parquet"
)

FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_{OUT_SUFFIX}.parquet"
)

PRIMARY_CONTRACT_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_{OUT_SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

TOL = 1e-10

print("=" * 100)
print("Cell 2G.DATA.4 — Mechanical sanity checks for full-history 1SD / 3SD call data")
print("=" * 100)
print(f"Universe:                    {UNIVERSE_PATH}")
print(f"Leg prices with fallback:    {LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Spread prices with fallback: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Fallback map:                {FALLBACK_MAP_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def near(a, b, tol=TOL):
    return (pd.to_numeric(a, errors="coerce") - pd.to_numeric(b, errors="coerce")).abs() <= tol

def add_check(rows, check_name, severity, passed, failed_count, detail):
    rows.append({
        "check_name": check_name,
        "severity": severity,
        "passed": bool(passed),
        "failed_count": int(failed_count) if pd.notna(failed_count) else np.nan,
        "detail": detail,
    })

def reason_list(row):
    reasons = []

    if not bool(row.get("complete_after_fallback", False)):
        reasons.append("missing_final_leg_price")

    if not bool(row.get("outcome_available", False)):
        reasons.append("missing_expiry_outcome")

    if pd.isna(row.get("entry_credit_after_fallback")):
        reasons.append("missing_entry_credit")
    elif row.get("entry_credit_after_fallback") <= 0:
        reasons.append("non_positive_entry_credit")

    if pd.isna(row.get("effective_width_after_fallback")):
        reasons.append("missing_effective_width")
    elif row.get("effective_width_after_fallback") <= 0:
        reasons.append("non_positive_effective_width")

    if pd.isna(row.get("max_loss_after_fallback")):
        reasons.append("missing_max_loss")
    elif row.get("max_loss_after_fallback") <= 0:
        reasons.append("non_positive_max_loss")

    if pd.isna(row.get("terminal_spread_intrinsic_after_fallback")):
        reasons.append("missing_terminal_intrinsic")

    if pd.isna(row.get("expiry_pnl_after_fallback")):
        reasons.append("missing_expiry_pnl")

    if pd.isna(row.get("return_on_max_loss_after_fallback")):
        reasons.append("missing_return_on_max_loss")

    if not reasons:
        reasons.append("unknown")

    return ";".join(reasons)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    LEG_PRICES_WITH_FALLBACK_PATH,
    SPREAD_PRICES_WITH_FALLBACK_PATH,
    FALLBACK_MAP_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
legs = pd.read_parquet(LEG_PRICES_WITH_FALLBACK_PATH).copy()
spreads = pd.read_parquet(SPREAD_PRICES_WITH_FALLBACK_PATH).copy()
fallback_map = pd.read_parquet(FALLBACK_MAP_PATH).copy()

primary_cache = pd.read_parquet(PRIMARY_CONTRACT_CACHE_PATH).copy() if PRIMARY_CONTRACT_CACHE_PATH.exists() else pd.DataFrame()
fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_CACHE_PATH).copy() if FALLBACK_CANDIDATE_CACHE_PATH.exists() else pd.DataFrame()

for df in [universe, legs, spreads, fallback_map, primary_cache, fallback_cache]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "leg_price_usable",
    "primary_leg_price_usable",
    "fallback_used",
    "fallback_found",
    "final_leg_price_usable",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
]:
    if c in legs.columns:
        legs[c] = bool_series(legs[c], index=legs.index)

for c in [
    "outcome_available",
    "exact_primary_complete",
    "exact_primary_pnl_complete",
    "complete_after_fallback",
    "fallback_pnl_complete",
    "fallback_used_any",
    "large_fallback_dollar_flag_any",
    "large_fallback_width_pct_flag_any",
    "trade_win_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "fallback_found",
    "fallback_direction_valid",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
]:
    if c in fallback_map.columns:
        fallback_map[c] = bool_series(fallback_map[c], index=fallback_map.index)

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Universe rows:             {len(universe):,}")
print(f"Leg rows:                  {len(legs):,}")
print(f"Spread rows:               {len(spreads):,}")
print(f"Fallback map rows:         {len(fallback_map):,}")
print(f"Primary cache rows:        {len(primary_cache):,}")
print(f"Fallback cache rows:       {len(fallback_cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Derived diagnostics
# --------------------------------------------------------------------------------------------------

# Leg counts per spread.
leg_counts = (
    legs
    .groupby("selected_trade_id", dropna=False)
    .agg(
        leg_count=("leg_label", "size"),
        short_legs=("leg_label", lambda x: int((x == "short_call_1sd").sum())),
        long_legs=("leg_label", lambda x: int((x == "long_call_3sd").sum())),
        final_usable_legs=("final_leg_price_usable", "sum"),
        fallback_used_legs=("fallback_used", "sum"),
    )
    .reset_index()
)

# Recompute formulas independently.
calc = spreads.copy()

calc["calc_effective_width"] = calc["long_effective_strike"] - calc["short_effective_strike"]
calc["calc_entry_credit"] = calc["short_bid_final"] - calc["long_ask_final"]
calc["calc_max_loss"] = calc["calc_effective_width"] - calc["calc_entry_credit"]

calc["calc_terminal_intrinsic"] = (
    np.maximum(calc["expiry_spy_close"] - calc["short_effective_strike"], 0.0)
    - np.maximum(calc["expiry_spy_close"] - calc["long_effective_strike"], 0.0)
)

calc.loc[~calc["outcome_available"], "calc_terminal_intrinsic"] = np.nan

calc["calc_expiry_pnl"] = calc["calc_entry_credit"] - calc["calc_terminal_intrinsic"]
calc["calc_return_on_max_loss"] = calc["calc_expiry_pnl"] / calc["calc_max_loss"]
calc["calc_trade_win"] = calc["calc_expiry_pnl"] > 0

# Non-P&L-complete row reason.
non_pnl = calc.loc[~calc["fallback_pnl_complete"]].copy()
if not non_pnl.empty:
    non_pnl["non_pnl_reason"] = non_pnl.apply(reason_list, axis=1)
else:
    non_pnl["non_pnl_reason"] = pd.Series(dtype=str)

non_pnl_reason_summary = (
    non_pnl["non_pnl_reason"]
    .value_counts(dropna=False)
    .rename_axis("non_pnl_reason")
    .reset_index(name="rows")
)

# Fallback direction checks at leg level.
fallback_used_legs = legs.loc[legs["fallback_used"]].copy()

short_fb = fallback_used_legs.loc[fallback_used_legs["leg_label"].eq("short_call_1sd")].copy()
long_fb = fallback_used_legs.loc[fallback_used_legs["leg_label"].eq("long_call_3sd")].copy()

short_direction_bad = short_fb.loc[
    ~(
        (short_fb["effective_strike"] >= short_fb["target_strike"])
        & (short_fb["effective_strike"] < short_fb["long_strike"])
    )
].copy()

long_direction_bad = long_fb.loc[
    ~(
        (long_fb["effective_strike"] <= long_fb["target_strike"])
        & (long_fb["effective_strike"] > long_fb["short_strike"])
    )
].copy()

exact_primary_legs = legs.loc[legs["final_price_source"].eq("exact_primary")].copy()

exact_primary_strike_changed = exact_primary_legs.loc[
    ~near(exact_primary_legs["effective_strike"], exact_primary_legs["target_strike"])
].copy()

# Large fallback diagnostics.
large_fallback_spreads = spreads.loc[
    spreads["large_fallback_dollar_flag_any"]
    | spreads["large_fallback_width_pct_flag_any"]
].copy()

fallback_slippage_by_leg = (
    fallback_used_legs
    .groupby("leg_label", dropna=False)
    .agg(
        fallback_used=("selected_trade_id", "size"),
        avg_abs_slip=("abs_strike_slip", "mean"),
        median_abs_slip=("abs_strike_slip", "median"),
        max_abs_slip=("abs_strike_slip", "max"),
        avg_slip_pct_original_width=("slip_pct_original_width", "mean"),
        median_slip_pct_original_width=("slip_pct_original_width", "median"),
        max_slip_pct_original_width=("slip_pct_original_width", "max"),
        large_dollar_flags=("large_fallback_dollar_flag", "sum"),
        large_width_pct_flags=("large_fallback_width_pct_flag", "sum"),
    )
    .reset_index()
)

# Coverage summaries.
global_summary = pd.DataFrame([{
    "universe_rows": int(len(universe)),
    "spread_rows": int(len(spreads)),
    "leg_rows": int(len(legs)),
    "fallback_map_rows": int(len(fallback_map)),
    "primary_cache_rows": int(len(primary_cache)),
    "fallback_cache_rows": int(len(fallback_cache)),
    "exact_primary_complete_spreads": int(spreads["exact_primary_complete"].sum()),
    "complete_after_fallback": int(spreads["complete_after_fallback"].sum()),
    "fallback_pnl_complete": int(spreads["fallback_pnl_complete"].sum()),
    "non_pnl_complete_rows": int((~spreads["fallback_pnl_complete"]).sum()),
    "fallback_used_any": int(spreads["fallback_used_any"].sum()),
    "large_fallback_dollar_flag_any": int(spreads["large_fallback_dollar_flag_any"].sum()),
    "large_fallback_width_pct_flag_any": int(spreads["large_fallback_width_pct_flag_any"].sum()),
    "avg_entry_credit_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "entry_credit_after_fallback"].mean()),
    "median_entry_credit_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "entry_credit_after_fallback"].median()),
    "avg_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].mean()),
    "median_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].median()),
    "worst_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].min()),
    "win_rate_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "trade_win_after_fallback"].mean()),
}])

coverage_by_tenor = (
    spreads
    .groupby("tenor", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        exact_primary_complete=("exact_primary_complete", "sum"),
        complete_after_fallback=("complete_after_fallback", "sum"),
        fallback_pnl_complete=("fallback_pnl_complete", "sum"),
        non_pnl_complete=("fallback_pnl_complete", lambda x: int((~x).sum())),
        fallback_used_any=("fallback_used_any", "sum"),
        large_fallback_dollar_flag_any=("large_fallback_dollar_flag_any", "sum"),
        large_fallback_width_pct_flag_any=("large_fallback_width_pct_flag_any", "sum"),
        avg_entry_credit=("entry_credit_after_fallback", "mean"),
        median_entry_credit=("entry_credit_after_fallback", "median"),
        avg_return=("return_on_max_loss_after_fallback", "mean"),
        median_return=("return_on_max_loss_after_fallback", "median"),
        worst_return=("return_on_max_loss_after_fallback", "min"),
        win_rate=("trade_win_after_fallback", "mean"),
    )
    .reset_index()
)

coverage_by_year = (
    spreads
    .groupby("trade_year", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        exact_primary_complete=("exact_primary_complete", "sum"),
        complete_after_fallback=("complete_after_fallback", "sum"),
        fallback_pnl_complete=("fallback_pnl_complete", "sum"),
        non_pnl_complete=("fallback_pnl_complete", lambda x: int((~x).sum())),
        fallback_used_any=("fallback_used_any", "sum"),
        large_fallback_dollar_flag_any=("large_fallback_dollar_flag_any", "sum"),
        large_fallback_width_pct_flag_any=("large_fallback_width_pct_flag_any", "sum"),
        avg_entry_credit=("entry_credit_after_fallback", "mean"),
        median_entry_credit=("entry_credit_after_fallback", "median"),
        avg_return=("return_on_max_loss_after_fallback", "mean"),
        median_return=("return_on_max_loss_after_fallback", "median"),
        worst_return=("return_on_max_loss_after_fallback", "min"),
        win_rate=("trade_win_after_fallback", "mean"),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Hard and soft checks
# --------------------------------------------------------------------------------------------------

checks = []

# Row identity / mapping.
add_check(
    checks,
    "spread row count equals universe row count",
    "hard",
    len(spreads) == len(universe),
    abs(len(spreads) - len(universe)),
    f"spreads={len(spreads):,}; universe={len(universe):,}",
)

add_check(
    checks,
    "unique selected_trade_id in universe",
    "hard",
    universe["selected_trade_id"].is_unique,
    int(universe["selected_trade_id"].duplicated().sum()),
    "Universe should have one row per trade_date × tenor spread candidate.",
)

add_check(
    checks,
    "unique selected_trade_id in spread panel",
    "hard",
    spreads["selected_trade_id"].is_unique,
    int(spreads["selected_trade_id"].duplicated().sum()),
    "Spread panel should have one row per spread candidate.",
)

missing_spreads_from_universe = set(universe["selected_trade_id"]) - set(spreads["selected_trade_id"])
extra_spreads_not_in_universe = set(spreads["selected_trade_id"]) - set(universe["selected_trade_id"])

add_check(
    checks,
    "spread panel selected_trade_id set matches universe",
    "hard",
    len(missing_spreads_from_universe) == 0 and len(extra_spreads_not_in_universe) == 0,
    len(missing_spreads_from_universe) + len(extra_spreads_not_in_universe),
    f"missing_from_spreads={len(missing_spreads_from_universe):,}; extra_in_spreads={len(extra_spreads_not_in_universe):,}",
)

bad_leg_counts = leg_counts.loc[
    (leg_counts["leg_count"] != 2)
    | (leg_counts["short_legs"] != 1)
    | (leg_counts["long_legs"] != 1)
].copy()

add_check(
    checks,
    "exactly one short leg and one long leg per spread",
    "hard",
    bad_leg_counts.empty and len(leg_counts) == len(spreads),
    int(len(bad_leg_counts) + abs(len(leg_counts) - len(spreads))),
    f"leg_count_rows={len(leg_counts):,}; spread_rows={len(spreads):,}; bad_leg_count_rows={len(bad_leg_counts):,}",
)

# Price completeness.
missing_final_leg_prices = legs.loc[~legs["final_leg_price_usable"]].copy()

add_check(
    checks,
    "all final legs have needed bid or ask after fallback",
    "hard",
    missing_final_leg_prices.empty,
    int(len(missing_final_leg_prices)),
    "Every short leg needs usable bid; every long leg needs usable ask.",
)

incomplete_after_fallback = spreads.loc[~spreads["complete_after_fallback"]].copy()

add_check(
    checks,
    "all spreads have complete final leg prices after fallback",
    "hard",
    incomplete_after_fallback.empty,
    int(len(incomplete_after_fallback)),
    "Both legs should be usable after fallback recovery.",
)

# Fallback direction.
add_check(
    checks,
    "short fallback direction valid",
    "hard",
    short_direction_bad.empty,
    int(len(short_direction_bad)),
    "Short fallback must be at/above target short strike and below long target strike.",
)

add_check(
    checks,
    "long fallback direction valid",
    "hard",
    long_direction_bad.empty,
    int(len(long_direction_bad)),
    "Long fallback must be at/below target long strike and above short target strike.",
)

add_check(
    checks,
    "exact-primary legs retain target strike",
    "hard",
    exact_primary_strike_changed.empty,
    int(len(exact_primary_strike_changed)),
    "Exact-primary legs should not change effective strike.",
)

fallback_map_bad_direction = fallback_map.loc[
    fallback_map["fallback_found"] & ~fallback_map["fallback_direction_valid"]
].copy()

add_check(
    checks,
    "fallback map directions valid",
    "hard",
    fallback_map_bad_direction.empty,
    int(len(fallback_map_bad_direction)),
    "Fallback map should contain only valid conservative fallback directions.",
)

# Strike and economics.
bad_width = calc.loc[calc["effective_width_after_fallback"] <= 0].copy()

add_check(
    checks,
    "long effective strike above short effective strike",
    "hard",
    bad_width.empty,
    int(len(bad_width)),
    "Effective width must be positive.",
)

bad_width_formula = calc.loc[
    calc["effective_width_after_fallback"].notna()
    & ~near(calc["effective_width_after_fallback"], calc["calc_effective_width"])
].copy()

add_check(
    checks,
    "effective width formula reconciles",
    "hard",
    bad_width_formula.empty,
    int(len(bad_width_formula)),
    "effective_width = long_effective_strike - short_effective_strike.",
)

bad_credit_formula = calc.loc[
    calc["entry_credit_after_fallback"].notna()
    & ~near(calc["entry_credit_after_fallback"], calc["calc_entry_credit"])
].copy()

add_check(
    checks,
    "entry credit formula reconciles",
    "hard",
    bad_credit_formula.empty,
    int(len(bad_credit_formula)),
    "entry_credit = short_bid_final - long_ask_final.",
)

bad_max_loss_formula = calc.loc[
    calc["max_loss_after_fallback"].notna()
    & ~near(calc["max_loss_after_fallback"], calc["calc_max_loss"])
].copy()

add_check(
    checks,
    "max loss formula reconciles",
    "hard",
    bad_max_loss_formula.empty,
    int(len(bad_max_loss_formula)),
    "max_loss = effective_width - entry_credit.",
)

bad_intrinsic_formula = calc.loc[
    calc["terminal_spread_intrinsic_after_fallback"].notna()
    & ~near(calc["terminal_spread_intrinsic_after_fallback"], calc["calc_terminal_intrinsic"])
].copy()

add_check(
    checks,
    "terminal intrinsic formula reconciles",
    "hard",
    bad_intrinsic_formula.empty,
    int(len(bad_intrinsic_formula)),
    "terminal_intrinsic = max(expiry - short, 0) - max(expiry - long, 0).",
)

bad_intrinsic_bounds = calc.loc[
    calc["terminal_spread_intrinsic_after_fallback"].notna()
    & (
        (calc["terminal_spread_intrinsic_after_fallback"] < -TOL)
        | (calc["terminal_spread_intrinsic_after_fallback"] - calc["effective_width_after_fallback"] > TOL)
    )
].copy()

add_check(
    checks,
    "terminal intrinsic bounded between zero and width",
    "hard",
    bad_intrinsic_bounds.empty,
    int(len(bad_intrinsic_bounds)),
    "Terminal intrinsic should be in [0, effective_width].",
)

bad_pnl_formula = calc.loc[
    calc["expiry_pnl_after_fallback"].notna()
    & ~near(calc["expiry_pnl_after_fallback"], calc["calc_expiry_pnl"])
].copy()

add_check(
    checks,
    "expiry P&L formula reconciles",
    "hard",
    bad_pnl_formula.empty,
    int(len(bad_pnl_formula)),
    "expiry_pnl = entry_credit - terminal_intrinsic.",
)

bad_return_formula = calc.loc[
    calc["return_on_max_loss_after_fallback"].notna()
    & ~near(calc["return_on_max_loss_after_fallback"], calc["calc_return_on_max_loss"])
].copy()

add_check(
    checks,
    "return on max loss formula reconciles",
    "hard",
    bad_return_formula.empty,
    int(len(bad_return_formula)),
    "return = expiry_pnl / max_loss.",
)

bad_win_formula = calc.loc[
    calc["fallback_pnl_complete"]
    & (calc["trade_win_after_fallback"] != calc["calc_trade_win"])
].copy()

add_check(
    checks,
    "win flag formula reconciles",
    "hard",
    bad_win_formula.empty,
    int(len(bad_win_formula)),
    "trade_win = expiry_pnl > 0.",
)

# Completion explanation.
add_check(
    checks,
    "non-P&L-complete rows are explainable",
    "hard",
    non_pnl.empty or (
        non_pnl["non_pnl_reason"].notna().all()
        and ~non_pnl["non_pnl_reason"].eq("unknown").any()
    ),
    int(non_pnl["non_pnl_reason"].eq("unknown").sum()) if not non_pnl.empty else 0,
    f"non_pnl_complete_rows={len(non_pnl):,}",
)

# Soft checks.
negative_or_zero_credit = calc.loc[
    calc["complete_after_fallback"]
    & calc["outcome_available"]
    & (
        calc["entry_credit_after_fallback"].isna()
        | (calc["entry_credit_after_fallback"] <= 0)
    )
].copy()

add_check(
    checks,
    "all complete rows have positive entry credit",
    "soft",
    negative_or_zero_credit.empty,
    int(len(negative_or_zero_credit)),
    "Rows with zero/negative credit are excluded from P&L-complete status but may explain the residual 15 rows.",
)

non_positive_max_loss = calc.loc[
    calc["complete_after_fallback"]
    & calc["outcome_available"]
    & (
        calc["max_loss_after_fallback"].isna()
        | (calc["max_loss_after_fallback"] <= 0)
    )
].copy()

add_check(
    checks,
    "all complete rows have positive max loss",
    "soft",
    non_positive_max_loss.empty,
    int(len(non_positive_max_loss)),
    "Rows with non-positive max loss are excluded from P&L-complete status.",
)

add_check(
    checks,
    "large fallback flags are present for review",
    "soft",
    len(large_fallback_spreads) > 0,
    0 if len(large_fallback_spreads) > 0 else 1,
    f"large_fallback_spreads={len(large_fallback_spreads):,}; flags are report-only, not exclusions.",
)

checks_df = pd.DataFrame(checks)

hard_failed = checks_df.loc[
    checks_df["severity"].eq("hard")
    & ~checks_df["passed"]
].copy()

soft_failed = checks_df.loc[
    checks_df["severity"].eq("soft")
    & ~checks_df["passed"]
].copy()

# --------------------------------------------------------------------------------------------------
# Save audits
# --------------------------------------------------------------------------------------------------

checks_path = AUDIT_DIR / f"02G_DATA_4_mechanical_sanity_checks_{OUT_SUFFIX}_{timestamp}.csv"
global_summary_path = AUDIT_DIR / f"02G_DATA_4_global_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_tenor_path = AUDIT_DIR / f"02G_DATA_4_coverage_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_year_path = AUDIT_DIR / f"02G_DATA_4_coverage_by_year_{OUT_SUFFIX}_{timestamp}.csv"
fallback_slippage_by_leg_path = AUDIT_DIR / f"02G_DATA_4_fallback_slippage_by_leg_{OUT_SUFFIX}_{timestamp}.csv"
non_pnl_reason_summary_path = AUDIT_DIR / f"02G_DATA_4_non_pnl_reason_summary_{OUT_SUFFIX}_{timestamp}.csv"
non_pnl_preview_path = AUDIT_DIR / f"02G_DATA_4_non_pnl_preview_{OUT_SUFFIX}_{timestamp}.csv"
large_fallback_preview_path = AUDIT_DIR / f"02G_DATA_4_large_fallback_preview_{OUT_SUFFIX}_{timestamp}.csv"
hard_failed_preview_path = AUDIT_DIR / f"02G_DATA_4_hard_failed_preview_{OUT_SUFFIX}_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02G_DATA_4_mechanical_sanity_{OUT_SUFFIX}_manifest_{timestamp}.json"

atomic_write_csv(checks_df, checks_path)
atomic_write_csv(global_summary, global_summary_path)
atomic_write_csv(coverage_by_tenor, coverage_by_tenor_path)
atomic_write_csv(coverage_by_year, coverage_by_year_path)
atomic_write_csv(fallback_slippage_by_leg, fallback_slippage_by_leg_path)
atomic_write_csv(non_pnl_reason_summary, non_pnl_reason_summary_path)
atomic_write_csv(non_pnl.head(1000), non_pnl_preview_path)
atomic_write_csv(large_fallback_spreads.head(1000), large_fallback_preview_path)

# Build a compact failed-row preview only when needed.
failed_preview_frames = []

for name, df in [
    ("bad_leg_counts", bad_leg_counts),
    ("missing_final_leg_prices", missing_final_leg_prices),
    ("incomplete_after_fallback", incomplete_after_fallback),
    ("short_direction_bad", short_direction_bad),
    ("long_direction_bad", long_direction_bad),
    ("fallback_map_bad_direction", fallback_map_bad_direction),
    ("bad_width", bad_width),
    ("bad_width_formula", bad_width_formula),
    ("bad_credit_formula", bad_credit_formula),
    ("bad_max_loss_formula", bad_max_loss_formula),
    ("bad_intrinsic_formula", bad_intrinsic_formula),
    ("bad_intrinsic_bounds", bad_intrinsic_bounds),
    ("bad_pnl_formula", bad_pnl_formula),
    ("bad_return_formula", bad_return_formula),
    ("bad_win_formula", bad_win_formula),
]:
    if len(df):
        tmp = df.head(100).copy()
        tmp.insert(0, "failed_preview_source", name)
        failed_preview_frames.append(tmp)

if failed_preview_frames:
    hard_failed_preview = pd.concat(failed_preview_frames, ignore_index=True, sort=False)
else:
    hard_failed_preview = pd.DataFrame({"failed_preview_source": []})

atomic_write_csv(hard_failed_preview, hard_failed_preview_path)

manifest = {
    "cell": "2G.DATA.4",
    "description": "Mechanical sanity checks for full-history 1SD/3SD SPY call data after fallback recovery",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "universe_path": str(UNIVERSE_PATH),
        "leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
        "spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
        "primary_contract_cache_path": str(PRIMARY_CONTRACT_CACHE_PATH),
        "fallback_candidate_cache_path": str(FALLBACK_CANDIDATE_CACHE_PATH),
    },
    "counts": {
        "universe_rows": int(len(universe)),
        "spread_rows": int(len(spreads)),
        "leg_rows": int(len(legs)),
        "fallback_map_rows": int(len(fallback_map)),
        "hard_checks": int(checks_df["severity"].eq("hard").sum()),
        "hard_failed": int(len(hard_failed)),
        "soft_checks": int(checks_df["severity"].eq("soft").sum()),
        "soft_failed": int(len(soft_failed)),
        "exact_primary_complete_spreads": int(spreads["exact_primary_complete"].sum()),
        "complete_after_fallback": int(spreads["complete_after_fallback"].sum()),
        "fallback_pnl_complete": int(spreads["fallback_pnl_complete"].sum()),
        "non_pnl_complete_rows": int((~spreads["fallback_pnl_complete"]).sum()),
        "fallback_used_any": int(spreads["fallback_used_any"].sum()),
        "large_fallback_dollar_flag_any": int(spreads["large_fallback_dollar_flag_any"].sum()),
        "large_fallback_width_pct_flag_any": int(spreads["large_fallback_width_pct_flag_any"].sum()),
    },
    "outputs": {
        "checks_path": str(checks_path),
        "global_summary_path": str(global_summary_path),
        "coverage_by_tenor_path": str(coverage_by_tenor_path),
        "coverage_by_year_path": str(coverage_by_year_path),
        "fallback_slippage_by_leg_path": str(fallback_slippage_by_leg_path),
        "non_pnl_reason_summary_path": str(non_pnl_reason_summary_path),
        "non_pnl_preview_path": str(non_pnl_preview_path),
        "large_fallback_preview_path": str(large_fallback_preview_path),
        "hard_failed_preview_path": str(hard_failed_preview_path),
        "manifest_path": str(manifest_path),
    },
    "status": "PASS" if len(hard_failed) == 0 else "FAIL",
    "next_step": "Run signal diagnostics from the fallback-adjusted full-history data panel if hard checks pass.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.4 complete — mechanical sanity checks")
print("=" * 100)

print()
print("Global summary")
display(global_summary)

print()
print("=" * 100)
print("Sanity checks")
print("=" * 100)
display(checks_df)

print()
print("=" * 100)
print("Hard check failures")
print("=" * 100)
if hard_failed.empty:
    print("No hard check failures.")
else:
    display(hard_failed)

print()
print("=" * 100)
print("Soft check failures")
print("=" * 100)
if soft_failed.empty:
    print("No soft check failures.")
else:
    display(soft_failed)

print()
print("=" * 100)
print("Non-P&L-complete reason summary")
print("=" * 100)
if non_pnl_reason_summary.empty:
    print("No non-P&L-complete rows.")
else:
    display(non_pnl_reason_summary)

print()
print("=" * 100)
print("Non-P&L-complete preview")
print("=" * 100)
if non_pnl.empty:
    print("No non-P&L-complete rows.")
else:
    display(
        non_pnl[
            [
                "selected_trade_id",
                "trade_date",
                "tenor",
                "expiration_date",
                "short_effective_strike",
                "long_effective_strike",
                "short_bid_final",
                "long_ask_final",
                "entry_credit_after_fallback",
                "effective_width_after_fallback",
                "max_loss_after_fallback",
                "outcome_available",
                "complete_after_fallback",
                "non_pnl_reason",
            ]
        ].head(100)
    )

print()
print("=" * 100)
print("Fallback slippage by leg")
print("=" * 100)
display(fallback_slippage_by_leg)

print()
print("=" * 100)
print("Coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Checks:                    {checks_path}")
print(f"Global summary:            {global_summary_path}")
print(f"Coverage by tenor:         {coverage_by_tenor_path}")
print(f"Coverage by year:          {coverage_by_year_path}")
print(f"Fallback slippage by leg:  {fallback_slippage_by_leg_path}")
print(f"Non-P&L reason summary:    {non_pnl_reason_summary_path}")
print(f"Non-P&L preview:           {non_pnl_preview_path}")
print(f"Large fallback preview:    {large_fallback_preview_path}")
print(f"Hard failed preview:       {hard_failed_preview_path}")
print(f"Manifest:                  {manifest_path}")

print()
if hard_failed.empty:
    print("Result: 2G.DATA.4 PASSED hard mechanical sanity checks.")
    print("Next step: run fixed-parameter DTE diagnostics from the full fallback-adjusted panel.")
else:
    print("Result: 2G.DATA.4 FAILED hard mechanical sanity checks.")
    print("Do not use this panel for signal diagnostics until the hard failures are repaired.")

Cell 2G.DATA.4 — Mechanical sanity checks for full-history 1SD / 3SD call data
Universe:                    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_universe_fullhist_1sd3sd_long5_cal_v1.parquet
Leg prices with fallback:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Spread prices with fallback: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback map:                C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_fullhist_1sd3sd_long5_cal_v1.parquet

Loaded inputs
Universe rows:             14,613
Leg rows:                  29,226
Spread rows:               14,613
Fallback map rows:         3,339
Primary 

,universe_rows,spread_rows,leg_rows,fallback_map_rows,primary_cache_rows,fallback_cache_rows,exact_primary_complete_spreads,complete_after_fallback,fallback_pnl_complete,non_pnl_complete_rows,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit_after_fallback,median_entry_credit_after_fallback,avg_return_on_max_loss_after_fallback,median_return_on_max_loss_after_fallback,worst_return_on_max_loss_after_fallback,win_rate_after_fallback
0,14613,14613,29226,3339,28523,50891,11562,14613,14598,15,3051,354,298,0.706971,0.65,-0.001906,0.013946,-0.668007,0.880121



Sanity checks


,check_name,severity,passed,failed_count,detail
0,spread row count equals universe row count,hard,True,0,"spreads=14,613; universe=14,613"
1,unique selected_trade_id in universe,hard,True,0,Universe should have one row per trade_date × ...
2,unique selected_trade_id in spread panel,hard,True,0,Spread panel should have one row per spread ca...
3,spread panel selected_trade_id set matches uni...,hard,True,0,missing_from_spreads=0; extra_in_spreads=0
4,exactly one short leg and one long leg per spread,hard,True,0,"leg_count_rows=14,613; spread_rows=14,613; bad..."
5,all final legs have needed bid or ask after fa...,hard,True,0,Every short leg needs usable bid; every long l...
6,all spreads have complete final leg prices aft...,hard,True,0,Both legs should be usable after fallback reco...
7,short fallback direction valid,hard,True,0,Short fallback must be at/above target short s...
8,long fallback direction valid,hard,True,0,Long fallback must be at/below target long str...
9,exact-primary legs retain target strike,hard,True,0,Exact-primary legs should not change effective...



Hard check failures


...


Coverage by tenor


,tenor,spread_rows,exact_primary_complete,complete_after_fallback,fallback_pnl_complete,non_pnl_complete,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit,median_entry_credit,avg_return,median_return,worst_return,win_rate
0,9,1632,1589,1632,1631,1,43,0,0,0.699969,0.620,-0.002720,0.020055,-0.668007,0.853554
1,12,1629,1544,1629,1629,0,85,0,0,0.769724,0.730,-0.003446,0.020408,-0.667509,0.859423
2,15,1628,1497,1628,1628,0,131,2,1,0.646947,0.600,-0.001916,0.014885,-0.591667,0.877150
3,18,1625,1398,1625,1623,2,227,14,12,0.726486,0.680,-0.002413,0.015228,-0.555219,0.878769
4,21,1624,1317,1624,1622,2,307,28,18,0.628073,0.590,-0.001631,0.012241,-0.481988,0.895320
5,24,1622,1214,1622,1621,1,408,38,33,0.708354,0.640,-0.002227,0.012889,-0.515539,0.888409
6,27,1620,1117,1620,1618,2,503,54,44,0.698630,0.645,-0.001693,0.011941,-0.477369,0.887654
7,30,1618,989,1618,1616,2,629,80,72,0.705828,0.650,-0.001232,0.011463,-0.458612,0.887515
8,33,1615,897,1615,1610,5,718,138,118,0.765455,0.710,-0.000572,0.012282,-1.000000,0.885449



Coverage by year


,trade_year,spread_rows,exact_primary_complete,complete_after_fallback,fallback_pnl_complete,non_pnl_complete,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit,median_entry_credit,avg_return,median_return,worst_return,win_rate
0,2020,2277,1801,2277,2277,0,476,199,196,0.549671,0.52,0.002984,0.012793,-0.523503,0.917874
1,2021,2268,1985,2268,2266,2,283,13,7,0.308761,0.29,-0.006186,0.007435,-1.000000,0.904321
2,2022,2259,2008,2259,2259,0,251,8,8,1.039380,1.00,0.007105,0.020160,-0.566478,0.902169
3,2023,2250,1928,2250,2250,0,322,1,1,0.766502,0.74,-0.010733,0.020942,-0.667509,0.807556
4,2024,2268,1560,2268,2262,6,708,26,13,0.756936,0.73,-0.003127,0.017409,-0.509074,0.832011
5,2025,2250,1419,2250,2245,5,831,107,73,0.783942,0.76,0.009164,0.014610,-0.435404,0.949333
6,2026,1041,861,1041,1039,2,180,0,0,0.772238,0.74,-0.026136,0.010486,-0.668007,0.796350



Saved
Checks:                    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_mechanical_sanity_checks_fullhist_1sd3sd_long5_cal_v1_20260714_192710.csv
Global summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_global_summary_fullhist_1sd3sd_long5_cal_v1_20260714_192710.csv
Coverage by tenor:         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_coverage_by_tenor_fullhist_1sd3sd_long5_cal_v1_20260714_192710.csv
Coverage by year:          C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_coverage_by_year_fullhist_1sd3sd_long5_cal_v1_20260714_192710.csv
Fallback slippage by leg:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_fallback_slippage_by_leg_fullhist_1sd3sd_long5_cal_v1_20260714_192710.csv
Non-P&L reason summary:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_non_pnl_reason_summary_fullhist_1sd3sd_long5_cal_v1_20260714_19271

In [2]:
# Cell 2G.DATA.4A — Repair joint fallback strike collisions
# Purpose:
#   Repair any fallback-adjusted call spreads where independently selected fallback legs
#   caused long_effective_strike <= short_effective_strike.
#
# This is a DATA LAYER repair:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule
#
# It keeps the conservative fallback directions:
#   short fallback: at/above target short, but below final long
#   long fallback:  at/below target long, but above final short
#
# It backs up the current fallback-adjusted leg/spread parquet files, repairs the collision,
# overwrites the canonical fallback-adjusted files, and then 2G.DATA.4 should be rerun.

from pathlib import Path
from datetime import datetime
import shutil
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

LEG_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

FALLBACK_CANDIDATE_PLAN_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_plan_{OUT_SUFFIX}.parquet"
)

FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_{OUT_SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_{OUT_SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

LARGE_FALLBACK_DOLLARS_FLAG = 10
LARGE_FALLBACK_WIDTH_PCT_FLAG = 0.25

print("=" * 100)
print("Cell 2G.DATA.4A — Repair joint fallback strike collisions")
print("=" * 100)
print(f"Leg prices with fallback:    {LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Spread prices with fallback: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Fallback candidate plan:     {FALLBACK_CANDIDATE_PLAN_PATH}")
print(f"Fallback candidate cache:    {FALLBACK_CANDIDATE_CACHE_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def backup_file(path, tag):
    path = Path(path)
    backup_path = AUDIT_DIR / f"{path.stem}_{tag}_{timestamp}{path.suffix}"
    shutil.copy2(path, backup_path)
    return backup_path

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = np.nan
    return df

def reason_list(row):
    reasons = []

    if not bool(row.get("complete_after_fallback", False)):
        reasons.append("missing_final_leg_price")

    if not bool(row.get("outcome_available", False)):
        reasons.append("missing_expiry_outcome")

    if pd.isna(row.get("entry_credit_after_fallback")):
        reasons.append("missing_entry_credit")
    elif row.get("entry_credit_after_fallback") <= 0:
        reasons.append("non_positive_entry_credit")

    if pd.isna(row.get("effective_width_after_fallback")):
        reasons.append("missing_effective_width")
    elif row.get("effective_width_after_fallback") <= 0:
        reasons.append("non_positive_effective_width")

    if pd.isna(row.get("max_loss_after_fallback")):
        reasons.append("missing_max_loss")
    elif row.get("max_loss_after_fallback") <= 0:
        reasons.append("non_positive_max_loss")

    if pd.isna(row.get("terminal_spread_intrinsic_after_fallback")):
        reasons.append("missing_terminal_intrinsic")

    if pd.isna(row.get("expiry_pnl_after_fallback")):
        reasons.append("missing_expiry_pnl")

    if pd.isna(row.get("return_on_max_loss_after_fallback")):
        reasons.append("missing_return_on_max_loss")

    if not reasons:
        reasons.append("unknown")

    return ";".join(reasons)

def build_spread_panel(universe, legs, old_spreads):
    spread_base_cols = [
        "selected_trade_id",
        "trade_date",
        "trade_year",
        "tenor",
        "expiration_date",
        "expiration_str",
        "outcome_available",
        "short_strike",
        "long_strike",
        "spy_close",
        "expiry_spy_close",
        "vix_style_vol_pct",
        "sd_move",
        "short_call_1sd_raw",
        "long_call_3sd_raw",
        "width",
    ]

    spread_base = universe[spread_base_cols].drop_duplicates("selected_trade_id").copy()

    exact_status = old_spreads[
        [
            "selected_trade_id",
            "exact_primary_complete",
            "exact_primary_pnl_complete",
        ]
    ].copy()

    spread_base = spread_base.merge(
        exact_status,
        on="selected_trade_id",
        how="left",
        validate="one_to_one",
    )

    for c in ["exact_primary_complete", "exact_primary_pnl_complete"]:
        spread_base[c] = bool_series(spread_base[c], index=spread_base.index)

    short_final = (
        legs
        .loc[legs["leg_label"].eq("short_call_1sd")]
        [
            [
                "selected_trade_id",
                "contract_request_id",
                "fallback_contract_request_id",
                "target_strike",
                "effective_strike",
                "final_bid",
                "final_ask",
                "final_mid",
                "final_leg_price_usable",
                "fallback_used",
                "fallback_found",
                "signed_strike_slip",
                "abs_strike_slip",
                "slip_pct_original_width",
                "large_fallback_dollar_flag",
                "large_fallback_width_pct_flag",
                "final_price_source",
            ]
        ]
        .rename(
            columns={
                "contract_request_id": "short_exact_contract_request_id",
                "fallback_contract_request_id": "short_fallback_contract_request_id",
                "target_strike": "short_target_strike",
                "effective_strike": "short_effective_strike",
                "final_bid": "short_bid_final",
                "final_ask": "short_ask_final",
                "final_mid": "short_mid_final",
                "final_leg_price_usable": "short_final_leg_price_usable",
                "fallback_used": "short_fallback_used",
                "fallback_found": "short_fallback_found",
                "signed_strike_slip": "short_signed_strike_slip",
                "abs_strike_slip": "short_abs_strike_slip",
                "slip_pct_original_width": "short_slip_pct_original_width",
                "large_fallback_dollar_flag": "short_large_fallback_dollar_flag",
                "large_fallback_width_pct_flag": "short_large_fallback_width_pct_flag",
                "final_price_source": "short_final_price_source",
            }
        )
    )

    long_final = (
        legs
        .loc[legs["leg_label"].eq("long_call_3sd")]
        [
            [
                "selected_trade_id",
                "contract_request_id",
                "fallback_contract_request_id",
                "target_strike",
                "effective_strike",
                "final_bid",
                "final_ask",
                "final_mid",
                "final_leg_price_usable",
                "fallback_used",
                "fallback_found",
                "signed_strike_slip",
                "abs_strike_slip",
                "slip_pct_original_width",
                "large_fallback_dollar_flag",
                "large_fallback_width_pct_flag",
                "final_price_source",
            ]
        ]
        .rename(
            columns={
                "contract_request_id": "long_exact_contract_request_id",
                "fallback_contract_request_id": "long_fallback_contract_request_id",
                "target_strike": "long_target_strike",
                "effective_strike": "long_effective_strike",
                "final_bid": "long_bid_final",
                "final_ask": "long_ask_final",
                "final_mid": "long_mid_final",
                "final_leg_price_usable": "long_final_leg_price_usable",
                "fallback_used": "long_fallback_used",
                "fallback_found": "long_fallback_found",
                "signed_strike_slip": "long_signed_strike_slip",
                "abs_strike_slip": "long_abs_strike_slip",
                "slip_pct_original_width": "long_slip_pct_original_width",
                "large_fallback_dollar_flag": "long_large_fallback_dollar_flag",
                "large_fallback_width_pct_flag": "long_large_fallback_width_pct_flag",
                "final_price_source": "long_final_price_source",
            }
        )
    )

    out = (
        spread_base
        .merge(short_final, on="selected_trade_id", how="left", validate="one_to_one")
        .merge(long_final, on="selected_trade_id", how="left", validate="one_to_one")
    )

    for c in [
        "short_final_leg_price_usable",
        "long_final_leg_price_usable",
        "short_fallback_used",
        "long_fallback_used",
        "short_large_fallback_dollar_flag",
        "long_large_fallback_dollar_flag",
        "short_large_fallback_width_pct_flag",
        "long_large_fallback_width_pct_flag",
        "outcome_available",
    ]:
        out[c] = bool_series(out[c], index=out.index)

    out["complete_after_fallback"] = (
        out["short_final_leg_price_usable"]
        & out["long_final_leg_price_usable"]
    )

    out["fallback_used_any"] = (
        out["short_fallback_used"]
        | out["long_fallback_used"]
    )

    out["large_fallback_dollar_flag_any"] = (
        out["short_large_fallback_dollar_flag"]
        | out["long_large_fallback_dollar_flag"]
    )

    out["large_fallback_width_pct_flag_any"] = (
        out["short_large_fallback_width_pct_flag"]
        | out["long_large_fallback_width_pct_flag"]
    )

    out["effective_width_after_fallback"] = (
        out["long_effective_strike"] - out["short_effective_strike"]
    )

    out["entry_credit_after_fallback"] = (
        out["short_bid_final"] - out["long_ask_final"]
    )

    out["max_loss_after_fallback"] = (
        out["effective_width_after_fallback"] - out["entry_credit_after_fallback"]
    )

    out["terminal_spread_intrinsic_after_fallback"] = (
        np.maximum(out["expiry_spy_close"] - out["short_effective_strike"], 0.0)
        - np.maximum(out["expiry_spy_close"] - out["long_effective_strike"], 0.0)
    )

    out.loc[
        ~out["outcome_available"],
        "terminal_spread_intrinsic_after_fallback",
    ] = np.nan

    out["expiry_pnl_after_fallback"] = (
        out["entry_credit_after_fallback"]
        - out["terminal_spread_intrinsic_after_fallback"]
    )

    out["return_on_max_loss_after_fallback"] = (
        out["expiry_pnl_after_fallback"] / out["max_loss_after_fallback"]
    )

    out["trade_win_after_fallback"] = out["expiry_pnl_after_fallback"] > 0

    out["fallback_pnl_complete"] = (
        out["complete_after_fallback"]
        & out["outcome_available"]
        & out["entry_credit_after_fallback"].notna()
        & (out["entry_credit_after_fallback"] > 0)
        & out["max_loss_after_fallback"].notna()
        & (out["max_loss_after_fallback"] > 0)
        & out["effective_width_after_fallback"].notna()
        & (out["effective_width_after_fallback"] > 0)
        & out["terminal_spread_intrinsic_after_fallback"].notna()
        & out["return_on_max_loss_after_fallback"].notna()
    )

    return out

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    LEG_PRICES_WITH_FALLBACK_PATH,
    SPREAD_PRICES_WITH_FALLBACK_PATH,
    FALLBACK_CANDIDATE_PLAN_PATH,
    FALLBACK_CANDIDATE_CACHE_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
legs = pd.read_parquet(LEG_PRICES_WITH_FALLBACK_PATH).copy()
spreads = pd.read_parquet(SPREAD_PRICES_WITH_FALLBACK_PATH).copy()
candidate_plan = pd.read_parquet(FALLBACK_CANDIDATE_PLAN_PATH).copy()
candidate_cache = pd.read_parquet(FALLBACK_CANDIDATE_CACHE_PATH).copy()

for df in [universe, legs, spreads, candidate_plan, candidate_cache]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "leg_price_usable",
    "primary_leg_price_usable",
    "fallback_used",
    "fallback_found",
    "final_leg_price_usable",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
]:
    if c in legs.columns:
        legs[c] = bool_series(legs[c], index=legs.index)

for c in [
    "outcome_available",
    "exact_primary_complete",
    "exact_primary_pnl_complete",
    "complete_after_fallback",
    "fallback_pnl_complete",
    "fallback_used_any",
    "large_fallback_dollar_flag_any",
    "large_fallback_width_pct_flag_any",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

candidate_cache = ensure_numeric(
    candidate_cache,
    ["strike", "bid", "ask", "mid", "iv", "delta", "gamma", "theta", "vega", "rho"],
)

candidate_cache["bid_usable"] = candidate_cache["bid"].notna() & (candidate_cache["bid"] >= 0)
candidate_cache["ask_usable"] = candidate_cache["ask"].notna() & (candidate_cache["ask"] >= 0)
candidate_cache["both_sides_usable"] = candidate_cache["bid_usable"] & candidate_cache["ask_usable"]

candidate_cache = (
    candidate_cache
    .sort_values(["contract_request_id", "both_sides_usable"], ascending=[True, False])
    .drop_duplicates("contract_request_id", keep="first")
    .reset_index(drop=True)
)

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Universe rows:          {len(universe):,}")
print(f"Leg rows:               {len(legs):,}")
print(f"Spread rows:            {len(spreads):,}")
print(f"Fallback candidate rows:{len(candidate_plan):,}")
print(f"Fallback cache rows:    {len(candidate_cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Detect and repair width collisions
# --------------------------------------------------------------------------------------------------

bad_width_before = spreads.loc[spreads["effective_width_after_fallback"] <= 0].copy()

print("=" * 100)
print("Width collisions before repair")
print("=" * 100)
print(f"Rows with effective_width_after_fallback <= 0: {len(bad_width_before):,}")

if not bad_width_before.empty:
    display(
        bad_width_before[
            [
                "selected_trade_id",
                "trade_date",
                "tenor",
                "expiration_date",
                "short_effective_strike",
                "long_effective_strike",
                "short_bid_final",
                "long_ask_final",
                "entry_credit_after_fallback",
                "effective_width_after_fallback",
            ]
        ]
    )

repair_log_rows = []

def get_priced_candidates(selected_trade_id, leg_label, current_short_eff, current_long_eff):
    rows = candidate_plan.loc[
        candidate_plan["selected_trade_id"].eq(selected_trade_id)
        & candidate_plan["leg_label"].eq(leg_label)
    ].copy()

    if rows.empty:
        return rows

    rows = rows.merge(
        candidate_cache[
            [
                "contract_request_id",
                "bid",
                "ask",
                "mid",
                "iv",
                "delta",
                "gamma",
                "theta",
                "vega",
                "rho",
                "bid_usable",
                "ask_usable",
                "price_found",
                "status",
                "status_code",
                "http_status_code",
                "query_attempted",
                "cache_source",
                "seed_cache_source",
                "endpoint",
            ]
        ],
        on="contract_request_id",
        how="left",
        validate="many_to_one",
    )

    if leg_label == "short_call_1sd":
        rows = rows.loc[
            rows["bid_usable"]
            & (rows["candidate_strike"] >= rows["target_strike"])
            & (rows["candidate_strike"] < current_long_eff)
        ].copy()
    elif leg_label == "long_call_3sd":
        rows = rows.loc[
            rows["ask_usable"]
            & (rows["candidate_strike"] <= rows["target_strike"])
            & (rows["candidate_strike"] > current_short_eff)
        ].copy()
    else:
        raise ValueError(f"Unknown leg_label: {leg_label}")

    rows = rows.sort_values(["candidate_rank", "abs_strike_slip", "candidate_strike"]).reset_index(drop=True)
    return rows

def update_leg_from_candidate(legs, selected_trade_id, leg_label, candidate):
    mask = legs["selected_trade_id"].eq(selected_trade_id) & legs["leg_label"].eq(leg_label)

    if int(mask.sum()) != 1:
        raise RuntimeError(f"Expected exactly one leg row for {selected_trade_id} / {leg_label}; found {int(mask.sum())}")

    idx = legs.index[mask][0]

    legs.loc[idx, "fallback_used"] = True
    legs.loc[idx, "fallback_found"] = True
    legs.loc[idx, "fallback_contract_request_id"] = candidate["contract_request_id"]
    legs.loc[idx, "fallback_candidate_rank"] = candidate["candidate_rank"]

    legs.loc[idx, "effective_strike"] = candidate["candidate_strike"]
    legs.loc[idx, "signed_strike_slip"] = candidate["signed_strike_slip"]
    legs.loc[idx, "abs_strike_slip"] = candidate["abs_strike_slip"]
    legs.loc[idx, "slip_pct_original_width"] = candidate["slip_pct_original_width"]

    legs.loc[idx, "final_bid"] = candidate["bid"]
    legs.loc[idx, "final_ask"] = candidate["ask"]
    legs.loc[idx, "final_mid"] = candidate["mid"]
    legs.loc[idx, "final_iv"] = candidate.get("iv", np.nan)
    legs.loc[idx, "final_delta"] = candidate.get("delta", np.nan)
    legs.loc[idx, "final_gamma"] = candidate.get("gamma", np.nan)
    legs.loc[idx, "final_theta"] = candidate.get("theta", np.nan)
    legs.loc[idx, "final_vega"] = candidate.get("vega", np.nan)

    legs.loc[idx, "large_fallback_dollar_flag"] = (
        float(candidate["abs_strike_slip"]) > LARGE_FALLBACK_DOLLARS_FLAG
    )
    legs.loc[idx, "large_fallback_width_pct_flag"] = (
        float(candidate["slip_pct_original_width"]) > LARGE_FALLBACK_WIDTH_PCT_FLAG
    )

    legs.loc[idx, "final_price_source"] = "fallback_conservative_jointsafe"

    legs.loc[idx, "final_bid_usable"] = pd.notna(candidate["bid"]) and float(candidate["bid"]) >= 0
    legs.loc[idx, "final_ask_usable"] = pd.notna(candidate["ask"]) and float(candidate["ask"]) >= 0

    needed = legs.loc[idx, "needed_price_field"]
    if needed == "bid":
        legs.loc[idx, "final_leg_price_usable"] = bool(legs.loc[idx, "final_bid_usable"])
    elif needed == "ask":
        legs.loc[idx, "final_leg_price_usable"] = bool(legs.loc[idx, "final_ask_usable"])
    else:
        raise RuntimeError(f"Unknown needed_price_field={needed}")

    return legs

for _, bad in bad_width_before.iterrows():
    sid = bad["selected_trade_id"]

    short_row = legs.loc[
        legs["selected_trade_id"].eq(sid)
        & legs["leg_label"].eq("short_call_1sd")
    ].iloc[0]

    long_row = legs.loc[
        legs["selected_trade_id"].eq(sid)
        & legs["leg_label"].eq("long_call_3sd")
    ].iloc[0]

    short_eff = float(short_row["effective_strike"])
    long_eff = float(long_row["effective_strike"])

    repaired = False
    repair_action = None

    # Prefer repairing the long leg upward relative to the current short effective strike
    # while remaining conservative versus the original long target.
    long_candidates = get_priced_candidates(
        selected_trade_id=sid,
        leg_label="long_call_3sd",
        current_short_eff=short_eff,
        current_long_eff=long_eff,
    )

    if not long_candidates.empty:
        chosen = long_candidates.iloc[0]
        old_long_eff = long_eff

        legs = update_leg_from_candidate(
            legs=legs,
            selected_trade_id=sid,
            leg_label="long_call_3sd",
            candidate=chosen,
        )

        long_eff = float(chosen["candidate_strike"])
        repaired = long_eff > short_eff
        repair_action = "reselected_long_fallback"

        repair_log_rows.append({
            "selected_trade_id": sid,
            "repair_action": repair_action,
            "old_short_effective_strike": short_eff,
            "old_long_effective_strike": old_long_eff,
            "new_short_effective_strike": short_eff,
            "new_long_effective_strike": long_eff,
            "chosen_contract_request_id": chosen["contract_request_id"],
            "chosen_candidate_rank": int(chosen["candidate_rank"]),
            "chosen_abs_strike_slip": float(chosen["abs_strike_slip"]),
            "chosen_slip_pct_original_width": float(chosen["slip_pct_original_width"]),
            "repaired": bool(repaired),
        })

    # If long repair did not work, try lowering the short fallback below the current long.
    if not repaired:
        short_candidates = get_priced_candidates(
            selected_trade_id=sid,
            leg_label="short_call_1sd",
            current_short_eff=short_eff,
            current_long_eff=long_eff,
        )

        if not short_candidates.empty:
            chosen = short_candidates.iloc[0]
            old_short_eff = short_eff

            legs = update_leg_from_candidate(
                legs=legs,
                selected_trade_id=sid,
                leg_label="short_call_1sd",
                candidate=chosen,
            )

            short_eff = float(chosen["candidate_strike"])
            repaired = long_eff > short_eff
            repair_action = "reselected_short_fallback"

            repair_log_rows.append({
                "selected_trade_id": sid,
                "repair_action": repair_action,
                "old_short_effective_strike": old_short_eff,
                "old_long_effective_strike": long_eff,
                "new_short_effective_strike": short_eff,
                "new_long_effective_strike": long_eff,
                "chosen_contract_request_id": chosen["contract_request_id"],
                "chosen_candidate_rank": int(chosen["candidate_rank"]),
                "chosen_abs_strike_slip": float(chosen["abs_strike_slip"]),
                "chosen_slip_pct_original_width": float(chosen["slip_pct_original_width"]),
                "repaired": bool(repaired),
            })

    if not repaired:
        repair_log_rows.append({
            "selected_trade_id": sid,
            "repair_action": "unrepaired_no_valid_joint_candidate",
            "old_short_effective_strike": float(short_row["effective_strike"]),
            "old_long_effective_strike": float(long_row["effective_strike"]),
            "new_short_effective_strike": short_eff,
            "new_long_effective_strike": long_eff,
            "chosen_contract_request_id": np.nan,
            "chosen_candidate_rank": np.nan,
            "chosen_abs_strike_slip": np.nan,
            "chosen_slip_pct_original_width": np.nan,
            "repaired": False,
        })

repair_log = pd.DataFrame(repair_log_rows)

# Rebuild spread panel from repaired legs.
spreads_repaired = build_spread_panel(universe, legs, spreads)

bad_width_after = spreads_repaired.loc[
    spreads_repaired["effective_width_after_fallback"] <= 0
].copy()

non_pnl_after = spreads_repaired.loc[~spreads_repaired["fallback_pnl_complete"]].copy()

if not non_pnl_after.empty:
    non_pnl_after["non_pnl_reason"] = non_pnl_after.apply(reason_list, axis=1)

non_pnl_reason_summary = (
    non_pnl_after["non_pnl_reason"]
    .value_counts(dropna=False)
    .rename_axis("non_pnl_reason")
    .reset_index(name="rows")
    if not non_pnl_after.empty
    else pd.DataFrame(columns=["non_pnl_reason", "rows"])
)

summary = pd.DataFrame([{
    "bad_width_before": int(len(bad_width_before)),
    "bad_width_after": int(len(bad_width_after)),
    "repair_log_rows": int(len(repair_log)),
    "repaired_rows": int(repair_log["repaired"].sum()) if not repair_log.empty else 0,
    "spread_rows": int(len(spreads_repaired)),
    "complete_after_fallback": int(spreads_repaired["complete_after_fallback"].sum()),
    "fallback_pnl_complete": int(spreads_repaired["fallback_pnl_complete"].sum()),
    "non_pnl_complete_rows": int((~spreads_repaired["fallback_pnl_complete"]).sum()),
    "fallback_used_any": int(spreads_repaired["fallback_used_any"].sum()),
    "large_fallback_dollar_flag_any": int(spreads_repaired["large_fallback_dollar_flag_any"].sum()),
    "large_fallback_width_pct_flag_any": int(spreads_repaired["large_fallback_width_pct_flag_any"].sum()),
    "avg_entry_credit_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "entry_credit_after_fallback"].mean()),
    "median_entry_credit_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "entry_credit_after_fallback"].median()),
    "avg_return_on_max_loss_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].mean()),
    "median_return_on_max_loss_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].median()),
    "worst_return_on_max_loss_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].min()),
    "win_rate_after_fallback": float(spreads_repaired.loc[spreads_repaired["fallback_pnl_complete"], "trade_win_after_fallback"].mean()),
}])

print()
print("=" * 100)
print("Repair result")
print("=" * 100)
display(summary)

if not repair_log.empty:
    print()
    print("Repair log")
    display(repair_log)

if not bad_width_after.empty:
    print()
    print("Still bad width rows after repair")
    display(
        bad_width_after[
            [
                "selected_trade_id",
                "trade_date",
                "tenor",
                "expiration_date",
                "short_effective_strike",
                "long_effective_strike",
                "effective_width_after_fallback",
            ]
        ]
    )

print()
print("Non-P&L-complete reason summary after repair")
display(non_pnl_reason_summary)

# --------------------------------------------------------------------------------------------------
# Save backups and patched outputs
# --------------------------------------------------------------------------------------------------

leg_backup_path = backup_file(LEG_PRICES_WITH_FALLBACK_PATH, "pre_jointsafe_repair")
spread_backup_path = backup_file(SPREAD_PRICES_WITH_FALLBACK_PATH, "pre_jointsafe_repair")

atomic_write_parquet(legs, LEG_PRICES_WITH_FALLBACK_PATH)
atomic_write_parquet(spreads_repaired, SPREAD_PRICES_WITH_FALLBACK_PATH)

repair_log_path = AUDIT_DIR / f"02G_DATA_4A_joint_fallback_repair_log_{OUT_SUFFIX}_{timestamp}.csv"
summary_path = AUDIT_DIR / f"02G_DATA_4A_joint_fallback_repair_summary_{OUT_SUFFIX}_{timestamp}.csv"
non_pnl_reason_summary_path = AUDIT_DIR / f"02G_DATA_4A_non_pnl_reason_summary_after_repair_{OUT_SUFFIX}_{timestamp}.csv"
bad_width_after_path = AUDIT_DIR / f"02G_DATA_4A_bad_width_after_repair_{OUT_SUFFIX}_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02G_DATA_4A_joint_fallback_repair_{OUT_SUFFIX}_manifest_{timestamp}.json"

atomic_write_csv(repair_log, repair_log_path)
atomic_write_csv(summary, summary_path)
atomic_write_csv(non_pnl_reason_summary, non_pnl_reason_summary_path)
atomic_write_csv(bad_width_after, bad_width_after_path)

manifest = {
    "cell": "2G.DATA.4A",
    "description": "Joint-safe fallback repair for call-spread strike collisions",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "universe_path": str(UNIVERSE_PATH),
        "leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
        "spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "fallback_candidate_plan_path": str(FALLBACK_CANDIDATE_PLAN_PATH),
        "fallback_candidate_cache_path": str(FALLBACK_CANDIDATE_CACHE_PATH),
    },
    "backups": {
        "leg_backup_path": str(leg_backup_path),
        "spread_backup_path": str(spread_backup_path),
    },
    "counts": summary.iloc[0].to_dict(),
    "outputs": {
        "patched_leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
        "patched_spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "repair_log_path": str(repair_log_path),
        "summary_path": str(summary_path),
        "non_pnl_reason_summary_path": str(non_pnl_reason_summary_path),
        "bad_width_after_path": str(bad_width_after_path),
        "manifest_path": str(manifest_path),
    },
    "next_step": "Rerun Cell 2G.DATA.4 mechanical sanity checks.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Leg backup:              {leg_backup_path}")
print(f"Spread backup:           {spread_backup_path}")
print(f"Patched leg prices:      {LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Patched spread prices:   {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Repair log:              {repair_log_path}")
print(f"Summary:                 {summary_path}")
print(f"Manifest:                {manifest_path}")

print()
if bad_width_after.empty:
    print("Result: 2G.DATA.4A repaired all non-positive effective-width collisions.")
    print("Next step: rerun Cell 2G.DATA.4.")
else:
    print("Result: 2G.DATA.4A did not repair all effective-width collisions.")
    print("Do not proceed until these are fixed.")

Cell 2G.DATA.4A — Repair joint fallback strike collisions
Leg prices with fallback:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Spread prices with fallback: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback candidate plan:     C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_plan_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback candidate cache:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_fullhist_1sd3sd_long5_cal_v1.parquet

Loaded inputs
Universe rows:          14,613
Leg rows:               29,226
Spread rows:            14,613
Fallback candidate rows:71,25

,selected_trade_id,trade_date,tenor,expiration_date,short_effective_strike,long_effective_strike,short_bid_final,long_ask_final,entry_credit_after_fallback,effective_width_after_fallback
4382,SPY_CALL_20211206_T33_EXP20220114_S497.0_L575.0,2021-12-06,33,2022-01-14,500.0,500.0,0.28,0.31,-0.03,0.0



Repair result


,bad_width_before,bad_width_after,repair_log_rows,repaired_rows,spread_rows,complete_after_fallback,fallback_pnl_complete,non_pnl_complete_rows,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit_after_fallback,median_entry_credit_after_fallback,avg_return_on_max_loss_after_fallback,median_return_on_max_loss_after_fallback,worst_return_on_max_loss_after_fallback,win_rate_after_fallback
0,1,1,1,0,14613,14613,14598,15,3051,354,298,0.706971,0.65,-0.001906,0.013946,-0.668007,0.880121



Repair log


...


Still bad width rows after repair


,selected_trade_id,trade_date,tenor,expiration_date,short_effective_strike,long_effective_strike,effective_width_after_fallback
4382,SPY_CALL_20211206_T33_EXP20220114_S497.0_L575.0,2021-12-06,33,2022-01-14,500.0,500.0,0.0



Non-P&L-complete reason summary after repair


,non_pnl_reason,rows
0,non_positive_entry_credit,14
1,non_positive_entry_credit;non_positive_effecti...,1



Saved
Leg backup:              C:\Users\patri\vrp_project\data\audit\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1_pre_jointsafe_repair_20260714_193049.parquet
Spread backup:           C:\Users\patri\vrp_project\data\audit\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1_pre_jointsafe_repair_20260714_193049.parquet
Patched leg prices:      C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Patched spread prices:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Repair log:              C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4A_joint_fallback_repair_log_fullhist_1sd3sd_long5_cal_v

In [3]:
# Cell 2G.DATA.4B — Flag unrecoverable joint fallback rows
# Purpose:
#   Do not force non-conservative fallback repairs.
#   Flag any fallback-adjusted spread with non-positive effective width as structurally invalid,
#   keep it for audit, and exclude it from P&L-complete use.
#
# Data layer only:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule

from pathlib import Path
from datetime import datetime
import json
import shutil

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

LEG_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.DATA.4B — Flag unrecoverable joint fallback rows")
print("=" * 100)
print(f"Spread panel: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Leg panel:    {LEG_PRICES_WITH_FALLBACK_PATH}")
print()

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)
    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)
    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)
    if s.dtype == bool:
        return s.fillna(False).astype(bool)
    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

if not SPREAD_PRICES_WITH_FALLBACK_PATH.exists():
    raise FileNotFoundError(SPREAD_PRICES_WITH_FALLBACK_PATH)

if not LEG_PRICES_WITH_FALLBACK_PATH.exists():
    raise FileNotFoundError(LEG_PRICES_WITH_FALLBACK_PATH)

spreads = pd.read_parquet(SPREAD_PRICES_WITH_FALLBACK_PATH).copy()
legs = pd.read_parquet(LEG_PRICES_WITH_FALLBACK_PATH).copy()

for df in [spreads, legs]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "outcome_available",
    "exact_primary_complete",
    "exact_primary_pnl_complete",
    "complete_after_fallback",
    "fallback_pnl_complete",
    "fallback_used_any",
    "large_fallback_dollar_flag_any",
    "large_fallback_width_pct_flag_any",
    "trade_win_after_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "final_leg_price_usable",
    "fallback_used",
    "fallback_found",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
]:
    if c in legs.columns:
        legs[c] = bool_series(legs[c], index=legs.index)

# Add explicit status columns.
spreads["joint_structure_valid_after_fallback"] = (
    spreads["effective_width_after_fallback"].notna()
    & (spreads["effective_width_after_fallback"] > 0)
    & spreads["short_effective_strike"].notna()
    & spreads["long_effective_strike"].notna()
    & (spreads["long_effective_strike"] > spreads["short_effective_strike"])
)

spreads["unrecoverable_joint_fallback"] = (
    spreads["complete_after_fallback"]
    & ~spreads["joint_structure_valid_after_fallback"]
)

# Keep complete_after_fallback as leg-price completeness.
# P&L completeness must require valid joint structure.
spreads["fallback_pnl_complete"] = (
    spreads["complete_after_fallback"]
    & spreads["joint_structure_valid_after_fallback"]
    & spreads["outcome_available"]
    & spreads["entry_credit_after_fallback"].notna()
    & (spreads["entry_credit_after_fallback"] > 0)
    & spreads["max_loss_after_fallback"].notna()
    & (spreads["max_loss_after_fallback"] > 0)
    & spreads["effective_width_after_fallback"].notna()
    & (spreads["effective_width_after_fallback"] > 0)
    & spreads["terminal_spread_intrinsic_after_fallback"].notna()
    & spreads["return_on_max_loss_after_fallback"].notna()
)

def make_exclusion_reason(row):
    reasons = []

    if not bool(row.get("complete_after_fallback", False)):
        reasons.append("missing_final_leg_price")

    if not bool(row.get("joint_structure_valid_after_fallback", False)):
        reasons.append("invalid_joint_structure")

    if not bool(row.get("outcome_available", False)):
        reasons.append("missing_expiry_outcome")

    if pd.isna(row.get("entry_credit_after_fallback")):
        reasons.append("missing_entry_credit")
    elif row.get("entry_credit_after_fallback") <= 0:
        reasons.append("non_positive_entry_credit")

    if pd.isna(row.get("effective_width_after_fallback")):
        reasons.append("missing_effective_width")
    elif row.get("effective_width_after_fallback") <= 0:
        reasons.append("non_positive_effective_width")

    if pd.isna(row.get("max_loss_after_fallback")):
        reasons.append("missing_max_loss")
    elif row.get("max_loss_after_fallback") <= 0:
        reasons.append("non_positive_max_loss")

    if not reasons:
        reasons.append("pnl_complete")

    return ";".join(reasons)

spreads["pnl_exclusion_reason_after_fallback"] = spreads.apply(make_exclusion_reason, axis=1)

invalid = spreads.loc[spreads["unrecoverable_joint_fallback"]].copy()
non_pnl = spreads.loc[~spreads["fallback_pnl_complete"]].copy()

# Add leg-level marker for the affected spread(s).
bad_ids = set(invalid["selected_trade_id"].astype(str))

legs["joint_structure_valid_after_fallback"] = ~legs["selected_trade_id"].astype(str).isin(bad_ids)
legs["unrecoverable_joint_fallback"] = legs["selected_trade_id"].astype(str).isin(bad_ids)

summary = pd.DataFrame([{
    "spread_rows": int(len(spreads)),
    "complete_after_fallback": int(spreads["complete_after_fallback"].sum()),
    "joint_structure_valid_after_fallback": int(spreads["joint_structure_valid_after_fallback"].sum()),
    "unrecoverable_joint_fallback": int(spreads["unrecoverable_joint_fallback"].sum()),
    "fallback_pnl_complete": int(spreads["fallback_pnl_complete"].sum()),
    "non_pnl_complete_rows": int((~spreads["fallback_pnl_complete"]).sum()),
}])

reason_summary = (
    non_pnl["pnl_exclusion_reason_after_fallback"]
    .value_counts(dropna=False)
    .rename_axis("pnl_exclusion_reason_after_fallback")
    .reset_index(name="rows")
)

print("Summary")
display(summary)

print()
print("Unrecoverable joint fallback rows")
if invalid.empty:
    print("None.")
else:
    display(
        invalid[
            [
                "selected_trade_id",
                "trade_date",
                "tenor",
                "expiration_date",
                "short_effective_strike",
                "long_effective_strike",
                "effective_width_after_fallback",
                "entry_credit_after_fallback",
                "fallback_pnl_complete",
                "pnl_exclusion_reason_after_fallback",
            ]
        ]
    )

print()
print("P&L exclusion reason summary")
display(reason_summary)

# Back up current files.
spread_backup_path = AUDIT_DIR / f"{SPREAD_PRICES_WITH_FALLBACK_PATH.stem}_pre_unrecoverable_flag_{timestamp}.parquet"
leg_backup_path = AUDIT_DIR / f"{LEG_PRICES_WITH_FALLBACK_PATH.stem}_pre_unrecoverable_flag_{timestamp}.parquet"

shutil.copy2(SPREAD_PRICES_WITH_FALLBACK_PATH, spread_backup_path)
shutil.copy2(LEG_PRICES_WITH_FALLBACK_PATH, leg_backup_path)

# Save patched canonical files.
atomic_write_parquet(spreads, SPREAD_PRICES_WITH_FALLBACK_PATH)
atomic_write_parquet(legs, LEG_PRICES_WITH_FALLBACK_PATH)

summary_path = AUDIT_DIR / f"02G_DATA_4B_unrecoverable_joint_fallback_summary_{OUT_SUFFIX}_{timestamp}.csv"
invalid_path = AUDIT_DIR / f"02G_DATA_4B_unrecoverable_joint_fallback_rows_{OUT_SUFFIX}_{timestamp}.csv"
reason_summary_path = AUDIT_DIR / f"02G_DATA_4B_pnl_exclusion_reason_summary_{OUT_SUFFIX}_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02G_DATA_4B_unrecoverable_joint_fallback_{OUT_SUFFIX}_manifest_{timestamp}.json"

atomic_write_csv(summary, summary_path)
atomic_write_csv(invalid, invalid_path)
atomic_write_csv(reason_summary, reason_summary_path)

manifest = {
    "cell": "2G.DATA.4B",
    "description": "Flag unrecoverable joint fallback structures instead of forcing non-conservative repairs",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
    },
    "backups": {
        "spread_backup_path": str(spread_backup_path),
        "leg_backup_path": str(leg_backup_path),
    },
    "counts": summary.iloc[0].to_dict(),
    "outputs": {
        "patched_spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "patched_leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
        "summary_path": str(summary_path),
        "invalid_rows_path": str(invalid_path),
        "reason_summary_path": str(reason_summary_path),
        "manifest_path": str(manifest_path),
    },
    "policy": {
        "do_not_force_non_conservative_repair": True,
        "invalid_joint_structure_rows_kept_for_audit": True,
        "invalid_joint_structure_rows_excluded_from_pnl_complete": True,
    },
    "next_step": "Run patched Cell 2G.DATA.4 sanity checks that allow explicitly flagged unrecoverable joint fallback rows.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Spread backup:       {spread_backup_path}")
print(f"Leg backup:          {leg_backup_path}")
print(f"Patched spread file: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Patched leg file:    {LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Summary:             {summary_path}")
print(f"Invalid rows:        {invalid_path}")
print(f"Reason summary:      {reason_summary_path}")
print(f"Manifest:            {manifest_path}")

print()
print("Result: unrecoverable joint fallback rows are now explicitly flagged and excluded from P&L-complete status.")
print("Next step: run the patched 2G.DATA.4 sanity check.")

Cell 2G.DATA.4B — Flag unrecoverable joint fallback rows
Spread panel: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Leg panel:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet

Summary


,spread_rows,complete_after_fallback,joint_structure_valid_after_fallback,unrecoverable_joint_fallback,fallback_pnl_complete,non_pnl_complete_rows
0,14613,14613,14612,1,14598,15



Unrecoverable joint fallback rows


,selected_trade_id,trade_date,tenor,expiration_date,short_effective_strike,long_effective_strike,effective_width_after_fallback,entry_credit_after_fallback,fallback_pnl_complete,pnl_exclusion_reason_after_fallback
4382,SPY_CALL_20211206_T33_EXP20220114_S497.0_L575.0,2021-12-06,33,2022-01-14,500.0,500.0,0.0,-0.03,False,invalid_joint_structure;non_positive_entry_cre...



P&L exclusion reason summary


,pnl_exclusion_reason_after_fallback,rows
0,non_positive_entry_credit,14
1,invalid_joint_structure;non_positive_entry_cre...,1



Saved
Spread backup:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1_pre_unrecoverable_flag_20260714_194049.parquet
Leg backup:          C:\Users\patri\vrp_project\data\audit\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1_pre_unrecoverable_flag_20260714_194049.parquet
Patched spread file: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Patched leg file:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Summary:             C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4B_unrecoverable_joint_fallback_summary_fullhist_1sd3sd_long5_cal_v1_202

In [4]:
# Cell 2G.DATA.4 — PATCHED mechanical sanity checks for full-history 1SD / 3SD call data
# Purpose:
#   Validate the full-history SPY 1SD / 3SD call-spread data panel after exact-primary
#   pricing, conservative fallback recovery, and unrecoverable joint-fallback flagging.
#
# This is a DATA LAYER cell:
#   - no RSI
#   - no z-score
#   - no VRP filter
#   - no signal logic
#   - no selection rule
#
# Important patched rule:
#   A fallback row with non-positive effective width is allowed ONLY IF:
#     - unrecoverable_joint_fallback == True
#     - joint_structure_valid_after_fallback == False
#     - fallback_pnl_complete == False
#   Such rows remain in the panel for audit but are excluded from P&L-complete use.

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"

UNIVERSE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_universe_{OUT_SUFFIX}.parquet"
)

LEG_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

FALLBACK_MAP_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_{OUT_SUFFIX}.parquet"
)

PRIMARY_CONTRACT_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_primary_contract_price_cache_{OUT_SUFFIX}.parquet"
)

FALLBACK_CANDIDATE_CACHE_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_fallback_candidate_contract_price_cache_{OUT_SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
TOL = 1e-10

print("=" * 100)
print("Cell 2G.DATA.4 — PATCHED mechanical sanity checks")
print("=" * 100)
print(f"Universe:                    {UNIVERSE_PATH}")
print(f"Leg prices with fallback:    {LEG_PRICES_WITH_FALLBACK_PATH}")
print(f"Spread prices with fallback: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Fallback map:                {FALLBACK_MAP_PATH}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def near(a, b, tol=TOL):
    return (pd.to_numeric(a, errors="coerce") - pd.to_numeric(b, errors="coerce")).abs() <= tol

def add_check(rows, check_name, severity, passed, failed_count, detail):
    rows.append({
        "check_name": check_name,
        "severity": severity,
        "passed": bool(passed),
        "failed_count": int(failed_count) if pd.notna(failed_count) else np.nan,
        "detail": detail,
    })

def reason_list(row):
    if "pnl_exclusion_reason_after_fallback" in row.index and pd.notna(row["pnl_exclusion_reason_after_fallback"]):
        return row["pnl_exclusion_reason_after_fallback"]

    reasons = []

    if not bool(row.get("complete_after_fallback", False)):
        reasons.append("missing_final_leg_price")

    if not bool(row.get("joint_structure_valid_after_fallback", True)):
        reasons.append("invalid_joint_structure")

    if not bool(row.get("outcome_available", False)):
        reasons.append("missing_expiry_outcome")

    if pd.isna(row.get("entry_credit_after_fallback")):
        reasons.append("missing_entry_credit")
    elif row.get("entry_credit_after_fallback") <= 0:
        reasons.append("non_positive_entry_credit")

    if pd.isna(row.get("effective_width_after_fallback")):
        reasons.append("missing_effective_width")
    elif row.get("effective_width_after_fallback") <= 0:
        reasons.append("non_positive_effective_width")

    if pd.isna(row.get("max_loss_after_fallback")):
        reasons.append("missing_max_loss")
    elif row.get("max_loss_after_fallback") <= 0:
        reasons.append("non_positive_max_loss")

    if pd.isna(row.get("terminal_spread_intrinsic_after_fallback")):
        reasons.append("missing_terminal_intrinsic")

    if pd.isna(row.get("expiry_pnl_after_fallback")):
        reasons.append("missing_expiry_pnl")

    if pd.isna(row.get("return_on_max_loss_after_fallback")):
        reasons.append("missing_return_on_max_loss")

    if not reasons:
        reasons.append("unknown")

    return ";".join(reasons)

# --------------------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------------------

for p in [
    UNIVERSE_PATH,
    LEG_PRICES_WITH_FALLBACK_PATH,
    SPREAD_PRICES_WITH_FALLBACK_PATH,
    FALLBACK_MAP_PATH,
]:
    require_file(p)

universe = pd.read_parquet(UNIVERSE_PATH).copy()
legs = pd.read_parquet(LEG_PRICES_WITH_FALLBACK_PATH).copy()
spreads = pd.read_parquet(SPREAD_PRICES_WITH_FALLBACK_PATH).copy()
fallback_map = pd.read_parquet(FALLBACK_MAP_PATH).copy()

primary_cache = pd.read_parquet(PRIMARY_CONTRACT_CACHE_PATH).copy() if PRIMARY_CONTRACT_CACHE_PATH.exists() else pd.DataFrame()
fallback_cache = pd.read_parquet(FALLBACK_CANDIDATE_CACHE_PATH).copy() if FALLBACK_CANDIDATE_CACHE_PATH.exists() else pd.DataFrame()

for df in [universe, legs, spreads, fallback_map, primary_cache, fallback_cache]:
    for c in ["trade_date", "expiration_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

for c in [
    "leg_price_usable",
    "primary_leg_price_usable",
    "fallback_used",
    "fallback_found",
    "final_leg_price_usable",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
    "joint_structure_valid_after_fallback",
    "unrecoverable_joint_fallback",
]:
    if c in legs.columns:
        legs[c] = bool_series(legs[c], index=legs.index)

for c in [
    "outcome_available",
    "exact_primary_complete",
    "exact_primary_pnl_complete",
    "complete_after_fallback",
    "fallback_pnl_complete",
    "fallback_used_any",
    "large_fallback_dollar_flag_any",
    "large_fallback_width_pct_flag_any",
    "trade_win_after_fallback",
    "joint_structure_valid_after_fallback",
    "unrecoverable_joint_fallback",
]:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "fallback_found",
    "fallback_direction_valid",
    "large_fallback_dollar_flag",
    "large_fallback_width_pct_flag",
]:
    if c in fallback_map.columns:
        fallback_map[c] = bool_series(fallback_map[c], index=fallback_map.index)

# Required patched columns.
if "joint_structure_valid_after_fallback" not in spreads.columns:
    spreads["joint_structure_valid_after_fallback"] = (
        spreads["effective_width_after_fallback"].notna()
        & (spreads["effective_width_after_fallback"] > 0)
        & spreads["short_effective_strike"].notna()
        & spreads["long_effective_strike"].notna()
        & (spreads["long_effective_strike"] > spreads["short_effective_strike"])
    )

if "unrecoverable_joint_fallback" not in spreads.columns:
    spreads["unrecoverable_joint_fallback"] = (
        spreads["complete_after_fallback"]
        & ~spreads["joint_structure_valid_after_fallback"]
    )

print("=" * 100)
print("Loaded inputs")
print("=" * 100)
print(f"Universe rows:             {len(universe):,}")
print(f"Leg rows:                  {len(legs):,}")
print(f"Spread rows:               {len(spreads):,}")
print(f"Fallback map rows:         {len(fallback_map):,}")
print(f"Primary cache rows:        {len(primary_cache):,}")
print(f"Fallback cache rows:       {len(fallback_cache):,}")
print()

# --------------------------------------------------------------------------------------------------
# Derived diagnostics
# --------------------------------------------------------------------------------------------------

leg_counts = (
    legs
    .groupby("selected_trade_id", dropna=False)
    .agg(
        leg_count=("leg_label", "size"),
        short_legs=("leg_label", lambda x: int((x == "short_call_1sd").sum())),
        long_legs=("leg_label", lambda x: int((x == "long_call_3sd").sum())),
        final_usable_legs=("final_leg_price_usable", "sum"),
        fallback_used_legs=("fallback_used", "sum"),
    )
    .reset_index()
)

calc = spreads.copy()

calc["calc_effective_width"] = calc["long_effective_strike"] - calc["short_effective_strike"]
calc["calc_entry_credit"] = calc["short_bid_final"] - calc["long_ask_final"]
calc["calc_max_loss"] = calc["calc_effective_width"] - calc["calc_entry_credit"]

calc["calc_terminal_intrinsic"] = (
    np.maximum(calc["expiry_spy_close"] - calc["short_effective_strike"], 0.0)
    - np.maximum(calc["expiry_spy_close"] - calc["long_effective_strike"], 0.0)
)

calc.loc[~calc["outcome_available"], "calc_terminal_intrinsic"] = np.nan

calc["calc_expiry_pnl"] = calc["calc_entry_credit"] - calc["calc_terminal_intrinsic"]
calc["calc_return_on_max_loss"] = calc["calc_expiry_pnl"] / calc["calc_max_loss"]
calc["calc_trade_win"] = calc["calc_expiry_pnl"] > 0

non_pnl = calc.loc[~calc["fallback_pnl_complete"]].copy()
if not non_pnl.empty:
    non_pnl["non_pnl_reason"] = non_pnl.apply(reason_list, axis=1)
else:
    non_pnl["non_pnl_reason"] = pd.Series(dtype=str)

non_pnl_reason_summary = (
    non_pnl["non_pnl_reason"]
    .value_counts(dropna=False)
    .rename_axis("non_pnl_reason")
    .reset_index(name="rows")
)

fallback_used_legs = legs.loc[legs["fallback_used"]].copy()

short_fb = fallback_used_legs.loc[fallback_used_legs["leg_label"].eq("short_call_1sd")].copy()
long_fb = fallback_used_legs.loc[fallback_used_legs["leg_label"].eq("long_call_3sd")].copy()

short_direction_bad = short_fb.loc[
    ~(
        (short_fb["effective_strike"] >= short_fb["target_strike"])
        & (short_fb["effective_strike"] < short_fb["long_strike"])
    )
].copy()

long_direction_bad = long_fb.loc[
    ~(
        (long_fb["effective_strike"] <= long_fb["target_strike"])
        & (long_fb["effective_strike"] > long_fb["short_strike"])
    )
].copy()

exact_primary_legs = legs.loc[legs["final_price_source"].eq("exact_primary")].copy()

exact_primary_strike_changed = exact_primary_legs.loc[
    ~near(exact_primary_legs["effective_strike"], exact_primary_legs["target_strike"])
].copy()

fallback_map_bad_direction = fallback_map.loc[
    fallback_map["fallback_found"] & ~fallback_map["fallback_direction_valid"]
].copy()

# Patched structural categories.
invalid_joint_rows = calc.loc[~calc["joint_structure_valid_after_fallback"]].copy()

unflagged_invalid_joint_rows = invalid_joint_rows.loc[
    ~invalid_joint_rows["unrecoverable_joint_fallback"]
].copy()

flagged_invalid_but_pnl_complete = calc.loc[
    calc["unrecoverable_joint_fallback"]
    & calc["fallback_pnl_complete"]
].copy()

pnl_complete = calc.loc[calc["fallback_pnl_complete"]].copy()

pnl_complete_bad_width = pnl_complete.loc[
    pnl_complete["effective_width_after_fallback"] <= 0
].copy()

pnl_complete_bad_joint_structure = pnl_complete.loc[
    ~pnl_complete["joint_structure_valid_after_fallback"]
].copy()

pnl_complete_non_positive_credit = pnl_complete.loc[
    pnl_complete["entry_credit_after_fallback"] <= 0
].copy()

pnl_complete_non_positive_max_loss = pnl_complete.loc[
    pnl_complete["max_loss_after_fallback"] <= 0
].copy()

large_fallback_spreads = spreads.loc[
    spreads["large_fallback_dollar_flag_any"]
    | spreads["large_fallback_width_pct_flag_any"]
].copy()

fallback_slippage_by_leg = (
    fallback_used_legs
    .groupby("leg_label", dropna=False)
    .agg(
        fallback_used=("selected_trade_id", "size"),
        avg_abs_slip=("abs_strike_slip", "mean"),
        median_abs_slip=("abs_strike_slip", "median"),
        max_abs_slip=("abs_strike_slip", "max"),
        avg_slip_pct_original_width=("slip_pct_original_width", "mean"),
        median_slip_pct_original_width=("slip_pct_original_width", "median"),
        max_slip_pct_original_width=("slip_pct_original_width", "max"),
        large_dollar_flags=("large_fallback_dollar_flag", "sum"),
        large_width_pct_flags=("large_fallback_width_pct_flag", "sum"),
    )
    .reset_index()
)

global_summary = pd.DataFrame([{
    "universe_rows": int(len(universe)),
    "spread_rows": int(len(spreads)),
    "leg_rows": int(len(legs)),
    "fallback_map_rows": int(len(fallback_map)),
    "primary_cache_rows": int(len(primary_cache)),
    "fallback_cache_rows": int(len(fallback_cache)),
    "exact_primary_complete_spreads": int(spreads["exact_primary_complete"].sum()),
    "complete_after_fallback": int(spreads["complete_after_fallback"].sum()),
    "joint_structure_valid_after_fallback": int(spreads["joint_structure_valid_after_fallback"].sum()),
    "unrecoverable_joint_fallback": int(spreads["unrecoverable_joint_fallback"].sum()),
    "fallback_pnl_complete": int(spreads["fallback_pnl_complete"].sum()),
    "non_pnl_complete_rows": int((~spreads["fallback_pnl_complete"]).sum()),
    "fallback_used_any": int(spreads["fallback_used_any"].sum()),
    "large_fallback_dollar_flag_any": int(spreads["large_fallback_dollar_flag_any"].sum()),
    "large_fallback_width_pct_flag_any": int(spreads["large_fallback_width_pct_flag_any"].sum()),
    "avg_entry_credit_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "entry_credit_after_fallback"].mean()),
    "median_entry_credit_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "entry_credit_after_fallback"].median()),
    "avg_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].mean()),
    "median_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].median()),
    "worst_return_on_max_loss_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "return_on_max_loss_after_fallback"].min()),
    "win_rate_after_fallback": float(spreads.loc[spreads["fallback_pnl_complete"], "trade_win_after_fallback"].mean()),
}])

coverage_by_tenor = (
    spreads
    .groupby("tenor", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        exact_primary_complete=("exact_primary_complete", "sum"),
        complete_after_fallback=("complete_after_fallback", "sum"),
        joint_structure_valid_after_fallback=("joint_structure_valid_after_fallback", "sum"),
        unrecoverable_joint_fallback=("unrecoverable_joint_fallback", "sum"),
        fallback_pnl_complete=("fallback_pnl_complete", "sum"),
        non_pnl_complete=("fallback_pnl_complete", lambda x: int((~x).sum())),
        fallback_used_any=("fallback_used_any", "sum"),
        large_fallback_dollar_flag_any=("large_fallback_dollar_flag_any", "sum"),
        large_fallback_width_pct_flag_any=("large_fallback_width_pct_flag_any", "sum"),
        avg_entry_credit=("entry_credit_after_fallback", "mean"),
        median_entry_credit=("entry_credit_after_fallback", "median"),
        avg_return=("return_on_max_loss_after_fallback", "mean"),
        median_return=("return_on_max_loss_after_fallback", "median"),
        worst_return=("return_on_max_loss_after_fallback", "min"),
        win_rate=("trade_win_after_fallback", "mean"),
    )
    .reset_index()
)

coverage_by_year = (
    spreads
    .groupby("trade_year", dropna=False)
    .agg(
        spread_rows=("selected_trade_id", "size"),
        exact_primary_complete=("exact_primary_complete", "sum"),
        complete_after_fallback=("complete_after_fallback", "sum"),
        joint_structure_valid_after_fallback=("joint_structure_valid_after_fallback", "sum"),
        unrecoverable_joint_fallback=("unrecoverable_joint_fallback", "sum"),
        fallback_pnl_complete=("fallback_pnl_complete", "sum"),
        non_pnl_complete=("fallback_pnl_complete", lambda x: int((~x).sum())),
        fallback_used_any=("fallback_used_any", "sum"),
        large_fallback_dollar_flag_any=("large_fallback_dollar_flag_any", "sum"),
        large_fallback_width_pct_flag_any=("large_fallback_width_pct_flag_any", "sum"),
        avg_entry_credit=("entry_credit_after_fallback", "mean"),
        median_entry_credit=("entry_credit_after_fallback", "median"),
        avg_return=("return_on_max_loss_after_fallback", "mean"),
        median_return=("return_on_max_loss_after_fallback", "median"),
        worst_return=("return_on_max_loss_after_fallback", "min"),
        win_rate=("trade_win_after_fallback", "mean"),
    )
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Hard and soft checks
# --------------------------------------------------------------------------------------------------

checks = []

add_check(
    checks,
    "spread row count equals universe row count",
    "hard",
    len(spreads) == len(universe),
    abs(len(spreads) - len(universe)),
    f"spreads={len(spreads):,}; universe={len(universe):,}",
)

add_check(
    checks,
    "unique selected_trade_id in universe",
    "hard",
    universe["selected_trade_id"].is_unique,
    int(universe["selected_trade_id"].duplicated().sum()),
    "Universe should have one row per trade_date × tenor spread candidate.",
)

add_check(
    checks,
    "unique selected_trade_id in spread panel",
    "hard",
    spreads["selected_trade_id"].is_unique,
    int(spreads["selected_trade_id"].duplicated().sum()),
    "Spread panel should have one row per spread candidate.",
)

missing_spreads_from_universe = set(universe["selected_trade_id"]) - set(spreads["selected_trade_id"])
extra_spreads_not_in_universe = set(spreads["selected_trade_id"]) - set(universe["selected_trade_id"])

add_check(
    checks,
    "spread panel selected_trade_id set matches universe",
    "hard",
    len(missing_spreads_from_universe) == 0 and len(extra_spreads_not_in_universe) == 0,
    len(missing_spreads_from_universe) + len(extra_spreads_not_in_universe),
    f"missing_from_spreads={len(missing_spreads_from_universe):,}; extra_in_spreads={len(extra_spreads_not_in_universe):,}",
)

bad_leg_counts = leg_counts.loc[
    (leg_counts["leg_count"] != 2)
    | (leg_counts["short_legs"] != 1)
    | (leg_counts["long_legs"] != 1)
].copy()

add_check(
    checks,
    "exactly one short leg and one long leg per spread",
    "hard",
    bad_leg_counts.empty and len(leg_counts) == len(spreads),
    int(len(bad_leg_counts) + abs(len(leg_counts) - len(spreads))),
    f"leg_count_rows={len(leg_counts):,}; spread_rows={len(spreads):,}; bad_leg_count_rows={len(bad_leg_counts):,}",
)

missing_final_leg_prices = legs.loc[~legs["final_leg_price_usable"]].copy()

add_check(
    checks,
    "all final legs have needed bid or ask after fallback",
    "hard",
    missing_final_leg_prices.empty,
    int(len(missing_final_leg_prices)),
    "Every short leg needs usable bid; every long leg needs usable ask.",
)

incomplete_after_fallback = spreads.loc[~spreads["complete_after_fallback"]].copy()

add_check(
    checks,
    "all spreads have complete final leg prices after fallback",
    "hard",
    incomplete_after_fallback.empty,
    int(len(incomplete_after_fallback)),
    "Both legs should be usable after fallback recovery.",
)

add_check(
    checks,
    "short fallback direction valid",
    "hard",
    short_direction_bad.empty,
    int(len(short_direction_bad)),
    "Short fallback must be at/above target short strike and below long target strike.",
)

add_check(
    checks,
    "long fallback direction valid",
    "hard",
    long_direction_bad.empty,
    int(len(long_direction_bad)),
    "Long fallback must be at/below target long strike and above short target strike.",
)

add_check(
    checks,
    "exact-primary legs retain target strike",
    "hard",
    exact_primary_strike_changed.empty,
    int(len(exact_primary_strike_changed)),
    "Exact-primary legs should not change effective strike.",
)

add_check(
    checks,
    "fallback map directions valid",
    "hard",
    fallback_map_bad_direction.empty,
    int(len(fallback_map_bad_direction)),
    "Fallback map should contain only valid conservative fallback directions.",
)

add_check(
    checks,
    "invalid joint structures are explicitly flagged unrecoverable",
    "hard",
    unflagged_invalid_joint_rows.empty,
    int(len(unflagged_invalid_joint_rows)),
    "Any long_effective <= short_effective row must be marked unrecoverable_joint_fallback.",
)

add_check(
    checks,
    "unrecoverable joint fallback rows are excluded from P&L-complete rows",
    "hard",
    flagged_invalid_but_pnl_complete.empty,
    int(len(flagged_invalid_but_pnl_complete)),
    "Unrecoverable joint structures must remain audit rows only, not P&L-complete rows.",
)

add_check(
    checks,
    "all P&L-complete rows have positive effective width",
    "hard",
    pnl_complete_bad_width.empty,
    int(len(pnl_complete_bad_width)),
    "P&L-complete rows must have long_effective_strike > short_effective_strike.",
)

add_check(
    checks,
    "all P&L-complete rows have valid joint structure",
    "hard",
    pnl_complete_bad_joint_structure.empty,
    int(len(pnl_complete_bad_joint_structure)),
    "P&L-complete rows must have joint_structure_valid_after_fallback=True.",
)

bad_width_formula = calc.loc[
    calc["effective_width_after_fallback"].notna()
    & ~near(calc["effective_width_after_fallback"], calc["calc_effective_width"])
].copy()

add_check(
    checks,
    "effective width formula reconciles",
    "hard",
    bad_width_formula.empty,
    int(len(bad_width_formula)),
    "effective_width = long_effective_strike - short_effective_strike.",
)

bad_credit_formula = calc.loc[
    calc["entry_credit_after_fallback"].notna()
    & ~near(calc["entry_credit_after_fallback"], calc["calc_entry_credit"])
].copy()

add_check(
    checks,
    "entry credit formula reconciles",
    "hard",
    bad_credit_formula.empty,
    int(len(bad_credit_formula)),
    "entry_credit = short_bid_final - long_ask_final.",
)

bad_max_loss_formula = calc.loc[
    calc["max_loss_after_fallback"].notna()
    & ~near(calc["max_loss_after_fallback"], calc["calc_max_loss"])
].copy()

add_check(
    checks,
    "max loss formula reconciles",
    "hard",
    bad_max_loss_formula.empty,
    int(len(bad_max_loss_formula)),
    "max_loss = effective_width - entry_credit.",
)

bad_intrinsic_formula = calc.loc[
    calc["terminal_spread_intrinsic_after_fallback"].notna()
    & ~near(calc["terminal_spread_intrinsic_after_fallback"], calc["calc_terminal_intrinsic"])
].copy()

add_check(
    checks,
    "terminal intrinsic formula reconciles",
    "hard",
    bad_intrinsic_formula.empty,
    int(len(bad_intrinsic_formula)),
    "terminal_intrinsic = max(expiry - short, 0) - max(expiry - long, 0).",
)

bad_intrinsic_bounds = pnl_complete.loc[
    pnl_complete["terminal_spread_intrinsic_after_fallback"].notna()
    & (
        (pnl_complete["terminal_spread_intrinsic_after_fallback"] < -TOL)
        | (pnl_complete["terminal_spread_intrinsic_after_fallback"] - pnl_complete["effective_width_after_fallback"] > TOL)
    )
].copy()

add_check(
    checks,
    "terminal intrinsic bounded between zero and width for P&L-complete rows",
    "hard",
    bad_intrinsic_bounds.empty,
    int(len(bad_intrinsic_bounds)),
    "Terminal intrinsic should be in [0, effective_width] for usable rows.",
)

bad_pnl_formula = calc.loc[
    calc["expiry_pnl_after_fallback"].notna()
    & ~near(calc["expiry_pnl_after_fallback"], calc["calc_expiry_pnl"])
].copy()

add_check(
    checks,
    "expiry P&L formula reconciles",
    "hard",
    bad_pnl_formula.empty,
    int(len(bad_pnl_formula)),
    "expiry_pnl = entry_credit - terminal_intrinsic.",
)

bad_return_formula = calc.loc[
    calc["return_on_max_loss_after_fallback"].notna()
    & ~near(calc["return_on_max_loss_after_fallback"], calc["calc_return_on_max_loss"])
].copy()

add_check(
    checks,
    "return on max loss formula reconciles",
    "hard",
    bad_return_formula.empty,
    int(len(bad_return_formula)),
    "return = expiry_pnl / max_loss.",
)

bad_win_formula = calc.loc[
    calc["fallback_pnl_complete"]
    & (calc["trade_win_after_fallback"] != calc["calc_trade_win"])
].copy()

add_check(
    checks,
    "win flag formula reconciles",
    "hard",
    bad_win_formula.empty,
    int(len(bad_win_formula)),
    "trade_win = expiry_pnl > 0.",
)

add_check(
    checks,
    "non-P&L-complete rows are explainable",
    "hard",
    non_pnl.empty or (
        non_pnl["non_pnl_reason"].notna().all()
        and ~non_pnl["non_pnl_reason"].eq("unknown").any()
    ),
    int(non_pnl["non_pnl_reason"].eq("unknown").sum()) if not non_pnl.empty else 0,
    f"non_pnl_complete_rows={len(non_pnl):,}",
)

add_check(
    checks,
    "all P&L-complete rows have positive entry credit",
    "hard",
    pnl_complete_non_positive_credit.empty,
    int(len(pnl_complete_non_positive_credit)),
    "P&L-complete rows must have entry_credit > 0.",
)

add_check(
    checks,
    "all P&L-complete rows have positive max loss",
    "hard",
    pnl_complete_non_positive_max_loss.empty,
    int(len(pnl_complete_non_positive_max_loss)),
    "P&L-complete rows must have max_loss > 0.",
)

# Soft checks / audit flags.
add_check(
    checks,
    "non-P&L-complete rows exist and are excluded",
    "soft",
    len(non_pnl) > 0,
    0 if len(non_pnl) > 0 else 1,
    f"non_pnl_complete_rows={len(non_pnl):,}; these rows should not be used in P&L diagnostics.",
)

add_check(
    checks,
    "large fallback flags are present for review",
    "soft",
    len(large_fallback_spreads) > 0,
    0 if len(large_fallback_spreads) > 0 else 1,
    f"large_fallback_spreads={len(large_fallback_spreads):,}; flags are report-only, not exclusions.",
)

checks_df = pd.DataFrame(checks)

hard_failed = checks_df.loc[
    checks_df["severity"].eq("hard")
    & ~checks_df["passed"]
].copy()

soft_failed = checks_df.loc[
    checks_df["severity"].eq("soft")
    & ~checks_df["passed"]
].copy()

# --------------------------------------------------------------------------------------------------
# Save audits
# --------------------------------------------------------------------------------------------------

checks_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_mechanical_sanity_checks_{OUT_SUFFIX}_{timestamp}.csv"
global_summary_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_global_summary_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_tenor_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_coverage_by_tenor_{OUT_SUFFIX}_{timestamp}.csv"
coverage_by_year_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_coverage_by_year_{OUT_SUFFIX}_{timestamp}.csv"
fallback_slippage_by_leg_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_fallback_slippage_by_leg_{OUT_SUFFIX}_{timestamp}.csv"
non_pnl_reason_summary_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_non_pnl_reason_summary_{OUT_SUFFIX}_{timestamp}.csv"
non_pnl_preview_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_non_pnl_preview_{OUT_SUFFIX}_{timestamp}.csv"
large_fallback_preview_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_large_fallback_preview_{OUT_SUFFIX}_{timestamp}.csv"
hard_failed_preview_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_hard_failed_preview_{OUT_SUFFIX}_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02G_DATA_4_PATCHED_mechanical_sanity_{OUT_SUFFIX}_manifest_{timestamp}.json"

atomic_write_csv(checks_df, checks_path)
atomic_write_csv(global_summary, global_summary_path)
atomic_write_csv(coverage_by_tenor, coverage_by_tenor_path)
atomic_write_csv(coverage_by_year, coverage_by_year_path)
atomic_write_csv(fallback_slippage_by_leg, fallback_slippage_by_leg_path)
atomic_write_csv(non_pnl_reason_summary, non_pnl_reason_summary_path)
atomic_write_csv(non_pnl.head(1000), non_pnl_preview_path)
atomic_write_csv(large_fallback_spreads.head(1000), large_fallback_preview_path)

failed_preview_frames = []

for name, df in [
    ("bad_leg_counts", bad_leg_counts),
    ("missing_final_leg_prices", missing_final_leg_prices),
    ("incomplete_after_fallback", incomplete_after_fallback),
    ("short_direction_bad", short_direction_bad),
    ("long_direction_bad", long_direction_bad),
    ("fallback_map_bad_direction", fallback_map_bad_direction),
    ("unflagged_invalid_joint_rows", unflagged_invalid_joint_rows),
    ("flagged_invalid_but_pnl_complete", flagged_invalid_but_pnl_complete),
    ("pnl_complete_bad_width", pnl_complete_bad_width),
    ("pnl_complete_bad_joint_structure", pnl_complete_bad_joint_structure),
    ("bad_width_formula", bad_width_formula),
    ("bad_credit_formula", bad_credit_formula),
    ("bad_max_loss_formula", bad_max_loss_formula),
    ("bad_intrinsic_formula", bad_intrinsic_formula),
    ("bad_intrinsic_bounds", bad_intrinsic_bounds),
    ("bad_pnl_formula", bad_pnl_formula),
    ("bad_return_formula", bad_return_formula),
    ("bad_win_formula", bad_win_formula),
    ("pnl_complete_non_positive_credit", pnl_complete_non_positive_credit),
    ("pnl_complete_non_positive_max_loss", pnl_complete_non_positive_max_loss),
]:
    if len(df):
        tmp = df.head(100).copy()
        tmp.insert(0, "failed_preview_source", name)
        failed_preview_frames.append(tmp)

if failed_preview_frames:
    hard_failed_preview = pd.concat(failed_preview_frames, ignore_index=True, sort=False)
else:
    hard_failed_preview = pd.DataFrame({"failed_preview_source": []})

atomic_write_csv(hard_failed_preview, hard_failed_preview_path)

manifest = {
    "cell": "2G.DATA.4_PATCHED",
    "description": "Patched mechanical sanity checks allowing explicitly flagged unrecoverable joint fallback rows",
    "created_at": timestamp,
    "out_suffix": OUT_SUFFIX,
    "inputs": {
        "universe_path": str(UNIVERSE_PATH),
        "leg_prices_with_fallback_path": str(LEG_PRICES_WITH_FALLBACK_PATH),
        "spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "fallback_map_path": str(FALLBACK_MAP_PATH),
        "primary_contract_cache_path": str(PRIMARY_CONTRACT_CACHE_PATH),
        "fallback_candidate_cache_path": str(FALLBACK_CANDIDATE_CACHE_PATH),
    },
    "counts": {
        "universe_rows": int(len(universe)),
        "spread_rows": int(len(spreads)),
        "leg_rows": int(len(legs)),
        "fallback_map_rows": int(len(fallback_map)),
        "hard_checks": int(checks_df["severity"].eq("hard").sum()),
        "hard_failed": int(len(hard_failed)),
        "soft_checks": int(checks_df["severity"].eq("soft").sum()),
        "soft_failed": int(len(soft_failed)),
        "exact_primary_complete_spreads": int(spreads["exact_primary_complete"].sum()),
        "complete_after_fallback": int(spreads["complete_after_fallback"].sum()),
        "joint_structure_valid_after_fallback": int(spreads["joint_structure_valid_after_fallback"].sum()),
        "unrecoverable_joint_fallback": int(spreads["unrecoverable_joint_fallback"].sum()),
        "fallback_pnl_complete": int(spreads["fallback_pnl_complete"].sum()),
        "non_pnl_complete_rows": int((~spreads["fallback_pnl_complete"]).sum()),
        "fallback_used_any": int(spreads["fallback_used_any"].sum()),
        "large_fallback_dollar_flag_any": int(spreads["large_fallback_dollar_flag_any"].sum()),
        "large_fallback_width_pct_flag_any": int(spreads["large_fallback_width_pct_flag_any"].sum()),
    },
    "policy": {
        "invalid_joint_structure_allowed_only_when_flagged_unrecoverable": True,
        "unrecoverable_joint_fallback_excluded_from_pnl_complete": True,
        "non_positive_credit_excluded_from_pnl_complete": True,
    },
    "outputs": {
        "checks_path": str(checks_path),
        "global_summary_path": str(global_summary_path),
        "coverage_by_tenor_path": str(coverage_by_tenor_path),
        "coverage_by_year_path": str(coverage_by_year_path),
        "fallback_slippage_by_leg_path": str(fallback_slippage_by_leg_path),
        "non_pnl_reason_summary_path": str(non_pnl_reason_summary_path),
        "non_pnl_preview_path": str(non_pnl_preview_path),
        "large_fallback_preview_path": str(large_fallback_preview_path),
        "hard_failed_preview_path": str(hard_failed_preview_path),
        "manifest_path": str(manifest_path),
    },
    "status": "PASS" if len(hard_failed) == 0 else "FAIL",
    "next_step": "Run signal diagnostics from the fallback-adjusted full-history data panel if hard checks pass.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DATA.4 PATCHED complete — mechanical sanity checks")
print("=" * 100)

print()
print("Global summary")
display(global_summary)

print()
print("=" * 100)
print("Sanity checks")
print("=" * 100)
display(checks_df)

print()
print("=" * 100)
print("Hard check failures")
print("=" * 100)
if hard_failed.empty:
    print("No hard check failures.")
else:
    display(hard_failed)

print()
print("=" * 100)
print("Soft check failures")
print("=" * 100)
if soft_failed.empty:
    print("No soft check failures.")
else:
    display(soft_failed)

print()
print("=" * 100)
print("Non-P&L-complete reason summary")
print("=" * 100)
if non_pnl_reason_summary.empty:
    print("No non-P&L-complete rows.")
else:
    display(non_pnl_reason_summary)

print()
print("=" * 100)
print("Non-P&L-complete preview")
print("=" * 100)
if non_pnl.empty:
    print("No non-P&L-complete rows.")
else:
    preview_cols = [
        "selected_trade_id",
        "trade_date",
        "tenor",
        "expiration_date",
        "short_effective_strike",
        "long_effective_strike",
        "short_bid_final",
        "long_ask_final",
        "entry_credit_after_fallback",
        "effective_width_after_fallback",
        "max_loss_after_fallback",
        "joint_structure_valid_after_fallback",
        "unrecoverable_joint_fallback",
        "outcome_available",
        "complete_after_fallback",
        "non_pnl_reason",
    ]
    display(non_pnl[[c for c in preview_cols if c in non_pnl.columns]].head(100))

print()
print("=" * 100)
print("Fallback slippage by leg")
print("=" * 100)
display(fallback_slippage_by_leg)

print()
print("=" * 100)
print("Coverage by tenor")
print("=" * 100)
display(coverage_by_tenor)

print()
print("=" * 100)
print("Coverage by year")
print("=" * 100)
display(coverage_by_year)

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Checks:                    {checks_path}")
print(f"Global summary:            {global_summary_path}")
print(f"Coverage by tenor:         {coverage_by_tenor_path}")
print(f"Coverage by year:          {coverage_by_year_path}")
print(f"Fallback slippage by leg:  {fallback_slippage_by_leg_path}")
print(f"Non-P&L reason summary:    {non_pnl_reason_summary_path}")
print(f"Non-P&L preview:           {non_pnl_preview_path}")
print(f"Large fallback preview:    {large_fallback_preview_path}")
print(f"Hard failed preview:       {hard_failed_preview_path}")
print(f"Manifest:                  {manifest_path}")

print()
if hard_failed.empty:
    print("Result: 2G.DATA.4 PATCHED PASSED hard mechanical sanity checks.")
    print("Next step: run fixed-parameter DTE diagnostics from the full fallback-adjusted panel.")
else:
    print("Result: 2G.DATA.4 PATCHED FAILED hard mechanical sanity checks.")
    print("Do not use this panel for signal diagnostics until the hard failures are repaired.")

Cell 2G.DATA.4 — PATCHED mechanical sanity checks
Universe:                    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_universe_fullhist_1sd3sd_long5_cal_v1.parquet
Leg prices with fallback:    C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_leg_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Spread prices with fallback: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Fallback map:                C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_fallback_map_fullhist_1sd3sd_long5_cal_v1.parquet

Loaded inputs
Universe rows:             14,613
Leg rows:                  29,226
Spread rows:               14,613
Fallback map rows:         3,339
Primary cache rows:        28,523
Fal

,universe_rows,spread_rows,leg_rows,fallback_map_rows,primary_cache_rows,fallback_cache_rows,exact_primary_complete_spreads,complete_after_fallback,joint_structure_valid_after_fallback,unrecoverable_joint_fallback,...,non_pnl_complete_rows,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit_after_fallback,median_entry_credit_after_fallback,avg_return_on_max_loss_after_fallback,median_return_on_max_loss_after_fallback,worst_return_on_max_loss_after_fallback,win_rate_after_fallback
0,14613,14613,29226,3339,28523,50891,11562,14613,14612,1,...,15,3051,354,298,0.706971,0.65,-0.001906,0.013946,-0.668007,0.880121



Sanity checks


,check_name,severity,passed,failed_count,detail
0,spread row count equals universe row count,hard,True,0,"spreads=14,613; universe=14,613"
1,unique selected_trade_id in universe,hard,True,0,Universe should have one row per trade_date × ...
2,unique selected_trade_id in spread panel,hard,True,0,Spread panel should have one row per spread ca...
3,spread panel selected_trade_id set matches uni...,hard,True,0,missing_from_spreads=0; extra_in_spreads=0
4,exactly one short leg and one long leg per spread,hard,True,0,"leg_count_rows=14,613; spread_rows=14,613; bad..."
5,all final legs have needed bid or ask after fa...,hard,True,0,Every short leg needs usable bid; every long l...
6,all spreads have complete final leg prices aft...,hard,True,0,Both legs should be usable after fallback reco...
7,short fallback direction valid,hard,True,0,Short fallback must be at/above target short s...
8,long fallback direction valid,hard,True,0,Long fallback must be at/below target long str...
9,exact-primary legs retain target strike,hard,True,0,Exact-primary legs should not change effective...



Hard check failures
No hard check failures.

Soft check failures
No soft check failures.

Non-P&L-complete reason summary


...


Coverage by tenor


,tenor,spread_rows,exact_primary_complete,complete_after_fallback,joint_structure_valid_after_fallback,unrecoverable_joint_fallback,fallback_pnl_complete,non_pnl_complete,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit,median_entry_credit,avg_return,median_return,worst_return,win_rate
0,9,1632,1589,1632,1632,0,1631,1,43,0,0,0.699969,0.620,-0.002720,0.020055,-0.668007,0.853554
1,12,1629,1544,1629,1629,0,1629,0,85,0,0,0.769724,0.730,-0.003446,0.020408,-0.667509,0.859423
2,15,1628,1497,1628,1628,0,1628,0,131,2,1,0.646947,0.600,-0.001916,0.014885,-0.591667,0.877150
3,18,1625,1398,1625,1625,0,1623,2,227,14,12,0.726486,0.680,-0.002413,0.015228,-0.555219,0.878769
4,21,1624,1317,1624,1624,0,1622,2,307,28,18,0.628073,0.590,-0.001631,0.012241,-0.481988,0.895320
5,24,1622,1214,1622,1622,0,1621,1,408,38,33,0.708354,0.640,-0.002227,0.012889,-0.515539,0.888409
6,27,1620,1117,1620,1620,0,1618,2,503,54,44,0.698630,0.645,-0.001693,0.011941,-0.477369,0.887654
7,30,1618,989,1618,1618,0,1616,2,629,80,72,0.705828,0.650,-0.001232,0.011463,-0.458612,0.887515
8,33,1615,897,1615,1614,1,1610,5,718,138,118,0.765455,0.710,-0.000572,0.012282,-1.000000,0.885449



Coverage by year


,trade_year,spread_rows,exact_primary_complete,complete_after_fallback,joint_structure_valid_after_fallback,unrecoverable_joint_fallback,fallback_pnl_complete,non_pnl_complete,fallback_used_any,large_fallback_dollar_flag_any,large_fallback_width_pct_flag_any,avg_entry_credit,median_entry_credit,avg_return,median_return,worst_return,win_rate
0,2020,2277,1801,2277,2277,0,2277,0,476,199,196,0.549671,0.52,0.002984,0.012793,-0.523503,0.917874
1,2021,2268,1985,2268,2267,1,2266,2,283,13,7,0.308761,0.29,-0.006186,0.007435,-1.000000,0.904321
2,2022,2259,2008,2259,2259,0,2259,0,251,8,8,1.039380,1.00,0.007105,0.020160,-0.566478,0.902169
3,2023,2250,1928,2250,2250,0,2250,0,322,1,1,0.766502,0.74,-0.010733,0.020942,-0.667509,0.807556
4,2024,2268,1560,2268,2268,0,2262,6,708,26,13,0.756936,0.73,-0.003127,0.017409,-0.509074,0.832011
5,2025,2250,1419,2250,2250,0,2245,5,831,107,73,0.783942,0.76,0.009164,0.014610,-0.435404,0.949333
6,2026,1041,861,1041,1041,0,1039,2,180,0,0,0.772238,0.74,-0.026136,0.010486,-0.668007,0.796350



Saved
Checks:                    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_mechanical_sanity_checks_fullhist_1sd3sd_long5_cal_v1_20260714_194449.csv
Global summary:            C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_global_summary_fullhist_1sd3sd_long5_cal_v1_20260714_194449.csv
Coverage by tenor:         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_coverage_by_tenor_fullhist_1sd3sd_long5_cal_v1_20260714_194449.csv
Coverage by year:          C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_coverage_by_year_fullhist_1sd3sd_long5_cal_v1_20260714_194449.csv
Fallback slippage by leg:  C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_fallback_slippage_by_leg_fullhist_1sd3sd_long5_cal_v1_20260714_194449.csv
Non-P&L reason summary:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\02G_DATA_4_PATCHED_non_pnl_reason_sum

In [1]:
# Cell 2G.DIAG.1 — Fixed-parameter DTE diagnostic from full fallback-adjusted call panel
# Purpose:
#   Evaluate each DTE independently under the fixed filter:
#       z_3m > 0.0
#       z_1y > 0.0
#       RSI14 > 58
#
# Tenors:
#   9, 12, 15, 18, 21, 24, 27, 30, 33
#
# Important:
#   - no tenor selection
#   - each DTE is evaluated independently
#   - use only fallback_pnl_complete == True for P&L statistics
#   - report exact-primary-only and fallback sensitivity
#   - report all rows and excluding large fallback flags

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# --------------------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"
DIAG_NAME = "z0_rsi58"
DIAG_SUFFIX = f"{DIAG_NAME}_{OUT_SUFFIX}"

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

Z_THRESHOLD = 0.0
RSI_FLOOR = 58.0

SPREAD_PRICES_WITH_FALLBACK_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_{OUT_SUFFIX}.parquet"
)

SIGNAL_SOURCE_OVERRIDE = None

SIGNAL_SOURCE_CANDIDATES = [
    PROCESSED_DIR / "call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet",
    PROJECT_ROOT / "data" / "processed" / "vrp_repaired_wilder_rsi_signal" / "vrp_repaired_wilder_rsi_signal_base_v1.parquet",
    PROJECT_ROOT / "data" / "processed" / "vrp_final_signal" / "vrp_final_corsi_signal_base_panel_v1.parquet",
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet",
]

OUT_DIAG_PANEL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_panel_{DIAG_SUFFIX}.parquet"
)

OUT_DIAG_DETAIL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_fixed_filter_detail_{DIAG_SUFFIX}.parquet"
)

OUT_DIAG_PNL_DETAIL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_fixed_filter_pnl_detail_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_OVERALL_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_overall_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_BY_TENOR_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_tenor_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_BY_TENOR_YEAR_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_tenor_year_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_BY_YEAR_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_year_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_BY_VARIANT_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_variant_{DIAG_SUFFIX}.parquet"
)

OUT_SUMMARY_BY_TENOR_VARIANT_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_tenor_variant_{DIAG_SUFFIX}.parquet"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2G.DIAG.1 — Fixed-parameter DTE diagnostic")
print("=" * 100)
print(f"Spread panel: {SPREAD_PRICES_WITH_FALLBACK_PATH}")
print(f"Target tenors: {TARGET_TENORS}")
print(f"Filter: z_3m > {Z_THRESHOLD}, z_1y > {Z_THRESHOLD}, RSI14 > {RSI_FLOOR}")
print()

# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def atomic_write_parquet(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.parquet"
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)

def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.parent / f".{path.stem}.tmp.csv"
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def bool_series(s, index=None):
    if s is None:
        return pd.Series(False, index=index, dtype=bool)

    if isinstance(s, bool):
        return pd.Series(s, index=index, dtype=bool)

    if not isinstance(s, pd.Series):
        return pd.Series(s, index=index).fillna(False).astype(bool)

    if s.dtype == bool:
        return s.fillna(False).astype(bool)

    return (
        s.fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y", "priced"])
        .astype(bool)
    )

def first_existing_col(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise KeyError(
            f"Could not find {label}. Tried: {candidates}. "
            f"Available columns include: {list(df.columns)[:80]}"
        )

    return None

def load_signal_source():
    candidates = []

    if SIGNAL_SOURCE_OVERRIDE is not None:
        candidates.append(Path(SIGNAL_SOURCE_OVERRIDE))

    candidates.extend(SIGNAL_SOURCE_CANDIDATES)

    inspected = []

    for path in candidates:
        path = Path(path)

        if not path.exists():
            inspected.append({
                "path": str(path),
                "exists": False,
                "usable": False,
                "reason": "missing_file",
            })
            continue

        try:
            df = pd.read_parquet(path)

            trade_col = first_existing_col(df, ["trade_date", "date", "quote_date"], required=False, label="trade_date")
            tenor_col = first_existing_col(df, ["tenor", "target_tenor", "dte", "tenor_dte"], required=False, label="tenor")
            z3_col = first_existing_col(
                df,
                [
                    "corsi_vrp_z_3m",
                    "z_3m_final",
                    "model_vrp_z_3m_final",
                    "model_vrp_z_3m",
                    "vrp_z_3m",
                    "z_3m",
                ],
                required=False,
                label="z_3m",
            )
            z1_col = first_existing_col(
                df,
                [
                    "corsi_vrp_z_1y",
                    "z_1y_final",
                    "model_vrp_z_1y_final",
                    "model_vrp_z_1y",
                    "vrp_z_1y",
                    "z_1y",
                ],
                required=False,
                label="z_1y",
            )
            rsi_col = first_existing_col(
                df,
                [
                    "rsi14",
                    "rsi14_final",
                    "spy_wilder_rsi14",
                    "wilder_rsi14",
                    "rsi_14",
                ],
                required=False,
                label="rsi14",
            )

            usable = all(c is not None for c in [trade_col, tenor_col, z3_col, z1_col, rsi_col])

            inspected.append({
                "path": str(path),
                "exists": True,
                "usable": bool(usable),
                "rows": int(len(df)),
                "trade_col": trade_col,
                "tenor_col": tenor_col,
                "z3_col": z3_col,
                "z1_col": z1_col,
                "rsi_col": rsi_col,
                "reason": "usable" if usable else "missing_required_columns",
            })

            if usable:
                return path, df, pd.DataFrame(inspected)

        except Exception as e:
            inspected.append({
                "path": str(path),
                "exists": True,
                "usable": False,
                "reason": repr(e),
            })

    inspected_df = pd.DataFrame(inspected)
    display(inspected_df)
    raise RuntimeError("No usable signal source found.")

def summarize_candidate_group(g, label=None):
    pnl = g.loc[g["fallback_pnl_complete"]].copy()

    out = {
        "label": label if label is not None else "all",
        "candidate_rows": int(len(g)),
        "pnl_complete_rows": int(len(pnl)),
        "non_pnl_complete_rows": int((~g["fallback_pnl_complete"]).sum()) if len(g) else 0,
        "pnl_complete_rate": float(len(pnl) / len(g)) if len(g) else np.nan,
        "unique_trade_dates": int(g["trade_date"].nunique()) if len(g) else 0,
        "first_trade_date": g["trade_date"].min() if len(g) else pd.NaT,
        "last_trade_date": g["trade_date"].max() if len(g) else pd.NaT,
        "exact_primary_rows": int(pnl["exact_primary_complete"].sum()) if len(pnl) else 0,
        "fallback_used_rows": int(pnl["fallback_used_any"].sum()) if len(pnl) else 0,
        "fallback_used_rate": float(pnl["fallback_used_any"].mean()) if len(pnl) else np.nan,
        "large_fallback_rows": int((pnl["large_fallback_dollar_flag_any"] | pnl["large_fallback_width_pct_flag_any"]).sum()) if len(pnl) else 0,
        "large_fallback_rate": float((pnl["large_fallback_dollar_flag_any"] | pnl["large_fallback_width_pct_flag_any"]).mean()) if len(pnl) else np.nan,
        "unrecoverable_joint_fallback_rows": int(pnl["unrecoverable_joint_fallback"].sum()) if "unrecoverable_joint_fallback" in pnl.columns and len(pnl) else 0,
    }

    if len(pnl):
        credit_pct_width = pnl["entry_credit_after_fallback"] / pnl["effective_width_after_fallback"]

        out.update({
            "win_rate": float(pnl["trade_win_after_fallback"].mean()),
            "avg_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].mean()),
            "median_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].median()),
            "worst_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].min()),
            "p05_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].quantile(0.05)),
            "p10_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].quantile(0.10)),
            "p25_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].quantile(0.25)),
            "p75_return_on_max_loss": float(pnl["return_on_max_loss_after_fallback"].quantile(0.75)),
            "avg_entry_credit": float(pnl["entry_credit_after_fallback"].mean()),
            "median_entry_credit": float(pnl["entry_credit_after_fallback"].median()),
            "avg_credit_pct_width": float(credit_pct_width.mean()),
            "median_credit_pct_width": float(credit_pct_width.median()),
            "avg_effective_width": float(pnl["effective_width_after_fallback"].mean()),
            "median_effective_width": float(pnl["effective_width_after_fallback"].median()),
            "avg_short_effective_strike": float(pnl["short_effective_strike"].mean()),
            "avg_long_effective_strike": float(pnl["long_effective_strike"].mean()),
            "avg_z_3m": float(pnl["diag_z_3m"].mean()),
            "avg_z_1y": float(pnl["diag_z_1y"].mean()),
            "avg_rsi14": float(pnl["diag_rsi14"].mean()),
            "median_z_3m": float(pnl["diag_z_3m"].median()),
            "median_z_1y": float(pnl["diag_z_1y"].median()),
            "median_rsi14": float(pnl["diag_rsi14"].median()),
        })
    else:
        out.update({
            "win_rate": np.nan,
            "avg_return_on_max_loss": np.nan,
            "median_return_on_max_loss": np.nan,
            "worst_return_on_max_loss": np.nan,
            "p05_return_on_max_loss": np.nan,
            "p10_return_on_max_loss": np.nan,
            "p25_return_on_max_loss": np.nan,
            "p75_return_on_max_loss": np.nan,
            "avg_entry_credit": np.nan,
            "median_entry_credit": np.nan,
            "avg_credit_pct_width": np.nan,
            "median_credit_pct_width": np.nan,
            "avg_effective_width": np.nan,
            "median_effective_width": np.nan,
            "avg_short_effective_strike": np.nan,
            "avg_long_effective_strike": np.nan,
            "avg_z_3m": np.nan,
            "avg_z_1y": np.nan,
            "avg_rsi14": np.nan,
            "median_z_3m": np.nan,
            "median_z_1y": np.nan,
            "median_rsi14": np.nan,
        })

    return pd.Series(out)

def summarize_variant(df, variant_name):
    s = summarize_candidate_group(df, label=variant_name)
    out = pd.DataFrame([s])
    out = out.rename(columns={"label": "variant"})
    return out

# --------------------------------------------------------------------------------------------------
# Load fallback-adjusted spread panel
# --------------------------------------------------------------------------------------------------

require_file(SPREAD_PRICES_WITH_FALLBACK_PATH)

spreads = pd.read_parquet(SPREAD_PRICES_WITH_FALLBACK_PATH).copy()

for c in ["trade_date", "expiration_date"]:
    if c in spreads.columns:
        spreads[c] = pd.to_datetime(spreads[c], errors="coerce").dt.normalize()

if "trade_year" not in spreads.columns:
    spreads["trade_year"] = spreads["trade_date"].dt.year

bool_cols = [
    "outcome_available",
    "exact_primary_complete",
    "exact_primary_pnl_complete",
    "complete_after_fallback",
    "fallback_pnl_complete",
    "fallback_used_any",
    "large_fallback_dollar_flag_any",
    "large_fallback_width_pct_flag_any",
    "trade_win_after_fallback",
    "joint_structure_valid_after_fallback",
    "unrecoverable_joint_fallback",
]

for c in bool_cols:
    if c in spreads.columns:
        spreads[c] = bool_series(spreads[c], index=spreads.index)

for c in [
    "entry_credit_after_fallback",
    "effective_width_after_fallback",
    "max_loss_after_fallback",
    "expiry_pnl_after_fallback",
    "return_on_max_loss_after_fallback",
    "short_effective_strike",
    "long_effective_strike",
    "tenor",
]:
    if c in spreads.columns:
        spreads[c] = pd.to_numeric(spreads[c], errors="coerce")

spreads = spreads.loc[spreads["tenor"].isin(TARGET_TENORS)].copy()

print("=" * 100)
print("Loaded fallback-adjusted spread panel")
print("=" * 100)
print(f"Rows: {len(spreads):,}")
print(f"Date range: {spreads['trade_date'].min().date()} to {spreads['trade_date'].max().date()}")
print(f"P&L-complete rows: {int(spreads['fallback_pnl_complete'].sum()):,}")
print(f"Unrecoverable joint fallback rows: {int(spreads.get('unrecoverable_joint_fallback', pd.Series(False, index=spreads.index)).sum()):,}")
print()

# --------------------------------------------------------------------------------------------------
# Load and normalize signal source
# --------------------------------------------------------------------------------------------------

signal_source_path, signal_raw, signal_source_inspection = load_signal_source()

print("=" * 100)
print("Signal source inspection")
print("=" * 100)
display(signal_source_inspection)

trade_col = first_existing_col(signal_raw, ["trade_date", "date", "quote_date"], label="trade_date")
tenor_col = first_existing_col(signal_raw, ["tenor", "target_tenor", "dte", "tenor_dte"], label="tenor")

z3_col = first_existing_col(
    signal_raw,
    [
        "corsi_vrp_z_3m",
        "z_3m_final",
        "model_vrp_z_3m_final",
        "model_vrp_z_3m",
        "vrp_z_3m",
        "z_3m",
    ],
    label="z_3m",
)

z1_col = first_existing_col(
    signal_raw,
    [
        "corsi_vrp_z_1y",
        "z_1y_final",
        "model_vrp_z_1y_final",
        "model_vrp_z_1y",
        "vrp_z_1y",
        "z_1y",
    ],
    label="z_1y",
)

rsi_col = first_existing_col(
    signal_raw,
    [
        "rsi14",
        "rsi14_final",
        "spy_wilder_rsi14",
        "wilder_rsi14",
        "rsi_14",
    ],
    label="rsi14",
)

vrp_log_col = first_existing_col(
    signal_raw,
    [
        "corsi_vrp_log",
        "model_vrp_log_final",
        "model_vrp_log",
        "vrp_log",
        "model_vrp_log_source",
    ],
    required=False,
    label="vrp_log",
)

rv21d_col = first_existing_col(
    signal_raw,
    [
        "rv21d_vol_pct",
        "rv21d_vol_pct_final",
        "rv21d",
        "rv_21d",
    ],
    required=False,
    label="rv21d",
)

forecast_vol_col = first_existing_col(
    signal_raw,
    [
        "forecast_vol_pct",
        "forecast_vol_final",
        "forecast_vol",
    ],
    required=False,
    label="forecast_vol",
)

vix_vol_col = first_existing_col(
    signal_raw,
    [
        "vix_style_vol_pct",
        "vix_style_vol_final",
        "implied_vol_pct",
    ],
    required=False,
    label="vix_style_vol",
)

signal = pd.DataFrame({
    "trade_date": pd.to_datetime(signal_raw[trade_col], errors="coerce").dt.normalize(),
    "tenor": pd.to_numeric(signal_raw[tenor_col], errors="coerce"),
    "diag_z_3m": pd.to_numeric(signal_raw[z3_col], errors="coerce"),
    "diag_z_1y": pd.to_numeric(signal_raw[z1_col], errors="coerce"),
    "diag_rsi14": pd.to_numeric(signal_raw[rsi_col], errors="coerce"),
})

if vrp_log_col is not None:
    signal["diag_vrp_log"] = pd.to_numeric(signal_raw[vrp_log_col], errors="coerce")
else:
    signal["diag_vrp_log"] = np.nan

if rv21d_col is not None:
    signal["diag_rv21d_vol_pct"] = pd.to_numeric(signal_raw[rv21d_col], errors="coerce")
else:
    signal["diag_rv21d_vol_pct"] = np.nan

if forecast_vol_col is not None:
    signal["diag_forecast_vol_pct"] = pd.to_numeric(signal_raw[forecast_vol_col], errors="coerce")
else:
    signal["diag_forecast_vol_pct"] = np.nan

if vix_vol_col is not None:
    signal["diag_vix_style_vol_pct"] = pd.to_numeric(signal_raw[vix_vol_col], errors="coerce")
else:
    signal["diag_vix_style_vol_pct"] = np.nan

signal = (
    signal
    .loc[signal["trade_date"].notna() & signal["tenor"].isin(TARGET_TENORS)]
    .drop_duplicates(["trade_date", "tenor"], keep="last")
    .reset_index(drop=True)
)

print()
print("=" * 100)
print("Selected signal source")
print("=" * 100)
print(f"Path: {signal_source_path}")
print(f"Rows after normalize/dedupe: {len(signal):,}")
print(f"Date range: {signal['trade_date'].min().date()} to {signal['trade_date'].max().date()}")
print(f"Columns used: z3={z3_col}, z1={z1_col}, rsi={rsi_col}, vrp_log={vrp_log_col}")
print()

# --------------------------------------------------------------------------------------------------
# Join signal inputs to spread panel
# --------------------------------------------------------------------------------------------------

panel = spreads.merge(
    signal,
    on=["trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

panel["signal_inputs_complete"] = (
    panel["diag_z_3m"].notna()
    & panel["diag_z_1y"].notna()
    & panel["diag_rsi14"].notna()
)

panel["pass_z_3m"] = panel["diag_z_3m"] > Z_THRESHOLD
panel["pass_z_1y"] = panel["diag_z_1y"] > Z_THRESHOLD
panel["pass_rsi14"] = panel["diag_rsi14"] > RSI_FLOOR

panel["fixed_filter_pass"] = (
    panel["signal_inputs_complete"]
    & panel["pass_z_3m"]
    & panel["pass_z_1y"]
    & panel["pass_rsi14"]
)

panel["diag_use_for_pnl"] = (
    panel["fixed_filter_pass"]
    & panel["fallback_pnl_complete"]
)

panel["large_fallback_any"] = (
    panel["large_fallback_dollar_flag_any"]
    | panel["large_fallback_width_pct_flag_any"]
)

diagnostic_detail = panel.loc[panel["fixed_filter_pass"]].copy()
pnl_detail = diagnostic_detail.loc[diagnostic_detail["fallback_pnl_complete"]].copy()

print("=" * 100)
print("Fixed-filter diagnostic row counts")
print("=" * 100)
print(f"Full panel rows:                 {len(panel):,}")
print(f"Signal inputs complete:          {int(panel['signal_inputs_complete'].sum()):,}")
print(f"z_3m > {Z_THRESHOLD}:                 {int(panel['pass_z_3m'].sum()):,}")
print(f"z_1y > {Z_THRESHOLD}:                 {int(panel['pass_z_1y'].sum()):,}")
print(f"RSI14 > {RSI_FLOOR}:                 {int(panel['pass_rsi14'].sum()):,}")
print(f"Fixed-filter candidate rows:     {len(diagnostic_detail):,}")
print(f"P&L-complete candidate rows:     {len(pnl_detail):,}")
print(f"Non-P&L-complete candidates:     {len(diagnostic_detail) - len(pnl_detail):,}")
print()

# --------------------------------------------------------------------------------------------------
# Component pass summaries
# --------------------------------------------------------------------------------------------------

component_summary = pd.DataFrame([{
    "full_panel_rows": int(len(panel)),
    "signal_inputs_complete": int(panel["signal_inputs_complete"].sum()),
    "pass_z_3m": int(panel["pass_z_3m"].sum()),
    "pass_z_1y": int(panel["pass_z_1y"].sum()),
    "pass_rsi14": int(panel["pass_rsi14"].sum()),
    "fixed_filter_pass": int(panel["fixed_filter_pass"].sum()),
    "fixed_filter_pnl_complete": int(panel["diag_use_for_pnl"].sum()),
    "fixed_filter_rate_vs_panel": float(panel["fixed_filter_pass"].mean()),
    "pnl_complete_rate_vs_fixed_filter": float(panel.loc[panel["fixed_filter_pass"], "fallback_pnl_complete"].mean()) if panel["fixed_filter_pass"].any() else np.nan,
}])

component_by_tenor = (
    panel
    .groupby("tenor", dropna=False)
    .agg(
        panel_rows=("selected_trade_id", "size"),
        signal_inputs_complete=("signal_inputs_complete", "sum"),
        pass_z_3m=("pass_z_3m", "sum"),
        pass_z_1y=("pass_z_1y", "sum"),
        pass_rsi14=("pass_rsi14", "sum"),
        fixed_filter_pass=("fixed_filter_pass", "sum"),
        fixed_filter_pnl_complete=("diag_use_for_pnl", "sum"),
    )
    .reset_index()
)

component_by_tenor["fixed_filter_rate_vs_panel"] = (
    component_by_tenor["fixed_filter_pass"] / component_by_tenor["panel_rows"]
)

component_by_tenor["pnl_complete_rate_vs_fixed_filter"] = (
    component_by_tenor["fixed_filter_pnl_complete"] / component_by_tenor["fixed_filter_pass"].replace(0, np.nan)
)

# --------------------------------------------------------------------------------------------------
# Primary DTE summaries
# --------------------------------------------------------------------------------------------------

overall_summary = pd.DataFrame([summarize_candidate_group(diagnostic_detail, label="all_fixed_filter_candidates")])
overall_summary = overall_summary.rename(columns={"label": "summary"})

summary_by_tenor = pd.DataFrame([
    summarize_candidate_group(g, label=tenor)
    for tenor, g in diagnostic_detail.groupby("tenor", dropna=False)
]).rename(columns={"label": "tenor"})

tenor_denoms = (
    panel
    .groupby("tenor", dropna=False)
    .size()
    .rename("panel_rows")
    .reset_index()
)

summary_by_tenor = summary_by_tenor.merge(tenor_denoms, on="tenor", how="left")
summary_by_tenor["fixed_filter_rate_vs_panel_rows"] = (
    summary_by_tenor["candidate_rows"] / summary_by_tenor["panel_rows"]
)

summary_by_tenor = summary_by_tenor[
    ["tenor", "panel_rows", "fixed_filter_rate_vs_panel_rows"]
    + [c for c in summary_by_tenor.columns if c not in ["tenor", "panel_rows", "fixed_filter_rate_vs_panel_rows"]]
]

summary_by_tenor_year = pd.DataFrame([
    pd.concat([
        pd.Series({"tenor": tenor, "trade_year": year}),
        summarize_candidate_group(g, label="tenor_year").drop("label"),
    ])
    for (tenor, year), g in diagnostic_detail.groupby(["tenor", "trade_year"], dropna=False)
]).reset_index(drop=True)

tenor_year_denoms = (
    panel
    .groupby(["tenor", "trade_year"], dropna=False)
    .size()
    .rename("panel_rows")
    .reset_index()
)

summary_by_tenor_year = summary_by_tenor_year.merge(
    tenor_year_denoms,
    on=["tenor", "trade_year"],
    how="left",
)

summary_by_tenor_year["fixed_filter_rate_vs_panel_rows"] = (
    summary_by_tenor_year["candidate_rows"] / summary_by_tenor_year["panel_rows"]
)

summary_by_year = pd.DataFrame([
    pd.concat([
        pd.Series({"trade_year": year}),
        summarize_candidate_group(g, label="year").drop("label"),
    ])
    for year, g in diagnostic_detail.groupby("trade_year", dropna=False)
]).reset_index(drop=True)

year_denoms = (
    panel
    .groupby("trade_year", dropna=False)
    .size()
    .rename("panel_rows")
    .reset_index()
)

summary_by_year = summary_by_year.merge(
    year_denoms,
    on="trade_year",
    how="left",
)

summary_by_year["fixed_filter_rate_vs_panel_rows"] = (
    summary_by_year["candidate_rows"] / summary_by_year["panel_rows"]
)

# --------------------------------------------------------------------------------------------------
# Variant / sensitivity summaries
# --------------------------------------------------------------------------------------------------

variant_frames = []

variant_inputs = {
    "all_pnl_complete": diagnostic_detail.loc[diagnostic_detail["fallback_pnl_complete"]].copy(),
    "exact_primary_only": diagnostic_detail.loc[
        diagnostic_detail["fallback_pnl_complete"]
        & diagnostic_detail["exact_primary_complete"]
    ].copy(),
    "fallback_used_only": diagnostic_detail.loc[
        diagnostic_detail["fallback_pnl_complete"]
        & diagnostic_detail["fallback_used_any"]
    ].copy(),
    "exclude_large_fallback_flags": diagnostic_detail.loc[
        diagnostic_detail["fallback_pnl_complete"]
        & ~diagnostic_detail["large_fallback_any"]
    ].copy(),
    "large_fallback_only": diagnostic_detail.loc[
        diagnostic_detail["fallback_pnl_complete"]
        & diagnostic_detail["large_fallback_any"]
    ].copy(),
}

for variant_name, df_variant in variant_inputs.items():
    variant_frames.append(summarize_variant(df_variant, variant_name))

summary_by_variant = pd.concat(variant_frames, ignore_index=True)

tenor_variant_rows = []

for variant_name, df_variant in variant_inputs.items():
    if df_variant.empty:
        continue

    for tenor, g in df_variant.groupby("tenor", dropna=False):
        s = summarize_candidate_group(g, label=variant_name)
        row = s.to_dict()
        row["variant"] = row.pop("label")
        row["tenor"] = tenor
        tenor_variant_rows.append(row)

summary_by_tenor_variant = pd.DataFrame(tenor_variant_rows)

if not summary_by_tenor_variant.empty:
    summary_by_tenor_variant = summary_by_tenor_variant[
        ["variant", "tenor"] + [c for c in summary_by_tenor_variant.columns if c not in ["variant", "tenor"]]
    ]

# --------------------------------------------------------------------------------------------------
# Non-P&L-complete fixed-filter candidates
# --------------------------------------------------------------------------------------------------

fixed_non_pnl = diagnostic_detail.loc[~diagnostic_detail["fallback_pnl_complete"]].copy()

if "pnl_exclusion_reason_after_fallback" in fixed_non_pnl.columns:
    fixed_non_pnl["diag_exclusion_reason"] = fixed_non_pnl["pnl_exclusion_reason_after_fallback"].fillna("unknown")
else:
    fixed_non_pnl["diag_exclusion_reason"] = "not_pnl_complete"

fixed_non_pnl_summary = (
    fixed_non_pnl["diag_exclusion_reason"]
    .value_counts(dropna=False)
    .rename_axis("diag_exclusion_reason")
    .reset_index(name="rows")
)

# --------------------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------------------

atomic_write_parquet(panel, OUT_DIAG_PANEL_PATH)
atomic_write_parquet(diagnostic_detail, OUT_DIAG_DETAIL_PATH)
atomic_write_parquet(pnl_detail, OUT_DIAG_PNL_DETAIL_PATH)

atomic_write_parquet(overall_summary, OUT_SUMMARY_OVERALL_PATH)
atomic_write_parquet(summary_by_tenor, OUT_SUMMARY_BY_TENOR_PATH)
atomic_write_parquet(summary_by_tenor_year, OUT_SUMMARY_BY_TENOR_YEAR_PATH)
atomic_write_parquet(summary_by_year, OUT_SUMMARY_BY_YEAR_PATH)
atomic_write_parquet(summary_by_variant, OUT_SUMMARY_BY_VARIANT_PATH)
atomic_write_parquet(summary_by_tenor_variant, OUT_SUMMARY_BY_TENOR_VARIANT_PATH)

audit_paths = {
    "component_summary": AUDIT_DIR / f"02G_DIAG_1_component_summary_{DIAG_SUFFIX}_{timestamp}.csv",
    "component_by_tenor": AUDIT_DIR / f"02G_DIAG_1_component_by_tenor_{DIAG_SUFFIX}_{timestamp}.csv",
    "overall_summary": AUDIT_DIR / f"02G_DIAG_1_summary_overall_{DIAG_SUFFIX}_{timestamp}.csv",
    "summary_by_tenor": AUDIT_DIR / f"02G_DIAG_1_summary_by_tenor_{DIAG_SUFFIX}_{timestamp}.csv",
    "summary_by_tenor_year": AUDIT_DIR / f"02G_DIAG_1_summary_by_tenor_year_{DIAG_SUFFIX}_{timestamp}.csv",
    "summary_by_year": AUDIT_DIR / f"02G_DIAG_1_summary_by_year_{DIAG_SUFFIX}_{timestamp}.csv",
    "summary_by_variant": AUDIT_DIR / f"02G_DIAG_1_summary_by_variant_{DIAG_SUFFIX}_{timestamp}.csv",
    "summary_by_tenor_variant": AUDIT_DIR / f"02G_DIAG_1_summary_by_tenor_variant_{DIAG_SUFFIX}_{timestamp}.csv",
    "fixed_non_pnl_summary": AUDIT_DIR / f"02G_DIAG_1_fixed_non_pnl_summary_{DIAG_SUFFIX}_{timestamp}.csv",
    "fixed_non_pnl_preview": AUDIT_DIR / f"02G_DIAG_1_fixed_non_pnl_preview_{DIAG_SUFFIX}_{timestamp}.csv",
    "signal_source_inspection": AUDIT_DIR / f"02G_DIAG_1_signal_source_inspection_{DIAG_SUFFIX}_{timestamp}.csv",
}

atomic_write_csv(component_summary, audit_paths["component_summary"])
atomic_write_csv(component_by_tenor, audit_paths["component_by_tenor"])
atomic_write_csv(overall_summary, audit_paths["overall_summary"])
atomic_write_csv(summary_by_tenor, audit_paths["summary_by_tenor"])
atomic_write_csv(summary_by_tenor_year, audit_paths["summary_by_tenor_year"])
atomic_write_csv(summary_by_year, audit_paths["summary_by_year"])
atomic_write_csv(summary_by_variant, audit_paths["summary_by_variant"])
atomic_write_csv(summary_by_tenor_variant, audit_paths["summary_by_tenor_variant"])
atomic_write_csv(fixed_non_pnl_summary, audit_paths["fixed_non_pnl_summary"])
atomic_write_csv(fixed_non_pnl.head(1000), audit_paths["fixed_non_pnl_preview"])
atomic_write_csv(signal_source_inspection, audit_paths["signal_source_inspection"])

manifest_path = AUDIT_DIR / f"02G_DIAG_1_fixed_parameter_dte_diagnostic_{DIAG_SUFFIX}_manifest_{timestamp}.json"

manifest = {
    "cell": "2G.DIAG.1",
    "description": "Fixed-parameter DTE diagnostic from full fallback-adjusted call-spread panel",
    "created_at": timestamp,
    "diag_name": DIAG_NAME,
    "out_suffix": OUT_SUFFIX,
    "target_tenors": TARGET_TENORS,
    "filter": {
        "z_3m_operator": ">",
        "z_3m_threshold": Z_THRESHOLD,
        "z_1y_operator": ">",
        "z_1y_threshold": Z_THRESHOLD,
        "rsi14_operator": ">",
        "rsi14_threshold": RSI_FLOOR,
    },
    "rules": {
        "tenor_selection": "none; each DTE evaluated independently",
        "pnl_filter": "fallback_pnl_complete == True",
        "large_fallback_flags": "reported as sensitivity, not excluded from main summary",
    },
    "inputs": {
        "spread_prices_with_fallback_path": str(SPREAD_PRICES_WITH_FALLBACK_PATH),
        "signal_source_path": str(signal_source_path),
        "signal_columns": {
            "trade_date": trade_col,
            "tenor": tenor_col,
            "z_3m": z3_col,
            "z_1y": z1_col,
            "rsi14": rsi_col,
            "vrp_log": vrp_log_col,
            "rv21d": rv21d_col,
            "forecast_vol": forecast_vol_col,
            "vix_style_vol": vix_vol_col,
        },
    },
    "counts": {
        "panel_rows": int(len(panel)),
        "signal_inputs_complete": int(panel["signal_inputs_complete"].sum()),
        "fixed_filter_candidate_rows": int(len(diagnostic_detail)),
        "fixed_filter_pnl_complete_rows": int(len(pnl_detail)),
        "fixed_filter_non_pnl_complete_rows": int(len(diagnostic_detail) - len(pnl_detail)),
        "fixed_filter_rate_vs_panel": float(panel["fixed_filter_pass"].mean()),
        "pnl_complete_rate_vs_fixed_filter": float(panel.loc[panel["fixed_filter_pass"], "fallback_pnl_complete"].mean()) if panel["fixed_filter_pass"].any() else np.nan,
    },
    "outputs": {
        "diag_panel_path": str(OUT_DIAG_PANEL_PATH),
        "diag_detail_path": str(OUT_DIAG_DETAIL_PATH),
        "diag_pnl_detail_path": str(OUT_DIAG_PNL_DETAIL_PATH),
        "summary_overall_path": str(OUT_SUMMARY_OVERALL_PATH),
        "summary_by_tenor_path": str(OUT_SUMMARY_BY_TENOR_PATH),
        "summary_by_tenor_year_path": str(OUT_SUMMARY_BY_TENOR_YEAR_PATH),
        "summary_by_year_path": str(OUT_SUMMARY_BY_YEAR_PATH),
        "summary_by_variant_path": str(OUT_SUMMARY_BY_VARIANT_PATH),
        "summary_by_tenor_variant_path": str(OUT_SUMMARY_BY_TENOR_VARIANT_PATH),
        "manifest_path": str(manifest_path),
    },
    "next_step": "Review DTE results; optionally run threshold grid diagnostics or fallback-exclusion sensitivity.",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# --------------------------------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("Cell 2G.DIAG.1 complete — fixed-parameter DTE diagnostic")
print("=" * 100)

print()
print("Component summary")
display(component_summary)

print()
print("=" * 100)
print("Component by tenor")
print("=" * 100)
display(component_by_tenor)

print()
print("=" * 100)
print("Overall summary")
print("=" * 100)
display(overall_summary)

print()
print("=" * 100)
print("Summary by tenor")
print("=" * 100)
display(summary_by_tenor)

print()
print("=" * 100)
print("Summary by year")
print("=" * 100)
display(summary_by_year)

print()
print("=" * 100)
print("Summary by variant")
print("=" * 100)
display(summary_by_variant)

print()
print("=" * 100)
print("Summary by tenor × variant")
print("=" * 100)
display(summary_by_tenor_variant)

print()
print("=" * 100)
print("Fixed-filter non-P&L-complete rows")
print("=" * 100)
if fixed_non_pnl.empty:
    print("No fixed-filter rows excluded for P&L completeness.")
else:
    display(fixed_non_pnl_summary)
    display(
        fixed_non_pnl[
            [
                "selected_trade_id",
                "trade_date",
                "tenor",
                "expiration_date",
                "entry_credit_after_fallback",
                "effective_width_after_fallback",
                "joint_structure_valid_after_fallback",
                "unrecoverable_joint_fallback",
                "diag_exclusion_reason",
            ]
        ].head(100)
    )

print()
print("=" * 100)
print("Saved")
print("=" * 100)
print(f"Diagnostic panel:          {OUT_DIAG_PANEL_PATH}")
print(f"Fixed-filter detail:       {OUT_DIAG_DETAIL_PATH}")
print(f"Fixed-filter P&L detail:   {OUT_DIAG_PNL_DETAIL_PATH}")
print(f"Summary overall:           {OUT_SUMMARY_OVERALL_PATH}")
print(f"Summary by tenor:          {OUT_SUMMARY_BY_TENOR_PATH}")
print(f"Summary by tenor-year:     {OUT_SUMMARY_BY_TENOR_YEAR_PATH}")
print(f"Summary by year:           {OUT_SUMMARY_BY_YEAR_PATH}")
print(f"Summary by variant:        {OUT_SUMMARY_BY_VARIANT_PATH}")
print(f"Summary tenor × variant:   {OUT_SUMMARY_BY_TENOR_VARIANT_PATH}")
print(f"Manifest:                  {manifest_path}")

print()
print("Result: 2G.DIAG.1 complete.")
print("Review the DTE summary and variant sensitivity before running broader parameter grids.")

Cell 2G.DIAG.1 — Fixed-parameter DTE diagnostic
Spread panel: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_full_history_1sd3sd_spread_prices_with_fallback_fullhist_1sd3sd_long5_cal_v1.parquet
Target tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Filter: z_3m > 0.0, z_1y > 0.0, RSI14 > 58.0

Loaded fallback-adjusted spread panel
Rows: 14,613
Date range: 2020-01-02 to 2026-07-01
P&L-complete rows: 14,598
Unrecoverable joint fallback rows: 1

Signal source inspection


,path,exists,usable,rows,trade_col,tenor_col,z3_col,z1_col,rsi_col,reason
0,C:\Users\patri\vrp_project\data\processed\call...,True,True,14724,trade_date,tenor,corsi_vrp_z_3m,corsi_vrp_z_1y,rsi14,usable



Selected signal source
Path: C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_repaired_rsi_base_candidate_panel_repaired_rsi_long5_cal_v1.parquet
Rows after normalize/dedupe: 14,724
Date range: 2020-01-02 to 2026-07-09
Columns used: z3=corsi_vrp_z_3m, z1=corsi_vrp_z_1y, rsi=rsi14, vrp_log=corsi_vrp_log

Fixed-filter diagnostic row counts
Full panel rows:                 14,613
Signal inputs complete:          12,345
z_3m > 0.0:                 6,640
z_1y > 0.0:                 5,296
RSI14 > 58.0:                 6,972
Fixed-filter candidate rows:     1,941
P&L-complete candidate rows:     1,941
Non-P&L-complete candidates:     0


Cell 2G.DIAG.1 complete — fixed-parameter DTE diagnostic

Component summary


,full_panel_rows,signal_inputs_complete,pass_z_3m,pass_z_1y,pass_rsi14,fixed_filter_pass,fixed_filter_pnl_complete,fixed_filter_rate_vs_panel,pnl_complete_rate_vs_fixed_filter
0,14613,12345,6640,5296,6972,1941,1941,0.132827,1.0



Component by tenor


...


Summary by variant


,variant,candidate_rows,pnl_complete_rows,non_pnl_complete_rows,pnl_complete_rate,unique_trade_dates,first_trade_date,last_trade_date,exact_primary_rows,fallback_used_rows,...,avg_effective_width,median_effective_width,avg_short_effective_strike,avg_long_effective_strike,avg_z_3m,avg_z_1y,avg_rsi14,median_z_3m,median_z_1y,median_rsi14
0,all_pnl_complete,1941,1941,0,1.0,314,2020-12-31,2026-06-02,1209,732,...,39.485832,38.0,582.482226,621.968058,0.949152,0.968828,66.711370,0.791076,0.773988,66.563452
1,exact_primary_only,1209,1209,0,1.0,297,2020-12-31,2026-06-02,1209,0,...,36.531844,34.0,565.354012,601.885856,0.909340,0.838892,66.369120,0.764696,0.672838,66.114503
2,fallback_used_only,732,732,0,1.0,212,2020-12-31,2026-05-14,0,732,...,44.364754,45.0,610.771858,655.136612,1.014906,1.183436,67.276643,0.873942,0.984340,67.279373
3,exclude_large_fallback_flags,1919,1919,0,1.0,314,2020-12-31,2026-06-02,1209,710,...,39.417405,37.0,581.986972,621.404377,0.946226,0.957921,66.736657,0.789333,0.767502,66.588873
4,large_fallback_only,22,22,0,1.0,18,2024-10-04,2025-07-07,0,22,...,45.454545,45.0,625.681818,671.136364,1.204294,1.920241,64.505624,1.244568,1.923997,63.353249



Summary by tenor × variant


,variant,tenor,candidate_rows,pnl_complete_rows,non_pnl_complete_rows,pnl_complete_rate,unique_trade_dates,first_trade_date,last_trade_date,exact_primary_rows,...,avg_effective_width,median_effective_width,avg_short_effective_strike,avg_long_effective_strike,avg_z_3m,avg_z_1y,avg_rsi14,median_z_3m,median_z_1y,median_rsi14
0,all_pnl_complete,9,249,249,0,1.0,249,2020-12-31,2026-05-14,229,...,24.369478,24.0,548.180723,572.550201,0.936604,0.845740,66.530945,0.788714,0.641677,66.114503
1,all_pnl_complete,12,241,241,0,1.0,241,2020-12-31,2026-05-14,202,...,29.219917,29.0,555.883817,585.103734,0.913457,0.863338,66.349828,0.740297,0.678611,65.980420
2,all_pnl_complete,15,231,231,0,1.0,231,2021-01-25,2026-06-02,184,...,33.225108,34.0,568.787879,602.012987,0.889416,0.886317,66.730031,0.698484,0.696717,66.563452
3,all_pnl_complete,18,223,223,0,1.0,223,2020-12-31,2026-06-02,152,...,37.152466,37.0,576.524664,613.677130,0.917289,0.921846,66.716384,0.753135,0.732767,66.563452
4,all_pnl_complete,21,214,214,0,1.0,214,2020-12-31,2026-06-02,122,...,41.471963,40.0,589.485981,630.957944,0.948404,0.989429,66.913643,0.791037,0.769664,66.915151
5,all_pnl_complete,24,205,205,0,1.0,205,2020-12-31,2026-06-02,107,...,44.809756,45.0,597.117073,641.926829,0.988641,1.033183,66.913950,0.813932,0.814821,66.855494
6,all_pnl_complete,27,200,200,0,1.0,200,2020-12-31,2026-06-02,87,...,48.440000,50.0,603.260000,651.700000,0.988811,1.071823,66.883999,0.834398,0.854678,66.720786
7,all_pnl_complete,30,191,191,0,1.0,191,2020-12-31,2026-06-02,71,...,51.534031,50.0,608.335079,659.869110,0.997715,1.084636,66.683953,0.883016,0.924654,66.588873
8,all_pnl_complete,33,187,187,0,1.0,187,2020-12-31,2026-06-02,55,...,53.368984,54.0,613.770053,667.139037,0.989196,1.104065,66.778345,0.901896,0.921117,66.842168
9,exact_primary_only,9,229,229,0,1.0,229,2020-12-31,2026-05-14,229,...,24.384279,23.0,545.266376,569.650655,0.911286,0.803679,66.421162,0.775207,0.627918,66.015633



Fixed-filter non-P&L-complete rows
No fixed-filter rows excluded for P&L completeness.

Saved
Diagnostic panel:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_panel_z0_rsi58_fullhist_1sd3sd_long5_cal_v1.parquet
Fixed-filter detail:       C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_fixed_filter_detail_z0_rsi58_fullhist_1sd3sd_long5_cal_v1.parquet
Fixed-filter P&L detail:   C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_fixed_filter_pnl_detail_z0_rsi58_fullhist_1sd3sd_long5_cal_v1.parquet
Summary overall:           C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_summary_overall_z0_rsi58_fullhist_1sd3sd_long5_cal_v1.parquet
Summary by tenor:          C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_call_dte_diagnostic_summary_by_teno

In [2]:
# Cell 2G.DIAG.1A — Compact display of key DTE diagnostic results

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

OUT_SUFFIX = "fullhist_1sd3sd_long5_cal_v1"
DIAG_NAME = "z0_rsi58"
DIAG_SUFFIX = f"{DIAG_NAME}_{OUT_SUFFIX}"

SUMMARY_BY_TENOR_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_tenor_{DIAG_SUFFIX}.parquet"
)

SUMMARY_BY_YEAR_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_year_{DIAG_SUFFIX}.parquet"
)

SUMMARY_BY_VARIANT_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_variant_{DIAG_SUFFIX}.parquet"
)

SUMMARY_BY_TENOR_VARIANT_PATH = (
    PROCESSED_DIR / f"call_sleeve_corsi_call_dte_diagnostic_summary_by_tenor_variant_{DIAG_SUFFIX}.parquet"
)

by_tenor = pd.read_parquet(SUMMARY_BY_TENOR_PATH)
by_year = pd.read_parquet(SUMMARY_BY_YEAR_PATH)
by_variant = pd.read_parquet(SUMMARY_BY_VARIANT_PATH)
by_tenor_variant = pd.read_parquet(SUMMARY_BY_TENOR_VARIANT_PATH)

key_cols_tenor = [
    "tenor",
    "candidate_rows",
    "pnl_complete_rows",
    "fixed_filter_rate_vs_panel_rows",
    "win_rate",
    "avg_return_on_max_loss",
    "median_return_on_max_loss",
    "worst_return_on_max_loss",
    "p05_return_on_max_loss",
    "avg_entry_credit",
    "median_entry_credit",
    "avg_credit_pct_width",
    "median_credit_pct_width",
    "fallback_used_rows",
    "fallback_used_rate",
    "large_fallback_rows",
    "large_fallback_rate",
]

key_cols_year = [
    "trade_year",
    "candidate_rows",
    "pnl_complete_rows",
    "fixed_filter_rate_vs_panel_rows",
    "win_rate",
    "avg_return_on_max_loss",
    "median_return_on_max_loss",
    "worst_return_on_max_loss",
    "p05_return_on_max_loss",
    "avg_entry_credit",
    "median_entry_credit",
    "fallback_used_rows",
    "fallback_used_rate",
    "large_fallback_rows",
    "large_fallback_rate",
]

key_cols_variant = [
    "variant",
    "candidate_rows",
    "pnl_complete_rows",
    "win_rate",
    "avg_return_on_max_loss",
    "median_return_on_max_loss",
    "worst_return_on_max_loss",
    "p05_return_on_max_loss",
    "avg_entry_credit",
    "median_entry_credit",
    "fallback_used_rows",
    "fallback_used_rate",
    "large_fallback_rows",
    "large_fallback_rate",
]

print("=" * 100)
print("DTE summary — key columns")
print("=" * 100)
display(by_tenor[[c for c in key_cols_tenor if c in by_tenor.columns]])

print("=" * 100)
print("Year summary — key columns")
print("=" * 100)
display(by_year[[c for c in key_cols_year if c in by_year.columns]])

print("=" * 100)
print("Variant summary — key columns")
print("=" * 100)
display(by_variant[[c for c in key_cols_variant if c in by_variant.columns]])

print("=" * 100)
print("Tenor × variant — key columns")
print("=" * 100)
display(
    by_tenor_variant[
        [c for c in ["variant"] + key_cols_tenor if c in by_tenor_variant.columns]
    ].sort_values(["variant", "tenor"])
)

DTE summary — key columns


,tenor,candidate_rows,pnl_complete_rows,fixed_filter_rate_vs_panel_rows,win_rate,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,fallback_used_rows,fallback_used_rate,large_fallback_rows,large_fallback_rate
0,9,249,249,0.152574,0.875502,0.008439,0.022567,-0.223085,-0.139989,0.614578,0.57,0.025806,0.024815,20,0.080321,0,0.000000
1,12,241,241,0.147944,0.879668,0.008024,0.025391,-0.288960,-0.139011,0.811162,0.80,0.028442,0.028387,39,0.161826,0,0.000000
2,15,231,231,0.141892,0.900433,0.009124,0.017214,-0.210929,-0.069752,0.647619,0.62,0.020246,0.020000,47,0.203463,0,0.000000
3,18,223,223,0.137231,0.928251,0.010689,0.019368,-0.351724,-0.038376,0.755247,0.73,0.021047,0.020333,71,0.318386,0,0.000000
4,21,214,214,0.131773,0.939252,0.009339,0.015973,-0.310850,-0.019808,0.683925,0.69,0.017107,0.016450,92,0.429907,0,0.000000
5,24,205,205,0.126387,0.912195,0.007078,0.014816,-0.263600,-0.052936,0.730780,0.73,0.016903,0.016400,98,0.478049,1,0.004878
6,27,200,200,0.123457,0.895000,0.007374,0.014579,-0.208995,-0.056689,0.783550,0.80,0.016858,0.015442,113,0.565000,1,0.005000
7,30,191,191,0.118047,0.910995,0.007687,0.014199,-0.284773,-0.058146,0.776597,0.78,0.015634,0.015000,120,0.628272,6,0.031414
8,33,187,187,0.115789,0.930481,0.010693,0.015744,-0.200216,-0.051474,0.883209,0.86,0.017124,0.016571,132,0.705882,14,0.074866


Year summary — key columns


,trade_year,candidate_rows,pnl_complete_rows,fixed_filter_rate_vs_panel_rows,win_rate,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,fallback_used_rows,fallback_used_rate,large_fallback_rows,large_fallback_rate
0,2020,8,8,0.003513,1.000000,0.017374,0.012969,0.010338,0.010732,0.560000,0.540,2,0.250000,0,0.000000
1,2021,120,120,0.052910,0.983333,0.007882,0.008832,-0.169440,0.004869,0.298333,0.280,18,0.150000,0,0.000000
2,2022,118,118,0.052236,0.983051,0.024235,0.024345,-0.125954,0.008132,0.975254,0.965,26,0.220339,0,0.000000
3,2023,212,212,0.094222,0.797170,-0.002720,0.024590,-0.285714,-0.189477,0.738821,0.705,32,0.150943,0,0.000000
4,2024,737,737,0.324956,0.911805,0.010002,0.019714,-0.351724,-0.066870,0.725455,0.710,307,0.416554,12,0.016282
5,2025,564,564,0.250667,0.987589,0.015759,0.015403,-0.220853,0.007167,0.729167,0.720,286,0.507092,10,0.017730
6,2026,182,182,0.174832,0.659341,-0.014882,0.012217,-0.263600,-0.130442,0.961593,0.925,61,0.335165,0,0.000000


Variant summary — key columns


,variant,candidate_rows,pnl_complete_rows,win_rate,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,fallback_used_rows,fallback_used_rate,large_fallback_rows,large_fallback_rate
0,all_pnl_complete,1941,1941,0.906749,0.008717,0.017035,-0.351724,-0.075878,0.738233,0.720,732,0.377125,22,0.011334
1,exact_primary_only,1209,1209,0.880066,0.006827,0.019576,-0.351724,-0.105522,0.769504,0.760,0,0.000000,0,0.000000
2,fallback_used_only,732,732,0.950820,0.011837,0.015022,-0.310850,0.002109,0.686585,0.665,732,1.000000,22,0.030055
3,exclude_large_fallback_flags,1919,1919,0.905680,0.008622,0.017035,-0.351724,-0.076059,0.738077,0.720,710,0.369984,0,0.000000
4,large_fallback_only,22,22,1.000000,0.016989,0.015887,0.008742,0.009930,0.751818,0.715,22,1.000000,22,1.000000


Tenor × variant — key columns


,variant,tenor,candidate_rows,pnl_complete_rows,win_rate,avg_return_on_max_loss,median_return_on_max_loss,worst_return_on_max_loss,p05_return_on_max_loss,avg_entry_credit,median_entry_credit,avg_credit_pct_width,median_credit_pct_width,fallback_used_rows,fallback_used_rate,large_fallback_rows,large_fallback_rate
0,all_pnl_complete,9,249,249,0.875502,0.008439,0.022567,-0.223085,-0.139989,0.614578,0.570,0.025806,0.024815,20,0.080321,0,0.000000
1,all_pnl_complete,12,241,241,0.879668,0.008024,0.025391,-0.288960,-0.139011,0.811162,0.800,0.028442,0.028387,39,0.161826,0,0.000000
2,all_pnl_complete,15,231,231,0.900433,0.009124,0.017214,-0.210929,-0.069752,0.647619,0.620,0.020246,0.020000,47,0.203463,0,0.000000
3,all_pnl_complete,18,223,223,0.928251,0.010689,0.019368,-0.351724,-0.038376,0.755247,0.730,0.021047,0.020333,71,0.318386,0,0.000000
4,all_pnl_complete,21,214,214,0.939252,0.009339,0.015973,-0.310850,-0.019808,0.683925,0.690,0.017107,0.016450,92,0.429907,0,0.000000
5,all_pnl_complete,24,205,205,0.912195,0.007078,0.014816,-0.263600,-0.052936,0.730780,0.730,0.016903,0.016400,98,0.478049,1,0.004878
6,all_pnl_complete,27,200,200,0.895000,0.007374,0.014579,-0.208995,-0.056689,0.783550,0.800,0.016858,0.015442,113,0.565000,1,0.005000
7,all_pnl_complete,30,191,191,0.910995,0.007687,0.014199,-0.284773,-0.058146,0.776597,0.780,0.015634,0.015000,120,0.628272,6,0.031414
8,all_pnl_complete,33,187,187,0.930481,0.010693,0.015744,-0.200216,-0.051474,0.883209,0.860,0.017124,0.016571,132,0.705882,14,0.074866
9,exact_primary_only,9,229,229,0.864629,0.007078,0.022567,-0.223085,-0.154456,0.617642,0.580,0.026024,0.025217,0,0.000000,0,0.000000
